# The Clown Project — V0.1

## Notebook 00: Run Contract and Source Registration

**Notebook:** `00_V01_RUN_CONTRACT.ipynb`  
**Pipeline version:** `V0.1`  
**Initial status:** `NOT EVALUATED`

### Purpose

This notebook registers the immutable V0.0 source collection and freezes the operating contract for one V0.1 research run.

It establishes:

- The authoritative V0.0 raw-source files
- The V0.0 artifacts that may be used for reconciliation only
- The V0.1 output locations
- Source and run identities
- Timestamp and event-ordering conventions
- Chronological split boundaries
- Missing-data and rejection rules
- Downstream access permissions
- The statistical authority of the available dataset

### Source authority

`D:\Clown Project\V0.0` is the immutable source and provenance tree.

V0.1 may read from this tree but must never overwrite, rename, move, delete, repair in place, or otherwise modify any V0.0 file.

The V0.0 raw collection is the primary reconstruction authority. V0.0 processed datasets, models, diagnostics, and gate reports are reference artifacts only and cannot silently replace V0.1 reconstruction.

### Output authority

Only files written under:

`D:\Clown Project\V0.1`

may be treated as V0.1 outputs.

Every authoritative V0.1 artifact must record:

- The V0.0 source run prefix
- The V0.1 run ID
- The producing notebook
- The source files used
- Input checksums
- Schema version
- Row counts
- Acceptance status

### Scope restriction

This notebook does not parse or clean the complete raw streams, reconstruct the order book, align trades to book states, construct market events, estimate Hawkes models, validate intensity signals, simulate quotes, or evaluate market-making performance.

Its sole purpose is to determine whether the source collection is unambiguous, auditable, causally usable, and sufficiently documented for Notebook 01 to begin.

In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import platform
import random
import subprocess
import sys

import numpy as np
import pandas as pd


# ============================================================
# NOTEBOOK IDENTITY
# ============================================================

PIPELINE_NAME = "The Clown Project"
PIPELINE_VERSION = "V0.1"

NOTEBOOK_NUMBER = "00"
NOTEBOOK_FILENAME = "00_V01_RUN_CONTRACT.ipynb"
NOTEBOOK_PURPOSE = "Run contract and immutable V0.0 source registration"

CONTRACT_SCHEMA_VERSION = "v0.1.0"
INITIAL_STATUS = "NOT EVALUATED"


# ============================================================
# SOURCE AND OUTPUT ROOTS
# ============================================================

V0_0_ROOT = Path(r"D:\Clown Project\V0.0")
V0_1_ROOT = Path(r"D:\Clown Project\V0.1")

SOURCE_RUN_PREFIX = "BTCUSDT_spot_20260710T063746Z_c8b5bf12"

V0_0_RAW_ROOT = V0_0_ROOT / "data" / "raw"
V0_0_PROCESSED_ROOT = V0_0_ROOT / "data" / "processed"
V0_0_ARTIFACT_ROOT = V0_0_ROOT / "artifacts" / "v0_0"
V0_0_NOTEBOOK_PATH = V0_0_ROOT / "notebooks" / "V0.0_development.ipynb"

V0_1_CONFIG_ROOT = V0_1_ROOT / "config"
V0_1_ARTIFACT_ROOT = V0_1_ROOT / "artifacts"
V0_1_MANIFEST_ROOT = V0_1_ARTIFACT_ROOT / "manifests"
V0_1_AUDIT_ROOT = V0_1_ARTIFACT_ROOT / "audit_tables"


# ============================================================
# GLOBAL RUN CONVENTIONS
# ============================================================

CANONICAL_TIMEZONE = "UTC"
HASH_ALGORITHM = "sha256"
RANDOM_SEED = 20260710

EXECUTION_STARTED_UTC = (
    datetime.now(timezone.utc)
    .replace(microsecond=0)
    .isoformat()
)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


# ============================================================
# BOOTSTRAP IDENTITY
# ============================================================

BOOTSTRAP_IDENTITY = {
    "pipeline_name": PIPELINE_NAME,
    "pipeline_version": PIPELINE_VERSION,
    "notebook_number": NOTEBOOK_NUMBER,
    "notebook_filename": NOTEBOOK_FILENAME,
    "notebook_purpose": NOTEBOOK_PURPOSE,
    "contract_schema_version": CONTRACT_SCHEMA_VERSION,
    "initial_status": INITIAL_STATUS,
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "v0_0_root": str(V0_0_ROOT),
    "v0_1_root": str(V0_1_ROOT),
    "canonical_timezone": CANONICAL_TIMEZONE,
    "hash_algorithm": HASH_ALGORITHM,
    "random_seed": RANDOM_SEED,
    "execution_started_utc": EXECUTION_STARTED_UTC,
}

pd.DataFrame(
    BOOTSTRAP_IDENTITY.items(),
    columns=["field", "value"],
).style.hide(axis="index")

field,value
pipeline_name,The Clown Project
pipeline_version,V0.1
notebook_number,00
notebook_filename,00_V01_RUN_CONTRACT.ipynb
notebook_purpose,Run contract and immutable V0.0 source registration
contract_schema_version,v0.1.0
initial_status,NOT EVALUATED
source_run_prefix,BTCUSDT_spot_20260710T063746Z_c8b5bf12
v0_0_root,D:\Clown Project\V0.0
v0_1_root,D:\Clown Project\V0.1


In [2]:
# ============================================================
# PROJECT ROOT RESOLUTION AND SEPARATION AUDIT
# ============================================================

def resolve_path(path: Path) -> Path:
    """Return an absolute normalized path without requiring it to exist."""
    return path.expanduser().resolve(strict=False)


def is_within(child: Path, parent: Path) -> bool:
    """Return True when child is equal to or contained within parent."""
    child_normalized = Path(os.path.normcase(str(resolve_path(child))))
    parent_normalized = Path(os.path.normcase(str(resolve_path(parent))))

    try:
        child_normalized.relative_to(parent_normalized)
        return True
    except ValueError:
        return False


V0_0_ROOT_RESOLVED = resolve_path(V0_0_ROOT)
V0_1_ROOT_RESOLVED = resolve_path(V0_1_ROOT)

ROOTS_OVERLAP = (
    is_within(V0_0_ROOT_RESOLVED, V0_1_ROOT_RESOLVED)
    or is_within(V0_1_ROOT_RESOLVED, V0_0_ROOT_RESOLVED)
)

if ROOTS_OVERLAP:
    raise RuntimeError(
        "Critical path failure: V0.0 and V0.1 overlap.\n"
        f"V0.0: {V0_0_ROOT_RESOLVED}\n"
        f"V0.1: {V0_1_ROOT_RESOLVED}"
    )


# V0.1 directories may be created only after separation is proven.
V0_1_REQUIRED_DIRECTORIES = [
    V0_1_ROOT,
    V0_1_ROOT / "notebooks",
    V0_1_CONFIG_ROOT,
    V0_1_ARTIFACT_ROOT,
    V0_1_MANIFEST_ROOT,
    V0_1_AUDIT_ROOT,
    V0_1_ROOT / "logs",
]

for directory in V0_1_REQUIRED_DIRECTORIES:
    resolved_directory = resolve_path(directory)

    if not is_within(resolved_directory, V0_1_ROOT_RESOLVED):
        raise RuntimeError(
            f"Refusing to create output directory outside V0.1: "
            f"{resolved_directory}"
        )

    resolved_directory.mkdir(parents=True, exist_ok=True)


ROOT_AUDIT_ROWS = [
    {
        "role": "V0.0 project root",
        "resolved_path": str(V0_0_ROOT_RESOLVED),
        "exists": V0_0_ROOT_RESOLVED.exists(),
        "is_directory": V0_0_ROOT_RESOLVED.is_dir(),
        "authority": "SOURCE_AND_PROVENANCE",
        "pipeline_write_allowed": False,
    },
    {
        "role": "V0.0 raw root",
        "resolved_path": str(resolve_path(V0_0_RAW_ROOT)),
        "exists": resolve_path(V0_0_RAW_ROOT).exists(),
        "is_directory": resolve_path(V0_0_RAW_ROOT).is_dir(),
        "authority": "PRIMARY_RAW",
        "pipeline_write_allowed": False,
    },
    {
        "role": "V0.0 processed root",
        "resolved_path": str(resolve_path(V0_0_PROCESSED_ROOT)),
        "exists": resolve_path(V0_0_PROCESSED_ROOT).exists(),
        "is_directory": resolve_path(V0_0_PROCESSED_ROOT).is_dir(),
        "authority": "REFERENCE_ONLY",
        "pipeline_write_allowed": False,
    },
    {
        "role": "V0.0 artifact root",
        "resolved_path": str(resolve_path(V0_0_ARTIFACT_ROOT)),
        "exists": resolve_path(V0_0_ARTIFACT_ROOT).exists(),
        "is_directory": resolve_path(V0_0_ARTIFACT_ROOT).is_dir(),
        "authority": "REFERENCE_ONLY",
        "pipeline_write_allowed": False,
    },
    {
        "role": "V0.1 project root",
        "resolved_path": str(V0_1_ROOT_RESOLVED),
        "exists": V0_1_ROOT_RESOLVED.exists(),
        "is_directory": V0_1_ROOT_RESOLVED.is_dir(),
        "authority": "V0_1_OUTPUT",
        "pipeline_write_allowed": True,
    },
    {
        "role": "V0.1 config root",
        "resolved_path": str(resolve_path(V0_1_CONFIG_ROOT)),
        "exists": resolve_path(V0_1_CONFIG_ROOT).exists(),
        "is_directory": resolve_path(V0_1_CONFIG_ROOT).is_dir(),
        "authority": "V0_1_OUTPUT",
        "pipeline_write_allowed": True,
    },
    {
        "role": "V0.1 artifact root",
        "resolved_path": str(resolve_path(V0_1_ARTIFACT_ROOT)),
        "exists": resolve_path(V0_1_ARTIFACT_ROOT).exists(),
        "is_directory": resolve_path(V0_1_ARTIFACT_ROOT).is_dir(),
        "authority": "V0_1_OUTPUT",
        "pipeline_write_allowed": True,
    },
]

PROJECT_ROOT_AUDIT = pd.DataFrame(ROOT_AUDIT_ROWS)

PROJECT_ROOT_AUDIT["status"] = np.where(
    PROJECT_ROOT_AUDIT["exists"]
    & PROJECT_ROOT_AUDIT["is_directory"],
    "PASS",
    "FAIL",
)

critical_failures = PROJECT_ROOT_AUDIT.loc[
    PROJECT_ROOT_AUDIT["role"].isin(
        [
            "V0.0 project root",
            "V0.0 raw root",
            "V0.1 project root",
            "V0.1 config root",
            "V0.1 artifact root",
        ]
    )
    & PROJECT_ROOT_AUDIT["status"].eq("FAIL")
]

display(PROJECT_ROOT_AUDIT)

if not critical_failures.empty:
    failed_roles = critical_failures["role"].tolist()
    raise RuntimeError(
        "Critical project-root audit failure: "
        + ", ".join(failed_roles)
    )

print("Project-root separation and directory audit: PASS")

,role,resolved_path,exists,is_directory,authority,pipeline_write_allowed,status
0,V0.0 project root,D:\Clown Project\V0.0,True,True,SOURCE_AND_PROVENANCE,False,PASS
1,V0.0 raw root,D:\Clown Project\V0.0\data\raw,True,True,PRIMARY_RAW,False,PASS
2,V0.0 processed root,D:\Clown Project\V0.0\data\processed,True,True,REFERENCE_ONLY,False,PASS
3,V0.0 artifact root,D:\Clown Project\V0.0\artifacts\v0_0,True,True,REFERENCE_ONLY,False,PASS
4,V0.1 project root,D:\Clown Project\V0.1,True,True,V0_1_OUTPUT,True,PASS
5,V0.1 config root,D:\Clown Project\V0.1\config,True,True,V0_1_OUTPUT,True,PASS
6,V0.1 artifact root,D:\Clown Project\V0.1\artifacts,True,True,V0_1_OUTPUT,True,PASS


Project-root separation and directory audit: PASS


In [3]:
# ============================================================
# SOURCE AUTHORITY REGISTRY
# ============================================================

def register_path(
    logical_name: str,
    path: Path,
    authority_class: str,
    required: bool,
    object_type: str,
    allowed_operations: str,
) -> dict:
    """Create one normalized source-registry record."""
    resolved = resolve_path(path)

    return {
        "logical_name": logical_name,
        "resolved_path": str(resolved),
        "relative_path": (
            str(resolved.relative_to(V0_0_ROOT_RESOLVED))
            if is_within(resolved, V0_0_ROOT_RESOLVED)
            else (
                str(resolved.relative_to(V0_1_ROOT_RESOLVED))
                if is_within(resolved, V0_1_ROOT_RESOLVED)
                else None
            )
        ),
        "authority_class": authority_class,
        "required": required,
        "object_type": object_type,
        "allowed_operations": allowed_operations,
        "exists": resolved.exists(),
        "is_directory": resolved.is_dir(),
        "is_file": resolved.is_file(),
    }


SOURCE_REGISTRY_ROWS = [
    # --------------------------------------------------------
    # PRIMARY RAW AUTHORITY
    # --------------------------------------------------------
    register_path(
        logical_name="v0_0_raw_root",
        path=V0_0_RAW_ROOT,
        authority_class="PRIMARY_RAW",
        required=True,
        object_type="directory",
        allowed_operations="READ_ONLY",
    ),

    # --------------------------------------------------------
    # V0.0 REFERENCE AUTHORITY
    # --------------------------------------------------------
    register_path(
        logical_name="v0_0_processed_root",
        path=V0_0_PROCESSED_ROOT,
        authority_class="V0_0_REFERENCE",
        required=False,
        object_type="directory",
        allowed_operations="READ_AND_RECONCILE_ONLY",
    ),
    register_path(
        logical_name="v0_0_processed_book",
        path=V0_0_PROCESSED_ROOT / "book",
        authority_class="V0_0_REFERENCE",
        required=False,
        object_type="directory",
        allowed_operations="READ_AND_RECONCILE_ONLY",
    ),
    register_path(
        logical_name="v0_0_processed_trades",
        path=V0_0_PROCESSED_ROOT / "trades",
        authority_class="V0_0_REFERENCE",
        required=False,
        object_type="directory",
        allowed_operations="READ_AND_RECONCILE_ONLY",
    ),
    register_path(
        logical_name="v0_0_processed_events",
        path=V0_0_PROCESSED_ROOT / "events",
        authority_class="V0_0_REFERENCE",
        required=False,
        object_type="directory",
        allowed_operations="READ_AND_RECONCILE_ONLY",
    ),
    register_path(
        logical_name="v0_0_processed_point_process",
        path=V0_0_PROCESSED_ROOT / "point_process",
        authority_class="V0_0_REFERENCE",
        required=False,
        object_type="directory",
        allowed_operations="READ_AND_RECONCILE_ONLY",
    ),
    register_path(
        logical_name="v0_0_processed_hawkes",
        path=V0_0_PROCESSED_ROOT / "hawkes",
        authority_class="V0_0_REFERENCE",
        required=False,
        object_type="directory",
        allowed_operations="READ_AND_RECONCILE_ONLY",
    ),
    register_path(
        logical_name="v0_0_artifact_root",
        path=V0_0_ARTIFACT_ROOT,
        authority_class="V0_0_REFERENCE",
        required=False,
        object_type="directory",
        allowed_operations="READ_AND_RECONCILE_ONLY",
    ),
    register_path(
        logical_name="v0_0_development_notebook",
        path=V0_0_NOTEBOOK_PATH,
        authority_class="V0_0_REFERENCE",
        required=False,
        object_type="file",
        allowed_operations="READ_ONLY",
    ),

    # --------------------------------------------------------
    # V0.1 OUTPUT AUTHORITY
    # --------------------------------------------------------
    register_path(
        logical_name="v0_1_root",
        path=V0_1_ROOT,
        authority_class="V0_1_OUTPUT",
        required=True,
        object_type="directory",
        allowed_operations="READ_WRITE",
    ),
    register_path(
        logical_name="v0_1_config_root",
        path=V0_1_CONFIG_ROOT,
        authority_class="V0_1_OUTPUT",
        required=True,
        object_type="directory",
        allowed_operations="READ_WRITE",
    ),
    register_path(
        logical_name="v0_1_manifest_root",
        path=V0_1_MANIFEST_ROOT,
        authority_class="V0_1_OUTPUT",
        required=True,
        object_type="directory",
        allowed_operations="READ_WRITE",
    ),
    register_path(
        logical_name="v0_1_audit_root",
        path=V0_1_AUDIT_ROOT,
        authority_class="V0_1_OUTPUT",
        required=True,
        object_type="directory",
        allowed_operations="READ_WRITE",
    ),
]


SOURCE_AUTHORITY_REGISTRY = pd.DataFrame(SOURCE_REGISTRY_ROWS)


# ============================================================
# REGISTRY VALIDATION
# ============================================================

valid_authority_classes = {
    "PRIMARY_RAW",
    "V0_0_REFERENCE",
    "V0_1_OUTPUT",
}

invalid_authority_rows = SOURCE_AUTHORITY_REGISTRY.loc[
    ~SOURCE_AUTHORITY_REGISTRY["authority_class"].isin(
        valid_authority_classes
    )
]

duplicate_logical_names = SOURCE_AUTHORITY_REGISTRY.loc[
    SOURCE_AUTHORITY_REGISTRY["logical_name"].duplicated(keep=False)
]

required_missing = SOURCE_AUTHORITY_REGISTRY.loc[
    SOURCE_AUTHORITY_REGISTRY["required"]
    & ~SOURCE_AUTHORITY_REGISTRY["exists"]
]

type_mismatches = SOURCE_AUTHORITY_REGISTRY.loc[
    (
        SOURCE_AUTHORITY_REGISTRY["object_type"].eq("directory")
        & SOURCE_AUTHORITY_REGISTRY["exists"]
        & ~SOURCE_AUTHORITY_REGISTRY["is_directory"]
    )
    |
    (
        SOURCE_AUTHORITY_REGISTRY["object_type"].eq("file")
        & SOURCE_AUTHORITY_REGISTRY["exists"]
        & ~SOURCE_AUTHORITY_REGISTRY["is_file"]
    )
]


def registry_location_is_valid(row: pd.Series) -> bool:
    resolved = Path(row["resolved_path"])

    if row["authority_class"] in {"PRIMARY_RAW", "V0_0_REFERENCE"}:
        return is_within(resolved, V0_0_ROOT_RESOLVED)

    if row["authority_class"] == "V0_1_OUTPUT":
        return is_within(resolved, V0_1_ROOT_RESOLVED)

    return False


SOURCE_AUTHORITY_REGISTRY["location_valid"] = (
    SOURCE_AUTHORITY_REGISTRY.apply(
        registry_location_is_valid,
        axis=1,
    )
)

SOURCE_AUTHORITY_REGISTRY["status"] = np.select(
    [
        ~SOURCE_AUTHORITY_REGISTRY["location_valid"],
        SOURCE_AUTHORITY_REGISTRY["required"]
        & ~SOURCE_AUTHORITY_REGISTRY["exists"],
        SOURCE_AUTHORITY_REGISTRY["exists"],
    ],
    [
        "FAIL",
        "FAIL",
        "PASS",
    ],
    default="NOT_PRESENT_OPTIONAL",
)


registry_failures = SOURCE_AUTHORITY_REGISTRY.loc[
    SOURCE_AUTHORITY_REGISTRY["status"].eq("FAIL")
]


display(
    SOURCE_AUTHORITY_REGISTRY[
        [
            "logical_name",
            "authority_class",
            "required",
            "object_type",
            "exists",
            "allowed_operations",
            "location_valid",
            "status",
            "resolved_path",
        ]
    ]
)


if not invalid_authority_rows.empty:
    raise RuntimeError(
        "Source registry contains an invalid authority class."
    )

if not duplicate_logical_names.empty:
    raise RuntimeError(
        "Source registry contains duplicate logical names: "
        + ", ".join(
            duplicate_logical_names["logical_name"]
            .drop_duplicates()
            .tolist()
        )
    )

if not required_missing.empty:
    raise RuntimeError(
        "Required registered paths are missing: "
        + ", ".join(required_missing["logical_name"].tolist())
    )

if not type_mismatches.empty:
    raise RuntimeError(
        "Registered path type mismatch: "
        + ", ".join(type_mismatches["logical_name"].tolist())
    )

if not registry_failures.empty:
    raise RuntimeError(
        "Source authority registry validation failed: "
        + ", ".join(registry_failures["logical_name"].tolist())
    )


optional_missing_count = int(
    SOURCE_AUTHORITY_REGISTRY["status"]
    .eq("NOT_PRESENT_OPTIONAL")
    .sum()
)

print("Source authority registry: PASS")
print(f"Registered paths: {len(SOURCE_AUTHORITY_REGISTRY):,}")
print(f"Optional paths not present: {optional_missing_count:,}")

,logical_name,authority_class,required,object_type,exists,allowed_operations,location_valid,status,resolved_path
0,v0_0_raw_root,PRIMARY_RAW,True,directory,True,READ_ONLY,True,PASS,D:\Clown Project\V0.0\data\raw
1,v0_0_processed_root,V0_0_REFERENCE,False,directory,True,READ_AND_RECONCILE_ONLY,True,PASS,D:\Clown Project\V0.0\data\processed
2,v0_0_processed_book,V0_0_REFERENCE,False,directory,False,READ_AND_RECONCILE_ONLY,True,NOT_PRESENT_OPTIONAL,D:\Clown Project\V0.0\data\processed\book
3,v0_0_processed_trades,V0_0_REFERENCE,False,directory,False,READ_AND_RECONCILE_ONLY,True,NOT_PRESENT_OPTIONAL,D:\Clown Project\V0.0\data\processed\trades
4,v0_0_processed_events,V0_0_REFERENCE,False,directory,False,READ_AND_RECONCILE_ONLY,True,NOT_PRESENT_OPTIONAL,D:\Clown Project\V0.0\data\processed\events
5,v0_0_processed_point_process,V0_0_REFERENCE,False,directory,True,READ_AND_RECONCILE_ONLY,True,PASS,D:\Clown Project\V0.0\data\processed\point_pro...
6,v0_0_processed_hawkes,V0_0_REFERENCE,False,directory,True,READ_AND_RECONCILE_ONLY,True,PASS,D:\Clown Project\V0.0\data\processed\hawkes
7,v0_0_artifact_root,V0_0_REFERENCE,False,directory,True,READ_AND_RECONCILE_ONLY,True,PASS,D:\Clown Project\V0.0\artifacts\v0_0
8,v0_0_development_notebook,V0_0_REFERENCE,False,file,False,READ_ONLY,True,NOT_PRESENT_OPTIONAL,D:\Clown Project\V0.0\notebooks\V0.0_developme...
9,v0_1_root,V0_1_OUTPUT,True,directory,True,READ_WRITE,True,PASS,D:\Clown Project\V0.1


Source authority registry: PASS
Registered paths: 13
Optional paths not present: 4


In [4]:
# ============================================================
# RAW FILE DISCOVERY AND PRELIMINARY CLASSIFICATION
# ============================================================

import re


RUN_PREFIX_PATTERN = re.compile(
    r"BTCUSDT_spot_\d{8}T\d{6}Z_[A-Za-z0-9]+",
    flags=re.IGNORECASE,
)

TEXT_LIKE_SUFFIXES = {
    ".json",
    ".jsonl",
    ".ndjson",
    ".txt",
    ".log",
    ".csv",
    ".yaml",
    ".yml",
}

ROLE_PRIORITY = [
    "TRADE_STREAM",
    "DEPTH_STREAM",
    "REST_SNAPSHOT",
    "RUN_MANIFEST",
    "COLLECTOR_METADATA",
    "COLLECTOR_LOG",
    "ERROR_LOG",
    "UNCLASSIFIED",
]


def read_small_text_sample(
    path: Path,
    max_bytes: int = 16_384,
) -> str:
    """
    Read only a small leading sample for lightweight classification.

    This does not constitute full parsing or validation.
    """
    if path.suffix.lower() not in TEXT_LIKE_SUFFIXES:
        return ""

    try:
        with path.open("rb") as handle:
            raw = handle.read(max_bytes)

        return raw.decode("utf-8", errors="ignore").lower()

    except OSError:
        return ""


def detect_run_prefix(path: Path) -> str | None:
    """Extract a run prefix from the path when one is present."""
    match = RUN_PREFIX_PATTERN.search(str(path))

    if match is None:
        return None

    return match.group(0)


def classify_raw_file(path: Path) -> tuple[str, str]:
    """
    Assign a preliminary source role using filename, directory,
    extension, and a small non-authoritative content sample.
    """
    name = path.name.lower()
    full_path_text = str(path).lower()
    sample = read_small_text_sample(path)

    filename_tokens = {
        token
        for token in re.split(r"[^a-z0-9]+", name)
        if token
    }

    # --------------------------------------------------------
    # LOGS
    # --------------------------------------------------------
    if (
        path.suffix.lower() == ".log"
        or "log" in filename_tokens
        or "\\logs\\" in full_path_text
    ):
        if any(
            token in filename_tokens
            for token in {"error", "errors", "failure", "failures"}
        ):
            return "ERROR_LOG", "filename_or_directory_pattern"

        return "COLLECTOR_LOG", "filename_or_directory_pattern"

    # --------------------------------------------------------
    # RUN MANIFEST
    # --------------------------------------------------------
    if any(
        token in filename_tokens
        for token in {"manifest", "inventory", "registry"}
    ):
        return "RUN_MANIFEST", "filename_pattern"

    # --------------------------------------------------------
    # COLLECTOR METADATA
    # --------------------------------------------------------
    if any(
        token in filename_tokens
        for token in {
            "metadata",
            "meta",
            "session",
            "environment",
            "clock",
            "host",
            "config",
            "configuration",
        }
    ):
        return "COLLECTOR_METADATA", "filename_pattern"

    # --------------------------------------------------------
    # REST SNAPSHOT
    # --------------------------------------------------------
    if any(
        token in filename_tokens
        for token in {"snapshot", "rest", "initialbook", "initial"}
    ):
        if any(
            marker in sample
            for marker in {
                '"lastupdateid"',
                '"bids"',
                '"asks"',
            }
        ):
            return "REST_SNAPSHOT", "filename_and_content_signature"

        return "REST_SNAPSHOT", "filename_pattern"

    if (
        '"lastupdateid"' in sample
        and '"bids"' in sample
        and '"asks"' in sample
    ):
        return "REST_SNAPSHOT", "content_signature"

    # --------------------------------------------------------
    # DIFFERENTIAL DEPTH STREAM
    # --------------------------------------------------------
    depth_filename_markers = {
        "depth",
        "bookdepth",
        "diffdepth",
        "differential",
        "orderbook",
        "book",
    }

    if filename_tokens.intersection(depth_filename_markers):
        return "DEPTH_STREAM", "filename_pattern"

    if any(
        marker in sample
        for marker in {
            '"depthupdate"',
            '"e":"depthupdate"',
            '"first update id"',
            '"final update id"',
            '"u":',
            '"u": ',
        }
    ) and (
        '"b":' in sample
        or '"a":' in sample
        or '"bids"' in sample
        or '"asks"' in sample
    ):
        return "DEPTH_STREAM", "content_signature"

    # --------------------------------------------------------
    # TRADE STREAM
    # --------------------------------------------------------
    trade_filename_markers = {
        "trade",
        "trades",
        "aggtrade",
        "aggtrades",
        "executions",
        "fills",
    }

    if filename_tokens.intersection(trade_filename_markers):
        return "TRADE_STREAM", "filename_pattern"

    if any(
        marker in sample
        for marker in {
            '"e":"trade"',
            '"e": "trade"',
            '"e":"aggtrade"',
            '"e": "aggtrade"',
        }
    ):
        return "TRADE_STREAM", "content_signature"

    return "UNCLASSIFIED", "no_reliable_signature"


# ============================================================
# DISCOVER FILES
# ============================================================

RAW_FILES = sorted(
    (
        path
        for path in V0_0_RAW_ROOT.rglob("*")
        if path.is_file()
    ),
    key=lambda path: str(path).lower(),
)

if not RAW_FILES:
    raise RuntimeError(
        f"No files were discovered under the registered raw root: "
        f"{V0_0_RAW_ROOT}"
    )


RAW_FILE_ROWS = []

for path in RAW_FILES:
    resolved = resolve_path(path)
    stat = resolved.stat()

    detected_role, detection_basis = classify_raw_file(resolved)
    detected_run_prefix = detect_run_prefix(resolved)

    if detected_run_prefix is None:
        run_relation = "UNTAGGED"
    elif detected_run_prefix.lower() == SOURCE_RUN_PREFIX.lower():
        run_relation = "DECLARED_RUN"
    else:
        run_relation = "FOREIGN_RUN"

    RAW_FILE_ROWS.append(
        {
            "filename": resolved.name,
            "relative_path": str(
                resolved.relative_to(resolve_path(V0_0_RAW_ROOT))
            ),
            "resolved_path": str(resolved),
            "suffix": resolved.suffix.lower(),
            "size_bytes": int(stat.st_size),
            "modified_utc": datetime.fromtimestamp(
                stat.st_mtime,
                tz=timezone.utc,
            ).replace(microsecond=0).isoformat(),
            "detected_role": detected_role,
            "detection_basis": detection_basis,
            "detected_run_prefix": detected_run_prefix,
            "run_relation": run_relation,
            "eligible_for_source_review": (
                run_relation != "FOREIGN_RUN"
                and stat.st_size > 0
            ),
        }
    )


RAW_FILE_CLASSIFICATION = pd.DataFrame(RAW_FILE_ROWS)

RAW_FILE_CLASSIFICATION["role_order"] = (
    RAW_FILE_CLASSIFICATION["detected_role"]
    .map({role: index for index, role in enumerate(ROLE_PRIORITY)})
    .fillna(len(ROLE_PRIORITY))
)

RAW_FILE_CLASSIFICATION = (
    RAW_FILE_CLASSIFICATION
    .sort_values(
        ["role_order", "relative_path"],
        kind="stable",
    )
    .drop(columns="role_order")
    .reset_index(drop=True)
)


# ============================================================
# DISCOVERY SUMMARY
# ============================================================

RAW_DISCOVERY_SUMMARY = (
    RAW_FILE_CLASSIFICATION
    .groupby(
        ["detected_role", "run_relation"],
        dropna=False,
    )
    .agg(
        file_count=("filename", "size"),
        total_bytes=("size_bytes", "sum"),
        eligible_files=("eligible_for_source_review", "sum"),
    )
    .reset_index()
    .sort_values(
        ["detected_role", "run_relation"],
        kind="stable",
    )
    .reset_index(drop=True)
)


zero_byte_files = RAW_FILE_CLASSIFICATION.loc[
    RAW_FILE_CLASSIFICATION["size_bytes"].eq(0)
]

foreign_run_files = RAW_FILE_CLASSIFICATION.loc[
    RAW_FILE_CLASSIFICATION["run_relation"].eq("FOREIGN_RUN")
]

unclassified_files = RAW_FILE_CLASSIFICATION.loc[
    RAW_FILE_CLASSIFICATION["detected_role"].eq("UNCLASSIFIED")
]


display(RAW_DISCOVERY_SUMMARY)
display(
    RAW_FILE_CLASSIFICATION[
        [
            "filename",
            "relative_path",
            "detected_role",
            "detection_basis",
            "run_relation",
            "size_bytes",
            "eligible_for_source_review",
        ]
    ]
)

print(f"Raw files discovered: {len(RAW_FILE_CLASSIFICATION):,}")
print(f"Zero-byte files: {len(zero_byte_files):,}")
print(f"Foreign-run files: {len(foreign_run_files):,}")
print(f"Unclassified files: {len(unclassified_files):,}")

,detected_role,run_relation,file_count,total_bytes,eligible_files
0,COLLECTOR_METADATA,DECLARED_RUN,1,1777,1
1,COLLECTOR_METADATA,FOREIGN_RUN,2,2428,0
2,DEPTH_STREAM,DECLARED_RUN,1,63443258,1
3,DEPTH_STREAM,FOREIGN_RUN,1,5238188,0
4,REST_SNAPSHOT,DECLARED_RUN,1,360812,1
5,REST_SNAPSHOT,FOREIGN_RUN,1,360811,0
6,RUN_MANIFEST,DECLARED_RUN,1,1738,1
7,TRADE_STREAM,DECLARED_RUN,1,46082102,1
8,TRADE_STREAM,FOREIGN_RUN,1,4150217,0
9,UNCLASSIFIED,DECLARED_RUN,1,945,1


,filename,relative_path,detected_role,detection_basis,run_relation,size_bytes,eligible_for_source_review
0,BTCUSDT_spot_20260710T060324Z_aa755c97_trades....,trades\BTCUSDT_spot_20260710T060324Z_aa755c97_...,TRADE_STREAM,filename_pattern,FOREIGN_RUN,4150217,False
1,BTCUSDT_spot_20260710T063746Z_c8b5bf12_trades....,trades\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,TRADE_STREAM,filename_pattern,DECLARED_RUN,46082102,True
2,BTCUSDT_spot_20260710T060324Z_aa755c97_depth_u...,order_book\BTCUSDT_spot_20260710T060324Z_aa755...,DEPTH_STREAM,filename_pattern,FOREIGN_RUN,5238188,False
3,BTCUSDT_spot_20260710T063746Z_c8b5bf12_depth_u...,order_book\BTCUSDT_spot_20260710T063746Z_c8b5b...,DEPTH_STREAM,filename_pattern,DECLARED_RUN,63443258,True
4,BTCUSDT_spot_20260710T060324Z_aa755c97_snapsho...,order_book\BTCUSDT_spot_20260710T060324Z_aa755...,REST_SNAPSHOT,filename_and_content_signature,FOREIGN_RUN,360811,False
5,BTCUSDT_spot_20260710T063746Z_c8b5bf12_snapsho...,order_book\BTCUSDT_spot_20260710T063746Z_c8b5b...,REST_SNAPSHOT,filename_and_content_signature,DECLARED_RUN,360812,True
6,BTCUSDT_spot_20260710T063746Z_c8b5bf12_develop...,metadata\BTCUSDT_spot_20260710T063746Z_c8b5bf1...,RUN_MANIFEST,filename_pattern,DECLARED_RUN,1738,True
7,BTCUSDT_spot_20260710T060324Z_aa755c97_clock_o...,metadata\BTCUSDT_spot_20260710T060324Z_aa755c9...,COLLECTOR_METADATA,filename_pattern,FOREIGN_RUN,657,False
8,BTCUSDT_spot_20260710T060324Z_aa755c97_session...,metadata\BTCUSDT_spot_20260710T060324Z_aa755c9...,COLLECTOR_METADATA,filename_pattern,FOREIGN_RUN,1771,False
9,BTCUSDT_spot_20260710T063746Z_c8b5bf12_session...,metadata\BTCUSDT_spot_20260710T063746Z_c8b5bf1...,COLLECTOR_METADATA,filename_pattern,DECLARED_RUN,1777,True


Raw files discovered: 13
Zero-byte files: 0
Foreign-run files: 6
Unclassified files: 3


In [6]:
# ============================================================
# SOURCE ROLE AUDIT AND AUTHORITATIVE FILE SELECTION
# ============================================================

RAW_FILE_CLASSIFICATION_REFINED = RAW_FILE_CLASSIFICATION.copy()


def refine_source_role(row: pd.Series) -> str:
    """
    Refine files that were intentionally left unclassified during
    preliminary discovery.
    """
    filename = str(row["filename"]).lower()
    suffix = str(row["suffix"]).lower()
    current_role = str(row["detected_role"])

    if current_role != "UNCLASSIFIED":
        return current_role

    if (
        suffix == ".csv"
        and any(token in filename for token in ("sha256", "checksum", "checksums"))
    ):
        return "CHECKSUM_MANIFEST"

    if filename.startswith("readme"):
        return "SOURCE_DOCUMENTATION"

    return "UNCLASSIFIED"


RAW_FILE_CLASSIFICATION_REFINED["source_role"] = (
    RAW_FILE_CLASSIFICATION_REFINED.apply(
        refine_source_role,
        axis=1,
    )
)


# ============================================================
# SOURCE ROLE CONTRACT
# ============================================================

SOURCE_ROLE_CONTRACT = pd.DataFrame(
    [
        {
            "source_role": "TRADE_STREAM",
            "required": True,
            "expected_count": 1,
            "selection_authority": True,
        },
        {
            "source_role": "DEPTH_STREAM",
            "required": True,
            "expected_count": 1,
            "selection_authority": True,
        },
        {
            "source_role": "REST_SNAPSHOT",
            "required": True,
            "expected_count": 1,
            "selection_authority": True,
        },
        {
            "source_role": "RUN_MANIFEST",
            "required": True,
            "expected_count": 1,
            "selection_authority": True,
        },
        {
            "source_role": "COLLECTOR_METADATA",
            "required": True,
            "expected_count": 1,
            "selection_authority": True,
        },
        {
            "source_role": "CHECKSUM_MANIFEST",
            "required": False,
            "expected_count": 1,
            "selection_authority": False,
        },
        {
            "source_role": "COLLECTOR_LOG",
            "required": False,
            "expected_count": None,
            "selection_authority": False,
        },
        {
            "source_role": "ERROR_LOG",
            "required": False,
            "expected_count": None,
            "selection_authority": False,
        },
        {
            "source_role": "SOURCE_DOCUMENTATION",
            "required": False,
            "expected_count": None,
            "selection_authority": False,
        },
        {
            "source_role": "UNCLASSIFIED",
            "required": False,
            "expected_count": None,
            "selection_authority": False,
        },
    ]
)

# Preserve missing expected counts safely instead of converting them
# implicitly through float NaN values.
SOURCE_ROLE_CONTRACT["expected_count"] = pd.array(
    SOURCE_ROLE_CONTRACT["expected_count"],
    dtype="Int64",
)


# ============================================================
# BUILD ROLE AUDIT WITHOUT NaN-TO-INTEGER CONVERSION
# ============================================================

role_audit_rows = []

for contract_row in SOURCE_ROLE_CONTRACT.itertuples(index=False):
    role = contract_row.source_role
    required = bool(contract_row.required)
    selection_authority = bool(contract_row.selection_authority)

    expected_count = (
        None
        if pd.isna(contract_row.expected_count)
        else int(contract_row.expected_count)
    )

    role_files = RAW_FILE_CLASSIFICATION_REFINED.loc[
        RAW_FILE_CLASSIFICATION_REFINED["source_role"].eq(role)
    ]

    declared_run_files = role_files.loc[
        role_files["run_relation"].eq("DECLARED_RUN")
        & role_files["eligible_for_source_review"]
    ]

    untagged_files = role_files.loc[
        role_files["run_relation"].eq("UNTAGGED")
        & role_files["eligible_for_source_review"]
    ]

    foreign_run_files_for_role = role_files.loc[
        role_files["run_relation"].eq("FOREIGN_RUN")
    ]

    declared_count = int(len(declared_run_files))
    untagged_count = int(len(untagged_files))
    foreign_count = int(len(foreign_run_files_for_role))

    if required:
        if declared_count == 0:
            status = "FAIL_MISSING"
        elif expected_count is not None and declared_count > expected_count:
            status = "FAIL_AMBIGUOUS"
        elif expected_count is not None and declared_count < expected_count:
            status = "FAIL_COUNT_MISMATCH"
        else:
            status = "PASS_REQUIRED"
    else:
        eligible_optional_count = declared_count + untagged_count

        if eligible_optional_count == 0:
            status = "NOT_PRESENT_OPTIONAL"
        elif (
            expected_count is not None
            and eligible_optional_count > expected_count
        ):
            status = "WARNING_MULTIPLE_OPTIONAL"
        elif (
            expected_count is not None
            and eligible_optional_count < expected_count
        ):
            status = "WARNING_COUNT_MISMATCH"
        else:
            status = "PASS_OPTIONAL_PRESENT"

    role_audit_rows.append(
        {
            "source_role": role,
            "required": required,
            "expected_count": expected_count,
            "selection_authority": selection_authority,
            "declared_run_candidates": declared_count,
            "untagged_candidates": untagged_count,
            "foreign_run_files": foreign_count,
            "status": status,
        }
    )


SOURCE_ROLE_AUDIT = pd.DataFrame(role_audit_rows)


# ============================================================
# AUTHORITATIVE FILE SELECTION
# ============================================================

selection_rows = []

selection_contract = SOURCE_ROLE_CONTRACT.loc[
    SOURCE_ROLE_CONTRACT["selection_authority"]
]

for contract_row in selection_contract.itertuples(index=False):
    role = contract_row.source_role

    candidates = RAW_FILE_CLASSIFICATION_REFINED.loc[
        RAW_FILE_CLASSIFICATION_REFINED["source_role"].eq(role)
        & RAW_FILE_CLASSIFICATION_REFINED["run_relation"].eq("DECLARED_RUN")
        & RAW_FILE_CLASSIFICATION_REFINED["eligible_for_source_review"]
    ].copy()

    if len(candidates) != 1:
        continue

    selected = candidates.iloc[0]

    selection_rows.append(
        {
            "source_role": role,
            "filename": selected["filename"],
            "relative_path": selected["relative_path"],
            "resolved_path": selected["resolved_path"],
            "size_bytes": int(selected["size_bytes"]),
            "modified_utc": selected["modified_utc"],
            "detected_run_prefix": selected["detected_run_prefix"],
            "selection_reason": (
                "unique eligible file matching declared source run"
            ),
            "authority_class": "PRIMARY_RAW",
        }
    )


AUTHORITATIVE_SOURCE_SELECTION = pd.DataFrame(selection_rows)

if not AUTHORITATIVE_SOURCE_SELECTION.empty:
    AUTHORITATIVE_SOURCE_SELECTION = (
        AUTHORITATIVE_SOURCE_SELECTION
        .sort_values("source_role", kind="stable")
        .reset_index(drop=True)
    )


selected_paths = set(
    AUTHORITATIVE_SOURCE_SELECTION.get(
        "resolved_path",
        pd.Series(dtype="object"),
    ).tolist()
)

RAW_FILE_CLASSIFICATION_REFINED["selected_as_primary"] = (
    RAW_FILE_CLASSIFICATION_REFINED["resolved_path"].isin(selected_paths)
)


def exclusion_reason(row: pd.Series) -> str:
    if bool(row["selected_as_primary"]):
        return "SELECTED_PRIMARY"

    if row["run_relation"] == "FOREIGN_RUN":
        return "FOREIGN_RUN"

    if int(row["size_bytes"]) == 0:
        return "ZERO_BYTE_FILE"

    if row["source_role"] in {
        "CHECKSUM_MANIFEST",
        "SOURCE_DOCUMENTATION",
        "COLLECTOR_LOG",
        "ERROR_LOG",
    }:
        return "SUPPORTING_FILE_NOT_PRIMARY"

    if row["source_role"] == "UNCLASSIFIED":
        return "UNCLASSIFIED_REQUIRES_REVIEW"

    return "NOT_SELECTED"


RAW_FILE_CLASSIFICATION_REFINED["selection_status"] = (
    RAW_FILE_CLASSIFICATION_REFINED.apply(
        exclusion_reason,
        axis=1,
    )
)

SOURCE_SELECTION_LEDGER = (
    RAW_FILE_CLASSIFICATION_REFINED[
        [
            "source_role",
            "filename",
            "relative_path",
            "run_relation",
            "size_bytes",
            "selected_as_primary",
            "selection_status",
            "resolved_path",
        ]
    ]
    .sort_values(
        ["selected_as_primary", "source_role", "relative_path"],
        ascending=[False, True, True],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ============================================================
# VALIDATION
# ============================================================

required_role_failures = SOURCE_ROLE_AUDIT.loc[
    SOURCE_ROLE_AUDIT["required"]
    & SOURCE_ROLE_AUDIT["status"].str.startswith("FAIL")
]

required_roles = set(
    SOURCE_ROLE_CONTRACT.loc[
        SOURCE_ROLE_CONTRACT["required"],
        "source_role",
    ]
)

selected_required_roles = set(
    AUTHORITATIVE_SOURCE_SELECTION.get(
        "source_role",
        pd.Series(dtype="object"),
    )
)

missing_authoritative_roles = sorted(
    required_roles - selected_required_roles
)


display(SOURCE_ROLE_AUDIT)
display(AUTHORITATIVE_SOURCE_SELECTION)
display(SOURCE_SELECTION_LEDGER)


if not required_role_failures.empty:
    failure_text = ", ".join(
        f"{row.source_role}={row.status}"
        for row in required_role_failures.itertuples(index=False)
    )

    raise RuntimeError(
        "Required source-role audit failed: "
        + failure_text
    )

if missing_authoritative_roles:
    raise RuntimeError(
        "Authoritative source selection is incomplete: "
        + ", ".join(missing_authoritative_roles)
    )


print("Source role audit: PASS")
print(
    "Authoritative primary files selected: "
    f"{len(AUTHORITATIVE_SOURCE_SELECTION):,}"
)
print(
    "Foreign-run files excluded: "
    f"{int(RAW_FILE_CLASSIFICATION_REFINED['run_relation'].eq('FOREIGN_RUN').sum()):,}"
)
print(
    "Supporting files retained for later review: "
    f"{int(RAW_FILE_CLASSIFICATION_REFINED['selection_status'].eq('SUPPORTING_FILE_NOT_PRIMARY').sum()):,}"
)

,source_role,required,expected_count,selection_authority,declared_run_candidates,untagged_candidates,foreign_run_files,status
0,TRADE_STREAM,True,1.0,True,1,0,1,PASS_REQUIRED
1,DEPTH_STREAM,True,1.0,True,1,0,1,PASS_REQUIRED
2,REST_SNAPSHOT,True,1.0,True,1,0,1,PASS_REQUIRED
3,RUN_MANIFEST,True,1.0,True,1,0,0,PASS_REQUIRED
4,COLLECTOR_METADATA,True,1.0,True,1,0,2,PASS_REQUIRED
5,CHECKSUM_MANIFEST,False,1.0,False,1,0,1,PASS_OPTIONAL_PRESENT
6,COLLECTOR_LOG,False,NaN,False,0,0,0,NOT_PRESENT_OPTIONAL
7,ERROR_LOG,False,NaN,False,0,0,0,NOT_PRESENT_OPTIONAL
8,SOURCE_DOCUMENTATION,False,NaN,False,0,1,0,PASS_OPTIONAL_PRESENT
9,UNCLASSIFIED,False,NaN,False,0,0,0,NOT_PRESENT_OPTIONAL


,source_role,filename,relative_path,resolved_path,size_bytes,modified_utc,detected_run_prefix,selection_reason,authority_class
0,COLLECTOR_METADATA,BTCUSDT_spot_20260710T063746Z_c8b5bf12_session...,metadata\BTCUSDT_spot_20260710T063746Z_c8b5bf1...,D:\Clown Project\V0.0\data\raw\metadata\BTCUSD...,1777,2026-07-10T07:37:52+00:00,BTCUSDT_spot_20260710T063746Z_c8b5bf12,unique eligible file matching declared source run,PRIMARY_RAW
1,DEPTH_STREAM,BTCUSDT_spot_20260710T063746Z_c8b5bf12_depth_u...,order_book\BTCUSDT_spot_20260710T063746Z_c8b5b...,D:\Clown Project\V0.0\data\raw\order_book\BTCU...,63443258,2026-07-10T07:37:46+00:00,BTCUSDT_spot_20260710T063746Z_c8b5bf12,unique eligible file matching declared source run,PRIMARY_RAW
2,REST_SNAPSHOT,BTCUSDT_spot_20260710T063746Z_c8b5bf12_snapsho...,order_book\BTCUSDT_spot_20260710T063746Z_c8b5b...,D:\Clown Project\V0.0\data\raw\order_book\BTCU...,360812,2026-07-10T06:37:49+00:00,BTCUSDT_spot_20260710T063746Z_c8b5bf12,unique eligible file matching declared source run,PRIMARY_RAW
3,RUN_MANIFEST,BTCUSDT_spot_20260710T063746Z_c8b5bf12_develop...,metadata\BTCUSDT_spot_20260710T063746Z_c8b5bf1...,D:\Clown Project\V0.0\data\raw\metadata\BTCUSD...,1738,2026-07-10T07:37:52+00:00,BTCUSDT_spot_20260710T063746Z_c8b5bf12,unique eligible file matching declared source run,PRIMARY_RAW
4,TRADE_STREAM,BTCUSDT_spot_20260710T063746Z_c8b5bf12_trades....,trades\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,D:\Clown Project\V0.0\data\raw\trades\BTCUSDT_...,46082102,2026-07-10T07:37:46+00:00,BTCUSDT_spot_20260710T063746Z_c8b5bf12,unique eligible file matching declared source run,PRIMARY_RAW


,source_role,filename,relative_path,run_relation,size_bytes,selected_as_primary,selection_status,resolved_path
0,COLLECTOR_METADATA,BTCUSDT_spot_20260710T063746Z_c8b5bf12_session...,metadata\BTCUSDT_spot_20260710T063746Z_c8b5bf1...,DECLARED_RUN,1777,True,SELECTED_PRIMARY,D:\Clown Project\V0.0\data\raw\metadata\BTCUSD...
1,DEPTH_STREAM,BTCUSDT_spot_20260710T063746Z_c8b5bf12_depth_u...,order_book\BTCUSDT_spot_20260710T063746Z_c8b5b...,DECLARED_RUN,63443258,True,SELECTED_PRIMARY,D:\Clown Project\V0.0\data\raw\order_book\BTCU...
2,REST_SNAPSHOT,BTCUSDT_spot_20260710T063746Z_c8b5bf12_snapsho...,order_book\BTCUSDT_spot_20260710T063746Z_c8b5b...,DECLARED_RUN,360812,True,SELECTED_PRIMARY,D:\Clown Project\V0.0\data\raw\order_book\BTCU...
3,RUN_MANIFEST,BTCUSDT_spot_20260710T063746Z_c8b5bf12_develop...,metadata\BTCUSDT_spot_20260710T063746Z_c8b5bf1...,DECLARED_RUN,1738,True,SELECTED_PRIMARY,D:\Clown Project\V0.0\data\raw\metadata\BTCUSD...
4,TRADE_STREAM,BTCUSDT_spot_20260710T063746Z_c8b5bf12_trades....,trades\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,DECLARED_RUN,46082102,True,SELECTED_PRIMARY,D:\Clown Project\V0.0\data\raw\trades\BTCUSDT_...
5,CHECKSUM_MANIFEST,BTCUSDT_spot_20260710T060324Z_aa755c97_sha256.csv,checksums\BTCUSDT_spot_20260710T060324Z_aa755c...,FOREIGN_RUN,746,False,FOREIGN_RUN,D:\Clown Project\V0.0\data\raw\checksums\BTCUS...
6,CHECKSUM_MANIFEST,BTCUSDT_spot_20260710T063746Z_c8b5bf12_develop...,checksums\BTCUSDT_spot_20260710T063746Z_c8b5bf...,DECLARED_RUN,945,False,SUPPORTING_FILE_NOT_PRIMARY,D:\Clown Project\V0.0\data\raw\checksums\BTCUS...
7,COLLECTOR_METADATA,BTCUSDT_spot_20260710T060324Z_aa755c97_clock_o...,metadata\BTCUSDT_spot_20260710T060324Z_aa755c9...,FOREIGN_RUN,657,False,FOREIGN_RUN,D:\Clown Project\V0.0\data\raw\metadata\BTCUSD...
8,COLLECTOR_METADATA,BTCUSDT_spot_20260710T060324Z_aa755c97_session...,metadata\BTCUSDT_spot_20260710T060324Z_aa755c9...,FOREIGN_RUN,1771,False,FOREIGN_RUN,D:\Clown Project\V0.0\data\raw\metadata\BTCUSD...
9,DEPTH_STREAM,BTCUSDT_spot_20260710T060324Z_aa755c97_depth_u...,order_book\BTCUSDT_spot_20260710T060324Z_aa755...,FOREIGN_RUN,5238188,False,FOREIGN_RUN,D:\Clown Project\V0.0\data\raw\order_book\BTCU...


Source role audit: PASS
Authoritative primary files selected: 5
Foreign-run files excluded: 6
Supporting files retained for later review: 2


In [7]:
# ============================================================
# FILE HASHING, RAW-SOURCE MANIFEST, AND CHECKSUM VERIFICATION
# ============================================================

import csv


SHA256_PATTERN = re.compile(r"^[a-fA-F0-9]{64}$")
HASH_CHUNK_SIZE = 8 * 1024 * 1024


def inspect_file_bytes(
    path: Path,
    chunk_size: int = HASH_CHUNK_SIZE,
) -> dict:
    """
    Compute SHA-256 and basic byte-level diagnostics in one pass.

    The line count is diagnostic only. It is not treated as an
    authoritative market-record count.
    """
    digest = hashlib.sha256()
    byte_count = 0
    newline_count = 0
    final_byte = b""

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)
            byte_count += len(chunk)
            newline_count += chunk.count(b"\n")
            final_byte = chunk[-1:]

    physical_line_count = newline_count

    if byte_count > 0 and final_byte != b"\n":
        physical_line_count += 1

    return {
        "sha256": digest.hexdigest(),
        "bytes_read": byte_count,
        "physical_line_count": physical_line_count,
    }


# ============================================================
# HASH EVERY DISCOVERED RAW FILE
# ============================================================

hash_rows = []

for file_number, row in enumerate(
    RAW_FILE_CLASSIFICATION_REFINED.itertuples(index=False),
    start=1,
):
    path = Path(row.resolved_path)

    print(
        f"Hashing {file_number:02d}/"
        f"{len(RAW_FILE_CLASSIFICATION_REFINED):02d}: "
        f"{row.filename}"
    )

    try:
        inspection = inspect_file_bytes(path)

        hash_rows.append(
            {
                "resolved_path": str(path),
                "sha256": inspection["sha256"],
                "bytes_read": inspection["bytes_read"],
                "physical_line_count": inspection[
                    "physical_line_count"
                ],
                "hash_status": "PASS",
                "hash_error": None,
            }
        )

    except OSError as exc:
        hash_rows.append(
            {
                "resolved_path": str(path),
                "sha256": None,
                "bytes_read": None,
                "physical_line_count": None,
                "hash_status": "FAIL",
                "hash_error": str(exc),
            }
        )


RAW_FILE_HASHES = pd.DataFrame(hash_rows)

RAW_FILE_INVENTORY = (
    RAW_FILE_CLASSIFICATION_REFINED
    .merge(
        RAW_FILE_HASHES,
        on="resolved_path",
        how="left",
        validate="one_to_one",
    )
)


# ============================================================
# DUPLICATE AND CHECKSUM-CONFLICT AUDIT
# ============================================================

valid_hash_inventory = RAW_FILE_INVENTORY.loc[
    RAW_FILE_INVENTORY["hash_status"].eq("PASS")
    & RAW_FILE_INVENTORY["sha256"].notna()
].copy()

duplicate_hash_groups = (
    valid_hash_inventory
    .groupby("sha256", dropna=False)
    .filter(lambda group: len(group) > 1)
)

if duplicate_hash_groups.empty:
    EXACT_DUPLICATES = pd.DataFrame(
        columns=[
            "sha256",
            "file_count",
            "filenames",
            "run_relations",
            "resolved_paths",
        ]
    )
else:
    EXACT_DUPLICATES = (
        duplicate_hash_groups
        .groupby("sha256", as_index=False)
        .agg(
            file_count=("resolved_path", "size"),
            filenames=(
                "filename",
                lambda values: " | ".join(sorted(values)),
            ),
            run_relations=(
                "run_relation",
                lambda values: " | ".join(
                    sorted(set(values))
                ),
            ),
            resolved_paths=(
                "resolved_path",
                lambda values: " | ".join(sorted(values)),
            ),
        )
    )


same_name_different_hash = (
    valid_hash_inventory
    .groupby("filename", dropna=False)
    .filter(lambda group: group["sha256"].nunique() > 1)
)

if same_name_different_hash.empty:
    FILENAME_HASH_CONFLICTS = pd.DataFrame(
        columns=[
            "filename",
            "distinct_hashes",
            "run_relations",
            "resolved_paths",
        ]
    )
else:
    FILENAME_HASH_CONFLICTS = (
        same_name_different_hash
        .groupby("filename", as_index=False)
        .agg(
            distinct_hashes=("sha256", "nunique"),
            run_relations=(
                "run_relation",
                lambda values: " | ".join(
                    sorted(set(values))
                ),
            ),
            resolved_paths=(
                "resolved_path",
                lambda values: " | ".join(sorted(values)),
            ),
        )
    )


# ============================================================
# BUILD THE AUTHORITATIVE RAW-SOURCE MANIFEST
# ============================================================

AUTHORITATIVE_RAW_SOURCE_MANIFEST = (
    AUTHORITATIVE_SOURCE_SELECTION
    .merge(
        RAW_FILE_INVENTORY[
            [
                "resolved_path",
                "sha256",
                "bytes_read",
                "physical_line_count",
                "hash_status",
                "hash_error",
            ]
        ],
        on="resolved_path",
        how="left",
        validate="one_to_one",
    )
    .sort_values("source_role", kind="stable")
    .reset_index(drop=True)
)

AUTHORITATIVE_RAW_SOURCE_MANIFEST[
    "source_run_prefix"
] = SOURCE_RUN_PREFIX

AUTHORITATIVE_RAW_SOURCE_MANIFEST[
    "manifest_schema_version"
] = CONTRACT_SCHEMA_VERSION

AUTHORITATIVE_RAW_SOURCE_MANIFEST[
    "producing_notebook"
] = NOTEBOOK_FILENAME

AUTHORITATIVE_RAW_SOURCE_MANIFEST[
    "acceptance_status"
] = np.where(
    AUTHORITATIVE_RAW_SOURCE_MANIFEST["hash_status"].eq("PASS"),
    "REGISTERED",
    "FAIL",
)


# ============================================================
# READ THE DECLARED-RUN CHECKSUM MANIFEST, WHEN PRESENT
# ============================================================

declared_checksum_files = RAW_FILE_INVENTORY.loc[
    RAW_FILE_INVENTORY["source_role"].eq("CHECKSUM_MANIFEST")
    & RAW_FILE_INVENTORY["run_relation"].eq("DECLARED_RUN")
    & RAW_FILE_INVENTORY["eligible_for_source_review"]
].copy()


def extract_checksum_entries(path: Path) -> pd.DataFrame:
    """
    Extract filename/path and SHA-256 pairs from a loosely structured
    CSV checksum manifest.
    """
    extracted_rows = []

    with path.open(
        "r",
        encoding="utf-8-sig",
        errors="replace",
        newline="",
    ) as handle:
        reader = csv.reader(handle)

        for row_number, cells in enumerate(reader, start=1):
            cleaned_cells = [
                str(cell).strip()
                for cell in cells
                if str(cell).strip()
            ]

            detected_hashes = [
                cell.lower()
                for cell in cleaned_cells
                if SHA256_PATTERN.fullmatch(cell)
            ]

            if not detected_hashes:
                continue

            non_hash_cells = [
                cell
                for cell in cleaned_cells
                if not SHA256_PATTERN.fullmatch(cell)
            ]

            declared_path = (
                max(non_hash_cells, key=len)
                if non_hash_cells
                else None
            )

            for declared_hash in detected_hashes:
                extracted_rows.append(
                    {
                        "checksum_manifest_path": str(path),
                        "manifest_row_number": row_number,
                        "declared_path": declared_path,
                        "declared_sha256": declared_hash,
                    }
                )

    return pd.DataFrame(extracted_rows)


if len(declared_checksum_files) == 1:
    checksum_manifest_path = Path(
        declared_checksum_files.iloc[0]["resolved_path"]
    )

    DECLARED_CHECKSUM_ENTRIES = extract_checksum_entries(
        checksum_manifest_path
    )

elif len(declared_checksum_files) == 0:
    checksum_manifest_path = None

    DECLARED_CHECKSUM_ENTRIES = pd.DataFrame(
        columns=[
            "checksum_manifest_path",
            "manifest_row_number",
            "declared_path",
            "declared_sha256",
        ]
    )

else:
    raise RuntimeError(
        "More than one declared-run checksum manifest was found."
    )


# ============================================================
# VERIFY SELECTED PRIMARY FILES AGAINST DECLARED CHECKSUMS
# ============================================================

def normalize_manifest_path(value: object) -> str:
    if value is None or pd.isna(value):
        return ""

    return (
        str(value)
        .strip()
        .replace("/", "\\")
        .lower()
    )


checksum_verification_rows = []

for source_row in AUTHORITATIVE_RAW_SOURCE_MANIFEST.itertuples(
    index=False
):
    filename_normalized = normalize_manifest_path(
        source_row.filename
    )

    relative_path_normalized = normalize_manifest_path(
        source_row.relative_path
    )

    if DECLARED_CHECKSUM_ENTRIES.empty:
        matching_entries = DECLARED_CHECKSUM_ENTRIES.copy()
    else:
        normalized_declared_paths = (
            DECLARED_CHECKSUM_ENTRIES["declared_path"]
            .map(normalize_manifest_path)
        )

        matching_entries = DECLARED_CHECKSUM_ENTRIES.loc[
            normalized_declared_paths.str.endswith(
                filename_normalized,
                na=False,
            )
            |
            normalized_declared_paths.str.endswith(
                relative_path_normalized,
                na=False,
            )
        ].copy()

    declared_hashes = sorted(
        set(
            matching_entries.get(
                "declared_sha256",
                pd.Series(dtype="object"),
            )
            .dropna()
            .astype(str)
            .str.lower()
            .tolist()
        )
    )

    if checksum_manifest_path is None:
        verification_status = "NOT_AVAILABLE"
        declared_sha256 = None

    elif len(declared_hashes) == 0:
        verification_status = "NOT_LISTED"
        declared_sha256 = None

    elif len(declared_hashes) > 1:
        verification_status = "FAIL_AMBIGUOUS_DECLARED_HASH"
        declared_sha256 = " | ".join(declared_hashes)

    else:
        declared_sha256 = declared_hashes[0]

        verification_status = (
            "PASS"
            if declared_sha256 == source_row.sha256
            else "FAIL_HASH_MISMATCH"
        )

    checksum_verification_rows.append(
        {
            "source_role": source_row.source_role,
            "filename": source_row.filename,
            "computed_sha256": source_row.sha256,
            "declared_sha256": declared_sha256,
            "matching_manifest_rows": int(
                len(matching_entries)
            ),
            "verification_status": verification_status,
        }
    )


CHECKSUM_VERIFICATION = pd.DataFrame(
    checksum_verification_rows
)


# ============================================================
# DISPLAY AND CRITICAL VALIDATION
# ============================================================

display(
    AUTHORITATIVE_RAW_SOURCE_MANIFEST[
        [
            "source_role",
            "filename",
            "size_bytes",
            "physical_line_count",
            "sha256",
            "hash_status",
            "acceptance_status",
        ]
    ]
)

display(CHECKSUM_VERIFICATION)

if EXACT_DUPLICATES.empty:
    print("Exact duplicate files: none detected")
else:
    display(EXACT_DUPLICATES)

if FILENAME_HASH_CONFLICTS.empty:
    print("Same-name checksum conflicts: none detected")
else:
    display(FILENAME_HASH_CONFLICTS)


hash_failures = AUTHORITATIVE_RAW_SOURCE_MANIFEST.loc[
    ~AUTHORITATIVE_RAW_SOURCE_MANIFEST["hash_status"].eq("PASS")
]

checksum_failures = CHECKSUM_VERIFICATION.loc[
    CHECKSUM_VERIFICATION["verification_status"].str.startswith(
        "FAIL",
        na=False,
    )
]


if not hash_failures.empty:
    raise RuntimeError(
        "One or more authoritative source files could not be hashed: "
        + ", ".join(hash_failures["source_role"].tolist())
    )

if not checksum_failures.empty:
    failure_text = ", ".join(
        f"{row.source_role}={row.verification_status}"
        for row in checksum_failures.itertuples(index=False)
    )

    raise RuntimeError(
        "Declared checksum verification failed: "
        + failure_text
    )


print("Authoritative raw-source manifest: PASS")
print(
    "Authoritative files hashed: "
    f"{len(AUTHORITATIVE_RAW_SOURCE_MANIFEST):,}"
)
print(
    "Total authoritative bytes: "
    f"{int(AUTHORITATIVE_RAW_SOURCE_MANIFEST['size_bytes'].sum()):,}"
)
print(
    "Declared checksum manifest: "
    + (
        str(checksum_manifest_path)
        if checksum_manifest_path is not None
        else "not available"
    )
)

Hashing 01/13: BTCUSDT_spot_20260710T060324Z_aa755c97_trades.jsonl
Hashing 02/13: BTCUSDT_spot_20260710T063746Z_c8b5bf12_trades.jsonl
Hashing 03/13: BTCUSDT_spot_20260710T060324Z_aa755c97_depth_updates.jsonl
Hashing 04/13: BTCUSDT_spot_20260710T063746Z_c8b5bf12_depth_updates.jsonl
Hashing 05/13: BTCUSDT_spot_20260710T060324Z_aa755c97_snapshot.json
Hashing 06/13: BTCUSDT_spot_20260710T063746Z_c8b5bf12_snapshot.json
Hashing 07/13: BTCUSDT_spot_20260710T063746Z_c8b5bf12_development_manifest.json
Hashing 08/13: BTCUSDT_spot_20260710T060324Z_aa755c97_clock_offset_contract.json
Hashing 09/13: BTCUSDT_spot_20260710T060324Z_aa755c97_session.json
Hashing 10/13: BTCUSDT_spot_20260710T063746Z_c8b5bf12_session.json
Hashing 11/13: README_V0_0_DATA_STAGING.md
Hashing 12/13: BTCUSDT_spot_20260710T060324Z_aa755c97_sha256.csv
Hashing 13/13: BTCUSDT_spot_20260710T063746Z_c8b5bf12_development_sha256.csv


,source_role,filename,size_bytes,physical_line_count,sha256,hash_status,acceptance_status
0,COLLECTOR_METADATA,BTCUSDT_spot_20260710T063746Z_c8b5bf12_session...,1777,37,f7ea3210033b7e6d02d775fdc496422204194d7fd8a64a...,PASS,REGISTERED
1,DEPTH_STREAM,BTCUSDT_spot_20260710T063746Z_c8b5bf12_depth_u...,63443258,35994,608aed2608120c92fe4eaf485c43a4b26cf75d1b212041...,PASS,REGISTERED
2,REST_SNAPSHOT,BTCUSDT_spot_20260710T063746Z_c8b5bf12_snapsho...,360812,18,7921a051c2483ed04e4da82c9a1f58d847eab34b4de5a9...,PASS,REGISTERED
3,RUN_MANIFEST,BTCUSDT_spot_20260710T063746Z_c8b5bf12_develop...,1738,37,f8a3a278d6cd8b674f47d6ec1d5f4f434cb6b0d1fc05ec...,PASS,REGISTERED
4,TRADE_STREAM,BTCUSDT_spot_20260710T063746Z_c8b5bf12_trades....,46082102,67683,fcffc0cbd88badc5e2d938545ca864f09871e70e6f1f2c...,PASS,REGISTERED


,source_role,filename,computed_sha256,declared_sha256,matching_manifest_rows,verification_status
0,COLLECTOR_METADATA,BTCUSDT_spot_20260710T063746Z_c8b5bf12_session...,f7ea3210033b7e6d02d775fdc496422204194d7fd8a64a...,f7ea3210033b7e6d02d775fdc496422204194d7fd8a64a...,1,PASS
1,DEPTH_STREAM,BTCUSDT_spot_20260710T063746Z_c8b5bf12_depth_u...,608aed2608120c92fe4eaf485c43a4b26cf75d1b212041...,608aed2608120c92fe4eaf485c43a4b26cf75d1b212041...,1,PASS
2,REST_SNAPSHOT,BTCUSDT_spot_20260710T063746Z_c8b5bf12_snapsho...,7921a051c2483ed04e4da82c9a1f58d847eab34b4de5a9...,7921a051c2483ed04e4da82c9a1f58d847eab34b4de5a9...,1,PASS
3,RUN_MANIFEST,BTCUSDT_spot_20260710T063746Z_c8b5bf12_develop...,f8a3a278d6cd8b674f47d6ec1d5f4f434cb6b0d1fc05ec...,f8a3a278d6cd8b674f47d6ec1d5f4f434cb6b0d1fc05ec...,1,PASS
4,TRADE_STREAM,BTCUSDT_spot_20260710T063746Z_c8b5bf12_trades....,fcffc0cbd88badc5e2d938545ca864f09871e70e6f1f2c...,fcffc0cbd88badc5e2d938545ca864f09871e70e6f1f2c...,1,PASS


Exact duplicate files: none detected
Same-name checksum conflicts: none detected
Authoritative raw-source manifest: PASS
Authoritative files hashed: 5
Total authoritative bytes: 109,889,687
Declared checksum manifest: D:\Clown Project\V0.0\data\raw\checksums\BTCUSDT_spot_20260710T063746Z_c8b5bf12_development_sha256.csv


In [8]:
# ============================================================
# V0.0 REFERENCE ARTIFACT INVENTORY
# ============================================================

REFERENCE_ROOTS = [
    {
        "root_name": "processed",
        "root_path": V0_0_PROCESSED_ROOT,
    },
    {
        "root_name": "artifacts_v0_0",
        "root_path": V0_0_ARTIFACT_ROOT,
    },
    {
        "root_name": "notebooks",
        "root_path": V0_0_ROOT / "notebooks",
    },
]


def classify_reference_artifact(path: Path) -> str:
    """
    Classify a V0.0 processed or artifact file by path and filename.

    These classifications are descriptive only. Every discovered file
    remains REFERENCE_ONLY and cannot replace V0.1 reconstruction.
    """
    text = str(path).lower().replace("/", "\\")
    name = path.name.lower()

    if path.suffix.lower() == ".ipynb":
        return "DEVELOPMENT_NOTEBOOK"

    if any(token in text for token in ("\\book\\", "reconstructed_book")):
        return "BOOK_RECONSTRUCTION"

    if any(
        token in text
        for token in (
            "\\trades\\",
            "canonical_trade",
            "trade_book",
            "aligned_trade",
            "synchronized_trade",
        )
    ):
        return "TRADE_OR_ALIGNMENT_REFERENCE"

    if any(
        token in text
        for token in (
            "\\events\\",
            "event_stream",
            "burst_event",
            "market_event",
        )
    ):
        return "EVENT_STREAM_REFERENCE"

    if any(
        token in text
        for token in (
            "\\point_process\\",
            "point_process",
            "poisson",
            "fano",
            "interarrival",
        )
    ):
        return "POINT_PROCESS_REFERENCE"

    if any(
        token in text
        for token in (
            "\\hawkes\\",
            "hawkes",
            "kernel",
            "branching",
            "compensator",
            "intensity",
        )
    ):
        return "HAWKES_REFERENCE"

    if any(
        token in text
        for token in (
            "state_dependent",
            "state-dependent",
            "sdhawkes",
        )
    ):
        return "STATE_DEPENDENT_HAWKES_REFERENCE"

    if any(
        token in text
        for token in (
            "quote",
            "quoting",
            "shadow",
            "execution",
            "fill",
            "latency",
            "market_making",
        )
    ):
        return "POST_V0_1_PROVENANCE"

    if any(
        token in text
        for token in (
            "split",
            "holdout",
            "calibration_period",
            "validation_period",
        )
    ):
        return "SPLIT_METADATA"

    if any(
        token in text
        for token in (
            "contract",
            "schema",
            "specification",
        )
    ):
        return "CONTRACT_OR_SCHEMA"

    if any(
        token in text
        for token in (
            "manifest",
            "registry",
            "inventory",
            "checksum",
            "sha256",
        )
    ):
        return "MANIFEST_OR_PROVENANCE"

    if any(
        token in text
        for token in (
            "gate",
            "audit",
            "validation_report",
            "acceptance",
            "status",
        )
    ):
        return "AUDIT_OR_GATE_REPORT"

    if path.suffix.lower() in {".png", ".jpg", ".jpeg", ".svg", ".pdf"}:
        return "FIGURE_OR_REPORT"

    if path.suffix.lower() in {".json", ".yaml", ".yml", ".toml"}:
        return "STRUCTURED_METADATA"

    if path.suffix.lower() in {".csv", ".parquet", ".feather", ".npz", ".npy"}:
        return "DATA_OR_MODEL_ARTIFACT"

    return "UNCLASSIFIED_REFERENCE"


# ============================================================
# DISCOVER REFERENCE FILES
# ============================================================

reference_file_rows = []

for root_record in REFERENCE_ROOTS:
    root_name = root_record["root_name"]
    root_path = resolve_path(root_record["root_path"])

    if not root_path.exists():
        continue

    for path in sorted(
        (item for item in root_path.rglob("*") if item.is_file()),
        key=lambda item: str(item).lower(),
    ):
        resolved = resolve_path(path)
        stat = resolved.stat()

        reference_file_rows.append(
            {
                "reference_root": root_name,
                "filename": resolved.name,
                "suffix": resolved.suffix.lower(),
                "relative_path_from_reference_root": str(
                    resolved.relative_to(root_path)
                ),
                "relative_path_from_v0_0": str(
                    resolved.relative_to(V0_0_ROOT_RESOLVED)
                ),
                "resolved_path": str(resolved),
                "artifact_class": classify_reference_artifact(resolved),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    tz=timezone.utc,
                ).replace(microsecond=0).isoformat(),
                "detected_run_prefix": detect_run_prefix(resolved),
                "authority_class": "V0_0_REFERENCE",
                "allowed_use": "READ_AND_RECONCILE_ONLY",
            }
        )


V0_0_REFERENCE_DISCOVERY = pd.DataFrame(reference_file_rows)

if V0_0_REFERENCE_DISCOVERY.empty:
    V0_0_REFERENCE_DISCOVERY = pd.DataFrame(
        columns=[
            "reference_root",
            "filename",
            "suffix",
            "relative_path_from_reference_root",
            "relative_path_from_v0_0",
            "resolved_path",
            "artifact_class",
            "size_bytes",
            "modified_utc",
            "detected_run_prefix",
            "authority_class",
            "allowed_use",
        ]
    )


# Remove duplicate paths if the same file was reachable through more
# than one registered reference root.
V0_0_REFERENCE_DISCOVERY = (
    V0_0_REFERENCE_DISCOVERY
    .drop_duplicates(subset="resolved_path", keep="first")
    .sort_values(
        ["artifact_class", "relative_path_from_v0_0"],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ============================================================
# HASH REFERENCE ARTIFACTS
# ============================================================

reference_hash_rows = []

for file_number, row in enumerate(
    V0_0_REFERENCE_DISCOVERY.itertuples(index=False),
    start=1,
):
    path = Path(row.resolved_path)

    print(
        f"Hashing V0.0 reference "
        f"{file_number:03d}/{len(V0_0_REFERENCE_DISCOVERY):03d}: "
        f"{row.relative_path_from_v0_0}"
    )

    try:
        inspection = inspect_file_bytes(path)

        reference_hash_rows.append(
            {
                "resolved_path": str(path),
                "sha256": inspection["sha256"],
                "bytes_read": int(inspection["bytes_read"]),
                "physical_line_count": int(
                    inspection["physical_line_count"]
                ),
                "hash_status": "PASS",
                "hash_error": None,
            }
        )

    except OSError as exc:
        reference_hash_rows.append(
            {
                "resolved_path": str(path),
                "sha256": None,
                "bytes_read": None,
                "physical_line_count": None,
                "hash_status": "FAIL",
                "hash_error": str(exc),
            }
        )


REFERENCE_FILE_HASHES = pd.DataFrame(reference_hash_rows)

if REFERENCE_FILE_HASHES.empty:
    REFERENCE_FILE_HASHES = pd.DataFrame(
        columns=[
            "resolved_path",
            "sha256",
            "bytes_read",
            "physical_line_count",
            "hash_status",
            "hash_error",
        ]
    )


V0_0_REFERENCE_ARTIFACT_MANIFEST = (
    V0_0_REFERENCE_DISCOVERY
    .merge(
        REFERENCE_FILE_HASHES,
        on="resolved_path",
        how="left",
        validate="one_to_one",
    )
)

V0_0_REFERENCE_ARTIFACT_MANIFEST[
    "source_run_prefix"
] = SOURCE_RUN_PREFIX

V0_0_REFERENCE_ARTIFACT_MANIFEST[
    "manifest_schema_version"
] = CONTRACT_SCHEMA_VERSION

V0_0_REFERENCE_ARTIFACT_MANIFEST[
    "producing_notebook"
] = NOTEBOOK_FILENAME

V0_0_REFERENCE_ARTIFACT_MANIFEST[
    "acceptance_status"
] = np.select(
    [
        V0_0_REFERENCE_ARTIFACT_MANIFEST["hash_status"].eq("FAIL"),
        V0_0_REFERENCE_ARTIFACT_MANIFEST[
            "detected_run_prefix"
        ].notna()
        & ~V0_0_REFERENCE_ARTIFACT_MANIFEST[
            "detected_run_prefix"
        ].str.lower().eq(SOURCE_RUN_PREFIX.lower()),
    ],
    [
        "HASH_FAILURE",
        "FOREIGN_OR_DIFFERENT_RUN_REFERENCE",
    ],
    default="REGISTERED_REFERENCE_ONLY",
)


# ============================================================
# REFERENCE INVENTORY SUMMARY
# ============================================================

if V0_0_REFERENCE_ARTIFACT_MANIFEST.empty:
    V0_0_REFERENCE_SUMMARY = pd.DataFrame(
        columns=[
            "artifact_class",
            "file_count",
            "total_bytes",
            "hash_failures",
        ]
    )
else:
    V0_0_REFERENCE_SUMMARY = (
        V0_0_REFERENCE_ARTIFACT_MANIFEST
        .groupby("artifact_class", as_index=False)
        .agg(
            file_count=("resolved_path", "size"),
            total_bytes=("size_bytes", "sum"),
            hash_failures=(
                "hash_status",
                lambda values: int((values != "PASS").sum()),
            ),
        )
        .sort_values("artifact_class", kind="stable")
        .reset_index(drop=True)
    )


reference_hash_failures = (
    V0_0_REFERENCE_ARTIFACT_MANIFEST.loc[
        V0_0_REFERENCE_ARTIFACT_MANIFEST[
            "hash_status"
        ].eq("FAIL")
    ]
)

unclassified_reference_files = (
    V0_0_REFERENCE_ARTIFACT_MANIFEST.loc[
        V0_0_REFERENCE_ARTIFACT_MANIFEST[
            "artifact_class"
        ].eq("UNCLASSIFIED_REFERENCE")
    ]
)


display(V0_0_REFERENCE_SUMMARY)

display(
    V0_0_REFERENCE_ARTIFACT_MANIFEST[
        [
            "artifact_class",
            "reference_root",
            "relative_path_from_v0_0",
            "size_bytes",
            "detected_run_prefix",
            "sha256",
            "acceptance_status",
        ]
    ]
)


if not reference_hash_failures.empty:
    failed_paths = reference_hash_failures[
        "relative_path_from_v0_0"
    ].tolist()

    raise RuntimeError(
        "One or more V0.0 reference artifacts could not be hashed: "
        + ", ".join(failed_paths)
    )


print("V0.0 reference artifact inventory: PASS")
print(
    "Reference artifacts registered: "
    f"{len(V0_0_REFERENCE_ARTIFACT_MANIFEST):,}"
)
print(
    "Reference artifact bytes: "
    f"{int(V0_0_REFERENCE_ARTIFACT_MANIFEST['size_bytes'].sum()):,}"
)
print(
    "Unclassified reference artifacts: "
    f"{len(unclassified_reference_files):,}"
)
print(
    "Authority restriction: all registered files are "
    "REFERENCE_ONLY and may not replace V0.1 reconstruction."
)

Hashing V0.0 reference 001/290: artifacts\v0_0\BTCUSDT_spot_20260710T060324Z_aa755c97_clock_gate_report.csv
Hashing V0.0 reference 002/290: artifacts\v0_0\BTCUSDT_spot_20260710T060324Z_aa755c97_smoke_test_gate_report.csv
Hashing V0.0 reference 003/290: artifacts\v0_0\BTCUSDT_spot_20260710T063746Z_c8b5bf12_book_reconstruction_gates.csv
Hashing V0.0 reference 004/290: artifacts\v0_0\BTCUSDT_spot_20260710T063746Z_c8b5bf12_development_audit_checks.csv
Hashing V0.0 reference 005/290: artifacts\v0_0\BTCUSDT_spot_20260710T063746Z_c8b5bf12_development_audit_decision.json
Hashing V0.0 reference 006/290: artifacts\v0_0\BTCUSDT_spot_20260710T063746Z_c8b5bf12_development_audit_gates.csv
Hashing V0.0 reference 007/290: artifacts\v0_0\BTCUSDT_spot_20260710T063746Z_c8b5bf12_development_audit_metrics.csv
Hashing V0.0 reference 008/290: artifacts\v0_0\BTCUSDT_spot_20260710T063746Z_c8b5bf12_local_receipt_time_tie_gates.csv
Hashing V0.0 reference 009/290: artifacts\v0_0\BTCUSDT_spot_20260710T063746Z_c8b5

,artifact_class,file_count,total_bytes,hash_failures
0,AUDIT_OR_GATE_REPORT,23,11664600,0
1,CONTRACT_OR_SCHEMA,10,19071,0
2,DATA_OR_MODEL_ARTIFACT,65,10447323,0
3,EVENT_STREAM_REFERENCE,1,1875822,0
4,HAWKES_REFERENCE,111,57666410,0
5,MANIFEST_OR_PROVENANCE,2,3948,0
6,POINT_PROCESS_REFERENCE,16,9679359,0
7,POST_V0_1_PROVENANCE,48,10106246,0
8,STATE_DEPENDENT_HAWKES_REFERENCE,7,15604,0
9,STRUCTURED_METADATA,1,504,0


,artifact_class,reference_root,relative_path_from_v0_0,size_bytes,detected_run_prefix,sha256,acceptance_status
0,AUDIT_OR_GATE_REPORT,artifacts_v0_0,artifacts\v0_0\BTCUSDT_spot_20260710T060324Z_a...,184,BTCUSDT_spot_20260710T060324Z_aa755c97,05d92b54c9cb3ca252dc9192f3d3abe5826cda6662d0ef...,FOREIGN_OR_DIFFERENT_RUN_REFERENCE
1,AUDIT_OR_GATE_REPORT,artifacts_v0_0,artifacts\v0_0\BTCUSDT_spot_20260710T060324Z_a...,292,BTCUSDT_spot_20260710T060324Z_aa755c97,d165099020d3a615f2a297d5c1c9dafde72b0879bce673...,FOREIGN_OR_DIFFERENT_RUN_REFERENCE
2,AUDIT_OR_GATE_REPORT,artifacts_v0_0,artifacts\v0_0\BTCUSDT_spot_20260710T063746Z_c...,353,BTCUSDT_spot_20260710T063746Z_c8b5bf12,f3db0ba871c16b57110bb6ad97408ff8a1a24a549c2413...,REGISTERED_REFERENCE_ONLY
3,AUDIT_OR_GATE_REPORT,artifacts_v0_0,artifacts\v0_0\BTCUSDT_spot_20260710T063746Z_c...,3106,BTCUSDT_spot_20260710T063746Z_c8b5bf12,80c9270ba53e3331caf12761bfac52b295582516cee15a...,REGISTERED_REFERENCE_ONLY
4,AUDIT_OR_GATE_REPORT,artifacts_v0_0,artifacts\v0_0\BTCUSDT_spot_20260710T063746Z_c...,1296,BTCUSDT_spot_20260710T063746Z_c8b5bf12,0490e56e6d7c231cfbf8cef48378177ed77817dfaa3faf...,REGISTERED_REFERENCE_ONLY
...,...,...,...,...,...,...,...
285,TRADE_OR_ALIGNMENT_REFERENCE,artifacts_v0_0,artifacts\v0_0\BTCUSDT_spot_20260710T063746Z_c...,505,BTCUSDT_spot_20260710T063746Z_c8b5bf12,446aa55adabd0f96f25ed6fbbd090f89a51acb056577b7...,REGISTERED_REFERENCE_ONLY
286,TRADE_OR_ALIGNMENT_REFERENCE,artifacts_v0_0,artifacts\v0_0\BTCUSDT_spot_20260710T063746Z_c...,2573,BTCUSDT_spot_20260710T063746Z_c8b5bf12,646838197f3dd9ee922471b83b3c220aa83430c3a18ea1...,REGISTERED_REFERENCE_ONLY
287,TRADE_OR_ALIGNMENT_REFERENCE,artifacts_v0_0,artifacts\v0_0\BTCUSDT_spot_20260710T063746Z_c...,1648,BTCUSDT_spot_20260710T063746Z_c8b5bf12,a56af554d70ee9f8984d66d1844504f474fbc8261bda35...,REGISTERED_REFERENCE_ONLY
288,TRADE_OR_ALIGNMENT_REFERENCE,artifacts_v0_0,artifacts\v0_0\BTCUSDT_spot_20260710T063746Z_c...,280,BTCUSDT_spot_20260710T063746Z_c8b5bf12,03b45f7138f7fbed5e1ee381d3f0f44f911c9b4ff1807b...,REGISTERED_REFERENCE_ONLY


V0.0 reference artifact inventory: PASS
Reference artifacts registered: 290
Reference artifact bytes: 101,484,812
Unclassified reference artifacts: 0
Authority restriction: all registered files are REFERENCE_ONLY and may not replace V0.1 reconstruction.


In [9]:
# ============================================================
# IMMUTABLE V0.0 SOURCE IDENTITY
# ============================================================

def normalize_identity_path(value: object) -> str:
    """
    Normalize a relative Windows path for deterministic hashing.

    The identity remains independent of the machine's absolute path.
    """
    return (
        str(value)
        .strip()
        .replace("\\", "/")
        .strip("/")
        .lower()
    )


def canonical_json_bytes(value: object) -> bytes:
    """Serialize an object deterministically for cryptographic hashing."""
    return json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode("utf-8")


def sha256_of_canonical_object(value: object) -> str:
    """Return the SHA-256 digest of canonical JSON content."""
    return hashlib.sha256(
        canonical_json_bytes(value)
    ).hexdigest()


# ============================================================
# VALIDATE IDENTITY INPUTS
# ============================================================

required_identity_columns = {
    "source_role",
    "relative_path",
    "size_bytes",
    "sha256",
    "hash_status",
}

missing_identity_columns = (
    required_identity_columns
    - set(AUTHORITATIVE_RAW_SOURCE_MANIFEST.columns)
)

if missing_identity_columns:
    raise RuntimeError(
        "Authoritative source manifest is missing required identity "
        "columns: "
        + ", ".join(sorted(missing_identity_columns))
    )


identity_inputs = AUTHORITATIVE_RAW_SOURCE_MANIFEST.loc[
    AUTHORITATIVE_RAW_SOURCE_MANIFEST["hash_status"].eq("PASS")
].copy()

if identity_inputs.empty:
    raise RuntimeError(
        "No successfully hashed authoritative source files are "
        "available for source identity generation."
    )


duplicate_roles = identity_inputs.loc[
    identity_inputs["source_role"].duplicated(keep=False)
]

duplicate_paths = identity_inputs.loc[
    identity_inputs["resolved_path"].duplicated(keep=False)
]

duplicate_primary_hashes = identity_inputs.loc[
    identity_inputs["sha256"].duplicated(keep=False)
]

if not duplicate_roles.empty:
    raise RuntimeError(
        "Source identity cannot be generated because authoritative "
        "source roles are duplicated: "
        + ", ".join(
            sorted(
                duplicate_roles["source_role"]
                .drop_duplicates()
                .tolist()
            )
        )
    )

if not duplicate_paths.empty:
    raise RuntimeError(
        "Source identity cannot be generated because one path was "
        "selected more than once."
    )

if not duplicate_primary_hashes.empty:
    duplicate_hash_roles = (
        duplicate_primary_hashes
        .groupby("sha256")["source_role"]
        .apply(lambda values: " | ".join(sorted(values)))
        .tolist()
    )

    raise RuntimeError(
        "Different authoritative source roles contain identical file "
        "content: "
        + "; ".join(duplicate_hash_roles)
    )


# ============================================================
# BUILD CANONICAL SOURCE COMPONENTS
# ============================================================

SOURCE_IDENTITY_COMPONENTS = pd.DataFrame(
    {
        "source_role": identity_inputs["source_role"].astype(str),
        "relative_path_normalized": (
            identity_inputs["relative_path"]
            .map(normalize_identity_path)
        ),
        "size_bytes": (
            identity_inputs["size_bytes"]
            .astype("int64")
        ),
        "sha256": (
            identity_inputs["sha256"]
            .astype(str)
            .str.lower()
        ),
    }
)

SOURCE_IDENTITY_COMPONENTS = (
    SOURCE_IDENTITY_COMPONENTS
    .sort_values(
        ["source_role", "relative_path_normalized"],
        kind="stable",
    )
    .reset_index(drop=True)
)


source_component_records = (
    SOURCE_IDENTITY_COMPONENTS
    .to_dict(orient="records")
)

SOURCE_SET_HASH = sha256_of_canonical_object(
    source_component_records
)


# ============================================================
# REGISTER CHECKSUM-MANIFEST EVIDENCE
# ============================================================

if checksum_manifest_path is not None:
    checksum_manifest_match = RAW_FILE_INVENTORY.loc[
        RAW_FILE_INVENTORY["resolved_path"].eq(
            str(checksum_manifest_path)
        )
    ]

    if len(checksum_manifest_match) != 1:
        raise RuntimeError(
            "The declared checksum manifest could not be resolved "
            "uniquely in the raw-file inventory."
        )

    CHECKSUM_MANIFEST_SHA256 = str(
        checksum_manifest_match.iloc[0]["sha256"]
    ).lower()

    CHECKSUM_MANIFEST_RELATIVE_PATH = normalize_identity_path(
        checksum_manifest_match.iloc[0]["relative_path"]
    )

else:
    CHECKSUM_MANIFEST_SHA256 = None
    CHECKSUM_MANIFEST_RELATIVE_PATH = None


# ============================================================
# BUILD SOURCE IDENTITY MANIFEST
# ============================================================

SOURCE_IDENTITY_MANIFEST = {
    "identity_type": "V0_0_PRIMARY_SOURCE_SET",
    "identity_schema_version": CONTRACT_SCHEMA_VERSION,
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "hash_algorithm": HASH_ALGORITHM,
    "primary_file_count": int(
        len(SOURCE_IDENTITY_COMPONENTS)
    ),
    "primary_source_roles": (
        SOURCE_IDENTITY_COMPONENTS["source_role"]
        .tolist()
    ),
    "source_components": source_component_records,
    "source_set_hash": SOURCE_SET_HASH,
    "checksum_manifest_evidence": {
        "relative_path": CHECKSUM_MANIFEST_RELATIVE_PATH,
        "sha256": CHECKSUM_MANIFEST_SHA256,
        "included_in_source_set_hash": False,
        "purpose": (
            "Independent verification evidence for selected "
            "primary source files"
        ),
    },
    "absolute_paths_included_in_hash": False,
    "filesystem_timestamps_included_in_hash": False,
    "producing_notebook": NOTEBOOK_FILENAME,
}


# Hash the complete identity declaration separately from the
# underlying primary-source-set hash.
SOURCE_IDENTITY_DECLARATION_HASH = (
    sha256_of_canonical_object(
        SOURCE_IDENTITY_MANIFEST
    )
)

SOURCE_IDENTITY_MANIFEST[
    "source_identity_declaration_hash"
] = SOURCE_IDENTITY_DECLARATION_HASH


# ============================================================
# SOURCE IDENTITY SUMMARY
# ============================================================

SOURCE_IDENTITY_SUMMARY = pd.DataFrame(
    [
        {
            "field": "identity_type",
            "value": SOURCE_IDENTITY_MANIFEST[
                "identity_type"
            ],
        },
        {
            "field": "source_run_prefix",
            "value": SOURCE_RUN_PREFIX,
        },
        {
            "field": "primary_file_count",
            "value": len(SOURCE_IDENTITY_COMPONENTS),
        },
        {
            "field": "primary_source_roles",
            "value": " | ".join(
                SOURCE_IDENTITY_COMPONENTS[
                    "source_role"
                ].tolist()
            ),
        },
        {
            "field": "total_primary_bytes",
            "value": int(
                SOURCE_IDENTITY_COMPONENTS[
                    "size_bytes"
                ].sum()
            ),
        },
        {
            "field": "source_set_hash",
            "value": SOURCE_SET_HASH,
        },
        {
            "field": "source_identity_declaration_hash",
            "value": SOURCE_IDENTITY_DECLARATION_HASH,
        },
        {
            "field": "checksum_manifest_sha256",
            "value": CHECKSUM_MANIFEST_SHA256,
        },
        {
            "field": "absolute_paths_in_hash",
            "value": False,
        },
        {
            "field": "filesystem_timestamps_in_hash",
            "value": False,
        },
    ]
)


display(SOURCE_IDENTITY_COMPONENTS)
display(SOURCE_IDENTITY_SUMMARY)


# ============================================================
# FINAL IDENTITY GATES
# ============================================================

if len(SOURCE_SET_HASH) != 64:
    raise RuntimeError(
        "Source-set hash is not a valid SHA-256 digest."
    )

if len(SOURCE_IDENTITY_DECLARATION_HASH) != 64:
    raise RuntimeError(
        "Source-identity declaration hash is not a valid "
        "SHA-256 digest."
    )

if (
    CHECKSUM_MANIFEST_SHA256 is not None
    and len(CHECKSUM_MANIFEST_SHA256) != 64
):
    raise RuntimeError(
        "Checksum-manifest evidence hash is invalid."
    )


print("Immutable V0.0 source identity: PASS")
print(f"Source run prefix: {SOURCE_RUN_PREFIX}")
print(f"Primary source files: {len(SOURCE_IDENTITY_COMPONENTS):,}")
print(f"Source-set hash: {SOURCE_SET_HASH}")
print(
    "Source-identity declaration hash: "
    f"{SOURCE_IDENTITY_DECLARATION_HASH}"
)

,source_role,relative_path_normalized,size_bytes,sha256
0,COLLECTOR_METADATA,metadata/btcusdt_spot_20260710t063746z_c8b5bf1...,1777,f7ea3210033b7e6d02d775fdc496422204194d7fd8a64a...
1,DEPTH_STREAM,order_book/btcusdt_spot_20260710t063746z_c8b5b...,63443258,608aed2608120c92fe4eaf485c43a4b26cf75d1b212041...
2,REST_SNAPSHOT,order_book/btcusdt_spot_20260710t063746z_c8b5b...,360812,7921a051c2483ed04e4da82c9a1f58d847eab34b4de5a9...
3,RUN_MANIFEST,metadata/btcusdt_spot_20260710t063746z_c8b5bf1...,1738,f8a3a278d6cd8b674f47d6ec1d5f4f434cb6b0d1fc05ec...
4,TRADE_STREAM,trades/btcusdt_spot_20260710t063746z_c8b5bf12_...,46082102,fcffc0cbd88badc5e2d938545ca864f09871e70e6f1f2c...


,field,value
0,identity_type,V0_0_PRIMARY_SOURCE_SET
1,source_run_prefix,BTCUSDT_spot_20260710T063746Z_c8b5bf12
2,primary_file_count,5
3,primary_source_roles,COLLECTOR_METADATA | DEPTH_STREAM | REST_SNAPS...
4,total_primary_bytes,109889687
5,source_set_hash,132c83531eec615d279408b5c06f402973114ba3058dfa...
6,source_identity_declaration_hash,0f1c16f51c9a608252a4a45a5801d1b03da3034ab63c03...
7,checksum_manifest_sha256,aa84d80399a895c17a6a3c24d4eed392c6e30d4cbad2a0...
8,absolute_paths_in_hash,False
9,filesystem_timestamps_in_hash,False


Immutable V0.0 source identity: PASS
Source run prefix: BTCUSDT_spot_20260710T063746Z_c8b5bf12
Primary source files: 5
Source-set hash: 132c83531eec615d279408b5c06f402973114ba3058dfadd2fe58e2e67184c4b
Source-identity declaration hash: 0f1c16f51c9a608252a4a45a5801d1b03da3034ab63c03127eb78f7b89320e78


In [10]:
# ============================================================
# LIGHTWEIGHT CHRONOLOGICAL COVERAGE SCAN
# ============================================================

from collections import defaultdict


def selected_source_path(source_role: str) -> Path:
    """Return the uniquely selected authoritative file for one role."""
    matches = AUTHORITATIVE_RAW_SOURCE_MANIFEST.loc[
        AUTHORITATIVE_RAW_SOURCE_MANIFEST[
            "source_role"
        ].eq(source_role)
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one authoritative {source_role} file; "
            f"found {len(matches)}."
        )

    return Path(matches.iloc[0]["resolved_path"])


TRADE_SOURCE_PATH = selected_source_path("TRADE_STREAM")
DEPTH_SOURCE_PATH = selected_source_path("DEPTH_STREAM")
SESSION_METADATA_PATH = selected_source_path(
    "COLLECTOR_METADATA"
)
DEVELOPMENT_MANIFEST_PATH = selected_source_path(
    "RUN_MANIFEST"
)


# ============================================================
# SMALL METADATA READERS
# ============================================================

def load_json_mapping(path: Path) -> dict:
    """Load one JSON object without transforming its contents."""
    with path.open(
        "r",
        encoding="utf-8-sig",
        errors="strict",
    ) as handle:
        value = json.load(handle)

    if not isinstance(value, dict):
        raise TypeError(
            f"Expected a JSON object in {path.name}; "
            f"found {type(value).__name__}."
        )

    return value


def nested_value(
    mapping: dict,
    path: tuple[str, ...],
    default=None,
):
    """Retrieve a nested value without raising on absent keys."""
    current = mapping

    for key in path:
        if not isinstance(current, dict) or key not in current:
            return default

        current = current[key]

    return current


SESSION_METADATA_DOCUMENT = load_json_mapping(
    SESSION_METADATA_PATH
)

DEVELOPMENT_MANIFEST_DOCUMENT = load_json_mapping(
    DEVELOPMENT_MANIFEST_PATH
)


# ============================================================
# RAW FIELD EXTRACTION
# ============================================================

def first_present(
    mapping: dict,
    aliases: tuple[str, ...],
):
    """
    Return the first non-null value matching one of the supplied
    aliases. Exact matching is preferred.
    """
    for alias in aliases:
        if alias in mapping and mapping[alias] is not None:
            return mapping[alias]

    casefold_lookup = {
        str(key).casefold(): key
        for key in mapping
    }

    for alias in aliases:
        matched_key = casefold_lookup.get(alias.casefold())

        if (
            matched_key is not None
            and mapping[matched_key] is not None
        ):
            return mapping[matched_key]

    return None


def safe_integer(value) -> int | None:
    """Convert an integer-like scalar without silently truncating."""
    if value is None or isinstance(value, bool):
        return None

    try:
        numeric = int(value)
    except (TypeError, ValueError, OverflowError):
        return None

    try:
        if float(value) != float(numeric):
            return None
    except (TypeError, ValueError, OverflowError):
        pass

    return numeric


def timestamp_iso(
    value: int | None,
    unit: str,
) -> str | None:
    """Render an integer timestamp as UTC ISO-8601."""
    if value is None:
        return None

    parsed = pd.to_datetime(
        value,
        unit=unit,
        utc=True,
        errors="coerce",
    )

    if pd.isna(parsed):
        return None

    return parsed.isoformat()


TRADE_COVERAGE_FIELDS = {
    "collector_sequence": (
        "collector_sequence",
        "collector_seq",
        "local_sequence",
        "sequence",
    ),
    "local_receipt_time_ns": (
        "local_receipt_time_ns",
        "receipt_time_ns",
        "received_time_ns",
        "receive_time_ns",
        "recv_time_ns",
    ),
    "exchange_event_time_ms": (
        "exchange_event_time_raw",
        "exchange_event_time_ms",
        "event_time_ms",
        "event_time",
        "E",
    ),
    "trade_time_ms": (
        "trade_time_raw",
        "trade_time_ms",
        "trade_time",
        "transaction_time_ms",
        "transaction_time",
        "T",
    ),
    "connection_session_id": (
        "connection_session_id",
        "session_id",
        "collector_session_id",
    ),
}

DEPTH_COVERAGE_FIELDS = {
    "collector_sequence": (
        "collector_sequence",
        "collector_seq",
        "local_sequence",
        "sequence",
    ),
    "local_receipt_time_ns": (
        "local_receipt_time_ns",
        "receipt_time_ns",
        "received_time_ns",
        "receive_time_ns",
        "recv_time_ns",
    ),
    "exchange_event_time_ms": (
        "exchange_event_time_raw",
        "exchange_event_time_ms",
        "event_time_ms",
        "event_time",
        "E",
    ),
    "connection_session_id": (
        "connection_session_id",
        "session_id",
        "collector_session_id",
    ),
}


# ============================================================
# STREAM COVERAGE SCANNER
# ============================================================

def scan_jsonl_coverage(
    path: Path,
    stream_name: str,
    field_contract: dict[str, tuple[str, ...]],
) -> dict:
    """
    Scan only the fields needed to establish chronological coverage.

    This cell does not normalize records, repair timestamps, validate
    market values, or construct analytical datasets.
    """
    row_count = 0
    invalid_json_lines = 0
    non_object_lines = 0

    missing_field_counts = {
        field_name: 0
        for field_name in field_contract
    }

    numeric_ranges = {
        field_name: {
            "minimum": None,
            "maximum": None,
        }
        for field_name in (
            "collector_sequence",
            "local_receipt_time_ns",
            "exchange_event_time_ms",
            "trade_time_ms",
        )
        if field_name in field_contract
    }

    source_order_reversals = {
        field_name: 0
        for field_name in (
            "collector_sequence",
            "local_receipt_time_ns",
            "exchange_event_time_ms",
            "trade_time_ms",
        )
        if field_name in field_contract
    }

    previous_values = {
        field_name: None
        for field_name in source_order_reversals
    }

    collector_sequences: set[int] = set()
    duplicate_collector_sequences = 0

    session_statistics = defaultdict(
        lambda: {
            "row_count": 0,
            "receipt_min_ns": None,
            "receipt_max_ns": None,
            "sequence_min": None,
            "sequence_max": None,
        }
    )

    with path.open(
        "r",
        encoding="utf-8-sig",
        errors="strict",
    ) as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue

            row_count += 1

            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                invalid_json_lines += 1
                continue

            if not isinstance(record, dict):
                non_object_lines += 1
                continue

            extracted = {}

            for canonical_field, aliases in field_contract.items():
                value = first_present(record, aliases)
                extracted[canonical_field] = value

                if value is None:
                    missing_field_counts[canonical_field] += 1

            session_id_raw = extracted.get(
                "connection_session_id"
            )

            session_id = (
                str(session_id_raw)
                if session_id_raw is not None
                else "<MISSING_SESSION_ID>"
            )

            session_statistics[session_id]["row_count"] += 1

            for field_name in numeric_ranges:
                numeric_value = safe_integer(
                    extracted.get(field_name)
                )

                if numeric_value is None:
                    continue

                current_minimum = numeric_ranges[
                    field_name
                ]["minimum"]

                current_maximum = numeric_ranges[
                    field_name
                ]["maximum"]

                numeric_ranges[field_name]["minimum"] = (
                    numeric_value
                    if current_minimum is None
                    else min(current_minimum, numeric_value)
                )

                numeric_ranges[field_name]["maximum"] = (
                    numeric_value
                    if current_maximum is None
                    else max(current_maximum, numeric_value)
                )

                previous_value = previous_values[field_name]

                if (
                    previous_value is not None
                    and numeric_value < previous_value
                ):
                    source_order_reversals[field_name] += 1

                previous_values[field_name] = numeric_value

            collector_sequence = safe_integer(
                extracted.get("collector_sequence")
            )

            if collector_sequence is not None:
                if collector_sequence in collector_sequences:
                    duplicate_collector_sequences += 1

                collector_sequences.add(collector_sequence)

                session_record = session_statistics[session_id]

                session_record["sequence_min"] = (
                    collector_sequence
                    if session_record["sequence_min"] is None
                    else min(
                        session_record["sequence_min"],
                        collector_sequence,
                    )
                )

                session_record["sequence_max"] = (
                    collector_sequence
                    if session_record["sequence_max"] is None
                    else max(
                        session_record["sequence_max"],
                        collector_sequence,
                    )
                )

            receipt_time_ns = safe_integer(
                extracted.get("local_receipt_time_ns")
            )

            if receipt_time_ns is not None:
                session_record = session_statistics[session_id]

                session_record["receipt_min_ns"] = (
                    receipt_time_ns
                    if session_record["receipt_min_ns"] is None
                    else min(
                        session_record["receipt_min_ns"],
                        receipt_time_ns,
                    )
                )

                session_record["receipt_max_ns"] = (
                    receipt_time_ns
                    if session_record["receipt_max_ns"] is None
                    else max(
                        session_record["receipt_max_ns"],
                        receipt_time_ns,
                    )
                )

    return {
        "stream_name": stream_name,
        "path": str(path),
        "row_count": row_count,
        "invalid_json_lines": invalid_json_lines,
        "non_object_lines": non_object_lines,
        "missing_field_counts": missing_field_counts,
        "numeric_ranges": numeric_ranges,
        "source_order_reversals": source_order_reversals,
        "collector_sequences": collector_sequences,
        "duplicate_collector_sequences": (
            duplicate_collector_sequences
        ),
        "session_statistics": dict(session_statistics),
    }


TRADE_COVERAGE_SCAN = scan_jsonl_coverage(
    path=TRADE_SOURCE_PATH,
    stream_name="TRADE_STREAM",
    field_contract=TRADE_COVERAGE_FIELDS,
)

DEPTH_COVERAGE_SCAN = scan_jsonl_coverage(
    path=DEPTH_SOURCE_PATH,
    stream_name="DEPTH_STREAM",
    field_contract=DEPTH_COVERAGE_FIELDS,
)


# ============================================================
# STREAM COVERAGE TABLE
# ============================================================

def coverage_summary_row(scan: dict) -> dict:
    ranges = scan["numeric_ranges"]

    receipt_min = ranges[
        "local_receipt_time_ns"
    ]["minimum"]

    receipt_max = ranges[
        "local_receipt_time_ns"
    ]["maximum"]

    duration_seconds = (
        (receipt_max - receipt_min) / 1_000_000_000
        if receipt_min is not None and receipt_max is not None
        else None
    )

    event_range = ranges.get(
        "exchange_event_time_ms",
        {},
    )

    trade_range = ranges.get(
        "trade_time_ms",
        {},
    )

    sequence_range = ranges[
        "collector_sequence"
    ]

    return {
        "stream": scan["stream_name"],
        "row_count": int(scan["row_count"]),
        "collector_sequence_min": sequence_range.get(
            "minimum"
        ),
        "collector_sequence_max": sequence_range.get(
            "maximum"
        ),
        "distinct_collector_sequences": int(
            len(scan["collector_sequences"])
        ),
        "duplicate_collector_sequences": int(
            scan["duplicate_collector_sequences"]
        ),
        "first_local_receipt_utc": timestamp_iso(
            receipt_min,
            unit="ns",
        ),
        "last_local_receipt_utc": timestamp_iso(
            receipt_max,
            unit="ns",
        ),
        "coverage_seconds": duration_seconds,
        "first_exchange_event_utc": timestamp_iso(
            event_range.get("minimum"),
            unit="ms",
        ),
        "last_exchange_event_utc": timestamp_iso(
            event_range.get("maximum"),
            unit="ms",
        ),
        "first_trade_time_utc": timestamp_iso(
            trade_range.get("minimum"),
            unit="ms",
        ),
        "last_trade_time_utc": timestamp_iso(
            trade_range.get("maximum"),
            unit="ms",
        ),
        "session_count": int(
            len(scan["session_statistics"])
        ),
        "invalid_json_lines": int(
            scan["invalid_json_lines"]
        ),
        "non_object_lines": int(
            scan["non_object_lines"]
        ),
        "collector_sequence_reversals": int(
            scan["source_order_reversals"].get(
                "collector_sequence",
                0,
            )
        ),
        "receipt_time_reversals": int(
            scan["source_order_reversals"].get(
                "local_receipt_time_ns",
                0,
            )
        ),
    }


SOURCE_COVERAGE_SUMMARY = pd.DataFrame(
    [
        coverage_summary_row(TRADE_COVERAGE_SCAN),
        coverage_summary_row(DEPTH_COVERAGE_SCAN),
    ]
)


# ============================================================
# COMBINED COLLECTOR-SEQUENCE COVERAGE
# ============================================================

trade_sequences = TRADE_COVERAGE_SCAN[
    "collector_sequences"
]

depth_sequences = DEPTH_COVERAGE_SCAN[
    "collector_sequences"
]

combined_sequences = trade_sequences | depth_sequences
cross_stream_sequence_duplicates = (
    trade_sequences & depth_sequences
)

combined_sequence_min = (
    min(combined_sequences)
    if combined_sequences
    else None
)

combined_sequence_max = (
    max(combined_sequences)
    if combined_sequences
    else None
)

combined_expected_count = (
    combined_sequence_max - combined_sequence_min + 1
    if (
        combined_sequence_min is not None
        and combined_sequence_max is not None
    )
    else None
)

combined_missing_sequence_count = (
    combined_expected_count - len(combined_sequences)
    if combined_expected_count is not None
    else None
)

COLLECTOR_SEQUENCE_COVERAGE = pd.DataFrame(
    [
        {
            "collector_sequence_min": combined_sequence_min,
            "collector_sequence_max": combined_sequence_max,
            "expected_inclusive_count": combined_expected_count,
            "observed_distinct_count": int(
                len(combined_sequences)
            ),
            "missing_sequence_count": (
                combined_missing_sequence_count
            ),
            "cross_stream_duplicate_count": int(
                len(cross_stream_sequence_duplicates)
            ),
            "trade_sequence_count": int(
                len(trade_sequences)
            ),
            "depth_sequence_count": int(
                len(depth_sequences)
            ),
        }
    ]
)


# ============================================================
# SESSION-LEVEL COVERAGE
# ============================================================

all_session_ids = sorted(
    set(
        TRADE_COVERAGE_SCAN[
            "session_statistics"
        ].keys()
    )
    |
    set(
        DEPTH_COVERAGE_SCAN[
            "session_statistics"
        ].keys()
    )
)

session_rows = []

for session_id in all_session_ids:
    trade_session = TRADE_COVERAGE_SCAN[
        "session_statistics"
    ].get(session_id, {})

    depth_session = DEPTH_COVERAGE_SCAN[
        "session_statistics"
    ].get(session_id, {})

    receipt_min_candidates = [
        value
        for value in (
            trade_session.get("receipt_min_ns"),
            depth_session.get("receipt_min_ns"),
        )
        if value is not None
    ]

    receipt_max_candidates = [
        value
        for value in (
            trade_session.get("receipt_max_ns"),
            depth_session.get("receipt_max_ns"),
        )
        if value is not None
    ]

    receipt_min_ns = (
        min(receipt_min_candidates)
        if receipt_min_candidates
        else None
    )

    receipt_max_ns = (
        max(receipt_max_candidates)
        if receipt_max_candidates
        else None
    )

    session_rows.append(
        {
            "connection_session_id": session_id,
            "first_receipt_utc": timestamp_iso(
                receipt_min_ns,
                unit="ns",
            ),
            "last_receipt_utc": timestamp_iso(
                receipt_max_ns,
                unit="ns",
            ),
            "coverage_seconds": (
                (
                    receipt_max_ns - receipt_min_ns
                )
                / 1_000_000_000
                if (
                    receipt_min_ns is not None
                    and receipt_max_ns is not None
                )
                else None
            ),
            "trade_rows": int(
                trade_session.get("row_count", 0)
            ),
            "depth_rows": int(
                depth_session.get("row_count", 0)
            ),
            "combined_rows": int(
                trade_session.get("row_count", 0)
                + depth_session.get("row_count", 0)
            ),
            "trade_sequence_min": trade_session.get(
                "sequence_min"
            ),
            "trade_sequence_max": trade_session.get(
                "sequence_max"
            ),
            "depth_sequence_min": depth_session.get(
                "sequence_min"
            ),
            "depth_sequence_max": depth_session.get(
                "sequence_max"
            ),
        }
    )


SESSION_COVERAGE = pd.DataFrame(session_rows)


# ============================================================
# METADATA CROSS-CHECK
# ============================================================

declared_trade_count = safe_integer(
    nested_value(
        DEVELOPMENT_MANIFEST_DOCUMENT,
        ("message_counts", "trade_messages"),
    )
)

declared_depth_count = safe_integer(
    nested_value(
        DEVELOPMENT_MANIFEST_DOCUMENT,
        ("message_counts", "depth_messages"),
    )
)

declared_session_id = nested_value(
    SESSION_METADATA_DOCUMENT,
    ("connection_session_id",),
)

metadata_crosscheck_rows = [
    {
        "check": "trade_row_count",
        "declared_value": declared_trade_count,
        "observed_value": int(
            TRADE_COVERAGE_SCAN["row_count"]
        ),
    },
    {
        "check": "depth_row_count",
        "declared_value": declared_depth_count,
        "observed_value": int(
            DEPTH_COVERAGE_SCAN["row_count"]
        ),
    },
    {
        "check": "connection_session_id",
        "declared_value": (
            str(declared_session_id)
            if declared_session_id is not None
            else None
        ),
        "observed_value": (
            " | ".join(all_session_ids)
            if all_session_ids
            else None
        ),
    },
]

METADATA_COVERAGE_CROSSCHECK = pd.DataFrame(
    metadata_crosscheck_rows
)

METADATA_COVERAGE_CROSSCHECK["status"] = (
    METADATA_COVERAGE_CROSSCHECK.apply(
        lambda row: (
            "PASS"
            if (
                row["declared_value"] is not None
                and str(row["declared_value"])
                == str(row["observed_value"])
            )
            else (
                "NOT_DECLARED"
                if row["declared_value"] is None
                else "FAIL_MISMATCH"
            )
        ),
        axis=1,
    )
)


# ============================================================
# LIGHTWEIGHT COVERAGE GATES
# ============================================================

coverage_gate_rows = [
    {
        "gate": "trade_stream_readable",
        "passed": (
            TRADE_COVERAGE_SCAN["row_count"] > 0
            and TRADE_COVERAGE_SCAN[
                "invalid_json_lines"
            ] == 0
            and TRADE_COVERAGE_SCAN[
                "non_object_lines"
            ] == 0
        ),
        "evidence": (
            f"rows={TRADE_COVERAGE_SCAN['row_count']}; "
            f"invalid_json="
            f"{TRADE_COVERAGE_SCAN['invalid_json_lines']}; "
            f"non_objects="
            f"{TRADE_COVERAGE_SCAN['non_object_lines']}"
        ),
    },
    {
        "gate": "depth_stream_readable",
        "passed": (
            DEPTH_COVERAGE_SCAN["row_count"] > 0
            and DEPTH_COVERAGE_SCAN[
                "invalid_json_lines"
            ] == 0
            and DEPTH_COVERAGE_SCAN[
                "non_object_lines"
            ] == 0
        ),
        "evidence": (
            f"rows={DEPTH_COVERAGE_SCAN['row_count']}; "
            f"invalid_json="
            f"{DEPTH_COVERAGE_SCAN['invalid_json_lines']}; "
            f"non_objects="
            f"{DEPTH_COVERAGE_SCAN['non_object_lines']}"
        ),
    },
    {
        "gate": "receipt_coverage_established",
        "passed": bool(
            SOURCE_COVERAGE_SUMMARY[
                "first_local_receipt_utc"
            ].notna().all()
            and SOURCE_COVERAGE_SUMMARY[
                "last_local_receipt_utc"
            ].notna().all()
        ),
        "evidence": (
            f"streams_with_start="
            f"{SOURCE_COVERAGE_SUMMARY['first_local_receipt_utc'].notna().sum()}/2; "
            f"streams_with_end="
            f"{SOURCE_COVERAGE_SUMMARY['last_local_receipt_utc'].notna().sum()}/2"
        ),
    },
    {
        "gate": "collector_sequence_coverage_established",
        "passed": bool(combined_sequences),
        "evidence": (
            f"range={combined_sequence_min}.."
            f"{combined_sequence_max}; "
            f"observed={len(combined_sequences)}"
        ),
    },
    {
        "gate": "metadata_counts_match",
        "passed": bool(
            METADATA_COVERAGE_CROSSCHECK.loc[
                METADATA_COVERAGE_CROSSCHECK[
                    "check"
                ].isin(
                    [
                        "trade_row_count",
                        "depth_row_count",
                    ]
                ),
                "status",
            ]
            .eq("PASS")
            .all()
        ),
        "evidence": (
            "trade="
            f"{TRADE_COVERAGE_SCAN['row_count']}/"
            f"{declared_trade_count}; "
            "depth="
            f"{DEPTH_COVERAGE_SCAN['row_count']}/"
            f"{declared_depth_count}"
        ),
    },
]

CHRONOLOGICAL_COVERAGE_GATES = pd.DataFrame(
    coverage_gate_rows
)


display(SOURCE_COVERAGE_SUMMARY)
display(COLLECTOR_SEQUENCE_COVERAGE)
display(SESSION_COVERAGE)
display(METADATA_COVERAGE_CROSSCHECK)
display(CHRONOLOGICAL_COVERAGE_GATES)


failed_coverage_gates = CHRONOLOGICAL_COVERAGE_GATES.loc[
    ~CHRONOLOGICAL_COVERAGE_GATES["passed"]
]

if not failed_coverage_gates.empty:
    raise RuntimeError(
        "Chronological coverage scan failed: "
        + ", ".join(
            failed_coverage_gates["gate"].tolist()
        )
    )


print("Lightweight chronological coverage scan: PASS")
print(
    "Observed collection sessions: "
    f"{len(SESSION_COVERAGE):,}"
)
print(
    "Combined collector-sequence range: "
    f"{combined_sequence_min:,} to "
    f"{combined_sequence_max:,}"
)
print(
    "Combined raw messages: "
    f"{TRADE_COVERAGE_SCAN['row_count'] + DEPTH_COVERAGE_SCAN['row_count']:,}"
)

,stream,row_count,collector_sequence_min,collector_sequence_max,distinct_collector_sequences,duplicate_collector_sequences,first_local_receipt_utc,last_local_receipt_utc,coverage_seconds,first_exchange_event_utc,last_exchange_event_utc,first_trade_time_utc,last_trade_time_utc,session_count,invalid_json_lines,non_object_lines,collector_sequence_reversals,receipt_time_reversals
0,TRADE_STREAM,67683,14,103676,67683,0,2026-07-10T06:37:48.766951600+00:00,2026-07-10T07:37:46.707304900+00:00,3597.940353,2026-07-10T06:37:49.953000+00:00,2026-07-10T07:37:47.970000+00:00,2026-07-10T06:37:49.952000+00:00,2026-07-10T07:37:47.969000+00:00,1,0,0,0,0
1,DEPTH_STREAM,35994,1,103677,35994,0,2026-07-10T06:37:47.531985400+00:00,2026-07-10T07:37:46.749750800+00:00,3599.217765,2026-07-10T06:37:48.714000+00:00,2026-07-10T07:37:48.014000+00:00,None,None,1,0,0,0,0


,collector_sequence_min,collector_sequence_max,expected_inclusive_count,observed_distinct_count,missing_sequence_count,cross_stream_duplicate_count,trade_sequence_count,depth_sequence_count
0,1,103677,103677,103677,0,0,67683,35994


,connection_session_id,first_receipt_utc,last_receipt_utc,coverage_seconds,trade_rows,depth_rows,combined_rows,trade_sequence_min,trade_sequence_max,depth_sequence_min,depth_sequence_max
0,c8b5bf127a7a44669514acfda8634107,2026-07-10T06:37:47.531985400+00:00,2026-07-10T07:37:46.749750800+00:00,3599.217765,67683,35994,103677,14,103676,1,103677


,check,declared_value,observed_value,status
0,trade_row_count,67683,67683,PASS
1,depth_row_count,35994,35994,PASS
2,connection_session_id,c8b5bf127a7a44669514acfda8634107,c8b5bf127a7a44669514acfda8634107,PASS


,gate,passed,evidence
0,trade_stream_readable,True,rows=67683; invalid_json=0; non_objects=0
1,depth_stream_readable,True,rows=35994; invalid_json=0; non_objects=0
2,receipt_coverage_established,True,streams_with_start=2/2; streams_with_end=2/2
3,collector_sequence_coverage_established,True,range=1..103677; observed=103677
4,metadata_counts_match,True,trade=67683/67683; depth=35994/35994


Lightweight chronological coverage scan: PASS
Observed collection sessions: 1
Combined collector-sequence range: 1 to 103,677
Combined raw messages: 103,677


In [11]:
# ============================================================
# DATA SUFFICIENCY AND V0.1 OPERATING MODE
# ============================================================

# These thresholds are contract rules for this project run.
# They are not claims that four sessions are universally sufficient
# for statistical generalization.
FULL_MODE_MIN_INDEPENDENT_SESSIONS = 4
MEANINGFUL_SESSION_MIN_SECONDS = 300.0
MEANINGFUL_SESSION_MIN_TRADE_ROWS = 1_000
MEANINGFUL_SESSION_MIN_DEPTH_ROWS = 1_000


# ============================================================
# SOURCE AND COVERAGE READINESS
# ============================================================

required_source_roles_complete = (
    required_roles == selected_required_roles
)

authoritative_hashes_complete = bool(
    not AUTHORITATIVE_RAW_SOURCE_MANIFEST.empty
    and AUTHORITATIVE_RAW_SOURCE_MANIFEST[
        "hash_status"
    ].eq("PASS").all()
)

declared_checksums_valid = bool(
    not CHECKSUM_VERIFICATION.empty
    and ~CHECKSUM_VERIFICATION[
        "verification_status"
    ].str.startswith("FAIL", na=False).any()
)

chronological_coverage_valid = bool(
    not CHRONOLOGICAL_COVERAGE_GATES.empty
    and CHRONOLOGICAL_COVERAGE_GATES[
        "passed"
    ].all()
)

collector_sequence_complete = bool(
    len(COLLECTOR_SEQUENCE_COVERAGE) == 1
    and int(
        COLLECTOR_SEQUENCE_COVERAGE.iloc[0][
            "missing_sequence_count"
        ]
    ) == 0
    and int(
        COLLECTOR_SEQUENCE_COVERAGE.iloc[0][
            "cross_stream_duplicate_count"
        ]
    ) == 0
)


# ============================================================
# MEANINGFUL SESSION ASSESSMENT
# ============================================================

SESSION_SUFFICIENCY_AUDIT = SESSION_COVERAGE.copy()

if SESSION_SUFFICIENCY_AUDIT.empty:
    SESSION_SUFFICIENCY_AUDIT = pd.DataFrame(
        columns=[
            "connection_session_id",
            "coverage_seconds",
            "trade_rows",
            "depth_rows",
            "combined_rows",
        ]
    )

for column in (
    "coverage_seconds",
    "trade_rows",
    "depth_rows",
    "combined_rows",
):
    SESSION_SUFFICIENCY_AUDIT[column] = pd.to_numeric(
        SESSION_SUFFICIENCY_AUDIT[column],
        errors="coerce",
    )

SESSION_SUFFICIENCY_AUDIT[
    "duration_requirement_passed"
] = (
    SESSION_SUFFICIENCY_AUDIT["coverage_seconds"]
    .fillna(0)
    .ge(MEANINGFUL_SESSION_MIN_SECONDS)
)

SESSION_SUFFICIENCY_AUDIT[
    "trade_activity_requirement_passed"
] = (
    SESSION_SUFFICIENCY_AUDIT["trade_rows"]
    .fillna(0)
    .ge(MEANINGFUL_SESSION_MIN_TRADE_ROWS)
)

SESSION_SUFFICIENCY_AUDIT[
    "depth_activity_requirement_passed"
] = (
    SESSION_SUFFICIENCY_AUDIT["depth_rows"]
    .fillna(0)
    .ge(MEANINGFUL_SESSION_MIN_DEPTH_ROWS)
)

SESSION_SUFFICIENCY_AUDIT[
    "meaningful_session"
] = (
    SESSION_SUFFICIENCY_AUDIT[
        "duration_requirement_passed"
    ]
    & SESSION_SUFFICIENCY_AUDIT[
        "trade_activity_requirement_passed"
    ]
    & SESSION_SUFFICIENCY_AUDIT[
        "depth_activity_requirement_passed"
    ]
)

meaningful_session_count = int(
    SESSION_SUFFICIENCY_AUDIT[
        "meaningful_session"
    ].sum()
)

observed_session_count = int(
    len(SESSION_SUFFICIENCY_AUDIT)
)

total_collection_seconds = float(
    SESSION_SUFFICIENCY_AUDIT[
        "coverage_seconds"
    ].fillna(0).sum()
)


# ============================================================
# MODE ELIGIBILITY CONDITIONS
# ============================================================

base_engineering_requirements = {
    "required_source_roles_complete": (
        required_source_roles_complete
    ),
    "authoritative_hashes_complete": (
        authoritative_hashes_complete
    ),
    "declared_checksums_valid": (
        declared_checksums_valid
    ),
    "chronological_coverage_valid": (
        chronological_coverage_valid
    ),
    "collector_sequence_complete": (
        collector_sequence_complete
    ),
    "trade_stream_nonempty": bool(
        TRADE_COVERAGE_SCAN["row_count"] > 0
    ),
    "depth_stream_nonempty": bool(
        DEPTH_COVERAGE_SCAN["row_count"] > 0
    ),
}

engineering_ready = all(
    base_engineering_requirements.values()
)

full_statistical_requirements = {
    **base_engineering_requirements,
    "minimum_independent_sessions_available": (
        meaningful_session_count
        >= FULL_MODE_MIN_INDEPENDENT_SESSIONS
    ),
    "whole_session_four_way_split_possible": (
        meaningful_session_count
        >= FULL_MODE_MIN_INDEPENDENT_SESSIONS
    ),
    "development_session_available": (
        meaningful_session_count >= 1
    ),
    "calibration_session_available": (
        meaningful_session_count >= 2
    ),
    "validation_session_available": (
        meaningful_session_count >= 3
    ),
    "untouched_holdout_session_available": (
        meaningful_session_count >= 4
    ),
}

full_statistical_ready = all(
    full_statistical_requirements.values()
)


# ============================================================
# ASSIGN OPERATING MODE
# ============================================================

if full_statistical_ready:
    V0_1_OPERATING_MODE = "FULL_STATISTICAL_MODE"
    V0_1_CONTRACT_STATUS = "PASS"

    OPERATING_MODE_REASON = (
        "The source collection supports a whole-session "
        "development, calibration, validation, and untouched "
        "holdout design."
    )

elif engineering_ready:
    V0_1_OPERATING_MODE = "ENGINEERING_REPRODUCTION_MODE"
    V0_1_CONTRACT_STATUS = "CONDITIONAL PASS"

    OPERATING_MODE_REASON = (
        "The raw collection is complete, internally ordered, "
        "checksum-verified, and sufficient for causal pipeline "
        "reconstruction. It contains too few independent sessions "
        "for claim-bearing development, calibration, validation, "
        "and holdout partitions."
    )

else:
    V0_1_OPERATING_MODE = "INSUFFICIENT_SOURCE_DATA"
    V0_1_CONTRACT_STATUS = "FAIL"

    failed_base_requirements = [
        requirement
        for requirement, passed
        in base_engineering_requirements.items()
        if not passed
    ]

    OPERATING_MODE_REASON = (
        "Critical source or chronological requirements failed: "
        + ", ".join(failed_base_requirements)
    )


# ============================================================
# REQUIREMENT AUDIT
# ============================================================

DATA_SUFFICIENCY_REQUIREMENTS = pd.DataFrame(
    [
        {
            "requirement": requirement,
            "requirement_group": "ENGINEERING_BASE",
            "required_for_mode": (
                "ENGINEERING_REPRODUCTION_MODE"
            ),
            "passed": bool(passed),
        }
        for requirement, passed
        in base_engineering_requirements.items()
    ]
    +
    [
        {
            "requirement": requirement,
            "requirement_group": "FULL_STATISTICAL",
            "required_for_mode": "FULL_STATISTICAL_MODE",
            "passed": bool(passed),
        }
        for requirement, passed
        in full_statistical_requirements.items()
        if requirement not in base_engineering_requirements
    ]
)

DATA_SUFFICIENCY_REQUIREMENTS["status"] = np.where(
    DATA_SUFFICIENCY_REQUIREMENTS["passed"],
    "PASS",
    "FAIL",
)


# ============================================================
# CLAIM AND DOWNSTREAM PERMISSION MATRIX
# ============================================================

if V0_1_OPERATING_MODE == "FULL_STATISTICAL_MODE":
    permission_values = {
        "reconstruct_v0_1_market_data": True,
        "estimate_ordinary_bivariate_hawkes": True,
        "compare_candidate_hawkes_models": True,
        "validate_intensity_signal": True,
        "use_chronological_holdout_once": True,
        "make_independent_validation_claims": True,
        "make_out_of_sample_generalization_claims": True,
        "treat_partitions_as_independent": True,
        "proceed_to_notebook_01": True,
    }

elif V0_1_OPERATING_MODE == "ENGINEERING_REPRODUCTION_MODE":
    permission_values = {
        "reconstruct_v0_1_market_data": True,
        "estimate_ordinary_bivariate_hawkes": True,
        "compare_candidate_hawkes_models": True,
        "validate_intensity_signal": True,
        "use_chronological_holdout_once": False,
        "make_independent_validation_claims": False,
        "make_out_of_sample_generalization_claims": False,
        "treat_partitions_as_independent": False,
        "proceed_to_notebook_01": True,
    }

else:
    permission_values = {
        "reconstruct_v0_1_market_data": False,
        "estimate_ordinary_bivariate_hawkes": False,
        "compare_candidate_hawkes_models": False,
        "validate_intensity_signal": False,
        "use_chronological_holdout_once": False,
        "make_independent_validation_claims": False,
        "make_out_of_sample_generalization_claims": False,
        "treat_partitions_as_independent": False,
        "proceed_to_notebook_01": False,
    }


permission_explanations = {
    "reconstruct_v0_1_market_data": (
        "Rebuild V0.1 datasets from immutable V0.0 raw inputs."
    ),
    "estimate_ordinary_bivariate_hawkes": (
        "Estimate provisional buy/sell Hawkes parameters."
    ),
    "compare_candidate_hawkes_models": (
        "Model comparisons remain engineering evidence unless "
        "full statistical mode is available."
    ),
    "validate_intensity_signal": (
        "Signal diagnostics are permitted, subject to the declared "
        "mode's claim restrictions."
    ),
    "use_chronological_holdout_once": (
        "A claim-bearing untouched holdout requires an independent "
        "chronological session."
    ),
    "make_independent_validation_claims": (
        "Requires genuinely separated validation coverage."
    ),
    "make_out_of_sample_generalization_claims": (
        "Requires independent holdout evidence."
    ),
    "treat_partitions_as_independent": (
        "Slices of one continuous collection session are dependent."
    ),
    "proceed_to_notebook_01": (
        "Notebook 01 may begin only when engineering requirements pass."
    ),
}


CLAIM_PERMISSION_MATRIX = pd.DataFrame(
    [
        {
            "permission": permission,
            "allowed": bool(allowed),
            "explanation": permission_explanations[
                permission
            ],
        }
        for permission, allowed
        in permission_values.items()
    ]
)


# ============================================================
# FINAL DATA-SUFFICIENCY DECISION
# ============================================================

failed_full_mode_conditions = (
    DATA_SUFFICIENCY_REQUIREMENTS.loc[
        DATA_SUFFICIENCY_REQUIREMENTS[
            "requirement_group"
        ].eq("FULL_STATISTICAL")
        & ~DATA_SUFFICIENCY_REQUIREMENTS["passed"],
        "requirement",
    ].tolist()
)

DATA_SUFFICIENCY_DECISION = pd.DataFrame(
    [
        {
            "selected_operating_mode": V0_1_OPERATING_MODE,
            "contract_status": V0_1_CONTRACT_STATUS,
            "observed_sessions": observed_session_count,
            "meaningful_sessions": meaningful_session_count,
            "minimum_sessions_for_full_mode": (
                FULL_MODE_MIN_INDEPENDENT_SESSIONS
            ),
            "total_collection_seconds": (
                total_collection_seconds
            ),
            "total_collection_hours": (
                total_collection_seconds / 3600.0
            ),
            "engineering_requirements_passed": (
                engineering_ready
            ),
            "full_statistical_requirements_passed": (
                full_statistical_ready
            ),
            "claim_bearing_holdout_available": (
                permission_values[
                    "use_chronological_holdout_once"
                ]
            ),
            "failed_full_mode_conditions": (
                " | ".join(failed_full_mode_conditions)
                if failed_full_mode_conditions
                else None
            ),
            "decision_reason": OPERATING_MODE_REASON,
        }
    ]
)


display(
    SESSION_SUFFICIENCY_AUDIT[
        [
            "connection_session_id",
            "coverage_seconds",
            "trade_rows",
            "depth_rows",
            "duration_requirement_passed",
            "trade_activity_requirement_passed",
            "depth_activity_requirement_passed",
            "meaningful_session",
        ]
    ]
)

display(DATA_SUFFICIENCY_REQUIREMENTS)
display(DATA_SUFFICIENCY_DECISION)
display(CLAIM_PERMISSION_MATRIX)


if V0_1_OPERATING_MODE == "INSUFFICIENT_SOURCE_DATA":
    raise RuntimeError(
        "V0.1 cannot proceed because the source collection failed "
        "the minimum engineering requirements."
    )


print(f"V0.1 operating mode: {V0_1_OPERATING_MODE}")
print(f"Contract status: {V0_1_CONTRACT_STATUS}")
print(
    "Meaningful independent sessions: "
    f"{meaningful_session_count:,}"
)
print(
    "Claim-bearing untouched holdout available: "
    f"{permission_values['use_chronological_holdout_once']}"
)
print(
    "Notebook 01 permission: "
    f"{permission_values['proceed_to_notebook_01']}"
)

,connection_session_id,coverage_seconds,trade_rows,depth_rows,duration_requirement_passed,trade_activity_requirement_passed,depth_activity_requirement_passed,meaningful_session
0,c8b5bf127a7a44669514acfda8634107,3599.217765,67683,35994,True,True,True,True


,requirement,requirement_group,required_for_mode,passed,status
0,required_source_roles_complete,ENGINEERING_BASE,ENGINEERING_REPRODUCTION_MODE,True,PASS
1,authoritative_hashes_complete,ENGINEERING_BASE,ENGINEERING_REPRODUCTION_MODE,True,PASS
2,declared_checksums_valid,ENGINEERING_BASE,ENGINEERING_REPRODUCTION_MODE,True,PASS
3,chronological_coverage_valid,ENGINEERING_BASE,ENGINEERING_REPRODUCTION_MODE,True,PASS
4,collector_sequence_complete,ENGINEERING_BASE,ENGINEERING_REPRODUCTION_MODE,True,PASS
5,trade_stream_nonempty,ENGINEERING_BASE,ENGINEERING_REPRODUCTION_MODE,True,PASS
6,depth_stream_nonempty,ENGINEERING_BASE,ENGINEERING_REPRODUCTION_MODE,True,PASS
7,minimum_independent_sessions_available,FULL_STATISTICAL,FULL_STATISTICAL_MODE,False,FAIL
8,whole_session_four_way_split_possible,FULL_STATISTICAL,FULL_STATISTICAL_MODE,False,FAIL
9,development_session_available,FULL_STATISTICAL,FULL_STATISTICAL_MODE,True,PASS


,selected_operating_mode,contract_status,observed_sessions,meaningful_sessions,minimum_sessions_for_full_mode,total_collection_seconds,total_collection_hours,engineering_requirements_passed,full_statistical_requirements_passed,claim_bearing_holdout_available,failed_full_mode_conditions,decision_reason
0,ENGINEERING_REPRODUCTION_MODE,CONDITIONAL PASS,1,1,4,3599.217765,0.999783,True,False,False,minimum_independent_sessions_available | whole...,"The raw collection is complete, internally ord..."


,permission,allowed,explanation
0,reconstruct_v0_1_market_data,True,Rebuild V0.1 datasets from immutable V0.0 raw ...
1,estimate_ordinary_bivariate_hawkes,True,Estimate provisional buy/sell Hawkes parameters.
2,compare_candidate_hawkes_models,True,Model comparisons remain engineering evidence ...
3,validate_intensity_signal,True,"Signal diagnostics are permitted, subject to t..."
4,use_chronological_holdout_once,False,A claim-bearing untouched holdout requires an ...
5,make_independent_validation_claims,False,Requires genuinely separated validation coverage.
6,make_out_of_sample_generalization_claims,False,Requires independent holdout evidence.
7,treat_partitions_as_independent,False,Slices of one continuous collection session ar...
8,proceed_to_notebook_01,True,Notebook 01 may begin only when engineering re...


V0.1 operating mode: ENGINEERING_REPRODUCTION_MODE
Contract status: CONDITIONAL PASS
Meaningful independent sessions: 1
Claim-bearing untouched holdout available: False
Notebook 01 permission: True


In [14]:
# ============================================================
# REST SNAPSHOT WRAPPER AUDIT AND PAYLOAD REGISTRATION
# ============================================================

from decimal import Decimal, InvalidOperation


SNAPSHOT_SOURCE_PATH = selected_source_path("REST_SNAPSHOT")
SNAPSHOT_DOCUMENT = load_json_mapping(SNAPSHOT_SOURCE_PATH)


# ============================================================
# GENERIC JSON-CONTAINER HELPERS
# ============================================================

def decode_json_container(value):
    """
    Decode a JSON-like string or bytes object.

    Existing mappings and lists are returned unchanged. Values that
    are not valid JSON remain unchanged.
    """
    if isinstance(value, (dict, list)):
        return value

    if isinstance(value, bytes):
        try:
            value = value.decode("utf-8-sig")
        except UnicodeDecodeError:
            return value

    if isinstance(value, str):
        stripped = value.strip()

        if not stripped:
            return value

        if stripped[0] not in {"{", "["}:
            return value

        try:
            return json.loads(stripped)
        except json.JSONDecodeError:
            return value

    return value


def mapping_lookup(
    mapping: dict,
    aliases: tuple[str, ...],
):
    """Return the first matching key and value, case-insensitively."""
    for alias in aliases:
        if alias in mapping:
            return alias, mapping[alias]

    normalized_keys = {
        str(key).casefold(): key
        for key in mapping
    }

    for alias in aliases:
        matched_key = normalized_keys.get(alias.casefold())

        if matched_key is not None:
            return matched_key, mapping[matched_key]

    return None, None


def walk_json_mappings(
    value,
    path: str = "$",
    depth: int = 0,
    max_depth: int = 8,
):
    """
    Yield mappings contained in a JSON-compatible object.

    Bid and ask level arrays are not recursively traversed because
    their rows are data values rather than nested metadata objects.
    """
    if depth > max_depth:
        return

    decoded = decode_json_container(value)

    if isinstance(decoded, dict):
        yield path, decoded

        for key, child in decoded.items():
            if str(key).casefold() in {"bids", "asks"}:
                continue

            child_decoded = decode_json_container(child)

            if isinstance(child_decoded, (dict, list)):
                yield from walk_json_mappings(
                    child_decoded,
                    path=f"{path}.{key}",
                    depth=depth + 1,
                    max_depth=max_depth,
                )

    elif isinstance(decoded, list):
        for index, child in enumerate(decoded):
            child_decoded = decode_json_container(child)

            if isinstance(child_decoded, (dict, list)):
                yield from walk_json_mappings(
                    child_decoded,
                    path=f"{path}[{index}]",
                    depth=depth + 1,
                    max_depth=max_depth,
                )


# Decode raw_response explicitly when the collector stored the
# exchange response separately from request metadata.
RAW_RESPONSE_KEY, RAW_RESPONSE_VALUE = mapping_lookup(
    SNAPSHOT_DOCUMENT,
    ("raw_response", "response_body", "response", "payload"),
)

SNAPSHOT_RAW_RESPONSE = decode_json_container(
    RAW_RESPONSE_VALUE
)

SNAPSHOT_SEARCH_DOCUMENT = dict(SNAPSHOT_DOCUMENT)

if RAW_RESPONSE_KEY is not None:
    SNAPSHOT_SEARCH_DOCUMENT[RAW_RESPONSE_KEY] = (
        SNAPSHOT_RAW_RESPONSE
    )


SNAPSHOT_MAPPING_CANDIDATES = list(
    walk_json_mappings(SNAPSHOT_SEARCH_DOCUMENT)
)


# ============================================================
# LOCATE BID/ASK ARRAYS
# ============================================================

book_candidate_rows = []
book_candidate_objects = []

for candidate_path, candidate_mapping in (
    SNAPSHOT_MAPPING_CANDIDATES
):
    bids_key, bids_value = mapping_lookup(
        candidate_mapping,
        ("bids", "bid_levels"),
    )

    asks_key, asks_value = mapping_lookup(
        candidate_mapping,
        ("asks", "ask_levels"),
    )

    if bids_key is None or asks_key is None:
        continue

    bids_decoded = decode_json_container(bids_value)
    asks_decoded = decode_json_container(asks_value)

    bids_is_list = isinstance(bids_decoded, list)
    asks_is_list = isinstance(asks_decoded, list)

    bid_count = (
        len(bids_decoded)
        if bids_is_list
        else None
    )

    ask_count = (
        len(asks_decoded)
        if asks_is_list
        else None
    )

    book_candidate_rows.append(
        {
            "candidate_path": candidate_path,
            "bids_key": bids_key,
            "asks_key": asks_key,
            "bids_is_list": bids_is_list,
            "asks_is_list": asks_is_list,
            "bid_level_count": bid_count,
            "ask_level_count": ask_count,
            "combined_level_count": (
                bid_count + ask_count
                if (
                    bid_count is not None
                    and ask_count is not None
                )
                else None
            ),
        }
    )

    book_candidate_objects.append(
        {
            "candidate_path": candidate_path,
            "mapping": candidate_mapping,
            "bids": bids_decoded,
            "asks": asks_decoded,
        }
    )


SNAPSHOT_BOOK_CANDIDATES = pd.DataFrame(
    book_candidate_rows
)

valid_book_candidates = [
    candidate
    for candidate in book_candidate_objects
    if (
        isinstance(candidate["bids"], list)
        and isinstance(candidate["asks"], list)
        and len(candidate["bids"]) > 0
        and len(candidate["asks"]) > 0
    )
]

if not valid_book_candidates:
    raise RuntimeError(
        "No non-empty bid/ask arrays were found in the REST "
        "snapshot wrapper or its raw_response field. "
        f"Top-level keys: {sorted(SNAPSHOT_DOCUMENT.keys())}"
    )


# Determine whether multiple discovered candidates represent the
# same book payload or genuinely conflicting payloads.
def book_payload_fingerprint(
    bids: list,
    asks: list,
) -> str:
    return hashlib.sha256(
        canonical_json_bytes(
            {
                "bids": bids,
                "asks": asks,
            }
        )
    ).hexdigest()


for candidate in valid_book_candidates:
    candidate["payload_fingerprint"] = (
        book_payload_fingerprint(
            candidate["bids"],
            candidate["asks"],
        )
    )

distinct_book_fingerprints = {
    candidate["payload_fingerprint"]
    for candidate in valid_book_candidates
}

if len(distinct_book_fingerprints) > 1:
    conflicting_paths = [
        candidate["candidate_path"]
        for candidate in valid_book_candidates
    ]

    raise RuntimeError(
        "Multiple conflicting bid/ask payloads were discovered: "
        + ", ".join(conflicting_paths)
    )


# Prefer the shallowest valid mapping when identical payloads appear
# at more than one nested path.
SELECTED_BOOK_CANDIDATE = sorted(
    valid_book_candidates,
    key=lambda candidate: (
        candidate["candidate_path"].count(".")
        + candidate["candidate_path"].count("["),
        candidate["candidate_path"],
    ),
)[0]

SNAPSHOT_BIDS = SELECTED_BOOK_CANDIDATE["bids"]
SNAPSHOT_ASKS = SELECTED_BOOK_CANDIDATE["asks"]
SNAPSHOT_BOOK_PAYLOAD_PATH = (
    SELECTED_BOOK_CANDIDATE["candidate_path"]
)
SNAPSHOT_BOOK_PAYLOAD_FINGERPRINT = (
    SELECTED_BOOK_CANDIDATE["payload_fingerprint"]
)


# ============================================================
# LOCATE lastUpdateId INDEPENDENTLY OF BID/ASK ARRAYS
# ============================================================

update_id_candidate_rows = []

for candidate_path, candidate_mapping in (
    SNAPSHOT_MAPPING_CANDIDATES
):
    update_id_key, update_id_value = mapping_lookup(
        candidate_mapping,
        (
            "lastUpdateId",
            "last_update_id",
            "last_update_id_raw",
        ),
    )

    if update_id_key is None:
        continue

    parsed_update_id = safe_integer(update_id_value)

    update_id_candidate_rows.append(
        {
            "candidate_path": candidate_path,
            "update_id_key": update_id_key,
            "raw_value": update_id_value,
            "parsed_value": parsed_update_id,
            "valid_integer": parsed_update_id is not None,
        }
    )


SNAPSHOT_UPDATE_ID_CANDIDATES = pd.DataFrame(
    update_id_candidate_rows
)

valid_update_ids = sorted(
    {
        int(row["parsed_value"])
        for row in update_id_candidate_rows
        if row["parsed_value"] is not None
    }
)

if not valid_update_ids:
    raise RuntimeError(
        "No valid integer lastUpdateId was found in the REST "
        "snapshot wrapper or nested raw response."
    )

if len(valid_update_ids) > 1:
    raise RuntimeError(
        "Conflicting lastUpdateId values were found: "
        + ", ".join(
            str(value)
            for value in valid_update_ids
        )
    )

SNAPSHOT_LAST_UPDATE_ID = valid_update_ids[0]

SNAPSHOT_UPDATE_ID_PATHS = [
    row["candidate_path"]
    for row in update_id_candidate_rows
    if row["parsed_value"] == SNAPSHOT_LAST_UPDATE_ID
]


# ============================================================
# VALIDATE BINANCE PRICE/QUANTITY LEVELS
# ============================================================

def parse_decimal_scalar(
    value,
) -> Decimal | None:
    """Parse a finite decimal scalar without using binary float."""
    if value is None or isinstance(value, bool):
        return None

    try:
        parsed = Decimal(str(value))
    except (InvalidOperation, ValueError, TypeError):
        return None

    if not parsed.is_finite():
        return None

    return parsed


def audit_book_side(
    levels: list,
    side: str,
) -> tuple[pd.DataFrame, dict]:
    """
    Validate one Binance snapshot side without altering level order
    or values.
    """
    audit_rows = []
    parsed_prices = []
    parsed_quantities = []

    for level_index, level in enumerate(levels):
        level_is_sequence = isinstance(level, (list, tuple))
        field_count = len(level) if level_is_sequence else None

        price_raw = (
            level[0]
            if level_is_sequence and len(level) >= 1
            else None
        )

        quantity_raw = (
            level[1]
            if level_is_sequence and len(level) >= 2
            else None
        )

        price = parse_decimal_scalar(price_raw)
        quantity = parse_decimal_scalar(quantity_raw)

        valid_structure = bool(
            level_is_sequence
            and field_count == 2
        )

        valid_price = bool(
            price is not None
            and price > 0
        )

        valid_quantity = bool(
            quantity is not None
            and quantity > 0
        )

        row_valid = bool(
            valid_structure
            and valid_price
            and valid_quantity
        )

        audit_rows.append(
            {
                "side": side,
                "level_index": level_index,
                "field_count": field_count,
                "price_raw": price_raw,
                "quantity_raw": quantity_raw,
                "valid_structure": valid_structure,
                "valid_price": valid_price,
                "valid_quantity": valid_quantity,
                "row_valid": row_valid,
            }
        )

        if row_valid:
            parsed_prices.append(price)
            parsed_quantities.append(quantity)

    audit_table = pd.DataFrame(audit_rows)

    expected_descending = side == "BID"

    if len(parsed_prices) == len(levels):
        if expected_descending:
            ordering_valid = all(
                current_price > next_price
                for current_price, next_price
                in zip(
                    parsed_prices,
                    parsed_prices[1:],
                )
            )
        else:
            ordering_valid = all(
                current_price < next_price
                for current_price, next_price
                in zip(
                    parsed_prices,
                    parsed_prices[1:],
                )
            )
    else:
        ordering_valid = False

    summary = {
        "side": side,
        "level_count": int(len(levels)),
        "valid_level_count": int(
            audit_table["row_valid"].sum()
        ),
        "invalid_level_count": int(
            (~audit_table["row_valid"]).sum()
        ),
        "strict_price_ordering_valid": bool(
            ordering_valid
        ),
        "best_price": (
            str(parsed_prices[0])
            if parsed_prices
            else None
        ),
        "worst_price": (
            str(parsed_prices[-1])
            if parsed_prices
            else None
        ),
        "total_quantity": (
            str(sum(parsed_quantities))
            if parsed_quantities
            else None
        ),
    }

    return audit_table, summary


SNAPSHOT_BID_LEVEL_AUDIT, bid_summary = (
    audit_book_side(
        SNAPSHOT_BIDS,
        side="BID",
    )
)

SNAPSHOT_ASK_LEVEL_AUDIT, ask_summary = (
    audit_book_side(
        SNAPSHOT_ASKS,
        side="ASK",
    )
)

SNAPSHOT_LEVEL_AUDIT = pd.concat(
    [
        SNAPSHOT_BID_LEVEL_AUDIT,
        SNAPSHOT_ASK_LEVEL_AUDIT,
    ],
    ignore_index=True,
)

SNAPSHOT_SIDE_SUMMARY = pd.DataFrame(
    [
        bid_summary,
        ask_summary,
    ]
)


# ============================================================
# WRAPPER METADATA CROSS-CHECK
# ============================================================

def top_level_integer(
    aliases: tuple[str, ...],
) -> int | None:
    _, value = mapping_lookup(
        SNAPSHOT_DOCUMENT,
        aliases,
    )
    return safe_integer(value)


DECLARED_BID_LEVEL_COUNT = top_level_integer(
    ("bid_level_count", "bids_count"),
)

DECLARED_ASK_LEVEL_COUNT = top_level_integer(
    ("ask_level_count", "asks_count"),
)

HTTP_STATUS = top_level_integer(
    ("http_status", "status_code"),
)

_, WRAPPER_STATUS = mapping_lookup(
    SNAPSHOT_DOCUMENT,
    ("status", "request_status"),
)

_, CONNECTION_SESSION_ID = mapping_lookup(
    SNAPSHOT_DOCUMENT,
    (
        "connection_session_id",
        "session_id",
    ),
)

BUFFERED_DEPTH_BEFORE_SNAPSHOT = top_level_integer(
    (
        "buffered_depth_before_snapshot",
        "buffered_depth_count",
    ),
)


SNAPSHOT_WRAPPER_CROSSCHECK = pd.DataFrame(
    [
        {
            "check": "bid_level_count",
            "declared_value": DECLARED_BID_LEVEL_COUNT,
            "observed_value": len(SNAPSHOT_BIDS),
            "required": (
                DECLARED_BID_LEVEL_COUNT is not None
            ),
        },
        {
            "check": "ask_level_count",
            "declared_value": DECLARED_ASK_LEVEL_COUNT,
            "observed_value": len(SNAPSHOT_ASKS),
            "required": (
                DECLARED_ASK_LEVEL_COUNT is not None
            ),
        },
        {
            "check": "http_status",
            "declared_value": HTTP_STATUS,
            "observed_value": 200,
            "required": HTTP_STATUS is not None,
        },
    ]
)

SNAPSHOT_WRAPPER_CROSSCHECK["status"] = (
    SNAPSHOT_WRAPPER_CROSSCHECK.apply(
        lambda row: (
            "NOT_DECLARED"
            if not bool(row["required"])
            else (
                "PASS"
                if str(row["declared_value"])
                == str(row["observed_value"])
                else "FAIL_MISMATCH"
            )
        ),
        axis=1,
    )
)


# ============================================================
# TOP-OF-BOOK SANITY CHECK
# ============================================================

best_bid = parse_decimal_scalar(
    SNAPSHOT_BIDS[0][0]
)

best_ask = parse_decimal_scalar(
    SNAPSHOT_ASKS[0][0]
)

spread = (
    best_ask - best_bid
    if (
        best_bid is not None
        and best_ask is not None
    )
    else None
)

mid_price = (
    (best_bid + best_ask) / Decimal("2")
    if (
        best_bid is not None
        and best_ask is not None
    )
    else None
)


# ============================================================
# SNAPSHOT CONTRACT GATES
# ============================================================

SNAPSHOT_STRUCTURE_GATES = pd.DataFrame(
    [
        {
            "gate": "last_update_id_available",
            "passed": (
                SNAPSHOT_LAST_UPDATE_ID >= 0
            ),
            "evidence": str(
                SNAPSHOT_LAST_UPDATE_ID
            ),
        },
        {
            "gate": "bid_array_nonempty",
            "passed": len(SNAPSHOT_BIDS) > 0,
            "evidence": (
                f"levels={len(SNAPSHOT_BIDS)}"
            ),
        },
        {
            "gate": "ask_array_nonempty",
            "passed": len(SNAPSHOT_ASKS) > 0,
            "evidence": (
                f"levels={len(SNAPSHOT_ASKS)}"
            ),
        },
        {
            "gate": "all_bid_levels_valid",
            "passed": bool(
                SNAPSHOT_BID_LEVEL_AUDIT[
                    "row_valid"
                ].all()
            ),
            "evidence": (
                f"valid="
                f"{int(SNAPSHOT_BID_LEVEL_AUDIT['row_valid'].sum())}/"
                f"{len(SNAPSHOT_BID_LEVEL_AUDIT)}"
            ),
        },
        {
            "gate": "all_ask_levels_valid",
            "passed": bool(
                SNAPSHOT_ASK_LEVEL_AUDIT[
                    "row_valid"
                ].all()
            ),
            "evidence": (
                f"valid="
                f"{int(SNAPSHOT_ASK_LEVEL_AUDIT['row_valid'].sum())}/"
                f"{len(SNAPSHOT_ASK_LEVEL_AUDIT)}"
            ),
        },
        {
            "gate": "bids_strictly_descending",
            "passed": bool(
                bid_summary[
                    "strict_price_ordering_valid"
                ]
            ),
            "evidence": (
                f"best={bid_summary['best_price']}; "
                f"worst={bid_summary['worst_price']}"
            ),
        },
        {
            "gate": "asks_strictly_ascending",
            "passed": bool(
                ask_summary[
                    "strict_price_ordering_valid"
                ]
            ),
            "evidence": (
                f"best={ask_summary['best_price']}; "
                f"worst={ask_summary['worst_price']}"
            ),
        },
        {
            "gate": "positive_top_of_book_spread",
            "passed": bool(
                spread is not None
                and spread > 0
            ),
            "evidence": (
                f"best_bid={best_bid}; "
                f"best_ask={best_ask}; "
                f"spread={spread}"
            ),
        },
        {
            "gate": "wrapper_metadata_consistent",
            "passed": bool(
                ~SNAPSHOT_WRAPPER_CROSSCHECK[
                    "status"
                ].eq("FAIL_MISMATCH").any()
            ),
            "evidence": (
                " | ".join(
                    f"{row.check}={row.status}"
                    for row
                    in SNAPSHOT_WRAPPER_CROSSCHECK.itertuples(
                        index=False
                    )
                )
            ),
        },
    ]
)


# ============================================================
# REGISTER THE RESOLVED SNAPSHOT CONTRACT
# ============================================================

RESOLVED_SNAPSHOT_CONTRACT = {
    "source_path": str(SNAPSHOT_SOURCE_PATH),
    "source_sha256": str(
        AUTHORITATIVE_RAW_SOURCE_MANIFEST.loc[
            AUTHORITATIVE_RAW_SOURCE_MANIFEST[
                "source_role"
            ].eq("REST_SNAPSHOT"),
            "sha256",
        ].iloc[0]
    ),
    "wrapper_format": (
        "COLLECTOR_METADATA_WITH_EMBEDDED_RAW_RESPONSE"
        if RAW_RESPONSE_KEY is not None
        else "DIRECT_EXCHANGE_RESPONSE"
    ),
    "raw_response_key": RAW_RESPONSE_KEY,
    "book_payload_path": SNAPSHOT_BOOK_PAYLOAD_PATH,
    "update_id_paths": SNAPSHOT_UPDATE_ID_PATHS,
    "last_update_id": SNAPSHOT_LAST_UPDATE_ID,
    "bid_level_count": int(len(SNAPSHOT_BIDS)),
    "ask_level_count": int(len(SNAPSHOT_ASKS)),
    "best_bid": str(best_bid),
    "best_ask": str(best_ask),
    "spread": str(spread),
    "mid_price": str(mid_price),
    "book_payload_fingerprint": (
        SNAPSHOT_BOOK_PAYLOAD_FINGERPRINT
    ),
    "http_status": HTTP_STATUS,
    "wrapper_status": WRAPPER_STATUS,
    "connection_session_id": CONNECTION_SESSION_ID,
    "buffered_depth_before_snapshot": (
        BUFFERED_DEPTH_BEFORE_SNAPSHOT
    ),
    "authority_class": "PRIMARY_RAW",
    "modification_allowed": False,
}


display(SNAPSHOT_BOOK_CANDIDATES)
display(SNAPSHOT_UPDATE_ID_CANDIDATES)
display(SNAPSHOT_SIDE_SUMMARY)
display(SNAPSHOT_WRAPPER_CROSSCHECK)
display(SNAPSHOT_STRUCTURE_GATES)

failed_snapshot_gates = SNAPSHOT_STRUCTURE_GATES.loc[
    ~SNAPSHOT_STRUCTURE_GATES["passed"]
]

if not failed_snapshot_gates.empty:
    failure_text = ", ".join(
        f"{row.gate}: {row.evidence}"
        for row in failed_snapshot_gates.itertuples(
            index=False
        )
    )

    raise RuntimeError(
        "REST snapshot contract failed: "
        + failure_text
    )


print("REST snapshot wrapper and payload audit: PASS")
print(
    "Wrapper format: "
    f"{RESOLVED_SNAPSHOT_CONTRACT['wrapper_format']}"
)
print(
    "Book payload path: "
    f"{SNAPSHOT_BOOK_PAYLOAD_PATH}"
)
print(
    "lastUpdateId: "
    f"{SNAPSHOT_LAST_UPDATE_ID:,}"
)
print(
    "Snapshot levels: "
    f"{len(SNAPSHOT_BIDS):,} bids, "
    f"{len(SNAPSHOT_ASKS):,} asks"
)
print(
    "Top of book: "
    f"bid={best_bid}, ask={best_ask}, "
    f"spread={spread}"
)

,candidate_path,bids_key,asks_key,bids_is_list,asks_is_list,bid_level_count,ask_level_count,combined_level_count
0,$.raw_response,bids,asks,True,True,5000,5000,10000


,candidate_path,update_id_key,raw_value,parsed_value,valid_integer
0,$,lastUpdateId,97233590166,97233590166,True
1,$.raw_response,lastUpdateId,97233590166,97233590166,True


,side,level_count,valid_level_count,invalid_level_count,strict_price_ordering_valid,best_price,worst_price,total_quantity
0,BID,5000,5000,0,True,63914.36000000,62979.31000000,170.19791000
1,ASK,5000,5000,0,True,63914.37000000,64775.44000000,303.84917000


,check,declared_value,observed_value,required,status
0,bid_level_count,5000,5000,True,PASS
1,ask_level_count,5000,5000,True,PASS
2,http_status,200,200,True,PASS


,gate,passed,evidence
0,last_update_id_available,True,97233590166
1,bid_array_nonempty,True,levels=5000
2,ask_array_nonempty,True,levels=5000
3,all_bid_levels_valid,True,valid=5000/5000
4,all_ask_levels_valid,True,valid=5000/5000
5,bids_strictly_descending,True,best=63914.36000000; worst=62979.31000000
6,asks_strictly_ascending,True,best=63914.37000000; worst=64775.44000000
7,positive_top_of_book_spread,True,best_bid=63914.36000000; best_ask=63914.370000...
8,wrapper_metadata_consistent,True,bid_level_count=PASS | ask_level_count=PASS | ...


REST snapshot wrapper and payload audit: PASS
Wrapper format: COLLECTOR_METADATA_WITH_EMBEDDED_RAW_RESPONSE
Book payload path: $.raw_response
lastUpdateId: 97,233,590,166
Snapshot levels: 5,000 bids, 5,000 asks
Top of book: bid=63914.36000000, ask=63914.37000000, spread=0.01000000


In [15]:
# ============================================================
# OBSERVED RAW SCHEMA AND CANONICAL MARKET-DATA CONTRACT
# ============================================================

def read_first_jsonl_mapping(
    path: Path,
) -> tuple[int, dict]:
    """Return the first non-empty valid JSON object from a JSONL file."""
    with path.open(
        "r",
        encoding="utf-8-sig",
        errors="strict",
    ) as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue

            try:
                value = json.loads(line)
            except json.JSONDecodeError:
                continue

            if isinstance(value, dict):
                return line_number, value

    raise RuntimeError(
        f"No valid JSON object was found in {path.name}."
    )


def locate_field(
    document: dict,
    aliases: tuple[str, ...],
) -> dict:
    """
    Locate one canonical field anywhere inside a JSON wrapper.

    The first shallow matching mapping is selected. No value is
    transformed or repaired.
    """
    for mapping_path, mapping in walk_json_mappings(document):
        matched_key, matched_value = mapping_lookup(
            mapping,
            aliases,
        )

        if matched_key is not None:
            return {
                "found": True,
                "mapping_path": mapping_path,
                "source_key": str(matched_key),
                "source_path": (
                    f"{mapping_path}.{matched_key}"
                ),
                "sample_value": matched_value,
                "sample_type": type(
                    matched_value
                ).__name__,
            }

    return {
        "found": False,
        "mapping_path": None,
        "source_key": None,
        "source_path": None,
        "sample_value": None,
        "sample_type": None,
    }


TRADE_SAMPLE_LINE, TRADE_SAMPLE_DOCUMENT = (
    read_first_jsonl_mapping(TRADE_SOURCE_PATH)
)

DEPTH_SAMPLE_LINE, DEPTH_SAMPLE_DOCUMENT = (
    read_first_jsonl_mapping(DEPTH_SOURCE_PATH)
)


# ============================================================
# CANONICAL FIELD DEFINITIONS
# ============================================================

TRADE_SCHEMA_DEFINITION = [
    {
        "canonical_field": "collector_sequence",
        "aliases": (
            "collector_sequence",
            "collector_seq",
            "local_sequence",
            "sequence",
        ),
        "required": True,
        "expected_semantics": (
            "strict collector-wide causal ordering key"
        ),
    },
    {
        "canonical_field": "local_receipt_time_ns",
        "aliases": (
            "local_receipt_time_ns",
            "receipt_time_ns",
            "received_time_ns",
            "receive_time_ns",
            "recv_time_ns",
        ),
        "required": True,
        "expected_semantics": (
            "collector receipt timestamp in UTC nanoseconds"
        ),
    },
    {
        "canonical_field": "connection_session_id",
        "aliases": (
            "connection_session_id",
            "collector_session_id",
            "session_id",
        ),
        "required": True,
        "expected_semantics": (
            "collector connection-session identity"
        ),
    },
    {
        "canonical_field": "event_type",
        "aliases": (
            "event_type",
            "e",
        ),
        "required": True,
        "expected_semantics": (
            "Binance WebSocket event type"
        ),
    },
    {
        "canonical_field": "exchange_event_time_ms",
        "aliases": (
            "exchange_event_time_raw",
            "exchange_event_time_ms",
            "event_time_ms",
            "event_time",
            "E",
        ),
        "required": True,
        "expected_semantics": (
            "exchange event timestamp in UTC milliseconds"
        ),
    },
    {
        "canonical_field": "trade_time_ms",
        "aliases": (
            "trade_time_raw",
            "trade_time_ms",
            "trade_time",
            "transaction_time_ms",
            "transaction_time",
            "T",
        ),
        "required": True,
        "expected_semantics": (
            "exchange trade timestamp in UTC milliseconds"
        ),
    },
    {
        "canonical_field": "symbol",
        "aliases": (
            "symbol",
            "s",
        ),
        "required": True,
        "expected_semantics": (
            "Binance market symbol"
        ),
    },
    {
        "canonical_field": "trade_id",
        "aliases": (
            "trade_id",
            "tradeId",
            "t",
            "aggregate_trade_id",
            "agg_trade_id",
            "a",
        ),
        "required": True,
        "expected_semantics": (
            "exchange trade identifier"
        ),
    },
    {
        "canonical_field": "price",
        "aliases": (
            "price",
            "p",
        ),
        "required": True,
        "expected_semantics": (
            "executed trade price as decimal text or scalar"
        ),
    },
    {
        "canonical_field": "quantity",
        "aliases": (
            "quantity",
            "qty",
            "q",
        ),
        "required": True,
        "expected_semantics": (
            "executed base-asset quantity"
        ),
    },
    {
        "canonical_field": "buyer_is_maker",
        "aliases": (
            "buyer_is_maker",
            "is_buyer_maker",
            "m",
        ),
        "required": True,
        "expected_semantics": (
            "aggressor-side classification field"
        ),
    },
]


DEPTH_SCHEMA_DEFINITION = [
    {
        "canonical_field": "collector_sequence",
        "aliases": (
            "collector_sequence",
            "collector_seq",
            "local_sequence",
            "sequence",
        ),
        "required": True,
        "expected_semantics": (
            "strict collector-wide causal ordering key"
        ),
    },
    {
        "canonical_field": "local_receipt_time_ns",
        "aliases": (
            "local_receipt_time_ns",
            "receipt_time_ns",
            "received_time_ns",
            "receive_time_ns",
            "recv_time_ns",
        ),
        "required": True,
        "expected_semantics": (
            "collector receipt timestamp in UTC nanoseconds"
        ),
    },
    {
        "canonical_field": "connection_session_id",
        "aliases": (
            "connection_session_id",
            "collector_session_id",
            "session_id",
        ),
        "required": True,
        "expected_semantics": (
            "collector connection-session identity"
        ),
    },
    {
        "canonical_field": "event_type",
        "aliases": (
            "event_type",
            "e",
        ),
        "required": True,
        "expected_semantics": (
            "Binance differential-depth event type"
        ),
    },
    {
        "canonical_field": "exchange_event_time_ms",
        "aliases": (
            "exchange_event_time_raw",
            "exchange_event_time_ms",
            "event_time_ms",
            "event_time",
            "E",
        ),
        "required": True,
        "expected_semantics": (
            "exchange event timestamp in UTC milliseconds"
        ),
    },
    {
        "canonical_field": "symbol",
        "aliases": (
            "symbol",
            "s",
        ),
        "required": True,
        "expected_semantics": (
            "Binance market symbol"
        ),
    },
    {
        "canonical_field": "first_update_id",
        "aliases": (
            "first_update_id",
            "first_update_id_raw",
            "U",
        ),
        "required": True,
        "expected_semantics": (
            "first exchange update ID represented by the event"
        ),
    },
    {
        "canonical_field": "final_update_id",
        "aliases": (
            "final_update_id",
            "final_update_id_raw",
            "u",
        ),
        "required": True,
        "expected_semantics": (
            "final exchange update ID represented by the event"
        ),
    },
    {
        "canonical_field": "bid_updates",
        "aliases": (
            "bid_updates",
            "bids",
            "b",
        ),
        "required": True,
        "expected_semantics": (
            "market-by-price bid updates"
        ),
    },
    {
        "canonical_field": "ask_updates",
        "aliases": (
            "ask_updates",
            "asks",
            "a",
        ),
        "required": True,
        "expected_semantics": (
            "market-by-price ask updates"
        ),
    },
]


# ============================================================
# OBSERVED FIELD BINDINGS
# ============================================================

def build_schema_binding(
    stream_name: str,
    sample_line_number: int,
    sample_document: dict,
    definitions: list[dict],
) -> pd.DataFrame:
    rows = []

    for definition in definitions:
        located = locate_field(
            sample_document,
            definition["aliases"],
        )

        sample_value = located["sample_value"]

        if isinstance(sample_value, (dict, list)):
            displayed_sample = (
                f"<{type(sample_value).__name__}:"
                f"{len(sample_value)}>"
            )
        else:
            displayed_sample = sample_value

        rows.append(
            {
                "stream": stream_name,
                "sample_line_number": (
                    sample_line_number
                ),
                "canonical_field": definition[
                    "canonical_field"
                ],
                "required": bool(
                    definition["required"]
                ),
                "found": bool(located["found"]),
                "source_path": located["source_path"],
                "source_key": located["source_key"],
                "sample_type": located["sample_type"],
                "sample_value": displayed_sample,
                "expected_semantics": definition[
                    "expected_semantics"
                ],
            }
        )

    result = pd.DataFrame(rows)

    result["status"] = np.select(
        [
            result["required"] & ~result["found"],
            result["found"],
        ],
        [
            "FAIL_REQUIRED_MISSING",
            "PASS",
        ],
        default="NOT_PRESENT_OPTIONAL",
    )

    return result


TRADE_SCHEMA_BINDING = build_schema_binding(
    stream_name="TRADE_STREAM",
    sample_line_number=TRADE_SAMPLE_LINE,
    sample_document=TRADE_SAMPLE_DOCUMENT,
    definitions=TRADE_SCHEMA_DEFINITION,
)

DEPTH_SCHEMA_BINDING = build_schema_binding(
    stream_name="DEPTH_STREAM",
    sample_line_number=DEPTH_SAMPLE_LINE,
    sample_document=DEPTH_SAMPLE_DOCUMENT,
    definitions=DEPTH_SCHEMA_DEFINITION,
)

OBSERVED_STREAM_SCHEMA = pd.concat(
    [
        TRADE_SCHEMA_BINDING,
        DEPTH_SCHEMA_BINDING,
    ],
    ignore_index=True,
)


# ============================================================
# BASIC SAMPLE-LEVEL SEMANTIC CHECKS
# ============================================================

def bound_sample_value(
    binding: pd.DataFrame,
    canonical_field: str,
):
    matches = binding.loc[
        binding["canonical_field"].eq(
            canonical_field
        )
    ]

    if len(matches) != 1:
        return None

    source_path = matches.iloc[0]["source_path"]

    if source_path is None:
        return None

    definitions = (
        TRADE_SCHEMA_DEFINITION
        if binding is TRADE_SCHEMA_BINDING
        else DEPTH_SCHEMA_DEFINITION
    )

    sample_document = (
        TRADE_SAMPLE_DOCUMENT
        if binding is TRADE_SCHEMA_BINDING
        else DEPTH_SAMPLE_DOCUMENT
    )

    definition = next(
        item
        for item in definitions
        if item["canonical_field"]
        == canonical_field
    )

    return locate_field(
        sample_document,
        definition["aliases"],
    )["sample_value"]


TRADE_SAMPLE_SYMBOL = bound_sample_value(
    TRADE_SCHEMA_BINDING,
    "symbol",
)

DEPTH_SAMPLE_SYMBOL = bound_sample_value(
    DEPTH_SCHEMA_BINDING,
    "symbol",
)

TRADE_SAMPLE_SEQUENCE = safe_integer(
    bound_sample_value(
        TRADE_SCHEMA_BINDING,
        "collector_sequence",
    )
)

DEPTH_SAMPLE_SEQUENCE = safe_integer(
    bound_sample_value(
        DEPTH_SCHEMA_BINDING,
        "collector_sequence",
    )
)

DEPTH_SAMPLE_FIRST_UPDATE_ID = safe_integer(
    bound_sample_value(
        DEPTH_SCHEMA_BINDING,
        "first_update_id",
    )
)

DEPTH_SAMPLE_FINAL_UPDATE_ID = safe_integer(
    bound_sample_value(
        DEPTH_SCHEMA_BINDING,
        "final_update_id",
    )
)


# ============================================================
# FREEZE THE CANONICAL MARKET-DATA CONTRACT
# ============================================================

MARKET_DATA_CONTRACT = {
    "contract_schema_version": CONTRACT_SCHEMA_VERSION,
    "pipeline_version": PIPELINE_VERSION,
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "source_set_hash": SOURCE_SET_HASH,
    "operating_mode": V0_1_OPERATING_MODE,
    "market": {
        "venue": "BINANCE",
        "venue_segment": "SPOT",
        "symbol": "BTCUSDT",
        "base_asset": "BTC",
        "quote_asset": "USDT",
    },
    "book_representation": {
        "type": "VISIBLE_MARKET_BY_PRICE",
        "snapshot_levels_per_side": int(
            len(SNAPSHOT_BIDS)
        ),
        "queue_position_observable": False,
        "market_by_order_observable": False,
        "hidden_liquidity_observable": False,
        "snapshot_last_update_id": int(
            SNAPSHOT_LAST_UPDATE_ID
        ),
    },
    "time_contract": {
        "canonical_timezone": "UTC",
        "collector_receipt_unit": "nanoseconds",
        "exchange_event_time_unit": "milliseconds",
        "trade_time_unit": "milliseconds",
        "preserve_original_integer_fields": True,
        "timestamp_jitter_allowed": False,
    },
    "ordering_contract": {
        "primary_causal_order": (
            "collector_sequence"
        ),
        "secondary_diagnostic_time": (
            "local_receipt_time_ns"
        ),
        "exchange_timestamps_are_ordering_authority": (
            False
        ),
        "equal_timestamps_preserved": True,
        "cross_stream_ties_resolved_by": (
            "collector_sequence"
        ),
        "forward_reordering_allowed": False,
    },
    "snapshot_bridge_contract": {
        "discard_depth_events_when": (
            "final_update_id <= snapshot_last_update_id"
        ),
        "first_applied_event_requirement": (
            "first_update_id <= "
            "snapshot_last_update_id + 1 <= "
            "final_update_id"
        ),
        "subsequent_event_requirement": (
            "first_update_id == "
            "previous_final_update_id + 1"
        ),
        "gap_repair_allowed": False,
        "snapshot_activation_authority": (
            "collector receipt chronology and exchange update IDs"
        ),
    },
    "numeric_contract": {
        "price_representation": (
            "exact decimal text converted without binary-float "
            "authority"
        ),
        "quantity_representation": (
            "exact decimal text converted without binary-float "
            "authority"
        ),
        "zero_quantity_depth_update": (
            "delete the price level"
        ),
        "positive_quantity_depth_update": (
            "replace the visible quantity at the price level"
        ),
        "negative_price_or_quantity_allowed": False,
    },
    "causality_contract": {
        "trade_book_alignment": (
            "latest reconstructed book state strictly available "
            "before the trade in collector sequence"
        ),
        "forward_join_allowed": False,
        "future_label_in_feature_table_allowed": False,
        "same_sequence_cross_stream_collision_allowed": False,
        "split_intervals": "[start, end)",
    },
    "authority_contract": {
        "raw_reconstruction_authority": (
            "V0.0 PRIMARY_RAW"
        ),
        "v0_0_processed_authority": (
            "REFERENCE_ONLY"
        ),
        "v0_1_output_authority": (
            "V0.1 generated artifacts"
        ),
        "silent_reference_substitution_allowed": False,
    },
}


# ============================================================
# CONTRACT ACCEPTANCE GATES
# ============================================================

required_schema_failures = OBSERVED_STREAM_SCHEMA.loc[
    OBSERVED_STREAM_SCHEMA["status"].eq(
        "FAIL_REQUIRED_MISSING"
    )
]

MARKET_DATA_CONTRACT_GATES = pd.DataFrame(
    [
        {
            "gate": "required_trade_fields_bound",
            "passed": bool(
                ~TRADE_SCHEMA_BINDING[
                    "status"
                ].eq("FAIL_REQUIRED_MISSING").any()
            ),
            "evidence": (
                f"bound="
                f"{int(TRADE_SCHEMA_BINDING['found'].sum())}/"
                f"{len(TRADE_SCHEMA_BINDING)}"
            ),
        },
        {
            "gate": "required_depth_fields_bound",
            "passed": bool(
                ~DEPTH_SCHEMA_BINDING[
                    "status"
                ].eq("FAIL_REQUIRED_MISSING").any()
            ),
            "evidence": (
                f"bound="
                f"{int(DEPTH_SCHEMA_BINDING['found'].sum())}/"
                f"{len(DEPTH_SCHEMA_BINDING)}"
            ),
        },
        {
            "gate": "trade_symbol_matches_contract",
            "passed": (
                str(TRADE_SAMPLE_SYMBOL).upper()
                == "BTCUSDT"
            ),
            "evidence": (
                f"observed={TRADE_SAMPLE_SYMBOL}"
            ),
        },
        {
            "gate": "depth_symbol_matches_contract",
            "passed": (
                str(DEPTH_SAMPLE_SYMBOL).upper()
                == "BTCUSDT"
            ),
            "evidence": (
                f"observed={DEPTH_SAMPLE_SYMBOL}"
            ),
        },
        {
            "gate": "sample_collector_sequences_valid",
            "passed": bool(
                TRADE_SAMPLE_SEQUENCE is not None
                and TRADE_SAMPLE_SEQUENCE > 0
                and DEPTH_SAMPLE_SEQUENCE is not None
                and DEPTH_SAMPLE_SEQUENCE > 0
            ),
            "evidence": (
                f"trade={TRADE_SAMPLE_SEQUENCE}; "
                f"depth={DEPTH_SAMPLE_SEQUENCE}"
            ),
        },
        {
            "gate": "sample_depth_update_range_valid",
            "passed": bool(
                DEPTH_SAMPLE_FIRST_UPDATE_ID is not None
                and DEPTH_SAMPLE_FINAL_UPDATE_ID is not None
                and DEPTH_SAMPLE_FIRST_UPDATE_ID
                <= DEPTH_SAMPLE_FINAL_UPDATE_ID
            ),
            "evidence": (
                f"U={DEPTH_SAMPLE_FIRST_UPDATE_ID}; "
                f"u={DEPTH_SAMPLE_FINAL_UPDATE_ID}"
            ),
        },
        {
            "gate": "snapshot_contract_available",
            "passed": bool(
                SNAPSHOT_LAST_UPDATE_ID > 0
                and len(SNAPSHOT_BIDS) > 0
                and len(SNAPSHOT_ASKS) > 0
            ),
            "evidence": (
                f"lastUpdateId="
                f"{SNAPSHOT_LAST_UPDATE_ID}; "
                f"bids={len(SNAPSHOT_BIDS)}; "
                f"asks={len(SNAPSHOT_ASKS)}"
            ),
        },
        {
            "gate": "collector_sequence_is_primary_order",
            "passed": (
                MARKET_DATA_CONTRACT[
                    "ordering_contract"
                ]["primary_causal_order"]
                == "collector_sequence"
            ),
            "evidence": (
                MARKET_DATA_CONTRACT[
                    "ordering_contract"
                ]["primary_causal_order"]
            ),
        },
        {
            "gate": "forward_join_prohibited",
            "passed": bool(
                not MARKET_DATA_CONTRACT[
                    "causality_contract"
                ]["forward_join_allowed"]
            ),
            "evidence": "forward_join_allowed=False",
        },
    ]
)


MARKET_DATA_CONTRACT_SUMMARY = pd.DataFrame(
    [
        {
            "field": "venue",
            "value": "BINANCE SPOT",
        },
        {
            "field": "symbol",
            "value": "BTCUSDT",
        },
        {
            "field": "book_representation",
            "value": "VISIBLE_MARKET_BY_PRICE",
        },
        {
            "field": "snapshot_levels_per_side",
            "value": len(SNAPSHOT_BIDS),
        },
        {
            "field": "snapshot_last_update_id",
            "value": SNAPSHOT_LAST_UPDATE_ID,
        },
        {
            "field": "canonical_timezone",
            "value": "UTC",
        },
        {
            "field": "primary_causal_order",
            "value": "collector_sequence",
        },
        {
            "field": "cross_stream_tie_policy",
            "value": "collector_sequence",
        },
        {
            "field": "timestamp_jitter_allowed",
            "value": False,
        },
        {
            "field": "forward_join_allowed",
            "value": False,
        },
        {
            "field": "operating_mode",
            "value": V0_1_OPERATING_MODE,
        },
    ]
)


display(
    OBSERVED_STREAM_SCHEMA[
        [
            "stream",
            "canonical_field",
            "required",
            "source_path",
            "sample_type",
            "sample_value",
            "status",
        ]
    ]
)

display(MARKET_DATA_CONTRACT_SUMMARY)
display(MARKET_DATA_CONTRACT_GATES)


failed_market_contract_gates = (
    MARKET_DATA_CONTRACT_GATES.loc[
        ~MARKET_DATA_CONTRACT_GATES["passed"]
    ]
)

if not required_schema_failures.empty:
    missing_fields = ", ".join(
        f"{row.stream}.{row.canonical_field}"
        for row in required_schema_failures.itertuples(
            index=False
        )
    )

    raise RuntimeError(
        "Required raw schema fields are missing: "
        + missing_fields
    )

if not failed_market_contract_gates.empty:
    failure_text = ", ".join(
        f"{row.gate}: {row.evidence}"
        for row in failed_market_contract_gates.itertuples(
            index=False
        )
    )

    raise RuntimeError(
        "Canonical market-data contract failed: "
        + failure_text
    )


MARKET_DATA_CONTRACT_HASH = (
    sha256_of_canonical_object(
        MARKET_DATA_CONTRACT
    )
)

print("Canonical market-data contract: PASS")
print(
    "Observed schema bindings: "
    f"{len(OBSERVED_STREAM_SCHEMA):,}"
)
print(
    "Primary causal ordering authority: "
    "collector_sequence"
)
print(
    "Snapshot bridge anchor: "
    f"{SNAPSHOT_LAST_UPDATE_ID:,}"
)
print(
    "Market-data contract hash: "
    f"{MARKET_DATA_CONTRACT_HASH}"
)

,stream,canonical_field,required,source_path,sample_type,sample_value,status
0,TRADE_STREAM,collector_sequence,True,$.collector_sequence,int,14,PASS
1,TRADE_STREAM,local_receipt_time_ns,True,$.local_receipt_time_ns,int,1783665468766951600,PASS
2,TRADE_STREAM,connection_session_id,True,$.connection_session_id,str,c8b5bf127a7a44669514acfda8634107,PASS
3,TRADE_STREAM,event_type,True,$.event_type,str,trade,PASS
4,TRADE_STREAM,exchange_event_time_ms,True,$.exchange_event_time_raw,int,1783665469953,PASS
5,TRADE_STREAM,trade_time_ms,True,$.trade_time_raw,int,1783665469952,PASS
6,TRADE_STREAM,symbol,True,$.symbol,str,BTCUSDT,PASS
7,TRADE_STREAM,trade_id,True,$.trade_id,int,6494596041,PASS
8,TRADE_STREAM,price,True,$.raw_message.data.p,str,63914.37000000,PASS
9,TRADE_STREAM,quantity,True,$.raw_message.data.q,str,0.00046000,PASS


,field,value
0,venue,BINANCE SPOT
1,symbol,BTCUSDT
2,book_representation,VISIBLE_MARKET_BY_PRICE
3,snapshot_levels_per_side,5000
4,snapshot_last_update_id,97233590166
5,canonical_timezone,UTC
6,primary_causal_order,collector_sequence
7,cross_stream_tie_policy,collector_sequence
8,timestamp_jitter_allowed,False
9,forward_join_allowed,False


,gate,passed,evidence
0,required_trade_fields_bound,True,bound=11/11
1,required_depth_fields_bound,True,bound=10/10
2,trade_symbol_matches_contract,True,observed=BTCUSDT
3,depth_symbol_matches_contract,True,observed=BTCUSDT
4,sample_collector_sequences_valid,True,trade=14; depth=1
5,sample_depth_update_range_valid,True,U=97233590081; u=97233590085
6,snapshot_contract_available,True,lastUpdateId=97233590166; bids=5000; asks=5000
7,collector_sequence_is_primary_order,True,collector_sequence
8,forward_join_prohibited,True,forward_join_allowed=False


Canonical market-data contract: PASS
Observed schema bindings: 21
Primary causal ordering authority: collector_sequence
Snapshot bridge anchor: 97,233,590,166
Market-data contract hash: aadc573f7d891385115db3029a51b242f5c1213313b9b8b904ee6cdd34da9f15


In [16]:
# ============================================================
# MISSING-DATA, REJECTION, AND NO-SILENT-REPAIR CONTRACT
# ============================================================

ALLOWED_REJECTION_SCOPES = {
    "PIPELINE",
    "FILE",
    "STREAM",
    "RECORD",
    "ELEMENT",
    "METADATA",
    "REFERENCE",
}

ALLOWED_REJECTION_SEVERITIES = {
    "CRITICAL",
    "ERROR",
    "WARNING",
    "INFORMATIONAL",
}

ALLOWED_DISPOSITIONS = {
    "FAIL_PIPELINE",
    "FAIL_FILE",
    "FAIL_STREAM",
    "REJECT_RECORD",
    "PRESERVE_NULL_AND_WARN",
    "EXCLUDE_FROM_PRIMARY_AUTHORITY",
    "RECORD_DISCREPANCY",
    "APPLY_DEPTH_DELETE",
}


# ============================================================
# REJECTION-REASON REGISTRY
# ============================================================

REJECTION_REASON_ROWS = [
    # --------------------------------------------------------
    # PIPELINE-INTEGRITY FAILURES
    # --------------------------------------------------------
    {
        "reason_code": "V0_0_MODIFICATION_ATTEMPT",
        "stage": "ALL",
        "scope": "PIPELINE",
        "severity": "CRITICAL",
        "condition": (
            "Any attempted write, rename, deletion, move, or "
            "in-place alteration under the V0.0 root."
        ),
        "disposition": "FAIL_PIPELINE",
        "downstream_effect": "STOP_ALL_DOWNSTREAM_EXECUTION",
        "retention_rule": "PRESERVE_ALL_EVIDENCE",
    },
    {
        "reason_code": "UNDECLARED_REPAIR_ATTEMPT",
        "stage": "ALL",
        "scope": "PIPELINE",
        "severity": "CRITICAL",
        "condition": (
            "A record, timestamp, sequence, price, quantity, or "
            "payload is modified without an explicit contract rule."
        ),
        "disposition": "FAIL_PIPELINE",
        "downstream_effect": "STOP_ALL_DOWNSTREAM_EXECUTION",
        "retention_rule": "PRESERVE_ORIGINAL_AND_ATTEMPT_DETAILS",
    },
    {
        "reason_code": "TIMESTAMP_SYNTHESIS_ATTEMPT",
        "stage": "ALL",
        "scope": "PIPELINE",
        "severity": "CRITICAL",
        "condition": (
            "Missing or tied timestamps are replaced with invented "
            "values, arbitrary jitter, interpolation, or offsets."
        ),
        "disposition": "FAIL_PIPELINE",
        "downstream_effect": "STOP_ALL_DOWNSTREAM_EXECUTION",
        "retention_rule": "PRESERVE_ORIGINAL_TIMESTAMP_FIELDS",
    },
    {
        "reason_code": "FORWARD_LOOKING_JOIN_ATTEMPT",
        "stage": "ALIGNMENT_OR_FEATURES",
        "scope": "PIPELINE",
        "severity": "CRITICAL",
        "condition": (
            "A trade, event, or feature is aligned to a book state "
            "that was not causally available beforehand."
        ),
        "disposition": "FAIL_PIPELINE",
        "downstream_effect": "INVALIDATE_AFFECTED_OUTPUTS",
        "retention_rule": "PRESERVE_JOIN_AUDIT_EVIDENCE",
    },

    # --------------------------------------------------------
    # RAW-FILE FAILURES
    # --------------------------------------------------------
    {
        "reason_code": "RAW_FILE_UNREADABLE",
        "stage": "INGESTION",
        "scope": "FILE",
        "severity": "CRITICAL",
        "condition": (
            "An authoritative raw file cannot be opened or read "
            "completely."
        ),
        "disposition": "FAIL_FILE",
        "downstream_effect": "BLOCK_DEPENDENT_NOTEBOOKS",
        "retention_rule": "RETAIN_FILE_PATH_AND_EXCEPTION",
    },
    {
        "reason_code": "RAW_FILE_HASH_MISMATCH",
        "stage": "INGESTION",
        "scope": "FILE",
        "severity": "CRITICAL",
        "condition": (
            "A computed SHA-256 digest differs from the registered "
            "or declared digest."
        ),
        "disposition": "FAIL_FILE",
        "downstream_effect": "BLOCK_DEPENDENT_NOTEBOOKS",
        "retention_rule": "RETAIN_BOTH_HASH_VALUES",
    },
    {
        "reason_code": "PRIMARY_SOURCE_ROLE_AMBIGUOUS",
        "stage": "INGESTION",
        "scope": "FILE",
        "severity": "CRITICAL",
        "condition": (
            "More than one eligible file competes for one required "
            "primary source role."
        ),
        "disposition": "FAIL_FILE",
        "downstream_effect": "BLOCK_SOURCE_SELECTION",
        "retention_rule": "RETAIN_ALL_CANDIDATES",
    },

    # --------------------------------------------------------
    # COLLECTOR-ORDERING FAILURES
    # --------------------------------------------------------
    {
        "reason_code": "COLLECTOR_SEQUENCE_MISSING",
        "stage": "RAW_AUDIT",
        "scope": "STREAM",
        "severity": "CRITICAL",
        "condition": (
            "A trade or depth record lacks a valid collector "
            "sequence."
        ),
        "disposition": "FAIL_STREAM",
        "downstream_effect": "CAUSAL_ORDER_UNAVAILABLE",
        "retention_rule": "RETAIN_RECORD_AND_SOURCE_LINE",
    },
    {
        "reason_code": "COLLECTOR_SEQUENCE_DUPLICATE",
        "stage": "RAW_AUDIT",
        "scope": "STREAM",
        "severity": "CRITICAL",
        "condition": (
            "Two raw records share the same collector sequence."
        ),
        "disposition": "FAIL_STREAM",
        "downstream_effect": "CAUSAL_ORDER_AMBIGUOUS",
        "retention_rule": "RETAIN_ALL_COLLIDING_RECORDS",
    },
    {
        "reason_code": "COLLECTOR_SEQUENCE_NONMONOTONE",
        "stage": "RAW_AUDIT",
        "scope": "STREAM",
        "severity": "CRITICAL",
        "condition": (
            "Collector sequence decreases in physical source order."
        ),
        "disposition": "FAIL_STREAM",
        "downstream_effect": "CAUSAL_ORDER_INVALID",
        "retention_rule": "RETAIN_VIOLATING_ADJACENT_ROWS",
    },
    {
        "reason_code": "COLLECTOR_SEQUENCE_GAP",
        "stage": "RAW_AUDIT",
        "scope": "STREAM",
        "severity": "CRITICAL",
        "condition": (
            "The combined authoritative trade/depth stream omits "
            "one or more collector sequence values."
        ),
        "disposition": "FAIL_STREAM",
        "downstream_effect": "COLLECTION_CONTINUITY_UNPROVEN",
        "retention_rule": "RETAIN_MISSING_SEQUENCE_RANGES",
    },
    {
        "reason_code": "CONNECTION_SESSION_MISMATCH",
        "stage": "RAW_AUDIT",
        "scope": "STREAM",
        "severity": "CRITICAL",
        "condition": (
            "A primary record belongs to a connection session other "
            "than the registered authoritative session."
        ),
        "disposition": "FAIL_STREAM",
        "downstream_effect": "SOURCE_CONTAMINATION",
        "retention_rule": "RETAIN_OBSERVED_SESSION_IDENTITIES",
    },

    # --------------------------------------------------------
    # TRADE-STREAM REJECTIONS
    # --------------------------------------------------------
    {
        "reason_code": "TRADE_INVALID_JSON",
        "stage": "TRADE_INGESTION",
        "scope": "RECORD",
        "severity": "ERROR",
        "condition": "A non-empty trade JSONL line is invalid JSON.",
        "disposition": "REJECT_RECORD",
        "downstream_effect": "EXCLUDE_FROM_TRADE_DATASET",
        "retention_rule": "RETAIN_LINE_NUMBER_AND_RAW_TEXT_HASH",
    },
    {
        "reason_code": "TRADE_NON_OBJECT_RECORD",
        "stage": "TRADE_INGESTION",
        "scope": "RECORD",
        "severity": "ERROR",
        "condition": (
            "A decoded trade JSONL value is not a JSON mapping."
        ),
        "disposition": "REJECT_RECORD",
        "downstream_effect": "EXCLUDE_FROM_TRADE_DATASET",
        "retention_rule": "RETAIN_LINE_NUMBER_AND_VALUE_TYPE",
    },
    {
        "reason_code": "TRADE_REQUIRED_FIELD_MISSING",
        "stage": "TRADE_INGESTION",
        "scope": "RECORD",
        "severity": "ERROR",
        "condition": (
            "A required canonical trade field cannot be bound."
        ),
        "disposition": "REJECT_RECORD",
        "downstream_effect": "EXCLUDE_FROM_TRADE_DATASET",
        "retention_rule": "RETAIN_MISSING_FIELD_NAMES",
    },
    {
        "reason_code": "TRADE_EVENT_TYPE_INVALID",
        "stage": "TRADE_INGESTION",
        "scope": "RECORD",
        "severity": "ERROR",
        "condition": (
            "The event type is not the contracted Binance trade "
            "event type."
        ),
        "disposition": "REJECT_RECORD",
        "downstream_effect": "EXCLUDE_FROM_TRADE_DATASET",
        "retention_rule": "RETAIN_OBSERVED_EVENT_TYPE",
    },
    {
        "reason_code": "TRADE_SYMBOL_MISMATCH",
        "stage": "TRADE_INGESTION",
        "scope": "RECORD",
        "severity": "ERROR",
        "condition": "The record symbol is not BTCUSDT.",
        "disposition": "REJECT_RECORD",
        "downstream_effect": "EXCLUDE_FROM_TRADE_DATASET",
        "retention_rule": "RETAIN_OBSERVED_SYMBOL",
    },
    {
        "reason_code": "TRADE_ID_INVALID",
        "stage": "TRADE_INGESTION",
        "scope": "RECORD",
        "severity": "ERROR",
        "condition": (
            "The exchange trade identifier is absent, non-integer, "
            "or negative."
        ),
        "disposition": "REJECT_RECORD",
        "downstream_effect": "EXCLUDE_FROM_TRADE_DATASET",
        "retention_rule": "RETAIN_RAW_TRADE_ID",
    },
    {
        "reason_code": "TRADE_ID_EXACT_DUPLICATE",
        "stage": "TRADE_INGESTION",
        "scope": "RECORD",
        "severity": "WARNING",
        "condition": (
            "The same trade ID appears with identical economic and "
            "exchange-time fields."
        ),
        "disposition": "REJECT_RECORD",
        "downstream_effect": "RETAIN_LOWEST_COLLECTOR_SEQUENCE_ONLY",
        "retention_rule": "RETAIN_ALL_DUPLICATE_SEQUENCE_VALUES",
    },
    {
        "reason_code": "TRADE_ID_CONFLICT",
        "stage": "TRADE_INGESTION",
        "scope": "STREAM",
        "severity": "CRITICAL",
        "condition": (
            "The same trade ID appears with conflicting price, "
            "quantity, side, or exchange-time values."
        ),
        "disposition": "FAIL_STREAM",
        "downstream_effect": "TRADE_IDENTITY_AMBIGUOUS",
        "retention_rule": "RETAIN_ALL_CONFLICTING_RECORDS",
    },
    {
        "reason_code": "TRADE_PRICE_INVALID",
        "stage": "TRADE_INGESTION",
        "scope": "RECORD",
        "severity": "ERROR",
        "condition": (
            "Trade price is missing, non-finite, non-decimal, or "
            "not strictly positive."
        ),
        "disposition": "REJECT_RECORD",
        "downstream_effect": "EXCLUDE_FROM_TRADE_DATASET",
        "retention_rule": "RETAIN_RAW_PRICE",
    },
    {
        "reason_code": "TRADE_QUANTITY_INVALID",
        "stage": "TRADE_INGESTION",
        "scope": "RECORD",
        "severity": "ERROR",
        "condition": (
            "Trade quantity is missing, non-finite, non-decimal, or "
            "not strictly positive."
        ),
        "disposition": "REJECT_RECORD",
        "downstream_effect": "EXCLUDE_FROM_TRADE_DATASET",
        "retention_rule": "RETAIN_RAW_QUANTITY",
    },
    {
        "reason_code": "TRADE_AGGRESSOR_FLAG_INVALID",
        "stage": "TRADE_INGESTION",
        "scope": "RECORD",
        "severity": "ERROR",
        "condition": (
            "buyer_is_maker is missing or is not a Boolean value."
        ),
        "disposition": "REJECT_RECORD",
        "downstream_effect": "SIDE_CLASSIFICATION_UNAVAILABLE",
        "retention_rule": "RETAIN_RAW_FLAG",
    },
    {
        "reason_code": "TRADE_TIMESTAMP_INVALID",
        "stage": "TRADE_INGESTION",
        "scope": "RECORD",
        "severity": "ERROR",
        "condition": (
            "Required trade exchange or receipt timestamps are "
            "missing, non-integer, or outside representable range."
        ),
        "disposition": "REJECT_RECORD",
        "downstream_effect": "EXCLUDE_FROM_TRADE_DATASET",
        "retention_rule": "RETAIN_ALL_RAW_TIMESTAMP_FIELDS",
    },

    # --------------------------------------------------------
    # DEPTH-STREAM FAILURES
    # --------------------------------------------------------
    {
        "reason_code": "DEPTH_INVALID_JSON",
        "stage": "DEPTH_INGESTION",
        "scope": "STREAM",
        "severity": "CRITICAL",
        "condition": "A non-empty depth JSONL line is invalid JSON.",
        "disposition": "FAIL_STREAM",
        "downstream_effect": "BOOK_RECONSTRUCTION_UNSAFE",
        "retention_rule": "RETAIN_LINE_NUMBER_AND_RAW_TEXT_HASH",
    },
    {
        "reason_code": "DEPTH_NON_OBJECT_RECORD",
        "stage": "DEPTH_INGESTION",
        "scope": "STREAM",
        "severity": "CRITICAL",
        "condition": (
            "A decoded depth JSONL value is not a JSON mapping."
        ),
        "disposition": "FAIL_STREAM",
        "downstream_effect": "BOOK_RECONSTRUCTION_UNSAFE",
        "retention_rule": "RETAIN_LINE_NUMBER_AND_VALUE_TYPE",
    },
    {
        "reason_code": "DEPTH_REQUIRED_FIELD_MISSING",
        "stage": "DEPTH_INGESTION",
        "scope": "STREAM",
        "severity": "CRITICAL",
        "condition": (
            "A required canonical differential-depth field cannot "
            "be bound."
        ),
        "disposition": "FAIL_STREAM",
        "downstream_effect": "BOOK_RECONSTRUCTION_UNSAFE",
        "retention_rule": "RETAIN_MISSING_FIELD_NAMES",
    },
    {
        "reason_code": "DEPTH_EVENT_TYPE_INVALID",
        "stage": "DEPTH_INGESTION",
        "scope": "STREAM",
        "severity": "CRITICAL",
        "condition": (
            "The event type is not Binance depthUpdate."
        ),
        "disposition": "FAIL_STREAM",
        "downstream_effect": "SOURCE_ROLE_CONTAMINATED",
        "retention_rule": "RETAIN_OBSERVED_EVENT_TYPE",
    },
    {
        "reason_code": "DEPTH_SYMBOL_MISMATCH",
        "stage": "DEPTH_INGESTION",
        "scope": "STREAM",
        "severity": "CRITICAL",
        "condition": "The depth record symbol is not BTCUSDT.",
        "disposition": "FAIL_STREAM",
        "downstream_effect": "SOURCE_ROLE_CONTAMINATED",
        "retention_rule": "RETAIN_OBSERVED_SYMBOL",
    },
    {
        "reason_code": "DEPTH_UPDATE_ID_INVALID",
        "stage": "DEPTH_INGESTION",
        "scope": "STREAM",
        "severity": "CRITICAL",
        "condition": (
            "First or final update ID is missing, non-integer, or "
            "negative."
        ),
        "disposition": "FAIL_STREAM",
        "downstream_effect": "EXCHANGE_SEQUENCE_UNAVAILABLE",
        "retention_rule": "RETAIN_RAW_UPDATE_IDS",
    },
    {
        "reason_code": "DEPTH_UPDATE_RANGE_INVALID",
        "stage": "DEPTH_INGESTION",
        "scope": "STREAM",
        "severity": "CRITICAL",
        "condition": (
            "The first update ID is greater than the final update "
            "ID."
        ),
        "disposition": "FAIL_STREAM",
        "downstream_effect": "EXCHANGE_SEQUENCE_INVALID",
        "retention_rule": "RETAIN_RAW_UPDATE_IDS",
    },
    {
        "reason_code": "DEPTH_SNAPSHOT_BRIDGE_FAILURE",
        "stage": "BOOK_RECONSTRUCTION",
        "scope": "STREAM",
        "severity": "CRITICAL",
        "condition": (
            "No first post-snapshot event satisfies "
            "U <= lastUpdateId + 1 <= u."
        ),
        "disposition": "FAIL_STREAM",
        "downstream_effect": "BOOK_ACTIVATION_UNPROVEN",
        "retention_rule": "RETAIN_BRIDGE_CANDIDATE_TABLE",
    },
    {
        "reason_code": "DEPTH_SEQUENCE_GAP",
        "stage": "BOOK_RECONSTRUCTION",
        "scope": "STREAM",
        "severity": "CRITICAL",
        "condition": (
            "A subsequent applied event does not begin at the "
            "required successor update ID."
        ),
        "disposition": "FAIL_STREAM",
        "downstream_effect": "BOOK_STATE_INVALID_AFTER_GAP",
        "retention_rule": "RETAIN_PREVIOUS_AND_CURRENT_UPDATE_IDS",
    },
    {
        "reason_code": "DEPTH_LEVEL_MALFORMED",
        "stage": "DEPTH_INGESTION",
        "scope": "ELEMENT",
        "severity": "CRITICAL",
        "condition": (
            "A depth level is not exactly a two-field "
            "[price, quantity] sequence."
        ),
        "disposition": "FAIL_STREAM",
        "downstream_effect": "BOOK_RECONSTRUCTION_UNSAFE",
        "retention_rule": "RETAIN_EVENT_AND_LEVEL_INDEX",
    },
    {
        "reason_code": "DEPTH_PRICE_INVALID",
        "stage": "DEPTH_INGESTION",
        "scope": "ELEMENT",
        "severity": "CRITICAL",
        "condition": (
            "A depth-update price is non-decimal, non-finite, or "
            "not strictly positive."
        ),
        "disposition": "FAIL_STREAM",
        "downstream_effect": "BOOK_RECONSTRUCTION_UNSAFE",
        "retention_rule": "RETAIN_RAW_PRICE",
    },
    {
        "reason_code": "DEPTH_QUANTITY_NEGATIVE_OR_INVALID",
        "stage": "DEPTH_INGESTION",
        "scope": "ELEMENT",
        "severity": "CRITICAL",
        "condition": (
            "A depth-update quantity is non-decimal, non-finite, or "
            "negative."
        ),
        "disposition": "FAIL_STREAM",
        "downstream_effect": "BOOK_RECONSTRUCTION_UNSAFE",
        "retention_rule": "RETAIN_RAW_QUANTITY",
    },
    {
        "reason_code": "DEPTH_ZERO_QUANTITY_DELETE",
        "stage": "BOOK_RECONSTRUCTION",
        "scope": "ELEMENT",
        "severity": "INFORMATIONAL",
        "condition": (
            "A valid depth update contains quantity exactly equal "
            "to zero."
        ),
        "disposition": "APPLY_DEPTH_DELETE",
        "downstream_effect": "REMOVE_PRICE_LEVEL_IF_PRESENT",
        "retention_rule": "RECORD_DELETE_OPERATION",
    },
    {
        "reason_code": "DEPTH_DUPLICATE_PRICE_IN_EVENT",
        "stage": "DEPTH_INGESTION",
        "scope": "STREAM",
        "severity": "CRITICAL",
        "condition": (
            "One differential-depth event contains the same price "
            "more than once on the same side."
        ),
        "disposition": "FAIL_STREAM",
        "downstream_effect": "WITHIN_EVENT_ORDER_AMBIGUOUS",
        "retention_rule": "RETAIN_DUPLICATE_LEVELS",
    },
    {
        "reason_code": "DEPTH_BOTH_SIDES_EMPTY",
        "stage": "DEPTH_INGESTION",
        "scope": "STREAM",
        "severity": "CRITICAL",
        "condition": (
            "Both bid-update and ask-update arrays are empty."
        ),
        "disposition": "FAIL_STREAM",
        "downstream_effect": "EMPTY_MARKET_EVENT_INVALID",
        "retention_rule": "RETAIN_EVENT",
    },

    # --------------------------------------------------------
    # SNAPSHOT FAILURES
    # --------------------------------------------------------
    {
        "reason_code": "SNAPSHOT_PAYLOAD_MISSING",
        "stage": "SNAPSHOT_INGESTION",
        "scope": "FILE",
        "severity": "CRITICAL",
        "condition": (
            "No valid non-empty bid and ask arrays are found in the "
            "snapshot wrapper or embedded raw response."
        ),
        "disposition": "FAIL_FILE",
        "downstream_effect": "BOOK_RECONSTRUCTION_BLOCKED",
        "retention_rule": "RETAIN_WRAPPER_STRUCTURE_REPORT",
    },
    {
        "reason_code": "SNAPSHOT_UPDATE_ID_INVALID",
        "stage": "SNAPSHOT_INGESTION",
        "scope": "FILE",
        "severity": "CRITICAL",
        "condition": (
            "No unique valid integer lastUpdateId is available."
        ),
        "disposition": "FAIL_FILE",
        "downstream_effect": "BOOK_RECONSTRUCTION_BLOCKED",
        "retention_rule": "RETAIN_ALL_CANDIDATE_VALUES",
    },
    {
        "reason_code": "SNAPSHOT_LEVEL_MALFORMED",
        "stage": "SNAPSHOT_INGESTION",
        "scope": "ELEMENT",
        "severity": "CRITICAL",
        "condition": (
            "A snapshot level is not exactly a two-field "
            "[price, quantity] sequence."
        ),
        "disposition": "FAIL_FILE",
        "downstream_effect": "BOOK_RECONSTRUCTION_BLOCKED",
        "retention_rule": "RETAIN_SIDE_AND_LEVEL_INDEX",
    },
    {
        "reason_code": "SNAPSHOT_PRICE_INVALID",
        "stage": "SNAPSHOT_INGESTION",
        "scope": "ELEMENT",
        "severity": "CRITICAL",
        "condition": (
            "A snapshot price is non-decimal, non-finite, or not "
            "strictly positive."
        ),
        "disposition": "FAIL_FILE",
        "downstream_effect": "BOOK_RECONSTRUCTION_BLOCKED",
        "retention_rule": "RETAIN_RAW_PRICE",
    },
    {
        "reason_code": "SNAPSHOT_QUANTITY_INVALID",
        "stage": "SNAPSHOT_INGESTION",
        "scope": "ELEMENT",
        "severity": "CRITICAL",
        "condition": (
            "A snapshot quantity is non-decimal, non-finite, or not "
            "strictly positive."
        ),
        "disposition": "FAIL_FILE",
        "downstream_effect": "BOOK_RECONSTRUCTION_BLOCKED",
        "retention_rule": "RETAIN_RAW_QUANTITY",
    },
    {
        "reason_code": "SNAPSHOT_PRICE_DUPLICATE",
        "stage": "SNAPSHOT_INGESTION",
        "scope": "FILE",
        "severity": "CRITICAL",
        "condition": (
            "A price appears more than once on one snapshot side."
        ),
        "disposition": "FAIL_FILE",
        "downstream_effect": "INITIAL_BOOK_AMBIGUOUS",
        "retention_rule": "RETAIN_DUPLICATE_PRICE_ROWS",
    },
    {
        "reason_code": "SNAPSHOT_ORDER_INVALID",
        "stage": "SNAPSHOT_INGESTION",
        "scope": "FILE",
        "severity": "CRITICAL",
        "condition": (
            "Bids are not strictly descending or asks are not "
            "strictly ascending."
        ),
        "disposition": "FAIL_FILE",
        "downstream_effect": "INITIAL_BOOK_INVALID",
        "retention_rule": "RETAIN_ORDERING_VIOLATIONS",
    },
    {
        "reason_code": "SNAPSHOT_CROSSED_OR_LOCKED",
        "stage": "SNAPSHOT_INGESTION",
        "scope": "FILE",
        "severity": "CRITICAL",
        "condition": (
            "The best bid is greater than or equal to the best ask."
        ),
        "disposition": "FAIL_FILE",
        "downstream_effect": "INITIAL_BOOK_INVALID",
        "retention_rule": "RETAIN_TOP_OF_BOOK_VALUES",
    },

    # --------------------------------------------------------
    # MISSING OPTIONAL METADATA AND REFERENCE DIFFERENCES
    # --------------------------------------------------------
    {
        "reason_code": "OPTIONAL_METADATA_MISSING",
        "stage": "ANY_INGESTION",
        "scope": "METADATA",
        "severity": "WARNING",
        "condition": (
            "A field explicitly designated optional is absent."
        ),
        "disposition": "PRESERVE_NULL_AND_WARN",
        "downstream_effect": "NO_IMPUTATION",
        "retention_rule": "RETAIN_FIELD_NAME_AND_SOURCE",
    },
    {
        "reason_code": "UNCLASSIFIED_RAW_SUPPORT_FILE",
        "stage": "SOURCE_REGISTRATION",
        "scope": "FILE",
        "severity": "WARNING",
        "condition": (
            "A non-primary raw-directory file cannot be classified."
        ),
        "disposition": "EXCLUDE_FROM_PRIMARY_AUTHORITY",
        "downstream_effect": "RETAIN_FOR_MANUAL_REVIEW",
        "retention_rule": "RETAIN_PATH_HASH_AND_SIZE",
    },
    {
        "reason_code": "V0_0_REFERENCE_DISCREPANCY",
        "stage": "RECONCILIATION",
        "scope": "REFERENCE",
        "severity": "WARNING",
        "condition": (
            "A V0.1 reconstruction output differs from a V0.0 "
            "reference artifact."
        ),
        "disposition": "RECORD_DISCREPANCY",
        "downstream_effect": "REQUIRE_EXPLANATION_NOT_REPAIR",
        "retention_rule": "RETAIN_BOTH_VALUES_AND_COMPARISON_RULE",
    },
]


REJECTION_REASON_REGISTRY = pd.DataFrame(
    REJECTION_REASON_ROWS
)


# ============================================================
# FROZEN MISSING-DATA AND REJECTION POLICY
# ============================================================

MISSING_DATA_REJECTION_CONTRACT = {
    "contract_schema_version": CONTRACT_SCHEMA_VERSION,
    "pipeline_version": PIPELINE_VERSION,
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "source_set_hash": SOURCE_SET_HASH,
    "market_data_contract_hash": MARKET_DATA_CONTRACT_HASH,
    "operating_mode": V0_1_OPERATING_MODE,
    "missing_data_policy": {
        "numeric_imputation_allowed": False,
        "categorical_imputation_allowed": False,
        "forward_fill_allowed": False,
        "backward_fill_allowed": False,
        "interpolation_allowed": False,
        "timestamp_synthesis_allowed": False,
        "timestamp_jitter_allowed": False,
        "sequence_synthesis_allowed": False,
        "price_or_quantity_coercion_allowed": False,
        "optional_metadata_may_remain_null": True,
        "required_trade_field_missing_action": "REJECT_RECORD",
        "required_depth_field_missing_action": "FAIL_STREAM",
        "required_snapshot_field_missing_action": "FAIL_FILE",
    },
    "trade_rejection_policy": {
        "record_level_rejection_allowed": True,
        "rejected_rows_must_be_saved": True,
        "exact_trade_duplicate_retention_rule": (
            "retain the record with the lowest collector_sequence"
        ),
        "conflicting_trade_duplicate_action": "FAIL_STREAM",
        "zero_or_negative_trade_quantity_allowed": False,
        "zero_or_negative_trade_price_allowed": False,
    },
    "depth_rejection_policy": {
        "dropping_invalid_depth_events_allowed": False,
        "dropping_invalid_depth_levels_allowed": False,
        "one_empty_side_array_allowed": True,
        "both_side_arrays_empty_allowed": False,
        "zero_quantity_is_missing": False,
        "zero_quantity_semantics": "DELETE_PRICE_LEVEL",
        "negative_quantity_allowed": False,
        "duplicate_price_within_event_allowed": False,
        "sequence_gap_repair_allowed": False,
        "snapshot_bridge_repair_allowed": False,
    },
    "snapshot_policy": {
        "partial_snapshot_acceptance_allowed": False,
        "sorting_snapshot_levels_allowed": False,
        "deduplicating_snapshot_prices_allowed": False,
        "dropping_invalid_snapshot_levels_allowed": False,
        "locked_or_crossed_snapshot_allowed": False,
    },
    "reference_policy": {
        "v0_0_processed_files_are_primary_authority": False,
        "reference_difference_is_automatically_an_error": False,
        "reference_difference_must_be_recorded": True,
        "reference_difference_may_be_silently_repaired": False,
    },
    "status_escalation": {
        "any_FAIL_PIPELINE": "FAIL",
        "any_FAIL_FILE": "FAIL",
        "any_FAIL_STREAM": "FAIL",
        "trade_record_rejections_without_causal_failure": "WARNING",
        "optional_metadata_missing_only": "WARNING",
        "no_failures_rejections_or_warnings": "PASS",
    },
    "required_rejection_ledger_columns": [
        "v0_1_run_id",
        "source_run_prefix",
        "producing_notebook",
        "source_role",
        "source_path",
        "source_sha256",
        "source_line_number",
        "collector_sequence",
        "exchange_identifier",
        "reason_code",
        "scope",
        "severity",
        "disposition",
        "raw_record_hash",
        "details",
    ],
    "required_count_ledger_columns": [
        "v0_1_run_id",
        "producing_notebook",
        "dataset_name",
        "stage_name",
        "input_row_count",
        "accepted_row_count",
        "rejected_row_count",
        "output_row_count",
        "count_identity_passed",
    ],
    "count_identity_rule": (
        "input_row_count == accepted_row_count + rejected_row_count"
    ),
    "registry_records": (
        REJECTION_REASON_REGISTRY
        .sort_values("reason_code", kind="stable")
        .to_dict(orient="records")
    ),
    "producing_notebook": NOTEBOOK_FILENAME,
}


# ============================================================
# CONTRACT VALIDATION
# ============================================================

duplicate_reason_codes = REJECTION_REASON_REGISTRY.loc[
    REJECTION_REASON_REGISTRY[
        "reason_code"
    ].duplicated(keep=False)
]

invalid_scopes = REJECTION_REASON_REGISTRY.loc[
    ~REJECTION_REASON_REGISTRY[
        "scope"
    ].isin(ALLOWED_REJECTION_SCOPES)
]

invalid_severities = REJECTION_REASON_REGISTRY.loc[
    ~REJECTION_REASON_REGISTRY[
        "severity"
    ].isin(ALLOWED_REJECTION_SEVERITIES)
]

invalid_dispositions = REJECTION_REASON_REGISTRY.loc[
    ~REJECTION_REASON_REGISTRY[
        "disposition"
    ].isin(ALLOWED_DISPOSITIONS)
]

empty_required_registry_fields = (
    REJECTION_REASON_REGISTRY.loc[
        REJECTION_REASON_REGISTRY[
            [
                "reason_code",
                "stage",
                "scope",
                "severity",
                "condition",
                "disposition",
                "downstream_effect",
                "retention_rule",
            ]
        ]
        .isna()
        .any(axis=1)
    ]
)


forbidden_operation_checks = {
    "numeric_imputation_prohibited": (
        not MISSING_DATA_REJECTION_CONTRACT[
            "missing_data_policy"
        ]["numeric_imputation_allowed"]
    ),
    "forward_fill_prohibited": (
        not MISSING_DATA_REJECTION_CONTRACT[
            "missing_data_policy"
        ]["forward_fill_allowed"]
    ),
    "backward_fill_prohibited": (
        not MISSING_DATA_REJECTION_CONTRACT[
            "missing_data_policy"
        ]["backward_fill_allowed"]
    ),
    "interpolation_prohibited": (
        not MISSING_DATA_REJECTION_CONTRACT[
            "missing_data_policy"
        ]["interpolation_allowed"]
    ),
    "timestamp_synthesis_prohibited": (
        not MISSING_DATA_REJECTION_CONTRACT[
            "missing_data_policy"
        ]["timestamp_synthesis_allowed"]
    ),
    "timestamp_jitter_prohibited": (
        not MISSING_DATA_REJECTION_CONTRACT[
            "missing_data_policy"
        ]["timestamp_jitter_allowed"]
    ),
    "depth_event_dropping_prohibited": (
        not MISSING_DATA_REJECTION_CONTRACT[
            "depth_rejection_policy"
        ]["dropping_invalid_depth_events_allowed"]
    ),
    "snapshot_sorting_prohibited": (
        not MISSING_DATA_REJECTION_CONTRACT[
            "snapshot_policy"
        ]["sorting_snapshot_levels_allowed"]
    ),
    "reference_repair_prohibited": (
        not MISSING_DATA_REJECTION_CONTRACT[
            "reference_policy"
        ]["reference_difference_may_be_silently_repaired"]
    ),
}


MISSING_DATA_REJECTION_GATES = pd.DataFrame(
    [
        {
            "gate": "reason_codes_unique",
            "passed": duplicate_reason_codes.empty,
            "evidence": (
                f"rows={len(REJECTION_REASON_REGISTRY)}; "
                f"unique_codes="
                f"{REJECTION_REASON_REGISTRY['reason_code'].nunique()}"
            ),
        },
        {
            "gate": "scopes_valid",
            "passed": invalid_scopes.empty,
            "evidence": (
                "invalid="
                f"{len(invalid_scopes)}"
            ),
        },
        {
            "gate": "severities_valid",
            "passed": invalid_severities.empty,
            "evidence": (
                "invalid="
                f"{len(invalid_severities)}"
            ),
        },
        {
            "gate": "dispositions_valid",
            "passed": invalid_dispositions.empty,
            "evidence": (
                "invalid="
                f"{len(invalid_dispositions)}"
            ),
        },
        {
            "gate": "registry_fields_complete",
            "passed": empty_required_registry_fields.empty,
            "evidence": (
                "incomplete_rows="
                f"{len(empty_required_registry_fields)}"
            ),
        },
        {
            "gate": "all_forbidden_operations_disabled",
            "passed": all(
                forbidden_operation_checks.values()
            ),
            "evidence": (
                " | ".join(
                    f"{name}={passed}"
                    for name, passed
                    in forbidden_operation_checks.items()
                )
            ),
        },
        {
            "gate": "zero_depth_quantity_is_delete",
            "passed": (
                MISSING_DATA_REJECTION_CONTRACT[
                    "depth_rejection_policy"
                ]["zero_quantity_semantics"]
                == "DELETE_PRICE_LEVEL"
            ),
            "evidence": (
                MISSING_DATA_REJECTION_CONTRACT[
                    "depth_rejection_policy"
                ]["zero_quantity_semantics"]
            ),
        },
        {
            "gate": "invalid_depth_events_are_not_dropped",
            "passed": (
                MISSING_DATA_REJECTION_CONTRACT[
                    "depth_rejection_policy"
                ]["dropping_invalid_depth_events_allowed"]
                is False
            ),
            "evidence": "dropping_invalid_depth_events_allowed=False",
        },
        {
            "gate": "trade_rejections_require_ledger",
            "passed": bool(
                MISSING_DATA_REJECTION_CONTRACT[
                    "trade_rejection_policy"
                ]["rejected_rows_must_be_saved"]
            ),
            "evidence": "rejected_rows_must_be_saved=True",
        },
    ]
)


MISSING_DATA_REJECTION_SUMMARY = pd.DataFrame(
    [
        {
            "field": "registered_reason_codes",
            "value": int(
                len(REJECTION_REASON_REGISTRY)
            ),
        },
        {
            "field": "critical_reason_codes",
            "value": int(
                REJECTION_REASON_REGISTRY[
                    "severity"
                ].eq("CRITICAL").sum()
            ),
        },
        {
            "field": "record_rejection_codes",
            "value": int(
                REJECTION_REASON_REGISTRY[
                    "disposition"
                ].eq("REJECT_RECORD").sum()
            ),
        },
        {
            "field": "imputation_allowed",
            "value": False,
        },
        {
            "field": "timestamp_jitter_allowed",
            "value": False,
        },
        {
            "field": "invalid_depth_event_drop_allowed",
            "value": False,
        },
        {
            "field": "zero_depth_quantity_semantics",
            "value": "DELETE_PRICE_LEVEL",
        },
        {
            "field": "snapshot_partial_acceptance",
            "value": False,
        },
        {
            "field": "reference_silent_repair_allowed",
            "value": False,
        },
    ]
)


display(
    REJECTION_REASON_REGISTRY[
        [
            "reason_code",
            "stage",
            "scope",
            "severity",
            "disposition",
            "downstream_effect",
        ]
    ]
)

display(MISSING_DATA_REJECTION_SUMMARY)
display(MISSING_DATA_REJECTION_GATES)


failed_rejection_contract_gates = (
    MISSING_DATA_REJECTION_GATES.loc[
        ~MISSING_DATA_REJECTION_GATES["passed"]
    ]
)

if not failed_rejection_contract_gates.empty:
    failure_text = ", ".join(
        f"{row.gate}: {row.evidence}"
        for row
        in failed_rejection_contract_gates.itertuples(
            index=False
        )
    )

    raise RuntimeError(
        "Missing-data and rejection contract failed: "
        + failure_text
    )


MISSING_DATA_REJECTION_CONTRACT_HASH = (
    sha256_of_canonical_object(
        MISSING_DATA_REJECTION_CONTRACT
    )
)


print("Missing-data and rejection contract: PASS")
print(
    "Registered rejection reasons: "
    f"{len(REJECTION_REASON_REGISTRY):,}"
)
print("Imputation and timestamp synthesis: PROHIBITED")
print(
    "Invalid differential-depth event handling: "
    "FAIL_STREAM"
)
print(
    "Zero-quantity differential-depth update: "
    "DELETE_PRICE_LEVEL"
)
print(
    "Missing-data and rejection contract hash: "
    f"{MISSING_DATA_REJECTION_CONTRACT_HASH}"
)

,reason_code,stage,scope,severity,disposition,downstream_effect
0,V0_0_MODIFICATION_ATTEMPT,ALL,PIPELINE,CRITICAL,FAIL_PIPELINE,STOP_ALL_DOWNSTREAM_EXECUTION
1,UNDECLARED_REPAIR_ATTEMPT,ALL,PIPELINE,CRITICAL,FAIL_PIPELINE,STOP_ALL_DOWNSTREAM_EXECUTION
2,TIMESTAMP_SYNTHESIS_ATTEMPT,ALL,PIPELINE,CRITICAL,FAIL_PIPELINE,STOP_ALL_DOWNSTREAM_EXECUTION
3,FORWARD_LOOKING_JOIN_ATTEMPT,ALIGNMENT_OR_FEATURES,PIPELINE,CRITICAL,FAIL_PIPELINE,INVALIDATE_AFFECTED_OUTPUTS
4,RAW_FILE_UNREADABLE,INGESTION,FILE,CRITICAL,FAIL_FILE,BLOCK_DEPENDENT_NOTEBOOKS
5,RAW_FILE_HASH_MISMATCH,INGESTION,FILE,CRITICAL,FAIL_FILE,BLOCK_DEPENDENT_NOTEBOOKS
6,PRIMARY_SOURCE_ROLE_AMBIGUOUS,INGESTION,FILE,CRITICAL,FAIL_FILE,BLOCK_SOURCE_SELECTION
7,COLLECTOR_SEQUENCE_MISSING,RAW_AUDIT,STREAM,CRITICAL,FAIL_STREAM,CAUSAL_ORDER_UNAVAILABLE
8,COLLECTOR_SEQUENCE_DUPLICATE,RAW_AUDIT,STREAM,CRITICAL,FAIL_STREAM,CAUSAL_ORDER_AMBIGUOUS
9,COLLECTOR_SEQUENCE_NONMONOTONE,RAW_AUDIT,STREAM,CRITICAL,FAIL_STREAM,CAUSAL_ORDER_INVALID


,field,value
0,registered_reason_codes,50
1,critical_reason_codes,35
2,record_rejection_codes,11
3,imputation_allowed,False
4,timestamp_jitter_allowed,False
5,invalid_depth_event_drop_allowed,False
6,zero_depth_quantity_semantics,DELETE_PRICE_LEVEL
7,snapshot_partial_acceptance,False
8,reference_silent_repair_allowed,False


,gate,passed,evidence
0,reason_codes_unique,True,rows=50; unique_codes=50
1,scopes_valid,True,invalid=0
2,severities_valid,True,invalid=0
3,dispositions_valid,True,invalid=0
4,registry_fields_complete,True,incomplete_rows=0
5,all_forbidden_operations_disabled,True,numeric_imputation_prohibited=True | forward_f...
6,zero_depth_quantity_is_delete,True,DELETE_PRICE_LEVEL
7,invalid_depth_events_are_not_dropped,True,dropping_invalid_depth_events_allowed=False
8,trade_rejections_require_ledger,True,rejected_rows_must_be_saved=True


Missing-data and rejection contract: PASS
Registered rejection reasons: 50
Imputation and timestamp synthesis: PROHIBITED
Invalid differential-depth event handling: FAIL_STREAM
Zero-quantity differential-depth update: DELETE_PRICE_LEVEL
Missing-data and rejection contract hash: 27dafdd4a69308d7ff4d64fb0bdde5150c123c92cc425a80027e9a3d0f6eca49


In [21]:
# ============================================================
# CHRONOLOGICAL SPLIT CONTRACT
# ============================================================

ENGINEERING_SPLIT_WEIGHTS = {
    "DEVELOPMENT": 50,
    "CALIBRATION": 20,
    "VALIDATION": 15,
    "ENGINEERING_HOLDOUT": 15,
}

FULL_MODE_SPLIT_NAMES = [
    "DEVELOPMENT",
    "CALIBRATION",
    "VALIDATION",
    "UNTOUCHED_HOLDOUT",
]


# ============================================================
# EXTRACT THE MINIMUM SPLIT KEYS FROM BOTH RAW STREAMS
# ============================================================

def scan_stream_split_keys(
    path: Path,
    source_role: str,
) -> list[dict]:
    """
    Extract only collector sequence, local receipt time, and session ID.

    This does not construct the canonical analytical dataset.
    """
    rows = []

    with path.open(
        "r",
        encoding="utf-8-sig",
        errors="strict",
    ) as handle:
        for source_line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise RuntimeError(
                    f"Invalid JSON in {source_role} at line "
                    f"{source_line_number}: {exc}"
                ) from exc

            if not isinstance(record, dict):
                raise RuntimeError(
                    f"Non-object record in {source_role} at line "
                    f"{source_line_number}."
                )

            collector_sequence = safe_integer(
                first_present(
                    record,
                    (
                        "collector_sequence",
                        "collector_seq",
                        "local_sequence",
                        "sequence",
                    ),
                )
            )

            local_receipt_time_ns = safe_integer(
                first_present(
                    record,
                    (
                        "local_receipt_time_ns",
                        "receipt_time_ns",
                        "received_time_ns",
                        "receive_time_ns",
                        "recv_time_ns",
                    ),
                )
            )

            connection_session_id = first_present(
                record,
                (
                    "connection_session_id",
                    "collector_session_id",
                    "session_id",
                ),
            )

            if collector_sequence is None:
                raise RuntimeError(
                    f"Missing collector sequence in {source_role} "
                    f"at line {source_line_number}."
                )

            if local_receipt_time_ns is None:
                raise RuntimeError(
                    f"Missing local receipt timestamp in {source_role} "
                    f"at line {source_line_number}."
                )

            if connection_session_id is None:
                raise RuntimeError(
                    f"Missing connection session ID in {source_role} "
                    f"at line {source_line_number}."
                )

            rows.append(
                {
                    "source_role": source_role,
                    "source_line_number": source_line_number,
                    "collector_sequence": collector_sequence,
                    "local_receipt_time_ns": local_receipt_time_ns,
                    "connection_session_id": str(
                        connection_session_id
                    ),
                }
            )

    return rows


split_key_rows = []

split_key_rows.extend(
    scan_stream_split_keys(
        path=TRADE_SOURCE_PATH,
        source_role="TRADE_STREAM",
    )
)

split_key_rows.extend(
    scan_stream_split_keys(
        path=DEPTH_SOURCE_PATH,
        source_role="DEPTH_STREAM",
    )
)


CHRONOLOGICAL_SPLIT_KEYS = pd.DataFrame(split_key_rows)

CHRONOLOGICAL_SPLIT_KEYS = (
    CHRONOLOGICAL_SPLIT_KEYS
    .sort_values(
        "collector_sequence",
        kind="stable",
    )
    .reset_index(drop=True)
)


# ============================================================
# SPLIT-KEY INTEGRITY GATES
# ============================================================

duplicate_split_sequences = CHRONOLOGICAL_SPLIT_KEYS.loc[
    CHRONOLOGICAL_SPLIT_KEYS[
        "collector_sequence"
    ].duplicated(keep=False)
]

observed_split_sequences = (
    CHRONOLOGICAL_SPLIT_KEYS[
        "collector_sequence"
    ].astype("int64")
)

observed_sequence_min = int(
    observed_split_sequences.min()
)

observed_sequence_max = int(
    observed_split_sequences.max()
)

expected_sequence_count = (
    observed_sequence_max
    - observed_sequence_min
    + 1
)

observed_sequence_count = int(
    observed_split_sequences.nunique()
)

missing_sequence_count = (
    expected_sequence_count
    - observed_sequence_count
)

sequence_differences = (
    observed_split_sequences.diff()
)

sequence_reversal_count = int(
    sequence_differences.lt(0).sum()
)

sequence_gap_count = int(
    sequence_differences.gt(1).sum()
)

receipt_differences = (
    CHRONOLOGICAL_SPLIT_KEYS[
        "local_receipt_time_ns"
    ]
    .astype("int64")
    .diff()
)

receipt_time_reversal_count = int(
    receipt_differences.lt(0).sum()
)


if not duplicate_split_sequences.empty:
    raise RuntimeError(
        "Duplicate collector sequences were found while defining "
        "the chronological split contract."
    )

if missing_sequence_count != 0:
    raise RuntimeError(
        "Collector sequence coverage is incomplete while defining "
        f"splits: missing={missing_sequence_count}."
    )

if sequence_reversal_count != 0:
    raise RuntimeError(
        "Collector sequence reversals were found while defining "
        "the chronological split contract."
    )

if sequence_gap_count != 0:
    raise RuntimeError(
        "Collector sequence gaps were found while defining "
        "the chronological split contract."
    )

if receipt_time_reversal_count != 0:
    raise RuntimeError(
        "Local receipt time reverses in collector-sequence order. "
        "Split boundaries cannot be frozen safely."
    )


# ============================================================
# SESSION ORDER
# ============================================================

SESSION_ORDER_TABLE = (
    CHRONOLOGICAL_SPLIT_KEYS
    .groupby(
        "connection_session_id",
        as_index=False,
    )
    .agg(
        first_collector_sequence=(
            "collector_sequence",
            "min",
        ),
        last_collector_sequence=(
            "collector_sequence",
            "max",
        ),
        first_local_receipt_time_ns=(
            "local_receipt_time_ns",
            "min",
        ),
        last_local_receipt_time_ns=(
            "local_receipt_time_ns",
            "max",
        ),
        row_count=(
            "collector_sequence",
            "size",
        ),
    )
    .sort_values(
        "first_collector_sequence",
        kind="stable",
    )
    .reset_index(drop=True)
)

SESSION_ORDER_TABLE["session_order"] = (
    np.arange(
        1,
        len(SESSION_ORDER_TABLE) + 1,
    )
)


# ============================================================
# PARTITION ASSIGNMENT
# ============================================================

def allocate_weighted_counts(
    total_count: int,
    weights: dict[str, int],
) -> dict[str, int]:
    """
    Allocate integer partition sizes deterministically.

    Remaining rows are assigned from the earliest partition forward.
    """
    if total_count <= 0:
        raise ValueError("total_count must be positive.")

    weight_total = sum(weights.values())

    if weight_total != 100:
        raise ValueError(
            "Engineering split weights must sum to 100."
        )

    counts = {
        name: total_count * weight // 100
        for name, weight in weights.items()
    }

    remainder = total_count - sum(counts.values())

    for name in weights:
        if remainder == 0:
            break

        counts[name] += 1
        remainder -= 1

    if any(count <= 0 for count in counts.values()):
        raise RuntimeError(
            "At least one chronological partition would be empty."
        )

    return counts


partition_assignments = pd.Series(
    index=CHRONOLOGICAL_SPLIT_KEYS.index,
    dtype="object",
)

partition_authority = {}
partition_claim_authority = {}
partition_independence = {}


if V0_1_OPERATING_MODE == "FULL_STATISTICAL_MODE":
    if len(SESSION_ORDER_TABLE) < 4:
        raise RuntimeError(
            "FULL_STATISTICAL_MODE requires at least four "
            "chronologically ordered sessions."
        )

    ordered_sessions = (
        SESSION_ORDER_TABLE[
            "connection_session_id"
        ].tolist()
    )

    development_sessions = ordered_sessions[:-3]
    calibration_sessions = [ordered_sessions[-3]]
    validation_sessions = [ordered_sessions[-2]]
    holdout_sessions = [ordered_sessions[-1]]

    full_mode_session_map = {
        "DEVELOPMENT": development_sessions,
        "CALIBRATION": calibration_sessions,
        "VALIDATION": validation_sessions,
        "UNTOUCHED_HOLDOUT": holdout_sessions,
    }

    for partition_name, session_ids in (
        full_mode_session_map.items()
    ):
        mask = CHRONOLOGICAL_SPLIT_KEYS[
            "connection_session_id"
        ].isin(session_ids)

        partition_assignments.loc[mask] = (
            partition_name
        )

        partition_authority[partition_name] = (
            "WHOLE_CONNECTION_SESSIONS"
        )

        partition_claim_authority[partition_name] = True
        partition_independence[partition_name] = True

    split_method = "WHOLE_SESSION_CHRONOLOGICAL"

else:
    engineering_counts = allocate_weighted_counts(
        total_count=len(CHRONOLOGICAL_SPLIT_KEYS),
        weights=ENGINEERING_SPLIT_WEIGHTS,
    )

    start_index = 0

    for partition_name, partition_count in (
        engineering_counts.items()
    ):
        stop_index = start_index + partition_count

        partition_assignments.iloc[
            start_index:stop_index
        ] = partition_name

        partition_authority[partition_name] = (
            "INTRA_SESSION_COLLECTOR_SEQUENCE"
        )

        partition_claim_authority[partition_name] = False
        partition_independence[partition_name] = False

        start_index = stop_index

    if start_index != len(CHRONOLOGICAL_SPLIT_KEYS):
        raise RuntimeError(
            "Engineering partition allocation did not consume "
            "the complete source sequence."
        )

    split_method = "INTRA_SESSION_ENGINEERING"


CHRONOLOGICAL_SPLIT_KEYS["partition"] = (
    partition_assignments
)

if CHRONOLOGICAL_SPLIT_KEYS["partition"].isna().any():
    raise RuntimeError(
        "At least one source record was not assigned to a "
        "chronological partition."
    )


# ============================================================
# PARTITION BOUNDARY TABLE
# ============================================================

partition_order = (
    FULL_MODE_SPLIT_NAMES
    if V0_1_OPERATING_MODE == "FULL_STATISTICAL_MODE"
    else list(ENGINEERING_SPLIT_WEIGHTS.keys())
)

partition_rows = []

for partition_number, partition_name in enumerate(
    partition_order,
    start=1,
):
    partition_records = (
        CHRONOLOGICAL_SPLIT_KEYS.loc[
            CHRONOLOGICAL_SPLIT_KEYS[
                "partition"
            ].eq(partition_name)
        ]
        .sort_values(
            "collector_sequence",
            kind="stable",
        )
    )

    if partition_records.empty:
        raise RuntimeError(
            f"Chronological partition is empty: "
            f"{partition_name}"
        )

    start_sequence = int(
        partition_records[
            "collector_sequence"
        ].iloc[0]
    )

    last_sequence = int(
        partition_records[
            "collector_sequence"
        ].iloc[-1]
    )

    end_sequence_exclusive = last_sequence + 1

    start_receipt_ns = int(
        partition_records[
            "local_receipt_time_ns"
        ].iloc[0]
    )

    if partition_number < len(partition_order):
        next_partition_name = partition_order[
            partition_number
        ]

        next_partition_records = (
            CHRONOLOGICAL_SPLIT_KEYS.loc[
                CHRONOLOGICAL_SPLIT_KEYS[
                    "partition"
                ].eq(next_partition_name)
            ]
            .sort_values(
                "collector_sequence",
                kind="stable",
            )
        )

        end_receipt_ns_exclusive = int(
            next_partition_records[
                "local_receipt_time_ns"
            ].iloc[0]
        )

    else:
        end_receipt_ns_exclusive = (
            int(
                partition_records[
                    "local_receipt_time_ns"
                ].iloc[-1]
            )
            + 1
        )

    partition_rows.append(
        {
            "partition_order": partition_number,
            "partition": partition_name,
            "boundary_authority": (
                partition_authority[
                    partition_name
                ]
            ),
            "collector_sequence_start": (
                start_sequence
            ),
            "collector_sequence_end_exclusive": (
                end_sequence_exclusive
            ),
            "collector_sequence_last_included": (
                last_sequence
            ),
            "receipt_time_start_ns": (
                start_receipt_ns
            ),
            "receipt_time_end_ns_exclusive": (
                end_receipt_ns_exclusive
            ),
            "receipt_time_start_utc": timestamp_iso(
                start_receipt_ns,
                unit="ns",
            ),
            "receipt_time_end_utc_exclusive": (
                timestamp_iso(
                    end_receipt_ns_exclusive,
                    unit="ns",
                )
            ),
            "row_count": int(
                len(partition_records)
            ),
            "trade_row_count": int(
                partition_records[
                    "source_role"
                ].eq("TRADE_STREAM").sum()
            ),
            "depth_row_count": int(
                partition_records[
                    "source_role"
                ].eq("DEPTH_STREAM").sum()
            ),
            "session_count": int(
                partition_records[
                    "connection_session_id"
                ].nunique()
            ),
            "independent_partition": bool(
                partition_independence[
                    partition_name
                ]
            ),
            "claim_authority": bool(
                partition_claim_authority[
                    partition_name
                ]
            ),
            "interval_convention": (
                "[start, end)"
            ),
        }
    )


CHRONOLOGICAL_SPLIT_TABLE = pd.DataFrame(
    partition_rows
)


# ============================================================
# ACCESS AND HISTORY RULES
# ============================================================

access_rows = []

for partition_name in partition_order:
    if partition_name == "DEVELOPMENT":
        model_selection_access = True
        parameter_estimation_access = True
        final_evaluation_access = True

    elif partition_name == "CALIBRATION":
        model_selection_access = True
        parameter_estimation_access = True
        final_evaluation_access = True

    elif partition_name == "VALIDATION":
        model_selection_access = True
        parameter_estimation_access = False
        final_evaluation_access = True

    else:
        model_selection_access = False
        parameter_estimation_access = False
        final_evaluation_access = True

    access_rows.append(
        {
            "partition": partition_name,
            "model_selection_access": (
                model_selection_access
            ),
            "parameter_estimation_access": (
                parameter_estimation_access
            ),
            "final_evaluation_access": (
                final_evaluation_access
            ),
            "claim_authority": bool(
                partition_claim_authority[
                    partition_name
                ]
            ),
        }
    )


SPLIT_ACCESS_MATRIX = pd.DataFrame(access_rows)


SPLIT_HISTORY_RULES = pd.DataFrame(
    [
        {
            "stage": "BOOK_RECONSTRUCTION",
            "history_carry_allowed": True,
            "history_usage": (
                "Carry the last valid reconstructed book state "
                "across partition boundaries."
            ),
            "scored_pre_boundary_events": False,
        },
        {
            "stage": "TRADE_BOOK_ALIGNMENT",
            "history_carry_allowed": True,
            "history_usage": (
                "Use only book states causally available before "
                "each trade in collector-sequence order."
            ),
            "scored_pre_boundary_events": False,
        },
        {
            "stage": "MARKET_STATE_FEATURES",
            "history_carry_allowed": True,
            "history_usage": (
                "Pre-boundary observations may initialize rolling "
                "state, but cannot be scored inside the partition."
            ),
            "scored_pre_boundary_events": False,
        },
        {
            "stage": "HAWKES_INTENSITY",
            "history_carry_allowed": True,
            "history_usage": (
                "Earlier events may initialize excitation history; "
                "partition likelihood and diagnostics begin at the "
                "partition boundary."
            ),
            "scored_pre_boundary_events": False,
        },
        {
            "stage": "MODEL_SELECTION",
            "history_carry_allowed": False,
            "history_usage": (
                "The final reserved partition cannot influence "
                "model choice or parameter tuning."
            ),
            "scored_pre_boundary_events": False,
        },
    ]
)


# ============================================================
# FROZEN SPLIT CONTRACT
# ============================================================

CHRONOLOGICAL_SPLIT_CONTRACT = {
    "contract_schema_version": CONTRACT_SCHEMA_VERSION,
    "pipeline_version": PIPELINE_VERSION,
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "source_set_hash": SOURCE_SET_HASH,
    "market_data_contract_hash": MARKET_DATA_CONTRACT_HASH,
    "missing_data_rejection_contract_hash": (
        MISSING_DATA_REJECTION_CONTRACT_HASH
    ),
    "operating_mode": V0_1_OPERATING_MODE,
    "split_method": split_method,
    "primary_boundary_authority": (
        "collector_sequence"
    ),
    "interval_convention": "[start, end)",
    "canonical_timezone": "UTC",
    "partition_order": partition_order,
    "engineering_split_weights_percent": (
        ENGINEERING_SPLIT_WEIGHTS
        if V0_1_OPERATING_MODE
        == "ENGINEERING_REPRODUCTION_MODE"
        else None
    ),
    "partitions_are_independent": bool(
        V0_1_OPERATING_MODE
        == "FULL_STATISTICAL_MODE"
    ),
    "claim_bearing_holdout_available": bool(
        V0_1_OPERATING_MODE
        == "FULL_STATISTICAL_MODE"
    ),
    "final_partition_model_selection_access": False,
    "boundary_table": (
        CHRONOLOGICAL_SPLIT_TABLE
        .to_dict(orient="records")
    ),
    "access_matrix": (
        SPLIT_ACCESS_MATRIX
        .to_dict(orient="records")
    ),
    "history_rules": (
        SPLIT_HISTORY_RULES
        .to_dict(orient="records")
    ),
    "producing_notebook": NOTEBOOK_FILENAME,
}


# ============================================================
# SPLIT-CONTRACT GATES
# ============================================================

sequence_intervals = (
    CHRONOLOGICAL_SPLIT_TABLE
    .sort_values(
        "partition_order",
        kind="stable",
    )
    .reset_index(drop=True)
)

boundary_contiguity_passed = True

for row_index in range(
    len(sequence_intervals) - 1
):
    current_end = int(
        sequence_intervals.loc[
            row_index,
            "collector_sequence_end_exclusive",
        ]
    )

    next_start = int(
        sequence_intervals.loc[
            row_index + 1,
            "collector_sequence_start",
        ]
    )

    if current_end != next_start:
        boundary_contiguity_passed = False
        break


total_partition_rows = int(
    CHRONOLOGICAL_SPLIT_TABLE[
        "row_count"
    ].sum()
)

complete_sequence_coverage_passed = bool(
    int(
        CHRONOLOGICAL_SPLIT_TABLE.iloc[0][
            "collector_sequence_start"
        ]
    )
    == observed_sequence_min
    and int(
        CHRONOLOGICAL_SPLIT_TABLE.iloc[-1][
            "collector_sequence_end_exclusive"
        ]
    )
    == observed_sequence_max + 1
    and total_partition_rows
    == len(CHRONOLOGICAL_SPLIT_KEYS)
)


if V0_1_OPERATING_MODE == "ENGINEERING_REPRODUCTION_MODE":
    holdout_authority_rule_passed = bool(
        not CHRONOLOGICAL_SPLIT_TABLE.loc[
            CHRONOLOGICAL_SPLIT_TABLE[
                "partition"
            ].eq("ENGINEERING_HOLDOUT"),
            "claim_authority",
        ].any()
    )

else:
    holdout_authority_rule_passed = bool(
        CHRONOLOGICAL_SPLIT_TABLE.loc[
            CHRONOLOGICAL_SPLIT_TABLE[
                "partition"
            ].eq("UNTOUCHED_HOLDOUT"),
            "claim_authority",
        ].all()
    )


CHRONOLOGICAL_SPLIT_GATES = pd.DataFrame(
    [
        {
            "gate": "all_records_assigned",
            "passed": bool(
                CHRONOLOGICAL_SPLIT_KEYS[
                    "partition"
                ].notna().all()
            ),
            "evidence": (
                f"assigned="
                f"{CHRONOLOGICAL_SPLIT_KEYS['partition'].notna().sum()}/"
                f"{len(CHRONOLOGICAL_SPLIT_KEYS)}"
            ),
        },
        {
            "gate": "partitions_nonempty",
            "passed": bool(
                CHRONOLOGICAL_SPLIT_TABLE[
                    "row_count"
                ].gt(0).all()
            ),
            "evidence": (
                " | ".join(
                    f"{row.partition}={row.row_count}"
                    for row in (
                        CHRONOLOGICAL_SPLIT_TABLE
                        .itertuples(index=False)
                    )
                )
            ),
        },
        {
            "gate": "collector_sequence_intervals_contiguous",
            "passed": boundary_contiguity_passed,
            "evidence": (
                "half-open sequence intervals"
            ),
        },
        {
            "gate": "complete_sequence_coverage",
            "passed": complete_sequence_coverage_passed,
            "evidence": (
                f"source={observed_sequence_min}.."
                f"{observed_sequence_max}; "
                f"partition_rows={total_partition_rows}"
            ),
        },
        {
            "gate": "receipt_time_monotone",
            "passed": (
                receipt_time_reversal_count == 0
            ),
            "evidence": (
                f"reversals="
                f"{receipt_time_reversal_count}"
            ),
        },
        {
            "gate": "final_partition_excluded_from_selection",
            "passed": bool(
                not SPLIT_ACCESS_MATRIX.iloc[-1][
                    "model_selection_access"
                ]
            ),
            "evidence": (
                f"partition="
                f"{SPLIT_ACCESS_MATRIX.iloc[-1]['partition']}"
            ),
        },
        {
            "gate": "holdout_authority_matches_operating_mode",
            "passed": holdout_authority_rule_passed,
            "evidence": (
                f"operating_mode="
                f"{V0_1_OPERATING_MODE}"
            ),
        },
        {
            "gate": "half_open_interval_rule_frozen",
            "passed": bool(
                CHRONOLOGICAL_SPLIT_TABLE[
                    "interval_convention"
                ].eq("[start, end)").all()
            ),
            "evidence": "[start, end)",
        },
    ]
)


display(
    SESSION_ORDER_TABLE[
        [
            "session_order",
            "connection_session_id",
            "first_collector_sequence",
            "last_collector_sequence",
            "row_count",
        ]
    ]
)

display(CHRONOLOGICAL_SPLIT_TABLE)
display(SPLIT_ACCESS_MATRIX)
display(SPLIT_HISTORY_RULES)
display(CHRONOLOGICAL_SPLIT_GATES)


failed_split_gates = (
    CHRONOLOGICAL_SPLIT_GATES.loc[
        ~CHRONOLOGICAL_SPLIT_GATES[
            "passed"
        ]
    ]
)

if not failed_split_gates.empty:
    failure_text = ", ".join(
        f"{row.gate}: {row.evidence}"
        for row in failed_split_gates.itertuples(
            index=False
        )
    )

    raise RuntimeError(
        "Chronological split contract failed: "
        + failure_text
    )


CHRONOLOGICAL_SPLIT_CONTRACT_HASH = (
    sha256_of_canonical_object(
        CHRONOLOGICAL_SPLIT_CONTRACT
    )
)


print("Chronological split contract: PASS")
print(f"Split method: {split_method}")
print(
    "Partition order: "
    + " -> ".join(partition_order)
)
print(
    "Primary boundary authority: "
    "collector_sequence"
)
print(
    "Partitions treated as independent: "
    f"{CHRONOLOGICAL_SPLIT_CONTRACT['partitions_are_independent']}"
)
print(
    "Claim-bearing holdout available: "
    f"{CHRONOLOGICAL_SPLIT_CONTRACT['claim_bearing_holdout_available']}"
)
print(
    "Chronological split contract hash: "
    f"{CHRONOLOGICAL_SPLIT_CONTRACT_HASH}"
)

,session_order,connection_session_id,first_collector_sequence,last_collector_sequence,row_count
0,1,c8b5bf127a7a44669514acfda8634107,1,103677,103677


,partition_order,partition,boundary_authority,collector_sequence_start,collector_sequence_end_exclusive,collector_sequence_last_included,receipt_time_start_ns,receipt_time_end_ns_exclusive,receipt_time_start_utc,receipt_time_end_utc_exclusive,row_count,trade_row_count,depth_row_count,session_count,independent_partition,claim_authority,interval_convention
0,1,DEVELOPMENT,INTRA_SESSION_COLLECTOR_SEQUENCE,1,51840,51839,1783665467531985400,1783667269391572100,2026-07-10T06:37:47.531985400+00:00,2026-07-10T07:07:49.391572100+00:00,51839,33820,18019,1,False,False,"[start, end)"
1,2,CALIBRATION,INTRA_SESSION_COLLECTOR_SEQUENCE,51840,72576,72575,1783667269391572100,1783667989690751200,2026-07-10T07:07:49.391572100+00:00,2026-07-10T07:19:49.690751200+00:00,20736,13532,7204,1,False,False,"[start, end)"
2,3,VALIDATION,INTRA_SESSION_COLLECTOR_SEQUENCE,72576,88127,88126,1783667989690751200,1783668525057534700,2026-07-10T07:19:49.690751200+00:00,2026-07-10T07:28:45.057534700+00:00,15551,10198,5353,1,False,False,"[start, end)"
3,4,ENGINEERING_HOLDOUT,INTRA_SESSION_COLLECTOR_SEQUENCE,88127,103678,103677,1783668525057534700,1783669066749750801,2026-07-10T07:28:45.057534700+00:00,2026-07-10T07:37:46.749750801+00:00,15551,10133,5418,1,False,False,"[start, end)"


,partition,model_selection_access,parameter_estimation_access,final_evaluation_access,claim_authority
0,DEVELOPMENT,True,True,True,False
1,CALIBRATION,True,True,True,False
2,VALIDATION,True,False,True,False
3,ENGINEERING_HOLDOUT,False,False,True,False


,stage,history_carry_allowed,history_usage,scored_pre_boundary_events
0,BOOK_RECONSTRUCTION,True,Carry the last valid reconstructed book state ...,False
1,TRADE_BOOK_ALIGNMENT,True,Use only book states causally available before...,False
2,MARKET_STATE_FEATURES,True,Pre-boundary observations may initialize rolli...,False
3,HAWKES_INTENSITY,True,Earlier events may initialize excitation histo...,False
4,MODEL_SELECTION,False,The final reserved partition cannot influence ...,False


,gate,passed,evidence
0,all_records_assigned,True,assigned=103677/103677
1,partitions_nonempty,True,DEVELOPMENT=51839 | CALIBRATION=20736 | VALIDA...
2,collector_sequence_intervals_contiguous,True,half-open sequence intervals
3,complete_sequence_coverage,True,source=1..103677; partition_rows=103677
4,receipt_time_monotone,True,reversals=0
5,final_partition_excluded_from_selection,True,partition=ENGINEERING_HOLDOUT
6,holdout_authority_matches_operating_mode,True,operating_mode=ENGINEERING_REPRODUCTION_MODE
7,half_open_interval_rule_frozen,True,"[start, end)"


Chronological split contract: PASS
Split method: INTRA_SESSION_ENGINEERING
Partition order: DEVELOPMENT -> CALIBRATION -> VALIDATION -> ENGINEERING_HOLDOUT
Primary boundary authority: collector_sequence
Partitions treated as independent: False
Claim-bearing holdout available: False
Chronological split contract hash: 3b7e8d46f43d8a4dd7b7a1f3d9a61c2c92d3f3643042a8b01968ddc7e0fb4624


In [22]:
# ============================================================
# CHRONOLOGICAL SPLIT CONTRACT
# ============================================================

ENGINEERING_SPLIT_WEIGHTS = {
    "DEVELOPMENT": 50,
    "CALIBRATION": 20,
    "VALIDATION": 15,
    "ENGINEERING_HOLDOUT": 15,
}

FULL_MODE_SPLIT_NAMES = [
    "DEVELOPMENT",
    "CALIBRATION",
    "VALIDATION",
    "UNTOUCHED_HOLDOUT",
]


# ============================================================
# EXTRACT THE MINIMUM SPLIT KEYS FROM BOTH RAW STREAMS
# ============================================================

def scan_stream_split_keys(
    path: Path,
    source_role: str,
) -> list[dict]:
    """
    Extract only collector sequence, local receipt time, and session ID.

    This does not construct the canonical analytical dataset.
    """
    rows = []

    with path.open(
        "r",
        encoding="utf-8-sig",
        errors="strict",
    ) as handle:
        for source_line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise RuntimeError(
                    f"Invalid JSON in {source_role} at line "
                    f"{source_line_number}: {exc}"
                ) from exc

            if not isinstance(record, dict):
                raise RuntimeError(
                    f"Non-object record in {source_role} at line "
                    f"{source_line_number}."
                )

            collector_sequence = safe_integer(
                first_present(
                    record,
                    (
                        "collector_sequence",
                        "collector_seq",
                        "local_sequence",
                        "sequence",
                    ),
                )
            )

            local_receipt_time_ns = safe_integer(
                first_present(
                    record,
                    (
                        "local_receipt_time_ns",
                        "receipt_time_ns",
                        "received_time_ns",
                        "receive_time_ns",
                        "recv_time_ns",
                    ),
                )
            )

            connection_session_id = first_present(
                record,
                (
                    "connection_session_id",
                    "collector_session_id",
                    "session_id",
                ),
            )

            if collector_sequence is None:
                raise RuntimeError(
                    f"Missing collector sequence in {source_role} "
                    f"at line {source_line_number}."
                )

            if local_receipt_time_ns is None:
                raise RuntimeError(
                    f"Missing local receipt timestamp in {source_role} "
                    f"at line {source_line_number}."
                )

            if connection_session_id is None:
                raise RuntimeError(
                    f"Missing connection session ID in {source_role} "
                    f"at line {source_line_number}."
                )

            rows.append(
                {
                    "source_role": source_role,
                    "source_line_number": source_line_number,
                    "collector_sequence": collector_sequence,
                    "local_receipt_time_ns": local_receipt_time_ns,
                    "connection_session_id": str(
                        connection_session_id
                    ),
                }
            )

    return rows


split_key_rows = []

split_key_rows.extend(
    scan_stream_split_keys(
        path=TRADE_SOURCE_PATH,
        source_role="TRADE_STREAM",
    )
)

split_key_rows.extend(
    scan_stream_split_keys(
        path=DEPTH_SOURCE_PATH,
        source_role="DEPTH_STREAM",
    )
)


CHRONOLOGICAL_SPLIT_KEYS = pd.DataFrame(split_key_rows)

CHRONOLOGICAL_SPLIT_KEYS = (
    CHRONOLOGICAL_SPLIT_KEYS
    .sort_values(
        "collector_sequence",
        kind="stable",
    )
    .reset_index(drop=True)
)


# ============================================================
# SPLIT-KEY INTEGRITY GATES
# ============================================================

duplicate_split_sequences = CHRONOLOGICAL_SPLIT_KEYS.loc[
    CHRONOLOGICAL_SPLIT_KEYS[
        "collector_sequence"
    ].duplicated(keep=False)
]

observed_split_sequences = (
    CHRONOLOGICAL_SPLIT_KEYS[
        "collector_sequence"
    ].astype("int64")
)

observed_sequence_min = int(
    observed_split_sequences.min()
)

observed_sequence_max = int(
    observed_split_sequences.max()
)

expected_sequence_count = (
    observed_sequence_max
    - observed_sequence_min
    + 1
)

observed_sequence_count = int(
    observed_split_sequences.nunique()
)

missing_sequence_count = (
    expected_sequence_count
    - observed_sequence_count
)

sequence_differences = (
    observed_split_sequences.diff()
)

sequence_reversal_count = int(
    sequence_differences.lt(0).sum()
)

sequence_gap_count = int(
    sequence_differences.gt(1).sum()
)

receipt_differences = (
    CHRONOLOGICAL_SPLIT_KEYS[
        "local_receipt_time_ns"
    ]
    .astype("int64")
    .diff()
)

receipt_time_reversal_count = int(
    receipt_differences.lt(0).sum()
)


if not duplicate_split_sequences.empty:
    raise RuntimeError(
        "Duplicate collector sequences were found while defining "
        "the chronological split contract."
    )

if missing_sequence_count != 0:
    raise RuntimeError(
        "Collector sequence coverage is incomplete while defining "
        f"splits: missing={missing_sequence_count}."
    )

if sequence_reversal_count != 0:
    raise RuntimeError(
        "Collector sequence reversals were found while defining "
        "the chronological split contract."
    )

if sequence_gap_count != 0:
    raise RuntimeError(
        "Collector sequence gaps were found while defining "
        "the chronological split contract."
    )

if receipt_time_reversal_count != 0:
    raise RuntimeError(
        "Local receipt time reverses in collector-sequence order. "
        "Split boundaries cannot be frozen safely."
    )


# ============================================================
# SESSION ORDER
# ============================================================

SESSION_ORDER_TABLE = (
    CHRONOLOGICAL_SPLIT_KEYS
    .groupby(
        "connection_session_id",
        as_index=False,
    )
    .agg(
        first_collector_sequence=(
            "collector_sequence",
            "min",
        ),
        last_collector_sequence=(
            "collector_sequence",
            "max",
        ),
        first_local_receipt_time_ns=(
            "local_receipt_time_ns",
            "min",
        ),
        last_local_receipt_time_ns=(
            "local_receipt_time_ns",
            "max",
        ),
        row_count=(
            "collector_sequence",
            "size",
        ),
    )
    .sort_values(
        "first_collector_sequence",
        kind="stable",
    )
    .reset_index(drop=True)
)

SESSION_ORDER_TABLE["session_order"] = (
    np.arange(
        1,
        len(SESSION_ORDER_TABLE) + 1,
    )
)


# ============================================================
# PARTITION ASSIGNMENT
# ============================================================

def allocate_weighted_counts(
    total_count: int,
    weights: dict[str, int],
) -> dict[str, int]:
    """
    Allocate integer partition sizes deterministically.

    Remaining rows are assigned from the earliest partition forward.
    """
    if total_count <= 0:
        raise ValueError("total_count must be positive.")

    weight_total = sum(weights.values())

    if weight_total != 100:
        raise ValueError(
            "Engineering split weights must sum to 100."
        )

    counts = {
        name: total_count * weight // 100
        for name, weight in weights.items()
    }

    remainder = total_count - sum(counts.values())

    for name in weights:
        if remainder == 0:
            break

        counts[name] += 1
        remainder -= 1

    if any(count <= 0 for count in counts.values()):
        raise RuntimeError(
            "At least one chronological partition would be empty."
        )

    return counts


partition_assignments = pd.Series(
    index=CHRONOLOGICAL_SPLIT_KEYS.index,
    dtype="object",
)

partition_authority = {}
partition_claim_authority = {}
partition_independence = {}


if V0_1_OPERATING_MODE == "FULL_STATISTICAL_MODE":
    if len(SESSION_ORDER_TABLE) < 4:
        raise RuntimeError(
            "FULL_STATISTICAL_MODE requires at least four "
            "chronologically ordered sessions."
        )

    ordered_sessions = (
        SESSION_ORDER_TABLE[
            "connection_session_id"
        ].tolist()
    )

    development_sessions = ordered_sessions[:-3]
    calibration_sessions = [ordered_sessions[-3]]
    validation_sessions = [ordered_sessions[-2]]
    holdout_sessions = [ordered_sessions[-1]]

    full_mode_session_map = {
        "DEVELOPMENT": development_sessions,
        "CALIBRATION": calibration_sessions,
        "VALIDATION": validation_sessions,
        "UNTOUCHED_HOLDOUT": holdout_sessions,
    }

    for partition_name, session_ids in (
        full_mode_session_map.items()
    ):
        mask = CHRONOLOGICAL_SPLIT_KEYS[
            "connection_session_id"
        ].isin(session_ids)

        partition_assignments.loc[mask] = (
            partition_name
        )

        partition_authority[partition_name] = (
            "WHOLE_CONNECTION_SESSIONS"
        )

        partition_claim_authority[partition_name] = True
        partition_independence[partition_name] = True

    split_method = "WHOLE_SESSION_CHRONOLOGICAL"

else:
    engineering_counts = allocate_weighted_counts(
        total_count=len(CHRONOLOGICAL_SPLIT_KEYS),
        weights=ENGINEERING_SPLIT_WEIGHTS,
    )

    start_index = 0

    for partition_name, partition_count in (
        engineering_counts.items()
    ):
        stop_index = start_index + partition_count

        partition_assignments.iloc[
            start_index:stop_index
        ] = partition_name

        partition_authority[partition_name] = (
            "INTRA_SESSION_COLLECTOR_SEQUENCE"
        )

        partition_claim_authority[partition_name] = False
        partition_independence[partition_name] = False

        start_index = stop_index

    if start_index != len(CHRONOLOGICAL_SPLIT_KEYS):
        raise RuntimeError(
            "Engineering partition allocation did not consume "
            "the complete source sequence."
        )

    split_method = "INTRA_SESSION_ENGINEERING"


CHRONOLOGICAL_SPLIT_KEYS["partition"] = (
    partition_assignments
)

if CHRONOLOGICAL_SPLIT_KEYS["partition"].isna().any():
    raise RuntimeError(
        "At least one source record was not assigned to a "
        "chronological partition."
    )


# ============================================================
# PARTITION BOUNDARY TABLE
# ============================================================

partition_order = (
    FULL_MODE_SPLIT_NAMES
    if V0_1_OPERATING_MODE == "FULL_STATISTICAL_MODE"
    else list(ENGINEERING_SPLIT_WEIGHTS.keys())
)

partition_rows = []

for partition_number, partition_name in enumerate(
    partition_order,
    start=1,
):
    partition_records = (
        CHRONOLOGICAL_SPLIT_KEYS.loc[
            CHRONOLOGICAL_SPLIT_KEYS[
                "partition"
            ].eq(partition_name)
        ]
        .sort_values(
            "collector_sequence",
            kind="stable",
        )
    )

    if partition_records.empty:
        raise RuntimeError(
            f"Chronological partition is empty: "
            f"{partition_name}"
        )

    start_sequence = int(
        partition_records[
            "collector_sequence"
        ].iloc[0]
    )

    last_sequence = int(
        partition_records[
            "collector_sequence"
        ].iloc[-1]
    )

    end_sequence_exclusive = last_sequence + 1

    start_receipt_ns = int(
        partition_records[
            "local_receipt_time_ns"
        ].iloc[0]
    )

    if partition_number < len(partition_order):
        next_partition_name = partition_order[
            partition_number
        ]

        next_partition_records = (
            CHRONOLOGICAL_SPLIT_KEYS.loc[
                CHRONOLOGICAL_SPLIT_KEYS[
                    "partition"
                ].eq(next_partition_name)
            ]
            .sort_values(
                "collector_sequence",
                kind="stable",
            )
        )

        end_receipt_ns_exclusive = int(
            next_partition_records[
                "local_receipt_time_ns"
            ].iloc[0]
        )

    else:
        end_receipt_ns_exclusive = (
            int(
                partition_records[
                    "local_receipt_time_ns"
                ].iloc[-1]
            )
            + 1
        )

    partition_rows.append(
        {
            "partition_order": partition_number,
            "partition": partition_name,
            "boundary_authority": (
                partition_authority[
                    partition_name
                ]
            ),
            "collector_sequence_start": (
                start_sequence
            ),
            "collector_sequence_end_exclusive": (
                end_sequence_exclusive
            ),
            "collector_sequence_last_included": (
                last_sequence
            ),
            "receipt_time_start_ns": (
                start_receipt_ns
            ),
            "receipt_time_end_ns_exclusive": (
                end_receipt_ns_exclusive
            ),
            "receipt_time_start_utc": timestamp_iso(
                start_receipt_ns,
                unit="ns",
            ),
            "receipt_time_end_utc_exclusive": (
                timestamp_iso(
                    end_receipt_ns_exclusive,
                    unit="ns",
                )
            ),
            "row_count": int(
                len(partition_records)
            ),
            "trade_row_count": int(
                partition_records[
                    "source_role"
                ].eq("TRADE_STREAM").sum()
            ),
            "depth_row_count": int(
                partition_records[
                    "source_role"
                ].eq("DEPTH_STREAM").sum()
            ),
            "session_count": int(
                partition_records[
                    "connection_session_id"
                ].nunique()
            ),
            "independent_partition": bool(
                partition_independence[
                    partition_name
                ]
            ),
            "claim_authority": bool(
                partition_claim_authority[
                    partition_name
                ]
            ),
            "interval_convention": (
                "[start, end)"
            ),
        }
    )


CHRONOLOGICAL_SPLIT_TABLE = pd.DataFrame(
    partition_rows
)


# ============================================================
# ACCESS AND HISTORY RULES
# ============================================================

access_rows = []

for partition_name in partition_order:
    if partition_name == "DEVELOPMENT":
        model_selection_access = True
        parameter_estimation_access = True
        final_evaluation_access = True

    elif partition_name == "CALIBRATION":
        model_selection_access = True
        parameter_estimation_access = True
        final_evaluation_access = True

    elif partition_name == "VALIDATION":
        model_selection_access = True
        parameter_estimation_access = False
        final_evaluation_access = True

    else:
        model_selection_access = False
        parameter_estimation_access = False
        final_evaluation_access = True

    access_rows.append(
        {
            "partition": partition_name,
            "model_selection_access": (
                model_selection_access
            ),
            "parameter_estimation_access": (
                parameter_estimation_access
            ),
            "final_evaluation_access": (
                final_evaluation_access
            ),
            "claim_authority": bool(
                partition_claim_authority[
                    partition_name
                ]
            ),
        }
    )


SPLIT_ACCESS_MATRIX = pd.DataFrame(access_rows)


SPLIT_HISTORY_RULES = pd.DataFrame(
    [
        {
            "stage": "BOOK_RECONSTRUCTION",
            "history_carry_allowed": True,
            "history_usage": (
                "Carry the last valid reconstructed book state "
                "across partition boundaries."
            ),
            "scored_pre_boundary_events": False,
        },
        {
            "stage": "TRADE_BOOK_ALIGNMENT",
            "history_carry_allowed": True,
            "history_usage": (
                "Use only book states causally available before "
                "each trade in collector-sequence order."
            ),
            "scored_pre_boundary_events": False,
        },
        {
            "stage": "MARKET_STATE_FEATURES",
            "history_carry_allowed": True,
            "history_usage": (
                "Pre-boundary observations may initialize rolling "
                "state, but cannot be scored inside the partition."
            ),
            "scored_pre_boundary_events": False,
        },
        {
            "stage": "HAWKES_INTENSITY",
            "history_carry_allowed": True,
            "history_usage": (
                "Earlier events may initialize excitation history; "
                "partition likelihood and diagnostics begin at the "
                "partition boundary."
            ),
            "scored_pre_boundary_events": False,
        },
        {
            "stage": "MODEL_SELECTION",
            "history_carry_allowed": False,
            "history_usage": (
                "The final reserved partition cannot influence "
                "model choice or parameter tuning."
            ),
            "scored_pre_boundary_events": False,
        },
    ]
)


# ============================================================
# FROZEN SPLIT CONTRACT
# ============================================================

CHRONOLOGICAL_SPLIT_CONTRACT = {
    "contract_schema_version": CONTRACT_SCHEMA_VERSION,
    "pipeline_version": PIPELINE_VERSION,
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "source_set_hash": SOURCE_SET_HASH,
    "market_data_contract_hash": MARKET_DATA_CONTRACT_HASH,
    "missing_data_rejection_contract_hash": (
        MISSING_DATA_REJECTION_CONTRACT_HASH
    ),
    "operating_mode": V0_1_OPERATING_MODE,
    "split_method": split_method,
    "primary_boundary_authority": (
        "collector_sequence"
    ),
    "interval_convention": "[start, end)",
    "canonical_timezone": "UTC",
    "partition_order": partition_order,
    "engineering_split_weights_percent": (
        ENGINEERING_SPLIT_WEIGHTS
        if V0_1_OPERATING_MODE
        == "ENGINEERING_REPRODUCTION_MODE"
        else None
    ),
    "partitions_are_independent": bool(
        V0_1_OPERATING_MODE
        == "FULL_STATISTICAL_MODE"
    ),
    "claim_bearing_holdout_available": bool(
        V0_1_OPERATING_MODE
        == "FULL_STATISTICAL_MODE"
    ),
    "final_partition_model_selection_access": False,
    "boundary_table": (
        CHRONOLOGICAL_SPLIT_TABLE
        .to_dict(orient="records")
    ),
    "access_matrix": (
        SPLIT_ACCESS_MATRIX
        .to_dict(orient="records")
    ),
    "history_rules": (
        SPLIT_HISTORY_RULES
        .to_dict(orient="records")
    ),
    "producing_notebook": NOTEBOOK_FILENAME,
}


# ============================================================
# SPLIT-CONTRACT GATES
# ============================================================

sequence_intervals = (
    CHRONOLOGICAL_SPLIT_TABLE
    .sort_values(
        "partition_order",
        kind="stable",
    )
    .reset_index(drop=True)
)

boundary_contiguity_passed = True

for row_index in range(
    len(sequence_intervals) - 1
):
    current_end = int(
        sequence_intervals.loc[
            row_index,
            "collector_sequence_end_exclusive",
        ]
    )

    next_start = int(
        sequence_intervals.loc[
            row_index + 1,
            "collector_sequence_start",
        ]
    )

    if current_end != next_start:
        boundary_contiguity_passed = False
        break


total_partition_rows = int(
    CHRONOLOGICAL_SPLIT_TABLE[
        "row_count"
    ].sum()
)

complete_sequence_coverage_passed = bool(
    int(
        CHRONOLOGICAL_SPLIT_TABLE.iloc[0][
            "collector_sequence_start"
        ]
    )
    == observed_sequence_min
    and int(
        CHRONOLOGICAL_SPLIT_TABLE.iloc[-1][
            "collector_sequence_end_exclusive"
        ]
    )
    == observed_sequence_max + 1
    and total_partition_rows
    == len(CHRONOLOGICAL_SPLIT_KEYS)
)


if V0_1_OPERATING_MODE == "ENGINEERING_REPRODUCTION_MODE":
    holdout_authority_rule_passed = bool(
        not CHRONOLOGICAL_SPLIT_TABLE.loc[
            CHRONOLOGICAL_SPLIT_TABLE[
                "partition"
            ].eq("ENGINEERING_HOLDOUT"),
            "claim_authority",
        ].any()
    )

else:
    holdout_authority_rule_passed = bool(
        CHRONOLOGICAL_SPLIT_TABLE.loc[
            CHRONOLOGICAL_SPLIT_TABLE[
                "partition"
            ].eq("UNTOUCHED_HOLDOUT"),
            "claim_authority",
        ].all()
    )


CHRONOLOGICAL_SPLIT_GATES = pd.DataFrame(
    [
        {
            "gate": "all_records_assigned",
            "passed": bool(
                CHRONOLOGICAL_SPLIT_KEYS[
                    "partition"
                ].notna().all()
            ),
            "evidence": (
                f"assigned="
                f"{CHRONOLOGICAL_SPLIT_KEYS['partition'].notna().sum()}/"
                f"{len(CHRONOLOGICAL_SPLIT_KEYS)}"
            ),
        },
        {
            "gate": "partitions_nonempty",
            "passed": bool(
                CHRONOLOGICAL_SPLIT_TABLE[
                    "row_count"
                ].gt(0).all()
            ),
            "evidence": (
                " | ".join(
                    f"{row.partition}={row.row_count}"
                    for row in (
                        CHRONOLOGICAL_SPLIT_TABLE
                        .itertuples(index=False)
                    )
                )
            ),
        },
        {
            "gate": "collector_sequence_intervals_contiguous",
            "passed": boundary_contiguity_passed,
            "evidence": (
                "half-open sequence intervals"
            ),
        },
        {
            "gate": "complete_sequence_coverage",
            "passed": complete_sequence_coverage_passed,
            "evidence": (
                f"source={observed_sequence_min}.."
                f"{observed_sequence_max}; "
                f"partition_rows={total_partition_rows}"
            ),
        },
        {
            "gate": "receipt_time_monotone",
            "passed": (
                receipt_time_reversal_count == 0
            ),
            "evidence": (
                f"reversals="
                f"{receipt_time_reversal_count}"
            ),
        },
        {
            "gate": "final_partition_excluded_from_selection",
            "passed": bool(
                not SPLIT_ACCESS_MATRIX.iloc[-1][
                    "model_selection_access"
                ]
            ),
            "evidence": (
                f"partition="
                f"{SPLIT_ACCESS_MATRIX.iloc[-1]['partition']}"
            ),
        },
        {
            "gate": "holdout_authority_matches_operating_mode",
            "passed": holdout_authority_rule_passed,
            "evidence": (
                f"operating_mode="
                f"{V0_1_OPERATING_MODE}"
            ),
        },
        {
            "gate": "half_open_interval_rule_frozen",
            "passed": bool(
                CHRONOLOGICAL_SPLIT_TABLE[
                    "interval_convention"
                ].eq("[start, end)").all()
            ),
            "evidence": "[start, end)",
        },
    ]
)


display(
    SESSION_ORDER_TABLE[
        [
            "session_order",
            "connection_session_id",
            "first_collector_sequence",
            "last_collector_sequence",
            "row_count",
        ]
    ]
)

display(CHRONOLOGICAL_SPLIT_TABLE)
display(SPLIT_ACCESS_MATRIX)
display(SPLIT_HISTORY_RULES)
display(CHRONOLOGICAL_SPLIT_GATES)


failed_split_gates = (
    CHRONOLOGICAL_SPLIT_GATES.loc[
        ~CHRONOLOGICAL_SPLIT_GATES[
            "passed"
        ]
    ]
)

if not failed_split_gates.empty:
    failure_text = ", ".join(
        f"{row.gate}: {row.evidence}"
        for row in failed_split_gates.itertuples(
            index=False
        )
    )

    raise RuntimeError(
        "Chronological split contract failed: "
        + failure_text
    )


CHRONOLOGICAL_SPLIT_CONTRACT_HASH = (
    sha256_of_canonical_object(
        CHRONOLOGICAL_SPLIT_CONTRACT
    )
)


print("Chronological split contract: PASS")
print(f"Split method: {split_method}")
print(
    "Partition order: "
    + " -> ".join(partition_order)
)
print(
    "Primary boundary authority: "
    "collector_sequence"
)
print(
    "Partitions treated as independent: "
    f"{CHRONOLOGICAL_SPLIT_CONTRACT['partitions_are_independent']}"
)
print(
    "Claim-bearing holdout available: "
    f"{CHRONOLOGICAL_SPLIT_CONTRACT['claim_bearing_holdout_available']}"
)
print(
    "Chronological split contract hash: "
    f"{CHRONOLOGICAL_SPLIT_CONTRACT_HASH}"
)

,session_order,connection_session_id,first_collector_sequence,last_collector_sequence,row_count
0,1,c8b5bf127a7a44669514acfda8634107,1,103677,103677


,partition_order,partition,boundary_authority,collector_sequence_start,collector_sequence_end_exclusive,collector_sequence_last_included,receipt_time_start_ns,receipt_time_end_ns_exclusive,receipt_time_start_utc,receipt_time_end_utc_exclusive,row_count,trade_row_count,depth_row_count,session_count,independent_partition,claim_authority,interval_convention
0,1,DEVELOPMENT,INTRA_SESSION_COLLECTOR_SEQUENCE,1,51840,51839,1783665467531985400,1783667269391572100,2026-07-10T06:37:47.531985400+00:00,2026-07-10T07:07:49.391572100+00:00,51839,33820,18019,1,False,False,"[start, end)"
1,2,CALIBRATION,INTRA_SESSION_COLLECTOR_SEQUENCE,51840,72576,72575,1783667269391572100,1783667989690751200,2026-07-10T07:07:49.391572100+00:00,2026-07-10T07:19:49.690751200+00:00,20736,13532,7204,1,False,False,"[start, end)"
2,3,VALIDATION,INTRA_SESSION_COLLECTOR_SEQUENCE,72576,88127,88126,1783667989690751200,1783668525057534700,2026-07-10T07:19:49.690751200+00:00,2026-07-10T07:28:45.057534700+00:00,15551,10198,5353,1,False,False,"[start, end)"
3,4,ENGINEERING_HOLDOUT,INTRA_SESSION_COLLECTOR_SEQUENCE,88127,103678,103677,1783668525057534700,1783669066749750801,2026-07-10T07:28:45.057534700+00:00,2026-07-10T07:37:46.749750801+00:00,15551,10133,5418,1,False,False,"[start, end)"


,partition,model_selection_access,parameter_estimation_access,final_evaluation_access,claim_authority
0,DEVELOPMENT,True,True,True,False
1,CALIBRATION,True,True,True,False
2,VALIDATION,True,False,True,False
3,ENGINEERING_HOLDOUT,False,False,True,False


,stage,history_carry_allowed,history_usage,scored_pre_boundary_events
0,BOOK_RECONSTRUCTION,True,Carry the last valid reconstructed book state ...,False
1,TRADE_BOOK_ALIGNMENT,True,Use only book states causally available before...,False
2,MARKET_STATE_FEATURES,True,Pre-boundary observations may initialize rolli...,False
3,HAWKES_INTENSITY,True,Earlier events may initialize excitation histo...,False
4,MODEL_SELECTION,False,The final reserved partition cannot influence ...,False


,gate,passed,evidence
0,all_records_assigned,True,assigned=103677/103677
1,partitions_nonempty,True,DEVELOPMENT=51839 | CALIBRATION=20736 | VALIDA...
2,collector_sequence_intervals_contiguous,True,half-open sequence intervals
3,complete_sequence_coverage,True,source=1..103677; partition_rows=103677
4,receipt_time_monotone,True,reversals=0
5,final_partition_excluded_from_selection,True,partition=ENGINEERING_HOLDOUT
6,holdout_authority_matches_operating_mode,True,operating_mode=ENGINEERING_REPRODUCTION_MODE
7,half_open_interval_rule_frozen,True,"[start, end)"


Chronological split contract: PASS
Split method: INTRA_SESSION_ENGINEERING
Partition order: DEVELOPMENT -> CALIBRATION -> VALIDATION -> ENGINEERING_HOLDOUT
Primary boundary authority: collector_sequence
Partitions treated as independent: False
Claim-bearing holdout available: False
Chronological split contract hash: 3b7e8d46f43d8a4dd7b7a1f3d9a61c2c92d3f3643042a8b01968ddc7e0fb4624


In [23]:
# ============================================================
# FIXED DECISIONS, EXPLORATORY DECISIONS, AND TOLERANCES
# ============================================================

FIXED_DECISION_ROWS = [
    {
        "decision_id": "SOURCE_V0_0_IMMUTABLE",
        "decision_group": "SOURCE_AUTHORITY",
        "value": True,
        "locked": True,
        "authority": "NOTEBOOK_00",
        "description": (
            "All files under the registered V0.0 root are read-only "
            "inputs and provenance."
        ),
    },
    {
        "decision_id": "MARKET_VENUE",
        "decision_group": "MARKET",
        "value": "BINANCE_SPOT",
        "locked": True,
        "authority": "NOTEBOOK_00",
        "description": "The authoritative venue is Binance Spot.",
    },
    {
        "decision_id": "MARKET_SYMBOL",
        "decision_group": "MARKET",
        "value": "BTCUSDT",
        "locked": True,
        "authority": "NOTEBOOK_00",
        "description": "The authoritative market symbol is BTCUSDT.",
    },
    {
        "decision_id": "BOOK_REPRESENTATION",
        "decision_group": "MARKET",
        "value": "VISIBLE_MARKET_BY_PRICE",
        "locked": True,
        "authority": "NOTEBOOK_00",
        "description": (
            "The reconstructed book represents visible market-by-price "
            "depth only."
        ),
    },
    {
        "decision_id": "CANONICAL_TIMEZONE",
        "decision_group": "TIME",
        "value": "UTC",
        "locked": True,
        "authority": "NOTEBOOK_00",
        "description": "All rendered timestamps use UTC.",
    },
    {
        "decision_id": "PRIMARY_CAUSAL_ORDER",
        "decision_group": "ORDERING",
        "value": "collector_sequence",
        "locked": True,
        "authority": "NOTEBOOK_00",
        "description": (
            "Collector sequence is the primary cross-stream causal "
            "ordering authority."
        ),
    },
    {
        "decision_id": "TIMESTAMP_JITTER_ALLOWED",
        "decision_group": "ORDERING",
        "value": False,
        "locked": True,
        "authority": "NOTEBOOK_00",
        "description": (
            "Equal timestamps remain equal; arbitrary timestamp jitter "
            "is prohibited."
        ),
    },
    {
        "decision_id": "FORWARD_JOIN_ALLOWED",
        "decision_group": "CAUSALITY",
        "value": False,
        "locked": True,
        "authority": "NOTEBOOK_00",
        "description": (
            "A trade or event may not use a future book state."
        ),
    },
    {
        "decision_id": "SPLIT_INTERVAL_CONVENTION",
        "decision_group": "SPLITS",
        "value": "[start, end)",
        "locked": True,
        "authority": "NOTEBOOK_00",
        "description": (
            "All chronological partitions use half-open intervals."
        ),
    },
    {
        "decision_id": "SPLIT_METHOD",
        "decision_group": "SPLITS",
        "value": split_method,
        "locked": True,
        "authority": "NOTEBOOK_00",
        "description": (
            "The split method is fixed by the available session count "
            "and operating mode."
        ),
    },
    {
        "decision_id": "PARTITIONS_INDEPENDENT",
        "decision_group": "SPLITS",
        "value": bool(
            CHRONOLOGICAL_SPLIT_CONTRACT[
                "partitions_are_independent"
            ]
        ),
        "locked": True,
        "authority": "NOTEBOOK_00",
        "description": (
            "Intra-session engineering partitions are not independent "
            "statistical samples."
        ),
    },
    {
        "decision_id": "CLAIM_BEARING_HOLDOUT_AVAILABLE",
        "decision_group": "SPLITS",
        "value": bool(
            CHRONOLOGICAL_SPLIT_CONTRACT[
                "claim_bearing_holdout_available"
            ]
        ),
        "locked": True,
        "authority": "NOTEBOOK_00",
        "description": (
            "A claim-bearing holdout requires an independent session."
        ),
    },
    {
        "decision_id": "NUMERIC_IMPUTATION_ALLOWED",
        "decision_group": "MISSING_DATA",
        "value": False,
        "locked": True,
        "authority": "NOTEBOOK_00",
        "description": "Numeric imputation is prohibited.",
    },
    {
        "decision_id": "TIMESTAMP_SYNTHESIS_ALLOWED",
        "decision_group": "MISSING_DATA",
        "value": False,
        "locked": True,
        "authority": "NOTEBOOK_00",
        "description": "Missing timestamps may not be synthesized.",
    },
    {
        "decision_id": "DEPTH_ZERO_QUANTITY_SEMANTICS",
        "decision_group": "BOOK_RECONSTRUCTION",
        "value": "DELETE_PRICE_LEVEL",
        "locked": True,
        "authority": "NOTEBOOK_00",
        "description": (
            "A valid zero-quantity depth update deletes the price level."
        ),
    },
    {
        "decision_id": "DEPTH_GAP_REPAIR_ALLOWED",
        "decision_group": "BOOK_RECONSTRUCTION",
        "value": False,
        "locked": True,
        "authority": "NOTEBOOK_00",
        "description": (
            "Exchange update-ID gaps may not be silently repaired."
        ),
    },
    {
        "decision_id": "SNAPSHOT_SORTING_ALLOWED",
        "decision_group": "BOOK_RECONSTRUCTION",
        "value": False,
        "locked": True,
        "authority": "NOTEBOOK_00",
        "description": (
            "Invalid snapshot ordering cannot be repaired by sorting."
        ),
    },
    {
        "decision_id": "PRICE_QUANTITY_AUTHORITY",
        "decision_group": "NUMERIC",
        "value": "EXACT_DECIMAL",
        "locked": True,
        "authority": "NOTEBOOK_00",
        "description": (
            "Raw prices and quantities retain exact decimal authority."
        ),
    },
    {
        "decision_id": "RANDOM_SEED",
        "decision_group": "REPRODUCIBILITY",
        "value": int(RANDOM_SEED),
        "locked": True,
        "authority": "NOTEBOOK_00",
        "description": (
            "All stochastic downstream procedures must use the "
            "registered deterministic seed."
        ),
    },
]


EXPLORATORY_DECISION_ROWS = [
    {
        "decision_id": "TRADE_EVENT_REPRESENTATION",
        "decision_group": "EVENT_DEFINITION",
        "allowed_values": [
            "INDIVIDUAL_TRADE_PRINTS",
            "SAME_TIMESTAMP_SIDE_BURSTS",
        ],
        "default_value": "INDIVIDUAL_TRADE_PRINTS",
        "selection_partitions": [
            "DEVELOPMENT",
            "CALIBRATION",
        ],
        "final_partition_access": False,
        "description": (
            "Candidate point-process event definitions may be compared "
            "without altering the raw trade dataset."
        ),
    },
    {
        "decision_id": "HAWKES_KERNEL_FAMILY",
        "decision_group": "MODEL",
        "allowed_values": [
            "SINGLE_EXPONENTIAL",
            "FINITE_EXPONENTIAL_SUM",
        ],
        "default_value": "SINGLE_EXPONENTIAL",
        "selection_partitions": [
            "DEVELOPMENT",
            "CALIBRATION",
            "VALIDATION",
        ],
        "final_partition_access": False,
        "description": (
            "Ordinary bivariate Hawkes kernel candidates may be "
            "compared chronologically."
        ),
    },
    {
        "decision_id": "HAWKES_DECAY_GRID",
        "decision_group": "MODEL",
        "allowed_values": [
            "DATA_DRIVEN_CANDIDATE_GRID",
        ],
        "default_value": "DATA_DRIVEN_CANDIDATE_GRID",
        "selection_partitions": [
            "DEVELOPMENT",
            "CALIBRATION",
        ],
        "final_partition_access": False,
        "description": (
            "Decay candidates must be documented before validation "
            "comparison."
        ),
    },
    {
        "decision_id": "MARKET_STATE_FEATURE_SET",
        "decision_group": "FEATURES",
        "allowed_values": [
            "SPREAD",
            "TOP_LEVEL_IMBALANCE",
            "MICROPRICE_DISLOCATION",
            "SHORT_HORIZON_REALIZED_VOLATILITY",
            "RECENT_SIGNED_FLOW",
        ],
        "default_value": [
            "SPREAD",
            "TOP_LEVEL_IMBALANCE",
            "MICROPRICE_DISLOCATION",
        ],
        "selection_partitions": [
            "DEVELOPMENT",
            "CALIBRATION",
        ],
        "final_partition_access": False,
        "description": (
            "Feature selection must remain causal and cannot use future "
            "labels."
        ),
    },
    {
        "decision_id": "SIGNAL_NORMALIZATION",
        "decision_group": "SIGNAL",
        "allowed_values": [
            "RAW_INTENSITY",
            "BASELINE_RATIO",
            "LOG_INTENSITY_RATIO",
            "BUY_SELL_INTENSITY_IMBALANCE",
        ],
        "default_value": "BUY_SELL_INTENSITY_IMBALANCE",
        "selection_partitions": [
            "DEVELOPMENT",
            "CALIBRATION",
            "VALIDATION",
        ],
        "final_partition_access": False,
        "description": (
            "Intensity transformations may be compared before final "
            "engineering evaluation."
        ),
    },
    {
        "decision_id": "OPTIONAL_STATE_DEPENDENT_EXTENSION",
        "decision_group": "MODEL_EXTENSION",
        "allowed_values": [
            "DISABLED",
            "DIAGNOSTIC_ONLY",
        ],
        "default_value": "DISABLED",
        "selection_partitions": [
            "DEVELOPMENT",
            "CALIBRATION",
        ],
        "final_partition_access": False,
        "description": (
            "State-dependent Hawkes cannot replace the ordinary "
            "bivariate model unless the ordinary model first passes."
        ),
    },
]


FIXED_DECISION_REGISTRY = pd.DataFrame(
    FIXED_DECISION_ROWS
)

EXPLORATORY_DECISION_REGISTRY = pd.DataFrame(
    EXPLORATORY_DECISION_ROWS
)


# ============================================================
# NUMERICAL AND INTEGRITY TOLERANCE CONTRACT
# ============================================================

TOLERANCE_ROWS = [
    {
        "tolerance_id": "SHA256_MISMATCH_COUNT",
        "category": "SOURCE_INTEGRITY",
        "value": 0,
        "unit": "files",
        "comparison": "EQUAL",
        "critical": True,
    },
    {
        "tolerance_id": "COLLECTOR_SEQUENCE_GAP_COUNT",
        "category": "ORDERING",
        "value": 0,
        "unit": "sequence_values",
        "comparison": "EQUAL",
        "critical": True,
    },
    {
        "tolerance_id": "COLLECTOR_SEQUENCE_DUPLICATE_COUNT",
        "category": "ORDERING",
        "value": 0,
        "unit": "records",
        "comparison": "EQUAL",
        "critical": True,
    },
    {
        "tolerance_id": "COLLECTOR_SEQUENCE_REVERSAL_COUNT",
        "category": "ORDERING",
        "value": 0,
        "unit": "records",
        "comparison": "EQUAL",
        "critical": True,
    },
    {
        "tolerance_id": "LOCAL_RECEIPT_TIME_REVERSAL_COUNT",
        "category": "ORDERING",
        "value": 0,
        "unit": "records",
        "comparison": "EQUAL",
        "critical": True,
    },
    {
        "tolerance_id": "DEPTH_INVALID_JSON_COUNT",
        "category": "DEPTH_STREAM",
        "value": 0,
        "unit": "records",
        "comparison": "EQUAL",
        "critical": True,
    },
    {
        "tolerance_id": "DEPTH_INVALID_LEVEL_COUNT",
        "category": "DEPTH_STREAM",
        "value": 0,
        "unit": "levels",
        "comparison": "EQUAL",
        "critical": True,
    },
    {
        "tolerance_id": "DEPTH_UPDATE_ID_GAP_COUNT",
        "category": "BOOK_RECONSTRUCTION",
        "value": 0,
        "unit": "gaps",
        "comparison": "EQUAL",
        "critical": True,
    },
    {
        "tolerance_id": "SNAPSHOT_INVALID_LEVEL_COUNT",
        "category": "SNAPSHOT",
        "value": 0,
        "unit": "levels",
        "comparison": "EQUAL",
        "critical": True,
    },
    {
        "tolerance_id": "SNAPSHOT_CROSSED_OR_LOCKED_COUNT",
        "category": "SNAPSHOT",
        "value": 0,
        "unit": "snapshots",
        "comparison": "EQUAL",
        "critical": True,
    },
    {
        "tolerance_id": "FORWARD_JOIN_COUNT",
        "category": "CAUSALITY",
        "value": 0,
        "unit": "joins",
        "comparison": "EQUAL",
        "critical": True,
    },
    {
        "tolerance_id": "FUTURE_LABEL_FEATURE_COUNT",
        "category": "CAUSALITY",
        "value": 0,
        "unit": "features",
        "comparison": "EQUAL",
        "critical": True,
    },
    {
        "tolerance_id": "DERIVED_FLOAT_ABSOLUTE_TOLERANCE",
        "category": "NUMERICAL_RECONCILIATION",
        "value": 1e-12,
        "unit": "absolute",
        "comparison": "LESS_THAN_OR_EQUAL",
        "critical": False,
    },
    {
        "tolerance_id": "DERIVED_FLOAT_RELATIVE_TOLERANCE",
        "category": "NUMERICAL_RECONCILIATION",
        "value": 1e-10,
        "unit": "relative",
        "comparison": "LESS_THAN_OR_EQUAL",
        "critical": False,
    },
]


TOLERANCE_REGISTRY = pd.DataFrame(
    TOLERANCE_ROWS
)


DECISION_AND_TOLERANCE_CONTRACT = {
    "contract_schema_version": CONTRACT_SCHEMA_VERSION,
    "pipeline_version": PIPELINE_VERSION,
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "source_set_hash": SOURCE_SET_HASH,
    "operating_mode": V0_1_OPERATING_MODE,
    "market_data_contract_hash": MARKET_DATA_CONTRACT_HASH,
    "missing_data_rejection_contract_hash": (
        MISSING_DATA_REJECTION_CONTRACT_HASH
    ),
    "chronological_split_contract_hash": (
        CHRONOLOGICAL_SPLIT_CONTRACT_HASH
    ),
    "fixed_decisions": (
        FIXED_DECISION_REGISTRY
        .sort_values("decision_id", kind="stable")
        .to_dict(orient="records")
    ),
    "exploratory_decisions": (
        EXPLORATORY_DECISION_REGISTRY
        .sort_values("decision_id", kind="stable")
        .to_dict(orient="records")
    ),
    "tolerances": (
        TOLERANCE_REGISTRY
        .sort_values("tolerance_id", kind="stable")
        .to_dict(orient="records")
    ),
    "exploration_rules": {
        "may_override_fixed_decisions": False,
        "may_modify_raw_inputs": False,
        "may_use_future_labels": False,
        "may_access_final_partition_for_selection": False,
        "must_record_candidate_grid": True,
        "must_record_selected_candidate": True,
        "must_record_rejected_candidates": True,
    },
    "producing_notebook": NOTEBOOK_FILENAME,
}


# ============================================================
# VALIDATION
# ============================================================

duplicate_fixed_ids = FIXED_DECISION_REGISTRY.loc[
    FIXED_DECISION_REGISTRY[
        "decision_id"
    ].duplicated(keep=False)
]

duplicate_exploratory_ids = (
    EXPLORATORY_DECISION_REGISTRY.loc[
        EXPLORATORY_DECISION_REGISTRY[
            "decision_id"
        ].duplicated(keep=False)
    ]
)

decision_id_overlap = sorted(
    set(FIXED_DECISION_REGISTRY["decision_id"])
    & set(EXPLORATORY_DECISION_REGISTRY["decision_id"])
)

duplicate_tolerance_ids = TOLERANCE_REGISTRY.loc[
    TOLERANCE_REGISTRY[
        "tolerance_id"
    ].duplicated(keep=False)
]

invalid_tolerances = TOLERANCE_REGISTRY.loc[
    pd.to_numeric(
        TOLERANCE_REGISTRY["value"],
        errors="coerce",
    ).isna()
    | pd.to_numeric(
        TOLERANCE_REGISTRY["value"],
        errors="coerce",
    ).lt(0)
]

unlocked_fixed_decisions = FIXED_DECISION_REGISTRY.loc[
    ~FIXED_DECISION_REGISTRY["locked"]
]

exploratory_final_access_violations = (
    EXPLORATORY_DECISION_REGISTRY.loc[
        EXPLORATORY_DECISION_REGISTRY[
            "final_partition_access"
        ]
    ]
)

exact_integrity_tolerances = TOLERANCE_REGISTRY.loc[
    TOLERANCE_REGISTRY["critical"]
]

critical_tolerances_are_zero = bool(
    pd.to_numeric(
        exact_integrity_tolerances["value"],
        errors="coerce",
    ).eq(0).all()
)


DECISION_TOLERANCE_GATES = pd.DataFrame(
    [
        {
            "gate": "fixed_decision_ids_unique",
            "passed": duplicate_fixed_ids.empty,
            "evidence": (
                f"rows={len(FIXED_DECISION_REGISTRY)}; "
                f"unique="
                f"{FIXED_DECISION_REGISTRY['decision_id'].nunique()}"
            ),
        },
        {
            "gate": "exploratory_decision_ids_unique",
            "passed": duplicate_exploratory_ids.empty,
            "evidence": (
                f"rows={len(EXPLORATORY_DECISION_REGISTRY)}; "
                f"unique="
                f"{EXPLORATORY_DECISION_REGISTRY['decision_id'].nunique()}"
            ),
        },
        {
            "gate": "fixed_and_exploratory_ids_disjoint",
            "passed": len(decision_id_overlap) == 0,
            "evidence": (
                "none"
                if not decision_id_overlap
                else " | ".join(decision_id_overlap)
            ),
        },
        {
            "gate": "all_fixed_decisions_locked",
            "passed": unlocked_fixed_decisions.empty,
            "evidence": (
                f"unlocked={len(unlocked_fixed_decisions)}"
            ),
        },
        {
            "gate": "final_partition_protected_from_exploration",
            "passed": exploratory_final_access_violations.empty,
            "evidence": (
                f"violations="
                f"{len(exploratory_final_access_violations)}"
            ),
        },
        {
            "gate": "tolerance_ids_unique",
            "passed": duplicate_tolerance_ids.empty,
            "evidence": (
                f"rows={len(TOLERANCE_REGISTRY)}; "
                f"unique="
                f"{TOLERANCE_REGISTRY['tolerance_id'].nunique()}"
            ),
        },
        {
            "gate": "tolerances_nonnegative_and_numeric",
            "passed": invalid_tolerances.empty,
            "evidence": (
                f"invalid={len(invalid_tolerances)}"
            ),
        },
        {
            "gate": "critical_integrity_tolerances_are_zero",
            "passed": critical_tolerances_are_zero,
            "evidence": (
                f"critical_rows="
                f"{len(exact_integrity_tolerances)}"
            ),
        },
        {
            "gate": "operating_mode_consistent",
            "passed": (
                DECISION_AND_TOLERANCE_CONTRACT[
                    "operating_mode"
                ]
                == V0_1_OPERATING_MODE
            ),
            "evidence": V0_1_OPERATING_MODE,
        },
    ]
)


display(
    FIXED_DECISION_REGISTRY[
        [
            "decision_id",
            "decision_group",
            "value",
            "locked",
            "authority",
        ]
    ]
)

display(
    EXPLORATORY_DECISION_REGISTRY[
        [
            "decision_id",
            "decision_group",
            "default_value",
            "selection_partitions",
            "final_partition_access",
        ]
    ]
)

display(TOLERANCE_REGISTRY)
display(DECISION_TOLERANCE_GATES)


failed_decision_tolerance_gates = (
    DECISION_TOLERANCE_GATES.loc[
        ~DECISION_TOLERANCE_GATES["passed"]
    ]
)

if not failed_decision_tolerance_gates.empty:
    failure_text = ", ".join(
        f"{row.gate}: {row.evidence}"
        for row in failed_decision_tolerance_gates.itertuples(
            index=False
        )
    )

    raise RuntimeError(
        "Decision and tolerance contract failed: "
        + failure_text
    )


DECISION_AND_TOLERANCE_CONTRACT_HASH = (
    sha256_of_canonical_object(
        DECISION_AND_TOLERANCE_CONTRACT
    )
)


print("Decision and tolerance contract: PASS")
print(
    "Fixed decisions registered: "
    f"{len(FIXED_DECISION_REGISTRY):,}"
)
print(
    "Exploratory decisions registered: "
    f"{len(EXPLORATORY_DECISION_REGISTRY):,}"
)
print(
    "Tolerance rules registered: "
    f"{len(TOLERANCE_REGISTRY):,}"
)
print(
    "Final partition available for exploratory selection: False"
)
print(
    "Decision and tolerance contract hash: "
    f"{DECISION_AND_TOLERANCE_CONTRACT_HASH}"
)

,decision_id,decision_group,value,locked,authority
0,SOURCE_V0_0_IMMUTABLE,SOURCE_AUTHORITY,True,True,NOTEBOOK_00
1,MARKET_VENUE,MARKET,BINANCE_SPOT,True,NOTEBOOK_00
2,MARKET_SYMBOL,MARKET,BTCUSDT,True,NOTEBOOK_00
3,BOOK_REPRESENTATION,MARKET,VISIBLE_MARKET_BY_PRICE,True,NOTEBOOK_00
4,CANONICAL_TIMEZONE,TIME,UTC,True,NOTEBOOK_00
5,PRIMARY_CAUSAL_ORDER,ORDERING,collector_sequence,True,NOTEBOOK_00
6,TIMESTAMP_JITTER_ALLOWED,ORDERING,False,True,NOTEBOOK_00
7,FORWARD_JOIN_ALLOWED,CAUSALITY,False,True,NOTEBOOK_00
8,SPLIT_INTERVAL_CONVENTION,SPLITS,"[start, end)",True,NOTEBOOK_00
9,SPLIT_METHOD,SPLITS,INTRA_SESSION_ENGINEERING,True,NOTEBOOK_00


,decision_id,decision_group,default_value,selection_partitions,final_partition_access
0,TRADE_EVENT_REPRESENTATION,EVENT_DEFINITION,INDIVIDUAL_TRADE_PRINTS,"[DEVELOPMENT, CALIBRATION]",False
1,HAWKES_KERNEL_FAMILY,MODEL,SINGLE_EXPONENTIAL,"[DEVELOPMENT, CALIBRATION, VALIDATION]",False
2,HAWKES_DECAY_GRID,MODEL,DATA_DRIVEN_CANDIDATE_GRID,"[DEVELOPMENT, CALIBRATION]",False
3,MARKET_STATE_FEATURE_SET,FEATURES,"[SPREAD, TOP_LEVEL_IMBALANCE, MICROPRICE_DISLO...","[DEVELOPMENT, CALIBRATION]",False
4,SIGNAL_NORMALIZATION,SIGNAL,BUY_SELL_INTENSITY_IMBALANCE,"[DEVELOPMENT, CALIBRATION, VALIDATION]",False
5,OPTIONAL_STATE_DEPENDENT_EXTENSION,MODEL_EXTENSION,DISABLED,"[DEVELOPMENT, CALIBRATION]",False


,tolerance_id,category,value,unit,comparison,critical
0,SHA256_MISMATCH_COUNT,SOURCE_INTEGRITY,0.000000e+00,files,EQUAL,True
1,COLLECTOR_SEQUENCE_GAP_COUNT,ORDERING,0.000000e+00,sequence_values,EQUAL,True
2,COLLECTOR_SEQUENCE_DUPLICATE_COUNT,ORDERING,0.000000e+00,records,EQUAL,True
3,COLLECTOR_SEQUENCE_REVERSAL_COUNT,ORDERING,0.000000e+00,records,EQUAL,True
4,LOCAL_RECEIPT_TIME_REVERSAL_COUNT,ORDERING,0.000000e+00,records,EQUAL,True
5,DEPTH_INVALID_JSON_COUNT,DEPTH_STREAM,0.000000e+00,records,EQUAL,True
6,DEPTH_INVALID_LEVEL_COUNT,DEPTH_STREAM,0.000000e+00,levels,EQUAL,True
7,DEPTH_UPDATE_ID_GAP_COUNT,BOOK_RECONSTRUCTION,0.000000e+00,gaps,EQUAL,True
8,SNAPSHOT_INVALID_LEVEL_COUNT,SNAPSHOT,0.000000e+00,levels,EQUAL,True
9,SNAPSHOT_CROSSED_OR_LOCKED_COUNT,SNAPSHOT,0.000000e+00,snapshots,EQUAL,True


,gate,passed,evidence
0,fixed_decision_ids_unique,True,rows=19; unique=19
1,exploratory_decision_ids_unique,True,rows=6; unique=6
2,fixed_and_exploratory_ids_disjoint,True,none
3,all_fixed_decisions_locked,True,unlocked=0
4,final_partition_protected_from_exploration,True,violations=0
5,tolerance_ids_unique,True,rows=14; unique=14
6,tolerances_nonnegative_and_numeric,True,invalid=0
7,critical_integrity_tolerances_are_zero,True,critical_rows=12
8,operating_mode_consistent,True,ENGINEERING_REPRODUCTION_MODE


Decision and tolerance contract: PASS
Fixed decisions registered: 19
Exploratory decisions registered: 6
Tolerance rules registered: 14
Final partition available for exploratory selection: False
Decision and tolerance contract hash: 31e054874e4072f0d90a0f7dfd14e1531f54f22e3c5beec824534b53a75c43b2


In [24]:
# ============================================================
# SOFTWARE ENVIRONMENT AND REPOSITORY SNAPSHOT
# ============================================================

from importlib import metadata as importlib_metadata
import socket


# ============================================================
# SAFE COMMAND EXECUTION
# ============================================================

def run_command(
    command: list[str],
    cwd: Path | None = None,
    timeout_seconds: int = 15,
) -> dict:
    """Run one local command without raising on expected failures."""
    try:
        completed = subprocess.run(
            command,
            cwd=str(cwd) if cwd is not None else None,
            capture_output=True,
            text=True,
            check=False,
            timeout=timeout_seconds,
        )

        return {
            "available": True,
            "return_code": int(completed.returncode),
            "stdout": completed.stdout.strip(),
            "stderr": completed.stderr.strip(),
            "error": None,
        }

    except FileNotFoundError as exc:
        return {
            "available": False,
            "return_code": None,
            "stdout": "",
            "stderr": "",
            "error": str(exc),
        }

    except subprocess.TimeoutExpired as exc:
        return {
            "available": True,
            "return_code": None,
            "stdout": (
                exc.stdout.strip()
                if isinstance(exc.stdout, str)
                else ""
            ),
            "stderr": (
                exc.stderr.strip()
                if isinstance(exc.stderr, str)
                else ""
            ),
            "error": (
                f"Command timed out after "
                f"{timeout_seconds} seconds."
            ),
        }

    except OSError as exc:
        return {
            "available": False,
            "return_code": None,
            "stdout": "",
            "stderr": "",
            "error": str(exc),
        }


# ============================================================
# PYTHON AND OPERATING-SYSTEM SNAPSHOT
# ============================================================

PYTHON_VERSION_INFO = {
    "major": int(sys.version_info.major),
    "minor": int(sys.version_info.minor),
    "micro": int(sys.version_info.micro),
    "release_level": str(
        sys.version_info.releaselevel
    ),
    "serial": int(sys.version_info.serial),
}

PYTHON_VERSION_STRING = (
    f"{PYTHON_VERSION_INFO['major']}."
    f"{PYTHON_VERSION_INFO['minor']}."
    f"{PYTHON_VERSION_INFO['micro']}"
)

SYSTEM_ENVIRONMENT = {
    "python_version": PYTHON_VERSION_STRING,
    "python_implementation": (
        platform.python_implementation()
    ),
    "python_compiler": platform.python_compiler(),
    "python_executable": str(
        Path(sys.executable).resolve(strict=False)
    ),
    "operating_system": platform.system(),
    "operating_system_release": platform.release(),
    "operating_system_version": platform.version(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "architecture": platform.architecture()[0],
    "hostname": socket.gethostname(),
    "working_directory": str(
        Path.cwd().resolve(strict=False)
    ),
}


# ============================================================
# INSTALLED PACKAGE INVENTORY
# ============================================================

package_rows = []

for distribution in importlib_metadata.distributions():
    package_name = (
        distribution.metadata.get("Name")
        or distribution.metadata.get("Summary")
        or "UNKNOWN"
    )

    package_rows.append(
        {
            "package_name": str(package_name),
            "package_name_normalized": (
                str(package_name)
                .strip()
                .lower()
                .replace("_", "-")
            ),
            "version": str(distribution.version),
        }
    )


INSTALLED_PACKAGE_INVENTORY = pd.DataFrame(
    package_rows
)

if INSTALLED_PACKAGE_INVENTORY.empty:
    raise RuntimeError(
        "No installed Python packages could be inventoried."
    )

INSTALLED_PACKAGE_INVENTORY = (
    INSTALLED_PACKAGE_INVENTORY
    .sort_values(
        [
            "package_name_normalized",
            "version",
        ],
        kind="stable",
    )
    .drop_duplicates(
        subset=[
            "package_name_normalized",
            "version",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)


CORE_PACKAGE_NAMES = [
    "ipython",
    "jupyter",
    "jupyterlab",
    "numpy",
    "pandas",
    "pyarrow",
    "scipy",
    "matplotlib",
    "statsmodels",
    "scikit-learn",
]


core_package_rows = []

for package_name in CORE_PACKAGE_NAMES:
    matches = INSTALLED_PACKAGE_INVENTORY.loc[
        INSTALLED_PACKAGE_INVENTORY[
            "package_name_normalized"
        ].eq(package_name)
    ]

    if matches.empty:
        package_version = None
        package_status = "NOT_INSTALLED"
    else:
        package_version = " | ".join(
            matches["version"]
            .drop_duplicates()
            .tolist()
        )
        package_status = "AVAILABLE"

    core_package_rows.append(
        {
            "package_name": package_name,
            "version": package_version,
            "status": package_status,
        }
    )


CORE_PACKAGE_INVENTORY = pd.DataFrame(
    core_package_rows
)


# ============================================================
# GIT REPOSITORY SNAPSHOT
# ============================================================

git_version_result = run_command(
    ["git", "--version"]
)

git_repo_result = run_command(
    [
        "git",
        "-C",
        str(V0_1_ROOT_RESOLVED),
        "rev-parse",
        "--show-toplevel",
    ]
)


if (
    git_version_result["available"]
    and git_version_result["return_code"] == 0
    and git_repo_result["return_code"] == 0
):
    GIT_AVAILABLE = True

    GIT_REPOSITORY_ROOT = git_repo_result[
        "stdout"
    ]

    git_commit_result = run_command(
        [
            "git",
            "-C",
            str(V0_1_ROOT_RESOLVED),
            "rev-parse",
            "HEAD",
        ]
    )

    git_branch_result = run_command(
        [
            "git",
            "-C",
            str(V0_1_ROOT_RESOLVED),
            "branch",
            "--show-current",
        ]
    )

    git_status_result = run_command(
        [
            "git",
            "-C",
            str(V0_1_ROOT_RESOLVED),
            "status",
            "--porcelain",
            "--untracked-files=all",
        ]
    )

    git_commit_time_result = run_command(
        [
            "git",
            "-C",
            str(V0_1_ROOT_RESOLVED),
            "show",
            "-s",
            "--format=%cI",
            "HEAD",
        ]
    )

    GIT_COMMIT = (
        git_commit_result["stdout"]
        if git_commit_result["return_code"] == 0
        else None
    )

    GIT_BRANCH = (
        git_branch_result["stdout"]
        if git_branch_result["return_code"] == 0
        else None
    )

    GIT_STATUS_PORCELAIN = (
        git_status_result["stdout"]
        if git_status_result["return_code"] == 0
        else None
    )

    GIT_DIRTY = (
        bool(GIT_STATUS_PORCELAIN)
        if GIT_STATUS_PORCELAIN is not None
        else None
    )

    GIT_CHANGED_PATH_COUNT = (
        len(
            [
                line
                for line in (
                    GIT_STATUS_PORCELAIN or ""
                ).splitlines()
                if line.strip()
            ]
        )
    )

    GIT_COMMIT_TIME = (
        git_commit_time_result["stdout"]
        if git_commit_time_result["return_code"] == 0
        else None
    )

else:
    GIT_AVAILABLE = False
    GIT_REPOSITORY_ROOT = None
    GIT_COMMIT = None
    GIT_BRANCH = None
    GIT_STATUS_PORCELAIN = None
    GIT_DIRTY = None
    GIT_CHANGED_PATH_COUNT = None
    GIT_COMMIT_TIME = None


GIT_REPOSITORY_SNAPSHOT = {
    "git_available": GIT_AVAILABLE,
    "git_version": (
        git_version_result["stdout"]
        if git_version_result["return_code"] == 0
        else None
    ),
    "repository_detected": (
        GIT_REPOSITORY_ROOT is not None
    ),
    "repository_root": GIT_REPOSITORY_ROOT,
    "commit": GIT_COMMIT,
    "branch": GIT_BRANCH,
    "commit_time": GIT_COMMIT_TIME,
    "dirty": GIT_DIRTY,
    "changed_path_count": GIT_CHANGED_PATH_COUNT,
}


# ============================================================
# STABLE SOFTWARE-ENVIRONMENT IDENTITY
# ============================================================

SOFTWARE_ENVIRONMENT_IDENTITY = {
    "python": {
        "version": PYTHON_VERSION_STRING,
        "implementation": (
            platform.python_implementation()
        ),
        "compiler": platform.python_compiler(),
    },
    "platform": {
        "system": platform.system(),
        "release": platform.release(),
        "machine": platform.machine(),
        "architecture": platform.architecture()[0],
    },
    "installed_packages": (
        INSTALLED_PACKAGE_INVENTORY[
            [
                "package_name_normalized",
                "version",
            ]
        ]
        .to_dict(orient="records")
    ),
    "git": {
        "available": GIT_AVAILABLE,
        "repository_detected": (
            GIT_REPOSITORY_ROOT is not None
        ),
        "commit": GIT_COMMIT,
        "branch": GIT_BRANCH,
        "dirty": GIT_DIRTY,
    },
}


SOFTWARE_ENVIRONMENT_HASH = (
    sha256_of_canonical_object(
        SOFTWARE_ENVIRONMENT_IDENTITY
    )
)


# ============================================================
# COMPLETE EXECUTION SNAPSHOT
# ============================================================

EXECUTION_ENVIRONMENT_SNAPSHOT = {
    "snapshot_schema_version": (
        CONTRACT_SCHEMA_VERSION
    ),
    "pipeline_version": PIPELINE_VERSION,
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "execution_started_utc": EXECUTION_STARTED_UTC,
    "system_environment": SYSTEM_ENVIRONMENT,
    "software_environment_hash": (
        SOFTWARE_ENVIRONMENT_HASH
    ),
    "git_repository": GIT_REPOSITORY_SNAPSHOT,
    "package_count": int(
        len(INSTALLED_PACKAGE_INVENTORY)
    ),
    "core_packages": (
        CORE_PACKAGE_INVENTORY
        .to_dict(orient="records")
    ),
    "producing_notebook": NOTEBOOK_FILENAME,
}


# ============================================================
# ENVIRONMENT GATES
# ============================================================

minimum_python_passed = bool(
    sys.version_info >= (3, 11)
)

required_runtime_packages = {
    "numpy",
    "pandas",
}

available_package_names = set(
    INSTALLED_PACKAGE_INVENTORY[
        "package_name_normalized"
    ]
)

missing_required_runtime_packages = sorted(
    required_runtime_packages
    - available_package_names
)


ENVIRONMENT_GATES = pd.DataFrame(
    [
        {
            "gate": "python_version_supported",
            "passed": minimum_python_passed,
            "severity": "CRITICAL",
            "evidence": (
                f"observed={PYTHON_VERSION_STRING}; "
                "minimum=3.11"
            ),
        },
        {
            "gate": "required_runtime_packages_available",
            "passed": (
                len(
                    missing_required_runtime_packages
                )
                == 0
            ),
            "severity": "CRITICAL",
            "evidence": (
                "none"
                if not missing_required_runtime_packages
                else " | ".join(
                    missing_required_runtime_packages
                )
            ),
        },
        {
            "gate": "package_inventory_nonempty",
            "passed": bool(
                len(INSTALLED_PACKAGE_INVENTORY)
                > 0
            ),
            "severity": "CRITICAL",
            "evidence": (
                f"packages="
                f"{len(INSTALLED_PACKAGE_INVENTORY)}"
            ),
        },
        {
            "gate": "git_executable_available",
            "passed": GIT_AVAILABLE,
            "severity": "WARNING",
            "evidence": (
                GIT_REPOSITORY_SNAPSHOT[
                    "git_version"
                ]
                or git_version_result["error"]
                or git_version_result["stderr"]
                or "not available"
            ),
        },
        {
            "gate": "git_repository_detected",
            "passed": (
                GIT_REPOSITORY_ROOT is not None
            ),
            "severity": "WARNING",
            "evidence": (
                GIT_REPOSITORY_ROOT
                or git_repo_result["stderr"]
                or git_repo_result["error"]
                or "not detected"
            ),
        },
        {
            "gate": "git_worktree_clean",
            "passed": (
                GIT_DIRTY is False
                if GIT_DIRTY is not None
                else False
            ),
            "severity": "WARNING",
            "evidence": (
                f"dirty={GIT_DIRTY}; "
                f"changed_paths="
                f"{GIT_CHANGED_PATH_COUNT}"
            ),
        },
    ]
)


critical_environment_failures = (
    ENVIRONMENT_GATES.loc[
        ENVIRONMENT_GATES[
            "severity"
        ].eq("CRITICAL")
        & ~ENVIRONMENT_GATES["passed"]
    ]
)

environment_warnings = (
    ENVIRONMENT_GATES.loc[
        ENVIRONMENT_GATES[
            "severity"
        ].eq("WARNING")
        & ~ENVIRONMENT_GATES["passed"]
    ]
)


SYSTEM_ENVIRONMENT_TABLE = pd.DataFrame(
    [
        {
            "field": field,
            "value": value,
        }
        for field, value
        in SYSTEM_ENVIRONMENT.items()
    ]
)

GIT_ENVIRONMENT_TABLE = pd.DataFrame(
    [
        {
            "field": field,
            "value": value,
        }
        for field, value
        in GIT_REPOSITORY_SNAPSHOT.items()
    ]
)


display(SYSTEM_ENVIRONMENT_TABLE)
display(CORE_PACKAGE_INVENTORY)
display(GIT_ENVIRONMENT_TABLE)
display(ENVIRONMENT_GATES)


if not critical_environment_failures.empty:
    failure_text = ", ".join(
        f"{row.gate}: {row.evidence}"
        for row
        in critical_environment_failures.itertuples(
            index=False
        )
    )

    raise RuntimeError(
        "Software-environment registration failed: "
        + failure_text
    )


print("Software environment registration: PASS")
print(
    "Python version: "
    f"{PYTHON_VERSION_STRING}"
)
print(
    "Installed packages inventoried: "
    f"{len(INSTALLED_PACKAGE_INVENTORY):,}"
)
print(
    "Git repository detected: "
    f"{GIT_REPOSITORY_ROOT is not None}"
)
print(
    "Git worktree dirty: "
    f"{GIT_DIRTY}"
)
print(
    "Environment warnings: "
    f"{len(environment_warnings):,}"
)
print(
    "Software environment hash: "
    f"{SOFTWARE_ENVIRONMENT_HASH}"
)

,field,value
0,python_version,3.11.9
1,python_implementation,CPython
2,python_compiler,MSC v.1938 64 bit (AMD64)
3,python_executable,C:\Users\Bryan\AppData\Local\Microsoft\Windows...
4,operating_system,Windows
5,operating_system_release,10
6,operating_system_version,10.0.26200
7,machine,AMD64
8,processor,"AMD64 Family 25 Model 33 Stepping 2, AuthenticAMD"
9,architecture,64bit


,package_name,version,status
0,ipython,9.5.0,AVAILABLE
1,jupyter,1.1.1,AVAILABLE
2,jupyterlab,4.5.1,AVAILABLE
3,numpy,2.2.0,AVAILABLE
4,pandas,2.3.2,AVAILABLE
5,pyarrow,22.0.0,AVAILABLE
6,scipy,1.16.1,AVAILABLE
7,matplotlib,3.10.6,AVAILABLE
8,statsmodels,0.14.5,AVAILABLE
9,scikit-learn,1.7.1,AVAILABLE


,field,value
0,git_available,False
1,git_version,git version 2.55.0.windows.2
2,repository_detected,False
3,repository_root,None
4,commit,None
5,branch,None
6,commit_time,None
7,dirty,None
8,changed_path_count,None


,gate,passed,severity,evidence
0,python_version_supported,True,CRITICAL,observed=3.11.9; minimum=3.11
1,required_runtime_packages_available,True,CRITICAL,none
2,package_inventory_nonempty,True,CRITICAL,packages=268
3,git_executable_available,False,WARNING,git version 2.55.0.windows.2
4,git_repository_detected,False,WARNING,fatal: not a git repository (or any of the par...
5,git_worktree_clean,False,WARNING,dirty=None; changed_paths=None


Software environment registration: PASS
Python version: 3.11.9
Installed packages inventoried: 268
Git repository detected: False
Git worktree dirty: None
Environment warnings: 3
Software environment hash: c16166df27b3fe072839c2bbf60d35a41ec06a0ca5c0655a2440d82ed31f5073


In [25]:
# ============================================================
# FREEZE COMPLETE V0.1 RUN CONFIGURATION
# ============================================================

import math


DOWNSTREAM_NOTEBOOK_SEQUENCE = [
    "01_RAW_DATA_AUDIT.ipynb",
    "02_VISIBLE_BOOK_RECONSTRUCTION.ipynb",
    "03_CAUSAL_TRADE_BOOK_ALIGNMENT.ipynb",
    "04_EVENT_STREAM_CONSTRUCTION.ipynb",
    "05_MARKET_STATE_FEATURES.ipynb",
    "06_POINT_PROCESS_BASELINES.ipynb",
    "07_HAWKES_ESTIMATION.ipynb",
    "08_HAWKES_DIAGNOSTICS.ipynb",
    "09_INTENSITY_SIGNAL_VALIDATION.ipynb",
    "10_V01_FINAL_AUDIT_AND_HANDOFF.ipynb",
]


# ============================================================
# JSON-SAFE NORMALIZATION
# ============================================================

def to_json_safe(value):
    """
    Convert supported Python, NumPy, pandas, Decimal, datetime,
    and pathlib values into deterministic JSON-compatible values.

    Non-finite floating-point values are rejected rather than silently
    converted.
    """
    if value is None:
        return None

    if isinstance(value, (str, bool, int)):
        return value

    if isinstance(value, float):
        if not math.isfinite(value):
            raise ValueError(
                f"Non-finite float cannot enter frozen config: {value}"
            )

        return value

    if isinstance(value, np.generic):
        return to_json_safe(value.item())

    if isinstance(value, Decimal):
        if not value.is_finite():
            raise ValueError(
                f"Non-finite Decimal cannot enter frozen config: {value}"
            )

        return str(value)

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, pd.Timestamp):
        if pd.isna(value):
            raise ValueError(
                "NaT cannot enter the frozen configuration."
            )

        return value.isoformat()

    if isinstance(value, datetime):
        return value.isoformat()

    if isinstance(value, dict):
        return {
            str(key): to_json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            to_json_safe(item)
            for item in value
        ]

    if isinstance(value, set):
        return [
            to_json_safe(item)
            for item in sorted(
                value,
                key=lambda item: str(item),
            )
        ]

    raise TypeError(
        "Unsupported value in frozen configuration: "
        f"{type(value).__name__}"
    )


# ============================================================
# CONFIGURATION COMPONENT HASH REGISTRY
# ============================================================

RUN_CONFIG_COMPONENT_HASHES = {
    "source_set_hash": SOURCE_SET_HASH,
    "source_identity_declaration_hash": (
        SOURCE_IDENTITY_DECLARATION_HASH
    ),
    "market_data_contract_hash": (
        MARKET_DATA_CONTRACT_HASH
    ),
    "missing_data_rejection_contract_hash": (
        MISSING_DATA_REJECTION_CONTRACT_HASH
    ),
    "chronological_split_contract_hash": (
        CHRONOLOGICAL_SPLIT_CONTRACT_HASH
    ),
    "decision_and_tolerance_contract_hash": (
        DECISION_AND_TOLERANCE_CONTRACT_HASH
    ),
    "software_environment_hash": (
        SOFTWARE_ENVIRONMENT_HASH
    ),
}


invalid_component_hashes = {
    name: digest
    for name, digest in RUN_CONFIG_COMPONENT_HASHES.items()
    if (
        not isinstance(digest, str)
        or SHA256_PATTERN.fullmatch(digest) is None
    )
}

if invalid_component_hashes:
    raise RuntimeError(
        "Invalid component hashes prevent configuration freeze: "
        + ", ".join(
            sorted(invalid_component_hashes)
        )
    )


# ============================================================
# PRIMARY-SOURCE REGISTRATION FOR THE RUN CONFIG
# ============================================================

RUN_CONFIG_PRIMARY_SOURCES = (
    AUTHORITATIVE_RAW_SOURCE_MANIFEST[
        [
            "source_role",
            "relative_path",
            "size_bytes",
            "sha256",
            "acceptance_status",
        ]
    ]
    .sort_values(
        "source_role",
        kind="stable",
    )
    .to_dict(orient="records")
)


# ============================================================
# PERMISSION REGISTRATION
# ============================================================

RUN_CONFIG_PERMISSIONS = {
    str(row.permission): bool(row.allowed)
    for row in CLAIM_PERMISSION_MATRIX.itertuples(
        index=False
    )
}


# ============================================================
# FROZEN V0.1 RUN CONFIGURATION
# ============================================================

V0_1_RUN_CONFIG = {
    "config_type": "V0_1_RUN_CONFIGURATION",
    "config_schema_version": CONTRACT_SCHEMA_VERSION,
    "pipeline_name": PIPELINE_NAME,
    "pipeline_version": PIPELINE_VERSION,
    "producing_notebook": NOTEBOOK_FILENAME,

    "source_identity": {
        "source_run_prefix": SOURCE_RUN_PREFIX,
        "source_set_hash": SOURCE_SET_HASH,
        "source_identity_declaration_hash": (
            SOURCE_IDENTITY_DECLARATION_HASH
        ),
        "primary_file_count": int(
            len(AUTHORITATIVE_RAW_SOURCE_MANIFEST)
        ),
        "primary_sources": RUN_CONFIG_PRIMARY_SOURCES,
    },

    "operating_authority": {
        "operating_mode": V0_1_OPERATING_MODE,
        "contract_status": V0_1_CONTRACT_STATUS,
        "partitions_are_independent": bool(
            CHRONOLOGICAL_SPLIT_CONTRACT[
                "partitions_are_independent"
            ]
        ),
        "claim_bearing_holdout_available": bool(
            CHRONOLOGICAL_SPLIT_CONTRACT[
                "claim_bearing_holdout_available"
            ]
        ),
        "permissions": RUN_CONFIG_PERMISSIONS,
    },

    "market": {
        "venue": MARKET_DATA_CONTRACT[
            "market"
        ]["venue"],
        "venue_segment": MARKET_DATA_CONTRACT[
            "market"
        ]["venue_segment"],
        "symbol": MARKET_DATA_CONTRACT[
            "market"
        ]["symbol"],
        "book_representation": MARKET_DATA_CONTRACT[
            "book_representation"
        ]["type"],
        "canonical_timezone": MARKET_DATA_CONTRACT[
            "time_contract"
        ]["canonical_timezone"],
        "primary_causal_order": MARKET_DATA_CONTRACT[
            "ordering_contract"
        ]["primary_causal_order"],
        "snapshot_last_update_id": int(
            SNAPSHOT_LAST_UPDATE_ID
        ),
    },

    "chronological_partitions": {
        "split_method": split_method,
        "interval_convention": "[start, end)",
        "partition_order": list(partition_order),
        "boundary_table": (
            CHRONOLOGICAL_SPLIT_TABLE
            .sort_values(
                "partition_order",
                kind="stable",
            )
            .to_dict(orient="records")
        ),
        "access_matrix": (
            SPLIT_ACCESS_MATRIX
            .to_dict(orient="records")
        ),
        "history_rules": (
            SPLIT_HISTORY_RULES
            .to_dict(orient="records")
        ),
    },

    "reproducibility": {
        "random_seed": int(RANDOM_SEED),
        "hash_algorithm": HASH_ALGORITHM,
        "software_environment_hash": (
            SOFTWARE_ENVIRONMENT_HASH
        ),
        "python_version": PYTHON_VERSION_STRING,
        "git_repository_detected": bool(
            GIT_REPOSITORY_ROOT is not None
        ),
        "git_commit": GIT_COMMIT,
        "git_branch": GIT_BRANCH,
        "git_dirty": GIT_DIRTY,
    },

    "output_contract": {
        "authoritative_root": str(
            V0_1_ROOT_RESOLVED
        ),
        "config_root": str(
            resolve_path(V0_1_CONFIG_ROOT)
        ),
        "manifest_root": str(
            resolve_path(V0_1_MANIFEST_ROOT)
        ),
        "audit_root": str(
            resolve_path(V0_1_AUDIT_ROOT)
        ),
        "naming_template": (
            "{source_run_prefix}__{v0_1_run_id}__"
            "{producing_notebook}__{artifact_name}."
            "{extension}"
        ),
        "required_artifact_fields": [
            "source_run_prefix",
            "v0_1_run_id",
            "producing_notebook",
            "schema_version",
            "source_files",
            "input_checksums",
            "input_row_count",
            "output_row_count",
            "acceptance_status",
        ],
        "atomic_write_required": True,
        "overwrite_v0_0_allowed": False,
    },

    "pipeline_sequence": {
        "current_notebook": NOTEBOOK_FILENAME,
        "downstream_notebooks": (
            DOWNSTREAM_NOTEBOOK_SEQUENCE
        ),
        "next_notebook": (
            DOWNSTREAM_NOTEBOOK_SEQUENCE[0]
        ),
        "market_making_stage_included": False,
        "v0_1_terminal_stage": (
            "CLEAN_INTENSITY_SIGNAL_HANDOFF"
        ),
    },

    "component_hashes": RUN_CONFIG_COMPONENT_HASHES,
}


V0_1_RUN_CONFIG = to_json_safe(
    V0_1_RUN_CONFIG
)

V0_1_RUN_CONFIG_HASH = (
    sha256_of_canonical_object(
        V0_1_RUN_CONFIG
    )
)


# ============================================================
# CONFIGURATION FREEZE SUMMARY
# ============================================================

RUN_CONFIG_HASH_TABLE = pd.DataFrame(
    [
        {
            "component": component,
            "sha256": digest,
        }
        for component, digest
        in RUN_CONFIG_COMPONENT_HASHES.items()
    ]
    + [
        {
            "component": "v0_1_run_config_hash",
            "sha256": V0_1_RUN_CONFIG_HASH,
        }
    ]
)


RUN_CONFIG_SUMMARY = pd.DataFrame(
    [
        {
            "field": "config_type",
            "value": V0_1_RUN_CONFIG[
                "config_type"
            ],
        },
        {
            "field": "config_schema_version",
            "value": CONTRACT_SCHEMA_VERSION,
        },
        {
            "field": "source_run_prefix",
            "value": SOURCE_RUN_PREFIX,
        },
        {
            "field": "operating_mode",
            "value": V0_1_OPERATING_MODE,
        },
        {
            "field": "contract_status",
            "value": V0_1_CONTRACT_STATUS,
        },
        {
            "field": "primary_source_files",
            "value": len(
                AUTHORITATIVE_RAW_SOURCE_MANIFEST
            ),
        },
        {
            "field": "chronological_partitions",
            "value": len(
                CHRONOLOGICAL_SPLIT_TABLE
            ),
        },
        {
            "field": "downstream_notebooks",
            "value": len(
                DOWNSTREAM_NOTEBOOK_SEQUENCE
            ),
        },
        {
            "field": "next_notebook",
            "value": DOWNSTREAM_NOTEBOOK_SEQUENCE[0],
        },
        {
            "field": "v0_1_run_config_hash",
            "value": V0_1_RUN_CONFIG_HASH,
        },
    ]
)


# ============================================================
# CONFIGURATION FREEZE GATES
# ============================================================

required_config_sections = {
    "source_identity",
    "operating_authority",
    "market",
    "chronological_partitions",
    "reproducibility",
    "output_contract",
    "pipeline_sequence",
    "component_hashes",
}

missing_config_sections = sorted(
    required_config_sections
    - set(V0_1_RUN_CONFIG)
)

config_primary_source_count_matches = bool(
    V0_1_RUN_CONFIG[
        "source_identity"
    ]["primary_file_count"]
    == len(AUTHORITATIVE_RAW_SOURCE_MANIFEST)
)

config_partition_count_matches = bool(
    len(
        V0_1_RUN_CONFIG[
            "chronological_partitions"
        ]["boundary_table"]
    )
    == len(CHRONOLOGICAL_SPLIT_TABLE)
)

config_permissions_match = bool(
    V0_1_RUN_CONFIG[
        "operating_authority"
    ]["permissions"]
    == RUN_CONFIG_PERMISSIONS
)

run_config_hash_valid = bool(
    SHA256_PATTERN.fullmatch(
        V0_1_RUN_CONFIG_HASH
    )
)


RUN_CONFIG_GATES = pd.DataFrame(
    [
        {
            "gate": "required_config_sections_present",
            "passed": len(
                missing_config_sections
            ) == 0,
            "evidence": (
                "none"
                if not missing_config_sections
                else " | ".join(
                    missing_config_sections
                )
            ),
        },
        {
            "gate": "component_hashes_valid",
            "passed": not invalid_component_hashes,
            "evidence": (
                f"valid="
                f"{len(RUN_CONFIG_COMPONENT_HASHES)}"
            ),
        },
        {
            "gate": "primary_source_count_matches",
            "passed": (
                config_primary_source_count_matches
            ),
            "evidence": (
                f"config="
                f"{V0_1_RUN_CONFIG['source_identity']['primary_file_count']}; "
                f"manifest="
                f"{len(AUTHORITATIVE_RAW_SOURCE_MANIFEST)}"
            ),
        },
        {
            "gate": "partition_count_matches",
            "passed": (
                config_partition_count_matches
            ),
            "evidence": (
                f"config="
                f"{len(V0_1_RUN_CONFIG['chronological_partitions']['boundary_table'])}; "
                f"table="
                f"{len(CHRONOLOGICAL_SPLIT_TABLE)}"
            ),
        },
        {
            "gate": "permission_matrix_matches",
            "passed": (
                config_permissions_match
            ),
            "evidence": (
                f"permissions="
                f"{len(RUN_CONFIG_PERMISSIONS)}"
            ),
        },
        {
            "gate": "v0_1_run_config_hash_valid",
            "passed": run_config_hash_valid,
            "evidence": V0_1_RUN_CONFIG_HASH,
        },
        {
            "gate": "v0_0_overwrite_prohibited",
            "passed": bool(
                not V0_1_RUN_CONFIG[
                    "output_contract"
                ]["overwrite_v0_0_allowed"]
            ),
            "evidence": (
                "overwrite_v0_0_allowed=False"
            ),
        },
        {
            "gate": "next_notebook_frozen",
            "passed": (
                V0_1_RUN_CONFIG[
                    "pipeline_sequence"
                ]["next_notebook"]
                == "01_RAW_DATA_AUDIT.ipynb"
            ),
            "evidence": (
                V0_1_RUN_CONFIG[
                    "pipeline_sequence"
                ]["next_notebook"]
            ),
        },
    ]
)


display(RUN_CONFIG_SUMMARY)
display(RUN_CONFIG_HASH_TABLE)
display(RUN_CONFIG_GATES)


failed_run_config_gates = RUN_CONFIG_GATES.loc[
    ~RUN_CONFIG_GATES["passed"]
]

if not failed_run_config_gates.empty:
    failure_text = ", ".join(
        f"{row.gate}: {row.evidence}"
        for row in failed_run_config_gates.itertuples(
            index=False
        )
    )

    raise RuntimeError(
        "V0.1 run-configuration freeze failed: "
        + failure_text
    )


print("V0.1 run configuration freeze: PASS")
print(
    "Operating mode: "
    f"{V0_1_OPERATING_MODE}"
)
print(
    "Primary source files registered: "
    f"{len(AUTHORITATIVE_RAW_SOURCE_MANIFEST):,}"
)
print(
    "Chronological partitions frozen: "
    f"{len(CHRONOLOGICAL_SPLIT_TABLE):,}"
)
print(
    "Next notebook: "
    f"{DOWNSTREAM_NOTEBOOK_SEQUENCE[0]}"
)
print(
    "V0.1 run configuration hash: "
    f"{V0_1_RUN_CONFIG_HASH}"
)

,field,value
0,config_type,V0_1_RUN_CONFIGURATION
1,config_schema_version,v0.1.0
2,source_run_prefix,BTCUSDT_spot_20260710T063746Z_c8b5bf12
3,operating_mode,ENGINEERING_REPRODUCTION_MODE
4,contract_status,CONDITIONAL PASS
5,primary_source_files,5
6,chronological_partitions,4
7,downstream_notebooks,10
8,next_notebook,01_RAW_DATA_AUDIT.ipynb
9,v0_1_run_config_hash,14aea0efb3c7b6a193b8c4575a440babe8265d2935c6a7...


,component,sha256
0,source_set_hash,132c83531eec615d279408b5c06f402973114ba3058dfa...
1,source_identity_declaration_hash,0f1c16f51c9a608252a4a45a5801d1b03da3034ab63c03...
2,market_data_contract_hash,aadc573f7d891385115db3029a51b242f5c1213313b9b8...
3,missing_data_rejection_contract_hash,27dafdd4a69308d7ff4d64fb0bdde5150c123c92cc425a...
4,chronological_split_contract_hash,3b7e8d46f43d8a4dd7b7a1f3d9a61c2c92d3f3643042a8...
5,decision_and_tolerance_contract_hash,31e054874e4072f0d90a0f7dfd14e1531f54f22e3c5bee...
6,software_environment_hash,c16166df27b3fe072839c2bbf60d35a41ec06a0ca5c065...
7,v0_1_run_config_hash,14aea0efb3c7b6a193b8c4575a440babe8265d2935c6a7...


,gate,passed,evidence
0,required_config_sections_present,True,none
1,component_hashes_valid,True,valid=7
2,primary_source_count_matches,True,config=5; manifest=5
3,partition_count_matches,True,config=4; table=4
4,permission_matrix_matches,True,permissions=9
5,v0_1_run_config_hash_valid,True,14aea0efb3c7b6a193b8c4575a440babe8265d2935c6a7...
6,v0_0_overwrite_prohibited,True,overwrite_v0_0_allowed=False
7,next_notebook_frozen,True,01_RAW_DATA_AUDIT.ipynb


V0.1 run configuration freeze: PASS
Operating mode: ENGINEERING_REPRODUCTION_MODE
Primary source files registered: 5
Chronological partitions frozen: 4
Next notebook: 01_RAW_DATA_AUDIT.ipynb
V0.1 run configuration hash: 14aea0efb3c7b6a193b8c4575a440babe8265d2935c6a770bee995f688d59617


In [26]:
# ============================================================
# GENERATE THE V0.1 RUN IDENTITY
# ============================================================

V0_1_RUN_STARTED_TIMESTAMP = pd.Timestamp(
    EXECUTION_STARTED_UTC
)

if V0_1_RUN_STARTED_TIMESTAMP.tzinfo is None:
    raise RuntimeError(
        "EXECUTION_STARTED_UTC must be timezone-aware."
    )

V0_1_RUN_STARTED_TIMESTAMP = (
    V0_1_RUN_STARTED_TIMESTAMP.tz_convert("UTC")
)

V0_1_RUN_STARTED_UTC = (
    V0_1_RUN_STARTED_TIMESTAMP.isoformat()
)

V0_1_RUN_STARTED_COMPACT = (
    V0_1_RUN_STARTED_TIMESTAMP.strftime(
        "%Y%m%dT%H%M%SZ"
    )
)


# ============================================================
# RUN-ID SEED
# ============================================================

V0_1_RUN_ID_SEED = {
    "identity_type": "V0_1_RUN_INSTANCE",
    "pipeline_name": PIPELINE_NAME,
    "pipeline_version": PIPELINE_VERSION,
    "contract_schema_version": (
        CONTRACT_SCHEMA_VERSION
    ),
    "execution_started_utc": (
        V0_1_RUN_STARTED_UTC
    ),
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "source_set_hash": SOURCE_SET_HASH,
    "v0_1_run_config_hash": (
        V0_1_RUN_CONFIG_HASH
    ),
    "software_environment_hash": (
        SOFTWARE_ENVIRONMENT_HASH
    ),
    "random_seed": int(RANDOM_SEED),
    "producing_notebook": NOTEBOOK_FILENAME,
}

V0_1_RUN_ID_SEED = to_json_safe(
    V0_1_RUN_ID_SEED
)

V0_1_RUN_ID_SEED_HASH = (
    sha256_of_canonical_object(
        V0_1_RUN_ID_SEED
    )
)


# ============================================================
# HUMAN-READABLE RUN ID
# ============================================================

V0_1_RUN_ID = (
    f"v0_1_{V0_1_RUN_STARTED_COMPACT}_"
    f"{V0_1_RUN_ID_SEED_HASH[:12]}"
)

RUN_OUTPUT_PREFIX = (
    f"{SOURCE_RUN_PREFIX}__{V0_1_RUN_ID}"
)


# ============================================================
# AUTHORITATIVE ARTIFACT-NAMING FUNCTION
# ============================================================

def sanitize_artifact_token(
    value: object,
) -> str:
    """
    Convert an artifact-name component into a portable filename token.
    """
    token = re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        str(value).strip(),
    )

    token = token.strip("._-")

    if not token:
        raise ValueError(
            "Artifact token cannot be empty after sanitization."
        )

    return token


def make_v0_1_artifact_name(
    artifact_name: str,
    extension: str,
    producing_notebook: str = NOTEBOOK_FILENAME,
) -> str:
    """
    Build one authoritative V0.1 artifact filename.

    The source run prefix and V0.1 run ID remain separate fields in
    both the filename and artifact metadata.
    """
    notebook_token = sanitize_artifact_token(
        Path(producing_notebook).stem
    )

    artifact_token = sanitize_artifact_token(
        artifact_name
    )

    extension_token = sanitize_artifact_token(
        extension.lstrip(".")
    )

    return (
        f"{RUN_OUTPUT_PREFIX}__"
        f"{notebook_token}__"
        f"{artifact_token}."
        f"{extension_token}"
    )


RUN_IDENTITY_ARTIFACT_NAME = (
    make_v0_1_artifact_name(
        artifact_name="run_identity",
        extension="json",
    )
)

RUN_CONFIG_ARTIFACT_NAME = (
    make_v0_1_artifact_name(
        artifact_name="run_config",
        extension="json",
    )
)

SOURCE_REGISTRY_ARTIFACT_NAME = (
    make_v0_1_artifact_name(
        artifact_name="v0_0_source_registry",
        extension="json",
    )
)


# ============================================================
# COMPLETE RUN-IDENTITY MANIFEST
# ============================================================

V0_1_RUN_IDENTITY_MANIFEST = {
    "identity_type": "V0_1_RUN_INSTANCE",
    "identity_schema_version": (
        CONTRACT_SCHEMA_VERSION
    ),
    "pipeline_name": PIPELINE_NAME,
    "pipeline_version": PIPELINE_VERSION,
    "v0_1_run_id": V0_1_RUN_ID,
    "execution_started_utc": (
        V0_1_RUN_STARTED_UTC
    ),

    "source_identity": {
        "source_run_prefix": SOURCE_RUN_PREFIX,
        "source_set_hash": SOURCE_SET_HASH,
        "source_identity_declaration_hash": (
            SOURCE_IDENTITY_DECLARATION_HASH
        ),
        "primary_file_count": int(
            len(AUTHORITATIVE_RAW_SOURCE_MANIFEST)
        ),
    },

    "configuration_identity": {
        "v0_1_run_config_hash": (
            V0_1_RUN_CONFIG_HASH
        ),
        "market_data_contract_hash": (
            MARKET_DATA_CONTRACT_HASH
        ),
        "missing_data_rejection_contract_hash": (
            MISSING_DATA_REJECTION_CONTRACT_HASH
        ),
        "chronological_split_contract_hash": (
            CHRONOLOGICAL_SPLIT_CONTRACT_HASH
        ),
        "decision_and_tolerance_contract_hash": (
            DECISION_AND_TOLERANCE_CONTRACT_HASH
        ),
    },

    "execution_authority": {
        "operating_mode": V0_1_OPERATING_MODE,
        "contract_status": V0_1_CONTRACT_STATUS,
        "claim_bearing_holdout_available": bool(
            CHRONOLOGICAL_SPLIT_CONTRACT[
                "claim_bearing_holdout_available"
            ]
        ),
        "partitions_are_independent": bool(
            CHRONOLOGICAL_SPLIT_CONTRACT[
                "partitions_are_independent"
            ]
        ),
        "next_notebook": (
            DOWNSTREAM_NOTEBOOK_SEQUENCE[0]
        ),
    },

    "reproducibility": {
        "random_seed": int(RANDOM_SEED),
        "python_version": PYTHON_VERSION_STRING,
        "software_environment_hash": (
            SOFTWARE_ENVIRONMENT_HASH
        ),
        "git_repository_detected": bool(
            GIT_REPOSITORY_ROOT is not None
        ),
        "git_commit": GIT_COMMIT,
        "git_branch": GIT_BRANCH,
        "git_dirty": GIT_DIRTY,
    },

    "naming_authority": {
        "run_output_prefix": RUN_OUTPUT_PREFIX,
        "run_identity_artifact_name": (
            RUN_IDENTITY_ARTIFACT_NAME
        ),
        "run_config_artifact_name": (
            RUN_CONFIG_ARTIFACT_NAME
        ),
        "source_registry_artifact_name": (
            SOURCE_REGISTRY_ARTIFACT_NAME
        ),
    },

    "run_id_seed_hash": (
        V0_1_RUN_ID_SEED_HASH
    ),
    "producing_notebook": NOTEBOOK_FILENAME,
}

V0_1_RUN_IDENTITY_MANIFEST = to_json_safe(
    V0_1_RUN_IDENTITY_MANIFEST
)


# The declaration hash is computed before adding itself to the
# manifest, preventing circular hashing.
V0_1_RUN_IDENTITY_HASH = (
    sha256_of_canonical_object(
        V0_1_RUN_IDENTITY_MANIFEST
    )
)

V0_1_RUN_IDENTITY_MANIFEST[
    "v0_1_run_identity_hash"
] = V0_1_RUN_IDENTITY_HASH


# ============================================================
# RUN-IDENTITY TABLES
# ============================================================

RUN_IDENTITY_COMPONENTS = pd.DataFrame(
    [
        {
            "identity_component": (
                "source_run_prefix"
            ),
            "value": SOURCE_RUN_PREFIX,
        },
        {
            "identity_component": (
                "source_set_hash"
            ),
            "value": SOURCE_SET_HASH,
        },
        {
            "identity_component": (
                "v0_1_run_config_hash"
            ),
            "value": V0_1_RUN_CONFIG_HASH,
        },
        {
            "identity_component": (
                "software_environment_hash"
            ),
            "value": SOFTWARE_ENVIRONMENT_HASH,
        },
        {
            "identity_component": (
                "execution_started_utc"
            ),
            "value": V0_1_RUN_STARTED_UTC,
        },
        {
            "identity_component": (
                "random_seed"
            ),
            "value": int(RANDOM_SEED),
        },
        {
            "identity_component": (
                "run_id_seed_hash"
            ),
            "value": V0_1_RUN_ID_SEED_HASH,
        },
        {
            "identity_component": (
                "v0_1_run_identity_hash"
            ),
            "value": V0_1_RUN_IDENTITY_HASH,
        },
    ]
)


RUN_IDENTITY_SUMMARY = pd.DataFrame(
    [
        {
            "field": "source_run_prefix",
            "value": SOURCE_RUN_PREFIX,
        },
        {
            "field": "v0_1_run_id",
            "value": V0_1_RUN_ID,
        },
        {
            "field": "execution_started_utc",
            "value": V0_1_RUN_STARTED_UTC,
        },
        {
            "field": "operating_mode",
            "value": V0_1_OPERATING_MODE,
        },
        {
            "field": "contract_status",
            "value": V0_1_CONTRACT_STATUS,
        },
        {
            "field": "run_output_prefix",
            "value": RUN_OUTPUT_PREFIX,
        },
        {
            "field": "run_identity_artifact",
            "value": RUN_IDENTITY_ARTIFACT_NAME,
        },
        {
            "field": "v0_1_run_identity_hash",
            "value": V0_1_RUN_IDENTITY_HASH,
        },
    ]
)


# ============================================================
# RUN-IDENTITY GATES
# ============================================================

run_id_pattern = re.compile(
    r"^v0_1_\d{8}T\d{6}Z_[a-f0-9]{12}$"
)

artifact_names = [
    RUN_IDENTITY_ARTIFACT_NAME,
    RUN_CONFIG_ARTIFACT_NAME,
    SOURCE_REGISTRY_ARTIFACT_NAME,
]

artifact_names_unique = (
    len(artifact_names)
    == len(set(artifact_names))
)

artifact_names_portable = all(
    re.fullmatch(
        r"[A-Za-z0-9._-]+",
        artifact_name,
    )
    is not None
    for artifact_name in artifact_names
)

RUN_IDENTITY_GATES = pd.DataFrame(
    [
        {
            "gate": "v0_1_run_id_format_valid",
            "passed": (
                run_id_pattern.fullmatch(
                    V0_1_RUN_ID
                )
                is not None
            ),
            "evidence": V0_1_RUN_ID,
        },
        {
            "gate": "source_and_v0_1_identities_separate",
            "passed": (
                V0_1_RUN_ID != SOURCE_RUN_PREFIX
            ),
            "evidence": (
                f"source={SOURCE_RUN_PREFIX}; "
                f"v0_1={V0_1_RUN_ID}"
            ),
        },
        {
            "gate": "run_id_seed_hash_valid",
            "passed": (
                SHA256_PATTERN.fullmatch(
                    V0_1_RUN_ID_SEED_HASH
                )
                is not None
            ),
            "evidence": V0_1_RUN_ID_SEED_HASH,
        },
        {
            "gate": "run_identity_hash_valid",
            "passed": (
                SHA256_PATTERN.fullmatch(
                    V0_1_RUN_IDENTITY_HASH
                )
                is not None
            ),
            "evidence": V0_1_RUN_IDENTITY_HASH,
        },
        {
            "gate": "run_config_hash_bound",
            "passed": (
                V0_1_RUN_IDENTITY_MANIFEST[
                    "configuration_identity"
                ]["v0_1_run_config_hash"]
                == V0_1_RUN_CONFIG_HASH
            ),
            "evidence": V0_1_RUN_CONFIG_HASH,
        },
        {
            "gate": "source_set_hash_bound",
            "passed": (
                V0_1_RUN_IDENTITY_MANIFEST[
                    "source_identity"
                ]["source_set_hash"]
                == SOURCE_SET_HASH
            ),
            "evidence": SOURCE_SET_HASH,
        },
        {
            "gate": "artifact_names_unique",
            "passed": artifact_names_unique,
            "evidence": (
                f"names={len(artifact_names)}; "
                f"unique={len(set(artifact_names))}"
            ),
        },
        {
            "gate": "artifact_names_portable",
            "passed": artifact_names_portable,
            "evidence": (
                " | ".join(artifact_names)
            ),
        },
        {
            "gate": "next_notebook_permission_preserved",
            "passed": bool(
                RUN_CONFIG_PERMISSIONS[
                    "proceed_to_notebook_01"
                ]
            ),
            "evidence": (
                "01_RAW_DATA_AUDIT.ipynb"
            ),
        },
    ]
)


display(RUN_IDENTITY_COMPONENTS)
display(RUN_IDENTITY_SUMMARY)
display(RUN_IDENTITY_GATES)


failed_run_identity_gates = (
    RUN_IDENTITY_GATES.loc[
        ~RUN_IDENTITY_GATES["passed"]
    ]
)

if not failed_run_identity_gates.empty:
    failure_text = ", ".join(
        f"{row.gate}: {row.evidence}"
        for row
        in failed_run_identity_gates.itertuples(
            index=False
        )
    )

    raise RuntimeError(
        "V0.1 run-identity generation failed: "
        + failure_text
    )


print("V0.1 run identity generation: PASS")
print(
    "Source run prefix: "
    f"{SOURCE_RUN_PREFIX}"
)
print(
    "V0.1 run ID: "
    f"{V0_1_RUN_ID}"
)
print(
    "Run output prefix: "
    f"{RUN_OUTPUT_PREFIX}"
)
print(
    "V0.1 run identity hash: "
    f"{V0_1_RUN_IDENTITY_HASH}"
)

,identity_component,value
0,source_run_prefix,BTCUSDT_spot_20260710T063746Z_c8b5bf12
1,source_set_hash,132c83531eec615d279408b5c06f402973114ba3058dfa...
2,v0_1_run_config_hash,14aea0efb3c7b6a193b8c4575a440babe8265d2935c6a7...
3,software_environment_hash,c16166df27b3fe072839c2bbf60d35a41ec06a0ca5c065...
4,execution_started_utc,2026-07-14T09:06:16+00:00
5,random_seed,20260710
6,run_id_seed_hash,e82325081a819aa69a24617cb9f79459dd074229a44389...
7,v0_1_run_identity_hash,5eb89cf073c036d70e3767df4b35ace7c31822e52e9b9d...


,field,value
0,source_run_prefix,BTCUSDT_spot_20260710T063746Z_c8b5bf12
1,v0_1_run_id,v0_1_20260714T090616Z_e82325081a81
2,execution_started_utc,2026-07-14T09:06:16+00:00
3,operating_mode,ENGINEERING_REPRODUCTION_MODE
4,contract_status,CONDITIONAL PASS
5,run_output_prefix,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_2...
6,run_identity_artifact,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_2...
7,v0_1_run_identity_hash,5eb89cf073c036d70e3767df4b35ace7c31822e52e9b9d...


,gate,passed,evidence
0,v0_1_run_id_format_valid,True,v0_1_20260714T090616Z_e82325081a81
1,source_and_v0_1_identities_separate,True,source=BTCUSDT_spot_20260710T063746Z_c8b5bf12;...
2,run_id_seed_hash_valid,True,e82325081a819aa69a24617cb9f79459dd074229a44389...
3,run_identity_hash_valid,True,5eb89cf073c036d70e3767df4b35ace7c31822e52e9b9d...
4,run_config_hash_bound,True,14aea0efb3c7b6a193b8c4575a440babe8265d2935c6a7...
5,source_set_hash_bound,True,132c83531eec615d279408b5c06f402973114ba3058dfa...
6,artifact_names_unique,True,names=3; unique=3
7,artifact_names_portable,True,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_2...
8,next_notebook_permission_preserved,True,01_RAW_DATA_AUDIT.ipynb


V0.1 run identity generation: PASS
Source run prefix: BTCUSDT_spot_20260710T063746Z_c8b5bf12
V0.1 run ID: v0_1_20260714T090616Z_e82325081a81
Run output prefix: BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81
V0.1 run identity hash: 5eb89cf073c036d70e3767df4b35ace7c31822e52e9b9df19204c42e37fe4198


In [27]:
# ============================================================
# NOTEBOOK-WIDE PRE-WRITE VALIDATION
# ============================================================

def register_prewrite_check(
    rows: list[dict],
    check_id: str,
    category: str,
    passed: bool,
    severity: str,
    evidence: str,
) -> None:
    """Append one normalized pre-write validation record."""
    rows.append(
        {
            "check_id": check_id,
            "category": category,
            "passed": bool(passed),
            "severity": severity,
            "evidence": str(evidence),
        }
    )


PREWRITE_VALIDATION_ROWS = []


# ============================================================
# RE-HASH AUTHORITATIVE PRIMARY SOURCES
# ============================================================

source_rehash_rows = []

for source_row in AUTHORITATIVE_RAW_SOURCE_MANIFEST.itertuples(
    index=False
):
    source_path = Path(source_row.resolved_path)

    try:
        current_inspection = inspect_file_bytes(
            source_path
        )

        current_sha256 = current_inspection[
            "sha256"
        ]

        current_size_bytes = int(
            current_inspection["bytes_read"]
        )

        hash_matches = (
            current_sha256
            == str(source_row.sha256).lower()
        )

        size_matches = (
            current_size_bytes
            == int(source_row.size_bytes)
        )

        source_status = (
            "PASS"
            if hash_matches and size_matches
            else "FAIL_SOURCE_DRIFT"
        )

        source_error = None

    except OSError as exc:
        current_sha256 = None
        current_size_bytes = None
        hash_matches = False
        size_matches = False
        source_status = "FAIL_UNREADABLE"
        source_error = str(exc)

    source_rehash_rows.append(
        {
            "source_role": source_row.source_role,
            "filename": source_row.filename,
            "registered_size_bytes": int(
                source_row.size_bytes
            ),
            "current_size_bytes": current_size_bytes,
            "registered_sha256": str(
                source_row.sha256
            ).lower(),
            "current_sha256": current_sha256,
            "size_matches": size_matches,
            "hash_matches": hash_matches,
            "status": source_status,
            "error": source_error,
        }
    )


PREWRITE_SOURCE_REHASH = pd.DataFrame(
    source_rehash_rows
)

source_rehash_passed = bool(
    not PREWRITE_SOURCE_REHASH.empty
    and PREWRITE_SOURCE_REHASH[
        "status"
    ].eq("PASS").all()
)

register_prewrite_check(
    rows=PREWRITE_VALIDATION_ROWS,
    check_id="PRIMARY_SOURCE_FILES_UNCHANGED",
    category="SOURCE_INTEGRITY",
    passed=source_rehash_passed,
    severity="CRITICAL",
    evidence=(
        f"unchanged="
        f"{int(PREWRITE_SOURCE_REHASH['status'].eq('PASS').sum())}/"
        f"{len(PREWRITE_SOURCE_REHASH)}"
    ),
)


# ============================================================
# RECOMPUTE ALL CONTRACT AND IDENTITY HASHES
# ============================================================

source_identity_without_hash = dict(
    SOURCE_IDENTITY_MANIFEST
)

source_identity_without_hash.pop(
    "source_identity_declaration_hash",
    None,
)

run_identity_without_hash = dict(
    V0_1_RUN_IDENTITY_MANIFEST
)

run_identity_without_hash.pop(
    "v0_1_run_identity_hash",
    None,
)


RECOMPUTED_HASHES = {
    "source_set_hash": (
        sha256_of_canonical_object(
            source_component_records
        )
    ),
    "source_identity_declaration_hash": (
        sha256_of_canonical_object(
            source_identity_without_hash
        )
    ),
    "market_data_contract_hash": (
        sha256_of_canonical_object(
            MARKET_DATA_CONTRACT
        )
    ),
    "missing_data_rejection_contract_hash": (
        sha256_of_canonical_object(
            MISSING_DATA_REJECTION_CONTRACT
        )
    ),
    "chronological_split_contract_hash": (
        sha256_of_canonical_object(
            CHRONOLOGICAL_SPLIT_CONTRACT
        )
    ),
    "decision_and_tolerance_contract_hash": (
        sha256_of_canonical_object(
            DECISION_AND_TOLERANCE_CONTRACT
        )
    ),
    "software_environment_hash": (
        sha256_of_canonical_object(
            SOFTWARE_ENVIRONMENT_IDENTITY
        )
    ),
    "v0_1_run_config_hash": (
        sha256_of_canonical_object(
            V0_1_RUN_CONFIG
        )
    ),
    "v0_1_run_id_seed_hash": (
        sha256_of_canonical_object(
            V0_1_RUN_ID_SEED
        )
    ),
    "v0_1_run_identity_hash": (
        sha256_of_canonical_object(
            run_identity_without_hash
        )
    ),
}


REGISTERED_HASHES = {
    "source_set_hash": SOURCE_SET_HASH,
    "source_identity_declaration_hash": (
        SOURCE_IDENTITY_DECLARATION_HASH
    ),
    "market_data_contract_hash": (
        MARKET_DATA_CONTRACT_HASH
    ),
    "missing_data_rejection_contract_hash": (
        MISSING_DATA_REJECTION_CONTRACT_HASH
    ),
    "chronological_split_contract_hash": (
        CHRONOLOGICAL_SPLIT_CONTRACT_HASH
    ),
    "decision_and_tolerance_contract_hash": (
        DECISION_AND_TOLERANCE_CONTRACT_HASH
    ),
    "software_environment_hash": (
        SOFTWARE_ENVIRONMENT_HASH
    ),
    "v0_1_run_config_hash": (
        V0_1_RUN_CONFIG_HASH
    ),
    "v0_1_run_id_seed_hash": (
        V0_1_RUN_ID_SEED_HASH
    ),
    "v0_1_run_identity_hash": (
        V0_1_RUN_IDENTITY_HASH
    ),
}


hash_reconciliation_rows = []

for hash_name in REGISTERED_HASHES:
    registered_hash = REGISTERED_HASHES[
        hash_name
    ]

    recomputed_hash = RECOMPUTED_HASHES[
        hash_name
    ]

    hash_reconciliation_rows.append(
        {
            "hash_name": hash_name,
            "registered_sha256": registered_hash,
            "recomputed_sha256": recomputed_hash,
            "matches": (
                registered_hash
                == recomputed_hash
            ),
        }
    )


PREWRITE_HASH_RECONCILIATION = pd.DataFrame(
    hash_reconciliation_rows
)

hash_reconciliation_passed = bool(
    PREWRITE_HASH_RECONCILIATION[
        "matches"
    ].all()
)

register_prewrite_check(
    rows=PREWRITE_VALIDATION_ROWS,
    check_id="ALL_CONTRACT_HASHES_REPRODUCIBLE",
    category="IDENTITY",
    passed=hash_reconciliation_passed,
    severity="CRITICAL",
    evidence=(
        f"matching="
        f"{int(PREWRITE_HASH_RECONCILIATION['matches'].sum())}/"
        f"{len(PREWRITE_HASH_RECONCILIATION)}"
    ),
)


# ============================================================
# PRIOR GATE-TABLE RECONCILIATION
# ============================================================

PRIOR_BOOLEAN_GATE_TABLES = {
    "CHRONOLOGICAL_COVERAGE_GATES": (
        CHRONOLOGICAL_COVERAGE_GATES
    ),
    "SNAPSHOT_STRUCTURE_GATES": (
        SNAPSHOT_STRUCTURE_GATES
    ),
    "MARKET_DATA_CONTRACT_GATES": (
        MARKET_DATA_CONTRACT_GATES
    ),
    "MISSING_DATA_REJECTION_GATES": (
        MISSING_DATA_REJECTION_GATES
    ),
    "CHRONOLOGICAL_SPLIT_GATES": (
        CHRONOLOGICAL_SPLIT_GATES
    ),
    "DECISION_TOLERANCE_GATES": (
        DECISION_TOLERANCE_GATES
    ),
    "RUN_CONFIG_GATES": (
        RUN_CONFIG_GATES
    ),
    "RUN_IDENTITY_GATES": (
        RUN_IDENTITY_GATES
    ),
}


prior_gate_rows = []

for table_name, gate_table in (
    PRIOR_BOOLEAN_GATE_TABLES.items()
):
    if "passed" not in gate_table.columns:
        raise RuntimeError(
            f"{table_name} does not contain a passed column."
        )

    gate_count = int(len(gate_table))

    passed_count = int(
        gate_table["passed"].fillna(False).sum()
    )

    all_passed = bool(
        gate_count > 0
        and passed_count == gate_count
    )

    prior_gate_rows.append(
        {
            "gate_table": table_name,
            "gate_count": gate_count,
            "passed_count": passed_count,
            "all_passed": all_passed,
        }
    )


PRIOR_GATE_RECONCILIATION = pd.DataFrame(
    prior_gate_rows
)

prior_boolean_gates_passed = bool(
    PRIOR_GATE_RECONCILIATION[
        "all_passed"
    ].all()
)

register_prewrite_check(
    rows=PREWRITE_VALIDATION_ROWS,
    check_id="PRIOR_BOOLEAN_GATE_TABLES_PASS",
    category="NOTEBOOK_STATE",
    passed=prior_boolean_gates_passed,
    severity="CRITICAL",
    evidence=(
        f"tables_passed="
        f"{int(PRIOR_GATE_RECONCILIATION['all_passed'].sum())}/"
        f"{len(PRIOR_GATE_RECONCILIATION)}"
    ),
)


# ============================================================
# STATUS-BASED TABLE RECONCILIATION
# ============================================================

project_root_status_passed = bool(
    PROJECT_ROOT_AUDIT[
        "status"
    ].eq("PASS").all()
)

source_registry_status_passed = bool(
    ~SOURCE_AUTHORITY_REGISTRY[
        "status"
    ].eq("FAIL").any()
)

source_role_status_passed = bool(
    ~SOURCE_ROLE_AUDIT[
        "status"
    ].str.startswith(
        "FAIL",
        na=False,
    ).any()
)

checksum_status_passed = bool(
    ~CHECKSUM_VERIFICATION[
        "verification_status"
    ].str.startswith(
        "FAIL",
        na=False,
    ).any()
)

reference_hash_status_passed = bool(
    V0_0_REFERENCE_ARTIFACT_MANIFEST[
        "hash_status"
    ].eq("PASS").all()
)


STATUS_TABLE_RECONCILIATION = pd.DataFrame(
    [
        {
            "table_name": "PROJECT_ROOT_AUDIT",
            "passed": project_root_status_passed,
        },
        {
            "table_name": "SOURCE_AUTHORITY_REGISTRY",
            "passed": source_registry_status_passed,
        },
        {
            "table_name": "SOURCE_ROLE_AUDIT",
            "passed": source_role_status_passed,
        },
        {
            "table_name": "CHECKSUM_VERIFICATION",
            "passed": checksum_status_passed,
        },
        {
            "table_name": (
                "V0_0_REFERENCE_ARTIFACT_MANIFEST"
            ),
            "passed": reference_hash_status_passed,
        },
    ]
)

status_tables_passed = bool(
    STATUS_TABLE_RECONCILIATION[
        "passed"
    ].all()
)

register_prewrite_check(
    rows=PREWRITE_VALIDATION_ROWS,
    check_id="STATUS_BASED_AUDITS_PASS",
    category="NOTEBOOK_STATE",
    passed=status_tables_passed,
    severity="CRITICAL",
    evidence=(
        f"tables_passed="
        f"{int(STATUS_TABLE_RECONCILIATION['passed'].sum())}/"
        f"{len(STATUS_TABLE_RECONCILIATION)}"
    ),
)


# ============================================================
# IDENTITY AND CONFIGURATION CONSISTENCY
# ============================================================

run_id_matches_manifest = bool(
    V0_1_RUN_IDENTITY_MANIFEST[
        "v0_1_run_id"
    ]
    == V0_1_RUN_ID
)

source_prefix_consistent = bool(
    V0_1_RUN_CONFIG[
        "source_identity"
    ]["source_run_prefix"]
    == SOURCE_RUN_PREFIX
    and V0_1_RUN_IDENTITY_MANIFEST[
        "source_identity"
    ]["source_run_prefix"]
    == SOURCE_RUN_PREFIX
)

operating_mode_consistent = bool(
    V0_1_RUN_CONFIG[
        "operating_authority"
    ]["operating_mode"]
    == V0_1_OPERATING_MODE
    and V0_1_RUN_IDENTITY_MANIFEST[
        "execution_authority"
    ]["operating_mode"]
    == V0_1_OPERATING_MODE
)

next_notebook_consistent = bool(
    V0_1_RUN_CONFIG[
        "pipeline_sequence"
    ]["next_notebook"]
    == "01_RAW_DATA_AUDIT.ipynb"
    and V0_1_RUN_IDENTITY_MANIFEST[
        "execution_authority"
    ]["next_notebook"]
    == "01_RAW_DATA_AUDIT.ipynb"
)

notebook_01_permission_valid = bool(
    RUN_CONFIG_PERMISSIONS[
        "proceed_to_notebook_01"
    ]
)


identity_config_checks = {
    "run_id_matches_manifest": (
        run_id_matches_manifest
    ),
    "source_prefix_consistent": (
        source_prefix_consistent
    ),
    "operating_mode_consistent": (
        operating_mode_consistent
    ),
    "next_notebook_consistent": (
        next_notebook_consistent
    ),
    "notebook_01_permission_valid": (
        notebook_01_permission_valid
    ),
}

identity_config_consistency_passed = all(
    identity_config_checks.values()
)

register_prewrite_check(
    rows=PREWRITE_VALIDATION_ROWS,
    check_id="IDENTITY_AND_CONFIG_CONSISTENT",
    category="IDENTITY",
    passed=identity_config_consistency_passed,
    severity="CRITICAL",
    evidence=(
        " | ".join(
            f"{name}={passed}"
            for name, passed
            in identity_config_checks.items()
        )
    ),
)


# ============================================================
# OUTPUT-ROOT SAFETY
# ============================================================

configured_output_paths = [
    V0_1_ROOT_RESOLVED,
    resolve_path(V0_1_CONFIG_ROOT),
    resolve_path(V0_1_MANIFEST_ROOT),
    resolve_path(V0_1_AUDIT_ROOT),
]

all_output_paths_inside_v0_1 = all(
    is_within(
        output_path,
        V0_1_ROOT_RESOLVED,
    )
    for output_path in configured_output_paths
)

no_output_path_inside_v0_0 = all(
    not is_within(
        output_path,
        V0_0_ROOT_RESOLVED,
    )
    for output_path in configured_output_paths
)

all_output_directories_exist = all(
    output_path.exists()
    and output_path.is_dir()
    for output_path in configured_output_paths
)

output_root_safety_passed = bool(
    all_output_paths_inside_v0_1
    and no_output_path_inside_v0_0
    and all_output_directories_exist
)

register_prewrite_check(
    rows=PREWRITE_VALIDATION_ROWS,
    check_id="OUTPUT_ROOTS_SAFE",
    category="OUTPUT_AUTHORITY",
    passed=output_root_safety_passed,
    severity="CRITICAL",
    evidence=(
        f"inside_v0_1="
        f"{all_output_paths_inside_v0_1}; "
        f"outside_v0_0="
        f"{no_output_path_inside_v0_0}; "
        f"directories_exist="
        f"{all_output_directories_exist}"
    ),
)


# ============================================================
# MODE AND ENVIRONMENT WARNINGS
# ============================================================

environment_warning_count = int(
    len(environment_warnings)
)

register_prewrite_check(
    rows=PREWRITE_VALIDATION_ROWS,
    check_id="GIT_PROVENANCE_AVAILABLE",
    category="ENVIRONMENT",
    passed=bool(
        GIT_REPOSITORY_ROOT is not None
    ),
    severity="WARNING",
    evidence=(
        f"repository_root="
        f"{GIT_REPOSITORY_ROOT}"
    ),
)

register_prewrite_check(
    rows=PREWRITE_VALIDATION_ROWS,
    check_id="FULL_STATISTICAL_MODE_AVAILABLE",
    category="STATISTICAL_AUTHORITY",
    passed=bool(
        V0_1_OPERATING_MODE
        == "FULL_STATISTICAL_MODE"
    ),
    severity="WARNING",
    evidence=(
        f"operating_mode="
        f"{V0_1_OPERATING_MODE}; "
        f"meaningful_sessions="
        f"{meaningful_session_count}"
    ),
)

register_prewrite_check(
    rows=PREWRITE_VALIDATION_ROWS,
    check_id="CLAIM_BEARING_HOLDOUT_AVAILABLE",
    category="STATISTICAL_AUTHORITY",
    passed=bool(
        CHRONOLOGICAL_SPLIT_CONTRACT[
            "claim_bearing_holdout_available"
        ]
    ),
    severity="WARNING",
    evidence=(
        f"available="
        f"{CHRONOLOGICAL_SPLIT_CONTRACT['claim_bearing_holdout_available']}"
    ),
)


# ============================================================
# FINAL PRE-WRITE DECISION
# ============================================================

PREWRITE_VALIDATION_LEDGER = pd.DataFrame(
    PREWRITE_VALIDATION_ROWS
)

PREWRITE_VALIDATION_LEDGER["status"] = (
    np.where(
        PREWRITE_VALIDATION_LEDGER["passed"],
        "PASS",
        np.where(
            PREWRITE_VALIDATION_LEDGER[
                "severity"
            ].eq("CRITICAL"),
            "FAIL",
            "WARNING",
        ),
    )
)


critical_prewrite_failures = (
    PREWRITE_VALIDATION_LEDGER.loc[
        PREWRITE_VALIDATION_LEDGER[
            "severity"
        ].eq("CRITICAL")
        & ~PREWRITE_VALIDATION_LEDGER[
            "passed"
        ]
    ]
)

prewrite_warnings = (
    PREWRITE_VALIDATION_LEDGER.loc[
        PREWRITE_VALIDATION_LEDGER[
            "severity"
        ].eq("WARNING")
        & ~PREWRITE_VALIDATION_LEDGER[
            "passed"
        ]
    ]
)


if not critical_prewrite_failures.empty:
    NOTEBOOK_00_PREWRITE_STATUS = "FAIL"

elif (
    V0_1_CONTRACT_STATUS
    == "CONDITIONAL PASS"
):
    NOTEBOOK_00_PREWRITE_STATUS = (
        "CONDITIONAL PASS"
    )

elif not prewrite_warnings.empty:
    NOTEBOOK_00_PREWRITE_STATUS = "WARNING"

else:
    NOTEBOOK_00_PREWRITE_STATUS = "PASS"


NOTEBOOK_00_PREWRITE_DECISION = pd.DataFrame(
    [
        {
            "v0_1_run_id": V0_1_RUN_ID,
            "source_run_prefix": (
                SOURCE_RUN_PREFIX
            ),
            "operating_mode": (
                V0_1_OPERATING_MODE
            ),
            "prewrite_status": (
                NOTEBOOK_00_PREWRITE_STATUS
            ),
            "critical_check_count": int(
                PREWRITE_VALIDATION_LEDGER[
                    "severity"
                ].eq("CRITICAL").sum()
            ),
            "critical_failure_count": int(
                len(critical_prewrite_failures)
            ),
            "warning_count": int(
                len(prewrite_warnings)
            ),
            "primary_sources_unchanged": (
                source_rehash_passed
            ),
            "contract_hashes_reproducible": (
                hash_reconciliation_passed
            ),
            "notebook_01_authorized": (
                notebook_01_permission_valid
            ),
            "next_notebook": (
                "01_RAW_DATA_AUDIT.ipynb"
            ),
        }
    ]
)


display(PREWRITE_SOURCE_REHASH)
display(PREWRITE_HASH_RECONCILIATION)
display(PRIOR_GATE_RECONCILIATION)
display(STATUS_TABLE_RECONCILIATION)
display(PREWRITE_VALIDATION_LEDGER)
display(NOTEBOOK_00_PREWRITE_DECISION)


if not critical_prewrite_failures.empty:
    failure_text = ", ".join(
        f"{row.check_id}: {row.evidence}"
        for row
        in critical_prewrite_failures.itertuples(
            index=False
        )
    )

    raise RuntimeError(
        "Notebook 00 pre-write validation failed: "
        + failure_text
    )


print("Notebook 00 pre-write validation: PASS")
print(
    "Pre-write status: "
    f"{NOTEBOOK_00_PREWRITE_STATUS}"
)
print(
    "Primary source files unchanged: "
    f"{source_rehash_passed}"
)
print(
    "Contract and identity hashes reproducible: "
    f"{hash_reconciliation_passed}"
)
print(
    "Critical failures: "
    f"{len(critical_prewrite_failures):,}"
)
print(
    "Warnings retained: "
    f"{len(prewrite_warnings):,}"
)
print(
    "Notebook 01 authorized: "
    f"{notebook_01_permission_valid}"
)

,source_role,filename,registered_size_bytes,current_size_bytes,registered_sha256,current_sha256,size_matches,hash_matches,status,error
0,COLLECTOR_METADATA,BTCUSDT_spot_20260710T063746Z_c8b5bf12_session...,1777,1777,f7ea3210033b7e6d02d775fdc496422204194d7fd8a64a...,f7ea3210033b7e6d02d775fdc496422204194d7fd8a64a...,True,True,PASS,None
1,DEPTH_STREAM,BTCUSDT_spot_20260710T063746Z_c8b5bf12_depth_u...,63443258,63443258,608aed2608120c92fe4eaf485c43a4b26cf75d1b212041...,608aed2608120c92fe4eaf485c43a4b26cf75d1b212041...,True,True,PASS,None
2,REST_SNAPSHOT,BTCUSDT_spot_20260710T063746Z_c8b5bf12_snapsho...,360812,360812,7921a051c2483ed04e4da82c9a1f58d847eab34b4de5a9...,7921a051c2483ed04e4da82c9a1f58d847eab34b4de5a9...,True,True,PASS,None
3,RUN_MANIFEST,BTCUSDT_spot_20260710T063746Z_c8b5bf12_develop...,1738,1738,f8a3a278d6cd8b674f47d6ec1d5f4f434cb6b0d1fc05ec...,f8a3a278d6cd8b674f47d6ec1d5f4f434cb6b0d1fc05ec...,True,True,PASS,None
4,TRADE_STREAM,BTCUSDT_spot_20260710T063746Z_c8b5bf12_trades....,46082102,46082102,fcffc0cbd88badc5e2d938545ca864f09871e70e6f1f2c...,fcffc0cbd88badc5e2d938545ca864f09871e70e6f1f2c...,True,True,PASS,None


,hash_name,registered_sha256,recomputed_sha256,matches
0,source_set_hash,132c83531eec615d279408b5c06f402973114ba3058dfa...,132c83531eec615d279408b5c06f402973114ba3058dfa...,True
1,source_identity_declaration_hash,0f1c16f51c9a608252a4a45a5801d1b03da3034ab63c03...,0f1c16f51c9a608252a4a45a5801d1b03da3034ab63c03...,True
2,market_data_contract_hash,aadc573f7d891385115db3029a51b242f5c1213313b9b8...,aadc573f7d891385115db3029a51b242f5c1213313b9b8...,True
3,missing_data_rejection_contract_hash,27dafdd4a69308d7ff4d64fb0bdde5150c123c92cc425a...,27dafdd4a69308d7ff4d64fb0bdde5150c123c92cc425a...,True
4,chronological_split_contract_hash,3b7e8d46f43d8a4dd7b7a1f3d9a61c2c92d3f3643042a8...,3b7e8d46f43d8a4dd7b7a1f3d9a61c2c92d3f3643042a8...,True
5,decision_and_tolerance_contract_hash,31e054874e4072f0d90a0f7dfd14e1531f54f22e3c5bee...,31e054874e4072f0d90a0f7dfd14e1531f54f22e3c5bee...,True
6,software_environment_hash,c16166df27b3fe072839c2bbf60d35a41ec06a0ca5c065...,c16166df27b3fe072839c2bbf60d35a41ec06a0ca5c065...,True
7,v0_1_run_config_hash,14aea0efb3c7b6a193b8c4575a440babe8265d2935c6a7...,14aea0efb3c7b6a193b8c4575a440babe8265d2935c6a7...,True
8,v0_1_run_id_seed_hash,e82325081a819aa69a24617cb9f79459dd074229a44389...,e82325081a819aa69a24617cb9f79459dd074229a44389...,True
9,v0_1_run_identity_hash,5eb89cf073c036d70e3767df4b35ace7c31822e52e9b9d...,5eb89cf073c036d70e3767df4b35ace7c31822e52e9b9d...,True


,gate_table,gate_count,passed_count,all_passed
0,CHRONOLOGICAL_COVERAGE_GATES,5,5,True
1,SNAPSHOT_STRUCTURE_GATES,9,9,True
2,MARKET_DATA_CONTRACT_GATES,9,9,True
3,MISSING_DATA_REJECTION_GATES,9,9,True
4,CHRONOLOGICAL_SPLIT_GATES,8,8,True
5,DECISION_TOLERANCE_GATES,9,9,True
6,RUN_CONFIG_GATES,8,8,True
7,RUN_IDENTITY_GATES,9,9,True


,table_name,passed
0,PROJECT_ROOT_AUDIT,True
1,SOURCE_AUTHORITY_REGISTRY,True
2,SOURCE_ROLE_AUDIT,True
3,CHECKSUM_VERIFICATION,True
4,V0_0_REFERENCE_ARTIFACT_MANIFEST,True


,check_id,category,passed,severity,evidence,status
0,PRIMARY_SOURCE_FILES_UNCHANGED,SOURCE_INTEGRITY,True,CRITICAL,unchanged=5/5,PASS
1,ALL_CONTRACT_HASHES_REPRODUCIBLE,IDENTITY,True,CRITICAL,matching=10/10,PASS
2,PRIOR_BOOLEAN_GATE_TABLES_PASS,NOTEBOOK_STATE,True,CRITICAL,tables_passed=8/8,PASS
3,STATUS_BASED_AUDITS_PASS,NOTEBOOK_STATE,True,CRITICAL,tables_passed=5/5,PASS
4,IDENTITY_AND_CONFIG_CONSISTENT,IDENTITY,True,CRITICAL,run_id_matches_manifest=True | source_prefix_c...,PASS
5,OUTPUT_ROOTS_SAFE,OUTPUT_AUTHORITY,True,CRITICAL,inside_v0_1=True; outside_v0_0=True; directori...,PASS
6,GIT_PROVENANCE_AVAILABLE,ENVIRONMENT,False,WARNING,repository_root=None,WARNING
7,FULL_STATISTICAL_MODE_AVAILABLE,STATISTICAL_AUTHORITY,False,WARNING,operating_mode=ENGINEERING_REPRODUCTION_MODE; ...,WARNING
8,CLAIM_BEARING_HOLDOUT_AVAILABLE,STATISTICAL_AUTHORITY,False,WARNING,available=False,WARNING


,v0_1_run_id,source_run_prefix,operating_mode,prewrite_status,critical_check_count,critical_failure_count,warning_count,primary_sources_unchanged,contract_hashes_reproducible,notebook_01_authorized,next_notebook
0,v0_1_20260714T090616Z_e82325081a81,BTCUSDT_spot_20260710T063746Z_c8b5bf12,ENGINEERING_REPRODUCTION_MODE,CONDITIONAL PASS,6,0,3,True,True,True,01_RAW_DATA_AUDIT.ipynb


Notebook 00 pre-write validation: PASS
Pre-write status: CONDITIONAL PASS
Primary source files unchanged: True
Contract and identity hashes reproducible: True
Critical failures: 0
Warnings retained: 3
Notebook 01 authorized: True


In [29]:
# ============================================================
# ATOMIC NOTEBOOK 00 ARTIFACT WRITES
# ============================================================

import math
import tempfile
from datetime import date
from datetime import datetime as datetime_type
from decimal import Decimal
from pathlib import Path


# ============================================================
# JSON-SAFE SERIALIZATION
# ============================================================

def artifact_json_safe(value):
    """
    Convert notebook objects into deterministic JSON-compatible values.

    Handles DataFrames, Series, Index objects, NumPy values, pandas
    missing values, Decimals, datetimes, Paths, arrays, tuples, and sets.
    """
    if value is None:
        return None

    if value is pd.NA or value is pd.NaT:
        return None

    if isinstance(value, (str, bool, int)):
        return value

    if isinstance(value, float):
        return value if math.isfinite(value) else None

    if isinstance(value, np.generic):
        return artifact_json_safe(value.item())

    if isinstance(value, Decimal):
        return str(value) if value.is_finite() else None

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, pd.Timestamp):
        if pd.isna(value):
            return None

        return value.isoformat()

    if isinstance(value, pd.Timedelta):
        if pd.isna(value):
            return None

        return value.isoformat()

    if isinstance(value, datetime_type):
        return value.isoformat()

    if isinstance(value, date):
        return value.isoformat()

    if isinstance(value, pd.DataFrame):
        columns = [
            str(column)
            for column in value.columns
        ]

        records = []

        for row in value.itertuples(
            index=False,
            name=None,
        ):
            records.append(
                {
                    column: artifact_json_safe(item)
                    for column, item in zip(columns, row)
                }
            )

        return records

    if isinstance(value, pd.Series):
        return {
            str(key): artifact_json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, pd.Index):
        return [
            artifact_json_safe(item)
            for item in value.tolist()
        ]

    if isinstance(value, np.ndarray):
        return artifact_json_safe(
            value.tolist()
        )

    if isinstance(value, dict):
        return {
            str(key): artifact_json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            artifact_json_safe(item)
            for item in value
        ]

    if isinstance(value, set):
        normalized_items = [
            artifact_json_safe(item)
            for item in value
        ]

        return sorted(
            normalized_items,
            key=lambda item: json.dumps(
                item,
                sort_keys=True,
                ensure_ascii=False,
            ),
        )

    try:
        missing = pd.isna(value)
    except (TypeError, ValueError):
        missing = False

    if (
        isinstance(missing, (bool, np.bool_))
        and bool(missing)
    ):
        return None

    raise TypeError(
        "Unsupported artifact value type: "
        f"{type(value).__name__}"
    )


def artifact_canonical_json_bytes(
    value,
) -> bytes:
    normalized = artifact_json_safe(value)

    return json.dumps(
        normalized,
        sort_keys=True,
        ensure_ascii=False,
        allow_nan=False,
        separators=(",", ":"),
    ).encode("utf-8")


def artifact_pretty_json_bytes(
    value,
) -> bytes:
    normalized = artifact_json_safe(value)

    text = json.dumps(
        normalized,
        sort_keys=True,
        ensure_ascii=False,
        allow_nan=False,
        indent=2,
    )

    return (text + "\n").encode("utf-8")


def artifact_payload_hash(
    payload,
) -> str:
    return hashlib.sha256(
        artifact_canonical_json_bytes(
            payload
        )
    ).hexdigest()


def dataframe_records_json_safe(
    frame: pd.DataFrame,
) -> list[dict]:
    if not isinstance(frame, pd.DataFrame):
        raise TypeError(
            "frame must be a pandas DataFrame."
        )

    return artifact_json_safe(frame)


# ============================================================
# ARTIFACT METADATA
# ============================================================

def attach_artifact_metadata(
    payload,
    artifact_type: str,
    artifact_schema_version: str,
    acceptance_status: str,
    row_count: int | None = None,
) -> dict:
    safe_payload = artifact_json_safe(
        payload
    )

    return {
        "artifact_metadata": {
            "artifact_type": artifact_type,
            "artifact_schema_version": (
                artifact_schema_version
            ),
            "pipeline_name": PIPELINE_NAME,
            "pipeline_version": PIPELINE_VERSION,
            "source_run_prefix": (
                SOURCE_RUN_PREFIX
            ),
            "source_set_hash": SOURCE_SET_HASH,
            "v0_1_run_id": V0_1_RUN_ID,
            "producing_notebook": (
                NOTEBOOK_FILENAME
            ),
            "created_utc": pd.Timestamp.now(
                tz="UTC"
            ).isoformat(),
            "acceptance_status": (
                acceptance_status
            ),
            "row_count": (
                int(row_count)
                if row_count is not None
                else None
            ),
            "payload_sha256": (
                artifact_payload_hash(
                    safe_payload
                )
            ),
        },
        "payload": safe_payload,
    }


# ============================================================
# SAFE ATOMIC WRITES
# ============================================================

def path_is_within(
    candidate: Path,
    root: Path,
) -> bool:
    candidate_resolved = Path(
        candidate
    ).resolve(strict=False)

    root_resolved = Path(
        root
    ).resolve(strict=False)

    try:
        candidate_resolved.relative_to(
            root_resolved
        )
        return True

    except ValueError:
        return False


def validate_v0_1_output_path(
    path: Path,
) -> Path:
    resolved_path = Path(path).resolve(
        strict=False
    )

    if not path_is_within(
        resolved_path,
        V0_1_ROOT_RESOLVED,
    ):
        raise RuntimeError(
            "Output path is outside the V0.1 root: "
            f"{resolved_path}"
        )

    if path_is_within(
        resolved_path,
        V0_0_ROOT_RESOLVED,
    ):
        raise RuntimeError(
            "Output path enters the immutable V0.0 root: "
            f"{resolved_path}"
        )

    return resolved_path


def atomic_write_bytes(
    path: Path,
    content: bytes,
) -> dict:
    target_path = validate_v0_1_output_path(
        path
    )

    target_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with tempfile.NamedTemporaryFile(
        mode="wb",
        prefix=f".{target_path.name}.",
        suffix=".tmp",
        dir=target_path.parent,
        delete=False,
    ) as temporary_file:
        temporary_path = Path(
            temporary_file.name
        )

        temporary_file.write(content)
        temporary_file.flush()

        os.fsync(
            temporary_file.fileno()
        )

    try:
        os.replace(
            temporary_path,
            target_path,
        )

    finally:
        if temporary_path.exists():
            temporary_path.unlink()

    written_bytes = target_path.read_bytes()

    v0_1_root_resolved = Path(
        V0_1_ROOT_RESOLVED
    ).resolve(strict=False)

    return {
        "resolved_path": str(target_path),
        "relative_path": str(
            target_path.relative_to(
                v0_1_root_resolved
            )
        ),
        "size_bytes": len(written_bytes),
        "sha256": hashlib.sha256(
            written_bytes
        ).hexdigest(),
    }


def atomic_write_json(
    path: Path,
    payload,
) -> dict:
    return atomic_write_bytes(
        path=path,
        content=artifact_pretty_json_bytes(
            payload
        ),
    )


def atomic_write_dataframe_csv(
    path: Path,
    frame: pd.DataFrame,
) -> dict:
    if not isinstance(frame, pd.DataFrame):
        raise TypeError(
            "frame must be a pandas DataFrame."
        )

    csv_text = frame.to_csv(
        index=False,
        lineterminator="\n",
    )

    return atomic_write_bytes(
        path=path,
        content=csv_text.encode("utf-8"),
    )


# ============================================================
# OUTPUT PAYLOADS
# ============================================================

V0_0_SOURCE_REGISTRY_OUTPUT = {
    "registry_type": "V0_0_SOURCE_REGISTRY",
    "registry_schema_version": (
        CONTRACT_SCHEMA_VERSION
    ),
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "source_set_hash": SOURCE_SET_HASH,
    "source_identity_declaration_hash": (
        SOURCE_IDENTITY_DECLARATION_HASH
    ),
    "v0_0_root": str(V0_0_ROOT_RESOLVED),
    "v0_1_root": str(V0_1_ROOT_RESOLVED),
    "v0_0_is_read_only": True,
    "registered_paths": (
        SOURCE_AUTHORITY_REGISTRY
    ),
    "authoritative_primary_sources": (
        AUTHORITATIVE_RAW_SOURCE_MANIFEST
    ),
    "source_selection_ledger": (
        SOURCE_SELECTION_LEDGER
    ),
    "reference_artifact_summary": (
        V0_0_REFERENCE_SUMMARY
    ),
    "producing_notebook": NOTEBOOK_FILENAME,
}

V0_0_SOURCE_REGISTRY_OUTPUT = (
    attach_artifact_metadata(
        payload=V0_0_SOURCE_REGISTRY_OUTPUT,
        artifact_type="V0_0_SOURCE_REGISTRY",
        artifact_schema_version=(
            CONTRACT_SCHEMA_VERSION
        ),
        acceptance_status=(
            NOTEBOOK_00_PREWRITE_STATUS
        ),
        row_count=len(
            AUTHORITATIVE_RAW_SOURCE_MANIFEST
        ),
    )
)


V0_1_RUN_CONFIG_OUTPUT = (
    attach_artifact_metadata(
        payload=V0_1_RUN_CONFIG,
        artifact_type="V0_1_RUN_CONFIG",
        artifact_schema_version=(
            CONTRACT_SCHEMA_VERSION
        ),
        acceptance_status=(
            NOTEBOOK_00_PREWRITE_STATUS
        ),
    )
)


CHRONOLOGICAL_SPLIT_OUTPUT = (
    attach_artifact_metadata(
        payload=(
            CHRONOLOGICAL_SPLIT_CONTRACT
        ),
        artifact_type=(
            "CHRONOLOGICAL_SPLIT_CONTRACT"
        ),
        artifact_schema_version=(
            CONTRACT_SCHEMA_VERSION
        ),
        acceptance_status=(
            NOTEBOOK_00_PREWRITE_STATUS
        ),
        row_count=len(
            CHRONOLOGICAL_SPLIT_TABLE
        ),
    )
)


MARKET_DATA_CONTRACT_OUTPUT = (
    attach_artifact_metadata(
        payload=MARKET_DATA_CONTRACT,
        artifact_type=(
            "MARKET_DATA_CONTRACT"
        ),
        artifact_schema_version=(
            CONTRACT_SCHEMA_VERSION
        ),
        acceptance_status=(
            NOTEBOOK_00_PREWRITE_STATUS
        ),
    )
)


MISSING_DATA_REJECTION_OUTPUT = (
    attach_artifact_metadata(
        payload=(
            MISSING_DATA_REJECTION_CONTRACT
        ),
        artifact_type=(
            "MISSING_DATA_REJECTION_CONTRACT"
        ),
        artifact_schema_version=(
            CONTRACT_SCHEMA_VERSION
        ),
        acceptance_status=(
            NOTEBOOK_00_PREWRITE_STATUS
        ),
        row_count=len(
            REJECTION_REASON_REGISTRY
        ),
    )
)


DECISION_TOLERANCE_OUTPUT = (
    attach_artifact_metadata(
        payload=(
            DECISION_AND_TOLERANCE_CONTRACT
        ),
        artifact_type=(
            "DECISION_AND_TOLERANCE_CONTRACT"
        ),
        artifact_schema_version=(
            CONTRACT_SCHEMA_VERSION
        ),
        acceptance_status=(
            NOTEBOOK_00_PREWRITE_STATUS
        ),
    )
)


RUN_IDENTITY_OUTPUT = (
    attach_artifact_metadata(
        payload=V0_1_RUN_IDENTITY_MANIFEST,
        artifact_type="V0_1_RUN_IDENTITY",
        artifact_schema_version=(
            CONTRACT_SCHEMA_VERSION
        ),
        acceptance_status=(
            NOTEBOOK_00_PREWRITE_STATUS
        ),
    )
)


ENVIRONMENT_SNAPSHOT_OUTPUT = (
    attach_artifact_metadata(
        payload=(
            EXECUTION_ENVIRONMENT_SNAPSHOT
        ),
        artifact_type=(
            "EXECUTION_ENVIRONMENT_SNAPSHOT"
        ),
        artifact_schema_version=(
            CONTRACT_SCHEMA_VERSION
        ),
        acceptance_status=(
            NOTEBOOK_00_PREWRITE_STATUS
        ),
        row_count=len(
            INSTALLED_PACKAGE_INVENTORY
        ),
    )
)


NOTEBOOK_00_AUDIT_OUTPUT = (
    attach_artifact_metadata(
        payload={
            "prewrite_decision": (
                NOTEBOOK_00_PREWRITE_DECISION
            ),
            "prewrite_validation_ledger": (
                PREWRITE_VALIDATION_LEDGER
            ),
            "source_rehash": (
                PREWRITE_SOURCE_REHASH
            ),
            "hash_reconciliation": (
                PREWRITE_HASH_RECONCILIATION
            ),
            "prior_gate_reconciliation": (
                PRIOR_GATE_RECONCILIATION
            ),
            "status_table_reconciliation": (
                STATUS_TABLE_RECONCILIATION
            ),
        },
        artifact_type=(
            "NOTEBOOK_00_FINAL_AUDIT"
        ),
        artifact_schema_version=(
            CONTRACT_SCHEMA_VERSION
        ),
        acceptance_status=(
            NOTEBOOK_00_PREWRITE_STATUS
        ),
    )
)


# ============================================================
# AUTHORITATIVE OUTPUT SPECIFICATION
# ============================================================

CONFIG_OUTPUT_SPECS = [
    {
        "artifact_type": (
            "V0_0_SOURCE_REGISTRY"
        ),
        "filename": (
            SOURCE_REGISTRY_ARTIFACT_NAME
        ),
        "payload": (
            V0_0_SOURCE_REGISTRY_OUTPUT
        ),
    },
    {
        "artifact_type": "V0_1_RUN_CONFIG",
        "filename": RUN_CONFIG_ARTIFACT_NAME,
        "payload": V0_1_RUN_CONFIG_OUTPUT,
    },
    {
        "artifact_type": (
            "CHRONOLOGICAL_SPLIT_CONTRACT"
        ),
        "filename": make_v0_1_artifact_name(
            artifact_name="split_contract",
            extension="json",
        ),
        "payload": CHRONOLOGICAL_SPLIT_OUTPUT,
    },
    {
        "artifact_type": (
            "MARKET_DATA_CONTRACT"
        ),
        "filename": make_v0_1_artifact_name(
            artifact_name=(
                "market_data_contract"
            ),
            extension="json",
        ),
        "payload": MARKET_DATA_CONTRACT_OUTPUT,
    },
    {
        "artifact_type": (
            "MISSING_DATA_REJECTION_CONTRACT"
        ),
        "filename": make_v0_1_artifact_name(
            artifact_name=(
                "missing_data_rejection_contract"
            ),
            extension="json",
        ),
        "payload": MISSING_DATA_REJECTION_OUTPUT,
    },
    {
        "artifact_type": (
            "DECISION_AND_TOLERANCE_CONTRACT"
        ),
        "filename": make_v0_1_artifact_name(
            artifact_name=(
                "decision_and_tolerance_contract"
            ),
            extension="json",
        ),
        "payload": DECISION_TOLERANCE_OUTPUT,
    },
]


MANIFEST_OUTPUT_SPECS = [
    {
        "artifact_type": (
            "V0_1_RUN_IDENTITY"
        ),
        "filename": RUN_IDENTITY_ARTIFACT_NAME,
        "payload": RUN_IDENTITY_OUTPUT,
    },
    {
        "artifact_type": (
            "EXECUTION_ENVIRONMENT_SNAPSHOT"
        ),
        "filename": make_v0_1_artifact_name(
            artifact_name=(
                "execution_environment_snapshot"
            ),
            extension="json",
        ),
        "payload": ENVIRONMENT_SNAPSHOT_OUTPUT,
    },
]


AUDIT_JSON_OUTPUT_SPECS = [
    {
        "artifact_type": (
            "NOTEBOOK_00_FINAL_AUDIT"
        ),
        "filename": make_v0_1_artifact_name(
            artifact_name=(
                "notebook_00_final_audit"
            ),
            extension="json",
        ),
        "payload": NOTEBOOK_00_AUDIT_OUTPUT,
    },
]


AUDIT_TABLE_OUTPUT_SPECS = [
    {
        "artifact_type": (
            "AUTHORITATIVE_RAW_SOURCE_MANIFEST"
        ),
        "artifact_name": (
            "authoritative_raw_source_manifest"
        ),
        "frame": (
            AUTHORITATIVE_RAW_SOURCE_MANIFEST
        ),
    },
    {
        "artifact_type": (
            "REFERENCE_ARTIFACT_MANIFEST"
        ),
        "artifact_name": (
            "v0_0_reference_artifact_manifest"
        ),
        "frame": (
            V0_0_REFERENCE_ARTIFACT_MANIFEST
        ),
    },
    {
        "artifact_type": (
            "CHRONOLOGICAL_SPLIT_TABLE"
        ),
        "artifact_name": (
            "chronological_split_table"
        ),
        "frame": CHRONOLOGICAL_SPLIT_TABLE,
    },
    {
        "artifact_type": (
            "REJECTION_REASON_REGISTRY"
        ),
        "artifact_name": (
            "rejection_reason_registry"
        ),
        "frame": REJECTION_REASON_REGISTRY,
    },
    {
        "artifact_type": (
            "FIXED_DECISION_REGISTRY"
        ),
        "artifact_name": (
            "fixed_decision_registry"
        ),
        "frame": FIXED_DECISION_REGISTRY,
    },
    {
        "artifact_type": (
            "EXPLORATORY_DECISION_REGISTRY"
        ),
        "artifact_name": (
            "exploratory_decision_registry"
        ),
        "frame": (
            EXPLORATORY_DECISION_REGISTRY
        ),
    },
    {
        "artifact_type": (
            "TOLERANCE_REGISTRY"
        ),
        "artifact_name": (
            "tolerance_registry"
        ),
        "frame": TOLERANCE_REGISTRY,
    },
    {
        "artifact_type": (
            "OBSERVED_STREAM_SCHEMA"
        ),
        "artifact_name": (
            "observed_stream_schema"
        ),
        "frame": OBSERVED_STREAM_SCHEMA,
    },
    {
        "artifact_type": (
            "PREWRITE_SOURCE_REHASH"
        ),
        "artifact_name": (
            "prewrite_source_rehash"
        ),
        "frame": PREWRITE_SOURCE_REHASH,
    },
    {
        "artifact_type": (
            "PREWRITE_HASH_RECONCILIATION"
        ),
        "artifact_name": (
            "prewrite_hash_reconciliation"
        ),
        "frame": (
            PREWRITE_HASH_RECONCILIATION
        ),
    },
    {
        "artifact_type": (
            "PREWRITE_VALIDATION_LEDGER"
        ),
        "artifact_name": (
            "prewrite_validation_ledger"
        ),
        "frame": (
            PREWRITE_VALIDATION_LEDGER
        ),
    },
    {
        "artifact_type": (
            "NOTEBOOK_00_PREWRITE_DECISION"
        ),
        "artifact_name": (
            "notebook_00_prewrite_decision"
        ),
        "frame": (
            NOTEBOOK_00_PREWRITE_DECISION
        ),
    },
]


# ============================================================
# WRITE AUTHORITATIVE OUTPUTS
# ============================================================

written_artifact_rows = []


for output_spec in CONFIG_OUTPUT_SPECS:
    target_path = (
        resolve_path(V0_1_CONFIG_ROOT)
        / output_spec["filename"]
    )

    write_result = atomic_write_json(
        path=target_path,
        payload=output_spec["payload"],
    )

    written_artifact_rows.append(
        {
            "artifact_type": (
                output_spec["artifact_type"]
            ),
            "format": "JSON",
            "acceptance_status": (
                NOTEBOOK_00_PREWRITE_STATUS
            ),
            **write_result,
        }
    )


for output_spec in MANIFEST_OUTPUT_SPECS:
    target_path = (
        resolve_path(V0_1_MANIFEST_ROOT)
        / output_spec["filename"]
    )

    write_result = atomic_write_json(
        path=target_path,
        payload=output_spec["payload"],
    )

    written_artifact_rows.append(
        {
            "artifact_type": (
                output_spec["artifact_type"]
            ),
            "format": "JSON",
            "acceptance_status": (
                NOTEBOOK_00_PREWRITE_STATUS
            ),
            **write_result,
        }
    )


for output_spec in AUDIT_JSON_OUTPUT_SPECS:
    target_path = (
        resolve_path(V0_1_AUDIT_ROOT)
        / output_spec["filename"]
    )

    write_result = atomic_write_json(
        path=target_path,
        payload=output_spec["payload"],
    )

    written_artifact_rows.append(
        {
            "artifact_type": (
                output_spec["artifact_type"]
            ),
            "format": "JSON",
            "acceptance_status": (
                NOTEBOOK_00_PREWRITE_STATUS
            ),
            **write_result,
        }
    )


for output_spec in AUDIT_TABLE_OUTPUT_SPECS:
    filename = make_v0_1_artifact_name(
        artifact_name=(
            output_spec["artifact_name"]
        ),
        extension="csv",
    )

    target_path = (
        resolve_path(V0_1_AUDIT_ROOT)
        / filename
    )

    write_result = (
        atomic_write_dataframe_csv(
            path=target_path,
            frame=output_spec["frame"],
        )
    )

    written_artifact_rows.append(
        {
            "artifact_type": (
                output_spec["artifact_type"]
            ),
            "format": "CSV",
            "acceptance_status": (
                NOTEBOOK_00_PREWRITE_STATUS
            ),
            "row_count": int(
                len(output_spec["frame"])
            ),
            **write_result,
        }
    )


NOTEBOOK_00_WRITTEN_ARTIFACTS = (
    pd.DataFrame(written_artifact_rows)
)


# ============================================================
# OUTPUT MANIFEST
# ============================================================

NOTEBOOK_00_OUTPUT_MANIFEST_PAYLOAD = {
    "manifest_type": (
        "NOTEBOOK_00_OUTPUT_MANIFEST"
    ),
    "manifest_schema_version": (
        CONTRACT_SCHEMA_VERSION
    ),
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "source_set_hash": SOURCE_SET_HASH,
    "v0_1_run_id": V0_1_RUN_ID,
    "v0_1_run_config_hash": (
        V0_1_RUN_CONFIG_HASH
    ),
    "v0_1_run_identity_hash": (
        V0_1_RUN_IDENTITY_HASH
    ),
    "operating_mode": V0_1_OPERATING_MODE,
    "acceptance_status": (
        NOTEBOOK_00_PREWRITE_STATUS
    ),
    "artifact_count": int(
        len(NOTEBOOK_00_WRITTEN_ARTIFACTS)
    ),
    "artifacts": (
        NOTEBOOK_00_WRITTEN_ARTIFACTS
    ),
    "next_notebook": (
        "01_RAW_DATA_AUDIT.ipynb"
    ),
    "producing_notebook": NOTEBOOK_FILENAME,
}


NOTEBOOK_00_OUTPUT_MANIFEST = (
    attach_artifact_metadata(
        payload=(
            NOTEBOOK_00_OUTPUT_MANIFEST_PAYLOAD
        ),
        artifact_type=(
            "NOTEBOOK_00_OUTPUT_MANIFEST"
        ),
        artifact_schema_version=(
            CONTRACT_SCHEMA_VERSION
        ),
        acceptance_status=(
            NOTEBOOK_00_PREWRITE_STATUS
        ),
        row_count=len(
            NOTEBOOK_00_WRITTEN_ARTIFACTS
        ),
    )
)


NOTEBOOK_00_OUTPUT_MANIFEST_NAME = (
    make_v0_1_artifact_name(
        artifact_name=(
            "notebook_00_output_manifest"
        ),
        extension="json",
    )
)


NOTEBOOK_00_OUTPUT_MANIFEST_PATH = (
    resolve_path(V0_1_MANIFEST_ROOT)
    / NOTEBOOK_00_OUTPUT_MANIFEST_NAME
)


manifest_write_result = atomic_write_json(
    path=NOTEBOOK_00_OUTPUT_MANIFEST_PATH,
    payload=NOTEBOOK_00_OUTPUT_MANIFEST,
)


NOTEBOOK_00_OUTPUT_MANIFEST_HASH = (
    manifest_write_result["sha256"]
)


# ============================================================
# POST-WRITE VERIFICATION
# ============================================================

postwrite_rows = []


for artifact_row in (
    NOTEBOOK_00_WRITTEN_ARTIFACTS.itertuples(
        index=False
    )
):
    artifact_path = Path(
        artifact_row.resolved_path
    )

    current_bytes = artifact_path.read_bytes()

    current_size_bytes = len(
        current_bytes
    )

    current_sha256 = hashlib.sha256(
        current_bytes
    ).hexdigest()

    postwrite_rows.append(
        {
            "artifact_type": (
                artifact_row.artifact_type
            ),
            "relative_path": (
                artifact_row.relative_path
            ),
            "expected_size_bytes": int(
                artifact_row.size_bytes
            ),
            "current_size_bytes": (
                current_size_bytes
            ),
            "expected_sha256": (
                artifact_row.sha256
            ),
            "current_sha256": (
                current_sha256
            ),
            "size_matches": (
                current_size_bytes
                == int(
                    artifact_row.size_bytes
                )
            ),
            "hash_matches": (
                current_sha256
                == artifact_row.sha256
            ),
        }
    )


NOTEBOOK_00_POSTWRITE_VERIFICATION = (
    pd.DataFrame(postwrite_rows)
)


postwrite_passed = bool(
    not NOTEBOOK_00_POSTWRITE_VERIFICATION.empty
    and NOTEBOOK_00_POSTWRITE_VERIFICATION[
        "size_matches"
    ].all()
    and NOTEBOOK_00_POSTWRITE_VERIFICATION[
        "hash_matches"
    ].all()
)


if not postwrite_passed:
    failed_outputs = (
        NOTEBOOK_00_POSTWRITE_VERIFICATION.loc[
            ~NOTEBOOK_00_POSTWRITE_VERIFICATION[
                "size_matches"
            ]
            | ~NOTEBOOK_00_POSTWRITE_VERIFICATION[
                "hash_matches"
            ],
            "relative_path",
        ]
        .astype(str)
        .tolist()
    )

    raise RuntimeError(
        "Post-write verification failed for: "
        + " | ".join(failed_outputs)
    )


display(NOTEBOOK_00_WRITTEN_ARTIFACTS)
display(NOTEBOOK_00_POSTWRITE_VERIFICATION)


print("Notebook 00 atomic artifact writes: PASS")
print(
    "Authoritative artifacts written: "
    f"{len(NOTEBOOK_00_WRITTEN_ARTIFACTS):,}"
)
print(
    "Output manifest: "
    f"{manifest_write_result['relative_path']}"
)
print(
    "Output manifest SHA-256: "
    f"{NOTEBOOK_00_OUTPUT_MANIFEST_HASH}"
)
print(
    "Post-write verification: "
    f"{postwrite_passed}"
)
print(
    "Notebook 00 status: "
    f"{NOTEBOOK_00_PREWRITE_STATUS}"
)
print(
    "Next notebook: "
    "01_RAW_DATA_AUDIT.ipynb"
)

,artifact_type,format,acceptance_status,resolved_path,relative_path,size_bytes,sha256,row_count
0,V0_0_SOURCE_REGISTRY,JSON,CONDITIONAL PASS,D:\Clown Project\V0.1\config\BTCUSDT_spot_2026...,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,21284,3131161456fdc9e6fcf2500d846ffcf12b9c609dfa1d97...,NaN
1,V0_1_RUN_CONFIG,JSON,CONDITIONAL PASS,D:\Clown Project\V0.1\config\BTCUSDT_spot_2026...,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,12529,cec9f2deb240944af04717657c1d35ffb093872e717849...,NaN
2,CHRONOLOGICAL_SPLIT_CONTRACT,JSON,CONDITIONAL PASS,D:\Clown Project\V0.1\config\BTCUSDT_spot_2026...,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,7330,1037581968095b5e262429fd86004d1abf3e5b691b330e...,NaN
3,MARKET_DATA_CONTRACT,JSON,CONDITIONAL PASS,D:\Clown Project\V0.1\config\BTCUSDT_spot_2026...,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,3556,8ef328c2f0d828c18ebfeb52a61ff06e9a13b1b6b975f4...,NaN
4,MISSING_DATA_REJECTION_CONTRACT,JSON,CONDITIONAL PASS,D:\Clown Project\V0.1\config\BTCUSDT_spot_2026...,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,24687,d027f32d05e04591badb8b7f71b37487e83872ee66b4b6...,NaN
5,DECISION_AND_TOLERANCE_CONTRACT,JSON,CONDITIONAL PASS,D:\Clown Project\V0.1\config\BTCUSDT_spot_2026...,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,13413,ba6df1590bc2355ca87771b8ee3b8c48eb06c5f5a82da2...,NaN
6,V0_1_RUN_IDENTITY,JSON,CONDITIONAL PASS,D:\Clown Project\V0.1\artifacts\manifests\BTCU...,artifacts\manifests\BTCUSDT_spot_20260710T0637...,3320,86775b812fc43571d030855ce5382a09357a31423c2455...,NaN
7,EXECUTION_ENVIRONMENT_SNAPSHOT,JSON,CONDITIONAL PASS,D:\Clown Project\V0.1\artifacts\manifests\BTCU...,artifacts\manifests\BTCUSDT_spot_20260710T0637...,3203,355c2ed73c10311264019cf054b6defe80c6de09a17bbb...,NaN
8,NOTEBOOK_00_FINAL_AUDIT,JSON,CONDITIONAL PASS,D:\Clown Project\V0.1\artifacts\audit_tables\B...,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,10801,1f8c90ffbe5dd3efa92c833f7134d40badb92d7da9e803...,NaN
9,AUTHORITATIVE_RAW_SOURCE_MANIFEST,CSV,CONDITIONAL PASS,D:\Clown Project\V0.1\artifacts\audit_tables\B...,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,2926,1cd4fd4165eab825562540df65534d82f5c1aba94df81c...,5.0


,artifact_type,relative_path,expected_size_bytes,current_size_bytes,expected_sha256,current_sha256,size_matches,hash_matches
0,V0_0_SOURCE_REGISTRY,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,21284,21284,3131161456fdc9e6fcf2500d846ffcf12b9c609dfa1d97...,3131161456fdc9e6fcf2500d846ffcf12b9c609dfa1d97...,True,True
1,V0_1_RUN_CONFIG,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,12529,12529,cec9f2deb240944af04717657c1d35ffb093872e717849...,cec9f2deb240944af04717657c1d35ffb093872e717849...,True,True
2,CHRONOLOGICAL_SPLIT_CONTRACT,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,7330,7330,1037581968095b5e262429fd86004d1abf3e5b691b330e...,1037581968095b5e262429fd86004d1abf3e5b691b330e...,True,True
3,MARKET_DATA_CONTRACT,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,3556,3556,8ef328c2f0d828c18ebfeb52a61ff06e9a13b1b6b975f4...,8ef328c2f0d828c18ebfeb52a61ff06e9a13b1b6b975f4...,True,True
4,MISSING_DATA_REJECTION_CONTRACT,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,24687,24687,d027f32d05e04591badb8b7f71b37487e83872ee66b4b6...,d027f32d05e04591badb8b7f71b37487e83872ee66b4b6...,True,True
5,DECISION_AND_TOLERANCE_CONTRACT,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,13413,13413,ba6df1590bc2355ca87771b8ee3b8c48eb06c5f5a82da2...,ba6df1590bc2355ca87771b8ee3b8c48eb06c5f5a82da2...,True,True
6,V0_1_RUN_IDENTITY,artifacts\manifests\BTCUSDT_spot_20260710T0637...,3320,3320,86775b812fc43571d030855ce5382a09357a31423c2455...,86775b812fc43571d030855ce5382a09357a31423c2455...,True,True
7,EXECUTION_ENVIRONMENT_SNAPSHOT,artifacts\manifests\BTCUSDT_spot_20260710T0637...,3203,3203,355c2ed73c10311264019cf054b6defe80c6de09a17bbb...,355c2ed73c10311264019cf054b6defe80c6de09a17bbb...,True,True
8,NOTEBOOK_00_FINAL_AUDIT,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,10801,10801,1f8c90ffbe5dd3efa92c833f7134d40badb92d7da9e803...,1f8c90ffbe5dd3efa92c833f7134d40badb92d7da9e803...,True,True
9,AUTHORITATIVE_RAW_SOURCE_MANIFEST,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,2926,2926,1cd4fd4165eab825562540df65534d82f5c1aba94df81c...,1cd4fd4165eab825562540df65534d82f5c1aba94df81c...,True,True


Notebook 00 atomic artifact writes: PASS
Authoritative artifacts written: 21
Output manifest: artifacts\manifests\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__00_V01_RUN_CONTRACT__notebook_00_output_manifest.json
Output manifest SHA-256: e12966301d92e61656715f2a6d397786820de836d8156dd70092250619cbb00e
Post-write verification: True
Notebook 00 status: CONDITIONAL PASS
Next notebook: 01_RAW_DATA_AUDIT.ipynb


In [30]:
# ============================================================
# FINAL NOTEBOOK 00 SYNTHESIS AND NOTEBOOK 01 HANDOFF
# ============================================================

NOTEBOOK_00_FINALIZED_UTC = pd.Timestamp.now(
    tz="UTC"
).isoformat()


# ============================================================
# RELOAD THE SAVED OUTPUT MANIFEST FROM DISK
# ============================================================

saved_output_manifest_path = Path(
    manifest_write_result["resolved_path"]
).resolve(strict=True)

with saved_output_manifest_path.open(
    "r",
    encoding="utf-8",
) as handle:
    SAVED_NOTEBOOK_00_OUTPUT_MANIFEST = json.load(
        handle
    )


if not isinstance(
    SAVED_NOTEBOOK_00_OUTPUT_MANIFEST,
    dict,
):
    raise RuntimeError(
        "Saved Notebook 00 output manifest is not a JSON mapping."
    )

saved_manifest_metadata = (
    SAVED_NOTEBOOK_00_OUTPUT_MANIFEST.get(
        "artifact_metadata"
    )
)

saved_manifest_payload = (
    SAVED_NOTEBOOK_00_OUTPUT_MANIFEST.get(
        "payload"
    )
)

if not isinstance(saved_manifest_metadata, dict):
    raise RuntimeError(
        "Saved Notebook 00 output manifest lacks artifact metadata."
    )

if not isinstance(saved_manifest_payload, dict):
    raise RuntimeError(
        "Saved Notebook 00 output manifest lacks a payload mapping."
    )

saved_manifest_artifacts = (
    saved_manifest_payload.get("artifacts")
)

if not isinstance(saved_manifest_artifacts, list):
    raise RuntimeError(
        "Saved Notebook 00 output manifest does not contain an "
        "artifact-record list."
    )


SAVED_OUTPUT_ARTIFACT_TABLE = pd.DataFrame(
    saved_manifest_artifacts
)

required_saved_artifact_columns = {
    "artifact_type",
    "format",
    "acceptance_status",
    "resolved_path",
    "relative_path",
    "size_bytes",
    "sha256",
}

missing_saved_artifact_columns = sorted(
    required_saved_artifact_columns
    - set(SAVED_OUTPUT_ARTIFACT_TABLE.columns)
)

if missing_saved_artifact_columns:
    raise RuntimeError(
        "Saved output manifest is missing artifact columns: "
        + " | ".join(missing_saved_artifact_columns)
    )


# ============================================================
# VERIFY EVERY MANIFESTED ARTIFACT FROM DISK
# ============================================================

final_artifact_verification_rows = []

for artifact_row in (
    SAVED_OUTPUT_ARTIFACT_TABLE.itertuples(
        index=False
    )
):
    artifact_path = Path(
        artifact_row.resolved_path
    ).resolve(strict=False)

    path_inside_v0_1 = path_is_within(
        artifact_path,
        V0_1_ROOT_RESOLVED,
    )

    path_inside_v0_0 = path_is_within(
        artifact_path,
        V0_0_ROOT_RESOLVED,
    )

    file_exists = (
        artifact_path.exists()
        and artifact_path.is_file()
    )

    if file_exists:
        file_bytes = artifact_path.read_bytes()

        observed_size_bytes = len(file_bytes)

        observed_sha256 = hashlib.sha256(
            file_bytes
        ).hexdigest()

    else:
        observed_size_bytes = None
        observed_sha256 = None

    expected_size_bytes = int(
        artifact_row.size_bytes
    )

    expected_sha256 = str(
        artifact_row.sha256
    ).lower()

    size_matches = bool(
        observed_size_bytes
        == expected_size_bytes
    )

    hash_matches = bool(
        observed_sha256
        == expected_sha256
    )

    artifact_passed = bool(
        file_exists
        and path_inside_v0_1
        and not path_inside_v0_0
        and size_matches
        and hash_matches
    )

    final_artifact_verification_rows.append(
        {
            "artifact_type": (
                artifact_row.artifact_type
            ),
            "format": artifact_row.format,
            "relative_path": (
                artifact_row.relative_path
            ),
            "file_exists": file_exists,
            "inside_v0_1": path_inside_v0_1,
            "inside_v0_0": path_inside_v0_0,
            "expected_size_bytes": (
                expected_size_bytes
            ),
            "observed_size_bytes": (
                observed_size_bytes
            ),
            "size_matches": size_matches,
            "expected_sha256": expected_sha256,
            "observed_sha256": observed_sha256,
            "hash_matches": hash_matches,
            "artifact_passed": artifact_passed,
        }
    )


NOTEBOOK_00_FINAL_ARTIFACT_VERIFICATION = (
    pd.DataFrame(
        final_artifact_verification_rows
    )
)


# ============================================================
# VERIFY THE OUTPUT MANIFEST ITSELF
# ============================================================

saved_manifest_bytes = (
    saved_output_manifest_path.read_bytes()
)

saved_manifest_size_bytes = len(
    saved_manifest_bytes
)

saved_manifest_sha256 = hashlib.sha256(
    saved_manifest_bytes
).hexdigest()

output_manifest_size_matches = bool(
    saved_manifest_size_bytes
    == int(manifest_write_result["size_bytes"])
)

output_manifest_hash_matches = bool(
    saved_manifest_sha256
    == str(
        manifest_write_result["sha256"]
    ).lower()
)

output_manifest_identity_matches = bool(
    saved_manifest_metadata.get(
        "artifact_type"
    )
    == "NOTEBOOK_00_OUTPUT_MANIFEST"
    and saved_manifest_payload.get(
        "source_run_prefix"
    )
    == SOURCE_RUN_PREFIX
    and saved_manifest_payload.get(
        "v0_1_run_id"
    )
    == V0_1_RUN_ID
    and saved_manifest_payload.get(
        "v0_1_run_config_hash"
    )
    == V0_1_RUN_CONFIG_HASH
    and saved_manifest_payload.get(
        "v0_1_run_identity_hash"
    )
    == V0_1_RUN_IDENTITY_HASH
)

manifest_declared_artifact_count = safe_integer(
    saved_manifest_payload.get(
        "artifact_count"
    )
)

manifest_actual_artifact_count = int(
    len(SAVED_OUTPUT_ARTIFACT_TABLE)
)

output_manifest_count_matches = bool(
    manifest_declared_artifact_count
    == manifest_actual_artifact_count
)


# ============================================================
# NOTEBOOK 01 REQUIRED INPUT HANDOFF
# ============================================================

NOTEBOOK_01_REQUIRED_ARTIFACT_TYPES = [
    "V0_0_SOURCE_REGISTRY",
    "V0_1_RUN_CONFIG",
    "CHRONOLOGICAL_SPLIT_CONTRACT",
    "MARKET_DATA_CONTRACT",
    "MISSING_DATA_REJECTION_CONTRACT",
    "DECISION_AND_TOLERANCE_CONTRACT",
    "V0_1_RUN_IDENTITY",
    "EXECUTION_ENVIRONMENT_SNAPSHOT",
]


handoff_rows = []

for required_artifact_type in (
    NOTEBOOK_01_REQUIRED_ARTIFACT_TYPES
):
    matches = SAVED_OUTPUT_ARTIFACT_TABLE.loc[
        SAVED_OUTPUT_ARTIFACT_TABLE[
            "artifact_type"
        ].eq(required_artifact_type)
    ]

    if len(matches) != 1:
        handoff_rows.append(
            {
                "artifact_type": (
                    required_artifact_type
                ),
                "available": False,
                "format": None,
                "relative_path": None,
                "resolved_path": None,
                "sha256": None,
            }
        )

        continue

    matched_row = matches.iloc[0]

    handoff_rows.append(
        {
            "artifact_type": (
                required_artifact_type
            ),
            "available": True,
            "format": matched_row["format"],
            "relative_path": (
                matched_row["relative_path"]
            ),
            "resolved_path": (
                matched_row["resolved_path"]
            ),
            "sha256": matched_row["sha256"],
        }
    )


handoff_rows.append(
    {
        "artifact_type": (
            "NOTEBOOK_00_OUTPUT_MANIFEST"
        ),
        "available": True,
        "format": "JSON",
        "relative_path": (
            manifest_write_result[
                "relative_path"
            ]
        ),
        "resolved_path": str(
            saved_output_manifest_path
        ),
        "sha256": saved_manifest_sha256,
    }
)


NOTEBOOK_01_HANDOFF_TABLE = pd.DataFrame(
    handoff_rows
)

notebook_01_required_inputs_available = bool(
    NOTEBOOK_01_HANDOFF_TABLE[
        "available"
    ].all()
)

notebook_01_required_inputs_unique = bool(
    NOTEBOOK_01_HANDOFF_TABLE[
        "artifact_type"
    ].is_unique
)


NOTEBOOK_01_HANDOFF = {
    "handoff_type": (
        "NOTEBOOK_00_TO_NOTEBOOK_01"
    ),
    "handoff_schema_version": (
        CONTRACT_SCHEMA_VERSION
    ),
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "source_set_hash": SOURCE_SET_HASH,
    "v0_1_run_id": V0_1_RUN_ID,
    "v0_1_run_config_hash": (
        V0_1_RUN_CONFIG_HASH
    ),
    "v0_1_run_identity_hash": (
        V0_1_RUN_IDENTITY_HASH
    ),
    "operating_mode": V0_1_OPERATING_MODE,
    "contract_status": (
        V0_1_CONTRACT_STATUS
    ),
    "claim_bearing_holdout_available": (
        False
    ),
    "partitions_are_independent": False,
    "next_notebook": (
        "01_RAW_DATA_AUDIT.ipynb"
    ),
    "required_saved_inputs": (
        artifact_json_safe(
            NOTEBOOK_01_HANDOFF_TABLE
        )
    ),
    "output_manifest_sha256": (
        saved_manifest_sha256
    ),
    "notebook_01_authorized": bool(
        RUN_CONFIG_PERMISSIONS[
            "proceed_to_notebook_01"
        ]
    ),
}


# ============================================================
# FINAL NOTEBOOK 00 GATES
# ============================================================

required_artifact_types_present = bool(
    set(
        NOTEBOOK_01_REQUIRED_ARTIFACT_TYPES
    ).issubset(
        set(
            SAVED_OUTPUT_ARTIFACT_TABLE[
                "artifact_type"
            ]
        )
    )
)

all_manifested_artifacts_valid = bool(
    not NOTEBOOK_00_FINAL_ARTIFACT_VERIFICATION.empty
    and NOTEBOOK_00_FINAL_ARTIFACT_VERIFICATION[
        "artifact_passed"
    ].all()
)

v0_0_source_state_preserved = bool(
    PREWRITE_SOURCE_REHASH[
        "status"
    ].eq("PASS").all()
)

notebook_01_authorized = bool(
    RUN_CONFIG_PERMISSIONS[
        "proceed_to_notebook_01"
    ]
)


NOTEBOOK_00_FINAL_GATES = pd.DataFrame(
    [
        {
            "gate": (
                "saved_output_manifest_readable"
            ),
            "passed": True,
            "severity": "CRITICAL",
            "evidence": str(
                saved_output_manifest_path
            ),
        },
        {
            "gate": (
                "output_manifest_identity_matches"
            ),
            "passed": (
                output_manifest_identity_matches
            ),
            "severity": "CRITICAL",
            "evidence": (
                f"source_run_prefix="
                f"{saved_manifest_payload.get('source_run_prefix')}; "
                f"v0_1_run_id="
                f"{saved_manifest_payload.get('v0_1_run_id')}"
            ),
        },
        {
            "gate": (
                "output_manifest_artifact_count_matches"
            ),
            "passed": (
                output_manifest_count_matches
            ),
            "severity": "CRITICAL",
            "evidence": (
                f"declared="
                f"{manifest_declared_artifact_count}; "
                f"observed="
                f"{manifest_actual_artifact_count}"
            ),
        },
        {
            "gate": (
                "output_manifest_size_matches"
            ),
            "passed": (
                output_manifest_size_matches
            ),
            "severity": "CRITICAL",
            "evidence": (
                f"expected="
                f"{manifest_write_result['size_bytes']}; "
                f"observed="
                f"{saved_manifest_size_bytes}"
            ),
        },
        {
            "gate": (
                "output_manifest_hash_matches"
            ),
            "passed": (
                output_manifest_hash_matches
            ),
            "severity": "CRITICAL",
            "evidence": saved_manifest_sha256,
        },
        {
            "gate": (
                "all_manifested_artifacts_valid"
            ),
            "passed": (
                all_manifested_artifacts_valid
            ),
            "severity": "CRITICAL",
            "evidence": (
                f"valid="
                f"{int(NOTEBOOK_00_FINAL_ARTIFACT_VERIFICATION['artifact_passed'].sum())}/"
                f"{len(NOTEBOOK_00_FINAL_ARTIFACT_VERIFICATION)}"
            ),
        },
        {
            "gate": (
                "required_notebook_01_artifacts_present"
            ),
            "passed": (
                required_artifact_types_present
            ),
            "severity": "CRITICAL",
            "evidence": (
                f"required="
                f"{len(NOTEBOOK_01_REQUIRED_ARTIFACT_TYPES)}"
            ),
        },
        {
            "gate": (
                "notebook_01_handoff_complete"
            ),
            "passed": bool(
                notebook_01_required_inputs_available
                and notebook_01_required_inputs_unique
            ),
            "severity": "CRITICAL",
            "evidence": (
                f"available="
                f"{notebook_01_required_inputs_available}; "
                f"unique="
                f"{notebook_01_required_inputs_unique}"
            ),
        },
        {
            "gate": (
                "v0_0_source_state_preserved"
            ),
            "passed": (
                v0_0_source_state_preserved
            ),
            "severity": "CRITICAL",
            "evidence": (
                f"unchanged="
                f"{int(PREWRITE_SOURCE_REHASH['status'].eq('PASS').sum())}/"
                f"{len(PREWRITE_SOURCE_REHASH)}"
            ),
        },
        {
            "gate": (
                "notebook_01_execution_authorized"
            ),
            "passed": notebook_01_authorized,
            "severity": "CRITICAL",
            "evidence": (
                "proceed_to_notebook_01="
                f"{notebook_01_authorized}"
            ),
        },
        {
            "gate": (
                "full_statistical_mode_available"
            ),
            "passed": bool(
                V0_1_OPERATING_MODE
                == "FULL_STATISTICAL_MODE"
            ),
            "severity": "WARNING",
            "evidence": V0_1_OPERATING_MODE,
        },
        {
            "gate": (
                "claim_bearing_holdout_available"
            ),
            "passed": bool(
                CHRONOLOGICAL_SPLIT_CONTRACT[
                    "claim_bearing_holdout_available"
                ]
            ),
            "severity": "WARNING",
            "evidence": (
                "available="
                f"{CHRONOLOGICAL_SPLIT_CONTRACT['claim_bearing_holdout_available']}"
            ),
        },
        {
            "gate": (
                "git_repository_provenance_available"
            ),
            "passed": bool(
                GIT_REPOSITORY_ROOT is not None
            ),
            "severity": "WARNING",
            "evidence": (
                f"repository_root="
                f"{GIT_REPOSITORY_ROOT}"
            ),
        },
    ]
)


NOTEBOOK_00_FINAL_GATES["status"] = np.select(
    [
        NOTEBOOK_00_FINAL_GATES["passed"],
        NOTEBOOK_00_FINAL_GATES[
            "severity"
        ].eq("CRITICAL"),
    ],
    [
        "PASS",
        "FAIL",
    ],
    default="WARNING",
)


final_critical_failures = (
    NOTEBOOK_00_FINAL_GATES.loc[
        NOTEBOOK_00_FINAL_GATES[
            "severity"
        ].eq("CRITICAL")
        & ~NOTEBOOK_00_FINAL_GATES[
            "passed"
        ]
    ]
)

final_warnings = (
    NOTEBOOK_00_FINAL_GATES.loc[
        NOTEBOOK_00_FINAL_GATES[
            "severity"
        ].eq("WARNING")
        & ~NOTEBOOK_00_FINAL_GATES[
            "passed"
        ]
    ]
)


if not final_critical_failures.empty:
    NOTEBOOK_00_FINAL_STATUS = "FAIL"

elif V0_1_CONTRACT_STATUS == "CONDITIONAL PASS":
    NOTEBOOK_00_FINAL_STATUS = (
        "CONDITIONAL PASS"
    )

elif not final_warnings.empty:
    NOTEBOOK_00_FINAL_STATUS = "WARNING"

else:
    NOTEBOOK_00_FINAL_STATUS = "PASS"


NOTEBOOK_00_FINAL_SYNTHESIS = pd.DataFrame(
    [
        {
            "pipeline_name": PIPELINE_NAME,
            "pipeline_version": PIPELINE_VERSION,
            "notebook": NOTEBOOK_FILENAME,
            "source_run_prefix": (
                SOURCE_RUN_PREFIX
            ),
            "source_set_hash": SOURCE_SET_HASH,
            "v0_1_run_id": V0_1_RUN_ID,
            "v0_1_run_config_hash": (
                V0_1_RUN_CONFIG_HASH
            ),
            "v0_1_run_identity_hash": (
                V0_1_RUN_IDENTITY_HASH
            ),
            "output_manifest_sha256": (
                saved_manifest_sha256
            ),
            "operating_mode": (
                V0_1_OPERATING_MODE
            ),
            "final_status": (
                NOTEBOOK_00_FINAL_STATUS
            ),
            "manifested_artifact_count": (
                manifest_actual_artifact_count
            ),
            "critical_failure_count": int(
                len(final_critical_failures)
            ),
            "warning_count": int(
                len(final_warnings)
            ),
            "notebook_01_authorized": (
                notebook_01_authorized
            ),
            "next_notebook": (
                "01_RAW_DATA_AUDIT.ipynb"
            ),
            "finalized_utc": (
                NOTEBOOK_00_FINALIZED_UTC
            ),
        }
    ]
)


display(
    NOTEBOOK_00_FINAL_ARTIFACT_VERIFICATION[
        [
            "artifact_type",
            "relative_path",
            "file_exists",
            "size_matches",
            "hash_matches",
            "artifact_passed",
        ]
    ]
)

display(NOTEBOOK_01_HANDOFF_TABLE)
display(NOTEBOOK_00_FINAL_GATES)
display(NOTEBOOK_00_FINAL_SYNTHESIS)


if not final_critical_failures.empty:
    failure_text = ", ".join(
        f"{row.gate}: {row.evidence}"
        for row
        in final_critical_failures.itertuples(
            index=False
        )
    )

    raise RuntimeError(
        "Notebook 00 finalization failed: "
        + failure_text
    )


print("Notebook 00 finalization: PASS")
print(
    "Final notebook status: "
    f"{NOTEBOOK_00_FINAL_STATUS}"
)
print(
    "Manifested artifacts verified: "
    f"{int(NOTEBOOK_00_FINAL_ARTIFACT_VERIFICATION['artifact_passed'].sum())}/"
    f"{len(NOTEBOOK_00_FINAL_ARTIFACT_VERIFICATION)}"
)
print(
    "Output manifest SHA-256: "
    f"{saved_manifest_sha256}"
)
print(
    "Critical failures: "
    f"{len(final_critical_failures):,}"
)
print(
    "Warnings retained: "
    f"{len(final_warnings):,}"
)
print(
    "Notebook 01 authorized: "
    f"{notebook_01_authorized}"
)
print(
    "Next notebook: "
    "01_RAW_DATA_AUDIT.ipynb"
)

,artifact_type,relative_path,file_exists,size_matches,hash_matches,artifact_passed
0,V0_0_SOURCE_REGISTRY,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,True,True,True,True
1,V0_1_RUN_CONFIG,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,True,True,True,True
2,CHRONOLOGICAL_SPLIT_CONTRACT,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,True,True,True,True
3,MARKET_DATA_CONTRACT,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,True,True,True,True
4,MISSING_DATA_REJECTION_CONTRACT,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,True,True,True,True
5,DECISION_AND_TOLERANCE_CONTRACT,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,True,True,True,True
6,V0_1_RUN_IDENTITY,artifacts\manifests\BTCUSDT_spot_20260710T0637...,True,True,True,True
7,EXECUTION_ENVIRONMENT_SNAPSHOT,artifacts\manifests\BTCUSDT_spot_20260710T0637...,True,True,True,True
8,NOTEBOOK_00_FINAL_AUDIT,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True
9,AUTHORITATIVE_RAW_SOURCE_MANIFEST,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True


,artifact_type,available,format,relative_path,resolved_path,sha256
0,V0_0_SOURCE_REGISTRY,True,JSON,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,D:\Clown Project\V0.1\config\BTCUSDT_spot_2026...,3131161456fdc9e6fcf2500d846ffcf12b9c609dfa1d97...
1,V0_1_RUN_CONFIG,True,JSON,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,D:\Clown Project\V0.1\config\BTCUSDT_spot_2026...,cec9f2deb240944af04717657c1d35ffb093872e717849...
2,CHRONOLOGICAL_SPLIT_CONTRACT,True,JSON,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,D:\Clown Project\V0.1\config\BTCUSDT_spot_2026...,1037581968095b5e262429fd86004d1abf3e5b691b330e...
3,MARKET_DATA_CONTRACT,True,JSON,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,D:\Clown Project\V0.1\config\BTCUSDT_spot_2026...,8ef328c2f0d828c18ebfeb52a61ff06e9a13b1b6b975f4...
4,MISSING_DATA_REJECTION_CONTRACT,True,JSON,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,D:\Clown Project\V0.1\config\BTCUSDT_spot_2026...,d027f32d05e04591badb8b7f71b37487e83872ee66b4b6...
5,DECISION_AND_TOLERANCE_CONTRACT,True,JSON,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,D:\Clown Project\V0.1\config\BTCUSDT_spot_2026...,ba6df1590bc2355ca87771b8ee3b8c48eb06c5f5a82da2...
6,V0_1_RUN_IDENTITY,True,JSON,artifacts\manifests\BTCUSDT_spot_20260710T0637...,D:\Clown Project\V0.1\artifacts\manifests\BTCU...,86775b812fc43571d030855ce5382a09357a31423c2455...
7,EXECUTION_ENVIRONMENT_SNAPSHOT,True,JSON,artifacts\manifests\BTCUSDT_spot_20260710T0637...,D:\Clown Project\V0.1\artifacts\manifests\BTCU...,355c2ed73c10311264019cf054b6defe80c6de09a17bbb...
8,NOTEBOOK_00_OUTPUT_MANIFEST,True,JSON,artifacts\manifests\BTCUSDT_spot_20260710T0637...,D:\Clown Project\V0.1\artifacts\manifests\BTCU...,e12966301d92e61656715f2a6d397786820de836d8156d...


,gate,passed,severity,evidence,status
0,saved_output_manifest_readable,True,CRITICAL,D:\Clown Project\V0.1\artifacts\manifests\BTCU...,PASS
1,output_manifest_identity_matches,True,CRITICAL,source_run_prefix=BTCUSDT_spot_20260710T063746...,PASS
2,output_manifest_artifact_count_matches,True,CRITICAL,declared=21; observed=21,PASS
3,output_manifest_size_matches,True,CRITICAL,expected=15290; observed=15290,PASS
4,output_manifest_hash_matches,True,CRITICAL,e12966301d92e61656715f2a6d397786820de836d8156d...,PASS
5,all_manifested_artifacts_valid,True,CRITICAL,valid=21/21,PASS
6,required_notebook_01_artifacts_present,True,CRITICAL,required=8,PASS
7,notebook_01_handoff_complete,True,CRITICAL,available=True; unique=True,PASS
8,v0_0_source_state_preserved,True,CRITICAL,unchanged=5/5,PASS
9,notebook_01_execution_authorized,True,CRITICAL,proceed_to_notebook_01=True,PASS


,pipeline_name,pipeline_version,notebook,source_run_prefix,source_set_hash,v0_1_run_id,v0_1_run_config_hash,v0_1_run_identity_hash,output_manifest_sha256,operating_mode,final_status,manifested_artifact_count,critical_failure_count,warning_count,notebook_01_authorized,next_notebook,finalized_utc
0,The Clown Project,V0.1,00_V01_RUN_CONTRACT.ipynb,BTCUSDT_spot_20260710T063746Z_c8b5bf12,132c83531eec615d279408b5c06f402973114ba3058dfa...,v0_1_20260714T090616Z_e82325081a81,14aea0efb3c7b6a193b8c4575a440babe8265d2935c6a7...,5eb89cf073c036d70e3767df4b35ace7c31822e52e9b9d...,e12966301d92e61656715f2a6d397786820de836d8156d...,ENGINEERING_REPRODUCTION_MODE,CONDITIONAL PASS,21,0,3,True,01_RAW_DATA_AUDIT.ipynb,2026-07-14T10:22:24.544990+00:00


Notebook 00 finalization: PASS
Final notebook status: CONDITIONAL PASS
Manifested artifacts verified: 21/21
Output manifest SHA-256: e12966301d92e61656715f2a6d397786820de836d8156dd70092250619cbb00e
Critical failures: 0
Warnings retained: 3
Notebook 01 authorized: True
Next notebook: 01_RAW_DATA_AUDIT.ipynb


In [31]:
# ============================================================
# NOTEBOOK 01 — RAW DATA AUDIT
# CLEAN-KERNEL BOOTSTRAP AND NOTEBOOK 00 HANDOFF LOAD
# ============================================================

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import platform
import random
import sys

import numpy as np
import pandas as pd


# ============================================================
# NOTEBOOK IDENTITY
# ============================================================

PIPELINE_NAME = "The Clown Project"
PIPELINE_VERSION = "V0.1"

NOTEBOOK_NUMBER = "01"
NOTEBOOK_FILENAME = "01_RAW_DATA_AUDIT.ipynb"
NOTEBOOK_PURPOSE = (
    "Independently audit the immutable V0.0 raw market-data inputs "
    "before visible-book reconstruction."
)

NOTEBOOK_STATUS = "IN PROGRESS"
CONTRACT_SCHEMA_VERSION = "v0.1.0"

SOURCE_RUN_PREFIX = (
    "BTCUSDT_spot_20260710T063746Z_c8b5bf12"
)

V0_1_RUN_ID = (
    "v0_1_20260714T090616Z_e82325081a81"
)

RANDOM_SEED = 20260710

NOTEBOOK_01_STARTED_UTC = datetime.now(
    timezone.utc
).isoformat()


# ============================================================
# PROJECT ROOTS
# ============================================================

V0_0_ROOT = Path(
    r"D:\Clown Project\V0.0"
)

V0_1_ROOT = Path(
    r"D:\Clown Project\V0.1"
)

V0_1_CONFIG_ROOT = (
    V0_1_ROOT / "config"
)

V0_1_MANIFEST_ROOT = (
    V0_1_ROOT
    / "artifacts"
    / "manifests"
)

V0_1_AUDIT_ROOT = (
    V0_1_ROOT
    / "artifacts"
    / "audit_tables"
)

V0_1_INTERIM_ROOT = (
    V0_1_ROOT
    / "data"
    / "interim"
)

V0_1_LOG_ROOT = (
    V0_1_ROOT
    / "logs"
)


V0_0_ROOT_RESOLVED = V0_0_ROOT.resolve(
    strict=True
)

V0_1_ROOT_RESOLVED = V0_1_ROOT.resolve(
    strict=True
)


# ============================================================
# GENERAL HELPERS
# ============================================================

def path_is_within(
    candidate: Path,
    root: Path,
) -> bool:
    candidate_resolved = Path(
        candidate
    ).resolve(strict=False)

    root_resolved = Path(
        root
    ).resolve(strict=False)

    try:
        candidate_resolved.relative_to(
            root_resolved
        )
        return True

    except ValueError:
        return False


def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def canonical_json_sha256(
    value,
) -> str:
    encoded = json.dumps(
        value,
        sort_keys=True,
        ensure_ascii=False,
        allow_nan=False,
        separators=(",", ":"),
    ).encode("utf-8")

    return hashlib.sha256(
        encoded
    ).hexdigest()


def load_json_mapping(
    path: Path,
) -> dict:
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        value = json.load(handle)

    if not isinstance(value, dict):
        raise RuntimeError(
            f"Expected a JSON mapping: {path}"
        )

    return value


def unwrap_saved_artifact(
    artifact_document: dict,
    expected_artifact_type: str,
) -> tuple[dict, dict]:
    metadata = artifact_document.get(
        "artifact_metadata"
    )

    payload = artifact_document.get(
        "payload"
    )

    if not isinstance(metadata, dict):
        raise RuntimeError(
            f"{expected_artifact_type} lacks artifact metadata."
        )

    if not isinstance(payload, dict):
        raise RuntimeError(
            f"{expected_artifact_type} lacks a payload mapping."
        )

    observed_artifact_type = metadata.get(
        "artifact_type"
    )

    if observed_artifact_type != expected_artifact_type:
        raise RuntimeError(
            f"Artifact-type mismatch for "
            f"{expected_artifact_type}: "
            f"{observed_artifact_type}"
        )

    if metadata.get(
        "source_run_prefix"
    ) != SOURCE_RUN_PREFIX:
        raise RuntimeError(
            f"Source-run mismatch for "
            f"{expected_artifact_type}."
        )

    if metadata.get(
        "v0_1_run_id"
    ) != V0_1_RUN_ID:
        raise RuntimeError(
            f"V0.1 run-ID mismatch for "
            f"{expected_artifact_type}."
        )

    expected_payload_hash = metadata.get(
        "payload_sha256"
    )

    observed_payload_hash = (
        canonical_json_sha256(payload)
    )

    if expected_payload_hash != observed_payload_hash:
        raise RuntimeError(
            f"Payload-hash mismatch for "
            f"{expected_artifact_type}."
        )

    return metadata, payload


# ============================================================
# DETERMINISTIC EXECUTION
# ============================================================

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

os.environ["PYTHONHASHSEED"] = str(
    RANDOM_SEED
)


# ============================================================
# LOCATE NOTEBOOK 00 OUTPUT MANIFEST
# ============================================================

NOTEBOOK_00_OUTPUT_MANIFEST_NAME = (
    f"{SOURCE_RUN_PREFIX}__"
    f"{V0_1_RUN_ID}__"
    "00_V01_RUN_CONTRACT__"
    "notebook_00_output_manifest.json"
)

NOTEBOOK_00_OUTPUT_MANIFEST_PATH = (
    V0_1_MANIFEST_ROOT
    / NOTEBOOK_00_OUTPUT_MANIFEST_NAME
).resolve(strict=True)


if not path_is_within(
    NOTEBOOK_00_OUTPUT_MANIFEST_PATH,
    V0_1_ROOT_RESOLVED,
):
    raise RuntimeError(
        "Notebook 00 output manifest is outside V0.1."
    )

if path_is_within(
    NOTEBOOK_00_OUTPUT_MANIFEST_PATH,
    V0_0_ROOT_RESOLVED,
):
    raise RuntimeError(
        "Notebook 00 output manifest enters V0.0."
    )


NOTEBOOK_00_OUTPUT_MANIFEST_DOCUMENT = (
    load_json_mapping(
        NOTEBOOK_00_OUTPUT_MANIFEST_PATH
    )
)

(
    NOTEBOOK_00_OUTPUT_MANIFEST_METADATA,
    NOTEBOOK_00_OUTPUT_MANIFEST_PAYLOAD,
) = unwrap_saved_artifact(
    artifact_document=(
        NOTEBOOK_00_OUTPUT_MANIFEST_DOCUMENT
    ),
    expected_artifact_type=(
        "NOTEBOOK_00_OUTPUT_MANIFEST"
    ),
)


# ============================================================
# VALIDATE NOTEBOOK 00 HANDOFF IDENTITY
# ============================================================

if (
    NOTEBOOK_00_OUTPUT_MANIFEST_PAYLOAD.get(
        "source_run_prefix"
    )
    != SOURCE_RUN_PREFIX
):
    raise RuntimeError(
        "Notebook 00 manifest source-run prefix mismatch."
    )

if (
    NOTEBOOK_00_OUTPUT_MANIFEST_PAYLOAD.get(
        "v0_1_run_id"
    )
    != V0_1_RUN_ID
):
    raise RuntimeError(
        "Notebook 00 manifest V0.1 run-ID mismatch."
    )

if (
    NOTEBOOK_00_OUTPUT_MANIFEST_PAYLOAD.get(
        "next_notebook"
    )
    != NOTEBOOK_FILENAME
):
    raise RuntimeError(
        "Notebook 00 did not authorize this notebook."
    )


manifest_artifact_records = (
    NOTEBOOK_00_OUTPUT_MANIFEST_PAYLOAD.get(
        "artifacts"
    )
)

if not isinstance(
    manifest_artifact_records,
    list,
):
    raise RuntimeError(
        "Notebook 00 manifest lacks an artifact list."
    )


NOTEBOOK_00_ARTIFACT_REGISTRY = pd.DataFrame(
    manifest_artifact_records
)


required_manifest_columns = {
    "artifact_type",
    "format",
    "resolved_path",
    "relative_path",
    "size_bytes",
    "sha256",
    "acceptance_status",
}

missing_manifest_columns = sorted(
    required_manifest_columns
    - set(
        NOTEBOOK_00_ARTIFACT_REGISTRY.columns
    )
)

if missing_manifest_columns:
    raise RuntimeError(
        "Notebook 00 artifact registry is missing columns: "
        + " | ".join(missing_manifest_columns)
    )


# ============================================================
# REQUIRED NOTEBOOK 01 INPUTS
# ============================================================

NOTEBOOK_01_REQUIRED_ARTIFACT_TYPES = [
    "V0_0_SOURCE_REGISTRY",
    "V0_1_RUN_CONFIG",
    "CHRONOLOGICAL_SPLIT_CONTRACT",
    "MARKET_DATA_CONTRACT",
    "MISSING_DATA_REJECTION_CONTRACT",
    "DECISION_AND_TOLERANCE_CONTRACT",
    "V0_1_RUN_IDENTITY",
    "EXECUTION_ENVIRONMENT_SNAPSHOT",
]


required_input_rows = []

for artifact_type in (
    NOTEBOOK_01_REQUIRED_ARTIFACT_TYPES
):
    matches = (
        NOTEBOOK_00_ARTIFACT_REGISTRY.loc[
            NOTEBOOK_00_ARTIFACT_REGISTRY[
                "artifact_type"
            ].eq(artifact_type)
        ]
    )

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one saved "
            f"{artifact_type} artifact; "
            f"found {len(matches)}."
        )

    row = matches.iloc[0]

    artifact_path = Path(
        row["resolved_path"]
    ).resolve(strict=False)

    file_exists = (
        artifact_path.exists()
        and artifact_path.is_file()
    )

    inside_v0_1 = path_is_within(
        artifact_path,
        V0_1_ROOT_RESOLVED,
    )

    inside_v0_0 = path_is_within(
        artifact_path,
        V0_0_ROOT_RESOLVED,
    )

    if file_exists:
        observed_size_bytes = int(
            artifact_path.stat().st_size
        )

        observed_sha256 = sha256_file(
            artifact_path
        )

    else:
        observed_size_bytes = None
        observed_sha256 = None

    expected_size_bytes = int(
        row["size_bytes"]
    )

    expected_sha256 = str(
        row["sha256"]
    ).lower()

    size_matches = bool(
        observed_size_bytes
        == expected_size_bytes
    )

    hash_matches = bool(
        observed_sha256
        == expected_sha256
    )

    accepted = bool(
        file_exists
        and inside_v0_1
        and not inside_v0_0
        and size_matches
        and hash_matches
    )

    required_input_rows.append(
        {
            "artifact_type": artifact_type,
            "format": row["format"],
            "acceptance_status": (
                row["acceptance_status"]
            ),
            "resolved_path": str(
                artifact_path
            ),
            "relative_path": (
                row["relative_path"]
            ),
            "expected_size_bytes": (
                expected_size_bytes
            ),
            "observed_size_bytes": (
                observed_size_bytes
            ),
            "size_matches": size_matches,
            "expected_sha256": expected_sha256,
            "observed_sha256": observed_sha256,
            "hash_matches": hash_matches,
            "inside_v0_1": inside_v0_1,
            "inside_v0_0": inside_v0_0,
            "accepted": accepted,
        }
    )


NOTEBOOK_01_INPUT_REGISTRY = pd.DataFrame(
    required_input_rows
)


if not NOTEBOOK_01_INPUT_REGISTRY[
    "accepted"
].all():
    failed_inputs = (
        NOTEBOOK_01_INPUT_REGISTRY.loc[
            ~NOTEBOOK_01_INPUT_REGISTRY[
                "accepted"
            ],
            "artifact_type",
        ]
        .astype(str)
        .tolist()
    )

    raise RuntimeError(
        "Notebook 01 input verification failed: "
        + " | ".join(failed_inputs)
    )


NOTEBOOK_01_INPUT_PATHS = {
    row.artifact_type: Path(
        row.resolved_path
    )
    for row in (
        NOTEBOOK_01_INPUT_REGISTRY
        .itertuples(index=False)
    )
}


# ============================================================
# LOAD AND UNWRAP REQUIRED SAVED CONTRACTS
# ============================================================

LOADED_ARTIFACT_DOCUMENTS = {}
LOADED_ARTIFACT_METADATA = {}
LOADED_ARTIFACT_PAYLOADS = {}

for artifact_type, artifact_path in (
    NOTEBOOK_01_INPUT_PATHS.items()
):
    document = load_json_mapping(
        artifact_path
    )

    metadata, payload = unwrap_saved_artifact(
        artifact_document=document,
        expected_artifact_type=artifact_type,
    )

    LOADED_ARTIFACT_DOCUMENTS[
        artifact_type
    ] = document

    LOADED_ARTIFACT_METADATA[
        artifact_type
    ] = metadata

    LOADED_ARTIFACT_PAYLOADS[
        artifact_type
    ] = payload


V0_0_SOURCE_REGISTRY = (
    LOADED_ARTIFACT_PAYLOADS[
        "V0_0_SOURCE_REGISTRY"
    ]
)

V0_1_RUN_CONFIG = (
    LOADED_ARTIFACT_PAYLOADS[
        "V0_1_RUN_CONFIG"
    ]
)

CHRONOLOGICAL_SPLIT_CONTRACT = (
    LOADED_ARTIFACT_PAYLOADS[
        "CHRONOLOGICAL_SPLIT_CONTRACT"
    ]
)

MARKET_DATA_CONTRACT = (
    LOADED_ARTIFACT_PAYLOADS[
        "MARKET_DATA_CONTRACT"
    ]
)

MISSING_DATA_REJECTION_CONTRACT = (
    LOADED_ARTIFACT_PAYLOADS[
        "MISSING_DATA_REJECTION_CONTRACT"
    ]
)

DECISION_AND_TOLERANCE_CONTRACT = (
    LOADED_ARTIFACT_PAYLOADS[
        "DECISION_AND_TOLERANCE_CONTRACT"
    ]
)

V0_1_RUN_IDENTITY = (
    LOADED_ARTIFACT_PAYLOADS[
        "V0_1_RUN_IDENTITY"
    ]
)

EXECUTION_ENVIRONMENT_SNAPSHOT = (
    LOADED_ARTIFACT_PAYLOADS[
        "EXECUTION_ENVIRONMENT_SNAPSHOT"
    ]
)


# ============================================================
# REGISTER AUTHORITATIVE RAW SOURCE PATHS
# ============================================================

primary_source_records = (
    V0_0_SOURCE_REGISTRY.get(
        "authoritative_primary_sources"
    )
)

if not isinstance(
    primary_source_records,
    list,
):
    raise RuntimeError(
        "V0.0 source registry lacks primary-source records."
    )


AUTHORITATIVE_RAW_SOURCE_MANIFEST = (
    pd.DataFrame(primary_source_records)
)


required_source_columns = {
    "source_role",
    "resolved_path",
    "size_bytes",
    "sha256",
}

missing_source_columns = sorted(
    required_source_columns
    - set(
        AUTHORITATIVE_RAW_SOURCE_MANIFEST.columns
    )
)

if missing_source_columns:
    raise RuntimeError(
        "Primary-source manifest is missing columns: "
        + " | ".join(missing_source_columns)
    )


def authoritative_source_path(
    source_role: str,
) -> Path:
    matches = (
        AUTHORITATIVE_RAW_SOURCE_MANIFEST.loc[
            AUTHORITATIVE_RAW_SOURCE_MANIFEST[
                "source_role"
            ].eq(source_role)
        ]
    )

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one authoritative "
            f"{source_role} source; "
            f"found {len(matches)}."
        )

    return Path(
        matches.iloc[0]["resolved_path"]
    ).resolve(strict=True)


TRADE_SOURCE_PATH = authoritative_source_path(
    "TRADE_STREAM"
)

DEPTH_SOURCE_PATH = authoritative_source_path(
    "DEPTH_STREAM"
)

REST_SNAPSHOT_PATH = authoritative_source_path(
    "REST_SNAPSHOT"
)

COLLECTOR_METADATA_PATH = authoritative_source_path(
    "COLLECTOR_METADATA"
)

RUN_MANIFEST_PATH = authoritative_source_path(
    "RUN_MANIFEST"
)


# ============================================================
# BOOTSTRAP GATES
# ============================================================

NOTEBOOK_01_BOOTSTRAP_GATES = pd.DataFrame(
    [
        {
            "gate": (
                "notebook_00_manifest_verified"
            ),
            "passed": True,
            "evidence": str(
                NOTEBOOK_00_OUTPUT_MANIFEST_PATH
            ),
        },
        {
            "gate": (
                "required_saved_inputs_verified"
            ),
            "passed": bool(
                NOTEBOOK_01_INPUT_REGISTRY[
                    "accepted"
                ].all()
            ),
            "evidence": (
                f"accepted="
                f"{int(NOTEBOOK_01_INPUT_REGISTRY['accepted'].sum())}/"
                f"{len(NOTEBOOK_01_INPUT_REGISTRY)}"
            ),
        },
        {
            "gate": (
                "saved_payload_hashes_verified"
            ),
            "passed": bool(
                len(LOADED_ARTIFACT_PAYLOADS)
                == len(
                    NOTEBOOK_01_REQUIRED_ARTIFACT_TYPES
                )
            ),
            "evidence": (
                f"verified="
                f"{len(LOADED_ARTIFACT_PAYLOADS)}"
            ),
        },
        {
            "gate": (
                "source_registry_loaded"
            ),
            "passed": bool(
                len(
                    AUTHORITATIVE_RAW_SOURCE_MANIFEST
                )
                == 5
            ),
            "evidence": (
                f"primary_sources="
                f"{len(AUTHORITATIVE_RAW_SOURCE_MANIFEST)}"
            ),
        },
        {
            "gate": (
                "raw_source_paths_inside_v0_0"
            ),
            "passed": all(
                path_is_within(
                    path,
                    V0_0_ROOT_RESOLVED,
                )
                for path in [
                    TRADE_SOURCE_PATH,
                    DEPTH_SOURCE_PATH,
                    REST_SNAPSHOT_PATH,
                    COLLECTOR_METADATA_PATH,
                    RUN_MANIFEST_PATH,
                ]
            ),
            "evidence": "all primary inputs under V0.0",
        },
        {
            "gate": (
                "notebook_01_authorized"
            ),
            "passed": bool(
                V0_1_RUN_CONFIG[
                    "operating_authority"
                ]["permissions"][
                    "proceed_to_notebook_01"
                ]
            ),
            "evidence": (
                "proceed_to_notebook_01=True"
            ),
        },
        {
            "gate": (
                "operating_mode_preserved"
            ),
            "passed": (
                V0_1_RUN_CONFIG[
                    "operating_authority"
                ]["operating_mode"]
                == "ENGINEERING_REPRODUCTION_MODE"
            ),
            "evidence": (
                V0_1_RUN_CONFIG[
                    "operating_authority"
                ]["operating_mode"]
            ),
        },
    ]
)


NOTEBOOK_01_BOOTSTRAP_SUMMARY = pd.DataFrame(
    [
        {
            "field": "pipeline_name",
            "value": PIPELINE_NAME,
        },
        {
            "field": "pipeline_version",
            "value": PIPELINE_VERSION,
        },
        {
            "field": "notebook",
            "value": NOTEBOOK_FILENAME,
        },
        {
            "field": "source_run_prefix",
            "value": SOURCE_RUN_PREFIX,
        },
        {
            "field": "v0_1_run_id",
            "value": V0_1_RUN_ID,
        },
        {
            "field": "operating_mode",
            "value": (
                V0_1_RUN_CONFIG[
                    "operating_authority"
                ]["operating_mode"]
            ),
        },
        {
            "field": "contract_status",
            "value": (
                V0_1_RUN_CONFIG[
                    "operating_authority"
                ]["contract_status"]
            ),
        },
        {
            "field": "primary_source_count",
            "value": len(
                AUTHORITATIVE_RAW_SOURCE_MANIFEST
            ),
        },
        {
            "field": "notebook_started_utc",
            "value": NOTEBOOK_01_STARTED_UTC,
        },
    ]
)


display(NOTEBOOK_01_BOOTSTRAP_SUMMARY)

display(
    NOTEBOOK_01_INPUT_REGISTRY[
        [
            "artifact_type",
            "format",
            "relative_path",
            "size_matches",
            "hash_matches",
            "accepted",
        ]
    ]
)

display(
    AUTHORITATIVE_RAW_SOURCE_MANIFEST[
        [
            "source_role",
            "resolved_path",
            "size_bytes",
            "sha256",
        ]
    ]
)

display(NOTEBOOK_01_BOOTSTRAP_GATES)


failed_bootstrap_gates = (
    NOTEBOOK_01_BOOTSTRAP_GATES.loc[
        ~NOTEBOOK_01_BOOTSTRAP_GATES[
            "passed"
        ]
    ]
)

if not failed_bootstrap_gates.empty:
    failure_text = ", ".join(
        f"{row.gate}: {row.evidence}"
        for row in (
            failed_bootstrap_gates
            .itertuples(index=False)
        )
    )

    raise RuntimeError(
        "Notebook 01 bootstrap failed: "
        + failure_text
    )


print("Notebook 01 clean-kernel bootstrap: PASS")
print(
    "Saved Notebook 00 inputs verified: "
    f"{len(NOTEBOOK_01_INPUT_REGISTRY):,}"
)
print(
    "Authoritative raw sources registered: "
    f"{len(AUTHORITATIVE_RAW_SOURCE_MANIFEST):,}"
)
print(
    "Operating mode: "
    f"{V0_1_RUN_CONFIG['operating_authority']['operating_mode']}"
)
print(
    "Notebook status: "
    f"{NOTEBOOK_STATUS}"
)

,field,value
0,pipeline_name,The Clown Project
1,pipeline_version,V0.1
2,notebook,01_RAW_DATA_AUDIT.ipynb
3,source_run_prefix,BTCUSDT_spot_20260710T063746Z_c8b5bf12
4,v0_1_run_id,v0_1_20260714T090616Z_e82325081a81
5,operating_mode,ENGINEERING_REPRODUCTION_MODE
6,contract_status,CONDITIONAL PASS
7,primary_source_count,5
8,notebook_started_utc,2026-07-14T10:24:45.273221+00:00


,artifact_type,format,relative_path,size_matches,hash_matches,accepted
0,V0_0_SOURCE_REGISTRY,JSON,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,True,True,True
1,V0_1_RUN_CONFIG,JSON,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,True,True,True
2,CHRONOLOGICAL_SPLIT_CONTRACT,JSON,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,True,True,True
3,MARKET_DATA_CONTRACT,JSON,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,True,True,True
4,MISSING_DATA_REJECTION_CONTRACT,JSON,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,True,True,True
5,DECISION_AND_TOLERANCE_CONTRACT,JSON,config\BTCUSDT_spot_20260710T063746Z_c8b5bf12_...,True,True,True
6,V0_1_RUN_IDENTITY,JSON,artifacts\manifests\BTCUSDT_spot_20260710T0637...,True,True,True
7,EXECUTION_ENVIRONMENT_SNAPSHOT,JSON,artifacts\manifests\BTCUSDT_spot_20260710T0637...,True,True,True


,source_role,resolved_path,size_bytes,sha256
0,COLLECTOR_METADATA,D:\Clown Project\V0.0\data\raw\metadata\BTCUSD...,1777,f7ea3210033b7e6d02d775fdc496422204194d7fd8a64a...
1,DEPTH_STREAM,D:\Clown Project\V0.0\data\raw\order_book\BTCU...,63443258,608aed2608120c92fe4eaf485c43a4b26cf75d1b212041...
2,REST_SNAPSHOT,D:\Clown Project\V0.0\data\raw\order_book\BTCU...,360812,7921a051c2483ed04e4da82c9a1f58d847eab34b4de5a9...
3,RUN_MANIFEST,D:\Clown Project\V0.0\data\raw\metadata\BTCUSD...,1738,f8a3a278d6cd8b674f47d6ec1d5f4f434cb6b0d1fc05ec...
4,TRADE_STREAM,D:\Clown Project\V0.0\data\raw\trades\BTCUSDT_...,46082102,fcffc0cbd88badc5e2d938545ca864f09871e70e6f1f2c...


,gate,passed,evidence
0,notebook_00_manifest_verified,True,D:\Clown Project\V0.1\artifacts\manifests\BTCU...
1,required_saved_inputs_verified,True,accepted=8/8
2,saved_payload_hashes_verified,True,verified=8
3,source_registry_loaded,True,primary_sources=5
4,raw_source_paths_inside_v0_0,True,all primary inputs under V0.0
5,notebook_01_authorized,True,proceed_to_notebook_01=True
6,operating_mode_preserved,True,ENGINEERING_REPRODUCTION_MODE


Notebook 01 clean-kernel bootstrap: PASS
Saved Notebook 00 inputs verified: 8
Authoritative raw sources registered: 5
Operating mode: ENGINEERING_REPRODUCTION_MODE
Notebook status: IN PROGRESS


In [32]:
# ============================================================
# AUTHORITATIVE RAW-FILE INTEGRITY AND JSON PARSE AUDIT
# ============================================================

from collections import Counter


# ============================================================
# EXPECTED STREAM COUNTS FROM THE FROZEN SPLIT CONTRACT
# ============================================================

split_boundary_records = (
    V0_1_RUN_CONFIG[
        "chronological_partitions"
    ]["boundary_table"]
)

if not isinstance(split_boundary_records, list):
    raise RuntimeError(
        "Frozen chronological boundary table is not a list."
    )


EXPECTED_TRADE_ROWS = int(
    sum(
        int(record["trade_row_count"])
        for record in split_boundary_records
    )
)

EXPECTED_DEPTH_ROWS = int(
    sum(
        int(record["depth_row_count"])
        for record in split_boundary_records
    )
)

EXPECTED_COMBINED_STREAM_ROWS = (
    EXPECTED_TRADE_ROWS
    + EXPECTED_DEPTH_ROWS
)


# ============================================================
# STREAMING JSONL AUDIT
# ============================================================

def audit_jsonl_source(
    path: Path,
    source_role: str,
    invalid_example_limit: int = 5,
) -> tuple[dict, list[dict]]:
    """
    Audit a JSONL source in one streaming pass.

    Computes the SHA-256 digest, byte count, physical-line count,
    JSON parse counts, top-level value types, and observed key
    frequencies without modifying the file.
    """
    digest = hashlib.sha256()

    size_bytes = 0
    physical_line_count = 0
    blank_line_count = 0
    decoded_line_count = 0
    decode_error_count = 0
    valid_json_count = 0
    invalid_json_count = 0

    mapping_record_count = 0
    list_record_count = 0
    scalar_record_count = 0

    top_level_key_frequency = Counter()
    invalid_examples = []

    with Path(path).open("rb") as handle:
        for physical_line_number, raw_line in enumerate(
            handle,
            start=1,
        ):
            physical_line_count += 1
            size_bytes += len(raw_line)
            digest.update(raw_line)

            if not raw_line.strip():
                blank_line_count += 1
                continue

            encoding = (
                "utf-8-sig"
                if physical_line_number == 1
                else "utf-8"
            )

            try:
                decoded_line = raw_line.decode(
                    encoding
                )

            except UnicodeDecodeError as exc:
                decode_error_count += 1

                if (
                    len(invalid_examples)
                    < invalid_example_limit
                ):
                    invalid_examples.append(
                        {
                            "source_role": source_role,
                            "physical_line_number": (
                                physical_line_number
                            ),
                            "failure_type": (
                                "UNICODE_DECODE_ERROR"
                            ),
                            "error": str(exc),
                            "raw_line_sha256": (
                                hashlib.sha256(
                                    raw_line
                                ).hexdigest()
                            ),
                        }
                    )

                continue

            decoded_line_count += 1

            try:
                value = json.loads(decoded_line)

            except json.JSONDecodeError as exc:
                invalid_json_count += 1

                if (
                    len(invalid_examples)
                    < invalid_example_limit
                ):
                    invalid_examples.append(
                        {
                            "source_role": source_role,
                            "physical_line_number": (
                                physical_line_number
                            ),
                            "failure_type": (
                                "JSON_DECODE_ERROR"
                            ),
                            "error": str(exc),
                            "raw_line_sha256": (
                                hashlib.sha256(
                                    raw_line
                                ).hexdigest()
                            ),
                        }
                    )

                continue

            valid_json_count += 1

            if isinstance(value, dict):
                mapping_record_count += 1

                top_level_key_frequency.update(
                    str(key)
                    for key in value.keys()
                )

            elif isinstance(value, list):
                list_record_count += 1

            else:
                scalar_record_count += 1

    trailing_newline = False

    if size_bytes > 0:
        with Path(path).open("rb") as handle:
            handle.seek(-1, os.SEEK_END)
            trailing_newline = (
                handle.read(1) == b"\n"
            )

    summary = {
        "source_role": source_role,
        "resolved_path": str(
            Path(path).resolve(strict=True)
        ),
        "observed_size_bytes": int(
            size_bytes
        ),
        "observed_sha256": digest.hexdigest(),
        "physical_line_count": int(
            physical_line_count
        ),
        "blank_line_count": int(
            blank_line_count
        ),
        "nonblank_line_count": int(
            physical_line_count
            - blank_line_count
        ),
        "decoded_line_count": int(
            decoded_line_count
        ),
        "decode_error_count": int(
            decode_error_count
        ),
        "valid_json_count": int(
            valid_json_count
        ),
        "invalid_json_count": int(
            invalid_json_count
        ),
        "mapping_record_count": int(
            mapping_record_count
        ),
        "list_record_count": int(
            list_record_count
        ),
        "scalar_record_count": int(
            scalar_record_count
        ),
        "unique_top_level_key_count": int(
            len(top_level_key_frequency)
        ),
        "top_level_keys": sorted(
            top_level_key_frequency
        ),
        "trailing_newline": bool(
            trailing_newline
        ),
    }

    return summary, invalid_examples


# ============================================================
# JSON-DOCUMENT AUDIT
# ============================================================

def audit_json_document_source(
    path: Path,
    source_role: str,
) -> tuple[dict, object]:
    """
    Audit a complete JSON document and return its parsed value.
    """
    raw_bytes = Path(path).read_bytes()

    observed_sha256 = hashlib.sha256(
        raw_bytes
    ).hexdigest()

    decode_error = None
    json_error = None
    parsed_value = None

    try:
        decoded_text = raw_bytes.decode(
            "utf-8-sig"
        )

    except UnicodeDecodeError as exc:
        decoded_text = None
        decode_error = str(exc)

    if decoded_text is not None:
        try:
            parsed_value = json.loads(
                decoded_text
            )

        except json.JSONDecodeError as exc:
            json_error = str(exc)

    if isinstance(parsed_value, dict):
        top_level_type = "mapping"
        top_level_key_count = len(
            parsed_value
        )
        top_level_keys = sorted(
            str(key)
            for key in parsed_value.keys()
        )

    elif isinstance(parsed_value, list):
        top_level_type = "list"
        top_level_key_count = None
        top_level_keys = []

    elif parsed_value is None:
        top_level_type = None
        top_level_key_count = None
        top_level_keys = []

    else:
        top_level_type = type(
            parsed_value
        ).__name__

        top_level_key_count = None
        top_level_keys = []

    summary = {
        "source_role": source_role,
        "resolved_path": str(
            Path(path).resolve(strict=True)
        ),
        "observed_size_bytes": int(
            len(raw_bytes)
        ),
        "observed_sha256": observed_sha256,
        "decode_passed": (
            decode_error is None
        ),
        "json_parse_passed": (
            json_error is None
            and parsed_value is not None
        ),
        "top_level_type": top_level_type,
        "top_level_key_count": (
            top_level_key_count
        ),
        "top_level_keys": top_level_keys,
        "decode_error": decode_error,
        "json_error": json_error,
    }

    return summary, parsed_value


# ============================================================
# EXECUTE FULL RAW-SOURCE AUDIT
# ============================================================

trade_jsonl_summary, trade_invalid_examples = (
    audit_jsonl_source(
        path=TRADE_SOURCE_PATH,
        source_role="TRADE_STREAM",
    )
)

depth_jsonl_summary, depth_invalid_examples = (
    audit_jsonl_source(
        path=DEPTH_SOURCE_PATH,
        source_role="DEPTH_STREAM",
    )
)


JSONL_PARSE_AUDIT = pd.DataFrame(
    [
        trade_jsonl_summary,
        depth_jsonl_summary,
    ]
)


RAW_JSONL_INVALID_EXAMPLES = pd.DataFrame(
    trade_invalid_examples
    + depth_invalid_examples
)


snapshot_document_summary, SNAPSHOT_DOCUMENT = (
    audit_json_document_source(
        path=REST_SNAPSHOT_PATH,
        source_role="REST_SNAPSHOT",
    )
)

collector_metadata_summary, COLLECTOR_METADATA_DOCUMENT = (
    audit_json_document_source(
        path=COLLECTOR_METADATA_PATH,
        source_role="COLLECTOR_METADATA",
    )
)

run_manifest_summary, RUN_MANIFEST_DOCUMENT = (
    audit_json_document_source(
        path=RUN_MANIFEST_PATH,
        source_role="RUN_MANIFEST",
    )
)


JSON_DOCUMENT_AUDIT = pd.DataFrame(
    [
        snapshot_document_summary,
        collector_metadata_summary,
        run_manifest_summary,
    ]
)


# ============================================================
# RECONCILE OBSERVED FILES WITH NOTEBOOK 00 REGISTRY
# ============================================================

observed_file_rows = []

for audit_row in (
    JSONL_PARSE_AUDIT.to_dict(
        orient="records"
    )
    + JSON_DOCUMENT_AUDIT.to_dict(
        orient="records"
    )
):
    source_role = audit_row[
        "source_role"
    ]

    manifest_matches = (
        AUTHORITATIVE_RAW_SOURCE_MANIFEST.loc[
            AUTHORITATIVE_RAW_SOURCE_MANIFEST[
                "source_role"
            ].eq(source_role)
        ]
    )

    if len(manifest_matches) != 1:
        raise RuntimeError(
            f"Expected one registered source for "
            f"{source_role}; found "
            f"{len(manifest_matches)}."
        )

    manifest_row = manifest_matches.iloc[0]

    resolved_path = Path(
        audit_row["resolved_path"]
    ).resolve(strict=True)

    registered_path = Path(
        manifest_row["resolved_path"]
    ).resolve(strict=True)

    observed_size_bytes = int(
        audit_row["observed_size_bytes"]
    )

    registered_size_bytes = int(
        manifest_row["size_bytes"]
    )

    observed_sha256 = str(
        audit_row["observed_sha256"]
    ).lower()

    registered_sha256 = str(
        manifest_row["sha256"]
    ).lower()

    observed_file_rows.append(
        {
            "source_role": source_role,
            "resolved_path": str(
                resolved_path
            ),
            "registered_path": str(
                registered_path
            ),
            "path_matches": (
                resolved_path
                == registered_path
            ),
            "inside_v0_0": path_is_within(
                resolved_path,
                V0_0_ROOT_RESOLVED,
            ),
            "inside_v0_1": path_is_within(
                resolved_path,
                V0_1_ROOT_RESOLVED,
            ),
            "registered_size_bytes": (
                registered_size_bytes
            ),
            "observed_size_bytes": (
                observed_size_bytes
            ),
            "size_matches": (
                registered_size_bytes
                == observed_size_bytes
            ),
            "registered_sha256": (
                registered_sha256
            ),
            "observed_sha256": (
                observed_sha256
            ),
            "hash_matches": (
                registered_sha256
                == observed_sha256
            ),
        }
    )


RAW_FILE_INTEGRITY_AUDIT = pd.DataFrame(
    observed_file_rows
)


# ============================================================
# CONTRACTED STREAM-COUNT RECONCILIATION
# ============================================================

STREAM_ROW_COUNT_RECONCILIATION = pd.DataFrame(
    [
        {
            "source_role": "TRADE_STREAM",
            "expected_row_count": (
                EXPECTED_TRADE_ROWS
            ),
            "observed_row_count": int(
                trade_jsonl_summary[
                    "mapping_record_count"
                ]
            ),
        },
        {
            "source_role": "DEPTH_STREAM",
            "expected_row_count": (
                EXPECTED_DEPTH_ROWS
            ),
            "observed_row_count": int(
                depth_jsonl_summary[
                    "mapping_record_count"
                ]
            ),
        },
    ]
)

STREAM_ROW_COUNT_RECONCILIATION[
    "difference"
] = (
    STREAM_ROW_COUNT_RECONCILIATION[
        "observed_row_count"
    ]
    - STREAM_ROW_COUNT_RECONCILIATION[
        "expected_row_count"
    ]
)

STREAM_ROW_COUNT_RECONCILIATION[
    "matches"
] = (
    STREAM_ROW_COUNT_RECONCILIATION[
        "difference"
    ].eq(0)
)


observed_combined_stream_rows = int(
    STREAM_ROW_COUNT_RECONCILIATION[
        "observed_row_count"
    ].sum()
)


# ============================================================
# RAW-FILE AUDIT GATES
# ============================================================

all_registered_roles_unique = bool(
    AUTHORITATIVE_RAW_SOURCE_MANIFEST[
        "source_role"
    ].is_unique
)

all_files_inside_v0_0 = bool(
    RAW_FILE_INTEGRITY_AUDIT[
        "inside_v0_0"
    ].all()
)

no_files_inside_v0_1 = bool(
    ~RAW_FILE_INTEGRITY_AUDIT[
        "inside_v0_1"
    ].any()
)

all_paths_match = bool(
    RAW_FILE_INTEGRITY_AUDIT[
        "path_matches"
    ].all()
)

all_sizes_match = bool(
    RAW_FILE_INTEGRITY_AUDIT[
        "size_matches"
    ].all()
)

all_hashes_match = bool(
    RAW_FILE_INTEGRITY_AUDIT[
        "hash_matches"
    ].all()
)

all_jsonl_decode_clean = bool(
    JSONL_PARSE_AUDIT[
        "decode_error_count"
    ].eq(0).all()
)

all_jsonl_parse_clean = bool(
    JSONL_PARSE_AUDIT[
        "invalid_json_count"
    ].eq(0).all()
)

all_jsonl_records_are_mappings = bool(
    JSONL_PARSE_AUDIT[
        "mapping_record_count"
    ].eq(
        JSONL_PARSE_AUDIT[
            "nonblank_line_count"
        ]
    ).all()
    and JSONL_PARSE_AUDIT[
        "list_record_count"
    ].eq(0).all()
    and JSONL_PARSE_AUDIT[
        "scalar_record_count"
    ].eq(0).all()
)

all_json_documents_valid = bool(
    JSON_DOCUMENT_AUDIT[
        "decode_passed"
    ].all()
    and JSON_DOCUMENT_AUDIT[
        "json_parse_passed"
    ].all()
    and JSON_DOCUMENT_AUDIT[
        "top_level_type"
    ].eq("mapping").all()
)

stream_row_counts_match = bool(
    STREAM_ROW_COUNT_RECONCILIATION[
        "matches"
    ].all()
)

combined_stream_count_matches = bool(
    observed_combined_stream_rows
    == EXPECTED_COMBINED_STREAM_ROWS
)


RAW_FILE_AUDIT_GATES = pd.DataFrame(
    [
        {
            "gate": (
                "registered_source_roles_unique"
            ),
            "passed": (
                all_registered_roles_unique
            ),
            "evidence": (
                f"roles="
                f"{AUTHORITATIVE_RAW_SOURCE_MANIFEST['source_role'].nunique()}"
            ),
        },
        {
            "gate": (
                "all_sources_remain_inside_v0_0"
            ),
            "passed": (
                all_files_inside_v0_0
                and no_files_inside_v0_1
            ),
            "evidence": (
                f"inside_v0_0="
                f"{all_files_inside_v0_0}; "
                f"inside_v0_1="
                f"{not no_files_inside_v0_1}"
            ),
        },
        {
            "gate": (
                "registered_paths_unchanged"
            ),
            "passed": all_paths_match,
            "evidence": (
                f"matching="
                f"{int(RAW_FILE_INTEGRITY_AUDIT['path_matches'].sum())}/"
                f"{len(RAW_FILE_INTEGRITY_AUDIT)}"
            ),
        },
        {
            "gate": (
                "registered_sizes_unchanged"
            ),
            "passed": all_sizes_match,
            "evidence": (
                f"matching="
                f"{int(RAW_FILE_INTEGRITY_AUDIT['size_matches'].sum())}/"
                f"{len(RAW_FILE_INTEGRITY_AUDIT)}"
            ),
        },
        {
            "gate": (
                "registered_hashes_unchanged"
            ),
            "passed": all_hashes_match,
            "evidence": (
                f"matching="
                f"{int(RAW_FILE_INTEGRITY_AUDIT['hash_matches'].sum())}/"
                f"{len(RAW_FILE_INTEGRITY_AUDIT)}"
            ),
        },
        {
            "gate": (
                "jsonl_utf8_decode_clean"
            ),
            "passed": all_jsonl_decode_clean,
            "evidence": (
                f"errors="
                f"{int(JSONL_PARSE_AUDIT['decode_error_count'].sum())}"
            ),
        },
        {
            "gate": (
                "jsonl_parse_clean"
            ),
            "passed": all_jsonl_parse_clean,
            "evidence": (
                f"invalid_json="
                f"{int(JSONL_PARSE_AUDIT['invalid_json_count'].sum())}"
            ),
        },
        {
            "gate": (
                "jsonl_records_are_mappings"
            ),
            "passed": (
                all_jsonl_records_are_mappings
            ),
            "evidence": (
                f"mapping_records="
                f"{int(JSONL_PARSE_AUDIT['mapping_record_count'].sum())}; "
                f"nonblank_lines="
                f"{int(JSONL_PARSE_AUDIT['nonblank_line_count'].sum())}"
            ),
        },
        {
            "gate": (
                "json_documents_valid_mappings"
            ),
            "passed": (
                all_json_documents_valid
            ),
            "evidence": (
                f"valid="
                f"{int((JSON_DOCUMENT_AUDIT['json_parse_passed'] & JSON_DOCUMENT_AUDIT['top_level_type'].eq('mapping')).sum())}/"
                f"{len(JSON_DOCUMENT_AUDIT)}"
            ),
        },
        {
            "gate": (
                "stream_row_counts_match_contract"
            ),
            "passed": (
                stream_row_counts_match
            ),
            "evidence": (
                " | ".join(
                    f"{row.source_role}="
                    f"{row.observed_row_count}/"
                    f"{row.expected_row_count}"
                    for row in (
                        STREAM_ROW_COUNT_RECONCILIATION
                        .itertuples(index=False)
                    )
                )
            ),
        },
        {
            "gate": (
                "combined_stream_count_matches_contract"
            ),
            "passed": (
                combined_stream_count_matches
            ),
            "evidence": (
                f"observed="
                f"{observed_combined_stream_rows}; "
                f"expected="
                f"{EXPECTED_COMBINED_STREAM_ROWS}"
            ),
        },
    ]
)


# ============================================================
# DISPLAY AND ENFORCE
# ============================================================

display(
    RAW_FILE_INTEGRITY_AUDIT[
        [
            "source_role",
            "resolved_path",
            "registered_size_bytes",
            "observed_size_bytes",
            "size_matches",
            "hash_matches",
            "inside_v0_0",
        ]
    ]
)

display(
    JSONL_PARSE_AUDIT[
        [
            "source_role",
            "physical_line_count",
            "blank_line_count",
            "nonblank_line_count",
            "decode_error_count",
            "valid_json_count",
            "invalid_json_count",
            "mapping_record_count",
            "list_record_count",
            "scalar_record_count",
            "trailing_newline",
        ]
    ]
)

display(
    JSON_DOCUMENT_AUDIT[
        [
            "source_role",
            "observed_size_bytes",
            "decode_passed",
            "json_parse_passed",
            "top_level_type",
            "top_level_key_count",
            "top_level_keys",
        ]
    ]
)

display(STREAM_ROW_COUNT_RECONCILIATION)
display(RAW_FILE_AUDIT_GATES)


if not RAW_JSONL_INVALID_EXAMPLES.empty:
    display(RAW_JSONL_INVALID_EXAMPLES)


failed_raw_file_gates = (
    RAW_FILE_AUDIT_GATES.loc[
        ~RAW_FILE_AUDIT_GATES[
            "passed"
        ]
    ]
)

if not failed_raw_file_gates.empty:
    failure_text = ", ".join(
        f"{row.gate}: {row.evidence}"
        for row in (
            failed_raw_file_gates
            .itertuples(index=False)
        )
    )

    raise RuntimeError(
        "Authoritative raw-file audit failed: "
        + failure_text
    )


print("Authoritative raw-file integrity audit: PASS")
print(
    "Primary source files verified: "
    f"{len(RAW_FILE_INTEGRITY_AUDIT):,}"
)
print(
    "Trade records parsed: "
    f"{trade_jsonl_summary['mapping_record_count']:,}"
)
print(
    "Depth records parsed: "
    f"{depth_jsonl_summary['mapping_record_count']:,}"
)
print(
    "Combined stream records: "
    f"{observed_combined_stream_rows:,}"
)
print(
    "Invalid JSONL records: "
    f"{int(JSONL_PARSE_AUDIT['invalid_json_count'].sum()):,}"
)
print(
    "Source hash mismatches: "
    f"{int((~RAW_FILE_INTEGRITY_AUDIT['hash_matches']).sum()):,}"
)

,source_role,resolved_path,registered_size_bytes,observed_size_bytes,size_matches,hash_matches,inside_v0_0
0,TRADE_STREAM,D:\Clown Project\V0.0\data\raw\trades\BTCUSDT_...,46082102,46082102,True,True,True
1,DEPTH_STREAM,D:\Clown Project\V0.0\data\raw\order_book\BTCU...,63443258,63443258,True,True,True
2,REST_SNAPSHOT,D:\Clown Project\V0.0\data\raw\order_book\BTCU...,360812,360812,True,True,True
3,COLLECTOR_METADATA,D:\Clown Project\V0.0\data\raw\metadata\BTCUSD...,1777,1777,True,True,True
4,RUN_MANIFEST,D:\Clown Project\V0.0\data\raw\metadata\BTCUSD...,1738,1738,True,True,True


,source_role,physical_line_count,blank_line_count,nonblank_line_count,decode_error_count,valid_json_count,invalid_json_count,mapping_record_count,list_record_count,scalar_record_count,trailing_newline
0,TRADE_STREAM,67683,0,67683,0,67683,0,67683,0,0,True
1,DEPTH_STREAM,35994,0,35994,0,35994,0,35994,0,0,True


,source_role,observed_size_bytes,decode_passed,json_parse_passed,top_level_type,top_level_key_count,top_level_keys
0,REST_SNAPSHOT,360812,True,True,mapping,16,"[ask_level_count, bid_level_count, buffered_de..."
1,COLLECTOR_METADATA,1777,True,True,mapping,24,"[actual_duration_seconds, collector_result_sta..."
2,RUN_MANIFEST,1738,True,True,mapping,17,"[actual_duration_seconds, collector_result_sta..."


,source_role,expected_row_count,observed_row_count,difference,matches
0,TRADE_STREAM,67683,67683,0,True
1,DEPTH_STREAM,35994,35994,0,True


,gate,passed,evidence
0,registered_source_roles_unique,True,roles=5
1,all_sources_remain_inside_v0_0,True,inside_v0_0=True; inside_v0_1=False
2,registered_paths_unchanged,True,matching=5/5
3,registered_sizes_unchanged,True,matching=5/5
4,registered_hashes_unchanged,True,matching=5/5
5,jsonl_utf8_decode_clean,True,errors=0
6,jsonl_parse_clean,True,invalid_json=0
7,jsonl_records_are_mappings,True,mapping_records=103677; nonblank_lines=103677
8,json_documents_valid_mappings,True,valid=3/3
9,stream_row_counts_match_contract,True,TRADE_STREAM=67683/67683 | DEPTH_STREAM=35994/...


Authoritative raw-file integrity audit: PASS
Primary source files verified: 5
Trade records parsed: 67,683
Depth records parsed: 35,994
Combined stream records: 103,677
Invalid JSONL records: 0
Source hash mismatches: 0


In [34]:
# ============================================================
# FULL-RECORD CANONICAL SCHEMA AUDIT
# RUNTIME PATH RESOLUTION WITH FLATTENED-FIELD SUPPORT
# ============================================================

from collections import Counter
from decimal import Decimal, InvalidOperation


# ============================================================
# LOAD AND VERIFY THE SAVED OBSERVED-SCHEMA ARTIFACT
# ============================================================

observed_schema_matches = (
    NOTEBOOK_00_ARTIFACT_REGISTRY.loc[
        NOTEBOOK_00_ARTIFACT_REGISTRY[
            "artifact_type"
        ].eq("OBSERVED_STREAM_SCHEMA")
    ]
)

if len(observed_schema_matches) != 1:
    raise RuntimeError(
        "Expected exactly one OBSERVED_STREAM_SCHEMA artifact; "
        f"found {len(observed_schema_matches)}."
    )

observed_schema_artifact_row = (
    observed_schema_matches.iloc[0]
)

OBSERVED_STREAM_SCHEMA_PATH = Path(
    observed_schema_artifact_row[
        "resolved_path"
    ]
).resolve(strict=True)

observed_schema_size_matches = bool(
    OBSERVED_STREAM_SCHEMA_PATH.stat().st_size
    == int(
        observed_schema_artifact_row[
            "size_bytes"
        ]
    )
)

observed_schema_hash_matches = bool(
    sha256_file(
        OBSERVED_STREAM_SCHEMA_PATH
    )
    == str(
        observed_schema_artifact_row[
            "sha256"
        ]
    ).lower()
)

if not (
    observed_schema_size_matches
    and observed_schema_hash_matches
):
    raise RuntimeError(
        "OBSERVED_STREAM_SCHEMA artifact failed size or hash "
        "verification."
    )

OBSERVED_STREAM_SCHEMA = pd.read_csv(
    OBSERVED_STREAM_SCHEMA_PATH
)

required_observed_schema_columns = {
    "stream",
    "canonical_field",
    "required",
    "source_path",
}

missing_observed_schema_columns = sorted(
    required_observed_schema_columns
    - set(
        OBSERVED_STREAM_SCHEMA.columns
    )
)

if missing_observed_schema_columns:
    raise RuntimeError(
        "OBSERVED_STREAM_SCHEMA is missing columns: "
        + " | ".join(
            missing_observed_schema_columns
        )
    )


def normalize_required_flag(
    value,
) -> bool:
    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    return str(value).strip().lower() in {
        "true",
        "1",
        "yes",
        "required",
    }


OBSERVED_STREAM_SCHEMA["required"] = (
    OBSERVED_STREAM_SCHEMA["required"].map(
        normalize_required_flag
    )
)


# ============================================================
# JSON-PATH AND TYPE HELPERS
# ============================================================

PATH_MISSING = object()


def decode_json_container(
    value,
):
    """
    Decode a JSON-encoded mapping or list when a collector preserved
    the original payload as a string.
    """
    if not isinstance(value, str):
        return value

    stripped = value.strip()

    if not stripped:
        return value

    if not (
        stripped.startswith("{")
        or stripped.startswith("[")
    ):
        return value

    try:
        return json.loads(stripped)

    except json.JSONDecodeError:
        return value


def prepare_raw_record(
    record: dict,
) -> dict:
    """
    Shallow-copy one record and decode known embedded JSON containers
    once per record.
    """
    prepared = dict(record)

    for key in (
        "raw_message",
        "raw_data",
        "data",
    ):
        if key in prepared:
            prepared[key] = decode_json_container(
                prepared[key]
            )

    raw_message = prepared.get(
        "raw_message"
    )

    if isinstance(raw_message, dict):
        raw_message = dict(raw_message)

        if "data" in raw_message:
            raw_message["data"] = (
                decode_json_container(
                    raw_message["data"]
                )
            )

        prepared["raw_message"] = raw_message

    raw_data = prepared.get(
        "raw_data"
    )

    if isinstance(raw_data, dict):
        raw_data = dict(raw_data)

        if "data" in raw_data:
            raw_data["data"] = (
                decode_json_container(
                    raw_data["data"]
                )
            )

        prepared["raw_data"] = raw_data

    return prepared


def resolve_json_path(
    record: dict,
    source_path: str,
):
    """
    Resolve a simple dotted JSON path such as $.raw_message.data.p.
    """
    if not isinstance(source_path, str):
        return PATH_MISSING

    normalized_path = source_path.strip()

    if normalized_path == "$":
        return record

    if not normalized_path.startswith("$."):
        return PATH_MISSING

    current = record

    for part in normalized_path[2:].split("."):
        current = decode_json_container(
            current
        )

        if isinstance(current, dict):
            if part not in current:
                return PATH_MISSING

            current = current[part]
            continue

        if isinstance(current, list):
            try:
                position = int(part)

            except ValueError:
                return PATH_MISSING

            if not (
                0 <= position < len(current)
            ):
                return PATH_MISSING

            current = current[position]
            continue

        return PATH_MISSING

    return current


def observed_type_name(
    value,
) -> str:
    if value is PATH_MISSING:
        return "missing"

    if value is None:
        return "null"

    if isinstance(value, bool):
        return "bool"

    if isinstance(value, int):
        return "int"

    if isinstance(value, float):
        return "float"

    if isinstance(value, str):
        return "str"

    if isinstance(value, list):
        return "list"

    if isinstance(value, dict):
        return "dict"

    return type(value).__name__


def parse_integer_value(
    value,
):
    if isinstance(value, bool):
        return None

    if isinstance(value, (int, np.integer)):
        return int(value)

    if isinstance(value, float):
        if np.isfinite(value) and value.is_integer():
            return int(value)

        return None

    if isinstance(value, str):
        stripped = value.strip()

        if not stripped:
            return None

        try:
            parsed = Decimal(stripped)

        except InvalidOperation:
            return None

        if not parsed.is_finite():
            return None

        if parsed != parsed.to_integral_value():
            return None

        return int(parsed)

    return None


def parse_decimal_value(
    value,
):
    if isinstance(value, bool):
        return None

    try:
        parsed = Decimal(
            str(value).strip()
        )

    except (
        InvalidOperation,
        AttributeError,
        ValueError,
    ):
        return None

    if not parsed.is_finite():
        return None

    return parsed


def normalize_boolean_value(
    value,
):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    if isinstance(value, str):
        normalized = value.strip().lower()

        if normalized in {
            "true",
            "1",
        }:
            return True

        if normalized in {
            "false",
            "0",
        }:
            return False

    if isinstance(value, (int, np.integer)):
        if int(value) == 1:
            return True

        if int(value) == 0:
            return False

    return None


def normalize_update_array(
    value,
):
    decoded = decode_json_container(
        value
    )

    if isinstance(decoded, list):
        return decoded

    return None


# ============================================================
# SEMANTIC FIELD VALIDATORS
# ============================================================

def validate_integer(
    value,
) -> bool:
    return parse_integer_value(value) is not None


def validate_nonnegative_integer(
    value,
) -> bool:
    parsed = parse_integer_value(value)

    return bool(
        parsed is not None
        and parsed >= 0
    )


def validate_positive_integer(
    value,
) -> bool:
    parsed = parse_integer_value(value)

    return bool(
        parsed is not None
        and parsed > 0
    )


def validate_nonempty_string(
    value,
) -> bool:
    return bool(
        isinstance(value, str)
        and value.strip()
    )


def validate_positive_decimal(
    value,
) -> bool:
    parsed = parse_decimal_value(value)

    return bool(
        parsed is not None
        and parsed > 0
    )


def validate_boolean(
    value,
) -> bool:
    return (
        normalize_boolean_value(value)
        is not None
    )


def validate_update_array(
    value,
) -> bool:
    return (
        normalize_update_array(value)
        is not None
    )


FIELD_VALIDATORS = {
    "collector_sequence": (
        validate_positive_integer
    ),
    "local_receipt_time_ns": (
        validate_positive_integer
    ),
    "connection_session_id": (
        validate_nonempty_string
    ),
    "event_type": (
        validate_nonempty_string
    ),
    "exchange_event_time_ms": (
        validate_positive_integer
    ),
    "trade_time_ms": (
        validate_positive_integer
    ),
    "symbol": validate_nonempty_string,
    "trade_id": validate_nonnegative_integer,
    "price": validate_positive_decimal,
    "quantity": validate_positive_decimal,
    "buyer_is_maker": validate_boolean,
    "first_update_id": (
        validate_nonnegative_integer
    ),
    "final_update_id": (
        validate_nonnegative_integer
    ),
    "bid_updates": validate_update_array,
    "ask_updates": validate_update_array,
}


# ============================================================
# CANDIDATE SOURCE-PATH REGISTRY
# ============================================================

FIELD_PATH_FALLBACKS = {
    (
        "TRADE_STREAM",
        "collector_sequence",
    ): [
        "$.collector_sequence",
        "$.collector_seq",
        "$.sequence",
    ],
    (
        "TRADE_STREAM",
        "local_receipt_time_ns",
    ): [
        "$.local_receipt_time_ns",
        "$.receipt_time_ns",
        "$.received_time_ns",
    ],
    (
        "TRADE_STREAM",
        "connection_session_id",
    ): [
        "$.connection_session_id",
        "$.collector_session_id",
        "$.session_id",
    ],
    (
        "TRADE_STREAM",
        "event_type",
    ): [
        "$.event_type",
        "$.raw_message.data.e",
        "$.raw_message.e",
    ],
    (
        "TRADE_STREAM",
        "exchange_event_time_ms",
    ): [
        "$.exchange_event_time_raw",
        "$.exchange_event_time_ms",
        "$.event_time_raw",
        "$.raw_message.data.E",
        "$.raw_message.E",
    ],
    (
        "TRADE_STREAM",
        "trade_time_ms",
    ): [
        "$.trade_time_raw",
        "$.trade_time_ms",
        "$.raw_message.data.T",
        "$.raw_message.T",
    ],
    (
        "TRADE_STREAM",
        "symbol",
    ): [
        "$.symbol",
        "$.raw_message.data.s",
        "$.raw_message.s",
    ],
    (
        "TRADE_STREAM",
        "trade_id",
    ): [
        "$.trade_id",
        "$.trade_id_raw",
        "$.raw_message.data.t",
        "$.raw_message.t",
    ],
    (
        "TRADE_STREAM",
        "price",
    ): [
        "$.price_raw",
        "$.price",
        "$.trade_price_raw",
        "$.raw_message.data.p",
        "$.raw_message.p",
        "$.raw_data.p",
        "$.p",
    ],
    (
        "TRADE_STREAM",
        "quantity",
    ): [
        "$.quantity_raw",
        "$.quantity",
        "$.qty_raw",
        "$.trade_quantity_raw",
        "$.raw_message.data.q",
        "$.raw_message.q",
        "$.raw_data.q",
        "$.q",
    ],
    (
        "TRADE_STREAM",
        "buyer_is_maker",
    ): [
        "$.buyer_is_maker_raw",
        "$.buyer_is_maker",
        "$.is_buyer_maker",
        "$.raw_message.data.m",
        "$.raw_message.m",
        "$.raw_data.m",
        "$.m",
    ],
    (
        "DEPTH_STREAM",
        "collector_sequence",
    ): [
        "$.collector_sequence",
        "$.collector_seq",
        "$.sequence",
    ],
    (
        "DEPTH_STREAM",
        "local_receipt_time_ns",
    ): [
        "$.local_receipt_time_ns",
        "$.receipt_time_ns",
        "$.received_time_ns",
    ],
    (
        "DEPTH_STREAM",
        "connection_session_id",
    ): [
        "$.connection_session_id",
        "$.collector_session_id",
        "$.session_id",
    ],
    (
        "DEPTH_STREAM",
        "event_type",
    ): [
        "$.event_type",
        "$.raw_message.data.e",
        "$.raw_message.e",
    ],
    (
        "DEPTH_STREAM",
        "exchange_event_time_ms",
    ): [
        "$.exchange_event_time_raw",
        "$.exchange_event_time_ms",
        "$.event_time_raw",
        "$.raw_message.data.E",
        "$.raw_message.E",
    ],
    (
        "DEPTH_STREAM",
        "symbol",
    ): [
        "$.symbol",
        "$.raw_message.data.s",
        "$.raw_message.s",
    ],
    (
        "DEPTH_STREAM",
        "first_update_id",
    ): [
        "$.first_update_id",
        "$.first_update_id_raw",
        "$.raw_message.data.U",
        "$.raw_message.U",
        "$.raw_data.U",
        "$.U",
    ],
    (
        "DEPTH_STREAM",
        "final_update_id",
    ): [
        "$.final_update_id",
        "$.final_update_id_raw",
        "$.raw_message.data.u",
        "$.raw_message.u",
        "$.raw_data.u",
        "$.u",
    ],
    (
        "DEPTH_STREAM",
        "bid_updates",
    ): [
        "$.bid_changes_raw",
        "$.bid_updates",
        "$.bids_raw",
        "$.bids",
        "$.raw_message.data.b",
        "$.raw_message.b",
        "$.raw_data.b",
        "$.b",
    ],
    (
        "DEPTH_STREAM",
        "ask_updates",
    ): [
        "$.ask_changes_raw",
        "$.ask_updates",
        "$.asks_raw",
        "$.asks",
        "$.raw_message.data.a",
        "$.raw_message.a",
        "$.raw_data.a",
        "$.a",
    ],
}


def unique_paths(
    values: list,
) -> list[str]:
    result = []
    seen = set()

    for value in values:
        if not isinstance(value, str):
            continue

        normalized = value.strip()

        if not normalized:
            continue

        if normalized in seen:
            continue

        seen.add(normalized)
        result.append(normalized)

    return result


candidate_definition_rows = []

for schema_row in (
    OBSERVED_STREAM_SCHEMA.itertuples(
        index=False
    )
):
    key = (
        str(schema_row.stream),
        str(schema_row.canonical_field),
    )

    contract_path = (
        str(schema_row.source_path).strip()
        if pd.notna(schema_row.source_path)
        else None
    )

    candidate_paths = unique_paths(
        [contract_path]
        + FIELD_PATH_FALLBACKS.get(
            key,
            [],
        )
    )

    if not candidate_paths:
        raise RuntimeError(
            "No candidate paths registered for "
            f"{key[0]}.{key[1]}."
        )

    for candidate_rank, candidate_path in enumerate(
        candidate_paths,
        start=1,
    ):
        candidate_definition_rows.append(
            {
                "stream": key[0],
                "canonical_field": key[1],
                "required": bool(
                    schema_row.required
                ),
                "contract_source_path": (
                    contract_path
                ),
                "candidate_rank": (
                    candidate_rank
                ),
                "candidate_path": (
                    candidate_path
                ),
            }
        )


CANONICAL_BINDING_CANDIDATES = pd.DataFrame(
    candidate_definition_rows
)


# ============================================================
# STREAMING CANDIDATE-COVERAGE AUDIT
# ============================================================

STREAM_PATHS = {
    "TRADE_STREAM": TRADE_SOURCE_PATH,
    "DEPTH_STREAM": DEPTH_SOURCE_PATH,
}

EXPECTED_STREAM_ROWS = {
    "TRADE_STREAM": EXPECTED_TRADE_ROWS,
    "DEPTH_STREAM": EXPECTED_DEPTH_ROWS,
}


candidate_stats = {}

for candidate_row in (
    CANONICAL_BINDING_CANDIDATES
    .itertuples(index=False)
):
    key = (
        candidate_row.stream,
        candidate_row.canonical_field,
        candidate_row.candidate_path,
    )

    candidate_stats[key] = {
        "row_count": 0,
        "present_count": 0,
        "null_count": 0,
        "valid_count": 0,
        "invalid_count": 0,
        "type_counts": Counter(),
    }


signature_counters = {
    "TRADE_STREAM": {
        "TOP_LEVEL_KEYS": Counter(),
        "RAW_MESSAGE_KEYS": Counter(),
        "RAW_MESSAGE_DATA_KEYS": Counter(),
        "RAW_DATA_KEYS": Counter(),
    },
    "DEPTH_STREAM": {
        "TOP_LEVEL_KEYS": Counter(),
        "RAW_MESSAGE_KEYS": Counter(),
        "RAW_MESSAGE_DATA_KEYS": Counter(),
        "RAW_DATA_KEYS": Counter(),
    },
}


for stream_name, stream_path in (
    STREAM_PATHS.items()
):
    stream_candidates = (
        CANONICAL_BINDING_CANDIDATES.loc[
            CANONICAL_BINDING_CANDIDATES[
                "stream"
            ].eq(stream_name)
        ]
    )

    with Path(stream_path).open(
        "r",
        encoding="utf-8-sig",
        errors="strict",
    ) as handle:
        for source_line_number, line in enumerate(
            handle,
            start=1,
        ):
            if not line.strip():
                continue

            try:
                raw_record = json.loads(line)

            except json.JSONDecodeError as exc:
                raise RuntimeError(
                    f"Invalid JSON in {stream_name} at line "
                    f"{source_line_number}: {exc}"
                ) from exc

            if not isinstance(raw_record, dict):
                raise RuntimeError(
                    f"Non-mapping record in {stream_name} at line "
                    f"{source_line_number}."
                )

            record = prepare_raw_record(
                raw_record
            )

            top_level_signature = " | ".join(
                sorted(
                    str(key)
                    for key in record.keys()
                )
            )

            signature_counters[
                stream_name
            ]["TOP_LEVEL_KEYS"][
                top_level_signature
            ] += 1

            raw_message = record.get(
                "raw_message"
            )

            if isinstance(raw_message, dict):
                raw_message_signature = " | ".join(
                    sorted(
                        str(key)
                        for key in raw_message.keys()
                    )
                )

                signature_counters[
                    stream_name
                ]["RAW_MESSAGE_KEYS"][
                    raw_message_signature
                ] += 1

                raw_message_data = raw_message.get(
                    "data"
                )

                if isinstance(
                    raw_message_data,
                    dict,
                ):
                    raw_message_data_signature = (
                        " | ".join(
                            sorted(
                                str(key)
                                for key
                                in raw_message_data.keys()
                            )
                        )
                    )

                    signature_counters[
                        stream_name
                    ]["RAW_MESSAGE_DATA_KEYS"][
                        raw_message_data_signature
                    ] += 1

            raw_data = record.get(
                "raw_data"
            )

            if isinstance(raw_data, dict):
                raw_data_signature = " | ".join(
                    sorted(
                        str(key)
                        for key in raw_data.keys()
                    )
                )

                signature_counters[
                    stream_name
                ]["RAW_DATA_KEYS"][
                    raw_data_signature
                ] += 1

            for candidate_row in (
                stream_candidates.itertuples(
                    index=False
                )
            ):
                stats_key = (
                    stream_name,
                    candidate_row.canonical_field,
                    candidate_row.candidate_path,
                )

                stats = candidate_stats[
                    stats_key
                ]

                stats["row_count"] += 1

                value = resolve_json_path(
                    record,
                    candidate_row.candidate_path,
                )

                if value is PATH_MISSING:
                    continue

                stats["present_count"] += 1

                if value is None:
                    stats["null_count"] += 1
                    continue

                stats["type_counts"][
                    observed_type_name(value)
                ] += 1

                validator = FIELD_VALIDATORS[
                    candidate_row.canonical_field
                ]

                if validator(value):
                    stats["valid_count"] += 1

                else:
                    stats["invalid_count"] += 1


candidate_audit_rows = []

for candidate_row in (
    CANONICAL_BINDING_CANDIDATES
    .itertuples(index=False)
):
    stats_key = (
        candidate_row.stream,
        candidate_row.canonical_field,
        candidate_row.candidate_path,
    )

    stats = candidate_stats[
        stats_key
    ]

    row_count = int(
        stats["row_count"]
    )

    present_count = int(
        stats["present_count"]
    )

    missing_count = (
        row_count
        - present_count
    )

    null_count = int(
        stats["null_count"]
    )

    valid_count = int(
        stats["valid_count"]
    )

    invalid_count = int(
        stats["invalid_count"]
    )

    full_present = bool(
        present_count == row_count
    )

    full_valid = bool(
        full_present
        and null_count == 0
        and invalid_count == 0
        and valid_count == row_count
    )

    candidate_audit_rows.append(
        {
            "stream": candidate_row.stream,
            "canonical_field": (
                candidate_row.canonical_field
            ),
            "required": bool(
                candidate_row.required
            ),
            "contract_source_path": (
                candidate_row.contract_source_path
            ),
            "candidate_rank": int(
                candidate_row.candidate_rank
            ),
            "candidate_path": (
                candidate_row.candidate_path
            ),
            "row_count": row_count,
            "present_count": (
                present_count
            ),
            "missing_count": (
                missing_count
            ),
            "null_count": null_count,
            "valid_count": valid_count,
            "invalid_count": (
                invalid_count
            ),
            "full_present": full_present,
            "full_valid": full_valid,
            "observed_types": (
                " | ".join(
                    f"{name}:{count}"
                    for name, count
                    in sorted(
                        stats[
                            "type_counts"
                        ].items()
                    )
                )
            ),
        }
    )


CANONICAL_BINDING_CANDIDATE_AUDIT = (
    pd.DataFrame(
        candidate_audit_rows
    )
)


# ============================================================
# SELECT ONE AUTHORITATIVE RUNTIME BINDING PER FIELD
# ============================================================

selected_binding_rows = []

for (
    stream_name,
    canonical_field,
), field_candidates in (
    CANONICAL_BINDING_CANDIDATE_AUDIT.groupby(
        [
            "stream",
            "canonical_field",
        ],
        sort=False,
    )
):
    ranked_candidates = (
        field_candidates.assign(
            full_valid_rank=(
                field_candidates[
                    "full_valid"
                ].astype(int)
            ),
            full_present_rank=(
                field_candidates[
                    "full_present"
                ].astype(int)
            ),
        )
        .sort_values(
            [
                "full_valid_rank",
                "full_present_rank",
                "valid_count",
                "present_count",
                "candidate_rank",
            ],
            ascending=[
                False,
                False,
                False,
                False,
                True,
            ],
            kind="stable",
        )
    )

    selected = ranked_candidates.iloc[0]

    selected_binding_rows.append(
        {
            "stream": stream_name,
            "canonical_field": (
                canonical_field
            ),
            "required": bool(
                selected["required"]
            ),
            "contract_source_path": (
                selected[
                    "contract_source_path"
                ]
            ),
            "selected_source_path": (
                selected["candidate_path"]
            ),
            "binding_changed": bool(
                selected[
                    "candidate_path"
                ]
                != selected[
                    "contract_source_path"
                ]
            ),
            "row_count": int(
                selected["row_count"]
            ),
            "present_count": int(
                selected[
                    "present_count"
                ]
            ),
            "missing_count": int(
                selected[
                    "missing_count"
                ]
            ),
            "null_count": int(
                selected["null_count"]
            ),
            "valid_count": int(
                selected["valid_count"]
            ),
            "invalid_count": int(
                selected[
                    "invalid_count"
                ]
            ),
            "observed_types": (
                selected["observed_types"]
            ),
            "binding_status": (
                "PASS"
                if bool(
                    selected["full_valid"]
                )
                else (
                    "FAIL_REQUIRED_MISSING"
                    if int(
                        selected[
                            "missing_count"
                        ]
                    ) > 0
                    else (
                        "FAIL_REQUIRED_NULL"
                        if int(
                            selected[
                                "null_count"
                            ]
                        ) > 0
                        else (
                            "FAIL_SEMANTIC_INVALID"
                        )
                    )
                )
            ),
        }
    )


RUNTIME_CANONICAL_FIELD_BINDINGS = (
    pd.DataFrame(
        selected_binding_rows
    )
    .sort_values(
        [
            "stream",
            "canonical_field",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ============================================================
# FULL-RECORD AUDIT USING THE SELECTED BINDINGS
# ============================================================

field_audit_state = {}

for binding_row in (
    RUNTIME_CANONICAL_FIELD_BINDINGS
    .itertuples(index=False)
):
    key = (
        binding_row.stream,
        binding_row.canonical_field,
    )

    field_audit_state[key] = {
        "row_count": 0,
        "present_count": 0,
        "null_count": 0,
        "valid_count": 0,
        "invalid_count": 0,
        "type_counts": Counter(),
    }


value_count_state = {
    (
        "TRADE_STREAM",
        "connection_session_id",
    ): Counter(),
    (
        "TRADE_STREAM",
        "event_type",
    ): Counter(),
    (
        "TRADE_STREAM",
        "symbol",
    ): Counter(),
    (
        "DEPTH_STREAM",
        "connection_session_id",
    ): Counter(),
    (
        "DEPTH_STREAM",
        "event_type",
    ): Counter(),
    (
        "DEPTH_STREAM",
        "symbol",
    ): Counter(),
}


depth_update_range_invalid_count = 0
depth_both_sides_empty_count = 0
depth_invalid_level_count = 0


def depth_level_is_valid(
    level,
) -> bool:
    decoded_level = decode_json_container(
        level
    )

    if not isinstance(
        decoded_level,
        (list, tuple),
    ):
        return False

    if len(decoded_level) != 2:
        return False

    price = parse_decimal_value(
        decoded_level[0]
    )

    quantity = parse_decimal_value(
        decoded_level[1]
    )

    return bool(
        price is not None
        and price > 0
        and quantity is not None
        and quantity >= 0
    )


for stream_name, stream_path in (
    STREAM_PATHS.items()
):
    stream_bindings = (
        RUNTIME_CANONICAL_FIELD_BINDINGS.loc[
            RUNTIME_CANONICAL_FIELD_BINDINGS[
                "stream"
            ].eq(stream_name)
        ]
    )

    selected_paths = {
        str(row.canonical_field): str(
            row.selected_source_path
        )
        for row in (
            stream_bindings.itertuples(
                index=False
            )
        )
    }

    with Path(stream_path).open(
        "r",
        encoding="utf-8-sig",
        errors="strict",
    ) as handle:
        for source_line_number, line in enumerate(
            handle,
            start=1,
        ):
            if not line.strip():
                continue

            record = prepare_raw_record(
                json.loads(line)
            )

            resolved_values = {}

            for binding_row in (
                stream_bindings.itertuples(
                    index=False
                )
            ):
                key = (
                    stream_name,
                    binding_row.canonical_field,
                )

                state = field_audit_state[
                    key
                ]

                state["row_count"] += 1

                value = resolve_json_path(
                    record,
                    binding_row.selected_source_path,
                )

                resolved_values[
                    binding_row.canonical_field
                ] = value

                if value is PATH_MISSING:
                    continue

                state["present_count"] += 1

                if value is None:
                    state["null_count"] += 1
                    continue

                state["type_counts"][
                    observed_type_name(value)
                ] += 1

                validator = FIELD_VALIDATORS[
                    binding_row.canonical_field
                ]

                if validator(value):
                    state["valid_count"] += 1

                else:
                    state["invalid_count"] += 1

                value_counter_key = (
                    stream_name,
                    binding_row.canonical_field,
                )

                if (
                    value_counter_key
                    in value_count_state
                ):
                    value_count_state[
                        value_counter_key
                    ][repr(value)] += 1

            if stream_name == "DEPTH_STREAM":
                first_update_id = (
                    parse_integer_value(
                        resolved_values[
                            "first_update_id"
                        ]
                    )
                )

                final_update_id = (
                    parse_integer_value(
                        resolved_values[
                            "final_update_id"
                        ]
                    )
                )

                if (
                    first_update_id is None
                    or final_update_id is None
                    or first_update_id
                    > final_update_id
                ):
                    depth_update_range_invalid_count += 1

                bid_updates = (
                    normalize_update_array(
                        resolved_values[
                            "bid_updates"
                        ]
                    )
                )

                ask_updates = (
                    normalize_update_array(
                        resolved_values[
                            "ask_updates"
                        ]
                    )
                )

                if (
                    bid_updates is not None
                    and ask_updates is not None
                    and len(bid_updates) == 0
                    and len(ask_updates) == 0
                ):
                    depth_both_sides_empty_count += 1

                for update_array in (
                    bid_updates,
                    ask_updates,
                ):
                    if update_array is None:
                        continue

                    for level in update_array:
                        if not depth_level_is_valid(
                            level
                        ):
                            depth_invalid_level_count += 1


canonical_field_audit_rows = []

for binding_row in (
    RUNTIME_CANONICAL_FIELD_BINDINGS
    .itertuples(index=False)
):
    key = (
        binding_row.stream,
        binding_row.canonical_field,
    )

    state = field_audit_state[key]

    row_count = int(
        state["row_count"]
    )

    present_count = int(
        state["present_count"]
    )

    missing_count = (
        row_count
        - present_count
    )

    null_count = int(
        state["null_count"]
    )

    valid_count = int(
        state["valid_count"]
    )

    invalid_count = int(
        state["invalid_count"]
    )

    if missing_count > 0:
        status = "FAIL_REQUIRED_MISSING"

    elif null_count > 0:
        status = "FAIL_REQUIRED_NULL"

    elif invalid_count > 0:
        status = "FAIL_SEMANTIC_INVALID"

    else:
        status = "PASS"

    canonical_field_audit_rows.append(
        {
            "stream": binding_row.stream,
            "canonical_field": (
                binding_row.canonical_field
            ),
            "required": bool(
                binding_row.required
            ),
            "contract_source_path": (
                binding_row.contract_source_path
            ),
            "source_path": (
                binding_row.selected_source_path
            ),
            "binding_changed": bool(
                binding_row.binding_changed
            ),
            "row_count": row_count,
            "present_count": (
                present_count
            ),
            "missing_count": (
                missing_count
            ),
            "null_count": null_count,
            "valid_count": valid_count,
            "invalid_count": (
                invalid_count
            ),
            "observed_types": (
                " | ".join(
                    f"{name}:{count}"
                    for name, count
                    in sorted(
                        state[
                            "type_counts"
                        ].items()
                    )
                )
            ),
            "status": status,
        }
    )


CANONICAL_FIELD_AUDIT = pd.DataFrame(
    canonical_field_audit_rows
)


# ============================================================
# LOW-CARDINALITY VALUE COUNTS
# ============================================================

canonical_value_count_rows = []

for (
    stream_name,
    canonical_field,
), counter in value_count_state.items():
    for observed_value, count in (
        counter.most_common()
    ):
        canonical_value_count_rows.append(
            {
                "stream": stream_name,
                "canonical_field": (
                    canonical_field
                ),
                "observed_value": (
                    observed_value
                ),
                "count": int(count),
            }
        )


CANONICAL_VALUE_COUNTS = pd.DataFrame(
    canonical_value_count_rows
)


# ============================================================
# RAW RECORD SIGNATURES
# ============================================================

raw_signature_rows = []

for stream_name, signature_groups in (
    signature_counters.items()
):
    for signature_type, counter in (
        signature_groups.items()
    ):
        for signature, record_count in (
            counter.most_common()
        ):
            raw_signature_rows.append(
                {
                    "stream": stream_name,
                    "signature_type": (
                        signature_type
                    ),
                    "signature": signature,
                    "record_count": int(
                        record_count
                    ),
                }
            )


RAW_RECORD_SIGNATURES = pd.DataFrame(
    raw_signature_rows
)


# ============================================================
# STREAM-LEVEL SUMMARY
# ============================================================

stream_summary_rows = []

for stream_name in STREAM_PATHS:
    stream_field_rows = (
        CANONICAL_FIELD_AUDIT.loc[
            CANONICAL_FIELD_AUDIT[
                "stream"
            ].eq(stream_name)
        ]
    )

    expected_rows = int(
        EXPECTED_STREAM_ROWS[
            stream_name
        ]
    )

    top_level_signature_count = int(
        RAW_RECORD_SIGNATURES.loc[
            RAW_RECORD_SIGNATURES[
                "stream"
            ].eq(stream_name)
            & RAW_RECORD_SIGNATURES[
                "signature_type"
            ].eq("TOP_LEVEL_KEYS"),
            "signature",
        ].nunique()
    )

    raw_message_signature_count = int(
        RAW_RECORD_SIGNATURES.loc[
            RAW_RECORD_SIGNATURES[
                "stream"
            ].eq(stream_name)
            & RAW_RECORD_SIGNATURES[
                "signature_type"
            ].eq("RAW_MESSAGE_KEYS"),
            "signature",
        ].nunique()
    )

    raw_data_signature_count = int(
        RAW_RECORD_SIGNATURES.loc[
            RAW_RECORD_SIGNATURES[
                "stream"
            ].eq(stream_name)
            & RAW_RECORD_SIGNATURES[
                "signature_type"
            ].eq("RAW_DATA_KEYS"),
            "signature",
        ].nunique()
    )

    stream_summary_rows.append(
        {
            "stream": stream_name,
            "row_count": expected_rows,
            "canonical_field_count": int(
                len(stream_field_rows)
            ),
            "required_field_count": int(
                stream_field_rows[
                    "required"
                ].sum()
            ),
            "changed_binding_count": int(
                stream_field_rows[
                    "binding_changed"
                ].sum()
            ),
            "missing_required_value_count": int(
                stream_field_rows.loc[
                    stream_field_rows[
                        "required"
                    ],
                    "missing_count",
                ].sum()
            ),
            "null_required_value_count": int(
                stream_field_rows.loc[
                    stream_field_rows[
                        "required"
                    ],
                    "null_count",
                ].sum()
            ),
            "invalid_required_value_count": int(
                stream_field_rows.loc[
                    stream_field_rows[
                        "required"
                    ],
                    "invalid_count",
                ].sum()
            ),
            "top_level_signature_count": (
                top_level_signature_count
            ),
            "raw_message_signature_count": (
                raw_message_signature_count
            ),
            "raw_data_signature_count": (
                raw_data_signature_count
            ),
            "depth_both_sides_empty_count": (
                depth_both_sides_empty_count
                if stream_name
                == "DEPTH_STREAM"
                else 0
            ),
            "depth_update_range_invalid_count": (
                depth_update_range_invalid_count
                if stream_name
                == "DEPTH_STREAM"
                else 0
            ),
            "depth_invalid_level_count": (
                depth_invalid_level_count
                if stream_name
                == "DEPTH_STREAM"
                else 0
            ),
        }
    )


STREAM_SCHEMA_SUMMARY = pd.DataFrame(
    stream_summary_rows
)


# ============================================================
# ACCEPTANCE GATES
# ============================================================

required_field_rows = (
    CANONICAL_FIELD_AUDIT.loc[
        CANONICAL_FIELD_AUDIT[
            "required"
        ]
    ]
)

expected_bound_value_count = int(
    required_field_rows[
        "row_count"
    ].sum()
)

observed_bound_value_count = int(
    required_field_rows[
        "present_count"
    ].sum()
)

canonical_bindings_unique = bool(
    not RUNTIME_CANONICAL_FIELD_BINDINGS[
        [
            "stream",
            "canonical_field",
        ]
    ].duplicated().any()
)

required_fields_present = bool(
    required_field_rows[
        "missing_count"
    ].eq(0).all()
)

required_fields_nonnull = bool(
    required_field_rows[
        "null_count"
    ].eq(0).all()
)

required_fields_valid = bool(
    required_field_rows[
        "invalid_count"
    ].eq(0).all()
    and required_field_rows[
        "valid_count"
    ].eq(
        required_field_rows[
            "row_count"
        ]
    ).all()
)

one_bound_value_per_required_field_per_row = bool(
    observed_bound_value_count
    == expected_bound_value_count
)

trade_row_count_preserved = bool(
    CANONICAL_FIELD_AUDIT.loc[
        CANONICAL_FIELD_AUDIT[
            "stream"
        ].eq("TRADE_STREAM"),
        "row_count",
    ].eq(EXPECTED_TRADE_ROWS).all()
)

depth_row_count_preserved = bool(
    CANONICAL_FIELD_AUDIT.loc[
        CANONICAL_FIELD_AUDIT[
            "stream"
        ].eq("DEPTH_STREAM"),
        "row_count",
    ].eq(EXPECTED_DEPTH_ROWS).all()
)


FULL_RECORD_SCHEMA_GATES = pd.DataFrame(
    [
        {
            "gate": (
                "observed_schema_artifact_verified"
            ),
            "passed": bool(
                observed_schema_size_matches
                and observed_schema_hash_matches
            ),
            "evidence": (
                f"size_matches="
                f"{observed_schema_size_matches}; "
                f"hash_matches="
                f"{observed_schema_hash_matches}"
            ),
        },
        {
            "gate": (
                "canonical_bindings_unique"
            ),
            "passed": canonical_bindings_unique,
            "evidence": (
                f"bindings="
                f"{len(RUNTIME_CANONICAL_FIELD_BINDINGS)}"
            ),
        },
        {
            "gate": (
                "required_fields_present"
            ),
            "passed": required_fields_present,
            "evidence": (
                f"missing_values="
                f"{int(required_field_rows['missing_count'].sum())}"
            ),
        },
        {
            "gate": (
                "required_fields_nonnull"
            ),
            "passed": required_fields_nonnull,
            "evidence": (
                f"null_values="
                f"{int(required_field_rows['null_count'].sum())}"
            ),
        },
        {
            "gate": (
                "required_fields_semantically_valid"
            ),
            "passed": required_fields_valid,
            "evidence": (
                f"invalid_values="
                f"{int(required_field_rows['invalid_count'].sum())}"
            ),
        },
        {
            "gate": (
                "one_bound_value_per_required_field_per_row"
            ),
            "passed": (
                one_bound_value_per_required_field_per_row
            ),
            "evidence": (
                f"observed="
                f"{observed_bound_value_count}; "
                f"expected="
                f"{expected_bound_value_count}"
            ),
        },
        {
            "gate": (
                "trade_row_count_preserved"
            ),
            "passed": trade_row_count_preserved,
            "evidence": (
                f"observed="
                f"{CANONICAL_FIELD_AUDIT.loc[CANONICAL_FIELD_AUDIT['stream'].eq('TRADE_STREAM'), 'row_count'].max()}; "
                f"expected={EXPECTED_TRADE_ROWS}"
            ),
        },
        {
            "gate": (
                "depth_row_count_preserved"
            ),
            "passed": depth_row_count_preserved,
            "evidence": (
                f"observed="
                f"{CANONICAL_FIELD_AUDIT.loc[CANONICAL_FIELD_AUDIT['stream'].eq('DEPTH_STREAM'), 'row_count'].max()}; "
                f"expected={EXPECTED_DEPTH_ROWS}"
            ),
        },
        {
            "gate": (
                "depth_update_ranges_valid"
            ),
            "passed": bool(
                depth_update_range_invalid_count
                == 0
            ),
            "evidence": (
                f"invalid_ranges="
                f"{depth_update_range_invalid_count}"
            ),
        },
        {
            "gate": (
                "depth_events_have_at_least_one_side"
            ),
            "passed": bool(
                depth_both_sides_empty_count
                == 0
            ),
            "evidence": (
                f"both_sides_empty="
                f"{depth_both_sides_empty_count}"
            ),
        },
        {
            "gate": (
                "depth_levels_structurally_valid"
            ),
            "passed": bool(
                depth_invalid_level_count
                == 0
            ),
            "evidence": (
                f"invalid_levels="
                f"{depth_invalid_level_count}"
            ),
        },
    ]
)


# ============================================================
# DISPLAY AND ENFORCE
# ============================================================

display(
    RUNTIME_CANONICAL_FIELD_BINDINGS[
        [
            "stream",
            "canonical_field",
            "required",
            "contract_source_path",
            "selected_source_path",
            "binding_changed",
            "binding_status",
        ]
    ]
)

display(
    CANONICAL_FIELD_AUDIT[
        [
            "stream",
            "canonical_field",
            "required",
            "source_path",
            "binding_changed",
            "row_count",
            "present_count",
            "missing_count",
            "null_count",
            "valid_count",
            "invalid_count",
            "observed_types",
            "status",
        ]
    ]
)

display(CANONICAL_VALUE_COUNTS)
display(RAW_RECORD_SIGNATURES)
display(STREAM_SCHEMA_SUMMARY)
display(FULL_RECORD_SCHEMA_GATES)


failed_schema_gates = (
    FULL_RECORD_SCHEMA_GATES.loc[
        ~FULL_RECORD_SCHEMA_GATES[
            "passed"
        ]
    ]
)

if not failed_schema_gates.empty:
    failure_text = ", ".join(
        f"{row.gate}: {row.evidence}"
        for row in (
            failed_schema_gates
            .itertuples(index=False)
        )
    )

    raise RuntimeError(
        "Full-record canonical schema audit failed: "
        + failure_text
    )


print("Full-record canonical schema audit: PASS")
print(
    "Canonical fields audited: "
    f"{len(CANONICAL_FIELD_AUDIT):,}"
)
print(
    "Runtime bindings changed from contract paths: "
    f"{int(CANONICAL_FIELD_AUDIT['binding_changed'].sum()):,}"
)
print(
    "Required values missing: "
    f"{int(required_field_rows['missing_count'].sum()):,}"
)
print(
    "Required values invalid: "
    f"{int(required_field_rows['invalid_count'].sum()):,}"
)
print(
    "Depth update ranges invalid: "
    f"{depth_update_range_invalid_count:,}"
)
print(
    "Depth events with both sides empty: "
    f"{depth_both_sides_empty_count:,}"
)
print(
    "Structurally invalid depth levels: "
    f"{depth_invalid_level_count:,}"
)

,stream,canonical_field,required,contract_source_path,selected_source_path,binding_changed,binding_status
0,DEPTH_STREAM,ask_updates,True,$.raw_message.data.a,$.raw_message.data.a,False,PASS
1,DEPTH_STREAM,bid_updates,True,$.raw_message.data.b,$.raw_message.data.b,False,PASS
2,DEPTH_STREAM,collector_sequence,True,$.collector_sequence,$.collector_sequence,False,PASS
3,DEPTH_STREAM,connection_session_id,True,$.connection_session_id,$.connection_session_id,False,PASS
4,DEPTH_STREAM,event_type,True,$.event_type,$.event_type,False,PASS
5,DEPTH_STREAM,exchange_event_time_ms,True,$.exchange_event_time_raw,$.exchange_event_time_raw,False,PASS
6,DEPTH_STREAM,final_update_id,True,$.final_update_id,$.final_update_id,False,PASS
7,DEPTH_STREAM,first_update_id,True,$.first_update_id,$.first_update_id,False,PASS
8,DEPTH_STREAM,local_receipt_time_ns,True,$.local_receipt_time_ns,$.local_receipt_time_ns,False,PASS
9,DEPTH_STREAM,symbol,True,$.symbol,$.symbol,False,PASS


,stream,canonical_field,required,source_path,binding_changed,row_count,present_count,missing_count,null_count,valid_count,invalid_count,observed_types,status
0,DEPTH_STREAM,ask_updates,True,$.raw_message.data.a,False,35994,35994,0,0,35994,0,list:35994,PASS
1,DEPTH_STREAM,bid_updates,True,$.raw_message.data.b,False,35994,35994,0,0,35994,0,list:35994,PASS
2,DEPTH_STREAM,collector_sequence,True,$.collector_sequence,False,35994,35994,0,0,35994,0,int:35994,PASS
3,DEPTH_STREAM,connection_session_id,True,$.connection_session_id,False,35994,35994,0,0,35994,0,str:35994,PASS
4,DEPTH_STREAM,event_type,True,$.event_type,False,35994,35994,0,0,35994,0,str:35994,PASS
5,DEPTH_STREAM,exchange_event_time_ms,True,$.exchange_event_time_raw,False,35994,35994,0,0,35994,0,int:35994,PASS
6,DEPTH_STREAM,final_update_id,True,$.final_update_id,False,35994,35994,0,0,35994,0,int:35994,PASS
7,DEPTH_STREAM,first_update_id,True,$.first_update_id,False,35994,35994,0,0,35994,0,int:35994,PASS
8,DEPTH_STREAM,local_receipt_time_ns,True,$.local_receipt_time_ns,False,35994,35994,0,0,35994,0,int:35994,PASS
9,DEPTH_STREAM,symbol,True,$.symbol,False,35994,35994,0,0,35994,0,str:35994,PASS


,stream,canonical_field,observed_value,count
0,TRADE_STREAM,connection_session_id,'c8b5bf127a7a44669514acfda8634107',67683
1,TRADE_STREAM,event_type,'trade',67683
2,TRADE_STREAM,symbol,'BTCUSDT',67683
3,DEPTH_STREAM,connection_session_id,'c8b5bf127a7a44669514acfda8634107',35994
4,DEPTH_STREAM,event_type,'depthUpdate',35994
5,DEPTH_STREAM,symbol,'BTCUSDT',35994


,stream,signature_type,signature,record_count
0,TRADE_STREAM,TOP_LEVEL_KEYS,buyer_is_maker_raw | collector_sequence | conn...,67683
1,TRADE_STREAM,RAW_MESSAGE_KEYS,data | stream,67683
2,TRADE_STREAM,RAW_MESSAGE_DATA_KEYS,E | M | T | e | m | p | q | s | t,67683
3,DEPTH_STREAM,TOP_LEVEL_KEYS,ask_changes_raw | bid_changes_raw | collector_...,35994
4,DEPTH_STREAM,RAW_MESSAGE_KEYS,data | stream,35994
5,DEPTH_STREAM,RAW_MESSAGE_DATA_KEYS,E | U | a | b | e | s | u,35994


,stream,row_count,canonical_field_count,required_field_count,changed_binding_count,missing_required_value_count,null_required_value_count,invalid_required_value_count,top_level_signature_count,raw_message_signature_count,raw_data_signature_count,depth_both_sides_empty_count,depth_update_range_invalid_count,depth_invalid_level_count
0,TRADE_STREAM,67683,11,11,0,0,0,0,1,1,0,0,0,0
1,DEPTH_STREAM,35994,10,10,0,0,0,0,1,1,0,0,0,0


,gate,passed,evidence
0,observed_schema_artifact_verified,True,size_matches=True; hash_matches=True
1,canonical_bindings_unique,True,bindings=21
2,required_fields_present,True,missing_values=0
3,required_fields_nonnull,True,null_values=0
4,required_fields_semantically_valid,True,invalid_values=0
5,one_bound_value_per_required_field_per_row,True,observed=1104453; expected=1104453
6,trade_row_count_preserved,True,observed=67683; expected=67683
7,depth_row_count_preserved,True,observed=35994; expected=35994
8,depth_update_ranges_valid,True,invalid_ranges=0
9,depth_events_have_at_least_one_side,True,both_sides_empty=0


Full-record canonical schema audit: PASS
Canonical fields audited: 21
Runtime bindings changed from contract paths: 0
Required values missing: 0
Required values invalid: 0
Depth update ranges invalid: 0
Depth events with both sides empty: 0
Structurally invalid depth levels: 0


In [36]:
# ============================================================
# RAW EVENT ORDER, TIMESTAMP, AND PARTITION AUDIT
# ============================================================

# This cell constructs every datetime column as a complete
# timezone-aware Series. It never assigns timezone-aware values
# into a timezone-naive or object column.


# ============================================================
# CANONICAL BINDING LOOKUP
# ============================================================

def selected_binding_path(
    stream: str,
    canonical_field: str,
) -> str:
    matches = RUNTIME_CANONICAL_FIELD_BINDINGS.loc[
        RUNTIME_CANONICAL_FIELD_BINDINGS[
            "stream"
        ].eq(stream)
        & RUNTIME_CANONICAL_FIELD_BINDINGS[
            "canonical_field"
        ].eq(canonical_field)
    ]

    if len(matches) != 1:
        raise RuntimeError(
            "Expected exactly one runtime binding for "
            f"{stream}.{canonical_field}; found {len(matches)}."
        )

    return str(
        matches.iloc[0][
            "selected_source_path"
        ]
    )


STREAM_BINDING_PATHS = {
    stream_name: {
        field_name: selected_binding_path(
            stream=stream_name,
            canonical_field=field_name,
        )
        for field_name in (
            RUNTIME_CANONICAL_FIELD_BINDINGS.loc[
                RUNTIME_CANONICAL_FIELD_BINDINGS[
                    "stream"
                ].eq(stream_name),
                "canonical_field",
            ]
            .astype(str)
            .tolist()
        )
    }
    for stream_name in (
        "TRADE_STREAM",
        "DEPTH_STREAM",
    )
}


# ============================================================
# SAFE EXTRACTION HELPERS
# ============================================================

def resolved_integer(
    record: dict,
    source_path: str,
):
    value = resolve_json_path(
        record,
        source_path,
    )

    if value is PATH_MISSING or value is None:
        return None

    return parse_integer_value(value)


def resolved_string(
    record: dict,
    source_path: str,
):
    value = resolve_json_path(
        record,
        source_path,
    )

    if value is PATH_MISSING or value is None:
        return None

    if not isinstance(value, str):
        return str(value)

    return value


def epoch_integer_series_to_utc(
    values: pd.Series,
    unit: str,
) -> pd.Series:
    """
    Convert nullable integer epoch values to datetime64[ns, UTC]
    without intermediate float conversion or incompatible dtype
    assignment.
    """
    nullable_values = pd.array(
        values,
        dtype="Int64",
    )

    missing_mask = nullable_values.isna()

    raw_values = nullable_values.to_numpy(
        dtype=np.int64,
        na_value=np.iinfo(np.int64).min,
    )

    converted = pd.to_datetime(
        raw_values,
        unit=unit,
        origin="unix",
        utc=True,
        errors="coerce",
    )

    result = pd.Series(
        converted,
        index=values.index,
        dtype="datetime64[ns, UTC]",
    )

    if missing_mask.any():
        result = result.mask(
            missing_mask,
            pd.NaT,
        )

    return result


# ============================================================
# STREAM RAW EVENT-ORDER FIELDS
# ============================================================

raw_event_order_rows = []


for stream_name, stream_path in (
    (
        "TRADE_STREAM",
        TRADE_SOURCE_PATH,
    ),
    (
        "DEPTH_STREAM",
        DEPTH_SOURCE_PATH,
    ),
):
    binding_paths = STREAM_BINDING_PATHS[
        stream_name
    ]

    with Path(stream_path).open(
        "r",
        encoding="utf-8-sig",
        errors="strict",
    ) as handle:
        for source_line_number, line in enumerate(
            handle,
            start=1,
        ):
            if not line.strip():
                continue

            try:
                raw_record = json.loads(line)

            except json.JSONDecodeError as exc:
                raise RuntimeError(
                    f"Invalid JSON in {stream_name} at line "
                    f"{source_line_number}: {exc}"
                ) from exc

            if not isinstance(raw_record, dict):
                raise RuntimeError(
                    f"Non-mapping record in {stream_name} at line "
                    f"{source_line_number}."
                )

            record = prepare_raw_record(
                raw_record
            )

            row = {
                "stream": stream_name,
                "source_line_number": (
                    source_line_number
                ),
                "collector_sequence": (
                    resolved_integer(
                        record,
                        binding_paths[
                            "collector_sequence"
                        ],
                    )
                ),
                "local_receipt_time_ns": (
                    resolved_integer(
                        record,
                        binding_paths[
                            "local_receipt_time_ns"
                        ],
                    )
                ),
                "exchange_event_time_ms": (
                    resolved_integer(
                        record,
                        binding_paths[
                            "exchange_event_time_ms"
                        ],
                    )
                ),
                "connection_session_id": (
                    resolved_string(
                        record,
                        binding_paths[
                            "connection_session_id"
                        ],
                    )
                ),
                "event_type": (
                    resolved_string(
                        record,
                        binding_paths[
                            "event_type"
                        ],
                    )
                ),
                "symbol": (
                    resolved_string(
                        record,
                        binding_paths[
                            "symbol"
                        ],
                    )
                ),
                "trade_time_ms": None,
                "trade_id": None,
                "first_update_id": None,
                "final_update_id": None,
            }

            if stream_name == "TRADE_STREAM":
                row["trade_time_ms"] = (
                    resolved_integer(
                        record,
                        binding_paths[
                            "trade_time_ms"
                        ],
                    )
                )

                row["trade_id"] = (
                    resolved_integer(
                        record,
                        binding_paths[
                            "trade_id"
                        ],
                    )
                )

            else:
                row["first_update_id"] = (
                    resolved_integer(
                        record,
                        binding_paths[
                            "first_update_id"
                        ],
                    )
                )

                row["final_update_id"] = (
                    resolved_integer(
                        record,
                        binding_paths[
                            "final_update_id"
                        ],
                    )
                )

            raw_event_order_rows.append(row)


RAW_EVENT_ORDER_INPUT_ORDER = pd.DataFrame(
    raw_event_order_rows
)


# ============================================================
# EXPLICIT CANONICAL DTYPES
# ============================================================

string_columns = [
    "stream",
    "connection_session_id",
    "event_type",
    "symbol",
]

integer_columns = [
    "source_line_number",
    "collector_sequence",
    "local_receipt_time_ns",
    "exchange_event_time_ms",
    "trade_time_ms",
    "trade_id",
    "first_update_id",
    "final_update_id",
]


for column in string_columns:
    RAW_EVENT_ORDER_INPUT_ORDER[column] = (
        RAW_EVENT_ORDER_INPUT_ORDER[
            column
        ].astype("string")
    )


for column in integer_columns:
    RAW_EVENT_ORDER_INPUT_ORDER[column] = (
        pd.array(
            RAW_EVENT_ORDER_INPUT_ORDER[
                column
            ],
            dtype="Int64",
        )
    )


# ============================================================
# WITHIN-FILE ORDER AUDIT BEFORE GLOBAL SORT
# ============================================================

within_stream_order_rows = []


for stream_name, stream_frame in (
    RAW_EVENT_ORDER_INPUT_ORDER.groupby(
        "stream",
        sort=False,
    )
):
    sequence_values = (
        stream_frame[
            "collector_sequence"
        ]
        .dropna()
        .to_numpy(dtype=np.int64)
    )

    local_time_values = (
        stream_frame[
            "local_receipt_time_ns"
        ]
        .dropna()
        .to_numpy(dtype=np.int64)
    )

    sequence_differences = np.diff(
        sequence_values
    )

    local_time_differences = np.diff(
        local_time_values
    )

    within_stream_order_rows.append(
        {
            "stream": stream_name,
            "row_count": int(
                len(stream_frame)
            ),
            "first_collector_sequence": (
                int(sequence_values[0])
                if len(sequence_values)
                else None
            ),
            "last_collector_sequence": (
                int(sequence_values[-1])
                if len(sequence_values)
                else None
            ),
            "collector_sequence_reversal_count": int(
                np.sum(
                    sequence_differences <= 0
                )
            ),
            "local_receipt_time_reversal_count": int(
                np.sum(
                    local_time_differences < 0
                )
            ),
        }
    )


WITHIN_STREAM_ORDER_AUDIT = pd.DataFrame(
    within_stream_order_rows
)


# ============================================================
# GLOBAL CAUSAL ORDER INDEX
# ============================================================

RAW_EVENT_ORDER_INDEX = (
    RAW_EVENT_ORDER_INPUT_ORDER
    .sort_values(
        [
            "collector_sequence",
            "stream",
            "source_line_number",
        ],
        kind="stable",
        na_position="last",
    )
    .reset_index(drop=True)
)


# Whole-column timezone-aware conversions.
RAW_EVENT_ORDER_INDEX[
    "local_receipt_time_utc"
] = epoch_integer_series_to_utc(
    RAW_EVENT_ORDER_INDEX[
        "local_receipt_time_ns"
    ],
    unit="ns",
)

RAW_EVENT_ORDER_INDEX[
    "exchange_event_time_utc"
] = epoch_integer_series_to_utc(
    RAW_EVENT_ORDER_INDEX[
        "exchange_event_time_ms"
    ],
    unit="ms",
)

RAW_EVENT_ORDER_INDEX[
    "trade_time_utc"
] = epoch_integer_series_to_utc(
    RAW_EVENT_ORDER_INDEX[
        "trade_time_ms"
    ],
    unit="ms",
)


trade_mask = RAW_EVENT_ORDER_INDEX[
    "stream"
].eq("TRADE_STREAM")

depth_mask = RAW_EVENT_ORDER_INDEX[
    "stream"
].eq("DEPTH_STREAM")


# ============================================================
# CLOCK-DIFFERENCE DIAGNOSTICS
# ============================================================

exchange_event_time_ns = (
    RAW_EVENT_ORDER_INDEX[
        "exchange_event_time_ms"
    ]
    * 1_000_000
)

trade_time_ns = (
    RAW_EVENT_ORDER_INDEX[
        "trade_time_ms"
    ]
    * 1_000_000
)


RAW_EVENT_ORDER_INDEX[
    "local_minus_exchange_ms"
] = (
    RAW_EVENT_ORDER_INDEX[
        "local_receipt_time_ns"
    ]
    - exchange_event_time_ns
).astype("Float64") / 1_000_000.0


RAW_EVENT_ORDER_INDEX[
    "exchange_minus_trade_ms"
] = (
    RAW_EVENT_ORDER_INDEX[
        "exchange_event_time_ms"
    ]
    - RAW_EVENT_ORDER_INDEX[
        "trade_time_ms"
    ]
).astype("Float64")


clock_summary_rows = []


for stream_name, stream_frame in (
    RAW_EVENT_ORDER_INDEX.groupby(
        "stream",
        sort=False,
    )
):
    exchange_lag = (
        stream_frame[
            "local_minus_exchange_ms"
        ]
        .dropna()
        .astype(float)
    )

    if exchange_lag.empty:
        lag_min = None
        lag_p01 = None
        lag_median = None
        lag_p99 = None
        lag_max = None
        exchange_ahead_count = 0

    else:
        lag_min = float(
            exchange_lag.min()
        )

        lag_p01 = float(
            exchange_lag.quantile(0.01)
        )

        lag_median = float(
            exchange_lag.median()
        )

        lag_p99 = float(
            exchange_lag.quantile(0.99)
        )

        lag_max = float(
            exchange_lag.max()
        )

        exchange_ahead_count = int(
            exchange_lag.lt(0).sum()
        )

    clock_summary_rows.append(
        {
            "stream": stream_name,
            "row_count": int(
                len(stream_frame)
            ),
            "local_minus_exchange_min_ms": (
                lag_min
            ),
            "local_minus_exchange_p01_ms": (
                lag_p01
            ),
            "local_minus_exchange_median_ms": (
                lag_median
            ),
            "local_minus_exchange_p99_ms": (
                lag_p99
            ),
            "local_minus_exchange_max_ms": (
                lag_max
            ),
            "exchange_clock_ahead_count": (
                exchange_ahead_count
            ),
        }
    )


CLOCK_DIAGNOSTIC_SUMMARY = pd.DataFrame(
    clock_summary_rows
)


trade_clock_difference = (
    RAW_EVENT_ORDER_INDEX.loc[
        trade_mask,
        "exchange_minus_trade_ms",
    ]
    .dropna()
    .astype(float)
)


TRADE_CLOCK_DIAGNOSTIC = pd.DataFrame(
    [
        {
            "trade_row_count": int(
                trade_mask.sum()
            ),
            "exchange_minus_trade_min_ms": (
                float(
                    trade_clock_difference.min()
                )
                if not trade_clock_difference.empty
                else None
            ),
            "exchange_minus_trade_median_ms": (
                float(
                    trade_clock_difference.median()
                )
                if not trade_clock_difference.empty
                else None
            ),
            "exchange_minus_trade_max_ms": (
                float(
                    trade_clock_difference.max()
                )
                if not trade_clock_difference.empty
                else None
            ),
            "exchange_before_trade_count": int(
                trade_clock_difference.lt(0).sum()
            ),
        }
    ]
)


# ============================================================
# GLOBAL COLLECTOR-SEQUENCE AUDIT
# ============================================================

collector_sequence_missing_count = int(
    RAW_EVENT_ORDER_INDEX[
        "collector_sequence"
    ].isna().sum()
)

collector_sequence_duplicate_count = int(
    RAW_EVENT_ORDER_INDEX[
        "collector_sequence"
    ].duplicated(
        keep=False
    ).sum()
)


if collector_sequence_missing_count == 0:
    collector_sequences = (
        RAW_EVENT_ORDER_INDEX[
            "collector_sequence"
        ].to_numpy(dtype=np.int64)
    )

    collector_sequence_differences = np.diff(
        collector_sequences
    )

    collector_sequence_gap_count = int(
        np.sum(
            collector_sequence_differences > 1
        )
    )

    collector_sequence_reversal_count = int(
        np.sum(
            collector_sequence_differences <= 0
        )
    )

    collector_sequence_first = int(
        collector_sequences[0]
    )

    collector_sequence_last = int(
        collector_sequences[-1]
    )

else:
    collector_sequences = np.array(
        [],
        dtype=np.int64,
    )

    collector_sequence_gap_count = None
    collector_sequence_reversal_count = None
    collector_sequence_first = None
    collector_sequence_last = None


local_receipt_missing_count = int(
    RAW_EVENT_ORDER_INDEX[
        "local_receipt_time_ns"
    ].isna().sum()
)


if local_receipt_missing_count == 0:
    local_receipt_values = (
        RAW_EVENT_ORDER_INDEX[
            "local_receipt_time_ns"
        ].to_numpy(dtype=np.int64)
    )

    local_receipt_differences = np.diff(
        local_receipt_values
    )

    local_receipt_reversal_count = int(
        np.sum(
            local_receipt_differences < 0
        )
    )

    local_receipt_duplicate_count = int(
        np.sum(
            local_receipt_differences == 0
        )
    )

else:
    local_receipt_reversal_count = None
    local_receipt_duplicate_count = None


GLOBAL_ORDER_AUDIT = pd.DataFrame(
    [
        {
            "row_count": int(
                len(RAW_EVENT_ORDER_INDEX)
            ),
            "expected_row_count": int(
                EXPECTED_COMBINED_STREAM_ROWS
            ),
            "collector_sequence_first": (
                collector_sequence_first
            ),
            "collector_sequence_last": (
                collector_sequence_last
            ),
            "collector_sequence_missing_count": (
                collector_sequence_missing_count
            ),
            "collector_sequence_duplicate_count": (
                collector_sequence_duplicate_count
            ),
            "collector_sequence_gap_count": (
                collector_sequence_gap_count
            ),
            "collector_sequence_reversal_count": (
                collector_sequence_reversal_count
            ),
            "local_receipt_missing_count": (
                local_receipt_missing_count
            ),
            "local_receipt_duplicate_count": (
                local_receipt_duplicate_count
            ),
            "local_receipt_reversal_count": (
                local_receipt_reversal_count
            ),
        }
    ]
)


# ============================================================
# TIMESTAMP VALIDITY AND CALENDAR AUDIT
# ============================================================

local_timestamp_invalid_count = int(
    RAW_EVENT_ORDER_INDEX[
        "local_receipt_time_utc"
    ].isna().sum()
)

exchange_timestamp_invalid_count = int(
    RAW_EVENT_ORDER_INDEX[
        "exchange_event_time_utc"
    ].isna().sum()
)

trade_timestamp_invalid_count = int(
    RAW_EVENT_ORDER_INDEX.loc[
        trade_mask,
        "trade_time_utc",
    ].isna().sum()
)

depth_trade_timestamp_nonnull_count = int(
    RAW_EVENT_ORDER_INDEX.loc[
        depth_mask,
        "trade_time_utc",
    ].notna().sum()
)


local_years = RAW_EVENT_ORDER_INDEX[
    "local_receipt_time_utc"
].dt.year

exchange_years = RAW_EVENT_ORDER_INDEX[
    "exchange_event_time_utc"
].dt.year

trade_years = (
    RAW_EVENT_ORDER_INDEX[
        "trade_time_utc"
    ]
    .dt.year
    .loc[trade_mask]
)


local_calendar_plausible = bool(
    local_timestamp_invalid_count == 0
    and local_years.between(
        2025,
        2030,
        inclusive="both",
    ).all()
)

exchange_calendar_plausible = bool(
    exchange_timestamp_invalid_count == 0
    and exchange_years.between(
        2025,
        2030,
        inclusive="both",
    ).all()
)

trade_calendar_plausible = bool(
    trade_timestamp_invalid_count == 0
    and trade_years.between(
        2025,
        2030,
        inclusive="both",
    ).all()
)


TIMESTAMP_AUDIT = pd.DataFrame(
    [
        {
            "timestamp_field": (
                "local_receipt_time_utc"
            ),
            "applicable_row_count": int(
                len(RAW_EVENT_ORDER_INDEX)
            ),
            "nonnull_count": int(
                RAW_EVENT_ORDER_INDEX[
                    "local_receipt_time_utc"
                ].notna().sum()
            ),
            "invalid_count": (
                local_timestamp_invalid_count
            ),
            "dtype": str(
                RAW_EVENT_ORDER_INDEX[
                    "local_receipt_time_utc"
                ].dtype
            ),
            "minimum_utc": (
                RAW_EVENT_ORDER_INDEX[
                    "local_receipt_time_utc"
                ].min()
            ),
            "maximum_utc": (
                RAW_EVENT_ORDER_INDEX[
                    "local_receipt_time_utc"
                ].max()
            ),
            "calendar_plausible": (
                local_calendar_plausible
            ),
        },
        {
            "timestamp_field": (
                "exchange_event_time_utc"
            ),
            "applicable_row_count": int(
                len(RAW_EVENT_ORDER_INDEX)
            ),
            "nonnull_count": int(
                RAW_EVENT_ORDER_INDEX[
                    "exchange_event_time_utc"
                ].notna().sum()
            ),
            "invalid_count": (
                exchange_timestamp_invalid_count
            ),
            "dtype": str(
                RAW_EVENT_ORDER_INDEX[
                    "exchange_event_time_utc"
                ].dtype
            ),
            "minimum_utc": (
                RAW_EVENT_ORDER_INDEX[
                    "exchange_event_time_utc"
                ].min()
            ),
            "maximum_utc": (
                RAW_EVENT_ORDER_INDEX[
                    "exchange_event_time_utc"
                ].max()
            ),
            "calendar_plausible": (
                exchange_calendar_plausible
            ),
        },
        {
            "timestamp_field": (
                "trade_time_utc"
            ),
            "applicable_row_count": int(
                trade_mask.sum()
            ),
            "nonnull_count": int(
                RAW_EVENT_ORDER_INDEX.loc[
                    trade_mask,
                    "trade_time_utc",
                ].notna().sum()
            ),
            "invalid_count": (
                trade_timestamp_invalid_count
            ),
            "dtype": str(
                RAW_EVENT_ORDER_INDEX[
                    "trade_time_utc"
                ].dtype
            ),
            "minimum_utc": (
                RAW_EVENT_ORDER_INDEX.loc[
                    trade_mask,
                    "trade_time_utc",
                ].min()
            ),
            "maximum_utc": (
                RAW_EVENT_ORDER_INDEX.loc[
                    trade_mask,
                    "trade_time_utc",
                ].max()
            ),
            "calendar_plausible": (
                trade_calendar_plausible
            ),
        },
    ]
)


# ============================================================
# CHRONOLOGICAL PARTITION ASSIGNMENT
# ============================================================

FROZEN_PARTITION_BOUNDARIES = pd.DataFrame(
    V0_1_RUN_CONFIG[
        "chronological_partitions"
    ]["boundary_table"]
).sort_values(
    "partition_order",
    kind="stable",
).reset_index(drop=True)


partition_conditions = []
partition_choices = []


for boundary_row in (
    FROZEN_PARTITION_BOUNDARIES.itertuples(
        index=False
    )
):
    partition_conditions.append(
        RAW_EVENT_ORDER_INDEX[
            "collector_sequence"
        ].ge(
            int(
                boundary_row.collector_sequence_start
            )
        )
        & RAW_EVENT_ORDER_INDEX[
            "collector_sequence"
        ].lt(
            int(
                boundary_row.collector_sequence_end_exclusive
            )
        )
    )

    partition_choices.append(
        str(boundary_row.partition)
    )


partition_values = np.select(
    partition_conditions,
    partition_choices,
    default="",
)


RAW_EVENT_ORDER_INDEX[
    "partition"
] = pd.Series(
    partition_values,
    index=RAW_EVENT_ORDER_INDEX.index,
    dtype="string",
).replace(
    "",
    pd.NA,
)


observed_partition_totals = (
    RAW_EVENT_ORDER_INDEX.groupby(
        "partition",
        observed=True,
        dropna=False,
    )
    .size()
    .rename("observed_row_count")
    .reset_index()
)


observed_trade_partition_totals = (
    RAW_EVENT_ORDER_INDEX.loc[
        trade_mask
    ]
    .groupby(
        "partition",
        observed=True,
        dropna=False,
    )
    .size()
    .rename(
        "observed_trade_row_count"
    )
    .reset_index()
)


observed_depth_partition_totals = (
    RAW_EVENT_ORDER_INDEX.loc[
        depth_mask
    ]
    .groupby(
        "partition",
        observed=True,
        dropna=False,
    )
    .size()
    .rename(
        "observed_depth_row_count"
    )
    .reset_index()
)


PARTITION_ASSIGNMENT_AUDIT = (
    FROZEN_PARTITION_BOUNDARIES[
        [
            "partition_order",
            "partition",
            "collector_sequence_start",
            "collector_sequence_end_exclusive",
            "row_count",
            "trade_row_count",
            "depth_row_count",
        ]
    ]
    .merge(
        observed_partition_totals,
        on="partition",
        how="left",
        validate="one_to_one",
    )
    .merge(
        observed_trade_partition_totals,
        on="partition",
        how="left",
        validate="one_to_one",
    )
    .merge(
        observed_depth_partition_totals,
        on="partition",
        how="left",
        validate="one_to_one",
    )
)


for column in (
    "observed_row_count",
    "observed_trade_row_count",
    "observed_depth_row_count",
):
    PARTITION_ASSIGNMENT_AUDIT[
        column
    ] = (
        PARTITION_ASSIGNMENT_AUDIT[
            column
        ]
        .fillna(0)
        .astype("int64")
    )


PARTITION_ASSIGNMENT_AUDIT[
    "row_count_matches"
] = (
    PARTITION_ASSIGNMENT_AUDIT[
        "observed_row_count"
    ]
    == PARTITION_ASSIGNMENT_AUDIT[
        "row_count"
    ].astype("int64")
)

PARTITION_ASSIGNMENT_AUDIT[
    "trade_row_count_matches"
] = (
    PARTITION_ASSIGNMENT_AUDIT[
        "observed_trade_row_count"
    ]
    == PARTITION_ASSIGNMENT_AUDIT[
        "trade_row_count"
    ].astype("int64")
)

PARTITION_ASSIGNMENT_AUDIT[
    "depth_row_count_matches"
] = (
    PARTITION_ASSIGNMENT_AUDIT[
        "observed_depth_row_count"
    ]
    == PARTITION_ASSIGNMENT_AUDIT[
        "depth_row_count"
    ].astype("int64")
)


# ============================================================
# LOW-CARDINALITY IDENTITY AUDIT
# ============================================================

identity_count_rows = []


for stream_name in (
    "TRADE_STREAM",
    "DEPTH_STREAM",
):
    stream_frame = RAW_EVENT_ORDER_INDEX.loc[
        RAW_EVENT_ORDER_INDEX[
            "stream"
        ].eq(stream_name)
    ]

    for field_name in (
        "connection_session_id",
        "event_type",
        "symbol",
    ):
        counts = (
            stream_frame[
                field_name
            ]
            .value_counts(
                dropna=False
            )
        )

        for observed_value, count in counts.items():
            identity_count_rows.append(
                {
                    "stream": stream_name,
                    "field": field_name,
                    "observed_value": (
                        observed_value
                    ),
                    "count": int(count),
                }
            )


RAW_EVENT_IDENTITY_COUNTS = pd.DataFrame(
    identity_count_rows
)


trade_event_type_valid = bool(
    RAW_EVENT_ORDER_INDEX.loc[
        trade_mask,
        "event_type",
    ].eq("trade").all()
)

depth_event_type_valid = bool(
    RAW_EVENT_ORDER_INDEX.loc[
        depth_mask,
        "event_type",
    ].eq("depthUpdate").all()
)

symbols_valid = bool(
    RAW_EVENT_ORDER_INDEX[
        "symbol"
    ].eq("BTCUSDT").all()
)

single_session_valid = bool(
    RAW_EVENT_ORDER_INDEX[
        "connection_session_id"
    ].nunique(
        dropna=True
    )
    == 1
    and RAW_EVENT_ORDER_INDEX[
        "connection_session_id"
    ].notna().all()
)


# ============================================================
# ACCEPTANCE GATES
# ============================================================

expected_first_sequence = int(
    FROZEN_PARTITION_BOUNDARIES[
        "collector_sequence_start"
    ].min()
)

expected_last_sequence = int(
    FROZEN_PARTITION_BOUNDARIES[
        "collector_sequence_end_exclusive"
    ].max()
    - 1
)


RAW_ORDER_TIMESTAMP_GATES = pd.DataFrame(
    [
        {
            "gate": (
                "combined_row_count_preserved"
            ),
            "passed": bool(
                len(RAW_EVENT_ORDER_INDEX)
                == EXPECTED_COMBINED_STREAM_ROWS
            ),
            "evidence": (
                f"observed="
                f"{len(RAW_EVENT_ORDER_INDEX)}; "
                f"expected="
                f"{EXPECTED_COMBINED_STREAM_ROWS}"
            ),
        },
        {
            "gate": (
                "collector_sequence_complete"
            ),
            "passed": bool(
                collector_sequence_missing_count
                == 0
                and collector_sequence_duplicate_count
                == 0
                and collector_sequence_gap_count
                == 0
                and collector_sequence_reversal_count
                == 0
                and collector_sequence_first
                == expected_first_sequence
                and collector_sequence_last
                == expected_last_sequence
            ),
            "evidence": (
                f"range="
                f"{collector_sequence_first}.."
                f"{collector_sequence_last}; "
                f"missing="
                f"{collector_sequence_missing_count}; "
                f"duplicates="
                f"{collector_sequence_duplicate_count}; "
                f"gaps="
                f"{collector_sequence_gap_count}; "
                f"reversals="
                f"{collector_sequence_reversal_count}"
            ),
        },
        {
            "gate": (
                "source_stream_order_monotone"
            ),
            "passed": bool(
                WITHIN_STREAM_ORDER_AUDIT[
                    "collector_sequence_reversal_count"
                ].eq(0).all()
            ),
            "evidence": (
                " | ".join(
                    f"{row.stream}="
                    f"{row.collector_sequence_reversal_count}"
                    for row in (
                        WITHIN_STREAM_ORDER_AUDIT
                        .itertuples(index=False)
                    )
                )
            ),
        },
        {
            "gate": (
                "local_receipt_order_monotone"
            ),
            "passed": bool(
                local_receipt_missing_count == 0
                and local_receipt_reversal_count == 0
            ),
            "evidence": (
                f"missing="
                f"{local_receipt_missing_count}; "
                f"reversals="
                f"{local_receipt_reversal_count}; "
                f"ties="
                f"{local_receipt_duplicate_count}"
            ),
        },
        {
            "gate": (
                "local_receipt_timestamps_valid"
            ),
            "passed": bool(
                local_timestamp_invalid_count == 0
                and local_calendar_plausible
            ),
            "evidence": (
                f"invalid="
                f"{local_timestamp_invalid_count}; "
                f"dtype="
                f"{RAW_EVENT_ORDER_INDEX['local_receipt_time_utc'].dtype}"
            ),
        },
        {
            "gate": (
                "exchange_event_timestamps_valid"
            ),
            "passed": bool(
                exchange_timestamp_invalid_count == 0
                and exchange_calendar_plausible
            ),
            "evidence": (
                f"invalid="
                f"{exchange_timestamp_invalid_count}; "
                f"dtype="
                f"{RAW_EVENT_ORDER_INDEX['exchange_event_time_utc'].dtype}"
            ),
        },
        {
            "gate": (
                "trade_timestamps_valid"
            ),
            "passed": bool(
                trade_timestamp_invalid_count == 0
                and depth_trade_timestamp_nonnull_count
                == 0
                and trade_calendar_plausible
            ),
            "evidence": (
                f"trade_invalid="
                f"{trade_timestamp_invalid_count}; "
                f"depth_nonnull="
                f"{depth_trade_timestamp_nonnull_count}; "
                f"dtype="
                f"{RAW_EVENT_ORDER_INDEX['trade_time_utc'].dtype}"
            ),
        },
        {
            "gate": (
                "stream_identity_values_valid"
            ),
            "passed": bool(
                trade_event_type_valid
                and depth_event_type_valid
                and symbols_valid
                and single_session_valid
            ),
            "evidence": (
                f"trade_type="
                f"{trade_event_type_valid}; "
                f"depth_type="
                f"{depth_event_type_valid}; "
                f"symbols="
                f"{symbols_valid}; "
                f"single_session="
                f"{single_session_valid}"
            ),
        },
        {
            "gate": (
                "all_rows_assigned_to_partition"
            ),
            "passed": bool(
                RAW_EVENT_ORDER_INDEX[
                    "partition"
                ].notna().all()
            ),
            "evidence": (
                f"unassigned="
                f"{int(RAW_EVENT_ORDER_INDEX['partition'].isna().sum())}"
            ),
        },
        {
            "gate": (
                "partition_counts_match_contract"
            ),
            "passed": bool(
                PARTITION_ASSIGNMENT_AUDIT[
                    "row_count_matches"
                ].all()
                and PARTITION_ASSIGNMENT_AUDIT[
                    "trade_row_count_matches"
                ].all()
                and PARTITION_ASSIGNMENT_AUDIT[
                    "depth_row_count_matches"
                ].all()
            ),
            "evidence": (
                f"partitions="
                f"{len(PARTITION_ASSIGNMENT_AUDIT)}"
            ),
        },
    ]
)


# ============================================================
# DISPLAY AND ENFORCE
# ============================================================

display(WITHIN_STREAM_ORDER_AUDIT)
display(GLOBAL_ORDER_AUDIT)
display(TIMESTAMP_AUDIT)
display(CLOCK_DIAGNOSTIC_SUMMARY)
display(TRADE_CLOCK_DIAGNOSTIC)
display(PARTITION_ASSIGNMENT_AUDIT)
display(RAW_EVENT_IDENTITY_COUNTS)
display(RAW_ORDER_TIMESTAMP_GATES)


failed_order_timestamp_gates = (
    RAW_ORDER_TIMESTAMP_GATES.loc[
        ~RAW_ORDER_TIMESTAMP_GATES[
            "passed"
        ]
    ]
)


if not failed_order_timestamp_gates.empty:
    failure_text = ", ".join(
        f"{row.gate}: {row.evidence}"
        for row in (
            failed_order_timestamp_gates
            .itertuples(index=False)
        )
    )

    raise RuntimeError(
        "Raw event-order and timestamp audit failed: "
        + failure_text
    )


print("Raw event-order and timestamp audit: PASS")
print(
    "Combined records ordered: "
    f"{len(RAW_EVENT_ORDER_INDEX):,}"
)
print(
    "Collector sequence range: "
    f"{collector_sequence_first:,}.."
    f"{collector_sequence_last:,}"
)
print(
    "Collector sequence gaps: "
    f"{collector_sequence_gap_count:,}"
)
print(
    "Local receipt-time reversals: "
    f"{local_receipt_reversal_count:,}"
)
print(
    "Timestamp dtypes: "
    f"local={RAW_EVENT_ORDER_INDEX['local_receipt_time_utc'].dtype}; "
    f"exchange={RAW_EVENT_ORDER_INDEX['exchange_event_time_utc'].dtype}; "
    f"trade={RAW_EVENT_ORDER_INDEX['trade_time_utc'].dtype}"
)
print(
    "Chronological partitions assigned: "
    f"{RAW_EVENT_ORDER_INDEX['partition'].nunique():,}"
)

,stream,row_count,first_collector_sequence,last_collector_sequence,collector_sequence_reversal_count,local_receipt_time_reversal_count
0,TRADE_STREAM,67683,14,103676,0,0
1,DEPTH_STREAM,35994,1,103677,0,0


,row_count,expected_row_count,collector_sequence_first,collector_sequence_last,collector_sequence_missing_count,collector_sequence_duplicate_count,collector_sequence_gap_count,collector_sequence_reversal_count,local_receipt_missing_count,local_receipt_duplicate_count,local_receipt_reversal_count
0,103677,103677,1,103677,0,0,0,0,0,51807,0


,timestamp_field,applicable_row_count,nonnull_count,invalid_count,dtype,minimum_utc,maximum_utc,calendar_plausible
0,local_receipt_time_utc,103677,103677,0,"datetime64[ns, UTC]",2026-07-10 06:37:47.531985400+00:00,2026-07-10 07:37:46.749750800+00:00,True
1,exchange_event_time_utc,103677,103677,0,"datetime64[ns, UTC]",2026-07-10 06:37:48.714000+00:00,2026-07-10 07:37:48.014000+00:00,True
2,trade_time_utc,67683,67683,0,"datetime64[ns, UTC]",2026-07-10 06:37:49.952000+00:00,2026-07-10 07:37:47.969000+00:00,True


,stream,row_count,local_minus_exchange_min_ms,local_minus_exchange_p01_ms,local_minus_exchange_median_ms,local_minus_exchange_p99_ms,local_minus_exchange_max_ms,exchange_clock_ahead_count
0,DEPTH_STREAM,35994,-1267.0121,-1263.804600,-1223.1544,-1182.38013,-1177.9191,35994
1,TRADE_STREAM,67683,-1270.9864,-1262.986172,-1220.5896,-1172.22150,-1118.1505,67683


,trade_row_count,exchange_minus_trade_min_ms,exchange_minus_trade_median_ms,exchange_minus_trade_max_ms,exchange_before_trade_count
0,67683,0.0,1.0,15.0,0


,partition_order,partition,collector_sequence_start,collector_sequence_end_exclusive,row_count,trade_row_count,depth_row_count,observed_row_count,observed_trade_row_count,observed_depth_row_count,row_count_matches,trade_row_count_matches,depth_row_count_matches
0,1,DEVELOPMENT,1,51840,51839,33820,18019,51839,33820,18019,True,True,True
1,2,CALIBRATION,51840,72576,20736,13532,7204,20736,13532,7204,True,True,True
2,3,VALIDATION,72576,88127,15551,10198,5353,15551,10198,5353,True,True,True
3,4,ENGINEERING_HOLDOUT,88127,103678,15551,10133,5418,15551,10133,5418,True,True,True


,stream,field,observed_value,count
0,TRADE_STREAM,connection_session_id,c8b5bf127a7a44669514acfda8634107,67683
1,TRADE_STREAM,event_type,trade,67683
2,TRADE_STREAM,symbol,BTCUSDT,67683
3,DEPTH_STREAM,connection_session_id,c8b5bf127a7a44669514acfda8634107,35994
4,DEPTH_STREAM,event_type,depthUpdate,35994
5,DEPTH_STREAM,symbol,BTCUSDT,35994


,gate,passed,evidence
0,combined_row_count_preserved,True,observed=103677; expected=103677
1,collector_sequence_complete,True,range=1..103677; missing=0; duplicates=0; gaps...
2,source_stream_order_monotone,True,TRADE_STREAM=0 | DEPTH_STREAM=0
3,local_receipt_order_monotone,True,missing=0; reversals=0; ties=51807
4,local_receipt_timestamps_valid,True,"invalid=0; dtype=datetime64[ns, UTC]"
5,exchange_event_timestamps_valid,True,"invalid=0; dtype=datetime64[ns, UTC]"
6,trade_timestamps_valid,True,trade_invalid=0; depth_nonnull=0; dtype=dateti...
7,stream_identity_values_valid,True,trade_type=True; depth_type=True; symbols=True...
8,all_rows_assigned_to_partition,True,unassigned=0
9,partition_counts_match_contract,True,partitions=4


Raw event-order and timestamp audit: PASS
Combined records ordered: 103,677
Collector sequence range: 1..103,677
Collector sequence gaps: 0
Local receipt-time reversals: 0
Timestamp dtypes: local=datetime64[ns, UTC]; exchange=datetime64[ns, UTC]; trade=datetime64[ns, UTC]
Chronological partitions assigned: 4


In [37]:
# ============================================================
# TRADE IDENTITY, PAYLOAD CONSISTENCY, AND SEMANTIC AUDIT
# ============================================================

from collections import Counter
from decimal import Decimal, InvalidOperation


# ============================================================
# TRADE FIELD PATHS
# ============================================================

TRADE_BINDINGS = STREAM_BINDING_PATHS[
    "TRADE_STREAM"
]

TRADE_PAYLOAD_RECONCILIATION_PATHS = {
    "event_type": {
        "collector_path": "$.event_type",
        "payload_path": "$.raw_message.data.e",
        "normalizer": "STRING",
    },
    "exchange_event_time_ms": {
        "collector_path": "$.exchange_event_time_raw",
        "payload_path": "$.raw_message.data.E",
        "normalizer": "INTEGER",
    },
    "trade_time_ms": {
        "collector_path": "$.trade_time_raw",
        "payload_path": "$.raw_message.data.T",
        "normalizer": "INTEGER",
    },
    "symbol": {
        "collector_path": "$.symbol",
        "payload_path": "$.raw_message.data.s",
        "normalizer": "STRING",
    },
    "trade_id": {
        "collector_path": "$.trade_id",
        "payload_path": "$.raw_message.data.t",
        "normalizer": "INTEGER",
    },
    "price": {
        "collector_path": "$.price_raw",
        "payload_path": "$.raw_message.data.p",
        "normalizer": "DECIMAL",
    },
    "quantity": {
        "collector_path": "$.quantity_raw",
        "payload_path": "$.raw_message.data.q",
        "normalizer": "DECIMAL",
    },
    "buyer_is_maker": {
        "collector_path": "$.buyer_is_maker_raw",
        "payload_path": "$.raw_message.data.m",
        "normalizer": "BOOLEAN",
    },
}


# ============================================================
# NORMALIZATION HELPERS
# ============================================================

def normalized_decimal_string(
    value,
):
    parsed = parse_decimal_value(value)

    if parsed is None:
        return None

    return format(parsed, "f")


def normalized_comparison_value(
    value,
    normalizer: str,
):
    if value is PATH_MISSING or value is None:
        return None

    if normalizer == "INTEGER":
        return parse_integer_value(value)

    if normalizer == "DECIMAL":
        return normalized_decimal_string(value)

    if normalizer == "BOOLEAN":
        return normalize_boolean_value(value)

    if normalizer == "STRING":
        return str(value)

    raise ValueError(
        f"Unsupported comparison normalizer: {normalizer}"
    )


def decimal_places(
    value,
):
    parsed = parse_decimal_value(value)

    if parsed is None:
        return None

    exponent = parsed.as_tuple().exponent

    return int(
        max(-exponent, 0)
    )


def assign_partition_from_sequence(
    collector_sequence,
):
    if collector_sequence is None:
        return None

    for boundary_row in (
        FROZEN_PARTITION_BOUNDARIES.itertuples(
            index=False
        )
    ):
        sequence_start = int(
            boundary_row.collector_sequence_start
        )

        sequence_end = int(
            boundary_row.collector_sequence_end_exclusive
        )

        if (
            sequence_start
            <= collector_sequence
            < sequence_end
        ):
            return str(
                boundary_row.partition
            )

    return None


# ============================================================
# AUDIT STATE
# ============================================================

payload_reconciliation_state = {
    field_name: {
        "row_count": 0,
        "collector_present_count": 0,
        "payload_present_count": 0,
        "both_present_count": 0,
        "collector_null_count": 0,
        "payload_null_count": 0,
        "match_count": 0,
        "mismatch_count": 0,
    }
    for field_name in (
        TRADE_PAYLOAD_RECONCILIATION_PATHS
    )
}


trade_rows = []
trade_rejection_rows = []


# ============================================================
# STREAM THE AUTHORITATIVE TRADE SOURCE
# ============================================================

with Path(TRADE_SOURCE_PATH).open(
    "r",
    encoding="utf-8-sig",
    errors="strict",
) as handle:
    for source_line_number, line in enumerate(
        handle,
        start=1,
    ):
        if not line.strip():
            continue

        try:
            raw_record = json.loads(line)

        except json.JSONDecodeError as exc:
            raise RuntimeError(
                "Invalid JSON in TRADE_STREAM at line "
                f"{source_line_number}: {exc}"
            ) from exc

        if not isinstance(raw_record, dict):
            raise RuntimeError(
                "Non-mapping TRADE_STREAM record at line "
                f"{source_line_number}."
            )

        record = prepare_raw_record(
            raw_record
        )

        collector_sequence = resolved_integer(
            record,
            TRADE_BINDINGS[
                "collector_sequence"
            ],
        )

        local_receipt_time_ns = resolved_integer(
            record,
            TRADE_BINDINGS[
                "local_receipt_time_ns"
            ],
        )

        exchange_event_time_ms = resolved_integer(
            record,
            TRADE_BINDINGS[
                "exchange_event_time_ms"
            ],
        )

        trade_time_ms = resolved_integer(
            record,
            TRADE_BINDINGS[
                "trade_time_ms"
            ],
        )

        trade_id = resolved_integer(
            record,
            TRADE_BINDINGS[
                "trade_id"
            ],
        )

        price_value = resolve_json_path(
            record,
            TRADE_BINDINGS["price"],
        )

        quantity_value = resolve_json_path(
            record,
            TRADE_BINDINGS["quantity"],
        )

        buyer_is_maker_value = resolve_json_path(
            record,
            TRADE_BINDINGS[
                "buyer_is_maker"
            ],
        )

        connection_session_id = resolved_string(
            record,
            TRADE_BINDINGS[
                "connection_session_id"
            ],
        )

        event_type = resolved_string(
            record,
            TRADE_BINDINGS[
                "event_type"
            ],
        )

        symbol = resolved_string(
            record,
            TRADE_BINDINGS[
                "symbol"
            ],
        )

        price_decimal = parse_decimal_value(
            price_value
        )

        quantity_decimal = parse_decimal_value(
            quantity_value
        )

        buyer_is_maker = normalize_boolean_value(
            buyer_is_maker_value
        )

        price_raw = (
            normalized_decimal_string(
                price_value
            )
        )

        quantity_raw = (
            normalized_decimal_string(
                quantity_value
            )
        )

        aggressor_side = (
            "SELL"
            if buyer_is_maker is True
            else (
                "BUY"
                if buyer_is_maker is False
                else None
            )
        )

        partition = assign_partition_from_sequence(
            collector_sequence
        )

        row_reasons = []

        if collector_sequence is None:
            row_reasons.append(
                "COLLECTOR_SEQUENCE_MISSING"
            )

        if local_receipt_time_ns is None:
            row_reasons.append(
                "TRADE_TIMESTAMP_INVALID"
            )

        if exchange_event_time_ms is None:
            row_reasons.append(
                "TRADE_TIMESTAMP_INVALID"
            )

        if trade_time_ms is None:
            row_reasons.append(
                "TRADE_TIMESTAMP_INVALID"
            )

        if trade_id is None or trade_id < 0:
            row_reasons.append(
                "TRADE_ID_INVALID"
            )

        if (
            price_decimal is None
            or price_decimal <= 0
        ):
            row_reasons.append(
                "TRADE_PRICE_INVALID"
            )

        if (
            quantity_decimal is None
            or quantity_decimal <= 0
        ):
            row_reasons.append(
                "TRADE_QUANTITY_INVALID"
            )

        if buyer_is_maker is None:
            row_reasons.append(
                "TRADE_AGGRESSOR_FLAG_INVALID"
            )

        if event_type != "trade":
            row_reasons.append(
                "TRADE_EVENT_TYPE_INVALID"
            )

        if symbol != "BTCUSDT":
            row_reasons.append(
                "TRADE_SYMBOL_MISMATCH"
            )

        if partition is None:
            row_reasons.append(
                "COLLECTOR_SEQUENCE_GAP"
            )

        for reason_code in sorted(
            set(row_reasons)
        ):
            trade_rejection_rows.append(
                {
                    "source_line_number": (
                        source_line_number
                    ),
                    "collector_sequence": (
                        collector_sequence
                    ),
                    "trade_id": trade_id,
                    "reason_code": reason_code,
                }
            )

        trade_rows.append(
            {
                "source_line_number": (
                    source_line_number
                ),
                "collector_sequence": (
                    collector_sequence
                ),
                "local_receipt_time_ns": (
                    local_receipt_time_ns
                ),
                "exchange_event_time_ms": (
                    exchange_event_time_ms
                ),
                "trade_time_ms": (
                    trade_time_ms
                ),
                "connection_session_id": (
                    connection_session_id
                ),
                "event_type": event_type,
                "symbol": symbol,
                "trade_id": trade_id,
                "price_raw": price_raw,
                "quantity_raw": quantity_raw,
                "buyer_is_maker": (
                    buyer_is_maker
                ),
                "aggressor_side": (
                    aggressor_side
                ),
                "partition": partition,
            }
        )

        for field_name, field_spec in (
            TRADE_PAYLOAD_RECONCILIATION_PATHS.items()
        ):
            state = payload_reconciliation_state[
                field_name
            ]

            state["row_count"] += 1

            collector_value = resolve_json_path(
                record,
                field_spec[
                    "collector_path"
                ],
            )

            payload_value = resolve_json_path(
                record,
                field_spec[
                    "payload_path"
                ],
            )

            collector_present = (
                collector_value is not PATH_MISSING
            )

            payload_present = (
                payload_value is not PATH_MISSING
            )

            if collector_present:
                state[
                    "collector_present_count"
                ] += 1

            if payload_present:
                state[
                    "payload_present_count"
                ] += 1

            if (
                collector_present
                and payload_present
            ):
                state[
                    "both_present_count"
                ] += 1

            if (
                collector_present
                and collector_value is None
            ):
                state[
                    "collector_null_count"
                ] += 1

            if (
                payload_present
                and payload_value is None
            ):
                state[
                    "payload_null_count"
                ] += 1

            collector_normalized = (
                normalized_comparison_value(
                    collector_value,
                    field_spec[
                        "normalizer"
                    ],
                )
            )

            payload_normalized = (
                normalized_comparison_value(
                    payload_value,
                    field_spec[
                        "normalizer"
                    ],
                )
            )

            if (
                collector_present
                and payload_present
                and collector_normalized
                == payload_normalized
            ):
                state["match_count"] += 1

            elif (
                collector_present
                and payload_present
            ):
                state[
                    "mismatch_count"
                ] += 1


# ============================================================
# CANONICAL RAW TRADE AUDIT TABLE
# ============================================================

RAW_TRADE_AUDIT_TABLE = pd.DataFrame(
    trade_rows
)


for column in (
    "source_line_number",
    "collector_sequence",
    "local_receipt_time_ns",
    "exchange_event_time_ms",
    "trade_time_ms",
    "trade_id",
):
    RAW_TRADE_AUDIT_TABLE[column] = pd.array(
        RAW_TRADE_AUDIT_TABLE[column],
        dtype="Int64",
    )


for column in (
    "connection_session_id",
    "event_type",
    "symbol",
    "price_raw",
    "quantity_raw",
    "aggressor_side",
    "partition",
):
    RAW_TRADE_AUDIT_TABLE[column] = (
        RAW_TRADE_AUDIT_TABLE[
            column
        ].astype("string")
    )


RAW_TRADE_AUDIT_TABLE[
    "buyer_is_maker"
] = pd.array(
    RAW_TRADE_AUDIT_TABLE[
        "buyer_is_maker"
    ],
    dtype="boolean",
)


TRADE_REJECTION_CANDIDATES = pd.DataFrame(
    trade_rejection_rows,
    columns=[
        "source_line_number",
        "collector_sequence",
        "trade_id",
        "reason_code",
    ],
)


# ============================================================
# COLLECTOR-WRAPPER VS RAW-PAYLOAD RECONCILIATION
# ============================================================

trade_payload_reconciliation_rows = []


for field_name, field_spec in (
    TRADE_PAYLOAD_RECONCILIATION_PATHS.items()
):
    state = payload_reconciliation_state[
        field_name
    ]

    trade_payload_reconciliation_rows.append(
        {
            "canonical_field": field_name,
            "collector_path": (
                field_spec[
                    "collector_path"
                ]
            ),
            "payload_path": (
                field_spec[
                    "payload_path"
                ]
            ),
            "row_count": int(
                state["row_count"]
            ),
            "collector_present_count": int(
                state[
                    "collector_present_count"
                ]
            ),
            "payload_present_count": int(
                state[
                    "payload_present_count"
                ]
            ),
            "both_present_count": int(
                state[
                    "both_present_count"
                ]
            ),
            "collector_null_count": int(
                state[
                    "collector_null_count"
                ]
            ),
            "payload_null_count": int(
                state[
                    "payload_null_count"
                ]
            ),
            "match_count": int(
                state["match_count"]
            ),
            "mismatch_count": int(
                state["mismatch_count"]
            ),
        }
    )


TRADE_PAYLOAD_RECONCILIATION = pd.DataFrame(
    trade_payload_reconciliation_rows
)


TRADE_PAYLOAD_RECONCILIATION[
    "status"
] = np.where(
    TRADE_PAYLOAD_RECONCILIATION[
        "collector_present_count"
    ].eq(
        TRADE_PAYLOAD_RECONCILIATION[
            "row_count"
        ]
    )
    & TRADE_PAYLOAD_RECONCILIATION[
        "payload_present_count"
    ].eq(
        TRADE_PAYLOAD_RECONCILIATION[
            "row_count"
        ]
    )
    & TRADE_PAYLOAD_RECONCILIATION[
        "collector_null_count"
    ].eq(0)
    & TRADE_PAYLOAD_RECONCILIATION[
        "payload_null_count"
    ].eq(0)
    & TRADE_PAYLOAD_RECONCILIATION[
        "mismatch_count"
    ].eq(0),
    "PASS",
    "FAIL",
)


# ============================================================
# TRADE IDENTITY FINGERPRINTS
# ============================================================

trade_fingerprints = []


for row in RAW_TRADE_AUDIT_TABLE.itertuples(
    index=False
):
    fingerprint_payload = [
        (
            int(row.exchange_event_time_ms)
            if pd.notna(
                row.exchange_event_time_ms
            )
            else None
        ),
        (
            int(row.trade_time_ms)
            if pd.notna(row.trade_time_ms)
            else None
        ),
        row.symbol,
        (
            int(row.trade_id)
            if pd.notna(row.trade_id)
            else None
        ),
        row.price_raw,
        row.quantity_raw,
        (
            bool(row.buyer_is_maker)
            if pd.notna(
                row.buyer_is_maker
            )
            else None
        ),
    ]

    fingerprint = hashlib.sha256(
        json.dumps(
            fingerprint_payload,
            ensure_ascii=False,
            allow_nan=False,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()

    trade_fingerprints.append(
        fingerprint
    )


RAW_TRADE_AUDIT_TABLE[
    "trade_identity_fingerprint"
] = pd.Series(
    trade_fingerprints,
    index=RAW_TRADE_AUDIT_TABLE.index,
    dtype="string",
)


trade_id_groups = (
    RAW_TRADE_AUDIT_TABLE.groupby(
        "trade_id",
        dropna=False,
        observed=True,
    )
    .agg(
        record_count=(
            "trade_id",
            "size",
        ),
        fingerprint_count=(
            "trade_identity_fingerprint",
            "nunique",
        ),
        first_collector_sequence=(
            "collector_sequence",
            "min",
        ),
        last_collector_sequence=(
            "collector_sequence",
            "max",
        ),
    )
    .reset_index()
)


duplicate_trade_id_groups = (
    trade_id_groups.loc[
        trade_id_groups[
            "record_count"
        ].gt(1)
    ]
    .copy()
)


conflicting_trade_id_groups = (
    duplicate_trade_id_groups.loc[
        duplicate_trade_id_groups[
            "fingerprint_count"
        ].gt(1)
    ]
    .copy()
)


exact_duplicate_trade_id_groups = (
    duplicate_trade_id_groups.loc[
        duplicate_trade_id_groups[
            "fingerprint_count"
        ].eq(1)
    ]
    .copy()
)


exact_duplicate_record_count = int(
    (
        exact_duplicate_trade_id_groups[
            "record_count"
        ]
        - 1
    ).sum()
)

conflicting_trade_id_count = int(
    len(
        conflicting_trade_id_groups
    )
)


# ============================================================
# REGISTER DUPLICATE REJECTION CANDIDATES
# ============================================================

if not duplicate_trade_id_groups.empty:
    duplicate_ids = set(
        int(value)
        for value in (
            duplicate_trade_id_groups[
                "trade_id"
            ]
            .dropna()
            .tolist()
        )
    )

    duplicate_records = (
        RAW_TRADE_AUDIT_TABLE.loc[
            RAW_TRADE_AUDIT_TABLE[
                "trade_id"
            ].isin(duplicate_ids)
        ]
        .sort_values(
            [
                "trade_id",
                "collector_sequence",
            ],
            kind="stable",
        )
    )

    for trade_id_value, group in (
        duplicate_records.groupby(
            "trade_id",
            sort=False,
        )
    ):
        group = group.sort_values(
            "collector_sequence",
            kind="stable",
        )

        fingerprints_in_group = int(
            group[
                "trade_identity_fingerprint"
            ].nunique()
        )

        if fingerprints_in_group == 1:
            rejected_group = group.iloc[1:]

            reason_code = (
                "TRADE_ID_EXACT_DUPLICATE"
            )

        else:
            rejected_group = group

            reason_code = (
                "TRADE_ID_CONFLICT"
            )

        for rejected_row in (
            rejected_group.itertuples(
                index=False
            )
        ):
            trade_rejection_rows.append(
                {
                    "source_line_number": int(
                        rejected_row[
                            RAW_TRADE_AUDIT_TABLE
                            .columns
                            .get_loc(
                                "source_line_number"
                            )
                        ]
                    )
                    if False
                    else int(
                        rejected_row.source_line_number
                    ),
                    "collector_sequence": int(
                        rejected_row.collector_sequence
                    ),
                    "trade_id": int(
                        trade_id_value
                    ),
                    "reason_code": reason_code,
                }
            )


TRADE_REJECTION_CANDIDATES = pd.DataFrame(
    trade_rejection_rows,
    columns=[
        "source_line_number",
        "collector_sequence",
        "trade_id",
        "reason_code",
    ],
).drop_duplicates(
    ignore_index=True
)


# ============================================================
# TRADE-ID ORDER DIAGNOSTICS
# ============================================================

ordered_trade_ids = (
    RAW_TRADE_AUDIT_TABLE[
        "trade_id"
    ]
    .dropna()
    .to_numpy(dtype=np.int64)
)

trade_id_differences = np.diff(
    ordered_trade_ids
)

trade_id_reversal_count = int(
    np.sum(
        trade_id_differences <= 0
    )
)

trade_id_gap_transition_count = int(
    np.sum(
        trade_id_differences > 1
    )
)

trade_id_missing_value_count = int(
    np.sum(
        np.maximum(
            trade_id_differences - 1,
            0,
        )
    )
)


TRADE_IDENTITY_AUDIT = pd.DataFrame(
    [
        {
            "row_count": int(
                len(RAW_TRADE_AUDIT_TABLE)
            ),
            "unique_trade_id_count": int(
                RAW_TRADE_AUDIT_TABLE[
                    "trade_id"
                ].nunique(
                    dropna=True
                )
            ),
            "first_trade_id": (
                int(ordered_trade_ids[0])
                if len(ordered_trade_ids)
                else None
            ),
            "last_trade_id": (
                int(ordered_trade_ids[-1])
                if len(ordered_trade_ids)
                else None
            ),
            "duplicate_trade_id_group_count": int(
                len(
                    duplicate_trade_id_groups
                )
            ),
            "exact_duplicate_record_count": (
                exact_duplicate_record_count
            ),
            "conflicting_trade_id_count": (
                conflicting_trade_id_count
            ),
            "trade_id_reversal_count": (
                trade_id_reversal_count
            ),
            "trade_id_gap_transition_count": (
                trade_id_gap_transition_count
            ),
            "trade_id_missing_value_count": (
                trade_id_missing_value_count
            ),
        }
    ]
)


# ============================================================
# EXACT-DECIMAL NUMERIC PROFILE
# ============================================================

price_decimals = [
    Decimal(value)
    for value in (
        RAW_TRADE_AUDIT_TABLE[
            "price_raw"
        ]
        .dropna()
        .astype(str)
        .tolist()
    )
]

quantity_decimals = [
    Decimal(value)
    for value in (
        RAW_TRADE_AUDIT_TABLE[
            "quantity_raw"
        ]
        .dropna()
        .astype(str)
        .tolist()
    )
]


TRADE_NUMERIC_PROFILE = pd.DataFrame(
    [
        {
            "field": "price",
            "row_count": len(
                price_decimals
            ),
            "minimum": (
                format(
                    min(price_decimals),
                    "f",
                )
                if price_decimals
                else None
            ),
            "maximum": (
                format(
                    max(price_decimals),
                    "f",
                )
                if price_decimals
                else None
            ),
            "minimum_decimal_places": (
                min(
                    decimal_places(value)
                    for value in (
                        RAW_TRADE_AUDIT_TABLE[
                            "price_raw"
                        ]
                        .dropna()
                    )
                )
                if price_decimals
                else None
            ),
            "maximum_decimal_places": (
                max(
                    decimal_places(value)
                    for value in (
                        RAW_TRADE_AUDIT_TABLE[
                            "price_raw"
                        ]
                        .dropna()
                    )
                )
                if price_decimals
                else None
            ),
            "nonpositive_count": int(
                sum(
                    value <= 0
                    for value in (
                        price_decimals
                    )
                )
            ),
        },
        {
            "field": "quantity",
            "row_count": len(
                quantity_decimals
            ),
            "minimum": (
                format(
                    min(quantity_decimals),
                    "f",
                )
                if quantity_decimals
                else None
            ),
            "maximum": (
                format(
                    max(quantity_decimals),
                    "f",
                )
                if quantity_decimals
                else None
            ),
            "minimum_decimal_places": (
                min(
                    decimal_places(value)
                    for value in (
                        RAW_TRADE_AUDIT_TABLE[
                            "quantity_raw"
                        ]
                        .dropna()
                    )
                )
                if quantity_decimals
                else None
            ),
            "maximum_decimal_places": (
                max(
                    decimal_places(value)
                    for value in (
                        RAW_TRADE_AUDIT_TABLE[
                            "quantity_raw"
                        ]
                        .dropna()
                    )
                )
                if quantity_decimals
                else None
            ),
            "nonpositive_count": int(
                sum(
                    value <= 0
                    for value in (
                        quantity_decimals
                    )
                )
            ),
        },
    ]
)


# ============================================================
# AGGRESSOR-SIDE SUMMARY
# ============================================================

side_accumulators = {
    "BUY": {
        "trade_count": 0,
        "total_quantity": Decimal("0"),
        "total_notional": Decimal("0"),
    },
    "SELL": {
        "trade_count": 0,
        "total_quantity": Decimal("0"),
        "total_notional": Decimal("0"),
    },
}


for row in RAW_TRADE_AUDIT_TABLE.itertuples(
    index=False
):
    if pd.isna(row.aggressor_side):
        continue

    side = str(
        row.aggressor_side
    )

    price = Decimal(
        str(row.price_raw)
    )

    quantity = Decimal(
        str(row.quantity_raw)
    )

    side_accumulators[
        side
    ]["trade_count"] += 1

    side_accumulators[
        side
    ]["total_quantity"] += quantity

    side_accumulators[
        side
    ]["total_notional"] += (
        price * quantity
    )


TRADE_AGGRESSOR_SIDE_SUMMARY = pd.DataFrame(
    [
        {
            "aggressor_side": side,
            "buyer_is_maker": (
                side == "SELL"
            ),
            "trade_count": int(
                values["trade_count"]
            ),
            "trade_share": (
                values["trade_count"]
                / len(
                    RAW_TRADE_AUDIT_TABLE
                )
            ),
            "total_quantity": format(
                values[
                    "total_quantity"
                ],
                "f",
            ),
            "total_notional": format(
                values[
                    "total_notional"
                ],
                "f",
            ),
        }
        for side, values in (
            side_accumulators.items()
        )
    ]
)


# ============================================================
# PARTITION SUMMARY
# ============================================================

TRADE_PARTITION_SUMMARY = (
    RAW_TRADE_AUDIT_TABLE.groupby(
        [
            "partition",
            "aggressor_side",
        ],
        observed=True,
        dropna=False,
    )
    .size()
    .rename("trade_count")
    .reset_index()
)


expected_trade_partition_counts = (
    FROZEN_PARTITION_BOUNDARIES[
        [
            "partition",
            "trade_row_count",
        ]
    ]
    .rename(
        columns={
            "trade_row_count": (
                "expected_trade_count"
            )
        }
    )
)


observed_trade_partition_counts = (
    RAW_TRADE_AUDIT_TABLE.groupby(
        "partition",
        observed=True,
        dropna=False,
    )
    .size()
    .rename(
        "observed_trade_count"
    )
    .reset_index()
)


TRADE_PARTITION_RECONCILIATION = (
    expected_trade_partition_counts.merge(
        observed_trade_partition_counts,
        on="partition",
        how="left",
        validate="one_to_one",
    )
)


TRADE_PARTITION_RECONCILIATION[
    "observed_trade_count"
] = (
    TRADE_PARTITION_RECONCILIATION[
        "observed_trade_count"
    ]
    .fillna(0)
    .astype("int64")
)

TRADE_PARTITION_RECONCILIATION[
    "expected_trade_count"
] = (
    TRADE_PARTITION_RECONCILIATION[
        "expected_trade_count"
    ].astype("int64")
)

TRADE_PARTITION_RECONCILIATION[
    "matches"
] = (
    TRADE_PARTITION_RECONCILIATION[
        "observed_trade_count"
    ]
    == TRADE_PARTITION_RECONCILIATION[
        "expected_trade_count"
    ]
)


# ============================================================
# ACCEPTANCE GATES
# ============================================================

payload_fields_complete = bool(
    TRADE_PAYLOAD_RECONCILIATION[
        "collector_present_count"
    ].eq(
        TRADE_PAYLOAD_RECONCILIATION[
            "row_count"
        ]
    ).all()
    and TRADE_PAYLOAD_RECONCILIATION[
        "payload_present_count"
    ].eq(
        TRADE_PAYLOAD_RECONCILIATION[
            "row_count"
        ]
    ).all()
)

payload_fields_consistent = bool(
    TRADE_PAYLOAD_RECONCILIATION[
        "mismatch_count"
    ].eq(0).all()
)

trade_ids_unique = bool(
    len(
        duplicate_trade_id_groups
    )
    == 0
)

trade_ids_conflict_free = bool(
    conflicting_trade_id_count
    == 0
)

trade_ids_strictly_increasing = bool(
    trade_id_reversal_count
    == 0
)

trade_numeric_values_valid = bool(
    TRADE_NUMERIC_PROFILE[
        "nonpositive_count"
    ].eq(0).all()
    and TRADE_NUMERIC_PROFILE[
        "row_count"
    ].eq(
        len(
            RAW_TRADE_AUDIT_TABLE
        )
    ).all()
)

aggressor_side_complete = bool(
    RAW_TRADE_AUDIT_TABLE[
        "aggressor_side"
    ].notna().all()
    and TRADE_AGGRESSOR_SIDE_SUMMARY[
        "trade_count"
    ].sum()
    == len(
        RAW_TRADE_AUDIT_TABLE
    )
)

trade_partitions_match = bool(
    TRADE_PARTITION_RECONCILIATION[
        "matches"
    ].all()
)

critical_trade_rejections = (
    TRADE_REJECTION_CANDIDATES.loc[
        ~TRADE_REJECTION_CANDIDATES[
            "reason_code"
        ].eq(
            "TRADE_ID_EXACT_DUPLICATE"
        )
    ]
    if not TRADE_REJECTION_CANDIDATES.empty
    else TRADE_REJECTION_CANDIDATES
)


TRADE_SEMANTIC_GATES = pd.DataFrame(
    [
        {
            "gate": (
                "trade_row_count_preserved"
            ),
            "passed": bool(
                len(
                    RAW_TRADE_AUDIT_TABLE
                )
                == EXPECTED_TRADE_ROWS
            ),
            "evidence": (
                f"observed="
                f"{len(RAW_TRADE_AUDIT_TABLE)}; "
                f"expected="
                f"{EXPECTED_TRADE_ROWS}"
            ),
        },
        {
            "gate": (
                "collector_and_payload_fields_complete"
            ),
            "passed": (
                payload_fields_complete
            ),
            "evidence": (
                f"fields="
                f"{len(TRADE_PAYLOAD_RECONCILIATION)}"
            ),
        },
        {
            "gate": (
                "collector_and_payload_values_match"
            ),
            "passed": (
                payload_fields_consistent
            ),
            "evidence": (
                f"mismatches="
                f"{int(TRADE_PAYLOAD_RECONCILIATION['mismatch_count'].sum())}"
            ),
        },
        {
            "gate": "trade_ids_unique",
            "passed": trade_ids_unique,
            "evidence": (
                f"duplicate_groups="
                f"{len(duplicate_trade_id_groups)}; "
                f"duplicate_records="
                f"{exact_duplicate_record_count}"
            ),
        },
        {
            "gate": (
                "trade_ids_conflict_free"
            ),
            "passed": (
                trade_ids_conflict_free
            ),
            "evidence": (
                f"conflicting_ids="
                f"{conflicting_trade_id_count}"
            ),
        },
        {
            "gate": (
                "trade_ids_strictly_increasing"
            ),
            "passed": (
                trade_ids_strictly_increasing
            ),
            "evidence": (
                f"reversals="
                f"{trade_id_reversal_count}; "
                f"gap_transitions="
                f"{trade_id_gap_transition_count}"
            ),
        },
        {
            "gate": (
                "trade_numeric_values_positive"
            ),
            "passed": (
                trade_numeric_values_valid
            ),
            "evidence": (
                f"nonpositive="
                f"{int(TRADE_NUMERIC_PROFILE['nonpositive_count'].sum())}"
            ),
        },
        {
            "gate": (
                "aggressor_side_mapping_complete"
            ),
            "passed": (
                aggressor_side_complete
            ),
            "evidence": (
                f"classified="
                f"{int(TRADE_AGGRESSOR_SIDE_SUMMARY['trade_count'].sum())}/"
                f"{len(RAW_TRADE_AUDIT_TABLE)}"
            ),
        },
        {
            "gate": (
                "trade_partition_counts_match"
            ),
            "passed": (
                trade_partitions_match
            ),
            "evidence": (
                f"partitions="
                f"{len(TRADE_PARTITION_RECONCILIATION)}"
            ),
        },
        {
            "gate": (
                "no_critical_trade_rejections"
            ),
            "passed": bool(
                critical_trade_rejections.empty
            ),
            "evidence": (
                f"critical_rejections="
                f"{len(critical_trade_rejections)}"
            ),
        },
    ]
)


# ============================================================
# DISPLAY AND ENFORCE
# ============================================================

display(
    TRADE_PAYLOAD_RECONCILIATION[
        [
            "canonical_field",
            "collector_path",
            "payload_path",
            "row_count",
            "collector_present_count",
            "payload_present_count",
            "match_count",
            "mismatch_count",
            "status",
        ]
    ]
)

display(TRADE_IDENTITY_AUDIT)
display(TRADE_NUMERIC_PROFILE)
display(TRADE_AGGRESSOR_SIDE_SUMMARY)
display(TRADE_PARTITION_SUMMARY)
display(TRADE_PARTITION_RECONCILIATION)
display(TRADE_SEMANTIC_GATES)


if not duplicate_trade_id_groups.empty:
    display(duplicate_trade_id_groups)


if not TRADE_REJECTION_CANDIDATES.empty:
    display(TRADE_REJECTION_CANDIDATES)


failed_trade_semantic_gates = (
    TRADE_SEMANTIC_GATES.loc[
        ~TRADE_SEMANTIC_GATES[
            "passed"
        ]
    ]
)


if not failed_trade_semantic_gates.empty:
    failure_text = ", ".join(
        f"{row.gate}: {row.evidence}"
        for row in (
            failed_trade_semantic_gates
            .itertuples(index=False)
        )
    )

    raise RuntimeError(
        "Trade identity and semantic audit failed: "
        + failure_text
    )


print("Trade identity and semantic audit: PASS")
print(
    "Trade records audited: "
    f"{len(RAW_TRADE_AUDIT_TABLE):,}"
)
print(
    "Unique trade IDs: "
    f"{RAW_TRADE_AUDIT_TABLE['trade_id'].nunique():,}"
)
print(
    "Trade ID range: "
    f"{int(ordered_trade_ids[0]):,}.."
    f"{int(ordered_trade_ids[-1]):,}"
)
print(
    "Trade ID gap transitions: "
    f"{trade_id_gap_transition_count:,}"
)
print(
    "Payload reconciliation mismatches: "
    f"{int(TRADE_PAYLOAD_RECONCILIATION['mismatch_count'].sum()):,}"
)
print(
    "BUY aggressor trades: "
    f"{side_accumulators['BUY']['trade_count']:,}"
)
print(
    "SELL aggressor trades: "
    f"{side_accumulators['SELL']['trade_count']:,}"
)

,canonical_field,collector_path,payload_path,row_count,collector_present_count,payload_present_count,match_count,mismatch_count,status
0,event_type,$.event_type,$.raw_message.data.e,67683,67683,67683,67683,0,PASS
1,exchange_event_time_ms,$.exchange_event_time_raw,$.raw_message.data.E,67683,67683,67683,67683,0,PASS
2,trade_time_ms,$.trade_time_raw,$.raw_message.data.T,67683,67683,67683,67683,0,PASS
3,symbol,$.symbol,$.raw_message.data.s,67683,67683,67683,67683,0,PASS
4,trade_id,$.trade_id,$.raw_message.data.t,67683,67683,67683,67683,0,PASS
5,price,$.price_raw,$.raw_message.data.p,67683,67683,67683,67683,0,PASS
6,quantity,$.quantity_raw,$.raw_message.data.q,67683,67683,67683,67683,0,PASS
7,buyer_is_maker,$.buyer_is_maker_raw,$.raw_message.data.m,67683,67683,67683,67683,0,PASS


,row_count,unique_trade_id_count,first_trade_id,last_trade_id,duplicate_trade_id_group_count,exact_duplicate_record_count,conflicting_trade_id_count,trade_id_reversal_count,trade_id_gap_transition_count,trade_id_missing_value_count
0,67683,67683,6494596041,6494663723,0,0,0,0,0,0


,field,row_count,minimum,maximum,minimum_decimal_places,maximum_decimal_places,nonpositive_count
0,price,67683,63802.02000000,64011.90000000,8,8,0
1,quantity,67683,0.00001000,2.50239000,8,8,0


,aggressor_side,buyer_is_maker,trade_count,trade_share,total_quantity,total_notional
0,BUY,False,30596,0.452049,131.78761000,8421584.8935796000000000
1,SELL,True,37087,0.547951,197.41086000,12614364.1336915000000000


,partition,aggressor_side,trade_count
0,CALIBRATION,BUY,7267
1,CALIBRATION,SELL,6265
2,DEVELOPMENT,BUY,15194
3,DEVELOPMENT,SELL,18626
4,ENGINEERING_HOLDOUT,BUY,2361
5,ENGINEERING_HOLDOUT,SELL,7772
6,VALIDATION,BUY,5774
7,VALIDATION,SELL,4424


,partition,expected_trade_count,observed_trade_count,matches
0,DEVELOPMENT,33820,33820,True
1,CALIBRATION,13532,13532,True
2,VALIDATION,10198,10198,True
3,ENGINEERING_HOLDOUT,10133,10133,True


,gate,passed,evidence
0,trade_row_count_preserved,True,observed=67683; expected=67683
1,collector_and_payload_fields_complete,True,fields=8
2,collector_and_payload_values_match,True,mismatches=0
3,trade_ids_unique,True,duplicate_groups=0; duplicate_records=0
4,trade_ids_conflict_free,True,conflicting_ids=0
5,trade_ids_strictly_increasing,True,reversals=0; gap_transitions=0
6,trade_numeric_values_positive,True,nonpositive=0
7,aggressor_side_mapping_complete,True,classified=67683/67683
8,trade_partition_counts_match,True,partitions=4
9,no_critical_trade_rejections,True,critical_rejections=0


Trade identity and semantic audit: PASS
Trade records audited: 67,683
Unique trade IDs: 67,683
Trade ID range: 6,494,596,041..6,494,663,723
Trade ID gap transitions: 0
Payload reconciliation mismatches: 0
BUY aggressor trades: 30,596
SELL aggressor trades: 37,087


In [38]:
# ============================================================
# DEPTH PAYLOAD, UPDATE-ID, LEVEL, AND SNAPSHOT-BRIDGE AUDIT
# ============================================================

from collections import Counter
from decimal import Decimal, InvalidOperation


# ============================================================
# DEPTH FIELD PATHS
# ============================================================

DEPTH_BINDINGS = STREAM_BINDING_PATHS[
    "DEPTH_STREAM"
]

DEPTH_PAYLOAD_RECONCILIATION_PATHS = {
    "event_type": {
        "collector_path": "$.event_type",
        "payload_path": "$.raw_message.data.e",
        "normalizer": "STRING",
    },
    "exchange_event_time_ms": {
        "collector_path": "$.exchange_event_time_raw",
        "payload_path": "$.raw_message.data.E",
        "normalizer": "INTEGER",
    },
    "symbol": {
        "collector_path": "$.symbol",
        "payload_path": "$.raw_message.data.s",
        "normalizer": "STRING",
    },
    "first_update_id": {
        "collector_path": "$.first_update_id",
        "payload_path": "$.raw_message.data.U",
        "normalizer": "INTEGER",
    },
    "final_update_id": {
        "collector_path": "$.final_update_id",
        "payload_path": "$.raw_message.data.u",
        "normalizer": "INTEGER",
    },
    "bid_updates": {
        "collector_path": "$.bid_changes_raw",
        "payload_path": "$.raw_message.data.b",
        "normalizer": "ARRAY",
    },
    "ask_updates": {
        "collector_path": "$.ask_changes_raw",
        "payload_path": "$.raw_message.data.a",
        "normalizer": "ARRAY",
    },
}


# ============================================================
# NORMALIZATION HELPERS
# ============================================================

def normalized_array_hash(
    value,
):
    decoded = decode_json_container(value)

    if not isinstance(decoded, list):
        return None

    encoded = json.dumps(
        decoded,
        sort_keys=False,
        ensure_ascii=False,
        allow_nan=False,
        separators=(",", ":"),
    ).encode("utf-8")

    return hashlib.sha256(
        encoded
    ).hexdigest()


def normalized_depth_comparison_value(
    value,
    normalizer: str,
):
    if value is PATH_MISSING or value is None:
        return None

    if normalizer == "INTEGER":
        return parse_integer_value(value)

    if normalizer == "STRING":
        return str(value)

    if normalizer == "ARRAY":
        return normalized_array_hash(value)

    raise ValueError(
        f"Unsupported depth normalizer: {normalizer}"
    )


def parse_snapshot_mapping(
    document: dict,
) -> tuple[dict, int]:
    raw_response = decode_json_container(
        document.get("raw_response")
    )

    if not isinstance(raw_response, dict):
        raw_response = {}

    top_level_update_id = parse_integer_value(
        document.get("lastUpdateId")
    )

    raw_response_update_id = parse_integer_value(
        raw_response.get("lastUpdateId")
    )

    update_ids = {
        value
        for value in (
            top_level_update_id,
            raw_response_update_id,
        )
        if value is not None
    }

    if len(update_ids) != 1:
        raise RuntimeError(
            "Snapshot lastUpdateId is missing or conflicting: "
            f"top_level={top_level_update_id}; "
            f"raw_response={raw_response_update_id}"
        )

    return raw_response, int(
        next(iter(update_ids))
    )


def initialize_level_profile() -> dict:
    return {
        "event_array_count": 0,
        "empty_array_count": 0,
        "level_count": 0,
        "valid_level_count": 0,
        "invalid_level_count": 0,
        "zero_quantity_count": 0,
        "positive_quantity_count": 0,
        "duplicate_price_count": 0,
        "minimum_price": None,
        "maximum_price": None,
        "minimum_quantity": None,
        "maximum_quantity": None,
        "price_decimal_places": Counter(),
        "quantity_decimal_places": Counter(),
    }


def update_minimum(
    current,
    candidate,
):
    if candidate is None:
        return current

    if current is None:
        return candidate

    return min(current, candidate)


def update_maximum(
    current,
    candidate,
):
    if candidate is None:
        return current

    if current is None:
        return candidate

    return max(current, candidate)


def audit_depth_update_array(
    update_array,
    side: str,
    level_profile: dict,
) -> dict:
    """
    Audit one bid or ask update array without modifying its contents.
    """
    result = {
        "array_valid": False,
        "level_count": 0,
        "invalid_level_count": 0,
        "zero_quantity_count": 0,
        "positive_quantity_count": 0,
        "duplicate_price_count": 0,
    }

    updates = normalize_update_array(
        update_array
    )

    if updates is None:
        return result

    result["array_valid"] = True
    result["level_count"] = len(updates)

    level_profile[
        "event_array_count"
    ] += 1

    level_profile[
        "level_count"
    ] += len(updates)

    if len(updates) == 0:
        level_profile[
            "empty_array_count"
        ] += 1

    observed_prices = set()

    for level in updates:
        decoded_level = decode_json_container(
            level
        )

        if (
            not isinstance(
                decoded_level,
                (list, tuple),
            )
            or len(decoded_level) != 2
        ):
            result[
                "invalid_level_count"
            ] += 1

            level_profile[
                "invalid_level_count"
            ] += 1

            continue

        price_value = parse_decimal_value(
            decoded_level[0]
        )

        quantity_value = parse_decimal_value(
            decoded_level[1]
        )

        valid_level = bool(
            price_value is not None
            and price_value > 0
            and quantity_value is not None
            and quantity_value >= 0
        )

        if not valid_level:
            result[
                "invalid_level_count"
            ] += 1

            level_profile[
                "invalid_level_count"
            ] += 1

            continue

        level_profile[
            "valid_level_count"
        ] += 1

        normalized_price = format(
            price_value,
            "f",
        )

        if normalized_price in observed_prices:
            result[
                "duplicate_price_count"
            ] += 1

            level_profile[
                "duplicate_price_count"
            ] += 1

        else:
            observed_prices.add(
                normalized_price
            )

        if quantity_value == 0:
            result[
                "zero_quantity_count"
            ] += 1

            level_profile[
                "zero_quantity_count"
            ] += 1

        else:
            result[
                "positive_quantity_count"
            ] += 1

            level_profile[
                "positive_quantity_count"
            ] += 1

        level_profile["minimum_price"] = (
            update_minimum(
                level_profile[
                    "minimum_price"
                ],
                price_value,
            )
        )

        level_profile["maximum_price"] = (
            update_maximum(
                level_profile[
                    "maximum_price"
                ],
                price_value,
            )
        )

        level_profile[
            "minimum_quantity"
        ] = update_minimum(
            level_profile[
                "minimum_quantity"
            ],
            quantity_value,
        )

        level_profile[
            "maximum_quantity"
        ] = update_maximum(
            level_profile[
                "maximum_quantity"
            ],
            quantity_value,
        )

        price_places = decimal_places(
            decoded_level[0]
        )

        quantity_places = decimal_places(
            decoded_level[1]
        )

        if price_places is not None:
            level_profile[
                "price_decimal_places"
            ][price_places] += 1

        if quantity_places is not None:
            level_profile[
                "quantity_decimal_places"
            ][quantity_places] += 1

    return result


# ============================================================
# SNAPSHOT REGISTRATION
# ============================================================

(
    SNAPSHOT_RAW_RESPONSE,
    SNAPSHOT_LAST_UPDATE_ID,
) = parse_snapshot_mapping(
    SNAPSHOT_DOCUMENT
)

CONTRACT_SNAPSHOT_LAST_UPDATE_ID = int(
    MARKET_DATA_CONTRACT[
        "book_representation"
    ]["snapshot_last_update_id"]
)

SNAPSHOT_RESPONSE_RECEIVED_TIME_NS = (
    parse_integer_value(
        SNAPSHOT_DOCUMENT.get(
            "response_received_time_ns"
        )
    )
)

SNAPSHOT_REQUEST_STARTED_TIME_NS = (
    parse_integer_value(
        SNAPSHOT_DOCUMENT.get(
            "request_started_time_ns"
        )
    )
)

DECLARED_BUFFERED_DEPTH_COUNT = (
    parse_integer_value(
        SNAPSHOT_DOCUMENT.get(
            "buffered_depth_before_snapshot"
        )
    )
)

SNAPSHOT_SESSION_ID = (
    SNAPSHOT_DOCUMENT.get(
        "connection_session_id"
    )
)


# ============================================================
# AUDIT STATE
# ============================================================

depth_payload_state = {
    field_name: {
        "row_count": 0,
        "collector_present_count": 0,
        "payload_present_count": 0,
        "both_present_count": 0,
        "collector_null_count": 0,
        "payload_null_count": 0,
        "match_count": 0,
        "mismatch_count": 0,
    }
    for field_name in (
        DEPTH_PAYLOAD_RECONCILIATION_PATHS
    )
}

depth_level_profiles = {
    "BID": initialize_level_profile(),
    "ASK": initialize_level_profile(),
}

depth_rows = []
depth_rejection_rows = []


# ============================================================
# STREAM THE AUTHORITATIVE DEPTH SOURCE
# ============================================================

with Path(DEPTH_SOURCE_PATH).open(
    "r",
    encoding="utf-8-sig",
    errors="strict",
) as handle:
    for source_line_number, line in enumerate(
        handle,
        start=1,
    ):
        if not line.strip():
            continue

        try:
            raw_record = json.loads(line)

        except json.JSONDecodeError as exc:
            raise RuntimeError(
                "Invalid JSON in DEPTH_STREAM at line "
                f"{source_line_number}: {exc}"
            ) from exc

        if not isinstance(raw_record, dict):
            raise RuntimeError(
                "Non-mapping DEPTH_STREAM record at line "
                f"{source_line_number}."
            )

        record = prepare_raw_record(
            raw_record
        )

        collector_sequence = resolved_integer(
            record,
            DEPTH_BINDINGS[
                "collector_sequence"
            ],
        )

        local_receipt_time_ns = resolved_integer(
            record,
            DEPTH_BINDINGS[
                "local_receipt_time_ns"
            ],
        )

        exchange_event_time_ms = resolved_integer(
            record,
            DEPTH_BINDINGS[
                "exchange_event_time_ms"
            ],
        )

        first_update_id = resolved_integer(
            record,
            DEPTH_BINDINGS[
                "first_update_id"
            ],
        )

        final_update_id = resolved_integer(
            record,
            DEPTH_BINDINGS[
                "final_update_id"
            ],
        )

        connection_session_id = resolved_string(
            record,
            DEPTH_BINDINGS[
                "connection_session_id"
            ],
        )

        event_type = resolved_string(
            record,
            DEPTH_BINDINGS[
                "event_type"
            ],
        )

        symbol = resolved_string(
            record,
            DEPTH_BINDINGS["symbol"],
        )

        bid_updates_value = resolve_json_path(
            record,
            DEPTH_BINDINGS[
                "bid_updates"
            ],
        )

        ask_updates_value = resolve_json_path(
            record,
            DEPTH_BINDINGS[
                "ask_updates"
            ],
        )

        bid_result = audit_depth_update_array(
            update_array=bid_updates_value,
            side="BID",
            level_profile=(
                depth_level_profiles["BID"]
            ),
        )

        ask_result = audit_depth_update_array(
            update_array=ask_updates_value,
            side="ASK",
            level_profile=(
                depth_level_profiles["ASK"]
            ),
        )

        partition = assign_partition_from_sequence(
            collector_sequence
        )

        update_range_valid = bool(
            first_update_id is not None
            and final_update_id is not None
            and first_update_id >= 0
            and final_update_id >= 0
            and first_update_id
            <= final_update_id
        )

        both_sides_empty = bool(
            bid_result["array_valid"]
            and ask_result["array_valid"]
            and bid_result["level_count"] == 0
            and ask_result["level_count"] == 0
        )

        duplicate_price_count = int(
            bid_result[
                "duplicate_price_count"
            ]
            + ask_result[
                "duplicate_price_count"
            ]
        )

        invalid_level_count = int(
            bid_result[
                "invalid_level_count"
            ]
            + ask_result[
                "invalid_level_count"
            ]
        )

        row_reasons = []

        if collector_sequence is None:
            row_reasons.append(
                "COLLECTOR_SEQUENCE_MISSING"
            )

        if (
            local_receipt_time_ns is None
            or exchange_event_time_ms is None
        ):
            row_reasons.append(
                "DEPTH_REQUIRED_FIELD_MISSING"
            )

        if event_type != "depthUpdate":
            row_reasons.append(
                "DEPTH_EVENT_TYPE_INVALID"
            )

        if symbol != "BTCUSDT":
            row_reasons.append(
                "DEPTH_SYMBOL_MISMATCH"
            )

        if not update_range_valid:
            row_reasons.append(
                "DEPTH_UPDATE_RANGE_INVALID"
            )

        if not (
            bid_result["array_valid"]
            and ask_result["array_valid"]
        ):
            row_reasons.append(
                "DEPTH_REQUIRED_FIELD_MISSING"
            )

        if invalid_level_count > 0:
            row_reasons.append(
                "DEPTH_LEVEL_MALFORMED"
            )

        if duplicate_price_count > 0:
            row_reasons.append(
                "DEPTH_DUPLICATE_PRICE_IN_EVENT"
            )

        if both_sides_empty:
            row_reasons.append(
                "DEPTH_BOTH_SIDES_EMPTY"
            )

        if partition is None:
            row_reasons.append(
                "COLLECTOR_SEQUENCE_GAP"
            )

        for reason_code in sorted(
            set(row_reasons)
        ):
            depth_rejection_rows.append(
                {
                    "source_line_number": (
                        source_line_number
                    ),
                    "collector_sequence": (
                        collector_sequence
                    ),
                    "first_update_id": (
                        first_update_id
                    ),
                    "final_update_id": (
                        final_update_id
                    ),
                    "reason_code": reason_code,
                }
            )

        depth_rows.append(
            {
                "source_line_number": (
                    source_line_number
                ),
                "collector_sequence": (
                    collector_sequence
                ),
                "local_receipt_time_ns": (
                    local_receipt_time_ns
                ),
                "exchange_event_time_ms": (
                    exchange_event_time_ms
                ),
                "connection_session_id": (
                    connection_session_id
                ),
                "event_type": event_type,
                "symbol": symbol,
                "first_update_id": (
                    first_update_id
                ),
                "final_update_id": (
                    final_update_id
                ),
                "bid_update_count": int(
                    bid_result["level_count"]
                ),
                "ask_update_count": int(
                    ask_result["level_count"]
                ),
                "zero_quantity_update_count": int(
                    bid_result[
                        "zero_quantity_count"
                    ]
                    + ask_result[
                        "zero_quantity_count"
                    ]
                ),
                "positive_quantity_update_count": int(
                    bid_result[
                        "positive_quantity_count"
                    ]
                    + ask_result[
                        "positive_quantity_count"
                    ]
                ),
                "invalid_level_count": (
                    invalid_level_count
                ),
                "duplicate_price_count": (
                    duplicate_price_count
                ),
                "both_sides_empty": (
                    both_sides_empty
                ),
                "update_range_valid": (
                    update_range_valid
                ),
                "partition": partition,
            }
        )

        for field_name, field_spec in (
            DEPTH_PAYLOAD_RECONCILIATION_PATHS.items()
        ):
            state = depth_payload_state[
                field_name
            ]

            state["row_count"] += 1

            collector_value = resolve_json_path(
                record,
                field_spec[
                    "collector_path"
                ],
            )

            payload_value = resolve_json_path(
                record,
                field_spec[
                    "payload_path"
                ],
            )

            collector_present = (
                collector_value
                is not PATH_MISSING
            )

            payload_present = (
                payload_value
                is not PATH_MISSING
            )

            if collector_present:
                state[
                    "collector_present_count"
                ] += 1

            if payload_present:
                state[
                    "payload_present_count"
                ] += 1

            if (
                collector_present
                and payload_present
            ):
                state[
                    "both_present_count"
                ] += 1

            if (
                collector_present
                and collector_value is None
            ):
                state[
                    "collector_null_count"
                ] += 1

            if (
                payload_present
                and payload_value is None
            ):
                state[
                    "payload_null_count"
                ] += 1

            collector_normalized = (
                normalized_depth_comparison_value(
                    collector_value,
                    field_spec[
                        "normalizer"
                    ],
                )
            )

            payload_normalized = (
                normalized_depth_comparison_value(
                    payload_value,
                    field_spec[
                        "normalizer"
                    ],
                )
            )

            if (
                collector_present
                and payload_present
                and collector_normalized
                == payload_normalized
            ):
                state["match_count"] += 1

            elif (
                collector_present
                and payload_present
            ):
                state[
                    "mismatch_count"
                ] += 1


# ============================================================
# EVENT-LEVEL DEPTH AUDIT TABLE
# ============================================================

RAW_DEPTH_EVENT_AUDIT_TABLE = pd.DataFrame(
    depth_rows
)


depth_integer_columns = [
    "source_line_number",
    "collector_sequence",
    "local_receipt_time_ns",
    "exchange_event_time_ms",
    "first_update_id",
    "final_update_id",
    "bid_update_count",
    "ask_update_count",
    "zero_quantity_update_count",
    "positive_quantity_update_count",
    "invalid_level_count",
    "duplicate_price_count",
]

depth_string_columns = [
    "connection_session_id",
    "event_type",
    "symbol",
    "partition",
]

depth_boolean_columns = [
    "both_sides_empty",
    "update_range_valid",
]


for column in depth_integer_columns:
    RAW_DEPTH_EVENT_AUDIT_TABLE[
        column
    ] = pd.array(
        RAW_DEPTH_EVENT_AUDIT_TABLE[
            column
        ],
        dtype="Int64",
    )


for column in depth_string_columns:
    RAW_DEPTH_EVENT_AUDIT_TABLE[
        column
    ] = (
        RAW_DEPTH_EVENT_AUDIT_TABLE[
            column
        ].astype("string")
    )


for column in depth_boolean_columns:
    RAW_DEPTH_EVENT_AUDIT_TABLE[
        column
    ] = pd.array(
        RAW_DEPTH_EVENT_AUDIT_TABLE[
            column
        ],
        dtype="boolean",
    )


RAW_DEPTH_EVENT_AUDIT_TABLE = (
    RAW_DEPTH_EVENT_AUDIT_TABLE
    .sort_values(
        "collector_sequence",
        kind="stable",
    )
    .reset_index(drop=True)
)


DEPTH_REJECTION_CANDIDATES = pd.DataFrame(
    depth_rejection_rows,
    columns=[
        "source_line_number",
        "collector_sequence",
        "first_update_id",
        "final_update_id",
        "reason_code",
    ],
).drop_duplicates(
    ignore_index=True
)


# ============================================================
# COLLECTOR-WRAPPER VS RAW-PAYLOAD RECONCILIATION
# ============================================================

depth_payload_rows = []

for field_name, field_spec in (
    DEPTH_PAYLOAD_RECONCILIATION_PATHS.items()
):
    state = depth_payload_state[
        field_name
    ]

    depth_payload_rows.append(
        {
            "canonical_field": field_name,
            "collector_path": (
                field_spec[
                    "collector_path"
                ]
            ),
            "payload_path": (
                field_spec[
                    "payload_path"
                ]
            ),
            "row_count": int(
                state["row_count"]
            ),
            "collector_present_count": int(
                state[
                    "collector_present_count"
                ]
            ),
            "payload_present_count": int(
                state[
                    "payload_present_count"
                ]
            ),
            "both_present_count": int(
                state[
                    "both_present_count"
                ]
            ),
            "collector_null_count": int(
                state[
                    "collector_null_count"
                ]
            ),
            "payload_null_count": int(
                state[
                    "payload_null_count"
                ]
            ),
            "match_count": int(
                state["match_count"]
            ),
            "mismatch_count": int(
                state["mismatch_count"]
            ),
        }
    )


DEPTH_PAYLOAD_RECONCILIATION = pd.DataFrame(
    depth_payload_rows
)


DEPTH_PAYLOAD_RECONCILIATION[
    "status"
] = np.where(
    DEPTH_PAYLOAD_RECONCILIATION[
        "collector_present_count"
    ].eq(
        DEPTH_PAYLOAD_RECONCILIATION[
            "row_count"
        ]
    )
    & DEPTH_PAYLOAD_RECONCILIATION[
        "payload_present_count"
    ].eq(
        DEPTH_PAYLOAD_RECONCILIATION[
            "row_count"
        ]
    )
    & DEPTH_PAYLOAD_RECONCILIATION[
        "collector_null_count"
    ].eq(0)
    & DEPTH_PAYLOAD_RECONCILIATION[
        "payload_null_count"
    ].eq(0)
    & DEPTH_PAYLOAD_RECONCILIATION[
        "mismatch_count"
    ].eq(0),
    "PASS",
    "FAIL",
)


# ============================================================
# LEVEL PROFILE
# ============================================================

depth_level_profile_rows = []

for side, profile in (
    depth_level_profiles.items()
):
    price_place_counts = profile[
        "price_decimal_places"
    ]

    quantity_place_counts = profile[
        "quantity_decimal_places"
    ]

    depth_level_profile_rows.append(
        {
            "side": side,
            "event_array_count": int(
                profile[
                    "event_array_count"
                ]
            ),
            "empty_array_count": int(
                profile[
                    "empty_array_count"
                ]
            ),
            "level_count": int(
                profile["level_count"]
            ),
            "valid_level_count": int(
                profile[
                    "valid_level_count"
                ]
            ),
            "invalid_level_count": int(
                profile[
                    "invalid_level_count"
                ]
            ),
            "zero_quantity_count": int(
                profile[
                    "zero_quantity_count"
                ]
            ),
            "positive_quantity_count": int(
                profile[
                    "positive_quantity_count"
                ]
            ),
            "duplicate_price_count": int(
                profile[
                    "duplicate_price_count"
                ]
            ),
            "minimum_price": (
                format(
                    profile[
                        "minimum_price"
                    ],
                    "f",
                )
                if profile[
                    "minimum_price"
                ] is not None
                else None
            ),
            "maximum_price": (
                format(
                    profile[
                        "maximum_price"
                    ],
                    "f",
                )
                if profile[
                    "maximum_price"
                ] is not None
                else None
            ),
            "minimum_quantity": (
                format(
                    profile[
                        "minimum_quantity"
                    ],
                    "f",
                )
                if profile[
                    "minimum_quantity"
                ] is not None
                else None
            ),
            "maximum_quantity": (
                format(
                    profile[
                        "maximum_quantity"
                    ],
                    "f",
                )
                if profile[
                    "maximum_quantity"
                ] is not None
                else None
            ),
            "minimum_price_decimal_places": (
                min(price_place_counts)
                if price_place_counts
                else None
            ),
            "maximum_price_decimal_places": (
                max(price_place_counts)
                if price_place_counts
                else None
            ),
            "minimum_quantity_decimal_places": (
                min(quantity_place_counts)
                if quantity_place_counts
                else None
            ),
            "maximum_quantity_decimal_places": (
                max(quantity_place_counts)
                if quantity_place_counts
                else None
            ),
        }
    )


DEPTH_LEVEL_PROFILE = pd.DataFrame(
    depth_level_profile_rows
)


# ============================================================
# UPDATE-ID ORDER AND SNAPSHOT BRIDGE
# ============================================================

first_update_ids = (
    RAW_DEPTH_EVENT_AUDIT_TABLE[
        "first_update_id"
    ].to_numpy(dtype=np.int64)
)

final_update_ids = (
    RAW_DEPTH_EVENT_AUDIT_TABLE[
        "final_update_id"
    ].to_numpy(dtype=np.int64)
)

collector_sequences = (
    RAW_DEPTH_EVENT_AUDIT_TABLE[
        "collector_sequence"
    ].to_numpy(dtype=np.int64)
)


first_update_id_differences = np.diff(
    first_update_ids
)

final_update_id_differences = np.diff(
    final_update_ids
)


first_update_id_reversal_count = int(
    np.sum(
        first_update_id_differences <= 0
    )
)

final_update_id_reversal_count = int(
    np.sum(
        final_update_id_differences <= 0
    )
)


snapshot_successor_id = (
    SNAPSHOT_LAST_UPDATE_ID + 1
)

stale_before_snapshot_mask = (
    final_update_ids
    <= SNAPSHOT_LAST_UPDATE_ID
)

post_snapshot_candidate_positions = np.flatnonzero(
    final_update_ids
    > SNAPSHOT_LAST_UPDATE_ID
)


if len(
    post_snapshot_candidate_positions
) == 0:
    first_post_snapshot_position = None
    first_post_snapshot_sequence = None
    first_post_snapshot_U = None
    first_post_snapshot_u = None
    snapshot_bridge_passed = False
    post_bridge_event_count = 0
    post_bridge_gap_count = None
    post_bridge_overlap_count = None
    post_bridge_exact_continuity_count = None

else:
    first_post_snapshot_position = int(
        post_snapshot_candidate_positions[0]
    )

    first_post_snapshot_sequence = int(
        collector_sequences[
            first_post_snapshot_position
        ]
    )

    first_post_snapshot_U = int(
        first_update_ids[
            first_post_snapshot_position
        ]
    )

    first_post_snapshot_u = int(
        final_update_ids[
            first_post_snapshot_position
        ]
    )

    snapshot_bridge_passed = bool(
        first_post_snapshot_U
        <= snapshot_successor_id
        <= first_post_snapshot_u
    )

    post_bridge_first_ids = (
        first_update_ids[
            first_post_snapshot_position:
        ]
    )

    post_bridge_final_ids = (
        final_update_ids[
            first_post_snapshot_position:
        ]
    )

    post_bridge_event_count = int(
        len(post_bridge_final_ids)
    )

    if post_bridge_event_count <= 1:
        post_bridge_gap_count = 0
        post_bridge_overlap_count = 0
        post_bridge_exact_continuity_count = 0

    else:
        expected_first_ids = (
            post_bridge_final_ids[:-1]
            + 1
        )

        observed_next_first_ids = (
            post_bridge_first_ids[1:]
        )

        post_bridge_gap_count = int(
            np.sum(
                observed_next_first_ids
                > expected_first_ids
            )
        )

        post_bridge_overlap_count = int(
            np.sum(
                observed_next_first_ids
                < expected_first_ids
            )
        )

        post_bridge_exact_continuity_count = int(
            np.sum(
                observed_next_first_ids
                == expected_first_ids
            )
        )


stale_event_count = int(
    stale_before_snapshot_mask.sum()
)

events_before_bridge_count = (
    first_post_snapshot_position
    if first_post_snapshot_position
    is not None
    else len(
        RAW_DEPTH_EVENT_AUDIT_TABLE
    )
)

all_events_before_bridge_stale = bool(
    first_post_snapshot_position
    is not None
    and stale_before_snapshot_mask[
        :first_post_snapshot_position
    ].all()
)

bridge_candidate_mask = (
    first_update_ids
    <= snapshot_successor_id
) & (
    final_update_ids
    >= snapshot_successor_id
)

bridge_candidate_count = int(
    bridge_candidate_mask.sum()
)


DEPTH_UPDATE_ID_AUDIT = pd.DataFrame(
    [
        {
            "depth_event_count": int(
                len(
                    RAW_DEPTH_EVENT_AUDIT_TABLE
                )
            ),
            "first_observed_U": int(
                first_update_ids[0]
            ),
            "first_observed_u": int(
                final_update_ids[0]
            ),
            "last_observed_U": int(
                first_update_ids[-1]
            ),
            "last_observed_u": int(
                final_update_ids[-1]
            ),
            "first_update_id_reversal_count": (
                first_update_id_reversal_count
            ),
            "final_update_id_reversal_count": (
                final_update_id_reversal_count
            ),
            "snapshot_last_update_id": (
                SNAPSHOT_LAST_UPDATE_ID
            ),
            "snapshot_successor_id": (
                snapshot_successor_id
            ),
            "stale_event_count": (
                stale_event_count
            ),
            "bridge_candidate_count": (
                bridge_candidate_count
            ),
            "first_post_snapshot_position": (
                first_post_snapshot_position
            ),
            "first_post_snapshot_collector_sequence": (
                first_post_snapshot_sequence
            ),
            "first_post_snapshot_U": (
                first_post_snapshot_U
            ),
            "first_post_snapshot_u": (
                first_post_snapshot_u
            ),
            "snapshot_bridge_passed": (
                snapshot_bridge_passed
            ),
            "post_bridge_event_count": (
                post_bridge_event_count
            ),
            "post_bridge_gap_count": (
                post_bridge_gap_count
            ),
            "post_bridge_overlap_count": (
                post_bridge_overlap_count
            ),
            "post_bridge_exact_continuity_count": (
                post_bridge_exact_continuity_count
            ),
        }
    ]
)


# ============================================================
# SNAPSHOT TIMING AND BUFFER RECONCILIATION
# ============================================================

if SNAPSHOT_RESPONSE_RECEIVED_TIME_NS is not None:
    observed_buffered_depth_count = int(
        RAW_DEPTH_EVENT_AUDIT_TABLE[
            "local_receipt_time_ns"
        ].le(
            SNAPSHOT_RESPONSE_RECEIVED_TIME_NS
        ).sum()
    )

else:
    observed_buffered_depth_count = None


if (
    SNAPSHOT_REQUEST_STARTED_TIME_NS is not None
    and SNAPSHOT_RESPONSE_RECEIVED_TIME_NS
    is not None
):
    snapshot_request_duration_ms_observed = (
        SNAPSHOT_RESPONSE_RECEIVED_TIME_NS
        - SNAPSHOT_REQUEST_STARTED_TIME_NS
    ) / 1_000_000.0

else:
    snapshot_request_duration_ms_observed = None


declared_snapshot_request_duration_ms = (
    SNAPSHOT_DOCUMENT.get(
        "request_duration_ms"
    )
)


SNAPSHOT_BUFFER_RECONCILIATION = pd.DataFrame(
    [
        {
            "snapshot_last_update_id": (
                SNAPSHOT_LAST_UPDATE_ID
            ),
            "contract_snapshot_last_update_id": (
                CONTRACT_SNAPSHOT_LAST_UPDATE_ID
            ),
            "snapshot_session_id": (
                SNAPSHOT_SESSION_ID
            ),
            "response_received_time_ns": (
                SNAPSHOT_RESPONSE_RECEIVED_TIME_NS
            ),
            "declared_buffered_depth_count": (
                DECLARED_BUFFERED_DEPTH_COUNT
            ),
            "observed_depth_received_by_response": (
                observed_buffered_depth_count
            ),
            "buffer_count_matches": (
                bool(
                    DECLARED_BUFFERED_DEPTH_COUNT
                    == observed_buffered_depth_count
                )
                if (
                    DECLARED_BUFFERED_DEPTH_COUNT
                    is not None
                    and observed_buffered_depth_count
                    is not None
                )
                else None
            ),
            "declared_request_duration_ms": (
                declared_snapshot_request_duration_ms
            ),
            "observed_request_duration_ms": (
                snapshot_request_duration_ms_observed
            ),
        }
    ]
)


# ============================================================
# PARTITION RECONCILIATION
# ============================================================

expected_depth_partition_counts = (
    FROZEN_PARTITION_BOUNDARIES[
        [
            "partition",
            "depth_row_count",
        ]
    ]
    .rename(
        columns={
            "depth_row_count": (
                "expected_depth_count"
            )
        }
    )
)


observed_depth_partition_counts = (
    RAW_DEPTH_EVENT_AUDIT_TABLE.groupby(
        "partition",
        observed=True,
        dropna=False,
    )
    .size()
    .rename(
        "observed_depth_count"
    )
    .reset_index()
)


DEPTH_PARTITION_RECONCILIATION = (
    expected_depth_partition_counts.merge(
        observed_depth_partition_counts,
        on="partition",
        how="left",
        validate="one_to_one",
    )
)


DEPTH_PARTITION_RECONCILIATION[
    "expected_depth_count"
] = (
    DEPTH_PARTITION_RECONCILIATION[
        "expected_depth_count"
    ].astype("int64")
)

DEPTH_PARTITION_RECONCILIATION[
    "observed_depth_count"
] = (
    DEPTH_PARTITION_RECONCILIATION[
        "observed_depth_count"
    ]
    .fillna(0)
    .astype("int64")
)

DEPTH_PARTITION_RECONCILIATION[
    "matches"
] = (
    DEPTH_PARTITION_RECONCILIATION[
        "expected_depth_count"
    ]
    == DEPTH_PARTITION_RECONCILIATION[
        "observed_depth_count"
    ]
)


# ============================================================
# EVENT ACTIVITY SUMMARY
# ============================================================

DEPTH_EVENT_ACTIVITY_SUMMARY = pd.DataFrame(
    [
        {
            "depth_event_count": int(
                len(
                    RAW_DEPTH_EVENT_AUDIT_TABLE
                )
            ),
            "bid_only_event_count": int(
                (
                    RAW_DEPTH_EVENT_AUDIT_TABLE[
                        "bid_update_count"
                    ].gt(0)
                    & RAW_DEPTH_EVENT_AUDIT_TABLE[
                        "ask_update_count"
                    ].eq(0)
                ).sum()
            ),
            "ask_only_event_count": int(
                (
                    RAW_DEPTH_EVENT_AUDIT_TABLE[
                        "bid_update_count"
                    ].eq(0)
                    & RAW_DEPTH_EVENT_AUDIT_TABLE[
                        "ask_update_count"
                    ].gt(0)
                ).sum()
            ),
            "both_side_event_count": int(
                (
                    RAW_DEPTH_EVENT_AUDIT_TABLE[
                        "bid_update_count"
                    ].gt(0)
                    & RAW_DEPTH_EVENT_AUDIT_TABLE[
                        "ask_update_count"
                    ].gt(0)
                ).sum()
            ),
            "both_sides_empty_count": int(
                RAW_DEPTH_EVENT_AUDIT_TABLE[
                    "both_sides_empty"
                ].sum()
            ),
            "total_bid_updates": int(
                RAW_DEPTH_EVENT_AUDIT_TABLE[
                    "bid_update_count"
                ].sum()
            ),
            "total_ask_updates": int(
                RAW_DEPTH_EVENT_AUDIT_TABLE[
                    "ask_update_count"
                ].sum()
            ),
            "zero_quantity_update_count": int(
                RAW_DEPTH_EVENT_AUDIT_TABLE[
                    "zero_quantity_update_count"
                ].sum()
            ),
            "positive_quantity_update_count": int(
                RAW_DEPTH_EVENT_AUDIT_TABLE[
                    "positive_quantity_update_count"
                ].sum()
            ),
            "invalid_level_count": int(
                RAW_DEPTH_EVENT_AUDIT_TABLE[
                    "invalid_level_count"
                ].sum()
            ),
            "duplicate_price_count": int(
                RAW_DEPTH_EVENT_AUDIT_TABLE[
                    "duplicate_price_count"
                ].sum()
            ),
        }
    ]
)


# ============================================================
# ACCEPTANCE GATES
# ============================================================

payload_complete = bool(
    DEPTH_PAYLOAD_RECONCILIATION[
        "collector_present_count"
    ].eq(
        DEPTH_PAYLOAD_RECONCILIATION[
            "row_count"
        ]
    ).all()
    and DEPTH_PAYLOAD_RECONCILIATION[
        "payload_present_count"
    ].eq(
        DEPTH_PAYLOAD_RECONCILIATION[
            "row_count"
        ]
    ).all()
)

payload_consistent = bool(
    DEPTH_PAYLOAD_RECONCILIATION[
        "mismatch_count"
    ].eq(0).all()
)

depth_levels_valid = bool(
    DEPTH_LEVEL_PROFILE[
        "invalid_level_count"
    ].eq(0).all()
)

depth_prices_unique_within_event = bool(
    DEPTH_LEVEL_PROFILE[
        "duplicate_price_count"
    ].eq(0).all()
)

depth_update_ranges_valid = bool(
    RAW_DEPTH_EVENT_AUDIT_TABLE[
        "update_range_valid"
    ].all()
)

depth_event_activity_valid = bool(
    ~RAW_DEPTH_EVENT_AUDIT_TABLE[
        "both_sides_empty"
    ].any()
)

depth_partitions_match = bool(
    DEPTH_PARTITION_RECONCILIATION[
        "matches"
    ].all()
)

snapshot_identity_matches = bool(
    SNAPSHOT_LAST_UPDATE_ID
    == CONTRACT_SNAPSHOT_LAST_UPDATE_ID
)

snapshot_session_matches = bool(
    str(SNAPSHOT_SESSION_ID)
    == str(
        RAW_DEPTH_EVENT_AUDIT_TABLE[
            "connection_session_id"
        ].iloc[0]
    )
)

post_bridge_continuity_passed = bool(
    snapshot_bridge_passed
    and post_bridge_gap_count == 0
    and post_bridge_overlap_count == 0
)

buffer_count_gate_passed = bool(
    DECLARED_BUFFERED_DEPTH_COUNT
    is None
    or observed_buffered_depth_count
    is None
    or DECLARED_BUFFERED_DEPTH_COUNT
    == observed_buffered_depth_count
)

critical_depth_rejections = (
    DEPTH_REJECTION_CANDIDATES
)


DEPTH_SEMANTIC_GATES = pd.DataFrame(
    [
        {
            "gate": (
                "depth_row_count_preserved"
            ),
            "passed": bool(
                len(
                    RAW_DEPTH_EVENT_AUDIT_TABLE
                )
                == EXPECTED_DEPTH_ROWS
            ),
            "evidence": (
                f"observed="
                f"{len(RAW_DEPTH_EVENT_AUDIT_TABLE)}; "
                f"expected={EXPECTED_DEPTH_ROWS}"
            ),
        },
        {
            "gate": (
                "collector_and_payload_fields_complete"
            ),
            "passed": payload_complete,
            "evidence": (
                f"fields="
                f"{len(DEPTH_PAYLOAD_RECONCILIATION)}"
            ),
        },
        {
            "gate": (
                "collector_and_payload_values_match"
            ),
            "passed": payload_consistent,
            "evidence": (
                f"mismatches="
                f"{int(DEPTH_PAYLOAD_RECONCILIATION['mismatch_count'].sum())}"
            ),
        },
        {
            "gate": (
                "depth_update_ranges_valid"
            ),
            "passed": depth_update_ranges_valid,
            "evidence": (
                f"invalid="
                f"{int((~RAW_DEPTH_EVENT_AUDIT_TABLE['update_range_valid']).sum())}"
            ),
        },
        {
            "gate": (
                "depth_levels_structurally_valid"
            ),
            "passed": depth_levels_valid,
            "evidence": (
                f"invalid_levels="
                f"{int(DEPTH_LEVEL_PROFILE['invalid_level_count'].sum())}"
            ),
        },
        {
            "gate": (
                "depth_prices_unique_within_events"
            ),
            "passed": (
                depth_prices_unique_within_event
            ),
            "evidence": (
                f"duplicate_prices="
                f"{int(DEPTH_LEVEL_PROFILE['duplicate_price_count'].sum())}"
            ),
        },
        {
            "gate": (
                "depth_events_have_activity"
            ),
            "passed": (
                depth_event_activity_valid
            ),
            "evidence": (
                f"both_sides_empty="
                f"{int(RAW_DEPTH_EVENT_AUDIT_TABLE['both_sides_empty'].sum())}"
            ),
        },
        {
            "gate": (
                "snapshot_identity_matches_contract"
            ),
            "passed": (
                snapshot_identity_matches
            ),
            "evidence": (
                f"snapshot="
                f"{SNAPSHOT_LAST_UPDATE_ID}; "
                f"contract="
                f"{CONTRACT_SNAPSHOT_LAST_UPDATE_ID}"
            ),
        },
        {
            "gate": (
                "snapshot_session_matches_depth_stream"
            ),
            "passed": (
                snapshot_session_matches
            ),
            "evidence": (
                f"snapshot_session="
                f"{SNAPSHOT_SESSION_ID}"
            ),
        },
        {
            "gate": (
                "first_post_snapshot_event_bridges"
            ),
            "passed": (
                snapshot_bridge_passed
            ),
            "evidence": (
                f"lastUpdateId="
                f"{SNAPSHOT_LAST_UPDATE_ID}; "
                f"U={first_post_snapshot_U}; "
                f"u={first_post_snapshot_u}"
            ),
        },
        {
            "gate": (
                "all_pre_bridge_events_are_stale"
            ),
            "passed": (
                all_events_before_bridge_stale
            ),
            "evidence": (
                f"events_before_bridge="
                f"{events_before_bridge_count}; "
                f"stale_events="
                f"{stale_event_count}"
            ),
        },
        {
            "gate": (
                "post_bridge_update_ids_contiguous"
            ),
            "passed": (
                post_bridge_continuity_passed
            ),
            "evidence": (
                f"events="
                f"{post_bridge_event_count}; "
                f"gaps="
                f"{post_bridge_gap_count}; "
                f"overlaps="
                f"{post_bridge_overlap_count}"
            ),
        },
        {
            "gate": (
                "snapshot_buffer_count_reconciles"
            ),
            "passed": (
                buffer_count_gate_passed
            ),
            "evidence": (
                f"declared="
                f"{DECLARED_BUFFERED_DEPTH_COUNT}; "
                f"observed="
                f"{observed_buffered_depth_count}"
            ),
        },
        {
            "gate": (
                "depth_partition_counts_match"
            ),
            "passed": (
                depth_partitions_match
            ),
            "evidence": (
                f"partitions="
                f"{len(DEPTH_PARTITION_RECONCILIATION)}"
            ),
        },
        {
            "gate": (
                "no_critical_depth_rejections"
            ),
            "passed": bool(
                critical_depth_rejections.empty
            ),
            "evidence": (
                f"critical_rejections="
                f"{len(critical_depth_rejections)}"
            ),
        },
    ]
)


# ============================================================
# DISPLAY AND ENFORCE
# ============================================================

display(
    DEPTH_PAYLOAD_RECONCILIATION[
        [
            "canonical_field",
            "collector_path",
            "payload_path",
            "row_count",
            "collector_present_count",
            "payload_present_count",
            "match_count",
            "mismatch_count",
            "status",
        ]
    ]
)

display(DEPTH_EVENT_ACTIVITY_SUMMARY)
display(DEPTH_LEVEL_PROFILE)
display(DEPTH_UPDATE_ID_AUDIT)
display(SNAPSHOT_BUFFER_RECONCILIATION)
display(DEPTH_PARTITION_RECONCILIATION)
display(DEPTH_SEMANTIC_GATES)


if not DEPTH_REJECTION_CANDIDATES.empty:
    display(DEPTH_REJECTION_CANDIDATES)


failed_depth_semantic_gates = (
    DEPTH_SEMANTIC_GATES.loc[
        ~DEPTH_SEMANTIC_GATES[
            "passed"
        ]
    ]
)


if not failed_depth_semantic_gates.empty:
    failure_text = ", ".join(
        f"{row.gate}: {row.evidence}"
        for row in (
            failed_depth_semantic_gates
            .itertuples(index=False)
        )
    )

    raise RuntimeError(
        "Depth payload and snapshot-bridge audit failed: "
        + failure_text
    )


print("Depth payload and snapshot-bridge audit: PASS")
print(
    "Depth events audited: "
    f"{len(RAW_DEPTH_EVENT_AUDIT_TABLE):,}"
)
print(
    "Snapshot lastUpdateId: "
    f"{SNAPSHOT_LAST_UPDATE_ID:,}"
)
print(
    "Stale pre-bridge depth events: "
    f"{stale_event_count:,}"
)
print(
    "First applied depth event: "
    f"sequence={first_post_snapshot_sequence:,}; "
    f"U={first_post_snapshot_U:,}; "
    f"u={first_post_snapshot_u:,}"
)
print(
    "Post-bridge depth events: "
    f"{post_bridge_event_count:,}"
)
print(
    "Post-bridge update-ID gaps: "
    f"{post_bridge_gap_count:,}"
)
print(
    "Post-bridge update-ID overlaps: "
    f"{post_bridge_overlap_count:,}"
)
print(
    "Depth update levels audited: "
    f"{int(DEPTH_LEVEL_PROFILE['level_count'].sum()):,}"
)
print(
    "Zero-quantity delete updates: "
    f"{int(DEPTH_LEVEL_PROFILE['zero_quantity_count'].sum()):,}"
)

,canonical_field,collector_path,payload_path,row_count,collector_present_count,payload_present_count,match_count,mismatch_count,status
0,event_type,$.event_type,$.raw_message.data.e,35994,35994,35994,35994,0,PASS
1,exchange_event_time_ms,$.exchange_event_time_raw,$.raw_message.data.E,35994,35994,35994,35994,0,PASS
2,symbol,$.symbol,$.raw_message.data.s,35994,35994,35994,35994,0,PASS
3,first_update_id,$.first_update_id,$.raw_message.data.U,35994,35994,35994,35994,0,PASS
4,final_update_id,$.final_update_id,$.raw_message.data.u,35994,35994,35994,35994,0,PASS
5,bid_updates,$.bid_changes_raw,$.raw_message.data.b,35994,35994,35994,35994,0,PASS
6,ask_updates,$.ask_changes_raw,$.raw_message.data.a,35994,35994,35994,35994,0,PASS


,depth_event_count,bid_only_event_count,ask_only_event_count,both_side_event_count,both_sides_empty_count,total_bid_updates,total_ask_updates,zero_quantity_update_count,positive_quantity_update_count,invalid_level_count,duplicate_price_count
0,35994,3252,0,32742,0,332920,277873,219617,391176,0,0


,side,event_array_count,empty_array_count,level_count,valid_level_count,invalid_level_count,zero_quantity_count,positive_quantity_count,duplicate_price_count,minimum_price,maximum_price,minimum_quantity,maximum_quantity,minimum_price_decimal_places,maximum_price_decimal_places,minimum_quantity_decimal_places,maximum_quantity_decimal_places
0,BID,35994,0,332920,332920,0,109491,223429,0,31948.00000000,64011.89000000,0.00000000,1365.14128000,8,8,8,8
1,ASK,35994,3252,277873,277873,0,110126,167747,0,63805.32000000,125000.00000000,0.00000000,125.21375000,8,8,8,8


,depth_event_count,first_observed_U,first_observed_u,last_observed_U,last_observed_u,first_update_id_reversal_count,final_update_id_reversal_count,snapshot_last_update_id,snapshot_successor_id,stale_event_count,bridge_candidate_count,first_post_snapshot_position,first_post_snapshot_collector_sequence,first_post_snapshot_U,first_post_snapshot_u,snapshot_bridge_passed,post_bridge_event_count,post_bridge_gap_count,post_bridge_overlap_count,post_bridge_exact_continuity_count
0,35994,97233590081,97233590085,97234812158,97234812218,0,0,97233590166,97233590167,9,1,9,10,97233590167,97233590170,True,35985,0,0,35984


,snapshot_last_update_id,contract_snapshot_last_update_id,snapshot_session_id,response_received_time_ns,declared_buffered_depth_count,observed_depth_received_by_response,buffer_count_matches,declared_request_duration_ms,observed_request_duration_ms
0,97233590166,97233590166,c8b5bf127a7a44669514acfda8634107,1783665469341933600,None,19,None,1808.9281,1808.9281


,partition,expected_depth_count,observed_depth_count,matches
0,DEVELOPMENT,18019,18019,True
1,CALIBRATION,7204,7204,True
2,VALIDATION,5353,5353,True
3,ENGINEERING_HOLDOUT,5418,5418,True


,gate,passed,evidence
0,depth_row_count_preserved,True,observed=35994; expected=35994
1,collector_and_payload_fields_complete,True,fields=7
2,collector_and_payload_values_match,True,mismatches=0
3,depth_update_ranges_valid,True,invalid=0
4,depth_levels_structurally_valid,True,invalid_levels=0
5,depth_prices_unique_within_events,True,duplicate_prices=0
6,depth_events_have_activity,True,both_sides_empty=0
7,snapshot_identity_matches_contract,True,snapshot=97233590166; contract=97233590166
8,snapshot_session_matches_depth_stream,True,snapshot_session=c8b5bf127a7a44669514acfda8634107
9,first_post_snapshot_event_bridges,True,lastUpdateId=97233590166; U=97233590167; u=972...


Depth payload and snapshot-bridge audit: PASS
Depth events audited: 35,994
Snapshot lastUpdateId: 97,233,590,166
Stale pre-bridge depth events: 9
First applied depth event: sequence=10; U=97,233,590,167; u=97,233,590,170
Post-bridge depth events: 35,985
Post-bridge update-ID gaps: 0
Post-bridge update-ID overlaps: 0
Depth update levels audited: 610,793
Zero-quantity delete updates: 219,617


In [39]:
# ============================================================
# COLLECTOR METADATA, RUN MANIFEST, AND REJECTION RECONCILIATION
# ============================================================

from collections import defaultdict


# ============================================================
# RECURSIVE DOCUMENT INSPECTION
# ============================================================

def walk_document_leaves(
    value,
    path: str = "$",
):
    """
    Yield scalar leaves from a nested JSON-compatible document.
    """
    decoded = decode_json_container(value)

    if isinstance(decoded, dict):
        for key, item in decoded.items():
            child_path = (
                f"{path}.{key}"
            )

            yield from walk_document_leaves(
                item,
                child_path,
            )

        return

    if isinstance(decoded, list):
        for position, item in enumerate(
            decoded
        ):
            child_path = (
                f"{path}[{position}]"
            )

            yield from walk_document_leaves(
                item,
                child_path,
            )

        return

    yield {
        "path": path,
        "key": (
            path.rsplit(".", 1)[-1]
            if "." in path
            else path
        ),
        "value": decoded,
        "value_type": (
            type(decoded).__name__
            if decoded is not None
            else "null"
        ),
    }


def document_leaf_table(
    document_name: str,
    document: dict,
) -> pd.DataFrame:
    rows = []

    for leaf in walk_document_leaves(
        document
    ):
        rows.append(
            {
                "document": document_name,
                "path": leaf["path"],
                "key": leaf["key"],
                "value": leaf["value"],
                "value_type": (
                    leaf["value_type"]
                ),
            }
        )

    return pd.DataFrame(
        rows,
        columns=[
            "document",
            "path",
            "key",
            "value",
            "value_type",
        ],
    )


COLLECTOR_METADATA_LEAVES = (
    document_leaf_table(
        document_name=(
            "COLLECTOR_METADATA"
        ),
        document=(
            COLLECTOR_METADATA_DOCUMENT
        ),
    )
)

RUN_MANIFEST_LEAVES = (
    document_leaf_table(
        document_name="RUN_MANIFEST",
        document=RUN_MANIFEST_DOCUMENT,
    )
)

METADATA_DOCUMENT_LEAVES = pd.concat(
    [
        COLLECTOR_METADATA_LEAVES,
        RUN_MANIFEST_LEAVES,
    ],
    ignore_index=True,
)


# ============================================================
# DECLARATION NORMALIZERS
# ============================================================

def normalize_metadata_integer(
    value,
):
    return parse_integer_value(value)


def normalize_metadata_float(
    value,
):
    if isinstance(value, bool):
        return None

    try:
        parsed = float(value)

    except (
        TypeError,
        ValueError,
        OverflowError,
    ):
        return None

    if not np.isfinite(parsed):
        return None

    return parsed


def normalize_metadata_string(
    value,
):
    if value is None:
        return None

    normalized = str(value).strip()

    if not normalized:
        return None

    return normalized


def normalize_metadata_status(
    value,
):
    normalized = normalize_metadata_string(
        value
    )

    if normalized is None:
        return None

    return normalized.upper()


def unique_normalized_values(
    values: list,
) -> list:
    unique_values = []
    seen = set()

    for value in values:
        key = json.dumps(
            value,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        if key in seen:
            continue

        seen.add(key)
        unique_values.append(value)

    return unique_values


# ============================================================
# OBSERVED AUTHORITATIVE VALUES
# ============================================================

observed_session_values = (
    RAW_EVENT_ORDER_INDEX[
        "connection_session_id"
    ]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

if len(observed_session_values) != 1:
    raise RuntimeError(
        "Expected exactly one observed connection session."
    )

OBSERVED_CONNECTION_SESSION_ID = (
    observed_session_values[0]
)

OBSERVED_SYMBOL = "BTCUSDT"

OBSERVED_TRADE_COUNT = int(
    len(RAW_TRADE_AUDIT_TABLE)
)

OBSERVED_DEPTH_COUNT = int(
    len(RAW_DEPTH_EVENT_AUDIT_TABLE)
)

OBSERVED_TOTAL_MESSAGE_COUNT = int(
    len(RAW_EVENT_ORDER_INDEX)
)

OBSERVED_FIRST_COLLECTOR_SEQUENCE = int(
    RAW_EVENT_ORDER_INDEX[
        "collector_sequence"
    ].min()
)

OBSERVED_LAST_COLLECTOR_SEQUENCE = int(
    RAW_EVENT_ORDER_INDEX[
        "collector_sequence"
    ].max()
)

OBSERVED_EVENT_COVERAGE_SECONDS = float(
    (
        RAW_EVENT_ORDER_INDEX[
            "local_receipt_time_utc"
        ].max()
        - RAW_EVENT_ORDER_INDEX[
            "local_receipt_time_utc"
        ].min()
    ).total_seconds()
)


# ============================================================
# SEMANTIC DECLARATION SPECIFICATION
# ============================================================

METADATA_DECLARATION_SPECS = {
    "connection_session_id": {
        "candidate_keys": {
            "connection_session_id",
            "collector_session_id",
            "session_id",
        },
        "normalizer": (
            normalize_metadata_string
        ),
        "expected_value": (
            OBSERVED_CONNECTION_SESSION_ID
        ),
        "comparison": "EXACT",
        "required_anywhere": True,
    },
    "symbol": {
        "candidate_keys": {
            "symbol",
            "market_symbol",
            "trading_symbol",
        },
        "normalizer": (
            normalize_metadata_string
        ),
        "expected_value": OBSERVED_SYMBOL,
        "comparison": "EXACT",
        "required_anywhere": True,
    },
    "source_run_prefix": {
        "candidate_keys": {
            "source_run_prefix",
            "run_prefix",
            "collection_run_prefix",
        },
        "normalizer": (
            normalize_metadata_string
        ),
        "expected_value": (
            SOURCE_RUN_PREFIX
        ),
        "comparison": "EXACT",
        "required_anywhere": False,
    },
    "trade_count": {
        "candidate_keys": {
            "trade_count",
            "trade_message_count",
            "trade_messages",
            "trade_row_count",
            "trades_count",
            "total_trade_messages",
        },
        "normalizer": (
            normalize_metadata_integer
        ),
        "expected_value": (
            OBSERVED_TRADE_COUNT
        ),
        "comparison": "EXACT",
        "required_anywhere": True,
    },
    "depth_count": {
        "candidate_keys": {
            "depth_count",
            "depth_message_count",
            "depth_messages",
            "depth_row_count",
            "total_depth_messages",
            "depth_update_count",
        },
        "normalizer": (
            normalize_metadata_integer
        ),
        "expected_value": (
            OBSERVED_DEPTH_COUNT
        ),
        "comparison": "EXACT",
        "required_anywhere": True,
    },
    "total_message_count": {
        "candidate_keys": {
            "total_message_count",
            "total_messages",
            "combined_message_count",
            "combined_row_count",
            "total_record_count",
            "collector_message_count",
        },
        "normalizer": (
            normalize_metadata_integer
        ),
        "expected_value": (
            OBSERVED_TOTAL_MESSAGE_COUNT
        ),
        "comparison": "EXACT",
        "required_anywhere": True,
    },
    "first_collector_sequence": {
        "candidate_keys": {
            "first_collector_sequence",
            "collector_sequence_first",
            "minimum_collector_sequence",
            "min_collector_sequence",
        },
        "normalizer": (
            normalize_metadata_integer
        ),
        "expected_value": (
            OBSERVED_FIRST_COLLECTOR_SEQUENCE
        ),
        "comparison": "EXACT",
        "required_anywhere": False,
    },
    "last_collector_sequence": {
        "candidate_keys": {
            "last_collector_sequence",
            "collector_sequence_last",
            "maximum_collector_sequence",
            "max_collector_sequence",
            "final_collector_sequence",
        },
        "normalizer": (
            normalize_metadata_integer
        ),
        "expected_value": (
            OBSERVED_LAST_COLLECTOR_SEQUENCE
        ),
        "comparison": "EXACT",
        "required_anywhere": False,
    },
    "snapshot_last_update_id": {
        "candidate_keys": {
            "snapshot_last_update_id",
            "rest_snapshot_last_update_id",
            "snapshot_lastupdateid",
        },
        "normalizer": (
            normalize_metadata_integer
        ),
        "expected_value": (
            SNAPSHOT_LAST_UPDATE_ID
        ),
        "comparison": "EXACT",
        "required_anywhere": False,
    },
    "actual_duration_seconds": {
        "candidate_keys": {
            "actual_duration_seconds",
            "elapsed_seconds",
            "collection_duration_seconds",
        },
        "normalizer": (
            normalize_metadata_float
        ),
        "expected_value": (
            OBSERVED_EVENT_COVERAGE_SECONDS
        ),
        "comparison": "DURATION_LOWER_BOUND",
        "required_anywhere": True,
    },
    "collector_result_status": {
        "candidate_keys": {
            "collector_result_status",
            "collection_result_status",
            "run_result_status",
        },
        "normalizer": (
            normalize_metadata_status
        ),
        "expected_value": None,
        "comparison": "ACCEPTED_STATUS",
        "required_anywhere": True,
    },
}


ACCEPTED_COLLECTOR_STATUSES = {
    "SUCCESS",
    "PASS",
    "PASSED",
    "COMPLETE",
    "COMPLETED",
    "OK",
}


# ============================================================
# FIND AND RECONCILE DECLARATIONS
# ============================================================

metadata_declaration_rows = []


for semantic_field, field_spec in (
    METADATA_DECLARATION_SPECS.items()
):
    candidate_keys = {
        key.lower()
        for key in field_spec[
            "candidate_keys"
        ]
    }

    for document_name, leaf_table in (
        (
            "COLLECTOR_METADATA",
            COLLECTOR_METADATA_LEAVES,
        ),
        (
            "RUN_MANIFEST",
            RUN_MANIFEST_LEAVES,
        ),
    ):
        matches = leaf_table.loc[
            leaf_table[
                "key"
            ].astype(str).str.lower().isin(
                candidate_keys
            )
        ]

        normalized_occurrences = []

        for match_row in matches.itertuples(
            index=False
        ):
            normalized_value = (
                field_spec["normalizer"](
                    match_row.value
                )
            )

            normalized_occurrences.append(
                {
                    "path": match_row.path,
                    "raw_value": (
                        match_row.value
                    ),
                    "normalized_value": (
                        normalized_value
                    ),
                }
            )

        valid_normalized_values = [
            occurrence[
                "normalized_value"
            ]
            for occurrence in (
                normalized_occurrences
            )
            if occurrence[
                "normalized_value"
            ] is not None
        ]

        unique_values = (
            unique_normalized_values(
                valid_normalized_values
            )
        )

        occurrence_count = int(
            len(normalized_occurrences)
        )

        normalized_value_count = int(
            len(valid_normalized_values)
        )

        conflict = bool(
            len(unique_values) > 1
        )

        declared_value = (
            unique_values[0]
            if len(unique_values) == 1
            else None
        )

        expected_value = field_spec[
            "expected_value"
        ]

        comparison = field_spec[
            "comparison"
        ]

        if occurrence_count == 0:
            comparison_passed = True
            status = "NOT_DECLARED"

        elif normalized_value_count == 0:
            comparison_passed = False
            status = "FAIL_UNPARSABLE"

        elif conflict:
            comparison_passed = False
            status = "FAIL_CONFLICT"

        elif comparison == "EXACT":
            comparison_passed = bool(
                declared_value
                == expected_value
            )

            status = (
                "PASS"
                if comparison_passed
                else "FAIL_MISMATCH"
            )

        elif comparison == (
            "DURATION_LOWER_BOUND"
        ):
            duration_tolerance_seconds = 5.0

            comparison_passed = bool(
                declared_value is not None
                and declared_value > 0
                and (
                    declared_value
                    + duration_tolerance_seconds
                    >= expected_value
                )
            )

            status = (
                "PASS"
                if comparison_passed
                else "FAIL_DURATION"
            )

        elif comparison == (
            "ACCEPTED_STATUS"
        ):
            comparison_passed = bool(
                declared_value
                in ACCEPTED_COLLECTOR_STATUSES
            )

            status = (
                "PASS"
                if comparison_passed
                else "FAIL_STATUS"
            )

        else:
            raise RuntimeError(
                "Unsupported metadata comparison: "
                f"{comparison}"
            )

        metadata_declaration_rows.append(
            {
                "document": document_name,
                "semantic_field": (
                    semantic_field
                ),
                "required_anywhere": bool(
                    field_spec[
                        "required_anywhere"
                    ]
                ),
                "comparison": comparison,
                "occurrence_count": (
                    occurrence_count
                ),
                "normalized_value_count": (
                    normalized_value_count
                ),
                "unique_value_count": int(
                    len(unique_values)
                ),
                "declared_value": (
                    declared_value
                ),
                "expected_value": (
                    expected_value
                ),
                "comparison_passed": (
                    comparison_passed
                ),
                "status": status,
                "paths": (
                    " | ".join(
                        occurrence["path"]
                        for occurrence in (
                            normalized_occurrences
                        )
                    )
                ),
            }
        )


METADATA_DECLARATION_AUDIT = pd.DataFrame(
    metadata_declaration_rows
)


# ============================================================
# CROSS-DOCUMENT CONSISTENCY
# ============================================================

cross_document_rows = []


for semantic_field, field_rows in (
    METADATA_DECLARATION_AUDIT.groupby(
        "semantic_field",
        sort=False,
    )
):
    declared_values = (
        field_rows.loc[
            ~field_rows[
                "status"
            ].eq("NOT_DECLARED")
            & field_rows[
                "declared_value"
            ].notna(),
            "declared_value",
        ]
        .tolist()
    )

    unique_values = unique_normalized_values(
        declared_values
    )

    required_anywhere = bool(
        field_rows[
            "required_anywhere"
        ].iloc[0]
    )

    declared_anywhere = bool(
        len(declared_values) > 0
    )

    cross_document_consistent = bool(
        len(unique_values) <= 1
    )

    cross_document_rows.append(
        {
            "semantic_field": (
                semantic_field
            ),
            "required_anywhere": (
                required_anywhere
            ),
            "declared_anywhere": (
                declared_anywhere
            ),
            "declaring_document_count": int(
                field_rows[
                    "status"
                ].ne("NOT_DECLARED").sum()
            ),
            "unique_declared_value_count": int(
                len(unique_values)
            ),
            "cross_document_consistent": (
                cross_document_consistent
            ),
            "declared_value": (
                unique_values[0]
                if len(unique_values) == 1
                else None
            ),
        }
    )


METADATA_CROSS_DOCUMENT_AUDIT = (
    pd.DataFrame(
        cross_document_rows
    )
)


# ============================================================
# TOP-LEVEL DOCUMENT SUMMARY
# ============================================================

METADATA_DOCUMENT_SUMMARY = pd.DataFrame(
    [
        {
            "document": (
                "COLLECTOR_METADATA"
            ),
            "top_level_type": type(
                COLLECTOR_METADATA_DOCUMENT
            ).__name__,
            "top_level_key_count": len(
                COLLECTOR_METADATA_DOCUMENT
            ),
            "leaf_count": len(
                COLLECTOR_METADATA_LEAVES
            ),
            "top_level_keys": (
                " | ".join(
                    sorted(
                        str(key)
                        for key in (
                            COLLECTOR_METADATA_DOCUMENT
                            .keys()
                        )
                    )
                )
            ),
        },
        {
            "document": "RUN_MANIFEST",
            "top_level_type": type(
                RUN_MANIFEST_DOCUMENT
            ).__name__,
            "top_level_key_count": len(
                RUN_MANIFEST_DOCUMENT
            ),
            "leaf_count": len(
                RUN_MANIFEST_LEAVES
            ),
            "top_level_keys": (
                " | ".join(
                    sorted(
                        str(key)
                        for key in (
                            RUN_MANIFEST_DOCUMENT
                            .keys()
                        )
                    )
                )
            ),
        },
    ]
)


# ============================================================
# COMBINED REJECTION LEDGER
# ============================================================

combined_rejection_rows = []


if not TRADE_REJECTION_CANDIDATES.empty:
    for row in (
        TRADE_REJECTION_CANDIDATES
        .itertuples(index=False)
    ):
        combined_rejection_rows.append(
            {
                "stream": "TRADE_STREAM",
                "source_line_number": (
                    int(
                        row.source_line_number
                    )
                    if pd.notna(
                        row.source_line_number
                    )
                    else None
                ),
                "collector_sequence": (
                    int(
                        row.collector_sequence
                    )
                    if pd.notna(
                        row.collector_sequence
                    )
                    else None
                ),
                "record_identity": (
                    int(row.trade_id)
                    if pd.notna(row.trade_id)
                    else None
                ),
                "reason_code": (
                    row.reason_code
                ),
            }
        )


if not DEPTH_REJECTION_CANDIDATES.empty:
    for row in (
        DEPTH_REJECTION_CANDIDATES
        .itertuples(index=False)
    ):
        depth_identity = None

        if (
            pd.notna(row.first_update_id)
            and pd.notna(row.final_update_id)
        ):
            depth_identity = (
                f"{int(row.first_update_id)}:"
                f"{int(row.final_update_id)}"
            )

        combined_rejection_rows.append(
            {
                "stream": "DEPTH_STREAM",
                "source_line_number": (
                    int(
                        row.source_line_number
                    )
                    if pd.notna(
                        row.source_line_number
                    )
                    else None
                ),
                "collector_sequence": (
                    int(
                        row.collector_sequence
                    )
                    if pd.notna(
                        row.collector_sequence
                    )
                    else None
                ),
                "record_identity": (
                    depth_identity
                ),
                "reason_code": (
                    row.reason_code
                ),
            }
        )


RAW_REJECTION_LEDGER = pd.DataFrame(
    combined_rejection_rows,
    columns=[
        "stream",
        "source_line_number",
        "collector_sequence",
        "record_identity",
        "reason_code",
    ],
)


if RAW_REJECTION_LEDGER.empty:
    RAW_REJECTION_REASON_SUMMARY = (
        pd.DataFrame(
            columns=[
                "stream",
                "reason_code",
                "rejection_count",
            ]
        )
    )

else:
    RAW_REJECTION_REASON_SUMMARY = (
        RAW_REJECTION_LEDGER.groupby(
            [
                "stream",
                "reason_code",
            ],
            dropna=False,
            observed=True,
        )
        .size()
        .rename("rejection_count")
        .reset_index()
    )


# ============================================================
# ACCEPTANCE GATES
# ============================================================

metadata_documents_are_mappings = bool(
    isinstance(
        COLLECTOR_METADATA_DOCUMENT,
        dict,
    )
    and isinstance(
        RUN_MANIFEST_DOCUMENT,
        dict,
    )
)

all_declared_values_parse = bool(
    ~METADATA_DECLARATION_AUDIT[
        "status"
    ].eq("FAIL_UNPARSABLE").any()
)

no_within_document_conflicts = bool(
    ~METADATA_DECLARATION_AUDIT[
        "status"
    ].eq("FAIL_CONFLICT").any()
)

all_declared_values_match = bool(
    METADATA_DECLARATION_AUDIT[
        "comparison_passed"
    ].all()
)

required_declarations_present = bool(
    METADATA_CROSS_DOCUMENT_AUDIT.loc[
        METADATA_CROSS_DOCUMENT_AUDIT[
            "required_anywhere"
        ],
        "declared_anywhere",
    ].all()
)

cross_document_values_consistent = bool(
    METADATA_CROSS_DOCUMENT_AUDIT[
        "cross_document_consistent"
    ].all()
)

event_coverage_positive = bool(
    OBSERVED_EVENT_COVERAGE_SECONDS > 0
)

no_raw_rejections = bool(
    RAW_REJECTION_LEDGER.empty
)


METADATA_AND_REJECTION_GATES = pd.DataFrame(
    [
        {
            "gate": (
                "metadata_documents_are_mappings"
            ),
            "passed": (
                metadata_documents_are_mappings
            ),
            "evidence": (
                f"collector="
                f"{type(COLLECTOR_METADATA_DOCUMENT).__name__}; "
                f"manifest="
                f"{type(RUN_MANIFEST_DOCUMENT).__name__}"
            ),
        },
        {
            "gate": (
                "metadata_declarations_parse"
            ),
            "passed": (
                all_declared_values_parse
            ),
            "evidence": (
                f"unparsable="
                f"{int(METADATA_DECLARATION_AUDIT['status'].eq('FAIL_UNPARSABLE').sum())}"
            ),
        },
        {
            "gate": (
                "no_within_document_declaration_conflicts"
            ),
            "passed": (
                no_within_document_conflicts
            ),
            "evidence": (
                f"conflicts="
                f"{int(METADATA_DECLARATION_AUDIT['status'].eq('FAIL_CONFLICT').sum())}"
            ),
        },
        {
            "gate": (
                "declared_metadata_matches_observed"
            ),
            "passed": (
                all_declared_values_match
            ),
            "evidence": (
                f"failed_comparisons="
                f"{int((~METADATA_DECLARATION_AUDIT['comparison_passed']).sum())}"
            ),
        },
        {
            "gate": (
                "required_metadata_declared"
            ),
            "passed": (
                required_declarations_present
            ),
            "evidence": (
                f"required_missing="
                f"{int((METADATA_CROSS_DOCUMENT_AUDIT['required_anywhere'] & ~METADATA_CROSS_DOCUMENT_AUDIT['declared_anywhere']).sum())}"
            ),
        },
        {
            "gate": (
                "cross_document_metadata_consistent"
            ),
            "passed": (
                cross_document_values_consistent
            ),
            "evidence": (
                f"inconsistent="
                f"{int((~METADATA_CROSS_DOCUMENT_AUDIT['cross_document_consistent']).sum())}"
            ),
        },
        {
            "gate": (
                "observed_event_coverage_positive"
            ),
            "passed": (
                event_coverage_positive
            ),
            "evidence": (
                f"coverage_seconds="
                f"{OBSERVED_EVENT_COVERAGE_SECONDS:.9f}"
            ),
        },
        {
            "gate": (
                "no_raw_record_rejections"
            ),
            "passed": no_raw_rejections,
            "evidence": (
                f"rejections="
                f"{len(RAW_REJECTION_LEDGER)}"
            ),
        },
    ]
)


# ============================================================
# DISPLAY AND ENFORCE
# ============================================================

display(METADATA_DOCUMENT_SUMMARY)

display(
    METADATA_DECLARATION_AUDIT[
        [
            "document",
            "semantic_field",
            "occurrence_count",
            "unique_value_count",
            "declared_value",
            "expected_value",
            "status",
            "paths",
        ]
    ]
)

display(METADATA_CROSS_DOCUMENT_AUDIT)
display(RAW_REJECTION_REASON_SUMMARY)
display(METADATA_AND_REJECTION_GATES)


if not RAW_REJECTION_LEDGER.empty:
    display(RAW_REJECTION_LEDGER)


failed_metadata_gates = (
    METADATA_AND_REJECTION_GATES.loc[
        ~METADATA_AND_REJECTION_GATES[
            "passed"
        ]
    ]
)


if not failed_metadata_gates.empty:
    failure_text = ", ".join(
        f"{row.gate}: {row.evidence}"
        for row in (
            failed_metadata_gates
            .itertuples(index=False)
        )
    )

    raise RuntimeError(
        "Collector metadata and run-manifest audit failed: "
        + failure_text
    )


print("Collector metadata and run-manifest audit: PASS")
print(
    "Collector metadata leaves inspected: "
    f"{len(COLLECTOR_METADATA_LEAVES):,}"
)
print(
    "Run manifest leaves inspected: "
    f"{len(RUN_MANIFEST_LEAVES):,}"
)
print(
    "Observed trade records: "
    f"{OBSERVED_TRADE_COUNT:,}"
)
print(
    "Observed depth records: "
    f"{OBSERVED_DEPTH_COUNT:,}"
)
print(
    "Observed total messages: "
    f"{OBSERVED_TOTAL_MESSAGE_COUNT:,}"
)
print(
    "Observed event coverage: "
    f"{OBSERVED_EVENT_COVERAGE_SECONDS:.6f} seconds"
)
print(
    "Raw record rejections: "
    f"{len(RAW_REJECTION_LEDGER):,}"
)

,document,top_level_type,top_level_key_count,leaf_count,top_level_keys
0,COLLECTOR_METADATA,dict,24,31,actual_duration_seconds | collector_result_sta...
1,RUN_MANIFEST,dict,17,29,actual_duration_seconds | collector_result_sta...


,document,semantic_field,occurrence_count,unique_value_count,declared_value,expected_value,status,paths
0,COLLECTOR_METADATA,connection_session_id,1,1,c8b5bf127a7a44669514acfda8634107,c8b5bf127a7a44669514acfda8634107,PASS,$.connection_session_id
1,RUN_MANIFEST,connection_session_id,1,1,c8b5bf127a7a44669514acfda8634107,c8b5bf127a7a44669514acfda8634107,PASS,$.connection_session_id
2,COLLECTOR_METADATA,symbol,1,1,BTCUSDT,BTCUSDT,PASS,$.symbol
3,RUN_MANIFEST,symbol,1,1,BTCUSDT,BTCUSDT,PASS,$.symbol
4,COLLECTOR_METADATA,source_run_prefix,0,0,None,BTCUSDT_spot_20260710T063746Z_c8b5bf12,NOT_DECLARED,
5,RUN_MANIFEST,source_run_prefix,0,0,None,BTCUSDT_spot_20260710T063746Z_c8b5bf12,NOT_DECLARED,
6,COLLECTOR_METADATA,trade_count,1,1,67683,67683,PASS,$.message_counts.trade_messages
7,RUN_MANIFEST,trade_count,1,1,67683,67683,PASS,$.message_counts.trade_messages
8,COLLECTOR_METADATA,depth_count,1,1,35994,35994,PASS,$.message_counts.depth_messages
9,RUN_MANIFEST,depth_count,1,1,35994,35994,PASS,$.message_counts.depth_messages


,semantic_field,required_anywhere,declared_anywhere,declaring_document_count,unique_declared_value_count,cross_document_consistent,declared_value
0,connection_session_id,True,True,2,1,True,c8b5bf127a7a44669514acfda8634107
1,symbol,True,True,2,1,True,BTCUSDT
2,source_run_prefix,False,False,0,0,True,None
3,trade_count,True,True,2,1,True,67683
4,depth_count,True,True,2,1,True,35994
5,total_message_count,True,True,2,1,True,103677
6,first_collector_sequence,False,False,0,0,True,None
7,last_collector_sequence,False,False,0,0,True,None
8,snapshot_last_update_id,False,True,2,1,True,97233590166
9,actual_duration_seconds,True,True,2,1,True,3605.225331


,stream,reason_code,rejection_count


,gate,passed,evidence
0,metadata_documents_are_mappings,True,collector=dict; manifest=dict
1,metadata_declarations_parse,True,unparsable=0
2,no_within_document_declaration_conflicts,True,conflicts=0
3,declared_metadata_matches_observed,True,failed_comparisons=0
4,required_metadata_declared,True,required_missing=0
5,cross_document_metadata_consistent,True,inconsistent=0
6,observed_event_coverage_positive,True,coverage_seconds=3599.217765000
7,no_raw_record_rejections,True,rejections=0


Collector metadata and run-manifest audit: PASS
Collector metadata leaves inspected: 31
Run manifest leaves inspected: 29
Observed trade records: 67,683
Observed depth records: 35,994
Observed total messages: 103,677
Observed event coverage: 3599.217765 seconds
Raw record rejections: 0


In [40]:
# ============================================================
# NOTEBOOK 01 PRE-WRITE VALIDATION AND RAW-AUDIT FREEZE
# ============================================================

from datetime import date
from decimal import Decimal
from pathlib import Path
import math
import re


# ============================================================
# JSON-SAFE CANONICALIZATION
# ============================================================

def notebook_01_json_safe(
    value,
):
    """
    Convert notebook objects into deterministic JSON-compatible values.
    """
    if value is None:
        return None

    if value is pd.NA or value is pd.NaT:
        return None

    if isinstance(value, (str, bool, int)):
        return value

    if isinstance(value, float):
        return (
            value
            if math.isfinite(value)
            else None
        )

    if isinstance(value, np.generic):
        return notebook_01_json_safe(
            value.item()
        )

    if isinstance(value, Decimal):
        return (
            str(value)
            if value.is_finite()
            else None
        )

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, pd.Timestamp):
        if pd.isna(value):
            return None

        return value.isoformat()

    if isinstance(value, pd.Timedelta):
        if pd.isna(value):
            return None

        return value.isoformat()

    if isinstance(value, datetime):
        return value.isoformat()

    if isinstance(value, date):
        return value.isoformat()

    if isinstance(value, pd.DataFrame):
        columns = [
            str(column)
            for column in value.columns
        ]

        records = []

        for row in value.itertuples(
            index=False,
            name=None,
        ):
            records.append(
                {
                    column: notebook_01_json_safe(
                        item
                    )
                    for column, item in zip(
                        columns,
                        row,
                    )
                }
            )

        return records

    if isinstance(value, pd.Series):
        return {
            str(key): notebook_01_json_safe(
                item
            )
            for key, item in value.items()
        }

    if isinstance(value, pd.Index):
        return [
            notebook_01_json_safe(item)
            for item in value.tolist()
        ]

    if isinstance(value, np.ndarray):
        return notebook_01_json_safe(
            value.tolist()
        )

    if isinstance(value, dict):
        return {
            str(key): notebook_01_json_safe(
                item
            )
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            notebook_01_json_safe(item)
            for item in value
        ]

    if isinstance(value, set):
        normalized_items = [
            notebook_01_json_safe(item)
            for item in value
        ]

        return sorted(
            normalized_items,
            key=lambda item: json.dumps(
                item,
                sort_keys=True,
                ensure_ascii=False,
            ),
        )

    try:
        missing = pd.isna(value)

    except (
        TypeError,
        ValueError,
    ):
        missing = False

    if (
        isinstance(
            missing,
            (bool, np.bool_),
        )
        and bool(missing)
    ):
        return None

    raise TypeError(
        "Unsupported Notebook 01 audit value type: "
        f"{type(value).__name__}"
    )


def notebook_01_canonical_json_bytes(
    value,
) -> bytes:
    return json.dumps(
        notebook_01_json_safe(value),
        sort_keys=True,
        ensure_ascii=False,
        allow_nan=False,
        separators=(",", ":"),
    ).encode("utf-8")


def notebook_01_object_hash(
    value,
) -> str:
    return hashlib.sha256(
        notebook_01_canonical_json_bytes(
            value
        )
    ).hexdigest()


def frame_records(
    frame: pd.DataFrame,
) -> list[dict]:
    if not isinstance(
        frame,
        pd.DataFrame,
    ):
        raise TypeError(
            "frame must be a pandas DataFrame."
        )

    return notebook_01_json_safe(
        frame
    )


# ============================================================
# RUN AND CONTRACT IDENTITIES
# ============================================================

NOTEBOOK_01_SOURCE_SET_HASH = str(
    V0_0_SOURCE_REGISTRY[
        "source_set_hash"
    ]
)

NOTEBOOK_01_RUN_CONFIG_HASH = str(
    NOTEBOOK_00_OUTPUT_MANIFEST_PAYLOAD[
        "v0_1_run_config_hash"
    ]
)

NOTEBOOK_01_RUN_IDENTITY_HASH = str(
    NOTEBOOK_00_OUTPUT_MANIFEST_PAYLOAD[
        "v0_1_run_identity_hash"
    ]
)

NOTEBOOK_01_OPERATING_MODE = str(
    V0_1_RUN_CONFIG[
        "operating_authority"
    ]["operating_mode"]
)

NOTEBOOK_01_PARENT_CONTRACT_STATUS = str(
    V0_1_RUN_CONFIG[
        "operating_authority"
    ]["contract_status"]
)

NOTEBOOK_02_FILENAME = (
    "02_VISIBLE_BOOK_RECONSTRUCTION.ipynb"
)


# ============================================================
# RECONCILE ALL PRIOR NOTEBOOK 01 GATE TABLES
# ============================================================

NOTEBOOK_01_PRIOR_GATE_TABLES = {
    "NOTEBOOK_01_BOOTSTRAP_GATES": (
        NOTEBOOK_01_BOOTSTRAP_GATES
    ),
    "RAW_FILE_AUDIT_GATES": (
        RAW_FILE_AUDIT_GATES
    ),
    "FULL_RECORD_SCHEMA_GATES": (
        FULL_RECORD_SCHEMA_GATES
    ),
    "RAW_ORDER_TIMESTAMP_GATES": (
        RAW_ORDER_TIMESTAMP_GATES
    ),
    "TRADE_SEMANTIC_GATES": (
        TRADE_SEMANTIC_GATES
    ),
    "DEPTH_SEMANTIC_GATES": (
        DEPTH_SEMANTIC_GATES
    ),
    "METADATA_AND_REJECTION_GATES": (
        METADATA_AND_REJECTION_GATES
    ),
}


prior_gate_rows = []


for gate_table_name, gate_table in (
    NOTEBOOK_01_PRIOR_GATE_TABLES.items()
):
    if not isinstance(
        gate_table,
        pd.DataFrame,
    ):
        raise TypeError(
            f"{gate_table_name} is not a DataFrame."
        )

    if "passed" not in gate_table.columns:
        raise RuntimeError(
            f"{gate_table_name} lacks a passed column."
        )

    gate_count = int(
        len(gate_table)
    )

    passed_count = int(
        gate_table[
            "passed"
        ]
        .fillna(False)
        .astype(bool)
        .sum()
    )

    prior_gate_rows.append(
        {
            "gate_table": gate_table_name,
            "gate_count": gate_count,
            "passed_count": passed_count,
            "failed_count": (
                gate_count
                - passed_count
            ),
            "all_passed": bool(
                gate_count > 0
                and gate_count
                == passed_count
            ),
        }
    )


NOTEBOOK_01_PRIOR_GATE_RECONCILIATION = (
    pd.DataFrame(
        prior_gate_rows
    )
)


# ============================================================
# FINAL SOURCE RE-HASH BEFORE WRITING
# ============================================================

source_rehash_rows = []


for source_row in (
    AUTHORITATIVE_RAW_SOURCE_MANIFEST
    .itertuples(index=False)
):
    source_path = Path(
        source_row.resolved_path
    ).resolve(strict=False)

    file_exists = bool(
        source_path.exists()
        and source_path.is_file()
    )

    inside_v0_0 = path_is_within(
        source_path,
        V0_0_ROOT_RESOLVED,
    )

    inside_v0_1 = path_is_within(
        source_path,
        V0_1_ROOT_RESOLVED,
    )

    if file_exists:
        observed_size_bytes = int(
            source_path.stat().st_size
        )

        observed_sha256 = sha256_file(
            source_path
        )

    else:
        observed_size_bytes = None
        observed_sha256 = None

    expected_size_bytes = int(
        source_row.size_bytes
    )

    expected_sha256 = str(
        source_row.sha256
    ).lower()

    size_matches = bool(
        observed_size_bytes
        == expected_size_bytes
    )

    hash_matches = bool(
        observed_sha256
        == expected_sha256
    )

    source_passed = bool(
        file_exists
        and inside_v0_0
        and not inside_v0_1
        and size_matches
        and hash_matches
    )

    source_rehash_rows.append(
        {
            "source_role": (
                source_row.source_role
            ),
            "resolved_path": str(
                source_path
            ),
            "file_exists": file_exists,
            "inside_v0_0": inside_v0_0,
            "inside_v0_1": inside_v0_1,
            "expected_size_bytes": (
                expected_size_bytes
            ),
            "observed_size_bytes": (
                observed_size_bytes
            ),
            "size_matches": size_matches,
            "expected_sha256": (
                expected_sha256
            ),
            "observed_sha256": (
                observed_sha256
            ),
            "hash_matches": hash_matches,
            "source_passed": (
                source_passed
            ),
        }
    )


NOTEBOOK_01_SOURCE_REHASH = pd.DataFrame(
    source_rehash_rows
)


# ============================================================
# RECONCILED AUDIT COUNTS
# ============================================================

NOTEBOOK_01_RECONCILED_COUNTS = {
    "primary_source_file_count": int(
        len(
            AUTHORITATIVE_RAW_SOURCE_MANIFEST
        )
    ),
    "trade_record_count": int(
        len(RAW_TRADE_AUDIT_TABLE)
    ),
    "depth_record_count": int(
        len(
            RAW_DEPTH_EVENT_AUDIT_TABLE
        )
    ),
    "combined_stream_record_count": int(
        len(RAW_EVENT_ORDER_INDEX)
    ),
    "canonical_field_count": int(
        len(CANONICAL_FIELD_AUDIT)
    ),
    "runtime_binding_count": int(
        len(
            RUNTIME_CANONICAL_FIELD_BINDINGS
        )
    ),
    "chronological_partition_count": int(
        len(
            PARTITION_ASSIGNMENT_AUDIT
        )
    ),
    "trade_id_count": int(
        RAW_TRADE_AUDIT_TABLE[
            "trade_id"
        ].nunique(
            dropna=True
        )
    ),
    "depth_update_level_count": int(
        DEPTH_LEVEL_PROFILE[
            "level_count"
        ].sum()
    ),
    "zero_quantity_delete_count": int(
        DEPTH_LEVEL_PROFILE[
            "zero_quantity_count"
        ].sum()
    ),
    "raw_rejection_count": int(
        len(RAW_REJECTION_LEDGER)
    ),
    "stale_pre_snapshot_depth_event_count": int(
        stale_event_count
    ),
    "post_snapshot_depth_event_count": int(
        post_bridge_event_count
    ),
}


# ============================================================
# FREEZE THE AUTHORITATIVE RAW-DATA AUDIT
# ============================================================

NOTEBOOK_01_RAW_DATA_AUDIT = {
    "audit_type": (
        "V0_1_RAW_DATA_AUDIT"
    ),
    "audit_schema_version": (
        CONTRACT_SCHEMA_VERSION
    ),
    "pipeline_name": PIPELINE_NAME,
    "pipeline_version": PIPELINE_VERSION,
    "producing_notebook": (
        NOTEBOOK_FILENAME
    ),
    "source_run_prefix": (
        SOURCE_RUN_PREFIX
    ),
    "source_set_hash": (
        NOTEBOOK_01_SOURCE_SET_HASH
    ),
    "v0_1_run_id": V0_1_RUN_ID,
    "v0_1_run_config_hash": (
        NOTEBOOK_01_RUN_CONFIG_HASH
    ),
    "v0_1_run_identity_hash": (
        NOTEBOOK_01_RUN_IDENTITY_HASH
    ),
    "operating_mode": (
        NOTEBOOK_01_OPERATING_MODE
    ),
    "parent_contract_status": (
        NOTEBOOK_01_PARENT_CONTRACT_STATUS
    ),
    "canonical_timezone": "UTC",
    "primary_ordering_authority": (
        "collector_sequence"
    ),
    "immutable_source_policy": {
        "v0_0_root": str(
            V0_0_ROOT_RESOLVED
        ),
        "v0_1_root": str(
            V0_1_ROOT_RESOLVED
        ),
        "v0_0_read_only": True,
        "source_files_copied": False,
        "source_files_modified": False,
    },
    "reconciled_counts": (
        NOTEBOOK_01_RECONCILED_COUNTS
    ),
    "source_manifest": frame_records(
        AUTHORITATIVE_RAW_SOURCE_MANIFEST
    ),
    "source_integrity_audit": frame_records(
        NOTEBOOK_01_SOURCE_REHASH
    ),
    "jsonl_parse_audit": frame_records(
        JSONL_PARSE_AUDIT
    ),
    "json_document_audit": frame_records(
        JSON_DOCUMENT_AUDIT
    ),
    "runtime_canonical_bindings": (
        frame_records(
            RUNTIME_CANONICAL_FIELD_BINDINGS
        )
    ),
    "canonical_field_audit": (
        frame_records(
            CANONICAL_FIELD_AUDIT
        )
    ),
    "raw_record_signatures": (
        frame_records(
            RAW_RECORD_SIGNATURES
        )
    ),
    "global_order_audit": frame_records(
        GLOBAL_ORDER_AUDIT
    ),
    "timestamp_audit": frame_records(
        TIMESTAMP_AUDIT
    ),
    "clock_diagnostic_summary": (
        frame_records(
            CLOCK_DIAGNOSTIC_SUMMARY
        )
    ),
    "trade_clock_diagnostic": (
        frame_records(
            TRADE_CLOCK_DIAGNOSTIC
        )
    ),
    "partition_assignment_audit": (
        frame_records(
            PARTITION_ASSIGNMENT_AUDIT
        )
    ),
    "trade_payload_reconciliation": (
        frame_records(
            TRADE_PAYLOAD_RECONCILIATION
        )
    ),
    "trade_identity_audit": (
        frame_records(
            TRADE_IDENTITY_AUDIT
        )
    ),
    "trade_numeric_profile": (
        frame_records(
            TRADE_NUMERIC_PROFILE
        )
    ),
    "trade_aggressor_side_summary": (
        frame_records(
            TRADE_AGGRESSOR_SIDE_SUMMARY
        )
    ),
    "depth_payload_reconciliation": (
        frame_records(
            DEPTH_PAYLOAD_RECONCILIATION
        )
    ),
    "depth_event_activity_summary": (
        frame_records(
            DEPTH_EVENT_ACTIVITY_SUMMARY
        )
    ),
    "depth_level_profile": (
        frame_records(
            DEPTH_LEVEL_PROFILE
        )
    ),
    "depth_update_id_audit": (
        frame_records(
            DEPTH_UPDATE_ID_AUDIT
        )
    ),
    "snapshot_buffer_reconciliation": (
        frame_records(
            SNAPSHOT_BUFFER_RECONCILIATION
        )
    ),
    "metadata_declaration_audit": (
        frame_records(
            METADATA_DECLARATION_AUDIT
        )
    ),
    "metadata_cross_document_audit": (
        frame_records(
            METADATA_CROSS_DOCUMENT_AUDIT
        )
    ),
    "raw_rejection_summary": (
        frame_records(
            RAW_REJECTION_REASON_SUMMARY
        )
    ),
    "acceptance_authority": {
        "raw_records_accepted": bool(
            RAW_REJECTION_LEDGER.empty
        ),
        "snapshot_bridge_accepted": bool(
            snapshot_bridge_passed
        ),
        "post_snapshot_update_ids_contiguous": bool(
            post_bridge_gap_count == 0
            and post_bridge_overlap_count == 0
        ),
        "trade_ids_unique": bool(
            RAW_TRADE_AUDIT_TABLE[
                "trade_id"
            ].is_unique
        ),
        "partitions_match_contract": bool(
            PARTITION_ASSIGNMENT_AUDIT[
                "row_count_matches"
            ].all()
            and PARTITION_ASSIGNMENT_AUDIT[
                "trade_row_count_matches"
            ].all()
            and PARTITION_ASSIGNMENT_AUDIT[
                "depth_row_count_matches"
            ].all()
        ),
        "claim_bearing_holdout_available": bool(
            CHRONOLOGICAL_SPLIT_CONTRACT.get(
                "claim_bearing_holdout_available",
                False,
            )
        ),
        "partitions_are_independent": bool(
            CHRONOLOGICAL_SPLIT_CONTRACT.get(
                "partitions_are_independent",
                False,
            )
        ),
    },
    "handoff": {
        "next_notebook": (
            NOTEBOOK_02_FILENAME
        ),
        "notebook_02_purpose": (
            "Reconstruct the visible market-by-price order book "
            "from the verified REST snapshot and contiguous "
            "post-snapshot differential-depth stream."
        ),
        "first_applicable_depth_collector_sequence": int(
            first_post_snapshot_sequence
        ),
        "first_applicable_depth_U": int(
            first_post_snapshot_U
        ),
        "first_applicable_depth_u": int(
            first_post_snapshot_u
        ),
        "stale_depth_events_to_discard": int(
            stale_event_count
        ),
    },
}


NOTEBOOK_01_RAW_DATA_AUDIT = (
    notebook_01_json_safe(
        NOTEBOOK_01_RAW_DATA_AUDIT
    )
)

NOTEBOOK_01_RAW_DATA_AUDIT_HASH = (
    notebook_01_object_hash(
        NOTEBOOK_01_RAW_DATA_AUDIT
    )
)


# ============================================================
# PRE-WRITE VALIDATION LEDGER
# ============================================================

prewrite_rows = []


def register_notebook_01_prewrite_check(
    check_id: str,
    category: str,
    passed: bool,
    severity: str,
    evidence: str,
) -> None:
    prewrite_rows.append(
        {
            "check_id": check_id,
            "category": category,
            "passed": bool(passed),
            "severity": severity,
            "evidence": str(evidence),
        }
    )


prior_gates_passed = bool(
    NOTEBOOK_01_PRIOR_GATE_RECONCILIATION[
        "all_passed"
    ].all()
)

sources_unchanged = bool(
    NOTEBOOK_01_SOURCE_REHASH[
        "source_passed"
    ].all()
)

all_sources_inside_v0_0 = bool(
    NOTEBOOK_01_SOURCE_REHASH[
        "inside_v0_0"
    ].all()
    and ~NOTEBOOK_01_SOURCE_REHASH[
        "inside_v0_1"
    ].any()
)

row_counts_preserved = bool(
    len(RAW_TRADE_AUDIT_TABLE)
    == EXPECTED_TRADE_ROWS
    and len(
        RAW_DEPTH_EVENT_AUDIT_TABLE
    )
    == EXPECTED_DEPTH_ROWS
    and len(RAW_EVENT_ORDER_INDEX)
    == EXPECTED_COMBINED_STREAM_ROWS
)

schema_complete = bool(
    CANONICAL_FIELD_AUDIT[
        "missing_count"
    ].eq(0).all()
    and CANONICAL_FIELD_AUDIT[
        "null_count"
    ].eq(0).all()
    and CANONICAL_FIELD_AUDIT[
        "invalid_count"
    ].eq(0).all()
)

sequence_integrity_passed = bool(
    collector_sequence_missing_count == 0
    and collector_sequence_duplicate_count == 0
    and collector_sequence_gap_count == 0
    and collector_sequence_reversal_count == 0
)

timestamp_integrity_passed = bool(
    local_timestamp_invalid_count == 0
    and exchange_timestamp_invalid_count == 0
    and trade_timestamp_invalid_count == 0
)

trade_integrity_passed = bool(
    RAW_TRADE_AUDIT_TABLE[
        "trade_id"
    ].is_unique
    and TRADE_PAYLOAD_RECONCILIATION[
        "mismatch_count"
    ].eq(0).all()
)

depth_integrity_passed = bool(
    snapshot_bridge_passed
    and post_bridge_gap_count == 0
    and post_bridge_overlap_count == 0
    and DEPTH_LEVEL_PROFILE[
        "invalid_level_count"
    ].eq(0).all()
)

metadata_integrity_passed = bool(
    METADATA_DECLARATION_AUDIT[
        "comparison_passed"
    ].all()
    and METADATA_CROSS_DOCUMENT_AUDIT[
        "cross_document_consistent"
    ].all()
)

partition_integrity_passed = bool(
    PARTITION_ASSIGNMENT_AUDIT[
        "row_count_matches"
    ].all()
    and PARTITION_ASSIGNMENT_AUDIT[
        "trade_row_count_matches"
    ].all()
    and PARTITION_ASSIGNMENT_AUDIT[
        "depth_row_count_matches"
    ].all()
)

audit_hash_valid = bool(
    re.fullmatch(
        r"[a-f0-9]{64}",
        NOTEBOOK_01_RAW_DATA_AUDIT_HASH,
    )
    is not None
)


register_notebook_01_prewrite_check(
    check_id=(
        "ALL_PRIOR_NOTEBOOK_01_GATES_PASS"
    ),
    category="NOTEBOOK_STATE",
    passed=prior_gates_passed,
    severity="CRITICAL",
    evidence=(
        f"tables_passed="
        f"{int(NOTEBOOK_01_PRIOR_GATE_RECONCILIATION['all_passed'].sum())}/"
        f"{len(NOTEBOOK_01_PRIOR_GATE_RECONCILIATION)}"
    ),
)

register_notebook_01_prewrite_check(
    check_id=(
        "AUTHORITATIVE_SOURCE_FILES_UNCHANGED"
    ),
    category="SOURCE_INTEGRITY",
    passed=sources_unchanged,
    severity="CRITICAL",
    evidence=(
        f"verified="
        f"{int(NOTEBOOK_01_SOURCE_REHASH['source_passed'].sum())}/"
        f"{len(NOTEBOOK_01_SOURCE_REHASH)}"
    ),
)

register_notebook_01_prewrite_check(
    check_id=(
        "SOURCE_PATH_AUTHORITY_PRESERVED"
    ),
    category="SOURCE_INTEGRITY",
    passed=all_sources_inside_v0_0,
    severity="CRITICAL",
    evidence=(
        f"inside_v0_0="
        f"{NOTEBOOK_01_SOURCE_REHASH['inside_v0_0'].all()}; "
        f"inside_v0_1="
        f"{NOTEBOOK_01_SOURCE_REHASH['inside_v0_1'].any()}"
    ),
)

register_notebook_01_prewrite_check(
    check_id="RAW_ROW_COUNTS_PRESERVED",
    category="ROW_AUTHORITY",
    passed=row_counts_preserved,
    severity="CRITICAL",
    evidence=(
        f"trades={len(RAW_TRADE_AUDIT_TABLE)}/{EXPECTED_TRADE_ROWS}; "
        f"depth={len(RAW_DEPTH_EVENT_AUDIT_TABLE)}/{EXPECTED_DEPTH_ROWS}; "
        f"combined={len(RAW_EVENT_ORDER_INDEX)}/"
        f"{EXPECTED_COMBINED_STREAM_ROWS}"
    ),
)

register_notebook_01_prewrite_check(
    check_id=(
        "CANONICAL_SCHEMA_COMPLETE"
    ),
    category="SCHEMA",
    passed=schema_complete,
    severity="CRITICAL",
    evidence=(
        f"fields="
        f"{len(CANONICAL_FIELD_AUDIT)}; "
        f"missing="
        f"{int(CANONICAL_FIELD_AUDIT['missing_count'].sum())}; "
        f"invalid="
        f"{int(CANONICAL_FIELD_AUDIT['invalid_count'].sum())}"
    ),
)

register_notebook_01_prewrite_check(
    check_id=(
        "COLLECTOR_SEQUENCE_INTEGRITY"
    ),
    category="ORDERING",
    passed=sequence_integrity_passed,
    severity="CRITICAL",
    evidence=(
        f"missing={collector_sequence_missing_count}; "
        f"duplicates={collector_sequence_duplicate_count}; "
        f"gaps={collector_sequence_gap_count}; "
        f"reversals={collector_sequence_reversal_count}"
    ),
)

register_notebook_01_prewrite_check(
    check_id="TIMESTAMP_INTEGRITY",
    category="TIMESTAMPS",
    passed=timestamp_integrity_passed,
    severity="CRITICAL",
    evidence=(
        f"local_invalid={local_timestamp_invalid_count}; "
        f"exchange_invalid={exchange_timestamp_invalid_count}; "
        f"trade_invalid={trade_timestamp_invalid_count}"
    ),
)

register_notebook_01_prewrite_check(
    check_id="TRADE_STREAM_INTEGRITY",
    category="TRADE_STREAM",
    passed=trade_integrity_passed,
    severity="CRITICAL",
    evidence=(
        f"unique_trade_ids="
        f"{RAW_TRADE_AUDIT_TABLE['trade_id'].is_unique}; "
        f"payload_mismatches="
        f"{int(TRADE_PAYLOAD_RECONCILIATION['mismatch_count'].sum())}"
    ),
)

register_notebook_01_prewrite_check(
    check_id="DEPTH_STREAM_INTEGRITY",
    category="DEPTH_STREAM",
    passed=depth_integrity_passed,
    severity="CRITICAL",
    evidence=(
        f"bridge={snapshot_bridge_passed}; "
        f"gaps={post_bridge_gap_count}; "
        f"overlaps={post_bridge_overlap_count}; "
        f"invalid_levels="
        f"{int(DEPTH_LEVEL_PROFILE['invalid_level_count'].sum())}"
    ),
)

register_notebook_01_prewrite_check(
    check_id=(
        "METADATA_AND_MANIFEST_RECONCILED"
    ),
    category="METADATA",
    passed=metadata_integrity_passed,
    severity="CRITICAL",
    evidence=(
        f"failed_comparisons="
        f"{int((~METADATA_DECLARATION_AUDIT['comparison_passed']).sum())}; "
        f"inconsistent_fields="
        f"{int((~METADATA_CROSS_DOCUMENT_AUDIT['cross_document_consistent']).sum())}"
    ),
)

register_notebook_01_prewrite_check(
    check_id="NO_RAW_RECORD_REJECTIONS",
    category="REJECTIONS",
    passed=bool(
        RAW_REJECTION_LEDGER.empty
    ),
    severity="CRITICAL",
    evidence=(
        f"rejections="
        f"{len(RAW_REJECTION_LEDGER)}"
    ),
)

register_notebook_01_prewrite_check(
    check_id=(
        "PARTITION_COUNTS_RECONCILED"
    ),
    category="PARTITIONS",
    passed=partition_integrity_passed,
    severity="CRITICAL",
    evidence=(
        f"partitions="
        f"{len(PARTITION_ASSIGNMENT_AUDIT)}"
    ),
)

register_notebook_01_prewrite_check(
    check_id=(
        "RAW_DATA_AUDIT_HASH_VALID"
    ),
    category="IDENTITY",
    passed=audit_hash_valid,
    severity="CRITICAL",
    evidence=(
        NOTEBOOK_01_RAW_DATA_AUDIT_HASH
    ),
)

register_notebook_01_prewrite_check(
    check_id=(
        "FULL_STATISTICAL_MODE_AVAILABLE"
    ),
    category="STATISTICAL_AUTHORITY",
    passed=bool(
        NOTEBOOK_01_OPERATING_MODE
        == "FULL_STATISTICAL_MODE"
    ),
    severity="WARNING",
    evidence=NOTEBOOK_01_OPERATING_MODE,
)

register_notebook_01_prewrite_check(
    check_id=(
        "CLAIM_BEARING_HOLDOUT_AVAILABLE"
    ),
    category="STATISTICAL_AUTHORITY",
    passed=bool(
        CHRONOLOGICAL_SPLIT_CONTRACT.get(
            "claim_bearing_holdout_available",
            False,
        )
    ),
    severity="WARNING",
    evidence=(
        "available="
        f"{CHRONOLOGICAL_SPLIT_CONTRACT.get('claim_bearing_holdout_available', False)}"
    ),
)


NOTEBOOK_01_PREWRITE_VALIDATION_LEDGER = (
    pd.DataFrame(
        prewrite_rows
    )
)


NOTEBOOK_01_PREWRITE_VALIDATION_LEDGER[
    "status"
] = np.select(
    [
        NOTEBOOK_01_PREWRITE_VALIDATION_LEDGER[
            "passed"
        ],
        NOTEBOOK_01_PREWRITE_VALIDATION_LEDGER[
            "severity"
        ].eq("CRITICAL"),
    ],
    [
        "PASS",
        "FAIL",
    ],
    default="WARNING",
)


notebook_01_critical_failures = (
    NOTEBOOK_01_PREWRITE_VALIDATION_LEDGER.loc[
        NOTEBOOK_01_PREWRITE_VALIDATION_LEDGER[
            "severity"
        ].eq("CRITICAL")
        & ~NOTEBOOK_01_PREWRITE_VALIDATION_LEDGER[
            "passed"
        ]
    ]
)

notebook_01_warnings = (
    NOTEBOOK_01_PREWRITE_VALIDATION_LEDGER.loc[
        NOTEBOOK_01_PREWRITE_VALIDATION_LEDGER[
            "severity"
        ].eq("WARNING")
        & ~NOTEBOOK_01_PREWRITE_VALIDATION_LEDGER[
            "passed"
        ]
    ]
)


if not notebook_01_critical_failures.empty:
    NOTEBOOK_01_PREWRITE_STATUS = "FAIL"

elif (
    NOTEBOOK_01_PARENT_CONTRACT_STATUS
    == "CONDITIONAL PASS"
):
    NOTEBOOK_01_PREWRITE_STATUS = (
        "CONDITIONAL PASS"
    )

elif not notebook_01_warnings.empty:
    NOTEBOOK_01_PREWRITE_STATUS = "WARNING"

else:
    NOTEBOOK_01_PREWRITE_STATUS = "PASS"


NOTEBOOK_02_AUTHORIZED = bool(
    notebook_01_critical_failures.empty
)


NOTEBOOK_01_PREWRITE_DECISION = pd.DataFrame(
    [
        {
            "pipeline_name": PIPELINE_NAME,
            "pipeline_version": (
                PIPELINE_VERSION
            ),
            "notebook": NOTEBOOK_FILENAME,
            "source_run_prefix": (
                SOURCE_RUN_PREFIX
            ),
            "source_set_hash": (
                NOTEBOOK_01_SOURCE_SET_HASH
            ),
            "v0_1_run_id": V0_1_RUN_ID,
            "operating_mode": (
                NOTEBOOK_01_OPERATING_MODE
            ),
            "prewrite_status": (
                NOTEBOOK_01_PREWRITE_STATUS
            ),
            "primary_source_count": int(
                len(
                    AUTHORITATIVE_RAW_SOURCE_MANIFEST
                )
            ),
            "trade_record_count": int(
                len(
                    RAW_TRADE_AUDIT_TABLE
                )
            ),
            "depth_record_count": int(
                len(
                    RAW_DEPTH_EVENT_AUDIT_TABLE
                )
            ),
            "combined_record_count": int(
                len(RAW_EVENT_ORDER_INDEX)
            ),
            "raw_rejection_count": int(
                len(RAW_REJECTION_LEDGER)
            ),
            "critical_failure_count": int(
                len(
                    notebook_01_critical_failures
                )
            ),
            "warning_count": int(
                len(notebook_01_warnings)
            ),
            "raw_data_audit_hash": (
                NOTEBOOK_01_RAW_DATA_AUDIT_HASH
            ),
            "notebook_02_authorized": (
                NOTEBOOK_02_AUTHORIZED
            ),
            "next_notebook": (
                NOTEBOOK_02_FILENAME
            ),
        }
    ]
)


# ============================================================
# DISPLAY AND ENFORCE
# ============================================================

display(
    NOTEBOOK_01_PRIOR_GATE_RECONCILIATION
)

display(
    NOTEBOOK_01_SOURCE_REHASH[
        [
            "source_role",
            "resolved_path",
            "file_exists",
            "size_matches",
            "hash_matches",
            "inside_v0_0",
            "inside_v0_1",
            "source_passed",
        ]
    ]
)

display(
    NOTEBOOK_01_PREWRITE_VALIDATION_LEDGER
)

display(
    NOTEBOOK_01_PREWRITE_DECISION
)


if not notebook_01_critical_failures.empty:
    failure_text = ", ".join(
        f"{row.check_id}: {row.evidence}"
        for row in (
            notebook_01_critical_failures
            .itertuples(index=False)
        )
    )

    raise RuntimeError(
        "Notebook 01 pre-write validation failed: "
        + failure_text
    )


print("Notebook 01 pre-write validation: PASS")
print(
    "Pre-write status: "
    f"{NOTEBOOK_01_PREWRITE_STATUS}"
)
print(
    "Prior gate tables passed: "
    f"{int(NOTEBOOK_01_PRIOR_GATE_RECONCILIATION['all_passed'].sum())}/"
    f"{len(NOTEBOOK_01_PRIOR_GATE_RECONCILIATION)}"
)
print(
    "Primary source files unchanged: "
    f"{int(NOTEBOOK_01_SOURCE_REHASH['source_passed'].sum())}/"
    f"{len(NOTEBOOK_01_SOURCE_REHASH)}"
)
print(
    "Raw record rejections: "
    f"{len(RAW_REJECTION_LEDGER):,}"
)
print(
    "Critical failures: "
    f"{len(notebook_01_critical_failures):,}"
)
print(
    "Warnings retained: "
    f"{len(notebook_01_warnings):,}"
)
print(
    "Raw-data audit SHA-256: "
    f"{NOTEBOOK_01_RAW_DATA_AUDIT_HASH}"
)
print(
    "Notebook 02 authorized: "
    f"{NOTEBOOK_02_AUTHORIZED}"
)
print(
    "Next notebook: "
    f"{NOTEBOOK_02_FILENAME}"
)

,gate_table,gate_count,passed_count,failed_count,all_passed
0,NOTEBOOK_01_BOOTSTRAP_GATES,7,7,0,True
1,RAW_FILE_AUDIT_GATES,11,11,0,True
2,FULL_RECORD_SCHEMA_GATES,11,11,0,True
3,RAW_ORDER_TIMESTAMP_GATES,10,10,0,True
4,TRADE_SEMANTIC_GATES,10,10,0,True
5,DEPTH_SEMANTIC_GATES,15,15,0,True
6,METADATA_AND_REJECTION_GATES,8,8,0,True


,source_role,resolved_path,file_exists,size_matches,hash_matches,inside_v0_0,inside_v0_1,source_passed
0,COLLECTOR_METADATA,D:\Clown Project\V0.0\data\raw\metadata\BTCUSD...,True,True,True,True,False,True
1,DEPTH_STREAM,D:\Clown Project\V0.0\data\raw\order_book\BTCU...,True,True,True,True,False,True
2,REST_SNAPSHOT,D:\Clown Project\V0.0\data\raw\order_book\BTCU...,True,True,True,True,False,True
3,RUN_MANIFEST,D:\Clown Project\V0.0\data\raw\metadata\BTCUSD...,True,True,True,True,False,True
4,TRADE_STREAM,D:\Clown Project\V0.0\data\raw\trades\BTCUSDT_...,True,True,True,True,False,True


,check_id,category,passed,severity,evidence,status
0,ALL_PRIOR_NOTEBOOK_01_GATES_PASS,NOTEBOOK_STATE,True,CRITICAL,tables_passed=7/7,PASS
1,AUTHORITATIVE_SOURCE_FILES_UNCHANGED,SOURCE_INTEGRITY,True,CRITICAL,verified=5/5,PASS
2,SOURCE_PATH_AUTHORITY_PRESERVED,SOURCE_INTEGRITY,True,CRITICAL,inside_v0_0=True; inside_v0_1=False,PASS
3,RAW_ROW_COUNTS_PRESERVED,ROW_AUTHORITY,True,CRITICAL,trades=67683/67683; depth=35994/35994; combine...,PASS
4,CANONICAL_SCHEMA_COMPLETE,SCHEMA,True,CRITICAL,fields=21; missing=0; invalid=0,PASS
5,COLLECTOR_SEQUENCE_INTEGRITY,ORDERING,True,CRITICAL,missing=0; duplicates=0; gaps=0; reversals=0,PASS
6,TIMESTAMP_INTEGRITY,TIMESTAMPS,True,CRITICAL,local_invalid=0; exchange_invalid=0; trade_inv...,PASS
7,TRADE_STREAM_INTEGRITY,TRADE_STREAM,True,CRITICAL,unique_trade_ids=True; payload_mismatches=0,PASS
8,DEPTH_STREAM_INTEGRITY,DEPTH_STREAM,True,CRITICAL,bridge=True; gaps=0; overlaps=0; invalid_levels=0,PASS
9,METADATA_AND_MANIFEST_RECONCILED,METADATA,True,CRITICAL,failed_comparisons=0; inconsistent_fields=0,PASS


,pipeline_name,pipeline_version,notebook,source_run_prefix,source_set_hash,v0_1_run_id,operating_mode,prewrite_status,primary_source_count,trade_record_count,depth_record_count,combined_record_count,raw_rejection_count,critical_failure_count,warning_count,raw_data_audit_hash,notebook_02_authorized,next_notebook
0,The Clown Project,V0.1,01_RAW_DATA_AUDIT.ipynb,BTCUSDT_spot_20260710T063746Z_c8b5bf12,132c83531eec615d279408b5c06f402973114ba3058dfa...,v0_1_20260714T090616Z_e82325081a81,ENGINEERING_REPRODUCTION_MODE,CONDITIONAL PASS,5,67683,35994,103677,0,0,2,fdfd1bc3ca907b1bf14286ca8f849d49a997bd5655f879...,True,02_VISIBLE_BOOK_RECONSTRUCTION.ipynb


Notebook 01 pre-write validation: PASS
Pre-write status: CONDITIONAL PASS
Prior gate tables passed: 7/7
Primary source files unchanged: 5/5
Raw record rejections: 0
Critical failures: 0
Warnings retained: 2
Raw-data audit SHA-256: fdfd1bc3ca907b1bf14286ca8f849d49a997bd5655f8793260838a34b1c9f56d
Notebook 02 authorized: True
Next notebook: 02_VISIBLE_BOOK_RECONSTRUCTION.ipynb


In [41]:
# ============================================================
# NOTEBOOK 01 ATOMIC ARTIFACT WRITES
# ============================================================

import tempfile


# ============================================================
# OUTPUT ROOTS
# ============================================================

V0_1_HANDOFF_ROOT = (
    V0_1_ROOT
    / "artifacts"
    / "handoff"
)

V0_1_MANIFEST_ROOT_RESOLVED = (
    V0_1_MANIFEST_ROOT.resolve(
        strict=False
    )
)

V0_1_AUDIT_ROOT_RESOLVED = (
    V0_1_AUDIT_ROOT.resolve(
        strict=False
    )
)

V0_1_HANDOFF_ROOT_RESOLVED = (
    V0_1_HANDOFF_ROOT.resolve(
        strict=False
    )
)


# ============================================================
# PORTABLE ARTIFACT NAMING
# ============================================================

def notebook_01_sanitize_token(
    value,
) -> str:
    token = re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        str(value).strip(),
    )

    token = token.strip("._-")

    if not token:
        raise ValueError(
            "Artifact token cannot be empty."
        )

    return token


NOTEBOOK_01_OUTPUT_PREFIX = (
    f"{SOURCE_RUN_PREFIX}__"
    f"{V0_1_RUN_ID}__"
    f"{notebook_01_sanitize_token(Path(NOTEBOOK_FILENAME).stem)}"
)


def notebook_01_artifact_name(
    artifact_name: str,
    extension: str,
) -> str:
    return (
        f"{NOTEBOOK_01_OUTPUT_PREFIX}__"
        f"{notebook_01_sanitize_token(artifact_name)}."
        f"{notebook_01_sanitize_token(extension.lstrip('.'))}"
    )


RAW_DATA_AUDIT_ARTIFACT_NAME = (
    notebook_01_artifact_name(
        artifact_name=(
            "v0_1_raw_data_audit"
        ),
        extension="json",
    )
)

NOTEBOOK_01_HANDOFF_ARTIFACT_NAME = (
    notebook_01_artifact_name(
        artifact_name=(
            "notebook_01_to_notebook_02_handoff"
        ),
        extension="json",
    )
)

NOTEBOOK_01_OUTPUT_MANIFEST_NAME = (
    notebook_01_artifact_name(
        artifact_name=(
            "notebook_01_output_manifest"
        ),
        extension="json",
    )
)


# ============================================================
# ARTIFACT WRAPPER
# ============================================================

def notebook_01_wrap_artifact(
    payload,
    artifact_type: str,
    row_count=None,
) -> dict:
    safe_payload = notebook_01_json_safe(
        payload
    )

    return {
        "artifact_metadata": {
            "artifact_type": artifact_type,
            "artifact_schema_version": (
                CONTRACT_SCHEMA_VERSION
            ),
            "pipeline_name": PIPELINE_NAME,
            "pipeline_version": (
                PIPELINE_VERSION
            ),
            "source_run_prefix": (
                SOURCE_RUN_PREFIX
            ),
            "source_set_hash": (
                NOTEBOOK_01_SOURCE_SET_HASH
            ),
            "v0_1_run_id": V0_1_RUN_ID,
            "v0_1_run_config_hash": (
                NOTEBOOK_01_RUN_CONFIG_HASH
            ),
            "v0_1_run_identity_hash": (
                NOTEBOOK_01_RUN_IDENTITY_HASH
            ),
            "producing_notebook": (
                NOTEBOOK_FILENAME
            ),
            "created_utc": (
                NOTEBOOK_01_STARTED_UTC
            ),
            "acceptance_status": (
                NOTEBOOK_01_PREWRITE_STATUS
            ),
            "row_count": (
                int(row_count)
                if row_count is not None
                else None
            ),
            "payload_sha256": (
                notebook_01_object_hash(
                    safe_payload
                )
            ),
        },
        "payload": safe_payload,
    }


# ============================================================
# DETERMINISTIC SERIALIZATION
# ============================================================

def notebook_01_pretty_json_bytes(
    value,
) -> bytes:
    normalized = notebook_01_json_safe(
        value
    )

    serialized = json.dumps(
        normalized,
        sort_keys=True,
        ensure_ascii=False,
        allow_nan=False,
        indent=2,
    )

    return (
        serialized + "\n"
    ).encode("utf-8")


def notebook_01_csv_bytes(
    frame: pd.DataFrame,
) -> bytes:
    if not isinstance(
        frame,
        pd.DataFrame,
    ):
        raise TypeError(
            "CSV artifact source must be a DataFrame."
        )

    csv_text = frame.to_csv(
        index=False,
        na_rep="",
        lineterminator="\n",
        date_format=(
            "%Y-%m-%dT%H:%M:%S.%f%z"
        ),
    )

    return csv_text.encode("utf-8")


# ============================================================
# OUTPUT PATH SAFETY
# ============================================================

def notebook_01_validate_output_path(
    path: Path,
) -> Path:
    resolved_path = Path(path).resolve(
        strict=False
    )

    if not path_is_within(
        resolved_path,
        V0_1_ROOT_RESOLVED,
    ):
        raise RuntimeError(
            "Output path is outside V0.1: "
            f"{resolved_path}"
        )

    if path_is_within(
        resolved_path,
        V0_0_ROOT_RESOLVED,
    ):
        raise RuntimeError(
            "Output path enters immutable V0.0: "
            f"{resolved_path}"
        )

    return resolved_path


# ============================================================
# ATOMIC WRITE HELPERS
# ============================================================

def notebook_01_atomic_write_bytes(
    path: Path,
    content: bytes,
) -> dict:
    target_path = (
        notebook_01_validate_output_path(
            path
        )
    )

    target_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with tempfile.NamedTemporaryFile(
        mode="wb",
        prefix=f".{target_path.name}.",
        suffix=".tmp",
        dir=target_path.parent,
        delete=False,
    ) as temporary_handle:
        temporary_path = Path(
            temporary_handle.name
        )

        temporary_handle.write(
            content
        )

        temporary_handle.flush()

        os.fsync(
            temporary_handle.fileno()
        )

    try:
        os.replace(
            temporary_path,
            target_path,
        )

    finally:
        if temporary_path.exists():
            temporary_path.unlink()

    written_bytes = target_path.read_bytes()

    return {
        "resolved_path": str(
            target_path
        ),
        "relative_path": str(
            target_path.relative_to(
                V0_1_ROOT_RESOLVED
            )
        ),
        "size_bytes": int(
            len(written_bytes)
        ),
        "sha256": hashlib.sha256(
            written_bytes
        ).hexdigest(),
    }


def notebook_01_atomic_write_json(
    path: Path,
    payload,
) -> dict:
    return notebook_01_atomic_write_bytes(
        path=path,
        content=(
            notebook_01_pretty_json_bytes(
                payload
            )
        ),
    )


def notebook_01_atomic_write_csv(
    path: Path,
    frame: pd.DataFrame,
) -> dict:
    return notebook_01_atomic_write_bytes(
        path=path,
        content=notebook_01_csv_bytes(
            frame
        ),
    )


# ============================================================
# NOTEBOOK 02 HANDOFF PAYLOAD
# ============================================================

NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF = {
    "handoff_type": (
        "NOTEBOOK_01_TO_NOTEBOOK_02"
    ),
    "handoff_schema_version": (
        CONTRACT_SCHEMA_VERSION
    ),
    "pipeline_name": PIPELINE_NAME,
    "pipeline_version": (
        PIPELINE_VERSION
    ),
    "source_run_prefix": (
        SOURCE_RUN_PREFIX
    ),
    "source_set_hash": (
        NOTEBOOK_01_SOURCE_SET_HASH
    ),
    "v0_1_run_id": V0_1_RUN_ID,
    "v0_1_run_config_hash": (
        NOTEBOOK_01_RUN_CONFIG_HASH
    ),
    "v0_1_run_identity_hash": (
        NOTEBOOK_01_RUN_IDENTITY_HASH
    ),
    "raw_data_audit_hash": (
        NOTEBOOK_01_RAW_DATA_AUDIT_HASH
    ),
    "operating_mode": (
        NOTEBOOK_01_OPERATING_MODE
    ),
    "acceptance_status": (
        NOTEBOOK_01_PREWRITE_STATUS
    ),
    "notebook_02_authorized": bool(
        NOTEBOOK_02_AUTHORIZED
    ),
    "next_notebook": (
        NOTEBOOK_02_FILENAME
    ),
    "authoritative_raw_sources": (
        frame_records(
            AUTHORITATIVE_RAW_SOURCE_MANIFEST[
                [
                    "source_role",
                    "resolved_path",
                    "size_bytes",
                    "sha256",
                ]
            ]
        )
    ),
    "snapshot_contract": {
        "snapshot_path": str(
            REST_SNAPSHOT_PATH
        ),
        "snapshot_last_update_id": int(
            SNAPSHOT_LAST_UPDATE_ID
        ),
        "snapshot_session_id": str(
            SNAPSHOT_SESSION_ID
        ),
        "snapshot_response_received_time_ns": (
            int(
                SNAPSHOT_RESPONSE_RECEIVED_TIME_NS
            )
            if (
                SNAPSHOT_RESPONSE_RECEIVED_TIME_NS
                is not None
            )
            else None
        ),
    },
    "depth_reconstruction_contract": {
        "depth_source_path": str(
            DEPTH_SOURCE_PATH
        ),
        "total_depth_event_count": int(
            len(
                RAW_DEPTH_EVENT_AUDIT_TABLE
            )
        ),
        "stale_pre_snapshot_event_count": int(
            stale_event_count
        ),
        "first_applicable_depth_position": int(
            first_post_snapshot_position
        ),
        "first_applicable_collector_sequence": int(
            first_post_snapshot_sequence
        ),
        "first_applicable_U": int(
            first_post_snapshot_U
        ),
        "first_applicable_u": int(
            first_post_snapshot_u
        ),
        "post_snapshot_event_count": int(
            post_bridge_event_count
        ),
        "post_snapshot_gap_count": int(
            post_bridge_gap_count
        ),
        "post_snapshot_overlap_count": int(
            post_bridge_overlap_count
        ),
        "zero_quantity_action": (
            "DELETE_PRICE_LEVEL"
        ),
        "ordering_authority": (
            "collector_sequence"
        ),
        "timezone": "UTC",
    },
    "partition_contract": (
        frame_records(
            FROZEN_PARTITION_BOUNDARIES
        )
    ),
    "raw_audit_summary": (
        NOTEBOOK_01_RECONCILED_COUNTS
    ),
    "raw_record_rejection_count": int(
        len(RAW_REJECTION_LEDGER)
    ),
    "claim_authority": {
        "claim_bearing_holdout_available": bool(
            CHRONOLOGICAL_SPLIT_CONTRACT.get(
                "claim_bearing_holdout_available",
                False,
            )
        ),
        "partitions_are_independent": bool(
            CHRONOLOGICAL_SPLIT_CONTRACT.get(
                "partitions_are_independent",
                False,
            )
        ),
    },
    "producing_notebook": (
        NOTEBOOK_FILENAME
    ),
}


NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF = (
    notebook_01_json_safe(
        NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF
    )
)

NOTEBOOK_01_HANDOFF_HASH = (
    notebook_01_object_hash(
        NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF
    )
)


# ============================================================
# JSON OUTPUT DOCUMENTS
# ============================================================

RAW_DATA_AUDIT_DOCUMENT = (
    notebook_01_wrap_artifact(
        payload=(
            NOTEBOOK_01_RAW_DATA_AUDIT
        ),
        artifact_type=(
            "V0_1_RAW_DATA_AUDIT"
        ),
        row_count=(
            EXPECTED_COMBINED_STREAM_ROWS
        ),
    )
)

NOTEBOOK_01_HANDOFF_DOCUMENT = (
    notebook_01_wrap_artifact(
        payload=(
            NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF
        ),
        artifact_type=(
            "NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF"
        ),
    )
)


# ============================================================
# AUDIT TABLE SPECIFICATION
# ============================================================

NOTEBOOK_01_AUDIT_TABLE_SPECS = [
    {
        "artifact_type": (
            "AUTHORITATIVE_RAW_SOURCE_MANIFEST"
        ),
        "artifact_name": (
            "authoritative_raw_source_manifest"
        ),
        "frame": (
            AUTHORITATIVE_RAW_SOURCE_MANIFEST
        ),
    },
    {
        "artifact_type": (
            "NOTEBOOK_01_SOURCE_REHASH"
        ),
        "artifact_name": (
            "notebook_01_source_rehash"
        ),
        "frame": (
            NOTEBOOK_01_SOURCE_REHASH
        ),
    },
    {
        "artifact_type": (
            "JSONL_PARSE_AUDIT"
        ),
        "artifact_name": (
            "jsonl_parse_audit"
        ),
        "frame": JSONL_PARSE_AUDIT,
    },
    {
        "artifact_type": (
            "JSON_DOCUMENT_AUDIT"
        ),
        "artifact_name": (
            "json_document_audit"
        ),
        "frame": JSON_DOCUMENT_AUDIT,
    },
    {
        "artifact_type": (
            "RUNTIME_CANONICAL_FIELD_BINDINGS"
        ),
        "artifact_name": (
            "runtime_canonical_field_bindings"
        ),
        "frame": (
            RUNTIME_CANONICAL_FIELD_BINDINGS
        ),
    },
    {
        "artifact_type": (
            "CANONICAL_FIELD_AUDIT"
        ),
        "artifact_name": (
            "canonical_field_audit"
        ),
        "frame": (
            CANONICAL_FIELD_AUDIT
        ),
    },
    {
        "artifact_type": (
            "RAW_RECORD_SIGNATURES"
        ),
        "artifact_name": (
            "raw_record_signatures"
        ),
        "frame": (
            RAW_RECORD_SIGNATURES
        ),
    },
    {
        "artifact_type": (
            "GLOBAL_ORDER_AUDIT"
        ),
        "artifact_name": (
            "global_order_audit"
        ),
        "frame": GLOBAL_ORDER_AUDIT,
    },
    {
        "artifact_type": (
            "TIMESTAMP_AUDIT"
        ),
        "artifact_name": (
            "timestamp_audit"
        ),
        "frame": TIMESTAMP_AUDIT,
    },
    {
        "artifact_type": (
            "CLOCK_DIAGNOSTIC_SUMMARY"
        ),
        "artifact_name": (
            "clock_diagnostic_summary"
        ),
        "frame": (
            CLOCK_DIAGNOSTIC_SUMMARY
        ),
    },
    {
        "artifact_type": (
            "TRADE_CLOCK_DIAGNOSTIC"
        ),
        "artifact_name": (
            "trade_clock_diagnostic"
        ),
        "frame": (
            TRADE_CLOCK_DIAGNOSTIC
        ),
    },
    {
        "artifact_type": (
            "PARTITION_ASSIGNMENT_AUDIT"
        ),
        "artifact_name": (
            "partition_assignment_audit"
        ),
        "frame": (
            PARTITION_ASSIGNMENT_AUDIT
        ),
    },
    {
        "artifact_type": (
            "TRADE_PAYLOAD_RECONCILIATION"
        ),
        "artifact_name": (
            "trade_payload_reconciliation"
        ),
        "frame": (
            TRADE_PAYLOAD_RECONCILIATION
        ),
    },
    {
        "artifact_type": (
            "TRADE_IDENTITY_AUDIT"
        ),
        "artifact_name": (
            "trade_identity_audit"
        ),
        "frame": TRADE_IDENTITY_AUDIT,
    },
    {
        "artifact_type": (
            "TRADE_NUMERIC_PROFILE"
        ),
        "artifact_name": (
            "trade_numeric_profile"
        ),
        "frame": TRADE_NUMERIC_PROFILE,
    },
    {
        "artifact_type": (
            "TRADE_AGGRESSOR_SIDE_SUMMARY"
        ),
        "artifact_name": (
            "trade_aggressor_side_summary"
        ),
        "frame": (
            TRADE_AGGRESSOR_SIDE_SUMMARY
        ),
    },
    {
        "artifact_type": (
            "TRADE_PARTITION_RECONCILIATION"
        ),
        "artifact_name": (
            "trade_partition_reconciliation"
        ),
        "frame": (
            TRADE_PARTITION_RECONCILIATION
        ),
    },
    {
        "artifact_type": (
            "DEPTH_PAYLOAD_RECONCILIATION"
        ),
        "artifact_name": (
            "depth_payload_reconciliation"
        ),
        "frame": (
            DEPTH_PAYLOAD_RECONCILIATION
        ),
    },
    {
        "artifact_type": (
            "DEPTH_EVENT_ACTIVITY_SUMMARY"
        ),
        "artifact_name": (
            "depth_event_activity_summary"
        ),
        "frame": (
            DEPTH_EVENT_ACTIVITY_SUMMARY
        ),
    },
    {
        "artifact_type": (
            "DEPTH_LEVEL_PROFILE"
        ),
        "artifact_name": (
            "depth_level_profile"
        ),
        "frame": DEPTH_LEVEL_PROFILE,
    },
    {
        "artifact_type": (
            "DEPTH_UPDATE_ID_AUDIT"
        ),
        "artifact_name": (
            "depth_update_id_audit"
        ),
        "frame": DEPTH_UPDATE_ID_AUDIT,
    },
    {
        "artifact_type": (
            "SNAPSHOT_BUFFER_RECONCILIATION"
        ),
        "artifact_name": (
            "snapshot_buffer_reconciliation"
        ),
        "frame": (
            SNAPSHOT_BUFFER_RECONCILIATION
        ),
    },
    {
        "artifact_type": (
            "DEPTH_PARTITION_RECONCILIATION"
        ),
        "artifact_name": (
            "depth_partition_reconciliation"
        ),
        "frame": (
            DEPTH_PARTITION_RECONCILIATION
        ),
    },
    {
        "artifact_type": (
            "METADATA_DECLARATION_AUDIT"
        ),
        "artifact_name": (
            "metadata_declaration_audit"
        ),
        "frame": (
            METADATA_DECLARATION_AUDIT
        ),
    },
    {
        "artifact_type": (
            "METADATA_CROSS_DOCUMENT_AUDIT"
        ),
        "artifact_name": (
            "metadata_cross_document_audit"
        ),
        "frame": (
            METADATA_CROSS_DOCUMENT_AUDIT
        ),
    },
    {
        "artifact_type": (
            "RAW_REJECTION_LEDGER"
        ),
        "artifact_name": (
            "raw_rejection_ledger"
        ),
        "frame": RAW_REJECTION_LEDGER,
    },
    {
        "artifact_type": (
            "RAW_REJECTION_REASON_SUMMARY"
        ),
        "artifact_name": (
            "raw_rejection_reason_summary"
        ),
        "frame": (
            RAW_REJECTION_REASON_SUMMARY
        ),
    },
    {
        "artifact_type": (
            "NOTEBOOK_01_PRIOR_GATE_RECONCILIATION"
        ),
        "artifact_name": (
            "notebook_01_prior_gate_reconciliation"
        ),
        "frame": (
            NOTEBOOK_01_PRIOR_GATE_RECONCILIATION
        ),
    },
    {
        "artifact_type": (
            "NOTEBOOK_01_PREWRITE_VALIDATION_LEDGER"
        ),
        "artifact_name": (
            "notebook_01_prewrite_validation_ledger"
        ),
        "frame": (
            NOTEBOOK_01_PREWRITE_VALIDATION_LEDGER
        ),
    },
    {
        "artifact_type": (
            "NOTEBOOK_01_PREWRITE_DECISION"
        ),
        "artifact_name": (
            "notebook_01_prewrite_decision"
        ),
        "frame": (
            NOTEBOOK_01_PREWRITE_DECISION
        ),
    },
]


# ============================================================
# VALIDATE OUTPUT SPECIFICATION
# ============================================================

audit_table_artifact_types = [
    specification[
        "artifact_type"
    ]
    for specification in (
        NOTEBOOK_01_AUDIT_TABLE_SPECS
    )
]

audit_table_filenames = [
    notebook_01_artifact_name(
        artifact_name=(
            specification[
                "artifact_name"
            ]
        ),
        extension="csv",
    )
    for specification in (
        NOTEBOOK_01_AUDIT_TABLE_SPECS
    )
]


if (
    len(audit_table_artifact_types)
    != len(
        set(
            audit_table_artifact_types
        )
    )
):
    raise RuntimeError(
        "Duplicate Notebook 01 audit artifact types detected."
    )


if (
    len(audit_table_filenames)
    != len(
        set(audit_table_filenames)
    )
):
    raise RuntimeError(
        "Duplicate Notebook 01 audit filenames detected."
    )


# ============================================================
# WRITE JSON ARTIFACTS
# ============================================================

written_artifact_rows = []


raw_audit_write_result = (
    notebook_01_atomic_write_json(
        path=(
            V0_1_MANIFEST_ROOT_RESOLVED
            / RAW_DATA_AUDIT_ARTIFACT_NAME
        ),
        payload=(
            RAW_DATA_AUDIT_DOCUMENT
        ),
    )
)

written_artifact_rows.append(
    {
        "artifact_type": (
            "V0_1_RAW_DATA_AUDIT"
        ),
        "format": "JSON",
        "acceptance_status": (
            NOTEBOOK_01_PREWRITE_STATUS
        ),
        "row_count": int(
            EXPECTED_COMBINED_STREAM_ROWS
        ),
        **raw_audit_write_result,
    }
)


handoff_write_result = (
    notebook_01_atomic_write_json(
        path=(
            V0_1_HANDOFF_ROOT_RESOLVED
            / NOTEBOOK_01_HANDOFF_ARTIFACT_NAME
        ),
        payload=(
            NOTEBOOK_01_HANDOFF_DOCUMENT
        ),
    )
)

written_artifact_rows.append(
    {
        "artifact_type": (
            "NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF"
        ),
        "format": "JSON",
        "acceptance_status": (
            NOTEBOOK_01_PREWRITE_STATUS
        ),
        "row_count": None,
        **handoff_write_result,
    }
)


# ============================================================
# WRITE AUDIT TABLES
# ============================================================

for specification in (
    NOTEBOOK_01_AUDIT_TABLE_SPECS
):
    filename = (
        notebook_01_artifact_name(
            artifact_name=(
                specification[
                    "artifact_name"
                ]
            ),
            extension="csv",
        )
    )

    write_result = (
        notebook_01_atomic_write_csv(
            path=(
                V0_1_AUDIT_ROOT_RESOLVED
                / filename
            ),
            frame=(
                specification["frame"]
            ),
        )
    )

    written_artifact_rows.append(
        {
            "artifact_type": (
                specification[
                    "artifact_type"
                ]
            ),
            "format": "CSV",
            "acceptance_status": (
                NOTEBOOK_01_PREWRITE_STATUS
            ),
            "row_count": int(
                len(
                    specification[
                        "frame"
                    ]
                )
            ),
            **write_result,
        }
    )


NOTEBOOK_01_WRITTEN_ARTIFACTS = (
    pd.DataFrame(
        written_artifact_rows
    )
)


# ============================================================
# OUTPUT MANIFEST
# ============================================================

NOTEBOOK_01_OUTPUT_MANIFEST_PAYLOAD = {
    "manifest_type": (
        "NOTEBOOK_01_OUTPUT_MANIFEST"
    ),
    "manifest_schema_version": (
        CONTRACT_SCHEMA_VERSION
    ),
    "pipeline_name": PIPELINE_NAME,
    "pipeline_version": (
        PIPELINE_VERSION
    ),
    "source_run_prefix": (
        SOURCE_RUN_PREFIX
    ),
    "source_set_hash": (
        NOTEBOOK_01_SOURCE_SET_HASH
    ),
    "v0_1_run_id": V0_1_RUN_ID,
    "v0_1_run_config_hash": (
        NOTEBOOK_01_RUN_CONFIG_HASH
    ),
    "v0_1_run_identity_hash": (
        NOTEBOOK_01_RUN_IDENTITY_HASH
    ),
    "raw_data_audit_hash": (
        NOTEBOOK_01_RAW_DATA_AUDIT_HASH
    ),
    "notebook_01_handoff_hash": (
        NOTEBOOK_01_HANDOFF_HASH
    ),
    "operating_mode": (
        NOTEBOOK_01_OPERATING_MODE
    ),
    "acceptance_status": (
        NOTEBOOK_01_PREWRITE_STATUS
    ),
    "artifact_count": int(
        len(
            NOTEBOOK_01_WRITTEN_ARTIFACTS
        )
    ),
    "artifacts": frame_records(
        NOTEBOOK_01_WRITTEN_ARTIFACTS
    ),
    "notebook_02_authorized": bool(
        NOTEBOOK_02_AUTHORIZED
    ),
    "next_notebook": (
        NOTEBOOK_02_FILENAME
    ),
    "producing_notebook": (
        NOTEBOOK_FILENAME
    ),
}


NOTEBOOK_01_OUTPUT_MANIFEST_DOCUMENT = (
    notebook_01_wrap_artifact(
        payload=(
            NOTEBOOK_01_OUTPUT_MANIFEST_PAYLOAD
        ),
        artifact_type=(
            "NOTEBOOK_01_OUTPUT_MANIFEST"
        ),
        row_count=len(
            NOTEBOOK_01_WRITTEN_ARTIFACTS
        ),
    )
)


NOTEBOOK_01_OUTPUT_MANIFEST_PATH = (
    V0_1_MANIFEST_ROOT_RESOLVED
    / NOTEBOOK_01_OUTPUT_MANIFEST_NAME
)


manifest_write_result = (
    notebook_01_atomic_write_json(
        path=(
            NOTEBOOK_01_OUTPUT_MANIFEST_PATH
        ),
        payload=(
            NOTEBOOK_01_OUTPUT_MANIFEST_DOCUMENT
        ),
    )
)


NOTEBOOK_01_OUTPUT_MANIFEST_HASH = (
    manifest_write_result[
        "sha256"
    ]
)


# ============================================================
# POST-WRITE VERIFICATION
# ============================================================

postwrite_rows = []


for artifact_row in (
    NOTEBOOK_01_WRITTEN_ARTIFACTS
    .itertuples(index=False)
):
    artifact_path = Path(
        artifact_row.resolved_path
    )

    file_exists = bool(
        artifact_path.exists()
        and artifact_path.is_file()
    )

    if file_exists:
        observed_bytes = (
            artifact_path.read_bytes()
        )

        observed_size_bytes = int(
            len(observed_bytes)
        )

        observed_sha256 = hashlib.sha256(
            observed_bytes
        ).hexdigest()

    else:
        observed_size_bytes = None
        observed_sha256 = None

    expected_size_bytes = int(
        artifact_row.size_bytes
    )

    expected_sha256 = str(
        artifact_row.sha256
    ).lower()

    size_matches = bool(
        observed_size_bytes
        == expected_size_bytes
    )

    hash_matches = bool(
        observed_sha256
        == expected_sha256
    )

    inside_v0_1 = path_is_within(
        artifact_path,
        V0_1_ROOT_RESOLVED,
    )

    inside_v0_0 = path_is_within(
        artifact_path,
        V0_0_ROOT_RESOLVED,
    )

    artifact_passed = bool(
        file_exists
        and size_matches
        and hash_matches
        and inside_v0_1
        and not inside_v0_0
    )

    postwrite_rows.append(
        {
            "artifact_type": (
                artifact_row.artifact_type
            ),
            "format": artifact_row.format,
            "relative_path": (
                artifact_row.relative_path
            ),
            "file_exists": file_exists,
            "inside_v0_1": inside_v0_1,
            "inside_v0_0": inside_v0_0,
            "expected_size_bytes": (
                expected_size_bytes
            ),
            "observed_size_bytes": (
                observed_size_bytes
            ),
            "size_matches": (
                size_matches
            ),
            "expected_sha256": (
                expected_sha256
            ),
            "observed_sha256": (
                observed_sha256
            ),
            "hash_matches": hash_matches,
            "artifact_passed": (
                artifact_passed
            ),
        }
    )


NOTEBOOK_01_POSTWRITE_VERIFICATION = (
    pd.DataFrame(
        postwrite_rows
    )
)


manifest_path = Path(
    manifest_write_result[
        "resolved_path"
    ]
)

manifest_bytes = (
    manifest_path.read_bytes()
)

manifest_size_verified = bool(
    len(manifest_bytes)
    == int(
        manifest_write_result[
            "size_bytes"
        ]
    )
)

manifest_hash_verified = bool(
    hashlib.sha256(
        manifest_bytes
    ).hexdigest()
    == NOTEBOOK_01_OUTPUT_MANIFEST_HASH
)

postwrite_artifacts_passed = bool(
    not NOTEBOOK_01_POSTWRITE_VERIFICATION.empty
    and NOTEBOOK_01_POSTWRITE_VERIFICATION[
        "artifact_passed"
    ].all()
)

NOTEBOOK_01_POSTWRITE_PASSED = bool(
    postwrite_artifacts_passed
    and manifest_size_verified
    and manifest_hash_verified
)


# ============================================================
# DISPLAY AND ENFORCE
# ============================================================

display(
    NOTEBOOK_01_WRITTEN_ARTIFACTS
)

display(
    NOTEBOOK_01_POSTWRITE_VERIFICATION[
        [
            "artifact_type",
            "format",
            "relative_path",
            "file_exists",
            "size_matches",
            "hash_matches",
            "artifact_passed",
        ]
    ]
)


if not NOTEBOOK_01_POSTWRITE_PASSED:
    failed_artifacts = (
        NOTEBOOK_01_POSTWRITE_VERIFICATION.loc[
            ~NOTEBOOK_01_POSTWRITE_VERIFICATION[
                "artifact_passed"
            ],
            "relative_path",
        ]
        .astype(str)
        .tolist()
    )

    failure_text = (
        " | ".join(
            failed_artifacts
        )
        if failed_artifacts
        else (
            "output manifest verification failed"
        )
    )

    raise RuntimeError(
        "Notebook 01 post-write verification failed: "
        + failure_text
    )


print("Notebook 01 atomic artifact writes: PASS")
print(
    "Authoritative artifacts written: "
    f"{len(NOTEBOOK_01_WRITTEN_ARTIFACTS):,}"
)
print(
    "Raw-data audit artifact: "
    f"{raw_audit_write_result['relative_path']}"
)
print(
    "Notebook 02 handoff artifact: "
    f"{handoff_write_result['relative_path']}"
)
print(
    "Output manifest: "
    f"{manifest_write_result['relative_path']}"
)
print(
    "Raw-data audit SHA-256: "
    f"{NOTEBOOK_01_RAW_DATA_AUDIT_HASH}"
)
print(
    "Handoff SHA-256: "
    f"{NOTEBOOK_01_HANDOFF_HASH}"
)
print(
    "Output manifest SHA-256: "
    f"{NOTEBOOK_01_OUTPUT_MANIFEST_HASH}"
)
print(
    "Post-write verification: "
    f"{NOTEBOOK_01_POSTWRITE_PASSED}"
)
print(
    "Notebook 01 status: "
    f"{NOTEBOOK_01_PREWRITE_STATUS}"
)
print(
    "Next notebook: "
    f"{NOTEBOOK_02_FILENAME}"
)

,artifact_type,format,acceptance_status,row_count,resolved_path,relative_path,size_bytes,sha256
0,V0_1_RAW_DATA_AUDIT,JSON,CONDITIONAL PASS,103677.0,D:\Clown Project\V0.1\artifacts\manifests\BTCU...,artifacts\manifests\BTCUSDT_spot_20260710T0637...,70251,a979107c7825e75d5ba151c6519c4b61b500695daaf803...
1,NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF,JSON,CONDITIONAL PASS,NaN,D:\Clown Project\V0.1\artifacts\handoff\BTCUSD...,artifacts\handoff\BTCUSDT_spot_20260710T063746...,8367,138704253d8821259a7be427620a11b1d469a7f009c13c...
2,AUTHORITATIVE_RAW_SOURCE_MANIFEST,CSV,CONDITIONAL PASS,5.0,D:\Clown Project\V0.1\artifacts\audit_tables\B...,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,2926,7c130c5b9d0e2861948a74a0ecc63bdad9d49309c3fe61...
3,NOTEBOOK_01_SOURCE_REHASH,CSV,CONDITIONAL PASS,5.0,D:\Clown Project\V0.1\artifacts\audit_tables\B...,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,1604,9c9c3563633ad934ba72a6fc3edfe42e9d1bc2e3752b2b...
4,JSONL_PARSE_AUDIT,CSV,CONDITIONAL PASS,2.0,D:\Clown Project\V0.1\artifacts\audit_tables\B...,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,1326,26eeeb2f1f576bc5d6b1608931e82213d079f137e40854...
5,JSON_DOCUMENT_AUDIT,CSV,CONDITIONAL PASS,3.0,D:\Clown Project\V0.1\artifacts\audit_tables\B...,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,2053,3c79ff2001245ada90f3e793f960dc7f9debd4354d0f44...
6,RUNTIME_CANONICAL_FIELD_BINDINGS,CSV,CONDITIONAL PASS,21.0,D:\Clown Project\V0.1\artifacts\audit_tables\B...,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,2644,910dfc68fef280519652164bc955c0fe2b8e768197c30b...
7,CANONICAL_FIELD_AUDIT,CSV,CONDITIONAL PASS,21.0,D:\Clown Project\V0.1\artifacts\audit_tables\B...,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,2627,a4d241ef3e053080574bfbda068868021d9eaf205b791e...
8,RAW_RECORD_SIGNATURES,CSV,CONDITIONAL PASS,6.0,D:\Clown Project\V0.1\artifacts\audit_tables\B...,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,870,49d26e349410af7d08ade69d85604c12a8cb1600d49fa4...
9,GLOBAL_ORDER_AUDIT,CSV,CONDITIONAL PASS,1.0,D:\Clown Project\V0.1\artifacts\audit_tables\B...,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,337,30f1f47c0fb09173cc8ce4707055712bed0f26bd8eb24c...


,artifact_type,format,relative_path,file_exists,size_matches,hash_matches,artifact_passed
0,V0_1_RAW_DATA_AUDIT,JSON,artifacts\manifests\BTCUSDT_spot_20260710T0637...,True,True,True,True
1,NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF,JSON,artifacts\handoff\BTCUSDT_spot_20260710T063746...,True,True,True,True
2,AUTHORITATIVE_RAW_SOURCE_MANIFEST,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True
3,NOTEBOOK_01_SOURCE_REHASH,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True
4,JSONL_PARSE_AUDIT,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True
5,JSON_DOCUMENT_AUDIT,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True
6,RUNTIME_CANONICAL_FIELD_BINDINGS,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True
7,CANONICAL_FIELD_AUDIT,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True
8,RAW_RECORD_SIGNATURES,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True
9,GLOBAL_ORDER_AUDIT,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True


Notebook 01 atomic artifact writes: PASS
Authoritative artifacts written: 32
Raw-data audit artifact: artifacts\manifests\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__01_RAW_DATA_AUDIT__v0_1_raw_data_audit.json
Notebook 02 handoff artifact: artifacts\handoff\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__01_RAW_DATA_AUDIT__notebook_01_to_notebook_02_handoff.json
Output manifest: artifacts\manifests\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__01_RAW_DATA_AUDIT__notebook_01_output_manifest.json
Raw-data audit SHA-256: fdfd1bc3ca907b1bf14286ca8f849d49a997bd5655f8793260838a34b1c9f56d
Handoff SHA-256: 80f3e2df95b6f2daebdf7b85324cbfcabdb64d22740e328962d1af078420b2e1
Output manifest SHA-256: 2bb561126bbfc20fd25aa80131c1c11307e651d38ed3d6ee26a11b2ff0f41a4f
Post-write verification: True
Notebook 01 status: CONDITIONAL PASS
Next notebook: 02_VISIBLE_BOOK_RECONSTRUCTION.ipynb


In [43]:
# ============================================================
# NOTEBOOK 01 ATOMIC ARTIFACT WRITES
# ============================================================

import os
import re
import json
import hashlib
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# OUTPUT DIRECTORIES
# ============================================================

V0_1_MANIFEST_ROOT_RESOLVED = Path(
    V0_1_MANIFEST_ROOT
).resolve(strict=False)

V0_1_AUDIT_ROOT_RESOLVED = Path(
    V0_1_AUDIT_ROOT
).resolve(strict=False)

V0_1_HANDOFF_ROOT_RESOLVED = (
    Path(V0_1_ROOT_RESOLVED)
    / "artifacts"
    / "handoff"
).resolve(strict=False)


for output_root in (
    V0_1_MANIFEST_ROOT_RESOLVED,
    V0_1_AUDIT_ROOT_RESOLVED,
    V0_1_HANDOFF_ROOT_RESOLVED,
):
    output_root.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# PATH AND NAME HELPERS
# ============================================================

def notebook_01_sanitize_token(
    value,
) -> str:
    token = re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        str(value).strip(),
    ).strip("._-")

    if not token:
        raise ValueError(
            "Artifact token cannot be empty."
        )

    return token


NOTEBOOK_01_OUTPUT_PREFIX = "__".join(
    [
        notebook_01_sanitize_token(
            SOURCE_RUN_PREFIX
        ),
        notebook_01_sanitize_token(
            V0_1_RUN_ID
        ),
        notebook_01_sanitize_token(
            Path(NOTEBOOK_FILENAME).stem
        ),
    ]
)


def notebook_01_artifact_name(
    artifact_name: str,
    extension: str,
) -> str:
    safe_artifact_name = (
        notebook_01_sanitize_token(
            artifact_name
        )
    )

    safe_extension = (
        notebook_01_sanitize_token(
            extension.lstrip(".")
        )
    )

    return (
        NOTEBOOK_01_OUTPUT_PREFIX
        + "__"
        + safe_artifact_name
        + "."
        + safe_extension
    )


def notebook_01_validate_output_path(
    path,
) -> Path:
    resolved_path = Path(path).resolve(
        strict=False
    )

    if not path_is_within(
        resolved_path,
        V0_1_ROOT_RESOLVED,
    ):
        raise RuntimeError(
            "Output path is outside V0.1: "
            + str(resolved_path)
        )

    if path_is_within(
        resolved_path,
        V0_0_ROOT_RESOLVED,
    ):
        raise RuntimeError(
            "Output path enters immutable V0.0: "
            + str(resolved_path)
        )

    return resolved_path


def registered_source_path(
    source_role: str,
) -> Path:
    matches = (
        AUTHORITATIVE_RAW_SOURCE_MANIFEST.loc[
            AUTHORITATIVE_RAW_SOURCE_MANIFEST[
                "source_role"
            ].eq(source_role)
        ]
    )

    if len(matches) != 1:
        raise RuntimeError(
            "Expected one source for role "
            + source_role
            + "; found "
            + str(len(matches))
            + "."
        )

    return Path(
        matches.iloc[0][
            "resolved_path"
        ]
    ).resolve(strict=True)


NOTEBOOK_01_TRADE_SOURCE_PATH = (
    registered_source_path(
        "TRADE_STREAM"
    )
)

NOTEBOOK_01_DEPTH_SOURCE_PATH = (
    registered_source_path(
        "DEPTH_STREAM"
    )
)

NOTEBOOK_01_SNAPSHOT_SOURCE_PATH = (
    registered_source_path(
        "REST_SNAPSHOT"
    )
)


# ============================================================
# SERIALIZATION HELPERS
# ============================================================

def notebook_01_pretty_json_bytes(
    value,
) -> bytes:
    normalized = notebook_01_json_safe(
        value
    )

    text = json.dumps(
        normalized,
        sort_keys=True,
        ensure_ascii=False,
        allow_nan=False,
        indent=2,
    )

    return (text + "\n").encode(
        "utf-8"
    )


def notebook_01_csv_bytes(
    frame: pd.DataFrame,
) -> bytes:
    if not isinstance(
        frame,
        pd.DataFrame,
    ):
        raise TypeError(
            "CSV artifact source must be a DataFrame."
        )

    text = frame.to_csv(
        index=False,
        na_rep="",
        lineterminator="\n",
        date_format="%Y-%m-%dT%H:%M:%S.%f%z",
    )

    return text.encode("utf-8")


def notebook_01_atomic_write_bytes(
    path,
    content: bytes,
) -> dict:
    target_path = (
        notebook_01_validate_output_path(
            path
        )
    )

    target_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = None

    try:
        with tempfile.NamedTemporaryFile(
            mode="wb",
            prefix="." + target_path.name + ".",
            suffix=".tmp",
            dir=target_path.parent,
            delete=False,
        ) as temporary_handle:
            temporary_path = Path(
                temporary_handle.name
            )

            temporary_handle.write(
                content
            )

            temporary_handle.flush()

            os.fsync(
                temporary_handle.fileno()
            )

        os.replace(
            temporary_path,
            target_path,
        )

    finally:
        if (
            temporary_path is not None
            and temporary_path.exists()
        ):
            temporary_path.unlink()

    written_bytes = target_path.read_bytes()

    return {
        "resolved_path": str(
            target_path
        ),
        "relative_path": str(
            target_path.relative_to(
                V0_1_ROOT_RESOLVED
            )
        ),
        "size_bytes": int(
            len(written_bytes)
        ),
        "sha256": hashlib.sha256(
            written_bytes
        ).hexdigest(),
    }


def notebook_01_atomic_write_json(
    path,
    payload,
) -> dict:
    return notebook_01_atomic_write_bytes(
        path=path,
        content=(
            notebook_01_pretty_json_bytes(
                payload
            )
        ),
    )


def notebook_01_atomic_write_csv(
    path,
    frame: pd.DataFrame,
) -> dict:
    return notebook_01_atomic_write_bytes(
        path=path,
        content=(
            notebook_01_csv_bytes(
                frame
            )
        ),
    )


# ============================================================
# ARTIFACT WRAPPER
# ============================================================

def notebook_01_wrap_artifact(
    payload,
    artifact_type: str,
    row_count=None,
) -> dict:
    safe_payload = notebook_01_json_safe(
        payload
    )

    metadata = {
        "artifact_type": (
            artifact_type
        ),
        "artifact_schema_version": (
            CONTRACT_SCHEMA_VERSION
        ),
        "pipeline_name": PIPELINE_NAME,
        "pipeline_version": (
            PIPELINE_VERSION
        ),
        "source_run_prefix": (
            SOURCE_RUN_PREFIX
        ),
        "source_set_hash": (
            NOTEBOOK_01_SOURCE_SET_HASH
        ),
        "v0_1_run_id": V0_1_RUN_ID,
        "v0_1_run_config_hash": (
            NOTEBOOK_01_RUN_CONFIG_HASH
        ),
        "v0_1_run_identity_hash": (
            NOTEBOOK_01_RUN_IDENTITY_HASH
        ),
        "producing_notebook": (
            NOTEBOOK_FILENAME
        ),
        "created_utc": (
            str(NOTEBOOK_01_STARTED_UTC)
        ),
        "acceptance_status": (
            NOTEBOOK_01_PREWRITE_STATUS
        ),
        "row_count": (
            int(row_count)
            if row_count is not None
            else None
        ),
        "payload_sha256": (
            notebook_01_object_hash(
                safe_payload
            )
        ),
    }

    return {
        "artifact_metadata": metadata,
        "payload": safe_payload,
    }


# ============================================================
# NOTEBOOK 02 HANDOFF
# ============================================================

NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF = {
    "handoff_type": (
        "NOTEBOOK_01_TO_NOTEBOOK_02"
    ),
    "handoff_schema_version": (
        CONTRACT_SCHEMA_VERSION
    ),
    "pipeline_name": PIPELINE_NAME,
    "pipeline_version": (
        PIPELINE_VERSION
    ),
    "source_run_prefix": (
        SOURCE_RUN_PREFIX
    ),
    "source_set_hash": (
        NOTEBOOK_01_SOURCE_SET_HASH
    ),
    "v0_1_run_id": V0_1_RUN_ID,
    "v0_1_run_config_hash": (
        NOTEBOOK_01_RUN_CONFIG_HASH
    ),
    "v0_1_run_identity_hash": (
        NOTEBOOK_01_RUN_IDENTITY_HASH
    ),
    "raw_data_audit_hash": (
        NOTEBOOK_01_RAW_DATA_AUDIT_HASH
    ),
    "operating_mode": (
        NOTEBOOK_01_OPERATING_MODE
    ),
    "acceptance_status": (
        NOTEBOOK_01_PREWRITE_STATUS
    ),
    "notebook_02_authorized": bool(
        NOTEBOOK_02_AUTHORIZED
    ),
    "next_notebook": (
        NOTEBOOK_02_FILENAME
    ),
    "authoritative_raw_sources": (
        frame_records(
            AUTHORITATIVE_RAW_SOURCE_MANIFEST[
                [
                    "source_role",
                    "resolved_path",
                    "size_bytes",
                    "sha256",
                ]
            ]
        )
    ),
    "snapshot_contract": {
        "snapshot_path": str(
            NOTEBOOK_01_SNAPSHOT_SOURCE_PATH
        ),
        "snapshot_last_update_id": int(
            SNAPSHOT_LAST_UPDATE_ID
        ),
        "snapshot_session_id": str(
            SNAPSHOT_SESSION_ID
        ),
        "snapshot_response_received_time_ns": (
            int(
                SNAPSHOT_RESPONSE_RECEIVED_TIME_NS
            )
            if SNAPSHOT_RESPONSE_RECEIVED_TIME_NS
            is not None
            else None
        ),
    },
    "depth_reconstruction_contract": {
        "depth_source_path": str(
            NOTEBOOK_01_DEPTH_SOURCE_PATH
        ),
        "total_depth_event_count": int(
            len(
                RAW_DEPTH_EVENT_AUDIT_TABLE
            )
        ),
        "stale_pre_snapshot_event_count": int(
            stale_event_count
        ),
        "first_applicable_depth_position": int(
            first_post_snapshot_position
        ),
        "first_applicable_collector_sequence": int(
            first_post_snapshot_sequence
        ),
        "first_applicable_U": int(
            first_post_snapshot_U
        ),
        "first_applicable_u": int(
            first_post_snapshot_u
        ),
        "post_snapshot_event_count": int(
            post_bridge_event_count
        ),
        "post_snapshot_gap_count": int(
            post_bridge_gap_count
        ),
        "post_snapshot_overlap_count": int(
            post_bridge_overlap_count
        ),
        "zero_quantity_action": (
            "DELETE_PRICE_LEVEL"
        ),
        "ordering_authority": (
            "collector_sequence"
        ),
        "timezone": "UTC",
    },
    "partition_contract": (
        frame_records(
            FROZEN_PARTITION_BOUNDARIES
        )
    ),
    "raw_audit_summary": (
        NOTEBOOK_01_RECONCILED_COUNTS
    ),
    "raw_record_rejection_count": int(
        len(RAW_REJECTION_LEDGER)
    ),
    "claim_authority": {
        "claim_bearing_holdout_available": bool(
            CHRONOLOGICAL_SPLIT_CONTRACT.get(
                "claim_bearing_holdout_available",
                False,
            )
        ),
        "partitions_are_independent": bool(
            CHRONOLOGICAL_SPLIT_CONTRACT.get(
                "partitions_are_independent",
                False,
            )
        ),
    },
    "producing_notebook": (
        NOTEBOOK_FILENAME
    ),
}


NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF = (
    notebook_01_json_safe(
        NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF
    )
)

NOTEBOOK_01_HANDOFF_HASH = (
    notebook_01_object_hash(
        NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF
    )
)


# ============================================================
# JSON DOCUMENTS
# ============================================================

RAW_DATA_AUDIT_DOCUMENT = (
    notebook_01_wrap_artifact(
        payload=(
            NOTEBOOK_01_RAW_DATA_AUDIT
        ),
        artifact_type=(
            "V0_1_RAW_DATA_AUDIT"
        ),
        row_count=(
            EXPECTED_COMBINED_STREAM_ROWS
        ),
    )
)

NOTEBOOK_01_HANDOFF_DOCUMENT = (
    notebook_01_wrap_artifact(
        payload=(
            NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF
        ),
        artifact_type=(
            "NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF"
        ),
    )
)


# ============================================================
# CSV ARTIFACT SPECIFICATION
# ============================================================

NOTEBOOK_01_AUDIT_TABLE_SPECS = [
    (
        "AUTHORITATIVE_RAW_SOURCE_MANIFEST",
        "authoritative_raw_source_manifest",
        AUTHORITATIVE_RAW_SOURCE_MANIFEST,
    ),
    (
        "NOTEBOOK_01_SOURCE_REHASH",
        "notebook_01_source_rehash",
        NOTEBOOK_01_SOURCE_REHASH,
    ),
    (
        "JSONL_PARSE_AUDIT",
        "jsonl_parse_audit",
        JSONL_PARSE_AUDIT,
    ),
    (
        "JSON_DOCUMENT_AUDIT",
        "json_document_audit",
        JSON_DOCUMENT_AUDIT,
    ),
    (
        "RUNTIME_CANONICAL_FIELD_BINDINGS",
        "runtime_canonical_field_bindings",
        RUNTIME_CANONICAL_FIELD_BINDINGS,
    ),
    (
        "CANONICAL_FIELD_AUDIT",
        "canonical_field_audit",
        CANONICAL_FIELD_AUDIT,
    ),
    (
        "RAW_RECORD_SIGNATURES",
        "raw_record_signatures",
        RAW_RECORD_SIGNATURES,
    ),
    (
        "GLOBAL_ORDER_AUDIT",
        "global_order_audit",
        GLOBAL_ORDER_AUDIT,
    ),
    (
        "TIMESTAMP_AUDIT",
        "timestamp_audit",
        TIMESTAMP_AUDIT,
    ),
    (
        "CLOCK_DIAGNOSTIC_SUMMARY",
        "clock_diagnostic_summary",
        CLOCK_DIAGNOSTIC_SUMMARY,
    ),
    (
        "TRADE_CLOCK_DIAGNOSTIC",
        "trade_clock_diagnostic",
        TRADE_CLOCK_DIAGNOSTIC,
    ),
    (
        "PARTITION_ASSIGNMENT_AUDIT",
        "partition_assignment_audit",
        PARTITION_ASSIGNMENT_AUDIT,
    ),
    (
        "TRADE_PAYLOAD_RECONCILIATION",
        "trade_payload_reconciliation",
        TRADE_PAYLOAD_RECONCILIATION,
    ),
    (
        "TRADE_IDENTITY_AUDIT",
        "trade_identity_audit",
        TRADE_IDENTITY_AUDIT,
    ),
    (
        "TRADE_NUMERIC_PROFILE",
        "trade_numeric_profile",
        TRADE_NUMERIC_PROFILE,
    ),
    (
        "TRADE_AGGRESSOR_SIDE_SUMMARY",
        "trade_aggressor_side_summary",
        TRADE_AGGRESSOR_SIDE_SUMMARY,
    ),
    (
        "TRADE_PARTITION_RECONCILIATION",
        "trade_partition_reconciliation",
        TRADE_PARTITION_RECONCILIATION,
    ),
    (
        "DEPTH_PAYLOAD_RECONCILIATION",
        "depth_payload_reconciliation",
        DEPTH_PAYLOAD_RECONCILIATION,
    ),
    (
        "DEPTH_EVENT_ACTIVITY_SUMMARY",
        "depth_event_activity_summary",
        DEPTH_EVENT_ACTIVITY_SUMMARY,
    ),
    (
        "DEPTH_LEVEL_PROFILE",
        "depth_level_profile",
        DEPTH_LEVEL_PROFILE,
    ),
    (
        "DEPTH_UPDATE_ID_AUDIT",
        "depth_update_id_audit",
        DEPTH_UPDATE_ID_AUDIT,
    ),
    (
        "SNAPSHOT_BUFFER_RECONCILIATION",
        "snapshot_buffer_reconciliation",
        SNAPSHOT_BUFFER_RECONCILIATION,
    ),
    (
        "DEPTH_PARTITION_RECONCILIATION",
        "depth_partition_reconciliation",
        DEPTH_PARTITION_RECONCILIATION,
    ),
    (
        "METADATA_DECLARATION_AUDIT",
        "metadata_declaration_audit",
        METADATA_DECLARATION_AUDIT,
    ),
    (
        "METADATA_CROSS_DOCUMENT_AUDIT",
        "metadata_cross_document_audit",
        METADATA_CROSS_DOCUMENT_AUDIT,
    ),
    (
        "RAW_REJECTION_LEDGER",
        "raw_rejection_ledger",
        RAW_REJECTION_LEDGER,
    ),
    (
        "RAW_REJECTION_REASON_SUMMARY",
        "raw_rejection_reason_summary",
        RAW_REJECTION_REASON_SUMMARY,
    ),
    (
        "NOTEBOOK_01_PRIOR_GATE_RECONCILIATION",
        "notebook_01_prior_gate_reconciliation",
        NOTEBOOK_01_PRIOR_GATE_RECONCILIATION,
    ),
    (
        "NOTEBOOK_01_PREWRITE_VALIDATION_LEDGER",
        "notebook_01_prewrite_validation_ledger",
        NOTEBOOK_01_PREWRITE_VALIDATION_LEDGER,
    ),
    (
        "NOTEBOOK_01_PREWRITE_DECISION",
        "notebook_01_prewrite_decision",
        NOTEBOOK_01_PREWRITE_DECISION,
    ),
]


artifact_types = [
    item[0]
    for item in (
        NOTEBOOK_01_AUDIT_TABLE_SPECS
    )
]

artifact_filenames = [
    notebook_01_artifact_name(
        item[1],
        "csv",
    )
    for item in (
        NOTEBOOK_01_AUDIT_TABLE_SPECS
    )
]


if len(artifact_types) != len(
    set(artifact_types)
):
    raise RuntimeError(
        "Duplicate CSV artifact types detected."
    )


if len(artifact_filenames) != len(
    set(artifact_filenames)
):
    raise RuntimeError(
        "Duplicate CSV artifact filenames detected."
    )


# ============================================================
# WRITE AUTHORITATIVE ARTIFACTS
# ============================================================

written_artifact_rows = []


RAW_DATA_AUDIT_ARTIFACT_NAME = (
    notebook_01_artifact_name(
        "v0_1_raw_data_audit",
        "json",
    )
)

NOTEBOOK_01_HANDOFF_ARTIFACT_NAME = (
    notebook_01_artifact_name(
        "notebook_01_to_notebook_02_handoff",
        "json",
    )
)

NOTEBOOK_01_OUTPUT_MANIFEST_NAME = (
    notebook_01_artifact_name(
        "notebook_01_output_manifest",
        "json",
    )
)


raw_audit_write_result = (
    notebook_01_atomic_write_json(
        path=(
            V0_1_MANIFEST_ROOT_RESOLVED
            / RAW_DATA_AUDIT_ARTIFACT_NAME
        ),
        payload=(
            RAW_DATA_AUDIT_DOCUMENT
        ),
    )
)

written_artifact_rows.append(
    {
        "artifact_type": (
            "V0_1_RAW_DATA_AUDIT"
        ),
        "format": "JSON",
        "acceptance_status": (
            NOTEBOOK_01_PREWRITE_STATUS
        ),
        "row_count": int(
            EXPECTED_COMBINED_STREAM_ROWS
        ),
        **raw_audit_write_result,
    }
)


handoff_write_result = (
    notebook_01_atomic_write_json(
        path=(
            V0_1_HANDOFF_ROOT_RESOLVED
            / NOTEBOOK_01_HANDOFF_ARTIFACT_NAME
        ),
        payload=(
            NOTEBOOK_01_HANDOFF_DOCUMENT
        ),
    )
)

written_artifact_rows.append(
    {
        "artifact_type": (
            "NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF"
        ),
        "format": "JSON",
        "acceptance_status": (
            NOTEBOOK_01_PREWRITE_STATUS
        ),
        "row_count": None,
        **handoff_write_result,
    }
)


for (
    artifact_type,
    artifact_name,
    artifact_frame,
) in NOTEBOOK_01_AUDIT_TABLE_SPECS:
    csv_filename = (
        notebook_01_artifact_name(
            artifact_name,
            "csv",
        )
    )

    csv_write_result = (
        notebook_01_atomic_write_csv(
            path=(
                V0_1_AUDIT_ROOT_RESOLVED
                / csv_filename
            ),
            frame=artifact_frame,
        )
    )

    written_artifact_rows.append(
        {
            "artifact_type": (
                artifact_type
            ),
            "format": "CSV",
            "acceptance_status": (
                NOTEBOOK_01_PREWRITE_STATUS
            ),
            "row_count": int(
                len(artifact_frame)
            ),
            **csv_write_result,
        }
    )


NOTEBOOK_01_WRITTEN_ARTIFACTS = (
    pd.DataFrame(
        written_artifact_rows
    )
)


# ============================================================
# WRITE OUTPUT MANIFEST
# ============================================================

NOTEBOOK_01_OUTPUT_MANIFEST_PAYLOAD = {
    "manifest_type": (
        "NOTEBOOK_01_OUTPUT_MANIFEST"
    ),
    "manifest_schema_version": (
        CONTRACT_SCHEMA_VERSION
    ),
    "pipeline_name": PIPELINE_NAME,
    "pipeline_version": (
        PIPELINE_VERSION
    ),
    "source_run_prefix": (
        SOURCE_RUN_PREFIX
    ),
    "source_set_hash": (
        NOTEBOOK_01_SOURCE_SET_HASH
    ),
    "v0_1_run_id": V0_1_RUN_ID,
    "v0_1_run_config_hash": (
        NOTEBOOK_01_RUN_CONFIG_HASH
    ),
    "v0_1_run_identity_hash": (
        NOTEBOOK_01_RUN_IDENTITY_HASH
    ),
    "raw_data_audit_hash": (
        NOTEBOOK_01_RAW_DATA_AUDIT_HASH
    ),
    "notebook_01_handoff_hash": (
        NOTEBOOK_01_HANDOFF_HASH
    ),
    "operating_mode": (
        NOTEBOOK_01_OPERATING_MODE
    ),
    "acceptance_status": (
        NOTEBOOK_01_PREWRITE_STATUS
    ),
    "artifact_count": int(
        len(
            NOTEBOOK_01_WRITTEN_ARTIFACTS
        )
    ),
    "artifacts": frame_records(
        NOTEBOOK_01_WRITTEN_ARTIFACTS
    ),
    "notebook_02_authorized": bool(
        NOTEBOOK_02_AUTHORIZED
    ),
    "next_notebook": (
        NOTEBOOK_02_FILENAME
    ),
    "producing_notebook": (
        NOTEBOOK_FILENAME
    ),
}


NOTEBOOK_01_OUTPUT_MANIFEST_DOCUMENT = (
    notebook_01_wrap_artifact(
        payload=(
            NOTEBOOK_01_OUTPUT_MANIFEST_PAYLOAD
        ),
        artifact_type=(
            "NOTEBOOK_01_OUTPUT_MANIFEST"
        ),
        row_count=int(
            len(
                NOTEBOOK_01_WRITTEN_ARTIFACTS
            )
        ),
    )
)


NOTEBOOK_01_OUTPUT_MANIFEST_PATH = (
    V0_1_MANIFEST_ROOT_RESOLVED
    / NOTEBOOK_01_OUTPUT_MANIFEST_NAME
)


manifest_write_result = (
    notebook_01_atomic_write_json(
        path=(
            NOTEBOOK_01_OUTPUT_MANIFEST_PATH
        ),
        payload=(
            NOTEBOOK_01_OUTPUT_MANIFEST_DOCUMENT
        ),
    )
)


NOTEBOOK_01_OUTPUT_MANIFEST_HASH = str(
    manifest_write_result[
        "sha256"
    ]
)


# ============================================================
# POST-WRITE VERIFICATION
# ============================================================

postwrite_rows = []


for artifact_row in (
    NOTEBOOK_01_WRITTEN_ARTIFACTS
    .itertuples(index=False)
):
    artifact_path = Path(
        artifact_row.resolved_path
    )

    file_exists = bool(
        artifact_path.exists()
        and artifact_path.is_file()
    )

    observed_size_bytes = None
    observed_sha256 = None

    if file_exists:
        observed_bytes = (
            artifact_path.read_bytes()
        )

        observed_size_bytes = int(
            len(observed_bytes)
        )

        observed_sha256 = (
            hashlib.sha256(
                observed_bytes
            ).hexdigest()
        )

    expected_size_bytes = int(
        artifact_row.size_bytes
    )

    expected_sha256 = str(
        artifact_row.sha256
    ).lower()

    size_matches = bool(
        observed_size_bytes
        == expected_size_bytes
    )

    hash_matches = bool(
        observed_sha256
        == expected_sha256
    )

    inside_v0_1 = path_is_within(
        artifact_path,
        V0_1_ROOT_RESOLVED,
    )

    inside_v0_0 = path_is_within(
        artifact_path,
        V0_0_ROOT_RESOLVED,
    )

    artifact_passed = bool(
        file_exists
        and size_matches
        and hash_matches
        and inside_v0_1
        and not inside_v0_0
    )

    postwrite_rows.append(
        {
            "artifact_type": (
                artifact_row.artifact_type
            ),
            "format": (
                artifact_row.format
            ),
            "relative_path": (
                artifact_row.relative_path
            ),
            "file_exists": file_exists,
            "inside_v0_1": inside_v0_1,
            "inside_v0_0": inside_v0_0,
            "expected_size_bytes": (
                expected_size_bytes
            ),
            "observed_size_bytes": (
                observed_size_bytes
            ),
            "size_matches": (
                size_matches
            ),
            "expected_sha256": (
                expected_sha256
            ),
            "observed_sha256": (
                observed_sha256
            ),
            "hash_matches": (
                hash_matches
            ),
            "artifact_passed": (
                artifact_passed
            ),
        }
    )


NOTEBOOK_01_POSTWRITE_VERIFICATION = (
    pd.DataFrame(
        postwrite_rows
    )
)


manifest_path = Path(
    manifest_write_result[
        "resolved_path"
    ]
)

manifest_bytes = (
    manifest_path.read_bytes()
)

manifest_size_verified = bool(
    len(manifest_bytes)
    == int(
        manifest_write_result[
            "size_bytes"
        ]
    )
)

manifest_hash_verified = bool(
    hashlib.sha256(
        manifest_bytes
    ).hexdigest()
    == NOTEBOOK_01_OUTPUT_MANIFEST_HASH
)

postwrite_artifacts_passed = bool(
    len(
        NOTEBOOK_01_POSTWRITE_VERIFICATION
    )
    == len(
        NOTEBOOK_01_WRITTEN_ARTIFACTS
    )
    and NOTEBOOK_01_POSTWRITE_VERIFICATION[
        "artifact_passed"
    ].all()
)

NOTEBOOK_01_POSTWRITE_PASSED = bool(
    postwrite_artifacts_passed
    and manifest_size_verified
    and manifest_hash_verified
)


# ============================================================
# DISPLAY AND ENFORCE
# ============================================================

display(
    NOTEBOOK_01_WRITTEN_ARTIFACTS
)

display(
    NOTEBOOK_01_POSTWRITE_VERIFICATION[
        [
            "artifact_type",
            "format",
            "relative_path",
            "file_exists",
            "size_matches",
            "hash_matches",
            "artifact_passed",
        ]
    ]
)


if not NOTEBOOK_01_POSTWRITE_PASSED:
    failed_paths = (
        NOTEBOOK_01_POSTWRITE_VERIFICATION.loc[
            ~NOTEBOOK_01_POSTWRITE_VERIFICATION[
                "artifact_passed"
            ],
            "relative_path",
        ]
        .astype(str)
        .tolist()
    )

    if failed_paths:
        failure_text = " | ".join(
            failed_paths
        )

    else:
        failure_text = (
            "output manifest verification failed"
        )

    raise RuntimeError(
        "Notebook 01 post-write verification failed: "
        + failure_text
    )


print(
    "Notebook 01 atomic artifact writes: PASS"
)

print(
    "Authoritative artifacts written: "
    + format(
        len(
            NOTEBOOK_01_WRITTEN_ARTIFACTS
        ),
        ",",
    )
)

print(
    "Raw-data audit artifact: "
    + raw_audit_write_result[
        "relative_path"
    ]
)

print(
    "Notebook 02 handoff artifact: "
    + handoff_write_result[
        "relative_path"
    ]
)

print(
    "Output manifest: "
    + manifest_write_result[
        "relative_path"
    ]
)

print(
    "Raw-data audit SHA-256: "
    + NOTEBOOK_01_RAW_DATA_AUDIT_HASH
)

print(
    "Handoff SHA-256: "
    + NOTEBOOK_01_HANDOFF_HASH
)

print(
    "Output manifest SHA-256: "
    + NOTEBOOK_01_OUTPUT_MANIFEST_HASH
)

print(
    "Post-write verification: "
    + str(
        NOTEBOOK_01_POSTWRITE_PASSED
    )
)

print(
    "Notebook 01 status: "
    + NOTEBOOK_01_PREWRITE_STATUS
)

print(
    "Next notebook: "
    + NOTEBOOK_02_FILENAME
)

,artifact_type,format,acceptance_status,row_count,resolved_path,relative_path,size_bytes,sha256
0,V0_1_RAW_DATA_AUDIT,JSON,CONDITIONAL PASS,103677.0,D:\Clown Project\V0.1\artifacts\manifests\BTCU...,artifacts\manifests\BTCUSDT_spot_20260710T0637...,70251,a979107c7825e75d5ba151c6519c4b61b500695daaf803...
1,NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF,JSON,CONDITIONAL PASS,NaN,D:\Clown Project\V0.1\artifacts\handoff\BTCUSD...,artifacts\handoff\BTCUSDT_spot_20260710T063746...,8367,138704253d8821259a7be427620a11b1d469a7f009c13c...
2,AUTHORITATIVE_RAW_SOURCE_MANIFEST,CSV,CONDITIONAL PASS,5.0,D:\Clown Project\V0.1\artifacts\audit_tables\B...,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,2926,7c130c5b9d0e2861948a74a0ecc63bdad9d49309c3fe61...
3,NOTEBOOK_01_SOURCE_REHASH,CSV,CONDITIONAL PASS,5.0,D:\Clown Project\V0.1\artifacts\audit_tables\B...,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,1604,9c9c3563633ad934ba72a6fc3edfe42e9d1bc2e3752b2b...
4,JSONL_PARSE_AUDIT,CSV,CONDITIONAL PASS,2.0,D:\Clown Project\V0.1\artifacts\audit_tables\B...,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,1326,26eeeb2f1f576bc5d6b1608931e82213d079f137e40854...
5,JSON_DOCUMENT_AUDIT,CSV,CONDITIONAL PASS,3.0,D:\Clown Project\V0.1\artifacts\audit_tables\B...,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,2053,3c79ff2001245ada90f3e793f960dc7f9debd4354d0f44...
6,RUNTIME_CANONICAL_FIELD_BINDINGS,CSV,CONDITIONAL PASS,21.0,D:\Clown Project\V0.1\artifacts\audit_tables\B...,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,2644,910dfc68fef280519652164bc955c0fe2b8e768197c30b...
7,CANONICAL_FIELD_AUDIT,CSV,CONDITIONAL PASS,21.0,D:\Clown Project\V0.1\artifacts\audit_tables\B...,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,2627,a4d241ef3e053080574bfbda068868021d9eaf205b791e...
8,RAW_RECORD_SIGNATURES,CSV,CONDITIONAL PASS,6.0,D:\Clown Project\V0.1\artifacts\audit_tables\B...,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,870,49d26e349410af7d08ade69d85604c12a8cb1600d49fa4...
9,GLOBAL_ORDER_AUDIT,CSV,CONDITIONAL PASS,1.0,D:\Clown Project\V0.1\artifacts\audit_tables\B...,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,337,30f1f47c0fb09173cc8ce4707055712bed0f26bd8eb24c...


,artifact_type,format,relative_path,file_exists,size_matches,hash_matches,artifact_passed
0,V0_1_RAW_DATA_AUDIT,JSON,artifacts\manifests\BTCUSDT_spot_20260710T0637...,True,True,True,True
1,NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF,JSON,artifacts\handoff\BTCUSDT_spot_20260710T063746...,True,True,True,True
2,AUTHORITATIVE_RAW_SOURCE_MANIFEST,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True
3,NOTEBOOK_01_SOURCE_REHASH,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True
4,JSONL_PARSE_AUDIT,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True
5,JSON_DOCUMENT_AUDIT,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True
6,RUNTIME_CANONICAL_FIELD_BINDINGS,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True
7,CANONICAL_FIELD_AUDIT,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True
8,RAW_RECORD_SIGNATURES,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True
9,GLOBAL_ORDER_AUDIT,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True


Notebook 01 atomic artifact writes: PASS
Authoritative artifacts written: 32
Raw-data audit artifact: artifacts\manifests\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__01_RAW_DATA_AUDIT__v0_1_raw_data_audit.json
Notebook 02 handoff artifact: artifacts\handoff\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__01_RAW_DATA_AUDIT__notebook_01_to_notebook_02_handoff.json
Output manifest: artifacts\manifests\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__01_RAW_DATA_AUDIT__notebook_01_output_manifest.json
Raw-data audit SHA-256: fdfd1bc3ca907b1bf14286ca8f849d49a997bd5655f8793260838a34b1c9f56d
Handoff SHA-256: 80f3e2df95b6f2daebdf7b85324cbfcabdb64d22740e328962d1af078420b2e1
Output manifest SHA-256: 2bb561126bbfc20fd25aa80131c1c11307e651d38ed3d6ee26a11b2ff0f41a4f
Post-write verification: True
Notebook 01 status: CONDITIONAL PASS
Next notebook: 02_VISIBLE_BOOK_RECONSTRUCTION.ipynb


In [44]:
# ============================================================
# NOTEBOOK 01 FINAL SYNTHESIS AND NOTEBOOK 02 HANDOFF
# ============================================================

from datetime import datetime, timezone
from pathlib import Path

import hashlib
import json
import numpy as np
import pandas as pd


# ============================================================
# LOAD THE SAVED NOTEBOOK 01 OUTPUT MANIFEST
# ============================================================

NOTEBOOK_01_FINALIZED_UTC = (
    datetime.now(
        timezone.utc
    ).isoformat()
)

saved_manifest_path = Path(
    manifest_write_result[
        "resolved_path"
    ]
).resolve(strict=True)

saved_manifest_bytes = (
    saved_manifest_path.read_bytes()
)

saved_manifest_size_bytes = int(
    len(saved_manifest_bytes)
)

saved_manifest_sha256 = (
    hashlib.sha256(
        saved_manifest_bytes
    ).hexdigest()
)


try:
    saved_manifest_document = json.loads(
        saved_manifest_bytes.decode(
            "utf-8"
        )
    )

except (
    UnicodeDecodeError,
    json.JSONDecodeError,
) as exc:
    raise RuntimeError(
        "Saved Notebook 01 output manifest is unreadable."
    ) from exc


if not isinstance(
    saved_manifest_document,
    dict,
):
    raise RuntimeError(
        "Saved Notebook 01 output manifest is not a mapping."
    )


saved_manifest_metadata = (
    saved_manifest_document.get(
        "artifact_metadata"
    )
)

saved_manifest_payload = (
    saved_manifest_document.get(
        "payload"
    )
)


if not isinstance(
    saved_manifest_metadata,
    dict,
):
    raise RuntimeError(
        "Saved Notebook 01 output manifest lacks artifact metadata."
    )


if not isinstance(
    saved_manifest_payload,
    dict,
):
    raise RuntimeError(
        "Saved Notebook 01 output manifest lacks a payload."
    )


saved_manifest_artifacts = (
    saved_manifest_payload.get(
        "artifacts"
    )
)


if not isinstance(
    saved_manifest_artifacts,
    list,
):
    raise RuntimeError(
        "Saved Notebook 01 output manifest lacks an artifact list."
    )


# ============================================================
# VERIFY EVERY MANIFESTED ARTIFACT FROM DISK
# ============================================================

final_artifact_verification_rows = []


for artifact_record in (
    saved_manifest_artifacts
):
    if not isinstance(
        artifact_record,
        dict,
    ):
        raise RuntimeError(
            "Output manifest contains a non-mapping artifact record."
        )

    artifact_type = str(
        artifact_record.get(
            "artifact_type"
        )
    )

    artifact_format = str(
        artifact_record.get(
            "format"
        )
    )

    relative_path_value = (
        artifact_record.get(
            "relative_path"
        )
    )

    expected_size_value = (
        artifact_record.get(
            "size_bytes"
        )
    )

    expected_sha256_value = (
        artifact_record.get(
            "sha256"
        )
    )

    if relative_path_value is None:
        raise RuntimeError(
            "Manifest artifact lacks relative_path: "
            + artifact_type
        )

    artifact_path = (
        Path(V0_1_ROOT_RESOLVED)
        / str(relative_path_value)
    ).resolve(strict=False)

    file_exists = bool(
        artifact_path.exists()
        and artifact_path.is_file()
    )

    inside_v0_1 = path_is_within(
        artifact_path,
        V0_1_ROOT_RESOLVED,
    )

    inside_v0_0 = path_is_within(
        artifact_path,
        V0_0_ROOT_RESOLVED,
    )

    observed_size_bytes = None
    observed_sha256 = None

    if file_exists:
        observed_bytes = (
            artifact_path.read_bytes()
        )

        observed_size_bytes = int(
            len(observed_bytes)
        )

        observed_sha256 = (
            hashlib.sha256(
                observed_bytes
            ).hexdigest()
        )

    expected_size_bytes = (
        int(expected_size_value)
        if expected_size_value
        is not None
        else None
    )

    expected_sha256 = (
        str(
            expected_sha256_value
        ).lower()
        if expected_sha256_value
        is not None
        else None
    )

    size_matches = bool(
        observed_size_bytes
        == expected_size_bytes
    )

    hash_matches = bool(
        observed_sha256
        == expected_sha256
    )

    artifact_passed = bool(
        file_exists
        and inside_v0_1
        and not inside_v0_0
        and size_matches
        and hash_matches
    )

    final_artifact_verification_rows.append(
        {
            "artifact_type": (
                artifact_type
            ),
            "format": (
                artifact_format
            ),
            "relative_path": str(
                relative_path_value
            ),
            "file_exists": (
                file_exists
            ),
            "inside_v0_1": (
                inside_v0_1
            ),
            "inside_v0_0": (
                inside_v0_0
            ),
            "expected_size_bytes": (
                expected_size_bytes
            ),
            "observed_size_bytes": (
                observed_size_bytes
            ),
            "size_matches": (
                size_matches
            ),
            "expected_sha256": (
                expected_sha256
            ),
            "observed_sha256": (
                observed_sha256
            ),
            "hash_matches": (
                hash_matches
            ),
            "artifact_passed": (
                artifact_passed
            ),
            "resolved_path": str(
                artifact_path
            ),
        }
    )


NOTEBOOK_01_FINAL_ARTIFACT_VERIFICATION = (
    pd.DataFrame(
        final_artifact_verification_rows
    )
)


# ============================================================
# LOAD AND VERIFY THE RAW-AUDIT AND HANDOFF DOCUMENTS
# ============================================================

def load_manifested_json_document(
    artifact_type: str,
) -> tuple[Path, dict]:
    matches = (
        NOTEBOOK_01_FINAL_ARTIFACT_VERIFICATION.loc[
            NOTEBOOK_01_FINAL_ARTIFACT_VERIFICATION[
                "artifact_type"
            ].eq(artifact_type)
        ]
    )

    if len(matches) != 1:
        raise RuntimeError(
            "Expected exactly one manifested artifact of type "
            + artifact_type
            + "; found "
            + str(len(matches))
            + "."
        )

    artifact_path = Path(
        matches.iloc[0][
            "resolved_path"
        ]
    ).resolve(strict=True)

    try:
        document = json.loads(
            artifact_path.read_text(
                encoding="utf-8"
            )
        )

    except (
        UnicodeDecodeError,
        json.JSONDecodeError,
    ) as exc:
        raise RuntimeError(
            "Manifested JSON artifact is unreadable: "
            + artifact_type
        ) from exc

    if not isinstance(
        document,
        dict,
    ):
        raise RuntimeError(
            "Manifested JSON artifact is not a mapping: "
            + artifact_type
        )

    return artifact_path, document


(
    saved_raw_audit_path,
    saved_raw_audit_document,
) = load_manifested_json_document(
    "V0_1_RAW_DATA_AUDIT"
)

(
    saved_handoff_path,
    saved_handoff_document,
) = load_manifested_json_document(
    "NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF"
)


saved_raw_audit_metadata = (
    saved_raw_audit_document.get(
        "artifact_metadata"
    )
)

saved_raw_audit_payload = (
    saved_raw_audit_document.get(
        "payload"
    )
)

saved_handoff_metadata = (
    saved_handoff_document.get(
        "artifact_metadata"
    )
)

saved_handoff_payload = (
    saved_handoff_document.get(
        "payload"
    )
)


if not isinstance(
    saved_raw_audit_metadata,
    dict,
):
    raise RuntimeError(
        "Saved raw-data audit metadata is missing."
    )


if not isinstance(
    saved_raw_audit_payload,
    dict,
):
    raise RuntimeError(
        "Saved raw-data audit payload is missing."
    )


if not isinstance(
    saved_handoff_metadata,
    dict,
):
    raise RuntimeError(
        "Saved Notebook 02 handoff metadata is missing."
    )


if not isinstance(
    saved_handoff_payload,
    dict,
):
    raise RuntimeError(
        "Saved Notebook 02 handoff payload is missing."
    )


observed_raw_audit_payload_hash = (
    notebook_01_object_hash(
        saved_raw_audit_payload
    )
)

observed_handoff_payload_hash = (
    notebook_01_object_hash(
        saved_handoff_payload
    )
)

declared_raw_audit_payload_hash = str(
    saved_raw_audit_metadata.get(
        "payload_sha256"
    )
)

declared_handoff_payload_hash = str(
    saved_handoff_metadata.get(
        "payload_sha256"
    )
)


raw_audit_hash_matches = bool(
    observed_raw_audit_payload_hash
    == NOTEBOOK_01_RAW_DATA_AUDIT_HASH
    and declared_raw_audit_payload_hash
    == NOTEBOOK_01_RAW_DATA_AUDIT_HASH
)

handoff_hash_matches = bool(
    observed_handoff_payload_hash
    == NOTEBOOK_01_HANDOFF_HASH
    and declared_handoff_payload_hash
    == NOTEBOOK_01_HANDOFF_HASH
)


# ============================================================
# MANIFEST IDENTITY AND COUNT CHECKS
# ============================================================

declared_manifest_artifact_count = int(
    saved_manifest_payload.get(
        "artifact_count",
        -1,
    )
)

observed_manifest_artifact_count = int(
    len(
        saved_manifest_artifacts
    )
)

verified_manifest_artifact_count = int(
    NOTEBOOK_01_FINAL_ARTIFACT_VERIFICATION[
        "artifact_passed"
    ].sum()
)

manifest_identity_matches = bool(
    saved_manifest_payload.get(
        "manifest_type"
    )
    == "NOTEBOOK_01_OUTPUT_MANIFEST"
    and saved_manifest_payload.get(
        "pipeline_name"
    )
    == PIPELINE_NAME
    and saved_manifest_payload.get(
        "pipeline_version"
    )
    == PIPELINE_VERSION
    and saved_manifest_payload.get(
        "source_run_prefix"
    )
    == SOURCE_RUN_PREFIX
    and saved_manifest_payload.get(
        "source_set_hash"
    )
    == NOTEBOOK_01_SOURCE_SET_HASH
    and saved_manifest_payload.get(
        "v0_1_run_id"
    )
    == V0_1_RUN_ID
    and saved_manifest_payload.get(
        "v0_1_run_config_hash"
    )
    == NOTEBOOK_01_RUN_CONFIG_HASH
    and saved_manifest_payload.get(
        "v0_1_run_identity_hash"
    )
    == NOTEBOOK_01_RUN_IDENTITY_HASH
    and saved_manifest_payload.get(
        "raw_data_audit_hash"
    )
    == NOTEBOOK_01_RAW_DATA_AUDIT_HASH
    and saved_manifest_payload.get(
        "notebook_01_handoff_hash"
    )
    == NOTEBOOK_01_HANDOFF_HASH
)

manifest_artifact_count_matches = bool(
    declared_manifest_artifact_count
    == observed_manifest_artifact_count
    == len(
        NOTEBOOK_01_WRITTEN_ARTIFACTS
    )
)

manifest_size_matches = bool(
    saved_manifest_size_bytes
    == int(
        manifest_write_result[
            "size_bytes"
        ]
    )
)

manifest_hash_matches = bool(
    saved_manifest_sha256
    == NOTEBOOK_01_OUTPUT_MANIFEST_HASH
    == str(
        manifest_write_result[
            "sha256"
        ]
    )
)


# ============================================================
# NOTEBOOK 02 HANDOFF VERIFICATION
# ============================================================

handoff_identity_matches = bool(
    saved_handoff_payload.get(
        "handoff_type"
    )
    == "NOTEBOOK_01_TO_NOTEBOOK_02"
    and saved_handoff_payload.get(
        "source_run_prefix"
    )
    == SOURCE_RUN_PREFIX
    and saved_handoff_payload.get(
        "source_set_hash"
    )
    == NOTEBOOK_01_SOURCE_SET_HASH
    and saved_handoff_payload.get(
        "v0_1_run_id"
    )
    == V0_1_RUN_ID
    and saved_handoff_payload.get(
        "raw_data_audit_hash"
    )
    == NOTEBOOK_01_RAW_DATA_AUDIT_HASH
    and saved_handoff_payload.get(
        "next_notebook"
    )
    == NOTEBOOK_02_FILENAME
)

saved_depth_reconstruction_contract = (
    saved_handoff_payload.get(
        "depth_reconstruction_contract"
    )
)


if not isinstance(
    saved_depth_reconstruction_contract,
    dict,
):
    raise RuntimeError(
        "Saved Notebook 02 handoff lacks the depth reconstruction contract."
    )


depth_handoff_matches = bool(
    int(
        saved_depth_reconstruction_contract.get(
            "total_depth_event_count",
            -1,
        )
    )
    == len(
        RAW_DEPTH_EVENT_AUDIT_TABLE
    )
    and int(
        saved_depth_reconstruction_contract.get(
            "stale_pre_snapshot_event_count",
            -1,
        )
    )
    == stale_event_count
    and int(
        saved_depth_reconstruction_contract.get(
            "first_applicable_collector_sequence",
            -1,
        )
    )
    == first_post_snapshot_sequence
    and int(
        saved_depth_reconstruction_contract.get(
            "first_applicable_U",
            -1,
        )
    )
    == first_post_snapshot_U
    and int(
        saved_depth_reconstruction_contract.get(
            "first_applicable_u",
            -1,
        )
    )
    == first_post_snapshot_u
    and int(
        saved_depth_reconstruction_contract.get(
            "post_snapshot_event_count",
            -1,
        )
    )
    == post_bridge_event_count
    and int(
        saved_depth_reconstruction_contract.get(
            "post_snapshot_gap_count",
            -1,
        )
    )
    == post_bridge_gap_count
    and int(
        saved_depth_reconstruction_contract.get(
            "post_snapshot_overlap_count",
            -1,
        )
    )
    == post_bridge_overlap_count
    and saved_depth_reconstruction_contract.get(
        "zero_quantity_action"
    )
    == "DELETE_PRICE_LEVEL"
    and saved_depth_reconstruction_contract.get(
        "ordering_authority"
    )
    == "collector_sequence"
)


required_notebook_02_artifact_types = {
    "V0_1_RAW_DATA_AUDIT",
    "NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF",
    "AUTHORITATIVE_RAW_SOURCE_MANIFEST",
    "NOTEBOOK_01_SOURCE_REHASH",
    "RUNTIME_CANONICAL_FIELD_BINDINGS",
    "CANONICAL_FIELD_AUDIT",
    "DEPTH_UPDATE_ID_AUDIT",
    "SNAPSHOT_BUFFER_RECONCILIATION",
    "NOTEBOOK_01_PREWRITE_DECISION",
}


observed_artifact_types = set(
    NOTEBOOK_01_FINAL_ARTIFACT_VERIFICATION[
        "artifact_type"
    ].astype(str)
)


required_notebook_02_artifacts_present = bool(
    required_notebook_02_artifact_types.issubset(
        observed_artifact_types
    )
)


source_state_preserved = bool(
    NOTEBOOK_01_SOURCE_REHASH[
        "source_passed"
    ].all()
)

all_manifested_artifacts_valid = bool(
    verified_manifest_artifact_count
    == observed_manifest_artifact_count
    and observed_manifest_artifact_count
    > 0
)

notebook_02_execution_authorized = bool(
    saved_manifest_payload.get(
        "notebook_02_authorized",
        False,
    )
    and saved_handoff_payload.get(
        "notebook_02_authorized",
        False,
    )
    and NOTEBOOK_02_AUTHORIZED
)


# ============================================================
# FINAL ACCEPTANCE GATES
# ============================================================

full_statistical_mode_available = bool(
    NOTEBOOK_01_OPERATING_MODE
    == "FULL_STATISTICAL_MODE"
)

claim_bearing_holdout_available = bool(
    CHRONOLOGICAL_SPLIT_CONTRACT.get(
        "claim_bearing_holdout_available",
        False,
    )
)


NOTEBOOK_01_FINAL_GATES = pd.DataFrame(
    [
        {
            "gate": (
                "saved_output_manifest_readable"
            ),
            "passed": True,
            "severity": "CRITICAL",
            "evidence": str(
                saved_manifest_path
            ),
        },
        {
            "gate": (
                "output_manifest_identity_matches"
            ),
            "passed": (
                manifest_identity_matches
            ),
            "severity": "CRITICAL",
            "evidence": (
                "source_run_prefix="
                + str(
                    saved_manifest_payload.get(
                        "source_run_prefix"
                    )
                )
                + "; v0_1_run_id="
                + str(
                    saved_manifest_payload.get(
                        "v0_1_run_id"
                    )
                )
            ),
        },
        {
            "gate": (
                "output_manifest_artifact_count_matches"
            ),
            "passed": (
                manifest_artifact_count_matches
            ),
            "severity": "CRITICAL",
            "evidence": (
                "declared="
                + str(
                    declared_manifest_artifact_count
                )
                + "; observed="
                + str(
                    observed_manifest_artifact_count
                )
            ),
        },
        {
            "gate": (
                "output_manifest_size_matches"
            ),
            "passed": (
                manifest_size_matches
            ),
            "severity": "CRITICAL",
            "evidence": (
                "expected="
                + str(
                    manifest_write_result[
                        "size_bytes"
                    ]
                )
                + "; observed="
                + str(
                    saved_manifest_size_bytes
                )
            ),
        },
        {
            "gate": (
                "output_manifest_hash_matches"
            ),
            "passed": (
                manifest_hash_matches
            ),
            "severity": "CRITICAL",
            "evidence": (
                saved_manifest_sha256
            ),
        },
        {
            "gate": (
                "all_manifested_artifacts_valid"
            ),
            "passed": (
                all_manifested_artifacts_valid
            ),
            "severity": "CRITICAL",
            "evidence": (
                "valid="
                + str(
                    verified_manifest_artifact_count
                )
                + "/"
                + str(
                    observed_manifest_artifact_count
                )
            ),
        },
        {
            "gate": (
                "raw_data_audit_hash_matches"
            ),
            "passed": (
                raw_audit_hash_matches
            ),
            "severity": "CRITICAL",
            "evidence": (
                observed_raw_audit_payload_hash
            ),
        },
        {
            "gate": (
                "notebook_02_handoff_hash_matches"
            ),
            "passed": (
                handoff_hash_matches
            ),
            "severity": "CRITICAL",
            "evidence": (
                observed_handoff_payload_hash
            ),
        },
        {
            "gate": (
                "notebook_02_handoff_identity_matches"
            ),
            "passed": (
                handoff_identity_matches
            ),
            "severity": "CRITICAL",
            "evidence": (
                str(
                    saved_handoff_path
                )
            ),
        },
        {
            "gate": (
                "depth_reconstruction_handoff_matches"
            ),
            "passed": (
                depth_handoff_matches
            ),
            "severity": "CRITICAL",
            "evidence": (
                "first_sequence="
                + str(
                    first_post_snapshot_sequence
                )
                + "; U="
                + str(
                    first_post_snapshot_U
                )
                + "; u="
                + str(
                    first_post_snapshot_u
                )
            ),
        },
        {
            "gate": (
                "required_notebook_02_artifacts_present"
            ),
            "passed": (
                required_notebook_02_artifacts_present
            ),
            "severity": "CRITICAL",
            "evidence": (
                "required="
                + str(
                    len(
                        required_notebook_02_artifact_types
                    )
                )
            ),
        },
        {
            "gate": (
                "v0_0_source_state_preserved"
            ),
            "passed": (
                source_state_preserved
            ),
            "severity": "CRITICAL",
            "evidence": (
                "unchanged="
                + str(
                    int(
                        NOTEBOOK_01_SOURCE_REHASH[
                            "source_passed"
                        ].sum()
                    )
                )
                + "/"
                + str(
                    len(
                        NOTEBOOK_01_SOURCE_REHASH
                    )
                )
            ),
        },
        {
            "gate": (
                "notebook_02_execution_authorized"
            ),
            "passed": (
                notebook_02_execution_authorized
            ),
            "severity": "CRITICAL",
            "evidence": (
                "authorized="
                + str(
                    notebook_02_execution_authorized
                )
            ),
        },
        {
            "gate": (
                "full_statistical_mode_available"
            ),
            "passed": (
                full_statistical_mode_available
            ),
            "severity": "WARNING",
            "evidence": (
                NOTEBOOK_01_OPERATING_MODE
            ),
        },
        {
            "gate": (
                "claim_bearing_holdout_available"
            ),
            "passed": (
                claim_bearing_holdout_available
            ),
            "severity": "WARNING",
            "evidence": (
                "available="
                + str(
                    claim_bearing_holdout_available
                )
            ),
        },
    ]
)


NOTEBOOK_01_FINAL_GATES[
    "status"
] = np.select(
    [
        NOTEBOOK_01_FINAL_GATES[
            "passed"
        ],
        NOTEBOOK_01_FINAL_GATES[
            "severity"
        ].eq("CRITICAL"),
    ],
    [
        "PASS",
        "FAIL",
    ],
    default="WARNING",
)


notebook_01_final_critical_failures = (
    NOTEBOOK_01_FINAL_GATES.loc[
        NOTEBOOK_01_FINAL_GATES[
            "severity"
        ].eq("CRITICAL")
        & ~NOTEBOOK_01_FINAL_GATES[
            "passed"
        ]
    ]
)

notebook_01_final_warnings = (
    NOTEBOOK_01_FINAL_GATES.loc[
        NOTEBOOK_01_FINAL_GATES[
            "severity"
        ].eq("WARNING")
        & ~NOTEBOOK_01_FINAL_GATES[
            "passed"
        ]
    ]
)


if not notebook_01_final_critical_failures.empty:
    NOTEBOOK_01_FINAL_STATUS = "FAIL"

elif (
    NOTEBOOK_01_PREWRITE_STATUS
    == "CONDITIONAL PASS"
):
    NOTEBOOK_01_FINAL_STATUS = (
        "CONDITIONAL PASS"
    )

elif not notebook_01_final_warnings.empty:
    NOTEBOOK_01_FINAL_STATUS = (
        "WARNING"
    )

else:
    NOTEBOOK_01_FINAL_STATUS = "PASS"


NOTEBOOK_02_FINAL_AUTHORIZATION = bool(
    notebook_01_final_critical_failures.empty
    and notebook_02_execution_authorized
)


# ============================================================
# FINAL NOTEBOOK 02 HANDOFF TABLE
# ============================================================

NOTEBOOK_02_HANDOFF_TABLE = pd.DataFrame(
    [
        {
            "artifact_type": (
                "V0_1_RAW_DATA_AUDIT"
            ),
            "available": bool(
                saved_raw_audit_path.exists()
            ),
            "format": "JSON",
            "relative_path": str(
                saved_raw_audit_path.relative_to(
                    V0_1_ROOT_RESOLVED
                )
            ),
            "sha256": str(
                NOTEBOOK_01_FINAL_ARTIFACT_VERIFICATION.loc[
                    NOTEBOOK_01_FINAL_ARTIFACT_VERIFICATION[
                        "artifact_type"
                    ].eq(
                        "V0_1_RAW_DATA_AUDIT"
                    ),
                    "observed_sha256",
                ].iloc[0]
            ),
        },
        {
            "artifact_type": (
                "NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF"
            ),
            "available": bool(
                saved_handoff_path.exists()
            ),
            "format": "JSON",
            "relative_path": str(
                saved_handoff_path.relative_to(
                    V0_1_ROOT_RESOLVED
                )
            ),
            "sha256": str(
                NOTEBOOK_01_FINAL_ARTIFACT_VERIFICATION.loc[
                    NOTEBOOK_01_FINAL_ARTIFACT_VERIFICATION[
                        "artifact_type"
                    ].eq(
                        "NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF"
                    ),
                    "observed_sha256",
                ].iloc[0]
            ),
        },
        {
            "artifact_type": (
                "NOTEBOOK_01_OUTPUT_MANIFEST"
            ),
            "available": bool(
                saved_manifest_path.exists()
            ),
            "format": "JSON",
            "relative_path": str(
                saved_manifest_path.relative_to(
                    V0_1_ROOT_RESOLVED
                )
            ),
            "sha256": (
                saved_manifest_sha256
            ),
        },
    ]
)


# ============================================================
# FINAL SYNTHESIS
# ============================================================

NOTEBOOK_01_FINAL_SYNTHESIS = pd.DataFrame(
    [
        {
            "pipeline_name": (
                PIPELINE_NAME
            ),
            "pipeline_version": (
                PIPELINE_VERSION
            ),
            "notebook": (
                NOTEBOOK_FILENAME
            ),
            "source_run_prefix": (
                SOURCE_RUN_PREFIX
            ),
            "source_set_hash": (
                NOTEBOOK_01_SOURCE_SET_HASH
            ),
            "v0_1_run_id": (
                V0_1_RUN_ID
            ),
            "raw_data_audit_hash": (
                NOTEBOOK_01_RAW_DATA_AUDIT_HASH
            ),
            "handoff_hash": (
                NOTEBOOK_01_HANDOFF_HASH
            ),
            "output_manifest_sha256": (
                saved_manifest_sha256
            ),
            "operating_mode": (
                NOTEBOOK_01_OPERATING_MODE
            ),
            "final_status": (
                NOTEBOOK_01_FINAL_STATUS
            ),
            "manifested_artifact_count": (
                observed_manifest_artifact_count
            ),
            "verified_artifact_count": (
                verified_manifest_artifact_count
            ),
            "trade_record_count": int(
                len(
                    RAW_TRADE_AUDIT_TABLE
                )
            ),
            "depth_record_count": int(
                len(
                    RAW_DEPTH_EVENT_AUDIT_TABLE
                )
            ),
            "combined_record_count": int(
                len(
                    RAW_EVENT_ORDER_INDEX
                )
            ),
            "raw_rejection_count": int(
                len(
                    RAW_REJECTION_LEDGER
                )
            ),
            "critical_failure_count": int(
                len(
                    notebook_01_final_critical_failures
                )
            ),
            "warning_count": int(
                len(
                    notebook_01_final_warnings
                )
            ),
            "notebook_02_authorized": (
                NOTEBOOK_02_FINAL_AUTHORIZATION
            ),
            "next_notebook": (
                NOTEBOOK_02_FILENAME
            ),
            "finalized_utc": (
                NOTEBOOK_01_FINALIZED_UTC
            ),
        }
    ]
)


# ============================================================
# DISPLAY AND ENFORCE
# ============================================================

display(
    NOTEBOOK_01_FINAL_ARTIFACT_VERIFICATION[
        [
            "artifact_type",
            "format",
            "relative_path",
            "file_exists",
            "size_matches",
            "hash_matches",
            "artifact_passed",
        ]
    ]
)

display(
    NOTEBOOK_02_HANDOFF_TABLE
)

display(
    NOTEBOOK_01_FINAL_GATES
)

display(
    NOTEBOOK_01_FINAL_SYNTHESIS
)


if not notebook_01_final_critical_failures.empty:
    failure_parts = []

    for row in (
        notebook_01_final_critical_failures
        .itertuples(index=False)
    ):
        failure_parts.append(
            str(row.gate)
            + ": "
            + str(row.evidence)
        )

    raise RuntimeError(
        "Notebook 01 finalization failed: "
        + ", ".join(
            failure_parts
        )
    )


print(
    "Notebook 01 finalization: PASS"
)

print(
    "Final notebook status: "
    + NOTEBOOK_01_FINAL_STATUS
)

print(
    "Manifested artifacts verified: "
    + str(
        verified_manifest_artifact_count
    )
    + "/"
    + str(
        observed_manifest_artifact_count
    )
)

print(
    "Raw-data audit SHA-256: "
    + NOTEBOOK_01_RAW_DATA_AUDIT_HASH
)

print(
    "Handoff SHA-256: "
    + NOTEBOOK_01_HANDOFF_HASH
)

print(
    "Output manifest SHA-256: "
    + saved_manifest_sha256
)

print(
    "Critical failures: "
    + str(
        len(
            notebook_01_final_critical_failures
        )
    )
)

print(
    "Warnings retained: "
    + str(
        len(
            notebook_01_final_warnings
        )
    )
)

print(
    "Notebook 02 authorized: "
    + str(
        NOTEBOOK_02_FINAL_AUTHORIZATION
    )
)

print(
    "Next notebook: "
    + NOTEBOOK_02_FILENAME
)

,artifact_type,format,relative_path,file_exists,size_matches,hash_matches,artifact_passed
0,V0_1_RAW_DATA_AUDIT,JSON,artifacts\manifests\BTCUSDT_spot_20260710T0637...,True,True,True,True
1,NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF,JSON,artifacts\handoff\BTCUSDT_spot_20260710T063746...,True,True,True,True
2,AUTHORITATIVE_RAW_SOURCE_MANIFEST,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True
3,NOTEBOOK_01_SOURCE_REHASH,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True
4,JSONL_PARSE_AUDIT,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True
5,JSON_DOCUMENT_AUDIT,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True
6,RUNTIME_CANONICAL_FIELD_BINDINGS,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True
7,CANONICAL_FIELD_AUDIT,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True
8,RAW_RECORD_SIGNATURES,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True
9,GLOBAL_ORDER_AUDIT,CSV,artifacts\audit_tables\BTCUSDT_spot_20260710T0...,True,True,True,True


,artifact_type,available,format,relative_path,sha256
0,V0_1_RAW_DATA_AUDIT,True,JSON,artifacts\manifests\BTCUSDT_spot_20260710T0637...,a979107c7825e75d5ba151c6519c4b61b500695daaf803...
1,NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF,True,JSON,artifacts\handoff\BTCUSDT_spot_20260710T063746...,138704253d8821259a7be427620a11b1d469a7f009c13c...
2,NOTEBOOK_01_OUTPUT_MANIFEST,True,JSON,artifacts\manifests\BTCUSDT_spot_20260710T0637...,2bb561126bbfc20fd25aa80131c1c11307e651d38ed3d6...


,gate,passed,severity,evidence,status
0,saved_output_manifest_readable,True,CRITICAL,D:\Clown Project\V0.1\artifacts\manifests\BTCU...,PASS
1,output_manifest_identity_matches,True,CRITICAL,source_run_prefix=BTCUSDT_spot_20260710T063746...,PASS
2,output_manifest_artifact_count_matches,True,CRITICAL,declared=32; observed=32,PASS
3,output_manifest_size_matches,True,CRITICAL,expected=23269; observed=23269,PASS
4,output_manifest_hash_matches,True,CRITICAL,2bb561126bbfc20fd25aa80131c1c11307e651d38ed3d6...,PASS
5,all_manifested_artifacts_valid,True,CRITICAL,valid=32/32,PASS
6,raw_data_audit_hash_matches,True,CRITICAL,fdfd1bc3ca907b1bf14286ca8f849d49a997bd5655f879...,PASS
7,notebook_02_handoff_hash_matches,True,CRITICAL,80f3e2df95b6f2daebdf7b85324cbfcabdb64d22740e32...,PASS
8,notebook_02_handoff_identity_matches,True,CRITICAL,D:\Clown Project\V0.1\artifacts\handoff\BTCUSD...,PASS
9,depth_reconstruction_handoff_matches,True,CRITICAL,first_sequence=10; U=97233590167; u=97233590170,PASS


,pipeline_name,pipeline_version,notebook,source_run_prefix,source_set_hash,v0_1_run_id,raw_data_audit_hash,handoff_hash,output_manifest_sha256,operating_mode,...,verified_artifact_count,trade_record_count,depth_record_count,combined_record_count,raw_rejection_count,critical_failure_count,warning_count,notebook_02_authorized,next_notebook,finalized_utc
0,The Clown Project,V0.1,01_RAW_DATA_AUDIT.ipynb,BTCUSDT_spot_20260710T063746Z_c8b5bf12,132c83531eec615d279408b5c06f402973114ba3058dfa...,v0_1_20260714T090616Z_e82325081a81,fdfd1bc3ca907b1bf14286ca8f849d49a997bd5655f879...,80f3e2df95b6f2daebdf7b85324cbfcabdb64d22740e32...,2bb561126bbfc20fd25aa80131c1c11307e651d38ed3d6...,ENGINEERING_REPRODUCTION_MODE,...,32,67683,35994,103677,0,0,2,True,02_VISIBLE_BOOK_RECONSTRUCTION.ipynb,2026-07-14T11:15:55.056686+00:00


Notebook 01 finalization: PASS
Final notebook status: CONDITIONAL PASS
Manifested artifacts verified: 32/32
Raw-data audit SHA-256: fdfd1bc3ca907b1bf14286ca8f849d49a997bd5655f8793260838a34b1c9f56d
Handoff SHA-256: 80f3e2df95b6f2daebdf7b85324cbfcabdb64d22740e328962d1af078420b2e1
Output manifest SHA-256: 2bb561126bbfc20fd25aa80131c1c11307e651d38ed3d6ee26a11b2ff0f41a4f
Critical failures: 0
Warnings retained: 2
Notebook 02 authorized: True
Next notebook: 02_VISIBLE_BOOK_RECONSTRUCTION.ipynb


In [ ]:
# ============================================================
# NOTEBOOK 02 — VISIBLE BOOK RECONSTRUCTION
# CLEAN-KERNEL BOOTSTRAP AND SAVED-HANDOFF VERIFICATION
# ============================================================

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import random

import numpy as np
import pandas as pd


# ============================================================
# NOTEBOOK IDENTITY
# ============================================================

PIPELINE_NAME = "The Clown Project"
PIPELINE_VERSION = "V0.1"

NOTEBOOK_NUMBER = "02"
NOTEBOOK_FILENAME = (
    "02_VISIBLE_BOOK_RECONSTRUCTION.ipynb"
)

NOTEBOOK_PURPOSE = (
    "Reconstruct the visible Binance Spot BTCUSDT "
    "market-by-price order book from the verified REST "
    "snapshot and contiguous post-snapshot differential-"
    "depth stream."
)

NOTEBOOK_STATUS = "IN PROGRESS"
CONTRACT_SCHEMA_VERSION = "v0.1.0"

SOURCE_RUN_PREFIX = (
    "BTCUSDT_spot_20260710T063746Z_c8b5bf12"
)

V0_1_RUN_ID = (
    "v0_1_20260714T090616Z_e82325081a81"
)

RANDOM_SEED = 20260710

NOTEBOOK_02_STARTED_UTC = (
    datetime.now(
        timezone.utc
    ).isoformat()
)


# ============================================================
# FROZEN IDENTITIES FROM NOTEBOOKS 00 AND 01
# ============================================================

EXPECTED_SOURCE_SET_HASH = (
    "132c83531eec615d279408b5c06f402973114ba3058dfa"
    "dd2fe58e2e67184c4b"
)

EXPECTED_RUN_CONFIG_HASH = (
    "14aea0efb3c7b6a193b8c4575a440babe8265d2935c6a7"
    "70bee995f688d59617"
)

EXPECTED_RUN_IDENTITY_HASH = (
    "5eb89cf073c036d70e3767df4b35ace7c31822e52e9b9d"
    "f19204c42e37fe4198"
)

EXPECTED_NOTEBOOK_00_MANIFEST_SHA256 = (
    "e12966301d92e61656715f2a6d397786820de836d8156dd"
    "70092250619cbb00e"
)

EXPECTED_RAW_DATA_AUDIT_HASH = (
    "fdfd1bc3ca907b1bf14286ca8f849d49a997bd5655f879"
    "3260838a34b1c9f56d"
)

EXPECTED_NOTEBOOK_01_HANDOFF_HASH = (
    "80f3e2df95b6f2daebdf7b85324cbfcabdb64d22740e328"
    "962d1af078420b2e1"
)

EXPECTED_NOTEBOOK_01_MANIFEST_SHA256 = (
    "2bb561126bbfc20fd25aa80131c1c11307e651d38ed3d6e"
    "e26a11b2ff0f41a4f"
)


# ============================================================
# PROJECT ROOTS
# ============================================================

V0_0_ROOT = Path(
    r"D:\Clown Project\V0.0"
)

V0_1_ROOT = Path(
    r"D:\Clown Project\V0.1"
)

V0_1_CONFIG_ROOT = (
    V0_1_ROOT
    / "config"
)

V0_1_MANIFEST_ROOT = (
    V0_1_ROOT
    / "artifacts"
    / "manifests"
)

V0_1_HANDOFF_ROOT = (
    V0_1_ROOT
    / "artifacts"
    / "handoff"
)

V0_1_AUDIT_ROOT = (
    V0_1_ROOT
    / "artifacts"
    / "audit_tables"
)

V0_1_INTERIM_ROOT = (
    V0_1_ROOT
    / "data"
    / "interim"
)

V0_1_PROCESSED_ROOT = (
    V0_1_ROOT
    / "data"
    / "processed"
)

V0_1_RECONCILIATION_ROOT = (
    V0_1_ROOT
    / "artifacts"
    / "reconciliation"
)

V0_1_LOG_ROOT = (
    V0_1_ROOT
    / "logs"
)


V0_0_ROOT_RESOLVED = (
    V0_0_ROOT.resolve(
        strict=True
    )
)

V0_1_ROOT_RESOLVED = (
    V0_1_ROOT.resolve(
        strict=True
    )
)


for output_root in (
    V0_1_INTERIM_ROOT,
    V0_1_PROCESSED_ROOT,
    V0_1_RECONCILIATION_ROOT,
    V0_1_LOG_ROOT,
):
    output_root.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# GENERAL HELPERS
# ============================================================

def path_is_within(
    candidate,
    root,
) -> bool:
    candidate_path = Path(
        candidate
    ).resolve(strict=False)

    root_path = Path(
        root
    ).resolve(strict=False)

    try:
        candidate_path.relative_to(
            root_path
        )

        return True

    except ValueError:
        return False


def sha256_file(
    path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def canonical_json_sha256(
    value,
) -> str:
    encoded = json.dumps(
        value,
        sort_keys=True,
        ensure_ascii=False,
        allow_nan=False,
        separators=(",", ":"),
    ).encode("utf-8")

    return hashlib.sha256(
        encoded
    ).hexdigest()


def load_json_mapping(
    path,
) -> dict:
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        value = json.load(handle)

    if not isinstance(
        value,
        dict,
    ):
        raise RuntimeError(
            "Expected a JSON mapping: "
            + str(path)
        )

    return value


def unwrap_saved_artifact(
    document: dict,
    expected_artifact_type: str,
) -> tuple[dict, dict]:
    metadata = document.get(
        "artifact_metadata"
    )

    payload = document.get(
        "payload"
    )

    if not isinstance(
        metadata,
        dict,
    ):
        raise RuntimeError(
            expected_artifact_type
            + " lacks artifact metadata."
        )

    if not isinstance(
        payload,
        dict,
    ):
        raise RuntimeError(
            expected_artifact_type
            + " lacks a payload mapping."
        )

    observed_artifact_type = (
        metadata.get(
            "artifact_type"
        )
    )

    if (
        observed_artifact_type
        != expected_artifact_type
    ):
        raise RuntimeError(
            "Artifact-type mismatch. Expected "
            + expected_artifact_type
            + "; observed "
            + str(
                observed_artifact_type
            )
            + "."
        )

    if (
        metadata.get(
            "source_run_prefix"
        )
        != SOURCE_RUN_PREFIX
    ):
        raise RuntimeError(
            "Source-run mismatch for "
            + expected_artifact_type
            + "."
        )

    if (
        metadata.get(
            "v0_1_run_id"
        )
        != V0_1_RUN_ID
    ):
        raise RuntimeError(
            "V0.1 run-ID mismatch for "
            + expected_artifact_type
            + "."
        )

    expected_payload_hash = str(
        metadata.get(
            "payload_sha256"
        )
    )

    observed_payload_hash = (
        canonical_json_sha256(
            payload
        )
    )

    if (
        expected_payload_hash
        != observed_payload_hash
    ):
        raise RuntimeError(
            "Payload-hash mismatch for "
            + expected_artifact_type
            + "."
        )

    return metadata, payload


def verified_manifest_artifact_path(
    registry: pd.DataFrame,
    artifact_type: str,
) -> Path:
    matches = registry.loc[
        registry[
            "artifact_type"
        ].eq(artifact_type)
    ]

    if len(matches) != 1:
        raise RuntimeError(
            "Expected exactly one manifested "
            + artifact_type
            + " artifact; found "
            + str(len(matches))
            + "."
        )

    row = matches.iloc[0]

    relative_path = row[
        "relative_path"
    ]

    if pd.isna(relative_path):
        raise RuntimeError(
            artifact_type
            + " lacks relative_path."
        )

    artifact_path = (
        V0_1_ROOT_RESOLVED
        / str(relative_path)
    ).resolve(strict=True)

    if not path_is_within(
        artifact_path,
        V0_1_ROOT_RESOLVED,
    ):
        raise RuntimeError(
            artifact_type
            + " is outside V0.1."
        )

    if path_is_within(
        artifact_path,
        V0_0_ROOT_RESOLVED,
    ):
        raise RuntimeError(
            artifact_type
            + " enters immutable V0.0."
        )

    expected_size = int(
        row["size_bytes"]
    )

    expected_hash = str(
        row["sha256"]
    ).lower()

    observed_size = int(
        artifact_path.stat().st_size
    )

    observed_hash = sha256_file(
        artifact_path
    )

    if observed_size != expected_size:
        raise RuntimeError(
            "Size mismatch for "
            + artifact_type
            + "."
        )

    if observed_hash != expected_hash:
        raise RuntimeError(
            "SHA-256 mismatch for "
            + artifact_type
            + "."
        )

    return artifact_path


def load_manifested_json_artifact(
    registry: pd.DataFrame,
    artifact_type: str,
) -> tuple[Path, dict, dict, dict]:
    artifact_path = (
        verified_manifest_artifact_path(
            registry=registry,
            artifact_type=artifact_type,
        )
    )

    document = load_json_mapping(
        artifact_path
    )

    metadata, payload = (
        unwrap_saved_artifact(
            document=document,
            expected_artifact_type=(
                artifact_type
            ),
        )
    )

    return (
        artifact_path,
        document,
        metadata,
        payload,
    )


def load_manifested_csv_artifact(
    registry: pd.DataFrame,
    artifact_type: str,
) -> tuple[Path, pd.DataFrame]:
    artifact_path = (
        verified_manifest_artifact_path(
            registry=registry,
            artifact_type=artifact_type,
        )
    )

    frame = pd.read_csv(
        artifact_path
    )

    return artifact_path, frame


# ============================================================
# DETERMINISTIC EXECUTION
# ============================================================

random.seed(
    RANDOM_SEED
)

np.random.seed(
    RANDOM_SEED
)

os.environ[
    "PYTHONHASHSEED"
] = str(
    RANDOM_SEED
)


# ============================================================
# NOTEBOOK 00 AND NOTEBOOK 01 OUTPUT MANIFEST PATHS
# ============================================================

NOTEBOOK_00_MANIFEST_NAME = (
    SOURCE_RUN_PREFIX
    + "__"
    + V0_1_RUN_ID
    + "__00_V01_RUN_CONTRACT__"
    + "notebook_00_output_manifest.json"
)

NOTEBOOK_01_MANIFEST_NAME = (
    SOURCE_RUN_PREFIX
    + "__"
    + V0_1_RUN_ID
    + "__01_RAW_DATA_AUDIT__"
    + "notebook_01_output_manifest.json"
)


NOTEBOOK_00_MANIFEST_PATH = (
    V0_1_MANIFEST_ROOT
    / NOTEBOOK_00_MANIFEST_NAME
).resolve(strict=True)

NOTEBOOK_01_MANIFEST_PATH = (
    V0_1_MANIFEST_ROOT
    / NOTEBOOK_01_MANIFEST_NAME
).resolve(strict=True)


NOTEBOOK_00_MANIFEST_FILE_HASH = (
    sha256_file(
        NOTEBOOK_00_MANIFEST_PATH
    )
)

NOTEBOOK_01_MANIFEST_FILE_HASH = (
    sha256_file(
        NOTEBOOK_01_MANIFEST_PATH
    )
)


if (
    NOTEBOOK_00_MANIFEST_FILE_HASH
    != EXPECTED_NOTEBOOK_00_MANIFEST_SHA256
):
    raise RuntimeError(
        "Notebook 00 output-manifest hash mismatch."
    )


if (
    NOTEBOOK_01_MANIFEST_FILE_HASH
    != EXPECTED_NOTEBOOK_01_MANIFEST_SHA256
):
    raise RuntimeError(
        "Notebook 01 output-manifest hash mismatch."
    )


# ============================================================
# LOAD AND UNWRAP BOTH OUTPUT MANIFESTS
# ============================================================

NOTEBOOK_00_MANIFEST_DOCUMENT = (
    load_json_mapping(
        NOTEBOOK_00_MANIFEST_PATH
    )
)

NOTEBOOK_01_MANIFEST_DOCUMENT = (
    load_json_mapping(
        NOTEBOOK_01_MANIFEST_PATH
    )
)


(
    NOTEBOOK_00_MANIFEST_METADATA,
    NOTEBOOK_00_MANIFEST_PAYLOAD,
) = unwrap_saved_artifact(
    document=(
        NOTEBOOK_00_MANIFEST_DOCUMENT
    ),
    expected_artifact_type=(
        "NOTEBOOK_00_OUTPUT_MANIFEST"
    ),
)


(
    NOTEBOOK_01_MANIFEST_METADATA,
    NOTEBOOK_01_MANIFEST_PAYLOAD,
) = unwrap_saved_artifact(
    document=(
        NOTEBOOK_01_MANIFEST_DOCUMENT
    ),
    expected_artifact_type=(
        "NOTEBOOK_01_OUTPUT_MANIFEST"
    ),
)


if (
    NOTEBOOK_01_MANIFEST_PAYLOAD.get(
        "next_notebook"
    )
    != NOTEBOOK_FILENAME
):
    raise RuntimeError(
        "Notebook 01 did not hand off to "
        + NOTEBOOK_FILENAME
        + "."
    )


if not bool(
    NOTEBOOK_01_MANIFEST_PAYLOAD.get(
        "notebook_02_authorized",
        False,
    )
):
    raise RuntimeError(
        "Notebook 02 was not authorized."
    )


# ============================================================
# MANIFEST REGISTRIES
# ============================================================

notebook_00_artifact_records = (
    NOTEBOOK_00_MANIFEST_PAYLOAD.get(
        "artifacts"
    )
)

notebook_01_artifact_records = (
    NOTEBOOK_01_MANIFEST_PAYLOAD.get(
        "artifacts"
    )
)


if not isinstance(
    notebook_00_artifact_records,
    list,
):
    raise RuntimeError(
        "Notebook 00 manifest lacks an artifact list."
    )


if not isinstance(
    notebook_01_artifact_records,
    list,
):
    raise RuntimeError(
        "Notebook 01 manifest lacks an artifact list."
    )


NOTEBOOK_00_ARTIFACT_REGISTRY = (
    pd.DataFrame(
        notebook_00_artifact_records
    )
)

NOTEBOOK_01_ARTIFACT_REGISTRY = (
    pd.DataFrame(
        notebook_01_artifact_records
    )
)


required_registry_columns = {
    "artifact_type",
    "format",
    "relative_path",
    "size_bytes",
    "sha256",
}


for registry_name, registry in (
    (
        "NOTEBOOK_00_ARTIFACT_REGISTRY",
        NOTEBOOK_00_ARTIFACT_REGISTRY,
    ),
    (
        "NOTEBOOK_01_ARTIFACT_REGISTRY",
        NOTEBOOK_01_ARTIFACT_REGISTRY,
    ),
):
    missing_columns = sorted(
        required_registry_columns
        - set(registry.columns)
    )

    if missing_columns:
        raise RuntimeError(
            registry_name
            + " is missing columns: "
            + " | ".join(
                missing_columns
            )
        )


# ============================================================
# LOAD NOTEBOOK 00 CONTRACTS
# ============================================================

(
    RUN_CONFIG_PATH,
    RUN_CONFIG_DOCUMENT,
    RUN_CONFIG_METADATA,
    V0_1_RUN_CONFIG,
) = load_manifested_json_artifact(
    registry=(
        NOTEBOOK_00_ARTIFACT_REGISTRY
    ),
    artifact_type=(
        "V0_1_RUN_CONFIG"
    ),
)


(
    MARKET_DATA_CONTRACT_PATH,
    MARKET_DATA_CONTRACT_DOCUMENT,
    MARKET_DATA_CONTRACT_METADATA,
    MARKET_DATA_CONTRACT,
) = load_manifested_json_artifact(
    registry=(
        NOTEBOOK_00_ARTIFACT_REGISTRY
    ),
    artifact_type=(
        "MARKET_DATA_CONTRACT"
    ),
)


(
    REJECTION_CONTRACT_PATH,
    REJECTION_CONTRACT_DOCUMENT,
    REJECTION_CONTRACT_METADATA,
    MISSING_DATA_REJECTION_CONTRACT,
) = load_manifested_json_artifact(
    registry=(
        NOTEBOOK_00_ARTIFACT_REGISTRY
    ),
    artifact_type=(
        "MISSING_DATA_REJECTION_CONTRACT"
    ),
)


(
    SPLIT_CONTRACT_PATH,
    SPLIT_CONTRACT_DOCUMENT,
    SPLIT_CONTRACT_METADATA,
    CHRONOLOGICAL_SPLIT_CONTRACT,
) = load_manifested_json_artifact(
    registry=(
        NOTEBOOK_00_ARTIFACT_REGISTRY
    ),
    artifact_type=(
        "CHRONOLOGICAL_SPLIT_CONTRACT"
    ),
)


(
    DECISION_CONTRACT_PATH,
    DECISION_CONTRACT_DOCUMENT,
    DECISION_CONTRACT_METADATA,
    DECISION_AND_TOLERANCE_CONTRACT,
) = load_manifested_json_artifact(
    registry=(
        NOTEBOOK_00_ARTIFACT_REGISTRY
    ),
    artifact_type=(
        "DECISION_AND_TOLERANCE_CONTRACT"
    ),
)


(
    RUN_IDENTITY_PATH,
    RUN_IDENTITY_DOCUMENT,
    RUN_IDENTITY_METADATA,
    V0_1_RUN_IDENTITY,
) = load_manifested_json_artifact(
    registry=(
        NOTEBOOK_00_ARTIFACT_REGISTRY
    ),
    artifact_type=(
        "V0_1_RUN_IDENTITY"
    ),
)


# ============================================================
# LOAD NOTEBOOK 01 RAW-AUDIT AND HANDOFF ARTIFACTS
# ============================================================

(
    RAW_DATA_AUDIT_PATH,
    RAW_DATA_AUDIT_DOCUMENT,
    RAW_DATA_AUDIT_METADATA,
    V0_1_RAW_DATA_AUDIT,
) = load_manifested_json_artifact(
    registry=(
        NOTEBOOK_01_ARTIFACT_REGISTRY
    ),
    artifact_type=(
        "V0_1_RAW_DATA_AUDIT"
    ),
)


(
    NOTEBOOK_01_HANDOFF_PATH,
    NOTEBOOK_01_HANDOFF_DOCUMENT,
    NOTEBOOK_01_HANDOFF_METADATA,
    NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF,
) = load_manifested_json_artifact(
    registry=(
        NOTEBOOK_01_ARTIFACT_REGISTRY
    ),
    artifact_type=(
        "NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF"
    ),
)


(
    RAW_SOURCE_MANIFEST_PATH,
    AUTHORITATIVE_RAW_SOURCE_MANIFEST,
) = load_manifested_csv_artifact(
    registry=(
        NOTEBOOK_01_ARTIFACT_REGISTRY
    ),
    artifact_type=(
        "AUTHORITATIVE_RAW_SOURCE_MANIFEST"
    ),
)


(
    SOURCE_REHASH_PATH,
    NOTEBOOK_01_SOURCE_REHASH,
) = load_manifested_csv_artifact(
    registry=(
        NOTEBOOK_01_ARTIFACT_REGISTRY
    ),
    artifact_type=(
        "NOTEBOOK_01_SOURCE_REHASH"
    ),
)


(
    DEPTH_UPDATE_ID_AUDIT_PATH,
    DEPTH_UPDATE_ID_AUDIT,
) = load_manifested_csv_artifact(
    registry=(
        NOTEBOOK_01_ARTIFACT_REGISTRY
    ),
    artifact_type=(
        "DEPTH_UPDATE_ID_AUDIT"
    ),
)


# ============================================================
# VERIFY SAVED PAYLOAD IDENTITIES
# ============================================================

observed_raw_data_audit_hash = (
    canonical_json_sha256(
        V0_1_RAW_DATA_AUDIT
    )
)

observed_handoff_hash = (
    canonical_json_sha256(
        NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF
    )
)


if (
    observed_raw_data_audit_hash
    != EXPECTED_RAW_DATA_AUDIT_HASH
):
    raise RuntimeError(
        "Raw-data audit payload hash mismatch."
    )


if (
    observed_handoff_hash
    != EXPECTED_NOTEBOOK_01_HANDOFF_HASH
):
    raise RuntimeError(
        "Notebook 01 handoff payload hash mismatch."
    )


for identity_name, observed_value, expected_value in (
    (
        "source_set_hash",
        NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF.get(
            "source_set_hash"
        ),
        EXPECTED_SOURCE_SET_HASH,
    ),
    (
        "v0_1_run_config_hash",
        NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF.get(
            "v0_1_run_config_hash"
        ),
        EXPECTED_RUN_CONFIG_HASH,
    ),
    (
        "v0_1_run_identity_hash",
        NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF.get(
            "v0_1_run_identity_hash"
        ),
        EXPECTED_RUN_IDENTITY_HASH,
    ),
    (
        "raw_data_audit_hash",
        NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF.get(
            "raw_data_audit_hash"
        ),
        EXPECTED_RAW_DATA_AUDIT_HASH,
    ),
):
    if observed_value != expected_value:
        raise RuntimeError(
            "Handoff identity mismatch for "
            + identity_name
            + "."
        )


# ============================================================
# REGISTER AUTHORITATIVE RAW SOURCES
# ============================================================

required_source_columns = {
    "source_role",
    "resolved_path",
    "size_bytes",
    "sha256",
}


missing_source_columns = sorted(
    required_source_columns
    - set(
        AUTHORITATIVE_RAW_SOURCE_MANIFEST.columns
    )
)


if missing_source_columns:
    raise RuntimeError(
        "Authoritative source manifest is missing columns: "
        + " | ".join(
            missing_source_columns
        )
    )


def authoritative_source_path(
    source_role: str,
) -> Path:
    matches = (
        AUTHORITATIVE_RAW_SOURCE_MANIFEST.loc[
            AUTHORITATIVE_RAW_SOURCE_MANIFEST[
                "source_role"
            ].eq(source_role)
        ]
    )

    if len(matches) != 1:
        raise RuntimeError(
            "Expected exactly one "
            + source_role
            + " source; found "
            + str(len(matches))
            + "."
        )

    path = Path(
        matches.iloc[0][
            "resolved_path"
        ]
    ).resolve(strict=True)

    if not path_is_within(
        path,
        V0_0_ROOT_RESOLVED,
    ):
        raise RuntimeError(
            source_role
            + " is outside immutable V0.0."
        )

    return path


TRADE_SOURCE_PATH = (
    authoritative_source_path(
        "TRADE_STREAM"
    )
)

DEPTH_SOURCE_PATH = (
    authoritative_source_path(
        "DEPTH_STREAM"
    )
)

REST_SNAPSHOT_PATH = (
    authoritative_source_path(
        "REST_SNAPSHOT"
    )
)

COLLECTOR_METADATA_PATH = (
    authoritative_source_path(
        "COLLECTOR_METADATA"
    )
)

RUN_MANIFEST_PATH = (
    authoritative_source_path(
        "RUN_MANIFEST"
    )
)


# ============================================================
# FINAL SOURCE RE-HASH
# ============================================================

notebook_02_source_rehash_rows = []


for source_row in (
    AUTHORITATIVE_RAW_SOURCE_MANIFEST
    .itertuples(index=False)
):
    source_path = Path(
        source_row.resolved_path
    ).resolve(strict=True)

    observed_size = int(
        source_path.stat().st_size
    )

    observed_hash = sha256_file(
        source_path
    )

    expected_size = int(
        source_row.size_bytes
    )

    expected_hash = str(
        source_row.sha256
    ).lower()

    source_passed = bool(
        observed_size == expected_size
        and observed_hash == expected_hash
        and path_is_within(
            source_path,
            V0_0_ROOT_RESOLVED,
        )
        and not path_is_within(
            source_path,
            V0_1_ROOT_RESOLVED,
        )
    )

    notebook_02_source_rehash_rows.append(
        {
            "source_role": (
                source_row.source_role
            ),
            "resolved_path": str(
                source_path
            ),
            "expected_size_bytes": (
                expected_size
            ),
            "observed_size_bytes": (
                observed_size
            ),
            "size_matches": bool(
                observed_size
                == expected_size
            ),
            "expected_sha256": (
                expected_hash
            ),
            "observed_sha256": (
                observed_hash
            ),
            "hash_matches": bool(
                observed_hash
                == expected_hash
            ),
            "source_passed": (
                source_passed
            ),
        }
    )


NOTEBOOK_02_SOURCE_REHASH = (
    pd.DataFrame(
        notebook_02_source_rehash_rows
    )
)


# ============================================================
# EXTRACT VISIBLE-BOOK RECONSTRUCTION CONTRACT
# ============================================================

SNAPSHOT_CONTRACT = (
    NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF.get(
        "snapshot_contract"
    )
)

DEPTH_RECONSTRUCTION_CONTRACT = (
    NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF.get(
        "depth_reconstruction_contract"
    )
)


if not isinstance(
    SNAPSHOT_CONTRACT,
    dict,
):
    raise RuntimeError(
        "Notebook 01 handoff lacks snapshot_contract."
    )


if not isinstance(
    DEPTH_RECONSTRUCTION_CONTRACT,
    dict,
):
    raise RuntimeError(
        "Notebook 01 handoff lacks "
        "depth_reconstruction_contract."
    )


SNAPSHOT_LAST_UPDATE_ID = int(
    SNAPSHOT_CONTRACT[
        "snapshot_last_update_id"
    ]
)

SNAPSHOT_SESSION_ID = str(
    SNAPSHOT_CONTRACT[
        "snapshot_session_id"
    ]
)

FIRST_APPLICABLE_DEPTH_POSITION = int(
    DEPTH_RECONSTRUCTION_CONTRACT[
        "first_applicable_depth_position"
    ]
)

FIRST_APPLICABLE_DEPTH_COLLECTOR_SEQUENCE = int(
    DEPTH_RECONSTRUCTION_CONTRACT[
        "first_applicable_collector_sequence"
    ]
)

FIRST_APPLICABLE_DEPTH_U = int(
    DEPTH_RECONSTRUCTION_CONTRACT[
        "first_applicable_U"
    ]
)

FIRST_APPLICABLE_DEPTH_u = int(
    DEPTH_RECONSTRUCTION_CONTRACT[
        "first_applicable_u"
    ]
)

STALE_PRE_SNAPSHOT_DEPTH_EVENT_COUNT = int(
    DEPTH_RECONSTRUCTION_CONTRACT[
        "stale_pre_snapshot_event_count"
    ]
)

TOTAL_DEPTH_EVENT_COUNT = int(
    DEPTH_RECONSTRUCTION_CONTRACT[
        "total_depth_event_count"
    ]
)

POST_SNAPSHOT_DEPTH_EVENT_COUNT = int(
    DEPTH_RECONSTRUCTION_CONTRACT[
        "post_snapshot_event_count"
    ]
)

POST_SNAPSHOT_GAP_COUNT = int(
    DEPTH_RECONSTRUCTION_CONTRACT[
        "post_snapshot_gap_count"
    ]
)

POST_SNAPSHOT_OVERLAP_COUNT = int(
    DEPTH_RECONSTRUCTION_CONTRACT[
        "post_snapshot_overlap_count"
    ]
)

ZERO_QUANTITY_ACTION = str(
    DEPTH_RECONSTRUCTION_CONTRACT[
        "zero_quantity_action"
    ]
)

PRIMARY_ORDERING_AUTHORITY = str(
    DEPTH_RECONSTRUCTION_CONTRACT[
        "ordering_authority"
    ]
)

CANONICAL_TIMEZONE = str(
    DEPTH_RECONSTRUCTION_CONTRACT[
        "timezone"
    ]
)


# ============================================================
# OPERATING AUTHORITY
# ============================================================

OPERATING_AUTHORITY = (
    V0_1_RUN_CONFIG.get(
        "operating_authority"
    )
)


if not isinstance(
    OPERATING_AUTHORITY,
    dict,
):
    raise RuntimeError(
        "Run config lacks operating_authority."
    )


V0_1_OPERATING_MODE = str(
    OPERATING_AUTHORITY[
        "operating_mode"
    ]
)

V0_1_CONTRACT_STATUS = str(
    OPERATING_AUTHORITY[
        "contract_status"
    ]
)


# ============================================================
# BOOTSTRAP GATES
# ============================================================

NOTEBOOK_02_BOOTSTRAP_GATES = pd.DataFrame(
    [
        {
            "gate": (
                "notebook_00_manifest_verified"
            ),
            "passed": bool(
                NOTEBOOK_00_MANIFEST_FILE_HASH
                == EXPECTED_NOTEBOOK_00_MANIFEST_SHA256
            ),
            "evidence": (
                NOTEBOOK_00_MANIFEST_FILE_HASH
            ),
        },
        {
            "gate": (
                "notebook_01_manifest_verified"
            ),
            "passed": bool(
                NOTEBOOK_01_MANIFEST_FILE_HASH
                == EXPECTED_NOTEBOOK_01_MANIFEST_SHA256
            ),
            "evidence": (
                NOTEBOOK_01_MANIFEST_FILE_HASH
            ),
        },
        {
            "gate": (
                "raw_data_audit_hash_verified"
            ),
            "passed": bool(
                observed_raw_data_audit_hash
                == EXPECTED_RAW_DATA_AUDIT_HASH
            ),
            "evidence": (
                observed_raw_data_audit_hash
            ),
        },
        {
            "gate": (
                "notebook_01_handoff_hash_verified"
            ),
            "passed": bool(
                observed_handoff_hash
                == EXPECTED_NOTEBOOK_01_HANDOFF_HASH
            ),
            "evidence": (
                observed_handoff_hash
            ),
        },
        {
            "gate": (
                "notebook_02_authorized"
            ),
            "passed": bool(
                NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF.get(
                    "notebook_02_authorized",
                    False,
                )
            ),
            "evidence": (
                "authorized="
                + str(
                    NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF.get(
                        "notebook_02_authorized",
                        False,
                    )
                )
            ),
        },
        {
            "gate": (
                "authoritative_sources_unchanged"
            ),
            "passed": bool(
                NOTEBOOK_02_SOURCE_REHASH[
                    "source_passed"
                ].all()
            ),
            "evidence": (
                "verified="
                + str(
                    int(
                        NOTEBOOK_02_SOURCE_REHASH[
                            "source_passed"
                        ].sum()
                    )
                )
                + "/"
                + str(
                    len(
                        NOTEBOOK_02_SOURCE_REHASH
                    )
                )
            ),
        },
        {
            "gate": (
                "snapshot_identity_preserved"
            ),
            "passed": bool(
                SNAPSHOT_LAST_UPDATE_ID
                == 97233590166
            ),
            "evidence": (
                "lastUpdateId="
                + str(
                    SNAPSHOT_LAST_UPDATE_ID
                )
            ),
        },
        {
            "gate": (
                "first_depth_event_bridges_snapshot"
            ),
            "passed": bool(
                FIRST_APPLICABLE_DEPTH_U
                <= SNAPSHOT_LAST_UPDATE_ID + 1
                <= FIRST_APPLICABLE_DEPTH_u
            ),
            "evidence": (
                "successor="
                + str(
                    SNAPSHOT_LAST_UPDATE_ID
                    + 1
                )
                + "; U="
                + str(
                    FIRST_APPLICABLE_DEPTH_U
                )
                + "; u="
                + str(
                    FIRST_APPLICABLE_DEPTH_u
                )
            ),
        },
        {
            "gate": (
                "post_snapshot_depth_contiguous"
            ),
            "passed": bool(
                POST_SNAPSHOT_GAP_COUNT == 0
                and POST_SNAPSHOT_OVERLAP_COUNT == 0
            ),
            "evidence": (
                "events="
                + str(
                    POST_SNAPSHOT_DEPTH_EVENT_COUNT
                )
                + "; gaps="
                + str(
                    POST_SNAPSHOT_GAP_COUNT
                )
                + "; overlaps="
                + str(
                    POST_SNAPSHOT_OVERLAP_COUNT
                )
            ),
        },
        {
            "gate": (
                "depth_event_counts_reconcile"
            ),
            "passed": bool(
                STALE_PRE_SNAPSHOT_DEPTH_EVENT_COUNT
                + POST_SNAPSHOT_DEPTH_EVENT_COUNT
                == TOTAL_DEPTH_EVENT_COUNT
            ),
            "evidence": (
                "stale="
                + str(
                    STALE_PRE_SNAPSHOT_DEPTH_EVENT_COUNT
                )
                + "; applicable="
                + str(
                    POST_SNAPSHOT_DEPTH_EVENT_COUNT
                )
                + "; total="
                + str(
                    TOTAL_DEPTH_EVENT_COUNT
                )
            ),
        },
        {
            "gate": (
                "zero_quantity_policy_preserved"
            ),
            "passed": bool(
                ZERO_QUANTITY_ACTION
                == "DELETE_PRICE_LEVEL"
            ),
            "evidence": (
                ZERO_QUANTITY_ACTION
            ),
        },
        {
            "gate": (
                "ordering_authority_preserved"
            ),
            "passed": bool(
                PRIMARY_ORDERING_AUTHORITY
                == "collector_sequence"
            ),
            "evidence": (
                PRIMARY_ORDERING_AUTHORITY
            ),
        },
        {
            "gate": (
                "canonical_timezone_preserved"
            ),
            "passed": bool(
                CANONICAL_TIMEZONE
                == "UTC"
            ),
            "evidence": (
                CANONICAL_TIMEZONE
            ),
        },
        {
            "gate": (
                "operating_mode_preserved"
            ),
            "passed": bool(
                V0_1_OPERATING_MODE
                == "ENGINEERING_REPRODUCTION_MODE"
            ),
            "evidence": (
                V0_1_OPERATING_MODE
            ),
        },
    ]
)


NOTEBOOK_02_BOOTSTRAP_SUMMARY = (
    pd.DataFrame(
        [
            {
                "field": "pipeline_name",
                "value": PIPELINE_NAME,
            },
            {
                "field": "pipeline_version",
                "value": PIPELINE_VERSION,
            },
            {
                "field": "notebook",
                "value": NOTEBOOK_FILENAME,
            },
            {
                "field": "source_run_prefix",
                "value": SOURCE_RUN_PREFIX,
            },
            {
                "field": "v0_1_run_id",
                "value": V0_1_RUN_ID,
            },
            {
                "field": "operating_mode",
                "value": V0_1_OPERATING_MODE,
            },
            {
                "field": "contract_status",
                "value": V0_1_CONTRACT_STATUS,
            },
            {
                "field": "snapshot_last_update_id",
                "value": (
                    SNAPSHOT_LAST_UPDATE_ID
                ),
            },
            {
                "field": (
                    "first_applicable_depth_sequence"
                ),
                "value": (
                    FIRST_APPLICABLE_DEPTH_COLLECTOR_SEQUENCE
                ),
            },
            {
                "field": (
                    "first_applicable_depth_U"
                ),
                "value": (
                    FIRST_APPLICABLE_DEPTH_U
                ),
            },
            {
                "field": (
                    "first_applicable_depth_u"
                ),
                "value": (
                    FIRST_APPLICABLE_DEPTH_u
                ),
            },
            {
                "field": (
                    "post_snapshot_depth_events"
                ),
                "value": (
                    POST_SNAPSHOT_DEPTH_EVENT_COUNT
                ),
            },
            {
                "field": (
                    "zero_quantity_action"
                ),
                "value": (
                    ZERO_QUANTITY_ACTION
                ),
            },
            {
                "field": (
                    "notebook_started_utc"
                ),
                "value": (
                    NOTEBOOK_02_STARTED_UTC
                ),
            },
        ]
    )
)


# ============================================================
# DISPLAY AND ENFORCE
# ============================================================

display(
    NOTEBOOK_02_BOOTSTRAP_SUMMARY
)

display(
    NOTEBOOK_02_SOURCE_REHASH[
        [
            "source_role",
            "resolved_path",
            "size_matches",
            "hash_matches",
            "source_passed",
        ]
    ]
)

display(
    NOTEBOOK_02_BOOTSTRAP_GATES
)


failed_bootstrap_gates = (
    NOTEBOOK_02_BOOTSTRAP_GATES.loc[
        ~NOTEBOOK_02_BOOTSTRAP_GATES[
            "passed"
        ]
    ]
)


if not failed_bootstrap_gates.empty:
    failure_parts = []

    for row in (
        failed_bootstrap_gates
        .itertuples(index=False)
    ):
        failure_parts.append(
            str(row.gate)
            + ": "
            + str(row.evidence)
        )

    raise RuntimeError(
        "Notebook 02 bootstrap failed: "
        + ", ".join(
            failure_parts
        )
    )


print(
    "Notebook 02 clean-kernel bootstrap: PASS"
)

print(
    "Notebook 00 manifest SHA-256: "
    + NOTEBOOK_00_MANIFEST_FILE_HASH
)

print(
    "Notebook 01 manifest SHA-256: "
    + NOTEBOOK_01_MANIFEST_FILE_HASH
)

print(
    "Raw-data audit SHA-256: "
    + observed_raw_data_audit_hash
)

print(
    "Notebook 01 handoff SHA-256: "
    + observed_handoff_hash
)

print(
    "Authoritative source files verified: "
    + str(
        int(
            NOTEBOOK_02_SOURCE_REHASH[
                "source_passed"
            ].sum()
        )
    )
    + "/"
    + str(
        len(
            NOTEBOOK_02_SOURCE_REHASH
        )
    )
)

print(
    "Snapshot lastUpdateId: "
    + format(
        SNAPSHOT_LAST_UPDATE_ID,
        ",",
    )
)

print(
    "First applicable depth event: "
    + "sequence="
    + format(
        FIRST_APPLICABLE_DEPTH_COLLECTOR_SEQUENCE,
        ",",
    )
    + "; U="
    + format(
        FIRST_APPLICABLE_DEPTH_U,
        ",",
    )
    + "; u="
    + format(
        FIRST_APPLICABLE_DEPTH_u,
        ",",
    )
)

print(
    "Post-snapshot depth events: "
    + format(
        POST_SNAPSHOT_DEPTH_EVENT_COUNT,
        ",",
    )
)

print(
    "Notebook status: "
    + NOTEBOOK_STATUS
)